# FPL VORTEX — Evidence-based Wildcard / Free Hit, fast Draft

Cell 0 exposes **exactly three controls**: Output (Draft/Final), Chips (Free Hit/Wildcard/Both), and Selection (Auto/Manual).

The model now blends season and recent official FPL xG/xA, uses current BPS as a bounded bonus proxy, and smooths observed DEFCON hit probabilities. These are statistical forecasts, not guaranteed points or a fully backtested match-level BPS simulator. Optimizer, scripts, pitch cards, fixture and captaincy slides read from the same refreshed projection contract. A per-chip `Projection Evidence.json` is saved with the scripts.

**Draft animation:** Narration-aligned section frames are cached and animated with lightweight pan/zoom at 480p/5fps. Draft keeps the complete Ryan narration, individual slide MP4s and a combined MP4, plus MP3/TXT exports. Final retains full scene and object animation, 4K output and four 4K PNGs.

For Auto: run the notebook in sequence. For Manual: run through Cell 10, save the selected 15 players, then continue Cells 11–17. Check Cell 17 for verified persistent Google Drive files.


In [1]:
#@title 🔴 CELL 0 - MANUAL SETUP {display-mode: "form"}
# 🔴 CELL 0 - MANUAL SETUP
VX_VIDEO_MODE = "Draft" #@param ["Draft", "Final"]
VX_CHIPS = "Both" #@param ["Free Hit", "Wildcard", "Both"]
VX_SELECTION_MODE = "Auto" #@param ["Auto", "Manual"]

# Automatic rendering controls: no preview-only or other manual settings.
VX_BACKGROUND_THEME = "auto"
VX_BACKGROUND_THEME_SEED = 0
VX_RENDER_VIDEOS = True
VX_INTRO_PREVIEW_ONLY = False
VX_INTRO_PREVIEW_CHIP = "Free Hit"
VX_DRAFT_FAST_MODE = True
VX_FINAL_FPS = 24 #@param [24, 30]
VX_SAVE_4K_PNG = VX_VIDEO_MODE == "Final"  # Final alone exports four native 3840x2160 slide PNGs.
VX_CACHE_POLICY = "Reuse cache"
VX_INCLUDE_POPUPS = VX_VIDEO_MODE == "Final"
VX_GENERATE_CHIPS = VX_CHIPS
VX_NARRATION_POLICY = "Keep edits"
VX_ASSETS_FOLDER_URL = "https://drive.google.com/drive/folders/1h3Loznr5RnsMyh-kUd-zMxEgOf98uFSt"
VX_WILDCARD_SELECTION_MODE = VX_SELECTION_MODE
VX_FREEHIT_SELECTION_MODE = VX_SELECTION_MODE
VX_WILDCARD_HORIZON = 6  # primary Wildcard horizon
VX_FPL_ENTRY_ID = "145784"  # User-confirmed FPL team; not another form input.
# Wildcard account inputs: 0 / -1 means unknown, never an invented account value.
VX_WC_BUDGET = 0  # resolve the available budget from current FPL rules
VX_WC_BENCH_BOOST = "Auto"
VX_WC_BB_GAMEWEEK = 0
VX_WC_FREE_TRANSFERS = -1
VX_WC_FDR_POLICY = "Strict"
VX_DAY3_FORECAST_PIN = "31e088378736a0f622eb51e7da6b3e2ab18ba55b"
VX_WC_NEWS_ENABLED = True
VX_WC_INTERNATIONAL_ENABLED = True
VX_WC_PRIMARY_NEWS_URLS = "Fpl.com"
VX_WC_SOLVER_SECONDS = 60
# Local speech keeps narration on this runtime. Microsoft mode sends its text
# to the same speech service used by the original notebook.
VX_WC_TTS_PROVIDER = "Microsoft speech (sends narration text)"
VX_TTS_VOICE = "en-GB-RyanNeural"
# Ryan presets requested by the user. Auto selects a profile from the scene data.
VX_NARRATION_PROFILE = "Default FPL Vortex"
VX_NARRATION_PROFILES = {
    'Default FPL Vortex': {'rate':'+7%','pitch':'+0Hz','volume':'+0%'},
    'Data-heavy player section': {'rate':'+7%','pitch':'+0Hz','volume':'+0%'},
    'Hook / captain reveal': {'rate':'+7%','pitch':'+0Hz','volume':'+0%'},
    'Careful uncertainty section': {'rate':'+7%','pitch':'+0Hz','volume':'+0%'},
    'Fast short-form video': {'rate':'+7%','pitch':'+0Hz','volume':'+0%'},
}
VX_RYAN_RATE = VX_NARRATION_PROFILES['Default FPL Vortex']['rate']
VX_RYAN_PITCH_HZ = 0
VX_VIDEO_MOODS = {
    'intro':VX_NARRATION_PROFILES['Hook / captain reveal'],
    'analysis':VX_NARRATION_PROFILES['Default FPL Vortex'],
    'key_stat':VX_NARRATION_PROFILES['Data-heavy player section'],
    'warning':VX_NARRATION_PROFILES['Careful uncertainty section'],
    'outro':VX_NARRATION_PROFILES['Hook / captain reveal'],
    'short':VX_NARRATION_PROFILES['Fast short-form video'],
}
VX_GENERATE_WILDCARD = VX_GENERATE_CHIPS in ("Wildcard", "Both")
VX_GENERATE_FREEHIT = VX_GENERATE_CHIPS in ("Free Hit", "Both")
USE_MANUAL_WILDCARD = VX_GENERATE_WILDCARD and VX_WILDCARD_SELECTION_MODE == "Manual"
USE_MANUAL_FREEHIT = VX_GENERATE_FREEHIT and VX_FREEHIT_SELECTION_MODE == "Manual"
# Re-running Cell 0 intentionally invalidates any earlier manual save.
VX_MANUAL_SELECTION_SAVED = False
VX_MANUAL_SAVED_VALUES = {}
VX_CELL0C_COMPLETE = False
VX_CELL1_COMPLETE = False
VX_CELL2_COMPLETE = False
"""Output profiles, durable editable scripts, and private Drive assets."""
import os
import re
import json
import hashlib
import shutil
import subprocess
from pathlib import Path

def final_video_encoder():
    """Use verified native acceleration on macOS; Colab keeps the software encoder."""
    import sys
    if sys.platform!='darwin':return 'libx264'
    if '_VX_FINAL_ENCODER' in globals():return globals()['_VX_FINAL_ENCODER']
    result=subprocess.run([ffmpeg_binary(),'-hide_banner','-loglevel','error','-f','lavfi','-i','color=c=black:s=3840x2160:r=30','-frames:v','1','-c:v','h264_videotoolbox','-b:v','40000000','-pix_fmt','yuv420p','-f','null','-'],capture_output=True,timeout=20)
    globals()['_VX_FINAL_ENCODER']='h264_videotoolbox' if result.returncode==0 else 'libx264'
    return globals()['_VX_FINAL_ENCODER']

def video_encode_args(args,profile=None):
    profile=profile or globals().get('VX_VIDEO_PROFILE',{})
    if profile.get('encoder')!='h264_videotoolbox':return list(args)
    result=[];i=0
    while i<len(args):
        key=str(args[i])
        if key=='-c:v' and i+1<len(args) and str(args[i+1])=='libx264':
            result.extend(['-c:v','h264_videotoolbox','-b:v','40000000','-profile:v','high','-spatial_aq','1']);i+=2
        elif key in ('-preset','-crf','-tune'):i+=2
        else:result.append(args[i]);i+=1
    return result

def video_profile(mode):
    if mode == 'Draft':
        # 854x480 with SAR 1280/1281 produces an exact 16:9 display aspect ratio.
        fast=bool(globals().get('VX_DRAFT_FAST_MODE',True))
        return dict(width=854,height=480,fps=5,crf=28,preset='ultrafast' if fast else 'veryfast',
                    sar='1280/1281',encoder='libx264',static_preview=False,
                    light_motion=True,scene_crossfades=False,draft_keyframes=True)
    if mode == 'Final':
        final_fps=int(globals().get('VX_FINAL_FPS',24))
        if final_fps not in (24,30):raise ValueError('Final FPS must be 24 or 30')
        return dict(width=3840,height=2160,fps=final_fps,crf=18,preset='medium',sar='1/1',
                    encoder=final_video_encoder(),static_preview=False,scene_crossfades=True)
    raise ValueError('Video mode must be Draft or Final')

def configure_outputs():
    try:
        import google.colab
        in_colab = True
    except ImportError:
        in_colab = False
    if in_colab:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive')
        root = Path('/content/drive/MyDrive/FPL_VORTEX/Wildcard_FreeHit')
        work = Path('/content/vx_wc_fh_work/final_v2')
        analysis = Path('/content/drive/MyDrive/FPL_VORTEX/Analysis/wc_fh')
    else:
        root = Path.cwd()/'FPL_VORTEX_OUTPUTS'/'Wildcard_FreeHit'
        work = Path.cwd()/'.media_work'/'final_v2'
        analysis = Path.cwd()/'FPL_VORTEX_OUTPUTS'/'wc_fh'
    for p in [root/'mp4',root/'data',root/'slide',work,analysis]:
        p.mkdir(parents=True,exist_ok=True)
    return root,work,analysis

def encode_script(segments):
    lines = ['# Editable narration. Edit the paragraphs; keep the SEGMENT headers.',
             '# Save this TXT and rerun the video cell. Headers are not spoken.', '']
    for s in segments:
        lines.extend([f"[[SEGMENT {s['id']} | GW {s['gameweek']} | {s.get('mood','analysis')}]]",s['text'].strip(),''])
    return '\n'.join(lines)

def decode_script(text):
    pattern = re.compile(r'^\[\[SEGMENT ([\w-]+) \| GW (\d+) \| ([\w-]+)\]\]\s*$',re.M)
    matches=list(pattern.finditer(text))
    if not matches:
        raise ValueError('Narration TXT has no SEGMENT headers. Restore headers or select Regenerate.')
    segments=[]
    for i,m in enumerate(matches):
        paragraph=text[m.end():matches[i+1].start() if i+1<len(matches) else len(text)]
        paragraph='\n'.join(line for line in paragraph.splitlines() if not line.lstrip().startswith('#')).strip()
        if not paragraph:
            raise ValueError(f'Empty narration in segment {m.group(1)}')
        segments.append(dict(id=m.group(1),gameweek=int(m.group(2)),mood=m.group(3),text=paragraph))
    if len({s['id'] for s in segments})!=len(segments):
        raise ValueError('Duplicate narration segment IDs')
    return segments

def sync_script(path, generated, policy='Keep edits'):
    if policy not in ('Keep edits','Regenerate'):
        raise ValueError('Unknown narration policy')
    if not path.exists() or policy=='Regenerate':
        path.write_text(encode_script(generated),encoding='utf-8')
    return decode_script(path.read_text(encoding='utf-8'))

def ffmpeg_binary():
    executable=shutil.which('ffmpeg')
    if executable:return executable
    import imageio_ffmpeg
    return imageio_ffmpeg.get_ffmpeg_exe()

def run_ffmpeg(args):
    result=subprocess.run([ffmpeg_binary(),'-hide_banner','-loglevel','error','-y',*map(str,video_encode_args(args))],capture_output=True,text=True)
    if result.returncode:
        raise RuntimeError('FFmpeg failed: '+result.stderr[-3000:])

def resolve_assets(folder_url,work):
    """Resolve video clips from the already-mounted Drive without a second auth prompt."""
    # Both four-slide videos begin with Slide 1 (include_opening=False).
    # Resolve only the clips that the current builders actually use.
    names=['outro.mp4']
    if VX_INCLUDE_POPUPS:
        names.append('popup.mp4')

    work=Path(work)
    work.mkdir(parents=True,exist_ok=True)

    local=Path(os.environ.get('FPL_MEDIA_ASSET_DIR',str(Path.cwd()/'media_assets')))
    if all((local/n).is_file() for n in names):
        return {n:local/n for n in names}

    drive_root=Path('/content/drive/MyDrive')
    if not drive_root.is_dir():
        raise RuntimeError('Google Drive is not mounted. Run Cell 0 first and approve the single Drive mount prompt.')

    asset_dir_candidates=[
        drive_root/'FPL_VORTEX'/'Assets ',
        drive_root/'FPL_VORTEX'/'Assets',
    ]
    asset_dir=next((candidate for candidate in asset_dir_candidates if candidate.is_dir()),None)
    if asset_dir is None:
        raise RuntimeError('Could not find the mounted FPL_VORTEX Assets folder in My Drive.')

    source_paths={
        'outro.mp4': asset_dir/'outro.mp4',
        'popup.mp4': asset_dir/'popup.mp4',
    }

    assets={}
    for name in names:
        source=source_paths[name]
        if not source.is_file():
            raise RuntimeError(
                f'Missing required video asset {name} at {source}. '
                'Restore it in Drive or set FPL_MEDIA_ASSET_DIR to a directory containing all required clips.'
            )

        target=work/name
        source_stat=source.stat()
        stamp=work/(name+'.source.json')
        expected={
            'source':str(source),
            'size':source_stat.st_size,
            'mtime_ns':source_stat.st_mtime_ns,
        }
        cached_ok=False
        if target.is_file() and stamp.is_file():
            try:
                cached_ok=(json.loads(stamp.read_text())==expected and target.stat().st_size==source_stat.st_size)
            except (ValueError,OSError):
                cached_ok=False

        if not cached_ok:
            temporary=work/(name+'.part')
            temporary.unlink(missing_ok=True)
            shutil.copy2(source,temporary)
            if temporary.stat().st_size<=0:
                temporary.unlink(missing_ok=True)
                raise RuntimeError(f'Copied video asset {name} is empty.')
            temporary.replace(target)
            stamp.write_text(json.dumps(expected))

        assets[name]=target
    return assets


def popup_times(opening_duration,body_duration,interval=90):
    """Every 90 seconds on the final timeline, only during analysis content."""
    end=opening_duration+body_duration
    return [float(t) for t in range(interval,int(end)+1,interval) if opening_duration<=t<end]


def clean_generated_caches(work, analysis):
    """Remove generated cache only, holding a lock while inspecting old work."""
    import fcntl,contextlib
    removed=[]
    with contextlib.ExitStack() as stack:
        for name in ('cleanup.lock','wildcard.render.lock','free_hit.render.lock'):
            lock=stack.enter_context((work/name).open('a'))
            fcntl.flock(lock.fileno(),fcntl.LOCK_EX|fcntl.LOCK_NB)
        # New namespaced work contains no user-authored content. Preserve lock files.
        for path in work.iterdir():
            # Loaded modules survive in memory; their source and approved assets must survive too.
            if path.name.endswith('.lock') or path.name in (
                'wildcard_engine', 'wildcard_visual_assets', 'approved_fdr_slide_02_runtime'
            ): continue
            if path.is_dir() and not path.is_symlink(): shutil.rmtree(path)
            else: path.unlink(missing_ok=True)
            removed.append(str(path))
        # Remove known legacy intermediates; never traverse arbitrary directories.
        legacy=work.parent
        with (legacy/'wildcard.render.lock').open('a') as legacy_lock:
            fcntl.flock(legacy_lock.fileno(),fcntl.LOCK_EX|fcntl.LOCK_NB)
            for name in ('audio_story_v4','audio_verified_wc','audio_local_wc','wildcard_player_media','day3_image_cache'):
                path=legacy/name
                if path.is_dir() and not path.is_symlink():shutil.rmtree(path);removed.append(str(path))
            for pattern in ('ryan_*.wav','*_manifest_*.mp4','*_crossfade_*.mp4','*.partial.mp4','*.part.mp3','*.part.wav'):
                for path in legacy.glob(pattern):
                    if path.is_file() and not path.is_symlink():path.unlink();removed.append(str(path))
    print(f'Cache cleanup: removed {len(removed)} generated entries')
    return removed

def save_verified_script(path, expected, data_hash, policy):
    """Preserve edits, track generated text, and archive replacements by content hash."""
    path=Path(path);prov_root=VX_ANALYSIS_ROOT/'narration_provenance';prov_root.mkdir(parents=True,exist_ok=True);stamp=prov_root/(path.name+'.provenance.json')
    previous=path.read_text(encoding='utf-8') if path.exists() else None
    provenance=json.loads(stamp.read_text()) if stamp.exists() else {}
    digest=lambda text:hashlib.sha256(text.encode()).hexdigest()
    edited=previous is not None and provenance.get('generated_sha256') and digest(previous)!=provenance['generated_sha256']
    if policy=='Keep edits' and edited:
        if provenance.get('data_hash')!=data_hash:
            raise RuntimeError(f'Edited narration preserved at {path}, but its data changed. Review it or select Regenerate; the previous text will be archived.')
        return decode_script(previous)
    if previous is not None and previous!=expected:
        archive=VX_ANALYSIS_ROOT/'previous_narration';archive.mkdir(exist_ok=True)
        backup=archive/(path.stem+'_'+digest(previous)[:16]+path.suffix)
        if not backup.exists():backup.write_text(previous,encoding='utf-8')
    temp=path.with_suffix(path.suffix+'.part');temp.write_text(expected,encoding='utf-8');temp.replace(path)
    stamp.write_text(json.dumps({'data_hash':data_hash,'generated_sha256':digest(expected)},indent=2))
    return decode_script(expected)

VX_DELIVERY_ROOT, VX_MEDIA_WORK, VX_ANALYSIS_ROOT = configure_outputs()
if VX_CACHE_POLICY=="Clean rebuild":clean_generated_caches(VX_MEDIA_WORK,VX_ANALYSIS_ROOT)
else:print("Reuse cache: unchanged Ryan audio and valid scene videos are retained; stale source data still expires.",flush=True)
VX_VIDEO_OUT_DIR = VX_DELIVERY_ROOT / 'mp4'
VX_NARRATION_OUT_DIR = VX_DELIVERY_ROOT / 'data'
VX_4K_PNG_OUT_DIR = VX_DELIVERY_ROOT / 'slide'
VX_AUDIO_OUT_DIR = VX_DELIVERY_ROOT / 'audio'

def _vx_runtime_cell_result(result):
    """Persist cell failures even when Colab has not saved the notebook output."""
    from datetime import datetime, timezone
    import traceback
    error = getattr(result,'error_in_exec',None) or getattr(result,'error_before_exec',None)
    info = getattr(result,'info',None)
    source = str(getattr(info,'raw_cell',''))
    label = next((line.strip('# ').strip() for line in source.splitlines()[:6] if re.search(r'Cell\s+\d+',line)), 'Notebook cell')
    record = {'timestamp':datetime.now(timezone.utc).isoformat(),'cell':label,
              'execution_count':getattr(result,'execution_count',None),'status':'failed' if error is not None else 'passed'}
    if error is not None:
        record.update(error_type=type(error).__name__,message=str(error),traceback=traceback.format_exception(type(error),error,error.__traceback__))
        target=VX_ANALYSIS_ROOT/'latest_runtime_error.json'
        temporary=target.with_suffix('.part.json');temporary.write_text(json.dumps(record,indent=2),encoding='utf-8');temporary.replace(target)
    with (VX_ANALYSIS_ROOT/'cell_execution_log.jsonl').open('a',encoding='utf-8') as log:
        log.write(json.dumps(record)+'\n')

try:
    _vx_ipython = get_ipython()
except NameError:
    _vx_ipython = None
if _vx_ipython is not None:
    _previous_monitor = globals().get('_VX_REGISTERED_CELL_MONITOR')
    if _previous_monitor is not None:
        try:_vx_ipython.events.unregister('post_run_cell',_previous_monitor)
        except ValueError:pass
    _vx_ipython.events.register('post_run_cell',_vx_runtime_cell_result)
    _VX_REGISTERED_CELL_MONITOR = _vx_runtime_cell_result
print('Cell monitoring ready: failures are saved with the full traceback in Analysis/wc_fh.',flush=True)
VX_VIDEO_PROFILE = video_profile(VX_VIDEO_MODE)
print(f"{VX_VIDEO_MODE}: {VX_VIDEO_PROFILE['width']}×{VX_VIDEO_PROFILE['height']}, {VX_VIDEO_PROFILE['fps']} fps, display aspect 16:9")
print('MP4 finals and individual slides:', VX_VIDEO_OUT_DIR)
print('Editable TXT scripts:', VX_NARRATION_OUT_DIR)
print('Narration MP3 exports:', VX_AUDIO_OUT_DIR)
print('4K PNGs (Final only):', VX_4K_PNG_OUT_DIR)
if VX_VIDEO_MODE=='Draft' and VX_DRAFT_FAST_MODE:
    print('Draft: 480p / 5 FPS / 16:9. Fast animated pan/zoom per narration section; no Draft 4K PNG or crossfade re-encode.')
print('Outputs per chip: combined + 4 individual MP4s, combined + 4 individual MP3s/TXTs; four 4K PNGs in Final only.')
print(f'Generate: {VX_GENERATE_CHIPS}')
print(f'Wildcard selection: {VX_WILDCARD_SELECTION_MODE} | Free Hit selection: {VX_FREEHIT_SELECTION_MODE}')
if USE_MANUAL_WILDCARD or USE_MANUAL_FREEHIT:
    print('Manual mode: after Cell 10 opens the player dropdowns, choose IN players, review OUT players, click SAVE; then run Cells 11–17.')
print("✅ CELL 0 COMPLETE")

Mounted at /content/drive
Reuse cache: unchanged Ryan audio and valid scene videos are retained; stale source data still expires.
Cell monitoring ready: failures are saved with the full traceback in Analysis/wc_fh.
Draft: 854×480, 5 fps, display aspect 16:9
MP4 finals and individual slides: /content/drive/MyDrive/FPL_VORTEX/Wildcard_FreeHit/mp4
Editable TXT scripts: /content/drive/MyDrive/FPL_VORTEX/Wildcard_FreeHit/data
Narration MP3 exports: /content/drive/MyDrive/FPL_VORTEX/Wildcard_FreeHit/audio
4K PNGs (Final only): /content/drive/MyDrive/FPL_VORTEX/Wildcard_FreeHit/slide
Draft: 480p / 5 FPS / 16:9. Fast animated pan/zoom per narration section; no Draft 4K PNG or crossfade re-encode.
Outputs per chip: combined + 4 individual MP4s, combined + 4 individual MP3s/TXTs; four 4K PNGs in Final only.
Generate: Both
Wildcard selection: Auto | Free Hit selection: Auto
✅ CELL 0 COMPLETE


In [2]:
#@title 🔴 CELL 1 - FDR METHODOLOGY {display-mode: "form"}
# 🔴 CELL 1 - FDR METHODOLOGY
# Canonical DAY-3 methodology is fetched from a pinned, checksum-verified GitHub commit.
import importlib.util,subprocess,sys
from pathlib import Path
for module,requirement in [('numpy','numpy'),('pandas','pandas'),('scipy','scipy'),('requests','requests'),('PIL','pillow')]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',requirement],check=True)
import hashlib as _vx_model_hashlib
import requests as _vx_model_requests
VX_DAY3_GITHUB_COMMIT = "31e088378736a0f622eb51e7da6b3e2ab18ba55b"
VX_DAY3_MODEL_URL = "https://raw.githubusercontent.com/mukundacity-glitch/FPL-VORTEX-AGENT/31e088378736a0f622eb51e7da6b3e2ab18ba55b/models/day3_fixture_forecast.py"
VX_DAY3_MODEL_SHA256 = "570481e1ec07d5c7c346b905fd58dac9d8f634db319c4bd000ef8a1fded11eef"
try:
    _vx_model_response = _vx_model_requests.get(VX_DAY3_MODEL_URL, timeout=35)
    _vx_model_response.raise_for_status()
except _vx_model_requests.RequestException as _vx_model_exc:
    raise RuntimeError("Canonical DAY-3 fixture model unavailable from GitHub; forecast disabled (no FPL FDR fallback)") from _vx_model_exc
_VX_MODULE_SOURCE = _vx_model_response.content.decode("utf-8")
if _vx_model_hashlib.sha256(_vx_model_response.content).hexdigest() != VX_DAY3_MODEL_SHA256:
    raise RuntimeError("Canonical DAY-3 GitHub model hash mismatch; refusing unverified model source")
_vx_module_dir=VX_MEDIA_WORK/'wildcard_engine'
_vx_module_dir.mkdir(parents=True,exist_ok=True)
(_vx_module_dir/'vortex_upstream_fdr.py').write_text(_VX_MODULE_SOURCE,encoding='utf-8')
if str(_vx_module_dir) not in sys.path:sys.path.insert(0,str(_vx_module_dir))
import importlib
sys.modules.pop('fixture_slide',None)
sys.modules.pop('fixture_visual',None)
importlib.invalidate_caches()
Path(importlib.util.cache_from_source(str(_vx_module_dir / 'vortex_upstream_fdr.py'))).unlink(missing_ok=True)

sys.modules.pop('vortex_upstream_fdr', None)
import vortex_upstream_fdr
print("✅ CELL 1 COMPLETE")

✅ CELL 1 COMPLETE


In [3]:
#@title 🔴 CELL 2 - WILDCARD OPTIMISER {display-mode: "form"}
# 🔴 CELL 2 - WILDCARD OPTIMISER
# This readable embedded module is materialised automatically; no extra download.
import importlib.util,subprocess,sys
for module,requirement in [('numpy','numpy'),('pandas','pandas'),('scipy','scipy'),('requests','requests'),('PIL','pillow')]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',requirement],check=True)
_VX_INTRO_SOURCE = '"""Wildcard Blueprint: evidence scoring, narration and deterministic broadcast HTML."""\nfrom __future__ import annotations\nimport html\nimport math\nimport statistics\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom typing import Any, Mapping, Sequence\n\nVERSION = \'wildcard-blueprint-six-gw-v2-day3-model\'\nAGENDA = (\'When to Play Wildcard?\', \'Six-Week Fixture Difficulty\', \'Wildcard Squad Plan\', \'Captaincy & Final Recommendations\')\nWEIGHTS = {\'scale\': .20, \'severity\': .25, \'relevance\': .20, \'duration\': .15, \'confidence\': .20}\nTHRESHOLD = 58.0\nSAMPLE_NARRATION = (\'Welcome to the Wildcard Blueprint for Gameweek [GW]. Over the next [N] Gameweeks, \'\n                    \'[TOP REASON: VERIFIED METRIC AND VISIBLE EVIDENCE]. [SECOND VISIBLE REASON]. \'\n                    \'First, let’s examine the fixture difficulty across the next [N] Gameweeks.\')\n\ndef numeric(value: Any) -> float | None:\n    try:\n        result = float(value)\n        return result if math.isfinite(result) else None\n    except (TypeError, ValueError):\n        return None\n\ndef timestamp(value: Any) -> datetime | None:\n    try:\n        result = datetime.fromisoformat(str(value).replace(\'Z\', \'+00:00\'))\n        return result.astimezone(timezone.utc) if result.tzinfo else None\n    except (ValueError, TypeError):\n        return None\n\ndef clamp(value: float) -> float:\n    return min(1., max(0., value))\n\ndef score_reason(dimensions: Mapping[str, float]) -> float:\n    return round(100 * sum(WEIGHTS[key] * clamp(dimensions[key]) for key in WEIGHTS), 1)\n\ndef reason(kind: str, label: str, evidence: str, metric: str, metric_label: str,\n           response: str, icon: str, dimensions: dict[str, float], summary: dict[str, Any]) -> dict[str, Any]:\n    if len(label.split()) > 4 or len(evidence.split()) > 20 or len(response.split()) > 12:\n        raise ValueError(\'Wildcard card copy exceeds its readable word budget\')\n    score = score_reason(dimensions)\n    confidence = dimensions[\'confidence\']\n    return {\'type\': kind, \'label\': label, \'impact_level\': \'High\' if score >= 78 and confidence >= .8 else \'Medium\' if score >= THRESHOLD else \'Watchlist\',\n            \'score\': score, \'evidence_text\': evidence, \'metric_value\': metric, \'metric_label\': metric_label,\n            \'recommendation\': response, \'icon\': icon, \'source_data_summary\': summary,\n            \'score_dimensions\': dimensions, \'is_neutral\': False}\n\ndef build_wildcard_case(bootstrap: Mapping[str, Any], fixtures: Sequence[Mapping[str, Any]], *,\n                        histories: Mapping[int, Sequence[Mapping[str, Any]]] | None = None,\n                        projections: Sequence[Mapping[str, Any]] | None = None,\n                        verified_events: Sequence[Mapping[str, Any]] | None = None,\n                        future_matches: Sequence[Mapping[str, Any]] | None = None,\n                        refreshed_at: str | None = None, now: datetime | None = None,\n                        fdr_records: Mapping[tuple[int, int], Mapping[str, Any]] | None = None) -> dict[str, Any]:\n    """Pass existing FPL core records here; no manager squad, network call or invented metric is needed."""\n    if fdr_records is None:\n        raise ValueError(\'Wildcard introduction requires the canonical DAY-3 fixture FDR records\')\n    current = now or datetime.now(timezone.utc)\n    if current.tzinfo is None:\n        raise ValueError(\'Analysis time must include a timezone\')\n    current = current.astimezone(timezone.utc)\n    events = sorted([event for event in bootstrap.get(\'events\', []) if timestamp(event.get(\'deadline_time\')) and timestamp(event[\'deadline_time\']) > current], key=lambda event: timestamp(event[\'deadline_time\']))\n    window = events[:6]\n    weeks = [int(event[\'id\']) for event in window]\n    health: dict[str, Any] = {\'warnings\': [], \'omitted_categories\': {}, \'window_complete\': len(weeks) == 6,\n                              \'refresh_timestamp\': refreshed_at, \'weights\': dict(WEIGHTS), \'threshold\': THRESHOLD}\n    if len(weeks) < 6:\n        health[\'warnings\'].append(f\'Only {len(weeks)} future Gameweeks remain; no fictional Gameweek is added.\')\n    refreshed = timestamp(refreshed_at)\n    age = (current - refreshed).total_seconds() / 3600 if refreshed else None\n    confidence = 1. if age is not None and -.25 <= age <= 6 else .8 if age is not None and 6 < age <= 24 else .55 if age is None or 24 < age <= 72 else .2\n    health[\'age_hours\'] = age\n    health[\'freshness_confidence\'] = confidence\n    health[\'data_label\'] = \'Live FPL Analysis\' if confidence == 1. else \'Latest FPL Analysis\'\n    if confidence < .55:\n        health[\'warnings\'].append(\'Core source is stale or future-dated; external triggers suppressed.\')\n    teams = {int(team[\'id\']): team for team in bootstrap.get(\'teams\', [])}\n    players = [player for player in bootstrap.get(\'elements\', []) if player.get(\'can_select\', True) and not player.get(\'removed\')]\n    popular = [player for player in players if (numeric(player.get(\'selected_by_percent\')) or 0) >= 10 and (numeric(player.get(\'minutes\')) or 0) > 0]\n    candidates: list[dict[str, Any]] = []\n    def dimensions(scale: float, severity: float, relevance: float, duration: float, quality: float = 1.) -> dict[str, float]:\n        return dict(scale=clamp(scale), severity=clamp(severity), relevance=clamp(relevance), duration=clamp(duration), confidence=clamp(confidence * quality))\n    def append(card: dict[str, Any]) -> None:\n        if card[\'score\'] >= THRESHOLD and card[\'score_dimensions\'][\'confidence\'] >= .55:\n            candidates.append(card)\n        else:\n            health[\'omitted_categories\'][card[\'type\']] = \'Below evidence or impact threshold\'\n    upcoming: dict[int, dict[int, list[float]]] = {team_id: {week: [] for week in weeks} for team_id in teams}\n    previous: dict[int, list[float]] = {team_id: [] for team_id in teams}\n    counts: dict[int, dict[int, int]] = {team_id: {week: 0 for week in weeks} for team_id in teams}\n    for fixture in fixtures:\n        week = fixture.get(\'event\')\n        for side in (\'h\', \'a\'):\n            team_id = fixture.get(\'team_\' + side)\n            if team_id not in teams:\n                continue\n            if week in weeks and not fixture.get(\'finished\'):\n                model_record = fdr_records.get((int(fixture[\'id\']), int(team_id)))\n                if model_record is None:\n                    raise ValueError(f\'DAY-3 attack FDR missing for Wildcard intro fixture {fixture["id"]}, team {team_id}\')\n                rating = numeric(model_record.get(\'attack_fdr\'))\n                if rating is None or not 1 <= rating <= 5:\n                    raise ValueError(f\'Invalid DAY-3 attack FDR for Wildcard intro fixture {fixture["id"]}, team {team_id}\')\n                counts[team_id][week] += 1\n                upcoming[team_id][week].append(rating)\n            # Historical model ratings are intentionally not substituted with official FPL difficulty.\n    rows: list[dict[str, Any]] = []\n    for team_id, values in upcoming.items():\n        ratings = [value for batch in values.values() for value in batch]\n        coverage = sum(bool(batch) for batch in values.values())\n        easy = sum(bool(batch) and statistics.mean(batch) <= 2.5 for batch in values.values())\n        if ratings and coverage >= min(3, len(weeks)):\n            ownership = sum(numeric(player.get(\'selected_by_percent\')) or 0 for player in popular if player.get(\'team\') == team_id)\n            rows.append({\'team_id\': team_id, \'club\': teams[team_id][\'name\'], \'future_fdr\': statistics.mean(ratings), \'easy_gws\': easy, \'covered_gws\': coverage, \'ownership_exposure\': ownership,\n                         \'previous_fdr\': statistics.mean(previous[team_id]) if len(previous[team_id]) >= 2 else None})\n    swings = [row for row in rows if row[\'previous_fdr\'] is not None and row[\'previous_fdr\'] - row[\'future_fdr\'] >= .65 and row[\'easy_gws\'] >= 3 and row[\'ownership_exposure\'] >= 10]\n    if swings and len(weeks) >= 3:\n        delta = statistics.mean(row[\'previous_fdr\'] - row[\'future_fdr\'] for row in swings)\n        append(reason(\'fixture_swing\', \'Major Fixture Swing\', f\'{len(swings)} clubs improve by {delta:.2f} FDR, with at least three favourable Gameweeks each.\', f\'−{delta:.2f}\', \'Average FDR change\', \'Target sustained favourable runs, while checking minutes and roles.\', \'calendar\',\n                      dimensions(len(swings)/4, delta/1.5, statistics.mean(row[\'ownership_exposure\'] for row in swings)/35, statistics.mean(row[\'easy_gws\'] for row in swings)/6), {\'source\': \'DAY-3 GitHub attack FDR\', \'clubs\': swings, \'baseline\': \'Previous three Gameweeks\', \'analysis_gws\': weeks}))\n    else:\n        health[\'omitted_categories\'][\'fixture_swing\'] = \'No evidenced persistent and relevant FDR improvement\'\n    flagged = [player for player in popular if player.get(\'status\') in (\'i\', \'s\', \'u\') or (player.get(\'status\') == \'d\' and numeric(player.get(\'chance_of_playing_next_round\')) is not None and float(player[\'chance_of_playing_next_round\']) <= 75)]\n    if flagged and weeks:\n        severity = statistics.mean(1. if player.get(\'status\') in (\'i\',\'s\',\'u\') else 1 - float(player[\'chance_of_playing_next_round\'])/100 for player in flagged)\n        append(reason(\'injury_reset\', \'Availability Reset\', f\'{len(flagged)} popular assets carry official availability flags; return dates need checking.\', str(len(flagged)), \'Flagged popular assets\', \'Review availability; replace persistent risks after confirmed team news.\', \'shield\',\n                      dimensions(len(flagged)/5, severity, statistics.mean(float(player[\'selected_by_percent\']) for player in flagged)/35, .25), {\'source\': \'Official FPL status and next-round chance\', \'players\': [{\'id\': player[\'id\'], \'name\': player[\'web_name\'], \'status\': player[\'status\'], \'chance\': player.get(\'chance_of_playing_next_round\'), \'news\': player.get(\'news\')} for player in flagged], \'duration\': \'Uncertain; no six-week absence assumed\'}))\n    else:\n        health[\'omitted_categories\'][\'injury_reset\'] = \'No qualifying popular official availability flags\'\n    declining = []\n    for player in popular:\n        history = (histories or {}).get(int(player[\'id\']))\n        if history is None:\n            history = (histories or {}).get(str(player[\'id\']))\n        matches = sorted([match for match in (history or []) if timestamp(match.get(\'kickoff_time\')) and timestamp(match[\'kickoff_time\']) <= current and numeric(match.get(\'minutes\')) is not None and numeric(match.get(\'starts\')) is not None], key=lambda match: timestamp(match[\'kickoff_time\']))[-6:]\n        if len(matches) < 6:\n            continue\n        before, recent = matches[:3], matches[3:]\n        old_minutes, new_minutes = statistics.mean(float(match[\'minutes\']) for match in before), statistics.mean(float(match[\'minutes\']) for match in recent)\n        start_drop = sum(float(match[\'starts\']) for match in before) - sum(float(match[\'starts\']) for match in recent)\n        if old_minutes >= 60 and new_minutes <= 60 and old_minutes - new_minutes >= 20 and start_drop >= 1:\n            declining.append({\'id\': player[\'id\'], \'name\': player[\'web_name\'], \'minutes_drop\': old_minutes-new_minutes, \'ownership\': float(player[\'selected_by_percent\']), \'start_drop\': start_drop})\n    if declining:\n        drop = statistics.mean(player[\'minutes_drop\'] for player in declining)\n        append(reason(\'minutes_risk\', \'Minutes Risk\', f\'{len(declining)} popular assets lost {drop:.0f} average minutes and starts across their latest three league matches.\', f\'−{drop:.0f} min\', \'Recent minutes decline\', \'Prefer secure roles; reassess observed declines before committing budget.\', \'clock\', dimensions(len(declining)/4, drop/45, statistics.mean(player[\'ownership\'] for player in declining)/35, .5), {\'source\': \'Official FPL per-match minutes and starts\', \'players\': declining, \'projection\': \'Observed decline; future rotation not asserted\'}))\n    else:\n        health[\'omitted_categories\'][\'minutes_risk\'] = \'Insufficient histories or no qualifying persistent decline\'\n    complete_schedule = bool(weeks and teams) and all(sum(counts[team_id][week] for team_id in teams) >= len(teams)*.6 for week in weeks)\n    schedule_changes = [{\'team_id\': team_id, \'club\': teams[team_id][\'name\'], \'blanks\': sum(value==0 for value in values.values()), \'doubles\': sum(value>1 for value in values.values()), \'fixtures\': sum(values.values())} for team_id, values in counts.items() if any(value != 1 for value in values.values())]\n    if complete_schedule and schedule_changes and len(weeks) == 6:\n        doubles = sum(row[\'doubles\'] for row in schedule_changes)\n        blanks = sum(row[\'blanks\'] for row in schedule_changes)\n        append(reason(\'schedule_opportunity\', \'Schedule Opportunity\', f\'The published window includes {doubles} club doubles and {blanks} club blanks; schedules can change.\', f\'{doubles}D · {blanks}B\', \'Club Gameweek counts\', \'Balance total fixtures with secure minutes across the window.\', \'calendar\', dimensions(len(schedule_changes)/6, min(1,(doubles+blanks)/6), .8, .5), {\'source\': \'Official FPL published event assignments\', \'clubs\': schedule_changes, \'analysis_gws\': weeks, \'counts_are_club_gameweeks\': True}))\n    else:\n        health[\'omitted_categories\'][\'schedule_opportunity\'] = \'No qualifying changes or incomplete published schedule\'\n    # Optional football feeds must supply dated, sourced future matches; no assumed cup schedules.\n    congested = []\n    end = timestamp(events[6][\'deadline_time\']) if len(events)>6 else max((timestamp(fixture.get(\'kickoff_time\')) for fixture in fixtures if fixture.get(\'event\') in weeks and timestamp(fixture.get(\'kickoff_time\'))),default=None)\n    start = timestamp(window[0][\'deadline_time\']) if window else None\n    schedules: dict[int, set[datetime]] = {}\n    for match in future_matches or []:\n        kickoff, checked = timestamp(match.get(\'kickoff_time\')), timestamp(match.get(\'checked_at\'))\n        if not (kickoff and checked and match.get(\'source_url\') and start and end and start <= kickoff < end and 0 <= (current-checked).total_seconds() <= 86400):\n            continue\n        for team_id in match.get(\'team_ids\', []):\n            if team_id in teams:\n                schedules.setdefault(team_id,set()).add(kickoff)\n    for team_id, dates in schedules.items():\n        for fixture in fixtures:\n            kickoff = timestamp(fixture.get(\'kickoff_time\'))\n            if fixture.get(\'event\') in weeks and team_id in (fixture.get(\'team_h\'),fixture.get(\'team_a\')) and kickoff:\n                dates.add(kickoff)\n        ordered = sorted(dates)\n        short_rest = sum((second-first).total_seconds()<4*86400 for first,second in zip(ordered,ordered[1:]))\n        if short_rest >= 3:\n            congested.append({\'team_id\':team_id,\'club\':teams[team_id][\'name\'],\'matches\':len(ordered),\'short_rest_gaps\':short_rest})\n    if congested:\n        append(reason(\'congestion\',\'Fixture Congestion\',f\'{len(congested)} clubs have at least three scheduled turnarounds shorter than four days.\',str(len(congested)),\'Congested clubs\',\'Check rotation exposure; favour dependable starters and playable cover.\',\'clock\',dimensions(len(congested)/4,.85,.7,.6),{\'source\':\'Sourced football schedules plus official FPL\',\'clubs\':congested}))\n    else:\n        health[\'omitted_categories\'][\'congestion\']=\'No qualifying sourced future cross-competition schedule\'\n    international = []\n    for event in verified_events or []:\n        checked=timestamp(event.get(\'checked_at\')); match=event.get(\'match\') or {}; played=timestamp(match.get(\'date\'))\n        competition=str(match.get(\'competition\',\'\')).lower()\n        if event.get(\'confirmed\') and event.get(\'monitor\') and event.get(\'source_url\') and checked and played and 0 <= (current-checked).total_seconds() <= 86400 and 0 <= (current-played).total_seconds() <= 7*86400 and any(term in competition for term in (\'international\',\'nations league\',\'world cup\',\'euro qualifier\')) and event.get(\'player_id\') in {player[\'id\'] for player in popular}:\n            international.append(event)\n    if international and start and 0 < (start-current).total_seconds() <= 5*86400:\n        count=len({event[\'player_id\'] for event in international})\n        append(reason(\'international\',\'International Monitoring\',f\'{count} popular assets have sourced international workload concerns before the next deadline.\',str(count),\'Monitored popular assets\',\'Wait for club updates; keep coverage flexible.\',\'globe\',dimensions(count/3,.6,.9,.25),{\'source\':\'Verified official football reports\',\'events\':international,\'confirmed_absence\':False}))\n    else:\n        health[\'omitted_categories\'][\'international\']=\'No fresh, relevant international workload evidence\'\n    # Pool value is compared within positions, never against an invented manager squad.\n    values=[]\n    for player in projections or []:\n        points=numeric(player.get(\'horizon_expected_points\'));price=numeric(player.get(\'price\'));minutes=numeric(player.get(\'expected_minutes\'));covered=player.get(\'analysis_gws\')\n        if points is not None and price and price>0 and minutes is not None and minutes>=65 and list(covered or [])==weeks and player.get(\'position\'):\n            values.append({\'id\':player.get(\'id\'),\'position\':player[\'position\'],\'points_per_million\':points/price})\n    spread=[]\n    for position in {row[\'position\'] for row in values}:\n        group=sorted(row[\'points_per_million\'] for row in values if row[\'position\']==position)\n        if len(group)>=8 and statistics.median(group)>0:\n            uplift=(group[int((len(group)-1)*.75)]/statistics.median(group)-1)*100\n            if uplift>=20:spread.append({\'position\':position,\'upper_quartile_vs_median_pct\':uplift,\'players\':len(group)})\n    if spread and len(weeks)>=3:\n        uplift=statistics.mean(row[\'upper_quartile_vs_median_pct\'] for row in spread)\n        append(reason(\'value_shift\',\'Projected Value Spread\',f\'Dependable candidates show {uplift:.0f}% stronger upper-quartile projected value across {len(spread)} positions.\',f\'+{uplift:.0f}%\',\'Model xPts/£m spread\',\'Compare value within positions; redirect budget toward sustained returns.\',\'chart\',dimensions(len(spread)/4,uplift/50,.8,len(weeks)/6,.8),{\'source\':\'Existing notebook projection model; estimates, not observed returns\',\'positions\':spread,\'analysis_gws\':weeks}))\n    else:\n        health[\'omitted_categories\'][\'value_shift\']=\'No complete, meaningful same-window projection spread\'\n    health[\'omitted_categories\'][\'structure_reset\']=\'No verified comparable optimisation baseline; no squad gain invented\'\n    if len(candidates)<2 and len(rows)>=4 and len(weeks)>=3:\n        easiest=sorted(rows,key=lambda row:row[\'future_fdr\'])[:3];hardest=sorted(rows,key=lambda row:row[\'future_fdr\'])[-3:]\n        gap=statistics.mean(row[\'future_fdr\'] for row in hardest)-statistics.mean(row[\'future_fdr\'] for row in easiest)\n        if gap>=.75 and all(row[\'easy_gws\']>=3 for row in easiest):\n            append(reason(\'optimisation_window\',\'Sustained Fixture Routes\',f\'The easiest and hardest club groups differ by {gap:.2f} average FDR across the window.\',f\'{gap:.2f}\',\'Fixture difficulty spread\',\'Build around sustained favourable runs and dependable player roles.\',\'calendar\',dimensions(.8,gap/1.5,.7,len(weeks)/6),{\'source\':\'Official FPL fixture difficulty\',\'easiest\':easiest,\'hardest\':hardest,\'analysis_gws\':weeks}))\n    ranked=sorted(candidates,key=lambda card:(-card[\'score\'],card[\'type\']))[:4]\n    fallback=None\n    window_label=f\'GW{weeks[0]}–GW{weeks[-1]}\' if weeks else \'Awaiting next deadline\'\n    if len(ranked)<2:\n        fallback=\'Fewer than two evidenced triggers qualified; neutral strategy is not a recommendation to activate now.\'\n        ranked.append({\'type\':\'neutral_optimisation\',\'label\':\'Six-Week Optimisation\' if len(weeks)==6 else \'Remaining-Season Planning\',\'impact_level\':\'Watchlist\',\'score\':0.,\'evidence_text\':(\'A Wildcard lets managers redirect budget toward the strongest roles and fixture runs across six Gameweeks.\' if len(weeks)==6 else \'Plan across the available future Gameweeks; the data does not support a complete six-week window.\'),\'metric_value\':window_label,\'metric_label\':\'Analysis window\',\'recommendation\':\'Build for sustained points, not short-term reactions.\',\'icon\':\'layers\',\'source_data_summary\':{\'source\':\'Neutral strategy guidance; not a live trigger\',\'analysis_gws\':weeks},\'score_dimensions\':{},\'is_neutral\':True})\n    next_gw=weeks[0] if weeks else None\n    high=any(card[\'impact_level\']==\'High\' for card in ranked)\n    headline=f\'Why a Wildcard Could Matter in GW{next_gw}\' if high and len(weeks)==6 else \'Your Six-Week Wildcard Opportunity\' if len(weeks)==6 else \'Your Remaining-Season Wildcard Window\' if weeks else \'Awaiting the Next Wildcard Window\'\n    subheadline=f\'{health["data_label"]} across {window_label}\' if weeks else \'No upcoming deadline is available; new recommendations are paused.\'\n    return {\'next_gw\':next_gw,\'analysis_gws\':weeks,\'reasons\':ranked,\'headline\':headline,\'subheadline\':subheadline,\'fallback_reason\':fallback,\'data_health\':health,\'sample_narration\':SAMPLE_NARRATION}\n\ndef generate_narration(case: Mapping[str, Any]) -> str:\n    weeks=case[\'analysis_gws\']; count=len(weeks)\n    if not weeks:\n        return \'Welcome to the Wildcard Blueprint. No upcoming deadline is available in the current data. We will resume the analysis when the next schedule is published.\'\n    window=\'six\' if count==6 else str(count)\n    cards=case[\'reasons\'][:3]\n    parts=[f\'Welcome to the Wildcard Blueprint for Gameweek {case["next_gw"]}. Over the next {window} Gameweeks, the aim is not simply to chase last week’s points. It is to position the squad for sustained opportunities.\']\n    for card in cards:\n        parts.append(f\'{card["label"]}. {card["evidence_text"]} {card["recommendation"]}\')\n    strong=any(not card.get(\'is_neutral\') for card in cards)\n    parts.append(\'These are opportunities to review, not an instruction to activate the chip. A reset can redirect the budget towards secure roles and favourable fixtures.\' if strong else \'There is no strong external trigger in the available evidence. Treat this as a planning window, rather than a reason to rush the chip.\')\n    parts.append(f\'First, let’s examine the fixture difficulty across the next {window} Gameweeks.\')\n    return \' \'.join(parts)\n\ndef animation_timing(case: Mapping[str, Any], duration: float) -> dict[str, Any]:\n    """Stretch holds to measured speech; never speed through cards to force a fixed duration."""\n    if not math.isfinite(duration) or duration<=0:\n        raise ValueError(\'Slide duration must be a positive measured or estimated time\')\n    if case.get(\'chip\')==\'freehit\':\n        from freehit_intro import generate_narration as one_week_narration\n        narration_text=case.get(\'narration_text\') or one_week_narration(case)\n    else:\n        narration_text=case.get(\'narration_text\') or generate_narration(case)\n    estimated=len(narration_text.split())/145*60\n    scale=duration/max(52.,estimated+.65)\n    return {\'title\':0.,\'agenda\':[scale*(3+index*1.2) for index in range(4)],\'timeline\':scale*10.,\n            \'cards\':[scale*(15+index*(22/max(1,len(case[\'reasons\'])))) for index in range(len(case[\'reasons\']))],\n            \'transition\':min(duration-1.,scale*max(40.,estimated-5.)), \'entrance\':max(.35,min(1.,scale*.75))}\n\nICON_PATHS={\'calendar\':\'M4 5h16v16H4zM8 2v6m8-6v6M4 10h16m-12 4h3m2 0h3m-8 4h3\',\'shield\':\'m12 2 9 4v7c0 6-9 9-9 9s-9-3-9-9V6z\',\'clock\':\'M12 4a9 9 0 1 0 .01 0M12 7v6l4 2\',\'globe\':\'M12 3a9 9 0 1 0 .01 0M3 12h18M12 3c-7 5-7 13 0 18 7-5 7-13 0-18\',\'chart\':\'M4 20V4m0 16h17M8 17v-5m5 5V8m5 9V4\',\'layers\':\'m12 3 10 5-10 5L2 8zM2 13l10 5 10-5M2 18l10 5 10-5\'}\n\ndef render_html(case: Mapping[str, Any], duration: float=52., player_media: Mapping[Any,Any] | None=None) -> str:\n    """HTML fragment fits the existing 3320×1360 safe area and theme CSS variables."""\n    def escape(value: Any) -> str:return html.escape(str(value),quote=True)\n    timeline=animation_timing(case,duration)\n    is_freehit=case.get(\'chip\')==\'freehit\'\n    agenda=(\'When to Play Free Hit?\',\'Gameweek Fixture Difficulty\',\'Free Hit Squad Plan\',\'Captain Pick & Final Recommendations\') if is_freehit else AGENDA\n    blueprint=\'Free Hit Blueprint\' if is_freehit else \'Wildcard Blueprint\'\n    kicker=\'THE ONE-WEEK PLAN\' if is_freehit else \'THE SIX-WEEK PLAN\'\n    agenda_note=\'SELECT · CAPTAIN · REVERT\' if is_freehit else \'RESET · REBALANCE · BUILD\'\n    items=\'\'.join(f\'<li data-wc-reveal="{start:g}" class="wc-agenda-item{(" active" if index==0 else "")}"><b>{index+1:02d}</b><span>{escape(label)}</span></li>\' for index,(label,start) in enumerate(zip(agenda,timeline[\'agenda\'])))\n    weeks=(f\'<span class="wc-week">GW{case["next_gw"]:02d} · TACTICAL WINDOW · ONE GAMEWEEK ONLY</span>\' if is_freehit and case[\'next_gw\'] else \'\'.join(f\'<span class="wc-week">GW{week:02d}</span>\' for week in case[\'analysis_gws\']))\n    cards=[]\n    for index,card in enumerate(case[\'reasons\']):\n        path=ICON_PATHS.get(card[\'icon\'],ICON_PATHS[\'layers\'])\n        summary=escape(card[\'source_data_summary\'].get(\'source\',\'Evidence summary\'))\n        injury_players=[player for player in card[\'source_data_summary\'].get(\'players\',[]) if player.get(\'status\')==\'i\' or (player.get(\'status\')==\'d\' and \'injur\' in str(player.get(\'news\',\'\')).lower())]\n        portraits=[]\n        for player in injury_players:\n            media=(player_media or {}).get(int(player[\'id\'])) or (player_media or {}).get(str(player[\'id\'])) or {}\n            path=Path(media[\'photo\']) if media.get(\'photo\') else None\n            if media.get(\'photo_kind\')!=\'player\' or path is None or not path.is_file():continue\n            portraits.append(f\'<figure data-wc-portrait="{len(portraits)}" style="display:{("flex" if len(portraits)<4 else "none")}"><img src="{escape(path.resolve().as_uri())}" alt="{escape(player["name"])}"><figcaption>{escape(player["name"])}</figcaption></figure>\')\n        gallery=\'<div class="wc-injuries">\'+\'\'.join(portraits)+\'</div>\' if portraits else \'\'\n\n        cards.append(f\'<article class="wc-reason" data-wc-reveal="{timeline["cards"][index]:g}" title="{summary}"><div class="wc-card-head"><svg viewBox="0 0 24 24" aria-hidden="true"><path d="{path}"/></svg><h3>{escape(card["label"])}</h3><span class="wc-impact">{escape(card["impact_level"])}</span></div><p class="wc-evidence">{escape(card["evidence_text"])}</p><div class="wc-metric"><strong>{escape(card["metric_value"])}</strong><span>{escape(card["metric_label"])}</span></div><div class="wc-score"><i style="width:{card["score"]}%"></i></div><p class="wc-response">{escape(card["recommendation"])}</p>{gallery}</article>\')\n    status=\'No strong live trigger • planning guidance\' if all(card.get(\'is_neutral\') for card in case[\'reasons\']) else \'Evidence ranked • projections remain estimates\'\n    transition=(f\'Next, we rank the best fixtures for Gameweek {case["next_gw"]}.\' if is_freehit else \'Next, we identify the teams with the best six-week fixture runs.\' if len(case[\'analysis_gws\'])==6 else \'Next, we examine the available fixture runs.\')\n    return f\'<section class="wc-blueprint" data-wc-transition="{timeline["transition"]:g}" data-wc-entrance="{timeline["entrance"]:g}" data-chip="{("freehit" if is_freehit else "wildcard")}"><aside class="wc-agenda" data-wc-reveal="{timeline["agenda"][0]:g}"><span class="wc-kicker">{kicker}</span><h2>{blueprint}</h2><ol>{items}</ol><div class="wc-agenda-note">{agenda_note}</div></aside><main class="wc-case"><header data-wc-reveal="0"><span class="wc-kicker">{escape(case["data_health"]["data_label"])}</span><h1>{escape(case["headline"])}</h1><p class="wc-subheadline">{escape(case["subheadline"])}</p></header><div class="wc-timeline" data-wc-reveal="{timeline["timeline"]:g}">{weeks or "Awaiting schedule"}</div><div class="wc-reasons wc-count-{len(cards)}">{"".join(cards)}</div><div class="wc-transition" data-wc-reveal="{timeline["transition"]:g}"><span>→</span>{escape(transition)}</div><div class="wc-status">{escape(status)}</div></main></section>\'\n'
_VX_MODULE_SOURCE = '\n"""VORTEX Wildcard engine. Embedded into the existing Colab; Free Hit stays separate.\n\nObserved data and model estimates are deliberately separate. No missing observed\nstatistic is replaced with a numerical guess. Estimates carry their assumptions.\n"""\nfrom __future__ import annotations\n\nfrom wildcard_intro import build_wildcard_case\nfrom wildcard_analyst_intro import enrich_case, generate_narration as blueprint_narration, generate_segments\nimport concurrent.futures as futures\nimport hashlib\nimport tempfile\nimport itertools\nimport json\nimport math\nimport os\nimport re\nimport unicodedata\nimport xml.etree.ElementTree as ET\nfrom dataclasses import dataclass, field\nfrom datetime import datetime, timezone, timedelta\nfrom decimal import Decimal, ROUND_HALF_UP\nfrom html.parser import HTMLParser\nfrom pathlib import Path\nfrom urllib.parse import urlencode, urlparse\n\nimport numpy as np\nimport pandas as pd\nimport requests\nfrom scipy.optimize import milp, Bounds, LinearConstraint\nfrom scipy.sparse import lil_matrix\nfrom scipy.stats import poisson\n\nVERSION = "vortex-wildcard-blueprint-six-gw-v5-2026-10-08"\nAPI = "https://fantasy.premierleague.com/api/"\nPOSITIONS = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}\nSQUAD_COUNTS = {"GK": 2, "DEF": 5, "MID": 5, "FWD": 3}\nPRIMARY_NEWS_DOMAINS = {\n    "premierleague.com", "uefa.com", "fifa.com", "thefa.com", "nff.no", "fotball.no", "englandfootball.com",\n    "arsenal.com", "avfc.co.uk", "afcb.co.uk", "brentfordfc.com",\n    "brightonandhovealbion.com", "chelseafc.com", "cpfc.co.uk",\n    "evertonfc.com", "fulhamfc.com", "leedsunited.com", "liverpoolfc.com",\n    "mancity.com", "manutd.com", "nufc.co.uk", "nottinghamforest.co.uk",\n    "safc.com", "tottenhamhotspur.com", "whufc.com", "wolves.co.uk",\n    "burnleyfootballclub.com", "southamptonfc.com", "itfc.co.uk",\n}\n\n\ndef utc(value=None):\n    if value is None:\n        return datetime.now(timezone.utc)\n    if isinstance(value, datetime):\n        return value.replace(tzinfo=timezone.utc) if value.tzinfo is None else value.astimezone(timezone.utc)\n    return datetime.fromisoformat(str(value).replace("Z", "+00:00")).astimezone(timezone.utc)\n\n\ndef norm(value):\n    s = unicodedata.normalize("NFKD", str(value or "")).encode("ascii", "ignore").decode().lower()\n    return re.sub(r"[^a-z0-9]+", " ", s).strip()\n\n\ndef number(value):\n    """Missing means None, including NaN. This is for observations, not estimates."""\n    try:\n        n = float(value)\n        return n if math.isfinite(n) else None\n    except (TypeError, ValueError):\n        return None\n\n\ndef clean_json(value):\n    if isinstance(value, dict):\n        return {str(k): clean_json(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple, np.ndarray)):\n        return [clean_json(v) for v in value]\n    if isinstance(value, np.integer):\n        return int(value)\n    if isinstance(value, (float, np.floating)):\n        return float(value) if math.isfinite(value) else None\n    return value\n\n\ndef write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(mode=\'w\',encoding=\'utf-8\',dir=path.parent,prefix=path.name+\'.\',suffix=\'.part\',delete=False) as handle:\n        temp=Path(handle.name)\n        handle.write(json.dumps(clean_json(obj),indent=2,ensure_ascii=False,allow_nan=False))\n    try:temp.replace(path)\n    finally:temp.unlink(missing_ok=True)\n\n\ndef attack_fdr(expected_team_goals):\n    return vx_phase4_attack_fdr(expected_team_goals)\n\n\n\ndef defence_fdr(clean_sheet_probability):\n    return vx_phase4_defence_fdr(clean_sheet_probability)\n\n\n\ndef fdr_display(raw):\n    """DAY-3 exact band thresholds, gradient endpoints and label colours."""\n    if raw is None:\n        return {"value": None, "label": "—", "background": "#233445", "text": "#FFFFFF", "band": "missing", "gradient": "none"}\n    v = number(raw)\n    if v is None or not 1 <= v <= 5:\n        raise ValueError(f"FDR out of range: {raw}")\n    from vortex_upstream_fdr import day3_fdr_style\n    style = day3_fdr_style(v)\n    gradient = style["gradient"]\n    q = float(Decimal(str(v)).quantize(Decimal("0.1"), rounding=ROUND_HALF_UP))\n    return {"value": q, "label": f"{q:g}", "background": gradient[0],\n            "gradient": f"linear-gradient(125deg,{gradient[0]},{gradient[1]})",\n            "gradient_endpoints": list(gradient), "text": style["text"], "band": style["band"].lower(),\n            "class_name": style["class_name"]}\n\n\n@dataclass\nclass Config:\n    horizon: int = 6\n    projection_weeks: int = 11  # five transfer weeks, each with a remaining lookahead\n    cache_hours: float = 1\n    budget: float | None = None\n    entry_id: str = ""\n    bench_boost: str = "Unknown"  # Auto / Available / Used / Unknown\n    bench_boost_gameweek: int | None = None\n    free_transfers: int | None = None\n    fdr_policy: str = "Strict"  # DAY-3 GitHub Phase 1–4 must be available\n    enable_news: bool = True\n    enable_espn: bool = True\n    # Explicit policy defaults implement the qualitative expert spec; model estimates\n    # are not certainty. Tight gates fail with a report rather than silently relax.\n    core_start_probability: float = .85\n    core_expected_minutes: float = 65.\n    cover_start_probability: float = .75\n    cover_expected_minutes: float = 60.\n    captain_start_probability: float = .90\n    captain_expected_minutes: float = 75.\n    minimum_history_appearances: int = 2\n    maximum_conditional_xi: int = 1\n    minimum_playable_weeks: int = 3\n    minimum_captaincy_weeks: int = 2\n    premium_price_quantile: float = .80\n    points_tie_tolerance: float = .05\n    solver_seconds: int = 60\n    transfer_threshold: float = 2.0\n    ft_option_value: float = .75\n    primary_news_urls: tuple = ()  # Optional explicit official sources, checked for publication freshness\n    root: Path = field(default_factory=lambda: Path("FPL_VORTEX_OUTPUTS/wc_fh"))\n    core_root: Path = field(default_factory=lambda: Path("/content/drive/MyDrive/FPL_VORTEX_DATA/00_SOURCE_ARCHIVE/FPL_CORE/data"))\n    model_inputs: Path = field(default_factory=lambda: Path("/content/drive/MyDrive/FPL_VORTEX/MODEL_INPUTS"))\n\n\nclass DataClient:\n    def __init__(self, directory, now=None, hours=1):\n        self.directory = Path(directory)\n        self.directory.mkdir(parents=True, exist_ok=True)\n        self.now = utc(now)\n        self.hours = hours\n        self.audit = []\n\n    def get(self, url, required=False, hours=None, timeout=12):\n        key = hashlib.sha256(url.encode()).hexdigest()[:24]\n        path = self.directory / (key + ".json")\n        limit = self.hours if hours is None else hours\n        cache=None\n        try:\n            if path.exists():\n                cache=json.loads(path.read_text())\n                if not isinstance(cache,dict) or \'fetched_at\' not in cache or \'data\' not in cache:raise ValueError(\'Invalid cache schema\')\n            age=(self.now-utc(cache[\'fetched_at\'])).total_seconds()/3600 if cache else math.inf\n        except (OSError,ValueError,KeyError,TypeError):\n            path.unlink(missing_ok=True);cache=None;age=math.inf\n        if cache is not None and not 0<=age<=limit:\n            path.unlink(missing_ok=True)\n        if 0 <= age <= limit:\n            self.audit.append({"url": url, "state": "fresh cache", "fetched_at": cache["fetched_at"]})\n            return cache["data"]\n        error = None\n        for _ in range(2):\n            try:\n                r = requests.get(url, timeout=timeout, headers={"User-Agent": "FPL-Vortex/2.0 (analytics)"})\n                r.raise_for_status()\n                data = r.json()\n                stamp = utc().isoformat()\n                write_json(path, {"fetched_at": stamp, "data": data})\n                self.audit.append({"url": url, "state": "live", "fetched_at": stamp})\n                return data\n            except (requests.RequestException, ValueError) as exc:\n                response = getattr(exc, \'response\', None)\n                error = type(exc).__name__ + (f\' HTTP {response.status_code}\' if response is not None else \'\')\n                if response is not None and 400<=response.status_code<500 and response.status_code not in (408,429):break\n        self.audit.append({"url": url, "state": "unavailable", "error": error,\n                           "rejected_cache_age_hours": age if math.isfinite(age) else None})\n        if required:\n            raise RuntimeError(f"Required current source unavailable: {url}; {error}. Stale cache was not used.")\n        return None\n\n\nclass FDRStore:\n    """Exactly one record per fixture/team, shared by every visual and explanation."""\n    def __init__(self, fixtures, teams, canonical=None, reason=None):\n        self.records = {}\n        if not canonical:\n            raise RuntimeError("DAY-3 GitHub fixture model returned no future fixture forecasts")\n        self.source = "VORTEX DAY-3 GitHub Phase 1–4"\n        self.reason = reason\n        model = {int(r["fixture_id"]): r for r in (canonical or [])}\n        for f in fixtures:\n            if f.get("event") is None or f.get("finished"):\n                continue\n            fid = int(f["id"])\n            m = model.get(fid)\n            if m is None:\n                raise ValueError(f"Canonical FDR missing future fixture {fid}")\n            if m and (int(m["home_team_id"]), int(m["away_team_id"]), int(m["gameweek"])) != (f["team_h"], f["team_a"], f["event"]):\n                raise ValueError(f"Canonical fixture identity mismatch {fid}")\n            for home in (True, False):\n                tid = f["team_h"] if home else f["team_a"]\n                opp = f["team_a"] if home else f["team_h"]\n                side = "home" if home else "away"\n                a = attack_fdr(m["model_lambda_" + side]) if m else None\n                cs = float(m[side + "_clean_sheet_probability"]) if m else None\n                d = defence_fdr(cs) if m else None\n                official = number(f.get("team_h_difficulty" if home else "team_a_difficulty"))\n                self.records[(fid, int(tid))] = {\n                    "fixture_id": fid, "gameweek": int(f["event"]), "team_id": int(tid),\n                    "opponent_id": int(opp), "opponent": teams[opp]["short_name"],\n                    "home": home, "kickoff": f.get("kickoff_time"), "source": self.source,\n                    "attack_fdr": a, "defence_fdr": d, "official_fdr": official,\n                    "team_goals": float(m["model_lambda_" + side]) if m else None,\n                    "opponent_goals": float(m["model_lambda_" + ("away" if home else "home")]) if m else None,\n                    "clean_sheet_probability": cs,\n                }\n\n    def for_team(self, tid, gw, position=None):\n        values = [dict(r) for (_, t), r in self.records.items() if t == int(tid) and r["gameweek"] == int(gw)]\n        values.sort(key=lambda r: r["kickoff"] or "")\n        for r in values:\n            channel = "defence" if position in ("GK", "DEF") else "attack"\n            raw = r[channel + "_fdr"]\n            r["fdr_kind"] = "D" if channel == "defence" else "A"\n            r["fdr"] = raw\n            r["fdr_style"] = fdr_display(raw)\n            r["attack_style"] = fdr_display(r["attack_fdr"])\n            r["defence_style"] = fdr_display(r["defence_fdr"])\n        return values\n\n\ndef canonical_fdr(boot, fixtures, cfg, now):\n    """Use only the pinned DAY-3 GitHub model and its verified live FPL-Core evidence."""\n    import vortex_upstream_fdr\n    source_client = DataClient(cfg.root / \'current_source_cache\', now, cfg.cache_hours)\n    try:\n        core_archive = vortex_upstream_fdr.refresh_fpl_core(boot, fixtures, cfg.model_inputs, now, source_client)\n        rows = vortex_upstream_fdr.rebuild(boot, fixtures, core_archive, cfg.model_inputs, now)\n    except (RuntimeError, ValueError, KeyError, OSError, requests.RequestException) as exc:\n        raise RuntimeError(\n            f"DAY-3 GitHub canonical forecast could not be rebuilt: {type(exc).__name__}: {exc}. "\n            "No official FPL FDR or FPL xG fallback is permitted."\n        ) from exc\n    return rows, None\n\n\ndef latest_pl(history, teams):\n    played = [h for h in history if number(h.get("minutes")) and h.get("kickoff_time")]\n    if not played:\n        return None\n    h = max(played, key=lambda h: h["kickoff_time"])\n    return {"date": h["kickoff_time"], "competition": "Premier League",\n            "opponent": teams.get(h.get("opponent_team"), {}).get("name", "Unknown opponent"),\n            "minutes": number(h.get("minutes")), "goals": number(h.get("goals_scored")),\n            "assists": number(h.get("assists")), "xg": number(h.get("expected_goals")),\n            "xa": number(h.get("expected_assists")), "points": number(h.get("total_points")),\n            "rating": None, "source": API + f"element-summary/{h.get(\'element\', \'\')}/",\n            "coverage": "league only"}\n\n\ndef evidence_line(match):\n    if not match:\n        return "Latest match evidence unavailable; no performance claim made."\n    metrics = []\n    for key, label in (("goals", "goals"), ("assists", "assists"), ("points", "FPL points")):\n        if match.get(key) is not None:\n            metrics.append(f"{match[key]:g} {label}")\n    if match.get("xg") is not None:\n        metrics.append(f"{match[\'xg\']:.2f} xG")\n    if match.get("rating") is not None:\n        metrics.append(f"{match[\'rating\']:g} provider rating")\n    mins = f"{match[\'minutes\']:g} minutes" if match.get("minutes") is not None else "minutes unavailable"\n    return f"{match[\'competition\']}, {match[\'date\'][:10]}, v {match[\'opponent\']}: {mins}" + ("; " + ", ".join(metrics) if metrics else "") + "."\n\n\ndef minutes_model(player, history):\n    """Multiple-match role evidence; availability is applied later per fixture."""\n    recent = sorted(history, key=lambda h: h.get("kickoff_time") or "")[-6:]\n    if not recent:\n        return {"p_start": None, "p_play": None, "p60": None, "conditional_minutes": None,\n                "base_minutes": None, "recent_minutes": [], "sample_size": 0, "uncertainty": "high"}\n    m = np.array([float(h["minutes"]) for h in recent])\n    w = np.arange(1, len(m) + 1, dtype=float)\n    starts = np.array([float(h["starts"]) > 0 if "starts" in h else h["minutes"] >= 60 for h in recent])\n    # Small samples must not imply certainty. Two prior appearances shrink\n    # these empirical estimates; priors are disclosed model assumptions.\n    n=len(m)\n    p_start = float((np.average(starts, weights=w)*n+2*.8)/(n+2))\n    p_play = float((np.average(m > 0, weights=w)*n+2*.9)/(n+2))\n    p60 = min(p_play,float((np.average(m >= 60, weights=w)*n+2*.75)/(n+2)))\n    conditional = float(np.average(m[m > 0], weights=w[m > 0])) if (m > 0).any() else 0.\n    return {"p_start": p_start, "p_play": p_play, "p60": p60,\n            "conditional_minutes": conditional, "base_minutes": p_play * conditional,\n            "recent_minutes": m.astype(int).tolist(), "sample_size": len(m),\n            "uncertainty": "high" if len(m) < 3 else "medium"}\n\n\ndef player_status(player):\n    status = player.get("status")\n    chance = number(player.get("chance_of_playing_next_round"))\n    label = {"i": "INJURED", "d": "DOUBTFUL", "s": "SUSPENDED", "u": "UNAVAILABLE", "n": "UNAVAILABLE"}.get(status)\n    if label is None and chance is not None and chance < 100:\n        label = f"{chance:g}%"\n    return {"code": status, "label": label or "AVAILABLE", "flagged": bool(label),\n            "chance_pct": chance, "news": str(player.get("news") or ""),\n            "news_at": player.get("news_added"), "source": API + "bootstrap-static/"}\n\n\ndef availability(player, fixture, now, verified_events):\n    """An official next-round flag only forecasts the first upcoming GW.\n\n    Without a confirmed return date future injury recovery is unknown, not 100%.\n    Carrying the current risk forward is a labelled conservative assumption.\n    Verified expiry/return evidence can change later fixtures independently.\n    """\n    status = player_status(player)\n    if not status["flagged"]:\n        return 1., "official available"\n    chance = status["chance_pct"]\n    health = chance / 100 if chance is not None else (0. if status["code"] in ("i", "s", "u", "n") else .5)\n    for e in verified_events:\n        if e.get("player_id") != int(player["id"]) or not e.get("confirmed"):\n            continue\n        if e.get("return_date") and fixture.get("kickoff") and utc(fixture["kickoff"]) >= utc(e["return_date"]):\n            return 1., "confirmed return date; role uncertainty remains"\n        if e.get("unavailable_until") and fixture.get("kickoff") and utc(fixture["kickoff"]) <= utc(e["unavailable_until"]):\n            return 0., "verified absence window"\n    return health, "official flag; future recovery unknown (conservative carry-forward)"\n\n\nclass NewsEvidence:\n    """Current official injury feed + source-linked headlines + optional exact match feeds.\n\n    Headlines are monitoring evidence. No allegation or sentiment is translated\n    into a made-up xPts penalty. Confirmed match minutes drive rest adjustments;\n    confirmed absence windows drive availability. Coverage gaps are visible.\n    """\n    def __init__(self, cfg, client, players, teams, now):\n        self.cfg, self.client, self.players, self.teams, self.now = cfg, client, players, teams, now\n        self.matches = {}\n        self.events = []\n        self.headlines = {}\n        self.coverage = {"fpl_availability": "official live", "all_competitions": "not yet checked", "club_news": "not yet checked"}\n\n    def primary_articles(self):\n        """Discover recent official PL reports; match linked Opta codes to FPL codes.\n\n        Only direct prose mentioning a player can flag that player. Exact\n        minutes are extracted only from an explicit \'played N minutes\' or\n        \'replaced in the Nth minute\' sentence. A missing number stays missing.\n        """\n        class Parser(HTMLParser):\n            def __init__(self):\n                super().__init__();self.paragraphs=[];self.links=[];self.dates=[];self.p=None;self.a=None\n            def handle_starttag(self,tag,attrs):\n                a=dict(attrs)\n                if tag==\'time\':\n                    for k in (\'datetime\',\'data-date\'):\n                        if a.get(k):self.dates.append(a[k])\n                if tag==\'meta\' and a.get(\'property\') in (\'article:published_time\',\'article:modified_time\'):\n                    self.dates.append(a.get(\'content\',\'\'))\n                if tag==\'p\':self.p={\'text\':\'\',\'codes\':[]}\n                if tag==\'a\':\n                    self.a={\'url\':a.get(\'href\',\'\'),\'text\':\'\'}\n                    code=re.search(r\'/players/(\\\\d+)/\',a.get(\'href\',\'\'))\n                    if self.p is not None and code:self.p[\'codes\'].append(code.group(1))\n            def handle_data(self,data):\n                if self.p is not None:self.p[\'text\']+=data\n                if self.a is not None:self.a[\'text\']+=data\n            def handle_endtag(self,tag):\n                if tag==\'p\' and self.p is not None:self.paragraphs.append(self.p);self.p=None\n                if tag==\'a\' and self.a is not None:self.links.append(self.a);self.a=None\n        urls=list(self.cfg.primary_news_urls)\n        try:\n            response=requests.get(\'https://www.premierleague.com/en/news\',timeout=8);response.raise_for_status()\n            index=Parser();index.feed(response.text)\n            for link in index.links:\n                if re.search(r\'international|injur|team news|appeal|commission|statement\',link[\'text\'],re.I):\n                    url=link[\'url\']\n                    if url.startswith(\'/en/news/\'):url=\'https://www.premierleague.com\'+url\n                    if url.startswith(\'https://www.premierleague.com/en/news/\'):urls.append(url)\n        except requests.RequestException:pass\n        checked=[]\n        codes={str(p[\'code\']):pid for pid,p in self.players.items()}\n        for url in list(dict.fromkeys(urls))[:10]:\n            host=urlparse(url).hostname or \'\'\n            if not any(host==d or host.endswith(\'.\'+d) for d in PRIMARY_NEWS_DOMAINS):continue\n            try:\n                response=requests.get(url,timeout=8);response.raise_for_status()\n                parser=Parser();parser.feed(response.text)\n                stamps=[]\n                for stamp in parser.dates:\n                    try:stamps.append(utc(stamp))\n                    except (ValueError,TypeError):\n                        try:stamps.append(datetime.strptime(stamp,\'%d %b %Y\').replace(tzinfo=timezone.utc))\n                        except (ValueError,TypeError):pass\n                # Some official pages encode publication date in the URL.\n                date=re.search(r\'/(20\\\\d{2})/(\\\\w{3})/(\\\\d{2})/\',url)\n                if date:\n                    try:stamps.append(datetime.strptime(\'-\'.join(date.groups()),\'%Y-%b-%d\').replace(tzinfo=timezone.utc))\n                    except ValueError:pass\n                if not stamps:continue\n                published=max(stamps)\n                if not 0 <= (self.now-published).total_seconds() <= 7*86400:continue\n                checked.append(url)\n                # Club proceedings are tracked separately from a player\'s fitness.\n                # A reported appeal/charge alone never supplies a points penalty.\n                article_text=\' \'.join(p[\'text\'] for p in parser.paragraphs)\n                if re.search(r\'appeal|commission|sanction|charges|postpon\',article_text,re.I):\n                    for tid,team in self.teams.items():\n                        if \' \'+norm(team[\'name\'])+\' \' in \' \'+norm(article_text)+\' \':\n                            self.events.append({\'team_id\':tid,\'source_url\':url,\n                                \'checked_at\':self.now.isoformat(),\'published_at\':published.isoformat(),\n                                \'confirmed\':True,\'monitor\':True,\'numeric_effect\':None,\n                                \'summary\':\'Recent official club proceedings or scheduling report; no automatic xPts change without a confirmed football consequence.\'})\n                for paragraph in parser.paragraphs:\n                    text=\' \'.join(paragraph[\'text\'].split())\n                    ids={codes[c] for c in paragraph[\'codes\'] if c in codes}\n                    # Strong exact-name fallback for unlinked opening paragraphs.\n                    for pid,p in self.players.items():\n                        full=norm(p[\'first_name\']+\' \'+p[\'second_name\'])\n                        if full and \' \'+full+\' \' in \' \'+norm(text)+\' \':ids.add(pid)\n                    for pid in ids:\n                        p=self.players[pid]\n                        sentences=re.split(r\'(?<=[.!?])\\\\s+\',text)\n                        attributed=[s for s in sentences if norm(p[\'second_name\']) in norm(s) or norm(p[\'web_name\']) in norm(s)]\n                        if not attributed:continue\n                        sentence=\' \'.join(attributed)\n                        concern=bool(re.search(r\'injur|fitness|withdraw|asked to be taken off|ended early|fatigue|ruled out\',sentence,re.I))\n                        self.events.append({\'player_id\':pid,\'team_id\':p[\'team\'],\'source_url\':url,\n                            \'checked_at\':self.now.isoformat(),\'published_at\':published.isoformat(),\'confirmed\':True,\n                            \'summary\':(\'Recent official report raises a fitness/workload concern; availability must be checked.\' if concern else \'Recent international involvement reported by an official source.\'),\n                            \'monitor\':concern,\'numeric_effect\':None})\n                        # Numeric performance remains exact. Avoid claiming match\n                        # date/opponent when the article does not resolve them.\n                        minutes=re.search(r\'(?:played|completed) (?:the (?:first|full) )?(\\\\d{1,3}) minutes|(?:replaced|substituted) in the (\\\\d{1,3})(?:st|nd|rd|th) minute\',sentence,re.I)\n                        if minutes:self.events[-1][\'reported_minutes\']=int(next(g for g in minutes.groups() if g))\n            except (requests.RequestException,ValueError):continue\n        self.coverage[\'primary_articles\']={\'checked_urls\':checked,\'automatic_effect\':\'sourced fitness monitoring; no invented xPts penalty\'}\n\n    def read_verified_file(self):\n        path = self.cfg.model_inputs / "verified_player_news.json"\n        if not path.exists():\n            return\n        data = json.loads(path.read_text())\n        for e in data.get("events", []):\n            try:\n                stamp = utc(e["checked_at"])\n                if not 0 <= (self.now - stamp).total_seconds() <= self.cfg.cache_hours * 3600:\n                    continue\n                domain = urlparse(e.get("source_url", "")).hostname or ""\n                if not any(domain == d or domain.endswith("." + d) for d in PRIMARY_NEWS_DOMAINS):\n                    continue\n                if not e.get("confirmed") or not e.get("source_url"):\n                    continue\n                pid = int(e.get("player_id", 0))\n                if pid and pid not in self.players:\n                    continue\n                self.events.append(e)\n                if pid and e.get("match"):\n                    m = dict(e["match"], source=e["source_url"], coverage="verified external match")\n                    if m.get("date") and utc(m["date"]) <= self.now:\n                        self.matches.setdefault(pid, []).append(m)\n            except (ValueError, KeyError, TypeError):\n                continue\n\n    @staticmethod\n    def _espn_roster_minutes(player, event):\n        """Read explicit minutes, or derive regulation minutes from timed substitutions."""\n        stats = {s.get(\'name\'): number(s.get(\'value\', s.get(\'displayValue\'))) for s in player.get(\'stats\', [])}\n        for key in (\'minutesPlayed\', \'minutes\', \'MIN\'):\n            if stats.get(key) is not None:\n                return stats[key], \'reported individual minutes\'\n        status = event.get(\'status\', {}).get(\'type\', {})\n        if not status.get(\'completed\') or status.get(\'name\') not in (\'STATUS_FULL_TIME\', \'STATUS_FINAL\', \'STATUS_FINAL_AET\', \'STATUS_FINAL_PEN\'):\n            return None, \'match completion or regulation duration unverified\'\n        end = 120. if \'AET\' in status.get(\'name\', \'\') or \'Extra Time\' in status.get(\'description\', \'\') or event.get(\'status\', {}).get(\'period\', 2) > 2 else 90.\n        substitutions = []\n        dismissals = []\n        for play in player.get(\'plays\', []):\n            clock = str(play.get(\'clock\', {}).get(\'displayValue\', \'\'))\n            match = re.fullmatch(r"(\\d+)(?:\\s*[\'’]\\s*\\+\\s*\\d+|\\s*\\+\\s*\\d+)?[\'’]?", clock.strip())\n            if not match:\n                continue\n            minute = min(end, float(match.group(1)))  # added time is excluded, as in FPL minutes\n            if play.get(\'substitution\'):\n                substitutions.append(minute)\n            if play.get(\'redCard\'):\n                dismissals.append(minute)\n        substitutions.sort()\n        if player.get(\'starter\'):\n            start = 0.\n        elif player.get(\'subbedIn\') and substitutions:\n            start = substitutions[0]\n        else:\n            return None, \'no verified appearance\'\n        if player.get(\'subbedOut\'):\n            if not substitutions or (player.get(\'subbedIn\') and len(substitutions) < 2):\n                return None, \'substitution timing incomplete\'\n            end = min(end, substitutions[-1])\n        if dismissals:\n            end = min(end, min(dismissals))\n        return max(0., end - start), \'official lineup and timed substitutions; regulation minutes, excludes added time\'\n\n    def espn(self):\n        if not self.cfg.enable_espn:\n            self.coverage["all_competitions"] = "disabled; league evidence only"\n            return\n        comps = ["eng.1", "eng.league_cup", "eng.fa", "uefa.champions", "uefa.europa",\n                 "uefa.europa.conf", "uefa.nations", "fifa.worldq.uefa", "fifa.friendly",\n                 "fifa.worldq.conmebol", "fifa.worldq.concacaf", "fifa.worldq.caf", "fifa.worldq.afc"]\n        # Soccer scoreboards reject the date-range form with HTTP 400. Query each\n        # calendar date, deduplicate event IDs, and expose partial-day failures.\n        days = [(self.now - timedelta(days=i)).strftime(\'%Y%m%d\') for i in range(22)]\n        boards = {c: {\'events\': {}} for c in comps}\n        checked = {c: [] for c in comps}; failed = {c: [] for c in comps}\n        def board(task):\n            comp, day = task\n            u = f"https://site.api.espn.com/apis/site/v2/sports/soccer/{comp}/scoreboard?dates={day}&limit=1000"\n            return comp, day, u, self.client.get(u, timeout=6)\n        with futures.ThreadPoolExecutor(max_workers=6) as ex:\n            for comp, day, url, data in ex.map(board, [(c,d) for c in comps for d in days]):\n                if not isinstance(data, dict) or not isinstance(data.get(\'events\'), list):\n                    failed[comp].append({\'date\': day, \'url\': url})\n                    continue\n                checked[comp].append(day)\n                for event in data[\'events\']:\n                    if event.get(\'id\'):\n                        boards[comp][\'events\'][str(event[\'id\'])] = event\n        self.coverage["all_competitions"] = {"provider": "ESPN", "lookback_days": 21,\n            "available_competitions": [c for c in comps if checked[c]],\n            "missing_competitions": [c for c in comps if not checked[c]],\n            "fully_checked_competitions": [c for c in comps if len(checked[c]) == len(days)],\n            "failed_days": {c:v for c,v in failed.items() if v},\n            "request_mode": "single calendar date; completed events only; event IDs deduplicated",\n            "claim": "Latest VERIFIED appearance in covered competitions; full career coverage is not claimed"}\n        names = {}\n        for pid, p in self.players.items():\n            full = norm(p[\'first_name\'] + \' \' + p[\'second_name\'])\n            for key in {full, \'tokens:\' + \' \'.join(sorted(full.split())), norm(p.get(\'web_name\'))} - {\'\'}:\n                names.setdefault(key, []).append(pid)\n        tasks = []\n        for c, b in boards.items():\n            for event in b[\'events\'].values():\n                if not event.get("status", {}).get("type", {}).get("completed") or not event.get(\'date\') or utc(event[\'date\']) > self.now:\n                    continue\n                tasks.append((c, event))\n        available_summaries = 0; verified_matches = 0; candidate_rows = []\n        # Match summaries include actual individual minutes, not guessed from scorelines.\n        def details(task):\n            c, event = task\n            u = f"https://site.api.espn.com/apis/site/v2/sports/soccer/{c}/summary?event={event[\'id\']}"\n            return c, event, u, self.client.get(u, timeout=6)\n        with futures.ThreadPoolExecutor(max_workers=4) as ex:\n            for c, event, source, detail in ex.map(details, tasks):\n                if not detail:\n                    continue\n                available_summaries += 1\n                competition = event.get("season", {}).get("displayName") or c\n                competitors = event.get("competitions", [{}])[0].get("competitors", [])\n                team_names = {str(x.get("team", {}).get("id")): x.get("team", {}).get("displayName") for x in competitors}\n                sides = list(detail.get(\'boxscore\', {}).get(\'players\', []))\n                # Soccer summaries publish individual players in rosters, not\n                # boxscore.players. Retain support for both observed schemas.\n                for roster in detail.get(\'rosters\', []):\n                    athletes = []\n                    for player in roster.get(\'roster\', []):\n                        values = {s.get(\'name\'): s.get(\'value\', s.get(\'displayValue\')) for s in player.get(\'stats\', [])}\n                        minutes, basis = self._espn_roster_minutes(player, event)\n                        values[\'minutesPlayed\'] = minutes\n                        athletes.append({\'athlete\': player.get(\'athlete\', {}), \'stats\': list(values.values()), \'_keys\': list(values), \'_minutes_basis\': basis})\n                    sides.append({\'team\': roster.get(\'team\', {}), \'statistics\': [{\'athletes\': athletes}]})\n                for side in sides:\n                    team_id = str(side.get("team", {}).get("id"))\n                    opponent = next((v for k, v in team_names.items() if k != team_id), "Unknown opponent")\n                    for block in side.get("statistics", []):\n                        labels = block.get("keys") or block.get("names") or block.get("labels") or []\n                        for a in block.get("athletes", []):\n                            identity = a.get("athlete", {})\n                            candidates = set()\n                            for label in (identity.get(\'fullName\'), identity.get(\'displayName\')):\n                                label = norm(label)\n                                candidates.update(names.get(label, []))\n                                candidates.update(names.get(\'tokens:\' + \' \'.join(sorted(label.split())), []))\n                            if not candidates:\n                                continue\n                            # Names only shortlist candidates. Exact date of birth\n                            # must resolve to one official player before facts enter.\n                            athlete_id = identity.get("id")\n                            if not athlete_id:\n                                continue\n                            stats = dict(zip(a.get(\'_keys\', labels), a.get("stats", [])))\n                            def stat(*keys):\n                                for k in keys:\n                                    if k in stats:\n                                        return number(stats[k])\n                                return None\n                            minutes = stat("minutesPlayed", "minutes", "MIN")\n                            if minutes is None or minutes <= 0:\n                                continue\n                            candidate_rows.append((str(athlete_id), c, identity, candidates, {"date": event["date"], "competition": competition,\n                                "opponent": opponent, "minutes": minutes, "goals": stat("goals", "totalGoals", "G"),\n                                "assists": stat("assists", "goalAssists", "A"), "rating": stat("rating"),\n                                "source": source, "minutes_basis": a.get(\'_minutes_basis\', \'reported individual minutes\'),\n                                "coverage": "official name or unique display name plus exact DOB; covered ESPN competitions"}))\n        # Resolve each athlete once, concurrently. Bench players with no verified\n        # minutes never trigger a metadata lookup; no name-only identity is used.\n        identities = {aid:(comp,identity) for aid,comp,identity,_,_ in candidate_rows}\n        def athlete_meta(item):\n            aid,(comp,identity)=item\n            meta=identity if identity.get(\'dateOfBirth\') else self.client.get(f"https://sports.core.api.espn.com/v2/sports/soccer/leagues/{comp}/athletes/{aid}?lang=en&region=us", timeout=6)\n            return aid, meta\n        with futures.ThreadPoolExecutor(max_workers=6) as ex:\n            metadata=dict(ex.map(athlete_meta,identities.items()))\n        unresolved=0\n        for aid,comp,identity,candidates,appearance in candidate_rows:\n            meta=metadata.get(aid) or {}\n            dob=str(meta.get(\'dateOfBirth\',\'\'))[:10]\n            resolved=[pid for pid in candidates if str(meta.get(\'id\',aid))==aid and dob and self.players[pid].get(\'birth_date\')==dob]\n            if len(resolved)!=1:unresolved+=1;continue\n            pid=resolved[0]\n            if any(m.get(\'source\')==appearance[\'source\'] for m in self.matches.get(pid,[])):continue\n            self.matches.setdefault(pid,[]).append(appearance);verified_matches+=1\n        self.coverage[\'all_competitions\'].update(completed_events=len(tasks), available_summaries=available_summaries,\n            verified_player_appearances=verified_matches, missing_summaries=len(tasks)-available_summaries,\n            unresolved_candidate_appearances=unresolved)\n\n    def rss(self):\n        if not self.cfg.enable_news:\n            self.coverage["club_news"] = "disabled"\n            return\n        query = "(" + " OR ".join("site:" + d for d in sorted(PRIMARY_NEWS_DOMAINS)) + ")"\n        def club(t):\n            q = f\'"{t["name"]}" (injury OR suspension OR team news OR sanctions OR charges OR postponement) when:7d {query}\'\n            url = "https://news.google.com/rss/search?" + urlencode({"q": q, "hl": "en-GB", "gl": "GB", "ceid": "GB:en"})\n            try:\n                r = requests.get(url, timeout=6); r.raise_for_status()\n                items = ET.fromstring(r.content).findall("./channel/item")\n                rows = []\n                for item in items[:8]:\n                    source = item.find("source")\n                    source_url = source.attrib.get("url", "") if source is not None else ""\n                    host = urlparse(source_url).hostname or ""\n                    if not any(host == d or host.endswith("." + d) for d in PRIMARY_NEWS_DOMAINS):\n                        continue\n                    title = item.findtext("title") or ""\n                    normalized = norm(title)\n                    # Search relevance is not evidence. Require the club name,\n                    # a squad player\'s exact surname, or that club\'s own publisher.\n                    club_words = norm(t["name"]).split()\n                    surnames = {norm(p["second_name"]) for p in self.players.values() if p["team"] == t["id"]}\n                    relevant_name = any(s and (" " + s + " ") in (" " + normalized + " ") for s in surnames)\n                    relevant_club = all(w in normalized.split() for w in club_words)\n                    concern = any(w in normalized for w in ("injur", "suspend", "team news", "international", "norway", "england", "sanction", "charges", "postpon", "ruled out", "fitness", "minutes", "press conference"))\n                    if not concern or not (relevant_name or relevant_club) or "women" in normalized:\n                        continue\n                    rows.append({"title": item.findtext("title"), "published": item.findtext("pubDate"),\n                                 "url": item.findtext("link"), "publisher_url": source_url,\n                                 "status": "headline only; read source before any tactical adjustment", "numeric_effect": None})\n                return int(t["id"]), rows, None\n            except (requests.RequestException, ET.ParseError) as exc:\n                return int(t["id"]), [], type(exc).__name__\n        errors = []\n        with futures.ThreadPoolExecutor(max_workers=4) as ex:\n            for tid, rows, err in ex.map(club, self.teams.values()):\n                self.headlines[tid] = rows\n                if err:\n                    errors.append(tid)\n        self.coverage["club_news"] = {"state": "partial" if errors else "headlines checked",\n                                     "failed_clubs": errors, "lookback_days": 7,\n                                     "numeric_effect": "none without a verified football consequence"}\n\n    def run(self):\n        self.read_verified_file()\n        self.primary_articles()\n        self.espn()\n        self.rss()\n        write_json(self.cfg.root / "news_evidence.json", {"checked_at": self.now.isoformat(),\n            "coverage": self.coverage, "matches": self.matches, "events": self.events, "club_headlines": self.headlines})\n        return self\n\n\ndef validate_squad(squad, budget, teams_limit=3):\n    ids = squad.player_id.astype(int).tolist()\n    assert len(ids) == 15, "Squad must have exactly fifteen players"\n    assert len(set(ids)) == 15, "Duplicate player"\n    assert squad.position.value_counts().to_dict() == SQUAD_COUNTS, "Invalid position counts"\n    assert squad.team_id.value_counts().max() <= teams_limit, "Club limit exceeded"\n    assert np.isfinite(squad.price).all(), "Missing squad prices"\n    assert int(round(float(squad.price.sum()) * 10)) <= int(round(budget * 10)), "Squad over budget"\n\n\ndef best_xi(squad, points, manual_captain=None, manual_vice=None, eligible_captains=None, eligible_starters=None, secure_starters=None, maximum_conditional=None, eligible_vice=None):\n    ids = squad.player_id.astype(int).tolist()\n    if len(ids) != 15 or set(ids) - set(points):\n        raise ValueError("XI inputs must resolve all fifteen projections")\n    groups = {p: squad.loc[squad.position.eq(p), "player_id"].astype(int).tolist() for p in SQUAD_COUNTS}\n    best, val = None, -math.inf\n    for gk in groups["GK"]:\n        for nd in range(3, 6):\n            for nm in range(2, 6):\n                nf = 10 - nd - nm\n                if not 1 <= nf <= 3:\n                    continue\n                for d in itertools.combinations(groups["DEF"], nd):\n                    for m in itertools.combinations(groups["MID"], nm):\n                        for f in itertools.combinations(groups["FWD"], nf):\n                            xi = (gk,) + d + m + f\n                            if eligible_starters is not None and any(i not in eligible_starters for i in xi):continue\n                            if maximum_conditional is not None and secure_starters is not None and sum(i not in secure_starters for i in xi)>maximum_conditional:continue\n                            if eligible_vice is not None and not any(i in eligible_vice for i in xi if i!=manual_captain):continue\n                            if manual_captain and eligible_captains is not None and manual_captain not in eligible_captains:continue\n                            if manual_vice and eligible_vice is not None and manual_vice not in eligible_vice:continue\n                            if manual_captain and manual_captain not in xi:\n                                continue\n                            if manual_vice and manual_vice not in xi:\n                                continue\n                            captains=[i for i in xi if eligible_captains is None or i in eligible_captains]\n                            if not captains and not manual_captain:continue\n                            cap = manual_captain or max(captains, key=lambda i: (points[i], -i))\n                            if eligible_vice is not None and not any(i!=cap and i in eligible_vice for i in xi):continue\n                            score = sum(points[i] for i in xi) + points[cap]\n                            if score > val:\n                                best, val = xi, score\n    if best is None:\n        raise ValueError("No legal XI includes the requested captain/vice captain")\n    ranked = sorted(best, key=lambda i: (-points[i], i))\n    captain = manual_captain or next(i for i in ranked if eligible_captains is None or i in eligible_captains)\n    vice = manual_vice or next(i for i in ranked if i != captain and (eligible_vice is None or i in eligible_vice))\n    if captain == vice:\n        raise ValueError("Captain and vice captain must differ")\n    bench = [i for i in ids if i not in best]\n    pos = squad.set_index("player_id").position.to_dict()\n    # Exact legal ordering: prioritize actual expected substitution value; GK separately.\n    outfield = sorted([i for i in bench if pos[i] != "GK"], key=lambda i: (-points[i], i))\n    bench_gk = next(i for i in bench if pos[i] == "GK")\n    counts = [sum(pos[i] == p for i in best) for p in ("DEF", "MID", "FWD")]\n    assert len(best) == 11 and counts[0] >= 3 and counts[1] >= 2 and counts[2] >= 1\n    return {"xi_ids": list(best), "bench_ids": outfield + [bench_gk], "captain_id": captain,\n            "vice_id": vice, "formation": "-".join(map(str, counts)), "xi_xpts": sum(points[i] for i in best),\n            "captain_increment": points[captain], "xi_plus_captain": val,\n            "bench_xpts": sum(points[i] for i in bench), "total_squad_xpts": sum(points[i] for i in ids)}\n\n\nclass WildcardEngine:\n    def __init__(self, cfg=None, now=None):\n        self.cfg = cfg or Config()\n        if self.cfg.horizon != 6:\n            raise ValueError("Wildcard primary horizon must be six gameweeks")\n        self.now = utc(now)\n        self.cfg.root.mkdir(parents=True, exist_ok=True)\n        self.client = DataClient(self.cfg.root / "current_source_cache", self.now, self.cfg.cache_hours)\n        self.audit = {"version": VERSION, "generated_at": self.now.isoformat(), "warnings": [],\n                      "model_estimates": ["expected minutes", "starting probability", "xPts", "team goals", "clean-sheet probability"],\n                      "assumptions": {"rate_prior_minutes": 270, "WC_horizon": 6,\n                                      "short_rest_days": 4, "short_rest_minutes_factor": .9,\n                                      "bench_option_weight": .12, "BB_bench_weight": 1.0,\n                                      "role_prior_appearances":2,"start_prior":.8,"play_prior":.9,"60_minute_prior":.75,\n                                      "transfer_threshold_xpts": self.cfg.transfer_threshold,\n                                      "FT_option_value_xpts": self.cfg.ft_option_value},\n                      "calibration": "Transparent model estimates; not backtested/calibrated probabilities"}\n\n    def load(self, boot=None, fixtures=None, histories=None, external_evidence=None):\n        self.boot = boot or self.client.get(API + "bootstrap-static/", required=True)\n        self.fixtures = fixtures if fixtures is not None else self.client.get(API + "fixtures/", required=True)\n        boot = self.boot\n        self.teams = {int(t["id"]): t for t in boot["teams"]}\n        self.players = {int(p["id"]): p for p in boot["elements"] if p.get("can_select", True) and not p.get("removed")}\n        future = sorted([e for e in boot["events"] if utc(e["deadline_time"]) > self.now], key=lambda e: e["id"])\n        if not future:\n            raise RuntimeError("Season has no future deadline; no past-GW recommendation generated")\n        self.gws = [int(e["id"]) for e in future[:self.cfg.projection_weeks]]\n        self.primary = self.gws[:self.cfg.horizon]\n        self.target = self.gws[0]\n        self.deadline = future[0]["deadline_time"]\n        season_year = min(int(e["deadline_time"][:4]) for e in boot["events"])\n        self.season = f"{season_year}/{str(season_year+1)[-2:]}"\n        if len(self.primary) < self.cfg.horizon:\n            self.audit["warnings"].append(f"Season end: only {len(self.primary)} future gameweeks remain")\n        self._manager()\n        raw_fdr, reason = canonical_fdr(boot, self.fixtures, self.cfg, self.now)\n        self.fdr = FDRStore(self.fixtures, self.teams, raw_fdr, reason)\n        self.audit["fdr"] = {"source": self.fdr.source, "fallback_reason": reason,\n                             "colour_policy": "DAY-3 canonical full-precision bands + original gradients",\n                             "player_channel": "D for GK/DEF; A for MID/FWD; DAY-3 model only"}\n        if reason:\n            self.audit["warnings"].append(reason)\n        if histories is None:\n            def history(pid):\n                data = self.client.get(API + f"element-summary/{pid}/", timeout=10)\n                return pid, data.get("history", []) if data else None\n            active = [i for i, p in self.players.items() if number(p.get("minutes")) and float(p["minutes"]) > 0]\n            with futures.ThreadPoolExecutor(max_workers=6) as ex:\n                histories = dict(ex.map(history, active))\n        self.histories = {int(k): v for k, v in histories.items() if v is not None}\n        missing = [i for i, p in self.players.items() if p.get("minutes", 0) > 0 and i not in self.histories]\n        self.audit["missing_histories"] = missing\n        if missing:\n            self.audit["warnings"].append(f"{len(missing)} player histories unavailable; those players excluded from automatic selection")\n        self.news = external_evidence or NewsEvidence(self.cfg, self.client, self.players, self.teams, self.now).run()\n        self.audit["news_coverage"] = self.news.coverage\n        self._projections()\n        self.audit["source_requests"] = self.client.audit\n        write_json(self.cfg.root / "wildcard_model_audit.json", self.audit)\n        write_json(self.cfg.root / "wildcard_current_bootstrap.json", self.boot)\n        write_json(self.cfg.root / "wildcard_current_fixtures.json", self.fixtures)\n        self.scores.to_csv(self.cfg.root / "wildcard_player_projections.csv", index=False)\n        write_json(self.cfg.root / "wildcard_fixture_projections.json", self.fixture_scores)\n        return self\n\n    def _manager(self):\n        rules = self.boot["game_settings"]\n        self.budget = float(self.cfg.budget) if self.cfg.budget is not None else rules["squad_total_spend"] / rules["ui_currency_multiplier"]\n        self.budget_source = "explicit manager budget" if self.cfg.budget is not None else "official starting-budget reference; personal selling budget unknown"\n        self.bb = self.cfg.bench_boost if self.cfg.bench_boost in ("Available", "Used") else "Unknown"\n        self.ft = self.cfg.free_transfers\n        self.ft_source = "explicit manager input" if self.ft is not None else "unknown; plan uses one conservative reference free transfer"\n        entry = self.cfg.entry_id or os.environ.get("FPL_ENTRY_ID", "")\n        self.entry_history = None\n        self.verified_manager_squad = None\n        if entry:\n            self.entry_history = self.client.get(API + f"entry/{int(entry)}/history/")\n            if self.entry_history:\n                # With multiple chips per season, used once != unavailable forever.\n                self.audit["chip_history"] = self.entry_history.get("chips", [])\n            cookie = os.environ.get("FPL_SESSION_COOKIE")\n            if cookie:\n                try:\n                    r = requests.get(API + f"my-team/{int(entry)}/", headers={"Cookie": cookie}, timeout=15)\n                    r.raise_for_status(); own = r.json()\n                    picks = own.get("picks", [])\n                    if len(picks) != 15:\n                        raise ValueError("Authenticated team incomplete")\n                    self.verified_manager_squad = {"picks": picks, "verified": True}\n                    self.budget = (sum(p["selling_price"] for p in picks) + own["transfers"]["bank"]) / rules["ui_currency_multiplier"]\n                    self.budget_source = "authenticated exact selling prices plus bank"\n                    transfer_limit = own.get("transfers", {}).get("limit")\n                    if transfer_limit is not None and self.ft is None:\n                        self.ft = int(transfer_limit); self.ft_source = "authenticated current transfer limit; retained on chip activation"\n                    chips = [c for c in own.get("chips", []) if c.get("name") == "bboost"]\n                    active = [c for c in chips if c.get("start_event", 0) <= self.target <= c.get("stop_event", 99)]\n                    if active and self.cfg.bench_boost == "Auto":\n                        self.bb = "Available" if any(c.get("status_for_entry") in ("available", "active") for c in active) else "Used"\n                except (requests.RequestException, ValueError, KeyError) as exc:\n                    self.audit["warnings"].append("Personal account state unavailable: " + type(exc).__name__)\n        if not math.isfinite(self.budget) or self.budget <= 0:\n            raise ValueError("Invalid budget")\n        if self.ft is not None and not 0 <= self.ft <= 1 + rules.get("max_extra_free_transfers", 4):\n            raise ValueError("Free transfers out of official range")\n        self.bb_relevant = self.bb == "Available" and self.cfg.bench_boost_gameweek in self.primary\n        self.audit["manager"] = {"budget": self.budget, "budget_source": self.budget_source,\n            "bench_boost": self.bb, "bench_boost_planned_gameweek": self.cfg.bench_boost_gameweek,\n            "strong_bench_relevant": self.bb_relevant, "free_transfers": self.ft,\n            "free_transfer_source": self.ft_source}\n        if self.bb == "Unknown":\n            self.audit["warnings"].append("Bench Boost availability unknown; conservative XI-focused strategy used")\n\n    def _projections(self):\n        players = pd.DataFrame(self.boot["elements"])\n        done = [f for f in self.fixtures if f.get("finished")]\n        counts = {tid: sum(tid in (f["team_h"], f["team_a"]) for f in done) for tid in self.teams}\n        league = pd.to_numeric(players.expected_goals).sum() / max(1, 2 * len(done))\n        self.team_stats = {}\n        for tid in self.teams:\n            tp = players[players.team.eq(tid)]\n            gk = tp[tp.element_type.eq(1)]\n            self.team_stats[tid] = {"attack": (pd.to_numeric(tp.expected_goals).sum() + 3 * league) / (counts[tid] + 3),\n                                    "concede": (pd.to_numeric(gk.expected_goals_conceded).sum() + 3 * league) / (counts[tid] + 3)}\n        self.fixture_scores, self.player_evidence = {}, {}\n        rows, pool = [], []\n        scoring = self.boot.get("game_config", {}).get("scoring")\n        if not scoring:\n            raise RuntimeError("Live scoring configuration missing; refusing fixed-season scoring assumptions")\n        top10k = self._top10k()\n        for pid, p in self.players.items():\n            tid, pos = int(p["team"]), POSITIONS.get(int(p["element_type"]))\n            if pos is None:\n                continue\n            hist = self.histories.get(pid, [])\n            role = minutes_model(p, hist)\n            latest = latest_pl(hist, self.teams)\n            if latest: latest["source"] = API + f"element-summary/{pid}/"\n            others = self.news.matches.get(pid, [])\n            verified = [m for m in ([latest] if latest else []) + others if m.get("date") and utc(m["date"]) <= self.now]\n            latest_any = max(verified, key=lambda m: m["date"]) if verified else None\n            self.player_evidence[pid] = {"latest_league_match": latest, "latest_verified_match": latest_any,\n                "role": role, "availability": player_status(p), "external_appearances": others,\n                "coverage": self.news.coverage["all_competitions"], "club_headlines": self.news.headlines.get(tid, [])}\n            if role["base_minutes"] is None:\n                continue\n            total_mins = float(p.get("minutes", 0))\n            peers = players[players.element_type.eq(p["element_type"])]\n            peer_mins = max(1., float(peers.minutes.sum()))\n            xg_prior = float(pd.to_numeric(peers.expected_goals).sum()) / peer_mins * 90\n            xa_prior = float(pd.to_numeric(peers.expected_assists).sum()) / peer_mins * 90\n            xg = number(p.get("expected_goals")); xa = number(p.get("expected_assists"))\n            if xg is None or xa is None:\n                continue\n            season_xg90 = (xg * 90 + 270 * xg_prior) / (total_mins + 270)\n            season_xa90 = (xa * 90 + 270 * xa_prior) / (total_mins + 270)\n            recent = [h for h in hist[-6:] if (number(h.get("minutes")) or 0) > 0]\n            # Use only observed match-level expected data; a missing metric is never zero.\n            def recent_rate(field, season_rate):\n                samples = [(number(item.get(field)), number(item.get("minutes"))) for item in recent]\n                samples = [(metric, minutes) for metric, minutes in samples if metric is not None and minutes is not None and minutes > 0]\n                if len(samples) < 2:\n                    return season_rate\n                recent_minutes = sum(minutes for _, minutes in samples)\n                rate90 = sum(metric for metric, _ in samples) * 90 / recent_minutes\n                weight = min(.35, recent_minutes / (recent_minutes + 900.))\n                return max(0., (1. - weight) * season_rate + weight * rate90)\n            xg90 = recent_rate("expected_goals", season_xg90)\n            xa90 = recent_rate("expected_assists", season_xa90)\n            # Current-season FPL BPS already reflects the live scoring rules.\n            peer_bonus90 = float(pd.to_numeric(peers.bonus).sum()) / peer_mins * 90\n            peer_bps90 = float(pd.to_numeric(peers.bps).sum()) / peer_mins * 90\n            bonus90 = (float(p.get("bonus",0)) * 90 + 450 * peer_bonus90) / (total_mins + 450)\n            bps90 = (float(p.get("bps",0)) * 90 + 450 * peer_bps90) / (total_mins + 450)\n            bps_factor = min(1.20, max(.80, .85 + .15 * bps90 / max(1.,peer_bps90)))\n            saves90 = float(p["saves"]) / max(1, total_mins) * 90\n            cards90 = (float(p["yellow_cards"]) + 3 * float(p["red_cards"])) / max(1, total_mins) * 90\n            threshold = 10 if pos == "DEF" else 12\n            dc_observed = [number(h.get("defensive_contribution")) for h in recent]\n            dc_observed = [v for v in dc_observed if v is not None]\n            # Four pseudomatches temper one-game certainty; no evidence means omitted.\n            dc_probability = (sum(v >= threshold for v in dc_observed) + 4*.25) / (len(dc_observed) + 4) if dc_observed and pos != "GK" else None\n            last_played = latest_any\n            for gw in self.gws:\n                fixtures = self.fdr.for_team(tid, gw, pos)\n                projections = []\n                for fx in fixtures:\n                    health, health_basis = availability(p, fx, self.now, self.news.events)\n                    rest_factor = 1.\n                    rest_days = None\n                    if last_played and last_played.get("date") and fx.get("kickoff"):\n                        rest_days = (utc(fx["kickoff"]) - utc(last_played["date"])).total_seconds() / 86400\n                        if 0 <= rest_days < 4 and (number(last_played.get("minutes")) or 0) >= 60:\n                            rest_factor = .9\n                    pplay, p60, pstart = [role[k] * health * rest_factor for k in ("p_play", "p60", "p_start")]\n                    xm = role["base_minutes"] * health * rest_factor\n                    cond = role["conditional_minutes"]\n                    opp = fx["opponent_id"]\n                    if fx["team_goals"] is not None:\n                        team_goals, conceded, cs = fx["team_goals"], fx["opponent_goals"], fx["clean_sheet_probability"]\n                        multiplier = team_goals / max(.1, self.team_stats[tid]["attack"])\n                        team_basis = "VORTEX upstream team-goal model"\n                    else:\n                        raise RuntimeError(f"DAY-3 canonical projected goals missing for fixture {fx[\'fixture_id\']}")\n                    key = "GKP" if pos == "GK" else pos\n                    exposure = xm / 90\n                    eg, ea = xg90 * exposure * multiplier, xa90 * exposure * multiplier\n                    components = {"appearance": scoring["short_play"] * pplay + (scoring["long_play"] - scoring["short_play"]) * p60,\n                        "goals": scoring["goals_scored"][key] * eg, "assists": scoring["assists"] * ea,\n                        "clean_sheet": scoring["clean_sheets"][key] * cs * p60,\n                        "saves": _vx_phase10_expected_floor_poisson(saves90*cond/90,3) * pplay if pos == "GK" else 0.,\n                        "bonus": min(3 * pplay, bonus90 * exposure * bps_factor),\n                        "cards": -cards90 * exposure,\n                        "conceded": scoring["goals_conceded"][key] * _vx_phase10_expected_floor_poisson(conceded*cond/90,2) * pplay}\n                    # Missing DC evidence is explicitly an omitted projection component,\n                    # never an observed zero. Model coverage is recorded on the card.\n                    if dc_probability is not None:\n                        components["defensive"] = scoring.get("defensive_contribution", {}).get(key, 0) * dc_probability * pplay\n                    xp = float(sum(components.values()))\n                    proj = dict(fx, xpts=xp, expected_minutes=xm, conditional_minutes=cond,\n                        p_start=pstart, p_play=pplay, p60=p60, availability_basis=health_basis,\n                        rest_days=rest_days, rest_factor=rest_factor, team_goals=team_goals,\n                        opponent_goals=conceded, clean_sheet_probability=cs, expected_goals=eg, expected_assists=ea,\n                        components=components, team_model_source=team_basis,\n                        model_version="rolling-xg-shrinkage-bps-2026-v1",\n                        model_inputs={"xg90":round(xg90,5),"xa90":round(xa90,5),\n                            "bps90":round(bps90,4),"bonus90":round(bonus90,4),"bps_factor":round(bps_factor,4),\n                            "defcon_probability":None if dc_probability is None else round(dc_probability,4),\n                            "defcon_threshold":threshold if pos != "GK" else None},\n                        missing_components=["defensive contributions"] if dc_probability is None and pos != "GK" else [])\n                    projections.append(proj)\n                    self.fixture_scores[(pid, fx["fixture_id"])] = proj\n                    # Scheduled PL rest for subsequent fixtures is a forecast scenario.\n                    last_played = {"date": fx["kickoff"], "minutes": xm} if fx.get("kickoff") else last_played\n                xp = sum(f["xpts"] for f in projections)\n                rows.append({"player_id": pid, "gameweek": gw, "gameweek_expected_points": xp,\n                    "expected_minutes": sum(f["expected_minutes"] for f in projections),\n                    "p_start": max((f["p_start"] for f in projections), default=0.),\n                    "p_play": max((f["p_play"] for f in projections), default=0.),\n                    "fixture_count": len(projections), "fixtures": ", ".join(f["opponent"] + (" (H)" if f["home"] else " (A)") for f in projections) or "BLANK",\n                    "xg90": xg90, "xa90": xa90,\n                    "model_version": "rolling-xg-shrinkage-bps-2026-v1"})\n            pool.append(dict(p, player_id=pid, player_name=p["web_name"], team_id=tid,\n                team_name=self.teams[tid]["name"], team_short=self.teams[tid]["short_name"], position=pos,\n                price=float(p["now_cost"])/self.boot["game_settings"]["ui_currency_multiplier"],\n                p_start=role["p_start"], expected_minutes=role["base_minutes"],\n                news_monitor=any(e.get(\'player_id\')==pid and e.get(\'monitor\') for e in self.news.events),\n                top10k_ownership=top10k.get(pid), availability=player_status(p)["label"]))\n        self.audit["projection_model"] = {\n            "version":"rolling-xg-shrinkage-bps-2026-v1",\n            "features":["FPL xG/xA recent-season shrinkage","FPL current BPS-informed bonus",\n                        "defcon probability with four pseudomatches","minutes and opponents"],\n            "limitations":["BPS is a bounded proxy, not an exact match-level bonus simulation",\n                           "DEFCON probability omitted without match observations",\n                           "Future lineups and xPts remain estimates"]}\n        self.scores = pd.DataFrame(rows)\n        self.pool = pd.DataFrame(pool)\n        if self.pool.empty:\n            raise RuntimeError("No players have sufficient current evidence to project")\n        totals = self.scores[self.scores.gameweek.isin(self.primary)].groupby("player_id").gameweek_expected_points.sum()\n        first = self.scores[self.scores.gameweek.eq(self.target)].set_index("player_id").gameweek_expected_points\n        self.pool["xpts_wildcard_horizon"] = self.pool.player_id.map(totals)\n        self.pool["xpts_freehit_gw"] = self.pool.player_id.map(first)  # compatibility only, not a FH engine change\n        self.pool["value_score"] = self.pool.xpts_wildcard_horizon / self.pool.price\n        self.pool["market_net"] = pd.to_numeric(self.pool.transfers_in_event) - pd.to_numeric(self.pool.transfers_out_event)\n        self.pool["minutes_security_score"] = self.pool.p_start\n        self._selection_features()  # role/minutes gates and structural evidence, before names\n        self.scores = self.scores.sort_values(["player_id", "gameweek"]).reset_index(drop=True)\n        assert not self.scores.duplicated(["player_id", "gameweek"]).any()\n        assert np.isfinite(self.scores.gameweek_expected_points).all()\n\n    def _top10k(self):\n        path = self.cfg.model_inputs / "top10k_ownership.json"\n        if not path.exists():\n            self.audit["top10k"] = "unavailable"\n            return {}\n        d = json.loads(path.read_text())\n        if int(d.get("gameweek", -1)) != self.target or not d.get("source_url") or not d.get("sample_size"):\n            self.audit["top10k"] = "rejected: wrong gameweek or missing source/sample size"\n            return {}\n        if not 0 <= (self.now - utc(d["fetched_at"])).total_seconds() <= self.cfg.cache_hours*3600:\n            self.audit["top10k"] = "rejected: stale"\n            return {}\n        self.audit["top10k"] = {k: d.get(k) for k in ("source_url", "sample_size", "fetched_at", "gameweek")}\n        values = {int(k): number(v) for k, v in d.get("ownership_pct", {}).items()}\n        if any(v is None or not 0 <= v <= 100 for v in values.values()):\n            raise ValueError("Invalid top-10K ownership values")\n        return values\n\n    def points(self, gw):\n        return self.scores[self.scores.gameweek.eq(gw)].set_index("player_id").gameweek_expected_points.to_dict()\n\n    def _selection_features(self):\n        """Availability -> minutes -> role -> fixtures -> horizon value.\n\n        Role labels require source evidence. Advanced positions, shots in the box,\n        crosses and central-striker status are not invented from FPL position codes.\n        Fixtures/team strength/role returns already enter xPts; no FDR or penalty\n        bonus is added to those points a second time.\n        """\n        p=self.pool.copy();records=[]\n        for _,player in p.iterrows():\n            pid=int(player.player_id);pos=str(player.position)\n            fs=[f for (player_id,_),f in self.fixture_scores.items() if player_id==pid and f[\'gameweek\'] in self.primary]\n            role=self.player_evidence[pid][\'role\']\n            def total(key):return sum(float(f.get(\'components\',{}).get(key) or 0) for f in fs)\n            def mean(key):\n                values=[float(f[key]) for f in fs if number(f.get(key)) is not None]\n                return sum(values)/len(values) if values else None\n            attacking=total(\'goals\')+total(\'assists\');cs=total(\'clean_sheet\');saves=total(\'saves\');defcon=total(\'defensive\')\n            routes=[]\n            if attacking>0:routes.append(\'goal/assist projection\')\n            if cs>0:routes.append(\'clean-sheet projection\')\n            if saves>0 and pos==\'GK\':routes.append(\'save-point projection\')\n            if defcon>0:routes.append(\'defensive-contribution projection\')\n            if number(player.get(\'penalties_order\'))==1:routes.append(\'official first-choice penalties\')\n            if number(player.get(\'corners_and_indirect_freekicks_order\'))==1 or number(player.get(\'direct_freekicks_order\'))==1:routes.append(\'official first-choice set pieces\')\n            available=str(player.status)==\'a\' and not bool(player.news_monitor)\n            sample=int(role.get(\'sample_size\') or 0)\n            cover=float(player.p_start)>=self.cfg.cover_start_probability and float(player.expected_minutes)>=self.cfg.cover_expected_minutes\n            core=float(player.p_start)>=self.cfg.core_start_probability and float(player.expected_minutes)>=self.cfg.core_expected_minutes\n            playable_weeks=sum(any(float(f.get(\'p_start\') or 0)>=self.cfg.cover_start_probability and float(f.get(\'expected_minutes\') or 0)>=self.cfg.cover_expected_minutes for f in fs if f[\'gameweek\']==gw) for gw in self.primary)\n            eligible=available and cover and sample>=self.cfg.minimum_history_appearances and bool(routes) and playable_weeks>=self.cfg.minimum_playable_weeks\n            reasons=[]\n            if not available:reasons.append(\'availability/news needs confirmation\')\n            if not cover:reasons.append(\'minutes below dependable-cover policy\')\n            if sample<self.cfg.minimum_history_appearances:reasons.append(\'insufficient recent playing-time evidence\')\n            if not routes:reasons.append(\'no evidenced route to points\')\n            if playable_weeks<self.cfg.minimum_playable_weeks:reasons.append(\'too few credible starts in the six-week run\')\n            records.append(dict(wc_eligible=eligible,wc_core=eligible and core,wc_routes=routes,wc_monitor_reason=\'; \'.join(reasons),\n                wc_fixture_mean=mean(\'fdr\'),wc_clean_sheet_mean=mean(\'clean_sheet_probability\'),wc_team_goals_mean=mean(\'team_goals\'),\n                wc_attacking_points=attacking,wc_defensive_points=cs+defcon,wc_save_points=saves,wc_playable_weeks=playable_weeks,\n                wc_role_evidence_score=len(routes),wc_history_sample=sample))\n        data=pd.DataFrame(records,index=p.index)\n        for col in data:p[col]=data[col]\n        self.pool=p\n        opportunities={int(pid):0 for pid in p.player_id}\n        for gw in self.primary:\n            masks=self._weekly_selection_masks(gw)\n            for pid in masks[\'captains\']:opportunities[pid]+=1\n        p[\'wc_captaincy_weeks\']=p.player_id.map(opportunities)\n        p[\'wc_price_tier\']=\'VALUE\';p[\'wc_premium_allowed\']=True\n        for pos in SQUAD_COUNTS:\n            peers=p[p.position.eq(pos)&p.wc_eligible]\n            if peers.empty:continue\n            premium_cut=float(peers.price.quantile(self.cfg.premium_price_quantile))\n            premium=p.position.eq(pos)&p.price.ge(premium_cut)\n            p.loc[premium,\'wc_price_tier\']=\'PREMIUM\'\n            mid=p.position.eq(pos)&~premium&p.price.gt(float(peers.price.median()))\n            p.loc[mid,\'wc_price_tier\']=\'MID_PRICE\'\n            if pos in (\'MID\',\'FWD\'):\n                exceptional=p.xpts_wildcard_horizon.ge(float(peers.xpts_wildcard_horizon.quantile(.90)))&p.wc_core\n                p.loc[premium,\'wc_premium_allowed\']=(p.wc_captaincy_weeks.ge(self.cfg.minimum_captaincy_weeks)|exceptional)[premium]\n            elif pos==\'GK\':\n                cs_cut=float(peers.wc_clean_sheet_mean.dropna().quantile(.75)) if peers.wc_clean_sheet_mean.notna().any() else float(\'inf\')\n                save_cut=float(peers.wc_save_points.quantile(.80));return_cut=float(peers.xpts_wildcard_horizon.quantile(.75))\n                worth=p.wc_core&(p.wc_clean_sheet_mean.ge(cs_cut)|(p.wc_save_points.gt(0)&p.wc_save_points.ge(save_cut)&p.xpts_wildcard_horizon.ge(return_cut)))\n                p.loc[premium,\'wc_premium_allowed\']=worth[premium]\n        p[\'wc_eligible\']=p.wc_eligible&p.wc_premium_allowed\n        p.loc[~p.wc_premium_allowed,\'wc_monitor_reason\']=p.loc[~p.wc_premium_allowed,\'wc_monitor_reason\'].apply(lambda x:(x+\'; \' if x else \'\')+\'premium opportunity cost not justified by captaincy or sustained output\')\n        p[\'wildcard_score\']=p.xpts_wildcard_horizon.where(p.wc_eligible,0.)\n        self.pool=p\n        self.audit[\'selection_policy\']={\'version\':VERSION,\'hierarchy\':[\'availability/minutes\',\'evidenced role/routes\',\'six-week fixtures\',\'same-horizon xPts\',\'price/value\',\'squad flexibility\',\'captaincy\',\'ownership/market context only\'],\n            \'thresholds\':{key:getattr(self.cfg,key) for key in (\'core_start_probability\',\'core_expected_minutes\',\'cover_start_probability\',\'cover_expected_minutes\',\'captain_start_probability\',\'captain_expected_minutes\',\'minimum_history_appearances\',\'maximum_conditional_xi\',\'minimum_playable_weeks\',\'minimum_captaincy_weeks\',\'premium_price_quantile\',\'points_tie_tolerance\')},\n            \'missing_role_data\':\'No advanced-position/shot-volume/cross/central-striker claim without source data\',\n            \'double_counting\':\'No fixture, role, ownership or price-rise bonus added to existing xPts\',\n            \'eligible_players\':int(p.wc_eligible.sum()),\'monitor_players\':int((~p.wc_eligible).sum())}\n        write_json(self.cfg.root/\'wildcard_selection_eligibility.json\',p[[\'player_id\',\'player_name\',\'position\',\'price\',\'wc_price_tier\',\'wc_eligible\',\'wc_core\',\'wc_monitor_reason\',\'wc_routes\',\'wc_fixture_mean\',\'wc_clean_sheet_mean\',\'wc_team_goals_mean\',\'wc_playable_weeks\',\'wc_captaincy_weeks\']].to_dict(\'records\'))\n\n    def _weekly_selection_masks(self,gw):\n        cover=set();core=set();captain_base=set();points=self.points(gw)\n        for _,p in self.pool.iterrows():\n            if not bool(p.get(\'wc_eligible\',False)):continue\n            pid=int(p.player_id);fixtures=[f for (player_id,_),f in self.fixture_scores.items() if player_id==pid and f[\'gameweek\']==gw]\n            def passes(start,minutes):return any(float(f.get(\'p_start\') or 0)>=start and float(f.get(\'expected_minutes\') or 0)>=minutes for f in fixtures)\n            if passes(self.cfg.cover_start_probability,self.cfg.cover_expected_minutes):cover.add(pid)\n            if passes(self.cfg.core_start_probability,self.cfg.core_expected_minutes):core.add(pid)\n            if p.position in (\'MID\',\'FWD\') and not bool(p.news_monitor) and passes(self.cfg.captain_start_probability,self.cfg.captain_expected_minutes):captain_base.add(pid)\n        # Captaincy candidates must compare well with other secure attackers this week.\n        values=[points.get(pid,0.) for pid in captain_base if points.get(pid,0.)>0]\n        threshold=float(np.quantile(values,.80)) if values else float(\'inf\')\n        captains={pid for pid in captain_base if points.get(pid,0.)>=threshold and points.get(pid,0.)>0}\n        return {\'cover\':cover,\'core\':core,\'captains\':captains}\n\n    def _structure_lineup(self,squad,gw,manual_captain=None,manual_vice=None):\n        if \'wc_eligible\' not in self.pool:self._selection_features()\n        masks=self._weekly_selection_masks(gw)\n        # Vice-captain may be any secure attacker, not only the highest projection tier.\n        secure=set(self.pool.loc[self.pool.position.isin([\'MID\',\'FWD\']),\'player_id\'].astype(int))&masks[\'core\']\n        return best_xi(squad,self.points(gw),manual_captain,manual_vice,\n            eligible_captains=masks[\'captains\'],eligible_starters=masks[\'cover\'],secure_starters=masks[\'core\'],\n            maximum_conditional=self.cfg.maximum_conditional_xi,eligible_vice=secure)\n\n    def optimize(self,budget=None):\n        from wildcard_slide3 import build_wildcard_squad\n        budget=self.budget if budget is None else float(budget)\n        from wildcard_slide3 import attach_unprojected_goalkeepers\n        self.pool=attach_unprojected_goalkeepers(self,self.pool)\n        identities=set(map(int,self.pool.player_id))\n        self.slide3_plan=build_wildcard_squad(self.boot[\'elements\'],\n            list(self.teams.values()),self.fixtures,self.target,budget=budget,\n            context={\'models\':self.pool.to_dict(\'records\'),\'weekly_models\':self.scores.to_dict(\'records\'),\'available_gameweeks\':self.primary,\n                \'currency_multiplier\':self.boot[\'game_settings\'][\'ui_currency_multiplier\'],\n                \'completed_gameweeks\':sum(bool(event.get(\'finished\')) for event in self.boot[\'events\']),\n                \'solver_seconds\':self.cfg.solver_seconds,\'bench_boost\':self.bb})\n        chosen={player[\'id\'] for player in self.slide3_plan[\'squad\']}\n        return self.pool[self.pool.player_id.isin(chosen)].copy(),self.slide3_plan[\'total_cost\']\n\n\n\n\n    def score_plan(self,squad,gameweeks=None):\n        from wildcard_slide3 import weekly_plan_rows\n        if not getattr(self,\'slide3_plan\',None) or {int(p[\'id\']) for p in self.slide3_plan[\'squad\']}!=set(map(int,squad.player_id)):\n            raise ValueError(\'Use the new automated Wildcard draft; regenerate selection first\')\n        rows=weekly_plan_rows(self.slide3_plan,squad,self.scores)\n        return pd.DataFrame([r for r in rows if gameweeks is None or r[\'gameweek\'] in gameweeks])\n\n\n\n    def transfer_plan(self,squad):\n        from wildcard_slide3 import build_transfer_rows\n        return build_transfer_rows(self.slide3_plan,self.ft)\n\n    def player_card(self, pid):\n        if self.pool.loc[self.pool.player_id.eq(int(pid)),"expected_minutes"].isna().all():\n            from wildcard_slide3 import unprojected_keeper_card\n            return unprojected_keeper_card(self,pid)\n        p = self.pool[self.pool.player_id.eq(int(pid))].iloc[0]\n        evidence = self.player_evidence[int(pid)]\n        evidence = dict(evidence, availability=dict(evidence[\'availability\']))\n        news_notes=[e for e in self.news.events if e.get(\'player_id\')==pid]\n        if any(e.get(\'monitor\') for e in news_notes) and not evidence[\'availability\'][\'flagged\']:\n            evidence[\'availability\'].update(label=\'MONITOR\',flagged=True,\n                news=\'Official recent report raises a fitness/workload concern; no confirmed absence or fabricated probability.\')\n        weeks = []\n        for gw in self.primary:\n            fixtures = []\n            for fx in self.fdr.for_team(int(p.team_id), gw, p.position):\n                fixtures.append(dict(self.fixture_scores[(int(pid), fx["fixture_id"])]))\n            weeks.append({"gameweek": gw, "fixtures": fixtures, "xpts": sum(f["xpts"] for f in fixtures),\n                          "expected_minutes": sum(f["expected_minutes"] for f in fixtures), "blank": not fixtures})\n        first = self.scores[self.scores.player_id.eq(pid)&self.scores.gameweek.eq(self.target)].iloc[0]\n        penalties = number(p.get("penalties_order"))\n        corners = number(p.get("corners_and_indirect_freekicks_order"))\n        freekicks = number(p.get("direct_freekicks_order"))\n        own = number(p.selected_by_percent)\n        latest = evidence["latest_verified_match"]\n        fixtures = [f for w in weeks for f in w["fixtures"]]\n        fdrmean = float(np.mean([f["fdr"] for f in fixtures])) if fixtures else None\n        card = {"id": int(pid), "name": str(p.player_name), "full_name": p.first_name+" "+p.second_name,\n            "club": str(p.team_short), "club_name": str(p.team_name), "team_id": int(p.team_id),\n            "position": str(p.position), "price": float(p.price), "photo_code": int(p.code),\n            "expected_points": float(first.gameweek_expected_points), "horizon_expected_points": float(p.xpts_wildcard_horizon),\n            "expected_minutes": float(first.expected_minutes), "base_expected_minutes": float(p.expected_minutes),\n            "conditional_minutes": evidence["role"]["conditional_minutes"], "starting_probability": float(first.p_start),\n            "ownership_pct": own, "top10k_ownership_pct": number(p.get("top10k_ownership")),\n            "transfers_in": int(p.transfers_in_event), "transfers_out": int(p.transfers_out_event),\n            "net_transfers": int(p.market_net), "market_direction": "IN" if p.market_net > 0 else "OUT" if p.market_net < 0 else "FLAT",\n            "price_change_event": number(p.get("cost_change_event")),\n            "availability": evidence["availability"], "injury_status": evidence["availability"]["label"],\n            "rotation_risk": "HIGH" if first.p_start < .65 else "MEDIUM" if first.p_start < .85 else "LOW",\n            "penalties_order": penalties, "corners_order": corners, "free_kicks_order": freekicks,\n            "role_note": "First penalty taker" if penalties == 1 else "Set-piece involvement" if corners == 1 or freekicks == 1 else "No first-choice set-piece duty confirmed",\n            "recent_minutes": evidence["role"]["recent_minutes"], "history_sample_size": evidence["role"]["sample_size"],\n            "latest_match": latest, "latest_match_text": evidence_line(latest), "latest_league_match": evidence["latest_league_match"],\n            "match_coverage": evidence["coverage"], "club_news": evidence["club_headlines"],\n            "verified_news": [e for e in self.news.events if e.get("player_id") == pid or (not e.get(\'player_id\') and e.get("team_id") == int(p.team_id))],\n            "weeks": weeks, "next_fixtures": [f["opponent"]+(" (H)" if f["home"] else " (A)") for f in fixtures],\n            "fdr": [f["fdr"] for f in fixtures], "fdr_mean": fdrmean, "fdr_source": self.fdr.source,\n            "value_score": float(p.value_score), "xg90": float(first.xg90), "xa90": float(first.xa90),\n            "projection_limitations": sorted(set(v for f in fixtures for v in f["missing_components"])),\n            "selection_policy":VERSION,"selection_eligible":bool(p.wc_eligible),"selection_classification":"CORE" if p.wc_core and p.wc_eligible else "PLAYABLE COVER" if p.wc_eligible else "MONITOR",\n            "selection_reason":str(p.wc_monitor_reason),"routes_to_points":list(p.wc_routes),"price_tier":str(p.wc_price_tier),"captaincy_gameweeks":int(p.wc_captaincy_weeks),\n            "alternative_ids": [], "is_bench": False, "is_captain": False, "is_vice_captain": False}\n        near = next((f for f in weeks[0]["fixtures"]), None)\n        opponent = near["opponent"]+(" at home" if near["home"] else " away") if near else "a blank gameweek"\n        role = card["role_note"].lower()\n        card["reason"] = (f"{card[\'name\']} offers {card[\'horizon_expected_points\']:.1f} model xPts over {len(self.primary)} GWs for £{card[\'price\']:.1f}m. "\n                          f"Recent minutes {\', \'.join(map(str,card[\'recent_minutes\']))} support {card[\'base_expected_minutes\']:.0f} expected minutes and {card[\'starting_probability\']:.0%} starting probability. "\n                          f"The opening fixture is {opponent}; {role}. "\n                          f"{card[\'xg90\']+card[\'xa90\']:.2f} shrunk xGI/90 and {card[\'value_score\']:.2f} horizon xPts per £m explain the price/role balance.")\n        if card["availability"]["flagged"]:\n            card["reason"] += " " + card["injury_status"] + ": " + card["availability"]["news"]\n        return card\n\n    def contract(self, squad, plan=None):\n        validate_squad(squad, self.budget)\n        from wildcard_slide3 import build_wildcard_squad\n        slide3_plan=getattr(self,\'slide3_plan\',None)\n        if slide3_plan is None or {p[\'id\'] for p in slide3_plan[\'squad\']}!=set(map(int,squad.player_id)):\n            slide3_plan=build_wildcard_squad(self.boot[\'elements\'],list(self.teams.values()),self.fixtures,\n                self.target,budget=self.budget,context={\'models\':self.pool.to_dict(\'records\'),\'weekly_models\':self.scores.to_dict(\'records\'),\n                    \'currency_multiplier\':self.boot[\'game_settings\'][\'ui_currency_multiplier\'],\n                    \'completed_gameweeks\':sum(bool(e.get(\'finished\')) for e in self.boot[\'events\']),\n                    \'available_gameweeks\':self.primary,\'solver_seconds\':self.cfg.solver_seconds,\'bench_boost\':self.bb})\n            raise ValueError(\'Manual Wildcard squad differs from the automated Slide 3 plan; use Auto for this standalone recommendation.\')\n        plan = self.score_plan(squad) if plan is None else plan\n        first = plan.iloc[0]\n        xiids = first.xi.player_id.astype(int).tolist()\n        benchids = first.bench.player_id.astype(int).tolist()\n        cards = [self.player_card(int(i)) for i in squad.player_id]\n        byid = {p["id"]: p for p in cards}\n        selected = set(byid)\n        future = self.transfer_plan(squad)\n        bank = round(self.budget-float(squad.price.sum()), 1)\n        for p in cards:\n            p["is_bench"] = p["id"] in benchids\n            p["is_captain"] = p["id"] == int(first.captain_player_id)\n            p["is_vice_captain"] = p["id"] == int(first.vice_captain_player_id)\n            p["bench_label"] = "BENCH GK" if p["is_bench"] and p["position"] == "GK" else "BENCH " + str(benchids.index(p["id"])+1) if p["is_bench"] else "STARTING XI"\n            if p["is_bench"]:\n                same = [byid[i] for i in xiids if byid[i]["position"] == p["position"]]\n                rivals = sorted(same, key=lambda q: -q["expected_points"])\n                if p.get(\'projection_status\')==\'unavailable\':p[\'bench_reason\']=\'Cheapest backup goalkeeper; playing-time and points projections unavailable\'\n                else:\n                    p["bench_reason"] = (f"{p[\'expected_points\']:.1f} current-GW xPts; {p[\'starting_probability\']:.0%} estimated starting chance. "\n                        + (f"{rivals[0][\'name\']} has the stronger legal-XI projection in this position." if rivals else "Formation and captaincy make this the better bench arrangement."))\n            rest = squad[squad.player_id.ne(p["id"])]\n            alts = self.pool[~self.pool.player_id.isin(selected) & self.pool.position.eq(p["position"])\n                             & self.pool.price.le(p["price"]) & self.pool.p_start.ge(.65) & self.pool.status.eq("a")]\n            counts = rest.team_id.value_counts()\n            alts = alts[alts.team_id.map(counts).fillna(0).lt(3)].nlargest(2, "value_score")\n            p["alternative_ids"] = alts.player_id.astype(int).tolist()\n        available = self.pool[~self.pool.player_id.isin(selected)&self.pool.wc_eligible]\n        watch_ids = available.nlargest(4, "xpts_wildcard_horizon").player_id.astype(int).tolist()\n        watch = [self.player_card(i) for i in watch_ids]\n        for p in watch:\n            p["reason"] = f"£{p[\'price\']:.1f}m, {p[\'horizon_expected_points\']:.1f} horizon xPts and {p[\'base_expected_minutes\']:.0f} base minutes. " + p["role_note"] + "; consider if squad budget or confirmed lineup news changes."\n        # Affordability is relative to actual positional prices, with credible\n        # minutes, above-median value and a legal replacement club allocation.\n        affordable=[]\n        for position in SQUAD_COUNTS:\n            peers=self.pool[self.pool.position.eq(position)]\n            options=available[available.position.eq(position)&available.wc_core\n                              &available.price.le(peers.price.median())\n                              &available.value_score.ge(peers.value_score.median())]\n            legal=[]\n            for _,option in options.iterrows():\n                for _,out in squad[squad.position.eq(position)&squad.price.ge(option.price)].iterrows():\n                    rest=squad[squad.player_id.ne(out.player_id)]\n                    if int(rest.team_id.eq(option.team_id).sum())<int(self.boot["game_settings"]["squad_team_limit"]):\n                        legal.append(int(option.player_id));break\n            candidates=options[options.player_id.isin(legal)]\n            if not candidates.empty:\n                p=self.player_card(int(candidates.nlargest(1,\'value_score\').iloc[0].player_id))\n                p[\'reason\']=f"{p[\'value_score\']:.2f} six-GW xPts per £m, {p[\'base_expected_minutes\']:.0f} base minutes and {p[\'starting_probability\']:.0%} starting estimate. Price is at or below the current positional median; compare the projected points sacrificed before replacing a starter."\n                affordable.append(p)\n        # Compare ownership within position, while requiring credible minutes,\n        # involvement, xPts and value. No fixed arbitrary 8% cutoff.\n        candidates = self.pool[self.pool.wc_eligible&self.pool.wc_core].copy()\n        candidates["ownership_rank"] = pd.to_numeric(candidates.selected_by_percent).groupby(candidates.position).rank(pct=True)\n        candidates["projection_rank"] = candidates.groupby("position").xpts_wildcard_horizon.rank(pct=True)\n        diff = candidates[candidates.ownership_rank.le(.5)&candidates.projection_rank.ge(.65)]\n        diff_ids = diff.nlargest(3, "value_score").player_id.astype(int).tolist()\n        differentials = [byid.get(i) or self.player_card(i) for i in diff_ids]\n        for p in differentials:\n            p["differential_reason"] = f"{p[\'ownership_pct\']:.1f}% overall ownership is in the lower half for this position; {p[\'horizon_expected_points\']:.1f} horizon xPts with {p[\'starting_probability\']:.0%} starting estimate. " + p["role_note"] + "; returns remain uncertain."\n            if p["top10k_ownership_pct"] is not None:\n                p["differential_reason"] += f" Sampled top-10K ownership: {p[\'top10k_ownership_pct\']:.1f}%."\n        bad = self.pool[~self.pool.wc_eligible & ~self.pool.player_id.isin(selected)].copy()\n        bad["ownership_number"] = pd.to_numeric(bad.selected_by_percent)\n        avoid = [self.player_card(int(i)) for i in bad.nlargest(5, "ownership_number").player_id]\n        for p in avoid:\n            p["selection_reason"] = str(self.pool.set_index("player_id").loc[p["id"],"wc_monitor_reason"])\n            p["classification"] = "AVOID" if p["injury_status"] in ("INJURED", "SUSPENDED", "UNAVAILABLE") or p["starting_probability"] < .35 else "MONITOR"\n            p["avoid_reason"] = (p["availability"]["news"] or p.get("selection_reason") or f"Recent minutes {p[\'recent_minutes\']} imply {p[\'starting_probability\']:.0%} starting probability.") + f" £{p[\'price\']:.1f}m for {p[\'expected_points\']:.1f} opening-GW xPts."\n        if len(avoid) < 5:\n            self.audit["warnings"].append(f"Only {len(avoid)} evidence-backed avoid/monitor candidates; no invented fifth candidate")\n        cap, vice = byid[int(first.captain_player_id)], byid[int(first.vice_captain_player_id)]\n        def captain(p):\n            return {"id": p["id"], "name": p["name"], "reason": f"{p[\'expected_points\']:.2f} current-GW xPts with {p[\'starting_probability\']:.0%} starting probability. {p[\'role_note\']}. Rotation: {p[\'rotation_risk\'].lower()}."}\n        entities = dict(byid)\n        for p in watch+differentials+avoid+affordable: entities[p["id"]] = p\n        for p in cards:\n            for i in p["alternative_ids"]: entities[i] = self.player_card(i)\n        for a in future:\n            for k in ("out_player_id", "in_player_id"):\n                if a.get(k): entities[a[k]] = self.player_card(a[k])\n        team_rows = []\n        for tid, t in self.teams.items():\n            values = [self.fdr.for_team(tid, gw, "MID") for gw in self.primary]\n            ratings = [f["fdr"] for fs in values for f in fs]\n            team_rows.append({"team_id": tid, "club": t["short_name"], "name": t["name"],\n                              "weeks": [{"gameweek": gw, "fixtures": fs} for gw, fs in zip(self.primary, values)],\n                              "average_fdr": float(np.mean(ratings)) if ratings else None})\n        team_rows.sort(key=lambda r: (r["average_fdr"] is None, r["average_fdr"] or 9))\n        from fixture_slide import build_fixture_slide\n        fixture_case = build_fixture_slide(team_rows, self.primary, self.boot[\'elements\'],\n            {\'chip\':\'wildcard\', \'player_models\':self.pool.to_dict(\'records\'),\n             \'currency_multiplier\':self.boot[\'game_settings\'].get(\'ui_currency_multiplier\',10), \'official_fixtures\':self.fixtures})\n        summary = {"formation": first.formation, "squad_value": float(squad.price.sum()), "money_in_bank": bank,\n            "starting_xi_xpts": float(first.xi_xpts), "bench_xpts": number(first.bench_xpts),\n            "total_squad_xpts": float(first.xi_xpts+first.bench_xpts) if number(first.bench_xpts) is not None else None,\n            "captain_increment": float(first.captain_incremental_xpts), "xi_with_captain_xpts": float(first.xi_plus_captain_xpts),\n            "total_expected_points": float(first.xi_plus_captain_xpts),\n            "horizon_xi_with_captain_xpts": float(plan.xi_plus_captain_xpts.sum())}\n        d = {"schema_version": VERSION, "chip": "wildcard", "gameweek": self.target,\n            "season": self.season, "deadline": self.deadline, "generated_at": self.now.isoformat(),\n            "horizon_gameweeks": self.primary, "planning_gameweeks": self.gws,\n            "budget_total": self.budget, "budget_used": summary["squad_value"], "budget_source": self.budget_source,\n            "formation": first.formation, "squad": cards, "starting_xi_ids": xiids,\n            "bench_ids": benchids, "bench_order": [byid[i]["name"] for i in benchids],\n            "captain_picks": {"captain": captain(cap), "vice_captain": captain(vice)},\n            "watchlist": watch, "differentials": differentials, "avoid_list": avoid,\n            "affordable_alternatives":affordable,\n            "transfer_plan": future, "slide3_plan": slide3_plan, "fixture_table": team_rows, "fixture_case": fixture_case, "summary_stats": summary,\n            "media_entities": {str(i): p for i, p in entities.items()},\n            "clubs": {t["short_name"]: {"id": t["id"], "name": t["name"], "code": t["code"]} for t in self.teams.values()},\n            "model_audit": dict(self.audit, solver=squad.attrs.get("solver")),\n            "data_sources": {"official": API, "fdr": self.fdr.source, "news": self.news.coverage},\n            "fdr_source": self.fdr.source, "fdr_fallback_reason": self.fdr.reason,\n            "bench_boost": self.bb, "free_transfers": self.ft,\n            "club_news_notes":[e for e in self.news.events if not e.get(\'player_id\')],\n            "bank_strategy":{"selection":f"Minimum spend within {self.cfg.points_tie_tolerance:g} model points of the best dependable six-GW structure",\n                "bank":bank,"price_assumption":"Current prices; future affordability is conditional",\n                "transfer_path":future},\n            "rotation_plan": [{"gameweek": int(r.gameweek), "xi_ids": r.xi.player_id.astype(int).tolist(),\n                               "bench_ids": r.bench.player_id.astype(int).tolist()} for _, r in plan.iterrows()]}\n        evidence = getattr(self, \'defender_two_way\', None)\n        if evidence is not None:\n            squad_ids = {int(p[\'id\']) for p in cards}\n            d[\'defender_two_way\'] = {\n                \'sample_gameweeks\': evidence[\'sample_gameweeks\'],\n                \'cohort_size\': evidence[\'cohort_size\'],\n                \'median_dc90\': evidence[\'median_dc90\'],\n                \'median_xgi90\': evidence[\'median_xgi90\'],\n                \'source\': evidence[\'source\'],\n                \'players\': {\n                    str(item[\'player_id\']): {\n                        \'player_id\': item[\'player_id\'], \'minutes\': item[\'minutes\'],\n                        \'dc90\': item[\'dc90\'], \'xgi90\': item[\'xgi90\'],\n                        \'quadrant\': item[\'quadrant\'], \'source\': item[\'data_source\']\n                    }\n                    for item in evidence[\'all_defenders\'] if item[\'player_id\'] in squad_ids\n                },\n            }\n        validate_contract(d)\n        stamps=[r[\'fetched_at\'] for r in self.client.audit if r.get(\'fetched_at\') and r.get(\'url\') in (API+\'bootstrap-static/\',API+\'fixtures/\')]\n        d[\'source_snapshot_at\']=min(stamps,key=utc) if stamps else self.now.isoformat()\n        # Feed the new intro from the existing core client, not from a manager squad.\n        intro_projections = [{\'id\': int(row.player_id), \'position\': row.position, \'price\': float(row.price),\n                              \'expected_minutes\': float(row.expected_minutes), \'horizon_expected_points\': float(row.xpts_wildcard_horizon),\n                              \'analysis_gws\': list(self.primary)} for row in self.pool.itertuples()]\n        d[\'wildcard_case\'] = build_wildcard_case(self.boot, self.fixtures, histories=self.histories,\n            projections=intro_projections, verified_events=getattr(self.news, \'events\', []),\n            future_matches=getattr(self.news, \'future_matches\', []), refreshed_at=d[\'source_snapshot_at\'], now=self.now,\n            fdr_records=self.fdr.records)\n        from wildcard_slide3 import enrich_plan\n        d[\'slide3_plan\']=enrich_plan(d[\'slide3_plan\'],d,self.boot,self.pool.to_dict(\'records\'),self.scores.to_dict(\'records\'))\n        d[\'wildcard_case\'] = enrich_case(d[\'wildcard_case\'],d,self.boot,getattr(self,\'verified_manager_squad\',None))\n        if d[\'wildcard_case\'][\'analysis_gws\'] != self.primary:\n            raise ValueError(\'Wildcard intro and squad optimisation use different Gameweek windows\')\n        d["data_hash"] = hashlib.sha256(json.dumps(clean_json(d), sort_keys=True, ensure_ascii=False).encode()).hexdigest()\n        self.audit["squad_hash"] = d["data_hash"]\n        write_json(self.cfg.root / "wildcard_data.json", d)\n        return clean_json(d)\n\n\ndef validate_contract(d):\n    assert d["chip"] == "wildcard"\n    squad = d["squad"]\n    byid = {p["id"]: p for p in squad}\n    assert len(squad) == len(byid) == 15, "Invalid squad identity/size"\n    assert pd.Series([p["position"] for p in squad]).value_counts().to_dict() == SQUAD_COUNTS\n    assert pd.Series([p["team_id"] for p in squad]).value_counts().max() <= 3\n    assert sum(round(p["price"]*10) for p in squad) <= round(d["budget_total"]*10)\n    xi, bench = d["starting_xi_ids"], d["bench_ids"]\n    assert len(xi) == len(set(xi)) == 11 and len(bench) == len(set(bench)) == 4\n    assert set(xi).isdisjoint(bench) and set(xi+bench) == set(byid)\n    pos = [byid[i]["position"] for i in xi]\n    assert pos.count("GK") == 1 and 3 <= pos.count("DEF") <= 5 and 2 <= pos.count("MID") <= 5 and 1 <= pos.count("FWD") <= 3\n    assert d["formation"] == "-".join(str(pos.count(p)) for p in ("DEF", "MID", "FWD"))\n    assert byid[bench[-1]]["position"] == "GK" and all(byid[i]["position"] != "GK" for i in bench[:3])\n    cap = d["captain_picks"]["captain"]["id"]; vice = d["captain_picks"]["vice_captain"]["id"]\n    assert cap != vice and cap in xi and vice in xi\n    s = d["summary_stats"]\n    near = lambda a,b: math.isclose(a,b,rel_tol=0,abs_tol=1e-8)\n    assert near(s["starting_xi_xpts"],sum(byid[i]["expected_points"] for i in xi))\n    if all(byid[i][\'expected_points\'] is not None for i in bench):\n        assert near(s[\'bench_xpts\'],sum(byid[i][\'expected_points\'] for i in bench))\n        assert near(s[\'total_squad_xpts\'],s[\'starting_xi_xpts\']+s[\'bench_xpts\'])\n    else:\n        assert s[\'bench_xpts\'] is None and s[\'total_squad_xpts\'] is None\n    assert near(s["xi_with_captain_xpts"],s["starting_xi_xpts"]+byid[cap]["expected_points"])\n    assert near(s["squad_value"],sum(p["price"] for p in squad))\n    assert near(s["money_in_bank"],round(d["budget_total"]-s["squad_value"],1))\n    fixture_values = {}\n    assert len(d[\'fixture_table\'])==len(d[\'clubs\'])\n    assert len({r[\'team_id\'] for r in d[\'fixture_table\']})==len(d[\'clubs\'])\n    lookup={}\n    for row in d[\'fixture_table\']:\n        for week in row[\'weeks\']:\n            assert week[\'gameweek\'] in d[\'horizon_gameweeks\']\n            for f in week[\'fixtures\']:\n                assert f[\'team_id\']==row[\'team_id\'] and f[\'gameweek\']==week[\'gameweek\']\n                assert f[\'fdr_style\']==fdr_display(f[\'fdr\'])\n                lookup[(f[\'fixture_id\'],row[\'team_id\'])]=f\n    for p in list(d["media_entities"].values())+squad:\n        assert p["id"] > 0 and p["team_id"] in {v["id"] for v in d["clubs"].values()}\n        assert [w["gameweek"] for w in p["weeks"]] == d["horizon_gameweeks"]\n        if p.get(\'projection_status\')==\'unavailable\':\n            assert p[\'position\']==\'GK\' and p[\'id\'] in bench and p[\'id\'] not in xi\n            assert p[\'expected_points\'] is None and all(w[\'xpts\'] is None for w in p[\'weeks\'])\n            for w in p[\'weeks\']:\n                for f in w[\'fixtures\']:\n                    assert f[\'team_id\']==p[\'team_id\'] and f[\'gameweek\']==w[\'gameweek\']\n                    assert f[\'fdr_style\']==fdr_display(f[\'fdr\']) and f[\'xpts\'] is None\n            continue\n        assert near(p["horizon_expected_points"],sum(w["xpts"] for w in p["weeks"]))\n        assert near(p["expected_points"],p["weeks"][0]["xpts"])\n        assert math.isfinite(p["expected_points"]) and 0 <= p["starting_probability"] <= 1\n        for w in p["weeks"]:\n            assert near(w["xpts"],sum(f["xpts"] for f in w["fixtures"]))\n            for f in w["fixtures"]:\n                assert f["team_id"] == p["team_id"] and f["opponent_id"] != p["team_id"]\n                assert f["gameweek"] == w["gameweek"] and 1 <= f["fdr"] <= 5\n                assert f["fdr_style"] == fdr_display(f["fdr"])\n                canonical=lookup[(f[\'fixture_id\'],p[\'team_id\'])]\n                raw=canonical[\'defence_fdr\'] if p[\'position\'] in (\'GK\',\'DEF\') else canonical[\'attack_fdr\']\n                assert d[\'fdr_source\'].startswith(\'VORTEX\'), \'Official FDR must never enter DAY-3 canonical rendering\'\n                assert near(f[\'fdr\'],raw), \'Card/table FDR source mismatch\'\n                assert near(f["xpts"],sum(f["components"].values()))\n                key=(f["fixture_id"],p["team_id"],p["position"] in ("GK","DEF"))\n                if key in fixture_values:assert fixture_values[key]==f["fdr"]\n                fixture_values[key]=f["fdr"]\n    bank = s["money_in_bank"]\n    state=set(byid)\n    entities={int(k):p for k,p in d[\'media_entities\'].items()}\n    for a in d["transfer_plan"]:\n        assert near(a["money_before"],bank)\n        if a["action"] == "TRANSFER":\n            assert a[\'out_player_id\'] in state and a[\'in_player_id\'] not in state\n            state.remove(a[\'out_player_id\']);state.add(a[\'in_player_id\'])\n            assert near(a["money_after"],round(bank+a["out_price"]-a["in_price"],1)) and a["money_after"] >= 0\n        else:assert a["transfers_required"]==0 and near(a["money_after"],bank)\n        bank=a["money_after"]\n        assert set(a[\'squad_after\'])==state and len(a[\'squad_after\'])==15\n        after=[entities[i] for i in state]\n        assert pd.Series([p[\'position\'] for p in after]).value_counts().to_dict()==SQUAD_COUNTS\n        assert pd.Series([p[\'team_id\'] for p in after]).value_counts().max()<=3\n        assert near(sum(p[\'price\'] for p in after)+bank,d[\'budget_total\'])\n    return True\n\n\n\n\ndef _short_date(value):\n    if not value:\n        return \'TBC\'\n    try:\n        dt = utc(value)\n    except Exception:\n        text = str(value).strip()\n        m = re.search(r\'(\\d{1,2}\\s+[A-Za-z]{3,9}\\s+\\d{4})\', text)\n        return m.group(1) if m else text[:30]\n    try:\n        return dt.strftime(\'%-d %b %Y\')\n    except Exception:\n        return dt.strftime(\'%d %b %Y\').lstrip(\'0\')\n\n\ndef _availability_status_label(player):\n    status = (player.get(\'availability\') or {}).get(\'label\') or player.get(\'injury_status\') or \'AVAILABLE\'\n    status = str(status).strip().upper()\n    return status if status else \'AVAILABLE\'\n\n\ndef _availability_reason(player):\n    notes = []\n    news = str((player.get(\'availability\') or {}).get(\'news\') or \'\').strip()\n    if news:\n        notes.append(\' \'.join(news.split()))\n    for event in player.get(\'verified_news\', []):\n        summary = str(event.get(\'summary\') or \'\').strip()\n        if summary:\n            notes.append(\' \'.join(summary.split()))\n    if notes:\n        return max(notes, key=len)[:120]\n    if _availability_status_label(player) == \'AVAILABLE\':\n        return \'No official availability flag from current tracked sources.\'\n    return \'Official flag detected; monitor the latest team news before locking the wildcard.\'\n\n\ndef _availability_return(player):\n    for event in player.get(\'verified_news\', []):\n        for key in (\'return_date\', \'unavailable_until\'):\n            if event.get(key):\n                return _short_date(event.get(key))\n    news = str((player.get(\'availability\') or {}).get(\'news\') or \'\')\n    m = re.search(r\'(?:Expected return|Back|Available|Fit)[: ]+([^.;]+)\', news, re.I)\n    if m:\n        return \' \'.join(m.group(1).split())[:28]\n    if _availability_status_label(player) == \'AVAILABLE\':\n        return \'Available now\'\n    return \'TBC\'\n\n\ndef _availability_event_type(event):\n    text = \' \'.join(str(event.get(k) or \'\') for k in (\'summary\', \'source_url\')).lower()\n    if \'friendly\' in text:\n        return \'Friendly\'\n    if \'international\' in text:\n        return \'International\'\n    return \'Club\'\n\n\ndef _availability_upcoming_watch(player, max_items=2):\n    items = []\n    seen = set()\n    for event in player.get(\'verified_news\', []):\n        summary = str(event.get(\'summary\') or \'\').strip()\n        if not summary:\n            continue\n        tag = _availability_event_type(event)\n        line = f"{tag}: {\' \'.join(summary.split())}"\n        if line not in seen:\n            items.append(line[:70]); seen.add(line)\n        if len(items) >= max_items:\n            return items\n    for week in player.get(\'weeks\', []):\n        gw = week.get(\'gameweek\')\n        for fx in week.get(\'fixtures\', []):\n            opp = str(fx.get(\'opponent\') or \'?\')\n            home = \'H\' if fx.get(\'home\') else \'A\'\n            kickoff = _short_date(fx.get(\'kickoff\')) if fx.get(\'kickoff\') else \'\'\n            line = f"Club: GW{gw} {opp} ({home})" + (f" • {kickoff}" if kickoff and kickoff != \'TBC\' else \'\')\n            if line not in seen:\n                items.append(line[:70]); seen.add(line)\n            if len(items) >= max_items:\n                return items\n    return items or [\'No immediate fixture context found\']\n\n\ndef _availability_risk(player):\n    label = _availability_status_label(player)\n    chance = number((player.get(\'availability\') or {}).get(\'chance_pct\'))\n    monitor = any(e.get(\'monitor\') for e in player.get(\'verified_news\', []))\n    if label in (\'INJURED\', \'SUSPENDED\', \'UNAVAILABLE\') or (chance is not None and chance <= 50):\n        return \'HIGH\'\n    if label != \'AVAILABLE\' or monitor or (chance is not None and chance < 100) or str(player.get(\'rotation_risk\', \'\')).upper() == \'HIGH\':\n        return \'MEDIUM\'\n    return \'LOW\'\n\n\n\n\n\ndef build_manifest(d):\n    """Build the finalized Wildcard visual sequence from the canonical contract.\n\n    The resulting manifest is embedded into the finalized Wildcard Scene JSON.\n    Wildcard narration never reads the optimization contract directly.\n    """\n    validate_contract(d)\n    scenes = []\n    def add(title, kind, players=None, extra=None, seconds=None):\n        scene = {"scene_id": f"wc_{len(scenes)+1:02d}", "title": title, "gameweek": d["gameweek"],\n                 "visual_type": kind, "players": players or [], "tables": [], "narration_text": "",\n                 "presentation_version": VERSION,\n                 "duration_seconds": seconds, "timing": {"status": "awaiting scene-json narration", "audio_padding_seconds": 1.5},\n                 "animation": {"crossfade_seconds": .5}, "source_data_hash": d.get("data_hash")}\n        if extra: scene.update(extra)\n        scenes.append(scene)\n\n    case=d[\'wildcard_case\']\n    scenes.append({\'scene_id\':\'wc_01\', \'title\':\'When Should You Play Your Wildcard?\', \'gameweek\':d[\'gameweek\'],\n        \'visual_type\':\'wc_blueprint\', \'players\':[], \'tables\':[], \'wildcard_case\':case,\n        \'narration_text\':blueprint_narration(case), \'narration_timeline\':generate_segments(case),\n        \'display_title\':\'WILDCARD ANALYSIS\', \'subheading\':\'Question · Evidence · Trade-off · Verdict\', \'presentation_version\':VERSION,\n        \'duration_seconds\':None, \'timing\':{\'status\':\'awaiting measured narration\',\'audio_padding_seconds\':.65},\n        \'animation\':{\'clock\':\'measured video seconds\',\'crossfade_seconds\':.5}, \'source_data_hash\':d.get(\'data_hash\')})\n    from wildcard_slide3 import generate_wildcard_slide3_narration\n    squad_plan=d[\'slide3_plan\']\n    timeline=generate_wildcard_slide3_narration(squad_plan)\n    add(f"Wildcard Plan: GW{squad_plan[\'next_gw\']}–GW{squad_plan[\'end_gw\']}","wc_squad_plan",extra={\n        \'squad_plan\':squad_plan,\'narration_timeline\':timeline,\n        \'narration_text\':\' \'.join(segment[\'narration_text\'] for segment in timeline),\n        \'subheading\':\'Built for sustained points, secure minutes and fixture value\'})\n    add(d[\'fixture_case\'][\'label\'], \'wc_analysis\', extra={\n        \'fixture_case\':d[\'fixture_case\'],\'narration_timeline\':d[\'fixture_case\'][\'narration_timeline\'],\n        \'subheading\':d[\'fixture_case\'][\'range_label\'],\'display_title\':\'FIXTURE ANALYSIS\'})\n    from wildcard_final_slide import from_contract, generate_wildcard_final_slide_narration\n    captaincy_plan=from_contract(d)\n    final_timeline=generate_wildcard_final_slide_narration(captaincy_plan,d[\'slide3_plan\'])\n    add(f"Captaincy Plan: GW{captaincy_plan[\'next_gw\']}–GW{captaincy_plan[\'end_gw\']}", \'wc_captaincy_final\', extra={\n        \'captaincy_plan\':captaincy_plan,\'narration_timeline\':final_timeline,\n        \'narration_text\':\' \'.join(item[\'narration_text\'] for item in final_timeline),\n        \'subheading\':f"Gameweek {captaincy_plan[\'next_gw\']} to Gameweek {captaincy_plan[\'end_gw\']}",\n        \'timing\':{\'status\':\'awaiting measured narration\',\'audio_padding_seconds\':2.5},\n        \'animation\':{\'clock\':\'measured narration segments\',\'final_hold_seconds\':2.5},\n        \'display_title\':\'CAPTAINCY PLAN\'})\n    # The intro transitions directly to the fixture analysis named in the agenda.\n    fixture_scene=next(scene for scene in scenes if scene[\'visual_type\']==\'wc_analysis\')\n    scenes.remove(fixture_scene);scenes.insert(1,fixture_scene)\n    for index,scene in enumerate(scenes,1):scene[\'scene_id\']=f\'wc_{index:02d}\'\n    return scenes\n\nWILDCARD_NARRATION_VERSION = "analyst-driven-eight-slides-v10"\nNARRATION_TARGET_WPM = 145\nNARRATION_MIN_SECONDS = 8 * 60\nNARRATION_MAX_SECONDS = 13 * 60\n\n\ndef build_scene_json(d, scenes):\n    """Freeze every Wildcard fact needed by visuals and narration into one authority."""\n    validate_contract(d)\n    scene_json = {\n        "scene_id": "wildcard_final",\n        "chip": "wildcard",\n        "gameweek": d["gameweek"],\n        "generated_at": d.get("generated_at"),\n        "data_hash": d.get("data_hash"),\n        "presentation_version": VERSION,\n        "narration_version": WILDCARD_NARRATION_VERSION,\n        "horizon_gameweeks": d["horizon_gameweeks"],\n        "fdr_source": d["fdr_source"],\n        "fdr_fallback_reason": d.get("fdr_fallback_reason"),\n        "budget_total": d["budget_total"],\n        "formation": d["formation"],\n        "summary_stats": d["summary_stats"],\n        "starting_xi_ids": d["starting_xi_ids"],\n        "bench_ids": d["bench_ids"],\n        "captain_picks": d["captain_picks"],\n        "bench_boost": d.get("bench_boost", "Unknown"),\n        "squad": d["squad"],\n        "clubs": d["clubs"],\n        "fixture_table": d["fixture_table"],\n        "watchlist": d.get("watchlist", []),\n        "affordable_alternatives": d.get("affordable_alternatives", []),\n        "differentials": d.get("differentials", []),\n        "avoid_list": d.get("avoid_list", []),\n        "transfer_plan": d.get("transfer_plan", []),\n        "club_news_notes": d.get("club_news_notes", []),\n        "bank_strategy": d.get("bank_strategy", {}),\n        "scenes": scenes,\n    }\n    validate_scene_json(scene_json)\n    return scene_json\n\n\ndef _scene_player_map(scene_json):\n    return {int(p["id"]): p for p in scene_json["squad"]}\n\n\ndef _near(a, b, tol=1e-8):\n    return math.isclose(float(a), float(b), rel_tol=0, abs_tol=tol)\n\n\ndef validate_scene_json(scene_json):\n    """Validate the finalized Scene JSON before any Wildcard narration/TTS/video."""\n    errors = []\n    try:\n        if scene_json.get("scene_id") != "wildcard_final": errors.append("source_scene_id must be wildcard_final")\n        if scene_json.get("chip") != "wildcard": errors.append("chip must be wildcard")\n        squad = scene_json.get("squad", [])\n        byid = {int(p["id"]): p for p in squad}\n        if len(squad) != 15 or len(byid) != 15: errors.append("Final squad must contain exactly 15 unique players")\n        counts = pd.Series([p.get("position") for p in squad]).value_counts().to_dict() if squad else {}\n        if counts != SQUAD_COUNTS: errors.append(f"Position counts invalid: {counts}")\n        if squad and pd.Series([p.get("team_id") for p in squad]).value_counts().max() > 3: errors.append("More than three players from one club")\n        squad_value = sum(float(p["price"]) for p in squad)\n        summary = scene_json.get("summary_stats", {})\n        if not _near(summary.get("squad_value", -1), squad_value): errors.append("Squad value does not equal finalized player prices")\n        bank = round(float(scene_json.get("budget_total", 0)) - squad_value, 1)\n        if not _near(summary.get("money_in_bank", -999), bank): errors.append("Money in bank does not equal budget minus squad value")\n        xi = list(map(int, scene_json.get("starting_xi_ids", [])))\n        bench = list(map(int, scene_json.get("bench_ids", [])))\n        if len(xi) != 11 or len(set(xi)) != 11: errors.append("Starting XI must contain 11 unique players")\n        if len(bench) != 4 or len(set(bench)) != 4: errors.append("Bench must contain four unique players")\n        if set(xi).intersection(bench) or set(xi + bench) != set(byid): errors.append("Starting XI/bench partition is invalid")\n        if xi and all(i in byid for i in xi):\n            pos = [byid[i]["position"] for i in xi]\n            legal = pos.count("GK") == 1 and 3 <= pos.count("DEF") <= 5 and 2 <= pos.count("MID") <= 5 and 1 <= pos.count("FWD") <= 3\n            if not legal: errors.append("Starting XI formation is illegal")\n            formation = "-".join(str(pos.count(p)) for p in ("DEF", "MID", "FWD"))\n            if formation != scene_json.get("formation"): errors.append("Formation label does not match Starting XI")\n        if bench and bench[-1] in byid and byid[bench[-1]]["position"] != "GK": errors.append("Bench goalkeeper is not the reserve goalkeeper")\n        cap = int(scene_json["captain_picks"]["captain"]["id"])\n        vice = int(scene_json["captain_picks"]["vice_captain"]["id"])\n        if cap == vice or cap not in xi or vice not in xi: errors.append("Captain and vice-captain must be different Starting XI players")\n        if xi and all(i in byid for i in xi):\n            xi_xpts = sum(float(byid[i]["expected_points"]) for i in xi)\n            if not _near(summary.get("starting_xi_xpts", -1), xi_xpts): errors.append("Starting XI xPts mismatch")\n        if bench and all(i in byid for i in bench):\n            if all(byid[i][\'expected_points\'] is not None for i in bench):\n                bench_xpts=sum(float(byid[i][\'expected_points\']) for i in bench)\n                if not _near(summary.get(\'bench_xpts\',-1),bench_xpts):errors.append(\'Bench xPts mismatch\')\n            elif summary.get(\'bench_xpts\') is not None:errors.append(\'Unavailable reserve projections must not become zero or invented points\')\n        # Every player scene must be a byte-for-byte factual projection of the finalized squad card for core fields.\n        expected_types=[\'wc_blueprint\',\'wc_analysis\',\'wc_squad_plan\',\'wc_captaincy_final\']\n        if [s.get(\'visual_type\') for s in scene_json.get(\'scenes\',[])]!=expected_types:\n            errors.append(\'Wildcard requires exactly four ordered slides\')\n        squad_scenes=[s for s in scene_json.get(\'scenes\',[]) if s.get(\'visual_type\')==\'wc_squad_plan\']\n        if len(squad_scenes)==1:\n            plan=squad_scenes[0][\'squad_plan\']\n            if {int(p[\'id\']) for p in plan[\'squad\']}!=set(byid):errors.append(\'Slide 3 squad differs from canonical squad\')\n            revealed=[pid for s in squad_scenes[0][\'narration_timeline\'] if s[\'type\'] in (\'players\',\'bench\') for pid in s[\'player_ids\']]\n            if len(revealed)!=15 or set(revealed)!=set(byid):errors.append(\'Slide 3 requires all 15 individual player reveals\')\n        # Transfer plan legality and budget validity.\n        state = set(byid); current_bank = float(summary.get("money_in_bank", 0))\n        all_entities = {int(p["id"]): p for p in squad + scene_json.get("watchlist", []) + scene_json.get("affordable_alternatives", []) + scene_json.get("differentials", []) + scene_json.get("avoid_list", [])}\n        for move in scene_json.get("transfer_plan", []):\n            if not _near(move.get("money_before", current_bank), current_bank): errors.append(f"GW{move.get(\'gameweek\')}: transfer-plan starting bank mismatch")\n            if move.get("action") == "TRANSFER":\n                out_id, in_id = int(move["out_player_id"]), int(move["in_player_id"])\n                if out_id not in state or in_id in state: errors.append(f"GW{move.get(\'gameweek\')}: illegal transfer identities")\n                if out_id in state: state.remove(out_id)\n                state.add(in_id)\n                expected_bank = round(current_bank + float(move["out_price"]) - float(move["in_price"]), 1)\n                if not _near(move.get("money_after", -999), expected_bank) or expected_bank < 0: errors.append(f"GW{move.get(\'gameweek\')}: transfer budget invalid")\n            elif move.get("action") == "ROLL":\n                if int(move.get("transfers_required", 0)) != 0: errors.append(f"GW{move.get(\'gameweek\')}: ROLL cannot require a transfer")\n            else: errors.append(f"GW{move.get(\'gameweek\')}: unknown transfer-plan action")\n            current_bank = float(move.get("money_after", current_bank))\n        # Scene hashes and Gameweeks must match the finalized authority.\n        for sc in scene_json.get("scenes", []):\n            if sc.get("gameweek") != scene_json.get("gameweek"): errors.append(f"{sc.get(\'scene_id\')}: Gameweek mismatch")\n            if sc.get("source_data_hash") != scene_json.get("data_hash"): errors.append(f"{sc.get(\'scene_id\')}: source hash mismatch")\n    except Exception as exc:\n        errors.append(f"Validation exception: {exc}")\n    return {"passed": not errors, "errors": errors, "checked_at": utc().isoformat(), "source_scene_id": scene_json.get("scene_id")}\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\ndef _section(scene, narration, source_fields, blocks=None, pause_before=0.0, pause_after=0.0):\n    seconds = len(narration.split()) / NARRATION_TARGET_WPM * 60 + pause_before + pause_after\n    return {"section_id": scene["scene_id"], "slide_id": scene["scene_id"], "visual_type": scene["visual_type"],\n            "narration": narration, "estimated_duration_seconds": round(seconds, 2),\n            "source_fields": sorted(set(source_fields)), "conditional_blocks_used": blocks or [],\n            "pause_before_seconds": pause_before, "pause_after_seconds": pause_after}\n\n\n\ndef narration_for_scene(scene, scene_json):\n    kind=scene[\'visual_type\']\n    if kind==\'wc_blueprint\':\n        text=blueprint_narration(scene[\'wildcard_case\'])\n        fields=[\'scenes.wildcard_case.reasons\',\'scenes.wildcard_case.analysis_gws\']\n    elif kind==\'wc_analysis\':\n        text=scene[\'fixture_case\'][\'narration_text\']\n        fields=[\'scenes.fixture_case.rows\',\'scenes.fixture_case.gameweeks\']\n    elif kind==\'wc_squad_plan\':\n        text=\' \'.join(segment[\'narration_text\'] for segment in scene[\'narration_timeline\'])\n        fields=[\'scenes.squad_plan\',\'scenes.narration_timeline\']\n    elif kind==\'wc_captaincy_final\':\n        text=\' \'.join(segment[\'narration_text\'] for segment in scene[\'narration_timeline\'])\n        fields=[\'scenes.captaincy_plan\',\'scenes.narration_timeline\']\n    else:raise ValueError(\'Unsupported slide in the four-slide Wildcard video: \'+kind)\n    return _section(scene,text,fields,[],pause_after=0.0)\n\ndef build_narration_object(scene_json, validation_report_path=""):\n    report = validate_scene_json(scene_json)\n    if not report["passed"]:\n        raise RuntimeError("Narration blocked: Wildcard Scene JSON validation failed.\\nResolve the listed data inconsistencies before generating video assets.\\n" + "\\n".join(report["errors"]))\n    sections=[narration_for_scene(scene,scene_json) for scene in scene_json["scenes"]]\n    total=round(sum(float(s["estimated_duration_seconds"]) for s in sections),2)\n    if not math.isfinite(total) or total <= 0:\n        raise RuntimeError("Narration must have a finite, positive natural duration")\n    summary=scene_json["summary_stats"]\n    obj={"gameweek":scene_json["gameweek"],"source_scene_id":"wildcard_final","validation_status":"passed",\n         "validation_report_path":str(validation_report_path),"total_estimated_duration_seconds":total,\n         "target_wpm":NARRATION_TARGET_WPM,"narration_version":WILDCARD_NARRATION_VERSION,"sections":sections,\n         "final_summary":{"formation":scene_json["formation"],"squad_value":summary["squad_value"],\n                          "money_in_bank":summary["money_in_bank"],"captain":scene_json["captain_picks"]["captain"]["name"],\n                          "vice_captain":scene_json["captain_picks"]["vice_captain"]["name"],\n                          "starting_xi_xpts":summary["starting_xi_xpts"]}}\n    validate_narration_object(scene_json,obj)\n    quality=_narration_quality_control(scene_json,obj)\n    obj["quality_control"]=quality\n    if not quality["passed"]:\n        failed=[name for name,passed in quality["checks"].items() if not passed]\n        raise RuntimeError("Narration blocked: Wildcard Scene JSON validation failed.\\nResolve the listed data inconsistencies before generating video assets.\\n"+"\\n".join(failed))\n    return obj\n\n\ndef validate_narration_object(scene_json, narration_object):\n    errors=[]\n    if narration_object.get("source_scene_id")!="wildcard_final":errors.append("Narration source_scene_id mismatch")\n    if narration_object.get("gameweek")!=scene_json.get("gameweek"):errors.append("Narration Gameweek mismatch")\n    if narration_object.get("validation_status")!="passed":errors.append("Narration validation status is not passed")\n    final=narration_object.get("final_summary",{});summary=scene_json["summary_stats"]\n    expected={"formation":scene_json["formation"],"squad_value":summary["squad_value"],"money_in_bank":summary["money_in_bank"],\n              "captain":scene_json["captain_picks"]["captain"]["name"],"vice_captain":scene_json["captain_picks"]["vice_captain"]["name"],\n              "starting_xi_xpts":summary["starting_xi_xpts"]}\n    if final!=expected:errors.append("Final narration summary does not match finalized Scene JSON")\n    scene_ids=[s["scene_id"] for s in scene_json["scenes"]]\n    if [s["slide_id"] for s in narration_object.get("sections",[])]!=scene_ids:errors.append("Narration section order does not match finalized scenes")\n    banned=("guaranteed","must own","must-own","certain haul","easy points","definitely starts")\n    for sec in narration_object.get("sections",[]):\n        lower=sec.get("narration","").lower()\n        if any(term in lower for term in banned):errors.append(f"{sec.get(\'section_id\')}: prohibited certainty language")\n        if not sec.get("source_fields"):errors.append(f"{sec.get(\'section_id\')}: missing source-field traceability")\n    if errors:raise RuntimeError("Narration blocked: Wildcard Scene JSON validation failed.\\nResolve the listed data inconsistencies before generating video assets.\\n"+"\\n".join(errors))\n    return True\n\n\n\ndef _narration_quality_control(scene_json, narration_object):\n    checks={}\n    checks["dynamic_gameweek"]=narration_object.get("gameweek")==scene_json.get("gameweek")\n    checks["single_source_scene"]=narration_object.get("source_scene_id")=="wildcard_final"\n    byid=_scene_player_map(scene_json);xi=set(map(int,scene_json.get("starting_xi_ids",[])))\n    cap=int(scene_json["captain_picks"]["captain"]["id"]);vice=int(scene_json["captain_picks"]["vice_captain"]["id"])\n    checks["captain_vice_legal"]=cap!=vice and cap in xi and vice in xi\n    squad_scene=next((s for s in scene_json.get(\'scenes\',[]) if s.get(\'visual_type\')==\'wc_squad_plan\'),{})\n    player_segments=[s for s in squad_scene.get(\'narration_timeline\',[]) if s.get(\'type\') in (\'players\',\'bench\')]\n    revealed=[pid for s in player_segments for pid in s.get(\'player_ids\',[])]\n    checks[\'all_player_reveals_traceable\']=(len(player_segments)==15 and len(revealed)==15 and set(revealed)==set(byid)\n        and all(len(s.get(\'player_ids\',[]))==1 and s.get(\'narration_text\') for s in player_segments))\n    checks["traceable_sections"]=all(sec.get("source_fields") and sec.get("narration") for sec in narration_object.get("sections",[]))\n    checks["conditional_blocks_structured"]=all(isinstance(sec.get("conditional_blocks_used"),list) for sec in narration_object.get("sections",[]))\n    checks["captain_names_synced"]=narration_object.get("final_summary",{}).get("captain")==byid[cap]["name"] and narration_object.get("final_summary",{}).get("vice_captain")==byid[vice]["name"]\n    return {"passed":all(checks.values()),"checks":checks}\n\n\ndef validate_render_script(narration_object, script_segments):\n    """Prevent editable TXT or downstream code from drifting away from validated Scene-JSON narration."""\n    expected=narration_segments(narration_object)\n    errors=[]\n    if [s.get("id") for s in script_segments] != [s.get("id") for s in expected]:\n        errors.append("Narration scene headers differ from validated narration object")\n    if len(script_segments)!=len(expected):errors.append("Narration section count differs from validated narration object")\n    for got,want in zip(script_segments,expected):\n        if got.get("text") != want.get("text"):\n            errors.append(f"{want.get(\'id\')}: narration text differs from finalized Scene JSON narration")\n    if errors:\n        raise RuntimeError("Narration blocked: Wildcard Scene JSON validation failed.\\nResolve the listed data inconsistencies before generating video assets.\\n"+"\\n".join(errors))\n    return True\n\ndef narration_segments(narration_object):\n    return [{"id":s["slide_id"],"gameweek":narration_object["gameweek"],"mood":"analysis","text":s["narration"],\n             "visual":s["visual_type"],"story_version":WILDCARD_NARRATION_VERSION,\n             "pause_before_seconds":s.get("pause_before_seconds",0),"pause_after_seconds":s.get("pause_after_seconds",0)}\n            for s in narration_object["sections"]]\n\n\ndef attach_verified_narration(scene_json, path, encode_script, decode_script, validation_report_path="", narration_json_path=None, script_saver=None, policy="Regenerate"):\n    """Generate canonical Wildcard narration from finalized Scene JSON only."""\n    narration_object=build_narration_object(scene_json,validation_report_path=validation_report_path)\n    segments=narration_segments(narration_object)\n    prefix=(f"# DATA_HASH: {scene_json[\'data_hash\']}\\n# STORY_VERSION: {WILDCARD_NARRATION_VERSION}\\n"\n            f"# SOURCE_SCENE_ID: wildcard_final\\n")\n    expected=prefix+encode_script(segments)\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    if script_saver is not None:\n        actual=script_saver(path,expected,scene_json[\'data_hash\'],policy)\n        for section,segment in zip(narration_object[\'sections\'],actual):\n            section[\'narration\']=segment[\'text\']\n            section[\'estimated_duration_seconds\']=round(len(segment[\'text\'].split())/NARRATION_TARGET_WPM*60+section.get(\'pause_before_seconds\',0)+section.get(\'pause_after_seconds\',0),2)\n        narration_object[\'total_estimated_duration_seconds\']=sum(x[\'estimated_duration_seconds\'] for x in narration_object[\'sections\'])\n        validate_narration_object(scene_json,narration_object)\n    else:path.write_text(expected,encoding=\'utf-8\')\n    if narration_json_path is not None:\n        jp=Path(narration_json_path);jp.parent.mkdir(parents=True,exist_ok=True);jp.write_text(json.dumps(narration_object,indent=2,ensure_ascii=False),encoding="utf-8")\n    return narration_object\n\n\n\n# Verbatim maths from 1000.day 3_update .ipynb, Drive ID 1zKWruOYPPstyyr42y6q-MKwE3-6qTh0s\ndef _vx_phase4_clip_fdr(value):\n    from vortex_upstream_fdr import _vx_phase4_clip_fdr as _canonical\n    return _canonical(value)\n\ndef vx_phase4_attack_fdr(lambda_for):\n    from vortex_upstream_fdr import vx_phase4_attack_fdr as _canonical\n    return _canonical(lambda_for)\n\ndef vx_phase4_defence_fdr(clean_sheet_probability):\n    from vortex_upstream_fdr import vx_phase4_defence_fdr as _canonical\n    return _canonical(clean_sheet_probability)\n\ndef vx_phase4_fdr_colour_band(fdr_value):\n    from vortex_upstream_fdr import vx_phase4_fdr_colour_band as _canonical\n    return _canonical(fdr_value)\n\ndef _vx_phase10_expected_floor_poisson(mu, divisor, tail=30):\n    mu = float(max(0.0, mu))\n    probs = [math.exp(-mu)]\n    for k in range(1, int(tail) + 1):\n        probs.append(probs[-1] * mu / k)\n    # Remaining tail is negligible for football-count means at chosen support.\n    return float(sum((k // int(divisor)) * probs[k] for k in range(len(probs))))\n\ndef _vx_phase10_prob_ge_poisson(mu, threshold, tail=40):\n    mu = float(max(0.0, mu))\n    p = math.exp(-mu)\n    cdf = p if threshold > 0 else 0.0\n    for k in range(1, int(threshold)):\n        p = p * mu / k\n        cdf += p\n    return float(np.clip(1.0 - cdf, 0.0, 1.0))\n'
_vx_module_dir=VX_MEDIA_WORK/'wildcard_engine'
_vx_module_dir.mkdir(parents=True,exist_ok=True)
_VX_FREEHIT_INTRO_SOURCE = '"""Independent single-Gameweek Free Hit evidence model and matching narration."""\nfrom __future__ import annotations\nimport math\nimport statistics\nfrom datetime import datetime, timezone\nfrom typing import Any, Mapping, Sequence\nfrom wildcard_intro import numeric, timestamp, clamp\n\nVERSION=\'free-hit-blueprint-single-gw-v2-day3-model\'\nWEIGHTS={\'scale\':.15,\'immediate_advantage\':.25,\'relevance\':.20,\'certainty\':.20,\'specificity\':.20}\nTHRESHOLD=58.\nSAMPLE_NARRATION=(\'Welcome to the Free Hit Blueprint for Gameweek [GW]. \'\n                  \'[VERIFIED ONE-WEEK REASON AND METRIC]. [SECOND VISIBLE REASON]. \'\n                  \'The original squad returns afterwards. Next, let’s rank this Gameweek’s fixtures.\')\n\ndef build_freehit_case(bootstrap: Mapping[str,Any], fixtures: Sequence[Mapping[str,Any]], *,\n                       projections: Sequence[Mapping[str,Any]] | None=None,\n                       refreshed_at: str | None=None, now: datetime | None=None,\n                       fdr_records: Mapping[tuple[int,int], Mapping[str,Any]] | None=None) -> dict[str,Any]:\n    """Pass core bootstrap/fixtures and model rows for NEXT_GW only; never pass horizon totals."""\n    if fdr_records is None:raise ValueError(\'Free Hit intro requires canonical DAY-3 fixture ratings\')\n    current=now or datetime.now(timezone.utc)\n    if current.tzinfo is None:raise ValueError(\'Free Hit analysis time requires a timezone\')\n    current=current.astimezone(timezone.utc)\n    events=sorted([event for event in bootstrap.get(\'events\',[]) if timestamp(event.get(\'deadline_time\')) and timestamp(event[\'deadline_time\'])>current],key=lambda event:timestamp(event[\'deadline_time\']))\n    next_gw=int(events[0][\'id\']) if events else None\n    refreshed=timestamp(refreshed_at);age=(current-refreshed).total_seconds()/3600 if refreshed else None\n    certainty=1. if age is not None and -.25<=age<=6 else .8 if age is not None and 6<age<=24 else .55 if age is None or 24<age<=72 else .2\n    health={\'analysis_gws\':[next_gw] if next_gw else [],\'future_gameweeks_used\':False,\'refresh_timestamp\':refreshed_at,\'age_hours\':age,\'weights\':dict(WEIGHTS),\'threshold\':THRESHOLD,\'warnings\':[],\'omitted_categories\':{},\'data_label\':\'Live FPL Analysis\' if certainty==1. else \'Latest FPL Analysis\'}\n    if certainty<.55:health[\'warnings\'].append(\'Stale or future-dated core snapshot: live triggers suppressed.\')\n    teams={int(team[\'id\']):team for team in bootstrap.get(\'teams\',[])}\n    players={int(player[\'id\']):player for player in bootstrap.get(\'elements\',[]) if player.get(\'can_select\',True) and not player.get(\'removed\')}\n    popular=[player for player in players.values() if (numeric(player.get(\'selected_by_percent\')) or 0)>=10 and (numeric(player.get(\'minutes\')) or 0)>0]\n    counts={team_id:0 for team_id in teams};ratings={team_id:[] for team_id in teams};candidates=[]\n    def add(kind,label,evidence,metric,metric_label,response,icon,scale,advantage,relevance,quality,summary,priority=1):\n        dimensions={\'scale\':clamp(scale),\'immediate_advantage\':clamp(advantage),\'relevance\':clamp(relevance),\'certainty\':clamp(certainty*quality),\'specificity\':1.}\n        score=round(100*sum(WEIGHTS[key]*dimensions[key] for key in WEIGHTS),1)\n        if len(label.split())>4 or len(evidence.split())>20 or len(response.split())>12:raise ValueError(\'Free Hit card copy exceeds word budget\')\n        if next_gw is None or score<THRESHOLD or dimensions[\'certainty\']<.55:\n            health[\'omitted_categories\'][kind]=\'Below evidence, freshness or impact threshold\';return\n        candidates.append({\'type\':kind,\'label\':label,\'impact_level\':\'High\' if score>=78 and dimensions[\'certainty\']>=.8 else \'Medium\',\'score\':score,\'evidence_text\':evidence,\'metric_value\':metric,\'metric_label\':metric_label,\'recommendation\':response,\'icon\':icon,\'source_data_summary\':summary,\'score_dimensions\':dimensions,\'priority\':priority,\'is_neutral\':False})\n    for fixture in fixtures:\n        if fixture.get(\'event\')!=next_gw or fixture.get(\'finished\'):continue\n        for side in (\'h\',\'a\'):\n            team_id=fixture.get(\'team_\'+side)\n            if team_id not in teams:continue\n            counts[team_id]+=1\n            model_record=fdr_records.get((int(fixture[\'id\']),int(team_id)))\n            if model_record is None:raise ValueError(f\'DAY-3 attack FDR missing for Free Hit intro fixture {fixture["id"]}, team {team_id}\')\n            rating=numeric(model_record.get(\'attack_fdr\'))\n            if rating is None or not 1<=rating<=5:raise ValueError(f\'Invalid DAY-3 attack FDR for Free Hit intro fixture {fixture["id"]}, team {team_id}\')\n            ratings[team_id].append(rating)\n    complete=bool(teams and next_gw) and sum(counts.values())>=len(teams)*.6\n    health[\'published_schedule_coverage\']=sum(count>0 for count in counts.values())/max(1,len(teams))\n    blank=[team_id for team_id,count in counts.items() if count==0]\n    doubles=[team_id for team_id,count in counts.items() if count>=2]\n    if complete and blank:\n        exposure=sum(numeric(player[\'selected_by_percent\']) or 0 for player in popular if player.get(\'team\') in blank)\n        add(\'blank\',\'Blank Gameweek\',f\'{len(blank)} clubs have no published fixture in Gameweek {next_gw}; schedules remain subject to change.\',str(len(blank)),\'Blank clubs\',\'Select active players; check the confirmed schedule before activation.\',\'calendar\',len(blank)/6,.95,exposure/60,1.,{\'source\':\'Official FPL event assignments\',\'gameweek\':next_gw,\'clubs\':[{\'id\':team_id,\'name\':teams[team_id][\'name\']} for team_id in blank]},3)\n    else:health[\'omitted_categories\'][\'blank\']=\'No published blank or insufficient schedule coverage\'\n    if complete and doubles:\n        add(\'double\',\'Double Gameweek\',f\'{len(doubles)} clubs have at least two published fixtures in Gameweek {next_gw}.\',str(len(doubles)),\'Double-fixture clubs\',\'Target double fixtures with credible starts and manageable rotation.\',\'calendar\',len(doubles)/4,1.,.85,1.,{\'source\':\'Official FPL event assignments\',\'gameweek\':next_gw,\'clubs\':[{\'id\':team_id,\'name\':teams[team_id][\'name\'],\'fixtures\':counts[team_id]} for team_id in doubles]},3)\n    else:health[\'omitted_categories\'][\'double\']=\'No published double or insufficient schedule coverage\'\n    available=[{\'team_id\':team_id,\'club\':teams[team_id][\'name\'],\'fdr\':statistics.mean(batch),\'fixtures\':counts[team_id]} for team_id,batch in ratings.items() if batch]\n    if len(available)>=max(4,len(teams)//2):\n        easy=[row for row in available if row[\'fdr\']<=2.5]\n        spread=max(row[\'fdr\'] for row in available)-min(row[\'fdr\'] for row in available)\n        if easy and spread>=2:\n            add(\'fixture_edge\',\'One-Week Fixture Edge\',f\'{len(easy)} clubs average FDR {statistics.mean(row["fdr"] for row in easy):.2f} in Gameweek {next_gw}; the difficulty spread is {spread:.1f}.\',f\'{spread:.1f}\',\'This-GW FDR spread\',\'Focus on favourable immediate matchups and secure player roles.\',\'calendar\',len(easy)/6,spread/3,.8,1.,{\'source\':\'DAY-3 GitHub attack FDR\',\'gameweek\':next_gw,\'favourable_clubs\':easy,\'fdr_spread\':spread})\n        else:health[\'omitted_categories\'][\'fixture_edge\']=\'No material immediate difficulty edge\'\n    else:health[\'omitted_categories\'][\'fixture_edge\']=\'Incomplete current-Gameweek difficulties\'\n    flagged=[player for player in popular if player.get(\'status\') in (\'i\',\'s\',\'u\') or (player.get(\'status\')==\'d\' and numeric(player.get(\'chance_of_playing_next_round\')) is not None and float(player[\'chance_of_playing_next_round\'])<=75)]\n    if flagged:\n        severity=statistics.mean(1. if player.get(\'status\') in (\'i\',\'s\',\'u\') else 1-float(player[\'chance_of_playing_next_round\'])/100 for player in flagged)\n        add(\'availability\',\'Availability Shock\',f\'{len(flagged)} popular assets carry official availability flags ahead of Gameweek {next_gw}.\',str(len(flagged)),\'Flagged popular assets\',\'Choose verified one-week replacements; reassess doubts near the deadline.\',\'shield\',len(flagged)/5,severity,statistics.mean(float(player[\'selected_by_percent\']) for player in flagged)/35,1.,{\'source\':\'Official FPL current status and next-round chance\',\'gameweek\':next_gw,\'players\':[{\'id\':player[\'id\'],\'name\':player[\'web_name\'],\'status\':player[\'status\'],\'chance\':player.get(\'chance_of_playing_next_round\'),\'news\':player.get(\'news\')} for player in flagged]})\n    else:health[\'omitted_categories\'][\'availability\']=\'No relevant official availability flags\'\n    scored=[]\n    for row in projections or []:\n        if row.get(\'gameweek\')!=next_gw:continue\n        points=numeric(row.get(\'gameweek_expected_points\'));minutes=numeric(row.get(\'expected_minutes\'));start=numeric(row.get(\'p_start\'));player_id=row.get(\'player_id\')\n        if points is not None and minutes is not None and start is not None and player_id in players:\n            scored.append({\'id\':player_id,\'name\':players[player_id][\'web_name\'],\'points\':points,\'minutes\':minutes,\'start\':start,\'ownership\':numeric(players[player_id].get(\'selected_by_percent\')) or 0})\n    eligible=sorted([row for row in scored if row[\'minutes\']>=65 and row[\'start\']>=.85],key=lambda row:-row[\'points\'])\n    if len(eligible)>=5:\n        best=eligible[0];comparison=statistics.median(row[\'points\'] for row in eligible[1:5]);gap=best[\'points\']-comparison\n        if best[\'points\']>=7 and gap>=1.5:\n            add(\'captaincy\',\'Captaincy Window\',f\'{best["name"]} projects {best["points"]:.1f} points, {gap:.1f} above the next four candidates’ median.\',f\'{best["points"]:.1f}\',\'Model xPts · before doubling\',\'Compare the leading captain route; verify minutes before locking.\',\'chart\',.5,gap/3,.95,.8,{\'source\':\'Existing next-Gameweek model projections; estimates\',\'gameweek\':next_gw,\'player\':best,\'comparison_median\':comparison})\n        else:health[\'omitted_categories\'][\'captaincy\']=\'No standout evidenced projection gap\'\n    else:health[\'omitted_categories\'][\'captaincy\']=\'Insufficient dependable single-GW projection candidates\'\n    risky=[row for row in scored if row[\'ownership\']>=10 and 0<row[\'start\']<.75 and row[\'minutes\']<65 and players[row[\'id\']].get(\'status\')==\'a\']\n    if risky:\n        add(\'rotation\',\'Rotation Avoidance\',f\'{len(risky)} popular, available assets have model start estimates below 75% for Gameweek {next_gw}.\',str(len(risky)),\'Model start-risk assets\',\'Prefer stronger start estimates; verify team news before selection.\',\'clock\',len(risky)/4,statistics.mean(1-row[\'start\'] for row in risky)*1.6,statistics.mean(row[\'ownership\'] for row in risky)/35,.8,{\'source\':\'Existing next-Gameweek start/minutes model; estimates\',\'gameweek\':next_gw,\'players\':risky})\n    else:health[\'omitted_categories\'][\'rotation\']=\'No relevant low-start model estimates\'\n    ranked=sorted(candidates,key=lambda card:(-card[\'priority\'],-card[\'score\'],card[\'type\']))[:4]\n    fallback=None\n    if len(ranked)<2:\n        fallback=\'Fewer than two qualifying immediate triggers; neutral planning is not an activation instruction.\'\n        ranked.append({\'type\':\'neutral_one_week\',\'label\':\'One-Week Optimisation\',\'impact_level\':\'Watchlist\',\'score\':0.,\'evidence_text\':\'A Free Hit allows a full squad selected purely for the strongest Gameweek fixtures and captaincy options.\',\'metric_value\':f\'GW{next_gw} ONLY\' if next_gw else \'AWAITING GW\',\'metric_label\':\'Temporary tactical squad\',\'recommendation\':\'Optimise every position for this Gameweek.\',\'icon\':\'layers\',\'source_data_summary\':{\'source\':\'Neutral strategy; not a verified activation trigger\',\'gameweek\':next_gw},\'score_dimensions\':{},\'is_neutral\':True})\n    headline=f\'Why a Free Hit Could Matter in GW{next_gw}\' if any(card[\'impact_level\']==\'High\' for card in ranked) else f\'Your GW{next_gw} Free Hit Opportunity\' if next_gw else \'Awaiting the Next Free Hit Window\'\n    return {\'chip\':\'freehit\',\'next_gw\':next_gw,\'analysis_gws\':[next_gw] if next_gw else [],\'reasons\':ranked,\'headline\':headline,\'subheadline\':f\'One-week tactical analysis for Gameweek {next_gw}\' if next_gw else \'No upcoming deadline is available.\',\'fallback_reason\':fallback,\'data_health\':health,\'sample_narration\':SAMPLE_NARRATION}\n\ndef generate_narration(case: Mapping[str,Any]) -> str:\n    gw=case[\'next_gw\']\n    if gw is None:return \'Welcome to the Free Hit Blueprint. No upcoming deadline is available. We will resume the one-week analysis when the next schedule is published.\'\n    parts=[f\'Welcome to the Free Hit Blueprint for Gameweek {gw}. A Free Hit is about maximising the upcoming Gameweek without compromising the longer-term squad.\']\n    for card in case[\'reasons\'][:3]:parts.append(f\'{card["label"]}. {card["evidence_text"]} {card["recommendation"]}\')\n    parts.append(\'These are tactical opportunities to assess, rather than a claim that the chip is necessary.\' if any(not card[\'is_neutral\'] for card in case[\'reasons\']) else \'No strong live trigger qualifies. Treat this as a planning exercise rather than a reason to rush the chip.\')\n    parts.append(\'Select a temporary starting eleven and bench for this Gameweek only. The original squad returns afterwards.\')\n    parts.append(f\'Next, let’s rank the fixture difficulty for Gameweek {gw}.\')\n    return \' \'.join(parts)\n'
_VX_SQUAD_PLAN_SOURCE = '"""Shared date/audio-clock utilities. Wildcard Slide 3 lives exclusively in wildcard_slide3."""\nimport math,copy\nfrom datetime import datetime,timezone\nfrom typing import Any,Mapping,Sequence\nfrom wildcard_slide3 import build_wildcard_squad,generate_wildcard_slide3_narration,weekly_plan_rows\n\ndef next_six_gameweeks(events: Sequence[Mapping[str,Any]], now: datetime | None=None) -> list[int]:\n    current=now or datetime.now(timezone.utc)\n    if current.tzinfo is None:raise ValueError(\'Analysis time requires a timezone\')\n    future=[]\n    for event in events:\n        try:deadline=datetime.fromisoformat(str(event[\'deadline_time\']).replace(\'Z\',\'+00:00\'))\n        except (ValueError,KeyError):continue\n        if deadline>current:future.append(int(event[\'id\']))\n    weeks=sorted(set(future))[:6]\n    if len(weeks)!=6 or weeks!=list(range(weeks[0],weeks[0]+6)):\n        raise ValueError(\'Six consecutive upcoming Gameweeks are not yet available; no fictional window generated\')\n    return weeks\n\n\ndef align_timeline(segments: Sequence[Mapping[str,Any]], duration: float,\n                   segment_durations: Sequence[float] | None=None) -> list[dict[str,Any]]:\n    if not math.isfinite(duration) or duration<=0:raise ValueError(\'Invalid audio duration\')\n    result=copy.deepcopy(list(segments));words=[len(segment[\'narration_text\'].split()) for segment in result]\n    durations=list(segment_durations) if segment_durations is not None else [duration*count/sum(words) for count in words]\n    if len(durations)!=len(result) or any(not math.isfinite(value) or value<=0 for value in durations) or abs(sum(durations)-duration)>.05:raise ValueError(\'Segment durations do not match audio\')\n    cursor=0.\n    for segment,seconds in zip(result,durations):\n        segment.update(start=cursor,end=cursor+seconds,alignment=\'measured_segments\' if segment_durations is not None else \'estimated_word_counts\',transition_overlap=.3)\n        cursor+=seconds\n    return result\n\n'
_VX_SQUAD_PLAN_VIEW_SOURCE = '"""Free Hit-only compatibility renderer. No Wildcard pitch or layout remains here."""\ndef render_html(plan,timeline=None,media=None,badges=None,views=None,logo=None):\n    if plan.get(\'chip\')!=\'freehit\':raise ValueError(\'Wildcard Slide 3 uses wildcard_slide3.render_html\')\n    from freehit_analyst import render_squad\n    return render_squad(plan,timeline,media,badges,views=views,logo=logo)\n'
(_vx_module_dir/'wildcard_slide3.py').write_text('"""Wildcard-only Slide 3: XI-first selection, one-goal pitch and conditional action plan."""\nfrom __future__ import annotations\nimport base64,copy,html,json,math\nfrom pathlib import Path\nfrom collections import Counter\nimport numpy as np\nfrom scipy.optimize import milp,Bounds,LinearConstraint\nfrom scipy.sparse import lil_matrix\nVERSION=\'wildcard-slide3-action-plan-v1\'\nVOICE={\'voice\':\'en-GB-RyanNeural\',\'rate\':\'+0%\',\'pitch\':\'+0Hz\',\'volume\':\'+0%\'}\nCOUNTS={\'GK\':2,\'DEF\':5,\'MID\':5,\'FWD\':3};POSITIONS={1:\'GK\',2:\'DEF\',3:\'MID\',4:\'FWD\'}\n\ndef num(x):\n    try:\n        v=float(x);return v if math.isfinite(v) else None\n    except (ValueError,TypeError):return None\n\ndef spoken(p):return p.get(\'spoken_name\') or p.get(\'full_name\') or p[\'name\']\ndef uri(path):\n    p=Path(path) if path else None\n    return \'data:image/png;base64,\'+base64.b64encode(p.read_bytes()).decode() if p and p.is_file() else \'\'\n\ndef prepare_pool(players,teams,context,weeks):\n    clubs={int(t[\'id\']):t for t in teams};models={int(m[\'player_id\']):m for m in context.get(\'models\',[])}\n    weekly={(int(w[\'player_id\']),int(w[\'gameweek\'])):w for w in context.get(\'weekly_models\',[])};out=[]\n    scale=float(context.get(\'currency_multiplier\',10))\n    for p in players:\n        if p.get(\'status\')!=\'a\' or p.get(\'removed\') or not p.get(\'can_select\',True):continue\n        if p.get(\'element_type\') not in POSITIONS or int(p.get(\'team\',-1)) not in clubs:continue\n        if num(p.get(\'chance_of_playing_next_round\')) not in (None,100):continue\n        cost=num(p.get(\'now_cost\'))\n        if cost is None or cost<=0 or cost!=int(cost):continue\n        pid=int(p[\'id\']);model=models.get(pid,{});pos=POSITIONS[p[\'element_type\']]\n        active=[weekly.get((pid,g),{}) for g in weeks if (num(weekly.get((pid,g),{}).get(\'fixture_count\')) or 0)>0]\n        start=num(model.get(\'p_start\'));minutes=num(model.get(\'expected_minutes\'))\n        if start is None and active:start=num(active[0].get(\'p_start\'))\n        if minutes is None and active:minutes=num(active[0].get(\'expected_minutes\'))/(num(active[0].get(\'fixture_count\')) or 1) if num(active[0].get(\'expected_minutes\')) is not None else None\n        # Do not invent future points or role estimates. Missing projection cannot rank an XI player.\n        projections=[num(weekly.get((pid,g),{}).get(\'gameweek_expected_points\')) for g in weeks]\n        total=sum(projections) if all(v is not None for v in projections) else num(model.get(\'xpts_wildcard_horizon\'))\n        secure=bool(start is not None and minutes is not None and start>=.8 and minutes>=65 and total is not None and not model.get(\'news_monitor\'))\n        cover=bool(start is not None and minutes is not None and start>=.75 and minutes>=60 and total is not None and not model.get(\'news_monitor\'))\n        nailed=bool(secure and pos==\'GK\')\n        viable=bool(start is not None and minutes is not None and start>=.35 and minutes>=25 and not model.get(\'news_monitor\'))\n        club=clubs[int(p[\'team\'])];first=str(p.get(\'first_name\',\'\')).split();last=str(p.get(\'second_name\',\'\'))\n        web=str(p[\'web_name\']);display=(first[0]+\' \'+web.split(\'.\',1)[1].strip()) if first and len(web.split(\'.\',1)[0])==1 and \'.\' in web else web\n        common=display if \' \' in display else ((first[0]+\' \'+display) if first else display)\n        out.append(dict(id=pid,name=display,spoken_name=common,full_name=(str(p.get(\'first_name\',\'\'))+\' \'+last).strip(),position=pos,team_id=int(p[\'team\']),club=club[\'short_name\'],club_name=club[\'name\'],photo_code=p.get(\'code\'),price=cost/scale,cost_units=int(cost),secure=secure,cover=cover,nailed=nailed,viable=viable,minutes=minutes,starting_probability=start,projected_points=total,first_projection=projections[0],score=total or 0,availability=\'AVAILABLE\',reason=\'Secure minutes and six-week model projection\' if secure else \'Low-cost reserve; playing time not confirmed\',weekly_points={g:projections[i] for i,g in enumerate(weeks)},analysis={\'role\':\'First-choice penalty taker\' if p.get(\'penalties_order\')==1 else \'Role and minutes subject to team news\',\'xgi90\':num(p.get(\'expected_goal_involvements_per_90\')),\'xg90\':num(p.get(\'expected_goals_per_90\')),\'xa90\':num(p.get(\'expected_assists_per_90\'))}))\n    return out\n\ndef build_wildcard_squad(players,teams,fixtures,next_gw,horizon=6,budget=None,max_bank=.3,bench_boost_mode=False,context=None):\n    """Fresh mixed-integer XI/bench selector. The backup GK is cheapest; bench = GK, DEF, DEF, MID/FWD."""\n    context=dict(context or {});budget=num(budget)\n    weeks=list(range(int(next_gw),int(next_gw)+6))\n    if horizon!=6 or bench_boost_mode or budget is None or budget<=0 or max_bank>.3:raise ValueError(\'Requires six weeks, a verified budget and no immediate Bench Boost\')\n    if context.get(\'available_gameweeks\',weeks)!=weeks:raise ValueError(\'Six consecutive projection Gameweeks required\')\n    pool=prepare_pool(players,teams,context,weeks);n=len(pool);size=3*n\n    if len({p[\'id\'] for p in pool})!=n:raise ValueError(\'Duplicate player identity\')\n    keeper_prices=[p[\'cost_units\'] for p in pool if p[\'position\']==\'GK\']\n    verified_keepers=[p for p in pool if p[\'position\']==\'GK\' and p[\'nailed\']]\n    if not verified_keepers:\n        best=max((p for p in pool if p[\'position\']==\'GK\'),key=lambda p:(p.get(\'starting_probability\') or 0,p.get(\'minutes\') or 0),default=None)\n        summary=\'none available\' if best is None else f"{best[\'name\']} p_start={best.get(\'starting_probability\')!r} expected_minutes={best.get(\'minutes\')!r}"\n        raise ValueError(f\'verified starting goalkeepers=0; Best goalkeeper evidence: {summary}\')\n    defender_prices=[p[\'cost_units\'] for p in pool if p[\'position\']==\'DEF\' and p[\'cover\']]\n    if len(keeper_prices)<2 or len(defender_prices)<5:raise ValueError(\'Insufficient verified goalkeeper/defender coverage\')\n    cheapest_gk=min(keeper_prices);cheap_def=min(defender_prices)+int(round(.5*context.get(\'currency_multiplier\',10)))\n    cheap_attack={pos:min((p[\'cost_units\'] for p in pool if p[\'position\']==pos and p[\'viable\']),default=-100)+int(round(.4*context.get(\'currency_multiplier\',10))) for pos in (\'MID\',\'FWD\')}\n    # y = squad, x = immediate XI, c = captain. A reserve is y-x.\n    rows=[];lo=[];hi=[]\n    def add(items,a,b):rows.append(items);lo.append(a);hi.append(b)\n    for pos,count in COUNTS.items():add({i:1 for i,p in enumerate(pool) if p[\'position\']==pos},count,count)\n    for team in {p[\'team_id\'] for p in pool}:add({i:1 for i,p in enumerate(pool) if p[\'team_id\']==team},0,3)\n    scale=float(context.get(\'currency_multiplier\',10));limit=int(math.floor(budget*scale+1e-6))\n    add({i:p[\'cost_units\'] for i,p in enumerate(pool)},0,limit)\n    add({n+i:1 for i in range(n)},11,11);add({2*n+i:1 for i in range(n)},1,1)\n    for pos,a,b in [(\'GK\',1,1),(\'DEF\',3,3),(\'MID\',4,5),(\'FWD\',2,3)]:add({n+i:1 for i,p in enumerate(pool) if p[\'position\']==pos},a,b)\n    upper=np.ones(size)\n    for i,p in enumerate(pool):\n        add({n+i:1,i:-1},-math.inf,0);add({2*n+i:1,n+i:-1},-math.inf,0)\n        if not p[\'secure\'] or (p[\'position\']==\'GK\' and not p[\'nailed\']):upper[n+i]=0\n        if not p[\'secure\'] or p[\'position\'] not in (\'MID\',\'FWD\'):upper[2*n+i]=0\n        reserve_ok=(p[\'position\']==\'GK\' and p[\'cost_units\']==cheapest_gk) or (p[\'position\']==\'DEF\' and p[\'cover\'] and p[\'cost_units\']<=cheap_def) or (p[\'position\'] in (\'MID\',\'FWD\') and p[\'viable\'] and p[\'cost_units\']<=cheap_attack[p[\'position\']])\n        if not reserve_ok:add({i:1,n+i:-1},0,0)\n    objective=np.zeros(size)\n    for i,p in enumerate(pool):\n        objective[n+i]=p[\'projected_points\'] or 0\n        # Only the incremental armband score; the XI points already appear above.\n        objective[2*n+i]=p[\'projected_points\'] or 0\n    def solve(c):\n        mat=lil_matrix((len(rows),size))\n        for row,items in enumerate(rows):\n            for col,v in items.items():mat[row,col]=v\n        return milp(c,integrality=np.ones(size),bounds=Bounds(np.zeros(size),upper),constraints=LinearConstraint(mat.tocsr(),lo,hi),options={\'time_limit\':float(context.get(\'solver_seconds\',45)),\'mip_rel_gap\':.0005})\n    # Spending within max_bank is a preference, never grounds to reject a legal cheaper squad.\n    add({i:p[\'cost_units\'] for i,p in enumerate(pool)},max(0,int(math.ceil((budget-max_bank)*scale-1e-6))),math.inf)\n    first=solve(-objective)\n    bank_preference_met=first.x is not None\n    if not bank_preference_met:\n        rows.pop();lo.pop();hi.pop()\n        first=solve(-objective)\n    if first.x is None:raise ValueError(\'No legal XI-first draft meets verified role confidence, budget and formation requirements. No weaker squad substituted.\')\n    optimum=float(objective@first.x);add({i:v for i,v in enumerate(objective) if v},optimum-.001,math.inf)\n    # Among equally strong starting elevens, prefer less spending on reserves.\n    tie=np.zeros(size)\n    for i,p in enumerate(pool):tie[i]=p[\'cost_units\'];tie[n+i]=-p[\'cost_units\'];tie[i]-=.00001*(p[\'minutes\'] or 0)\n    second=solve(tie);sol=second if second.x is not None else first\n    squad=[copy.deepcopy(p) for i,p in enumerate(pool) if sol.x[i]>.5];xi=[p for p in squad if sol.x[n+next(i for i,q in enumerate(pool) if q[\'id\']==p[\'id\'])]>.5]\n    xi.sort(key=lambda p:(list(COUNTS).index(p[\'position\']),-(p[\'projected_points\'] or 0),p[\'id\']))\n    bench=sorted([p for p in squad if p[\'id\'] not in {x[\'id\'] for x in xi}],key=lambda p:(0 if p[\'position\']==\'GK\' else 1 if p[\'position\']==\'DEF\' else 2,-(p[\'minutes\'] or 0),p[\'id\']))\n    plan=dict(chip=\'wildcard\',version=VERSION,squad=squad,starting_xi=xi,bench=bench,formation=formation(xi),total_cost=sum(p[\'price\'] for p in squad),bank=round(budget-sum(p[\'price\'] for p in squad),1),budget=budget,next_gw=weeks[0],end_gw=weeks[-1],analysis_gws=weeks,selection_summary={\'default_bench_boost\':False,\'secure_starters\':11,\'bench_policy\':\'Cheapest backup goalkeeper; two low-cost dependable-cover defenders; viable cheap attacker\',\'selection_policy\':\'bank-preference-met\' if bank_preference_met else \'bank-preference-relaxed\'},data_health={\'valid\':True,\'solver_status\':int(first.status),\'optimality_proven\':first.status==0,\'solver_gap\':num(getattr(first,\'mip_gap\',None)),\'warnings\':[]})\n    plan[\'gameweek_plans\']=weekly_plans(plan,context.get(\'weekly_models\',[]));opening=plan[\'gameweek_plans\'][0];byid={p[\'id\']:p for p in squad};plan[\'captain\']=byid[opening[\'captain_id\']];plan[\'vice_captain\']=byid[opening[\'vice_captain_id\']]\n    plan[\'projected_points_total\']=opening[\'xi_xpts\'];plan[\'captain_adjusted_xpts\']=opening[\'captain_adjusted_xpts\'];plan[\'player_reasons\']={str(p[\'id\']):p[\'reason\'] for p in squad}\n    plan[\'bench_boost_path\']=build_bench_boost_path(plan,pool,fixtures,context.get(\'bench_boost\',\'Unknown\'));plan[\'bench_boost_upgrade_target\']=next(iter(plan[\'bench_boost_path\'][\'targets\']),None)\n    validate(plan,cheapest_gk=cheapest_gk,scale=scale);return plan\n\ndef formation(xi):\n    c=Counter(p[\'position\'] for p in xi);return \'-\'.join(str(c[p]) for p in (\'DEF\',\'MID\',\'FWD\'))\n\ndef weekly_plans(plan,weekly_models):\n    from itertools import combinations\n    lookup={(int(r[\'player_id\']),int(r[\'gameweek\'])):r for r in weekly_models}\n    squad=list(plan[\'squad\']);initial_ids={int(p[\'id\']) for p in plan[\'starting_xi\']};rows=[]\n    if len(squad)!=15 or len({int(p[\'id\']) for p in squad})!=15:raise ValueError(\'Wildcard six-week squad must contain 15 unique FPL player IDs\')\n    for index,gameweek in enumerate(plan[\'analysis_gws\']):\n        def predicted(p):return num(lookup.get((int(p[\'id\']),gameweek),{}).get(\'gameweek_expected_points\'))\n        if index==0:\n            lineup=list(plan[\'starting_xi\'])\n        else:\n            # The squad is fixed across all six Gameweeks; only the legal XI and captain rotate.\n            opening_gk_id=next(p[\'id\'] for p in plan[\'starting_xi\'] if p[\'position\']==\'GK\')\n            eligible=[p for p in squad if predicted(p) is not None and p.get(\'cover\') and (p[\'position\']!=\'GK\' or p[\'id\']==opening_gk_id)]\n            by_pos={position:[p for p in eligible if p[\'position\']==position] for position in COUNTS}\n            best=None\n            for defenders in range(3,6):\n                for mids in range(2,6):\n                    forwards=10-defenders-mids\n                    if forwards<1 or forwards>3:continue\n                    if any(len(by_pos[pos])<needed for pos,needed in ((\'GK\',1),(\'DEF\',defenders),(\'MID\',mids),(\'FWD\',forwards))):continue\n                    for goalkeepers in combinations(by_pos[\'GK\'],1):\n                        for ds in combinations(by_pos[\'DEF\'],defenders):\n                            for ms in combinations(by_pos[\'MID\'],mids):\n                                for fs in combinations(by_pos[\'FWD\'],forwards):\n                                    choice=(*goalkeepers,*ds,*ms,*fs)\n                                    attacking=[p for p in choice if p[\'position\'] in (\'MID\',\'FWD\') and p.get(\'secure\')]\n                                    if len(attacking)<2:continue\n                                    captain=max(attacking,key=lambda p:(predicted(p),p.get(\'minutes\') or 0,-p[\'id\']))\n                                    value=sum(predicted(p) for p in choice)+predicted(captain)\n                                    key=(value,sum(p.get(\'minutes\') or 0 for p in choice),sum(p[\'id\'] in initial_ids for p in choice))\n                                    if best is None or key>best[0]:best=(key,choice)\n            if best is None:raise ValueError(f\'No legal, projected Wildcard XI for GW{gameweek} from the fixed squad\')\n            lineup=list(best[1])\n        lineup.sort(key=lambda p:(list(COUNTS).index(p[\'position\']),-(predicted(p) or 0),p[\'id\']))\n        starters=[dict(p,xpts=predicted(p)) for p in lineup]\n        if len(starters)!=11 or any(p[\'xpts\'] is None for p in starters):raise ValueError(\'Weekly XI projection unavailable\')\n        attackers=sorted((p for p in starters if p[\'position\'] in (\'MID\',\'FWD\') and p.get(\'secure\')),key=lambda p:(-p[\'xpts\'],-(p.get(\'minutes\') or 0),p[\'id\']))\n        if len(attackers)<2:raise ValueError(\'Insufficient secure captaincy choices\')\n        captain,vice=attackers[:2]\n        total=sum(p[\'xpts\'] for p in starters)\n        rows.append(dict(gameweek=gameweek,starting_xi=starters,captain_id=captain[\'id\'],vice_captain_id=vice[\'id\'],formation=formation(starters),xi_xpts=total,captain_adjusted_xpts=total+captain[\'xpts\'],status=\'Six-week fixed squad; Gameweek-optimized legal XI\',all_players=[{\'id\':p[\'id\'],\'xpts\':predicted(p)} for p in squad]))\n    return rows\n\ndef validate(plan,cheapest_gk=None,scale=10):\n    squad=plan[\'squad\'];xi=plan[\'starting_xi\'];bench=plan[\'bench\'];ids={p[\'id\'] for p in squad}\n    if len(squad)!=15 or len(ids)!=15 or Counter(p[\'position\'] for p in squad)!=Counter(COUNTS):raise ValueError(\'Illegal fifteen-player squad\')\n    if len(xi)!=11 or len(bench)!=4 or {p[\'id\'] for p in xi}&{p[\'id\'] for p in bench}:raise ValueError(\'Illegal XI/bench partition\')\n    if max(Counter(p[\'team_id\'] for p in squad).values())>3:raise ValueError(\'Club limit exceeded\')\n    if Counter(p[\'position\'] for p in bench)[\'DEF\']!=2 or bench[0][\'position\']!=\'GK\':raise ValueError(\'Requires GK, two defenders and one cheap attacker on bench\')\n    if not all(p[\'secure\'] for p in xi) or not next(p for p in xi if p[\'position\']==\'GK\')[\'nailed\']:raise ValueError(\'Starting role confidence insufficient\')\n    if not all(p[\'cover\'] for p in bench if p[\'position\']==\'DEF\'):raise ValueError(\'Bench defenders lack dependable cover\')\n    if cheapest_gk is not None and round(bench[0][\'price\']*scale)!=cheapest_gk:raise ValueError(\'Backup goalkeeper is not cheapest\')\n    if plan[\'bank\'] < -1e-6 or abs(sum(p[\'price\'] for p in squad)+plan[\'bank\']-plan[\'budget\'])>1e-5:raise ValueError(\'Bank or budget mismatch\')\n    if len(plan[\'gameweek_plans\'])!=6 or len({next(p[\'id\'] for p in w[\'starting_xi\'] if p[\'position\']==\'GK\') for w in plan[\'gameweek_plans\']})!=1:raise ValueError(\'Goalkeeper rotation prohibited\')\n    if any(w[\'captain_id\']==w[\'vice_captain_id\'] or not {w[\'captain_id\'],w[\'vice_captain_id\']}<={p[\'id\'] for p in w[\'starting_xi\']} for w in plan[\'gameweek_plans\']):raise ValueError(\'Captain pair invalid\')\n\n\ndef build_bench_boost_path(plan,pool,fixtures,status=\'Unknown\'):\n    result=dict(status=status,conditional=status!=\'Available\',candidate_gameweek=None,window_text=\'NO CONFIRMED BENCH BOOST WINDOW YET\',targets=[],steps=[\'UPGRADE BACKUP GK ONLY IF REQUIRED\',\'PRIORITISE TWO PLAYABLE OUTFIELD BENCH SLOTS\',\'USE FREE TRANSFERS WHERE POSSIBLE\',\'PLAY ONLY WHEN STARTERS AND FIXTURES ALIGN\'])\n    if status==\'Used\':result[\'window_text\']=\'BENCH BOOST ALREADY USED\';return result\n    selected={p[\'id\'] for p in plan[\'squad\']};clubs=Counter(p[\'team_id\'] for p in plan[\'squad\']);used=set()\n    for reserve in sorted(plan[\'bench\'],key=lambda p:(p[\'position\']!=\'GK\',p[\'minutes\'] or 0)):\n        options=[]\n        for p in pool:\n            if p[\'id\'] in selected or p[\'id\'] in used or p[\'position\']!=reserve[\'position\'] or not p[\'secure\']:continue\n            if clubs[p[\'team_id\']]-int(p[\'team_id\']==reserve[\'team_id\'])>=3:continue\n            future=[f for f in fixtures if f.get(\'event\') in plan[\'analysis_gws\'][1:] and p[\'team_id\'] in (f.get(\'team_h\'),f.get(\'team_a\')) and not f.get(\'finished\')]\n            easy=[f for f in future if (num(f.get(\'team_h_difficulty\' if f[\'team_h\']==p[\'team_id\'] else \'team_a_difficulty\')) or 6)<=2.5]\n            gain=(p[\'projected_points\'] or 0)-(reserve[\'projected_points\'] or 0);extra=round(p[\'price\']-reserve[\'price\'],1)\n            if easy and gain>0 and 0<extra<=1:options.append((gain/(extra+.1),dict(p,replaces_player_id=reserve[\'id\'],extra_cost=extra,future_option=True,upgrade_reason=f\'{len(easy)} favourable future fixtures; regular-minute estimate\',funding_note=f\'Needs £{extra:.1f}m extra; funding not assumed\')))\n        if options:\n            target=max(options,key=lambda x:(x[0],-x[1][\'id\']))[1];result[\'targets\'].append(target);used.add(target[\'id\'])\n        if len(result[\'targets\'])==2:break\n    # A future double alone is insufficient: all fifteen need credible playing time.\n    if all(p[\'secure\'] for p in plan[\'squad\']):\n        for g in plan[\'analysis_gws\'][1:]:\n            counts={p[\'id\']:sum(f.get(\'event\')==g and p[\'team_id\'] in (f.get(\'team_h\'),f.get(\'team_a\')) and not f.get(\'finished\') for f in fixtures) for p in plan[\'squad\']}\n            if all(counts.values()) and sum(v>1 for v in counts.values())>=2:\n                result[\'candidate_gameweek\']=g;result[\'window_text\']=f\'GW{g} CANDIDATE · RECHECK ALL FIFTEEN\';break\n    return result\n\ndef build_transfer_rows(plan,free_transfers=None):\n    # Conditional monitoring is not a committed transfer. No future injury is invented.\n    bank=plan[\'bank\'];rows=[];ft=free_transfers if isinstance(free_transfers,int) and free_transfers>=0 else 1\n    for g in plan[\'analysis_gws\'][1:]:\n        rows.append(dict(gameweek=g,action=\'ROLL\',transfers_required=0,free_transfers_before=ft,free_transfers_after=min(5,ft+1),money_before=bank,money_after=bank,hit_cost=0,xi_effect_xpts=0,lookahead_effect_xpts=0,reason=\'Roll if no injury, suspension or major fixture swing\',trigger=\'REASSESS IF INJURY, BAN OR MINUTES FALL\',squad_after=[p[\'id\'] for p in plan[\'squad\']],price_basis=\'current snapshot\',free_transfer_source=\'verified manager count\' if free_transfers is not None else \'one conservative reference transfer; actual count unknown\'))\n        ft=min(5,ft+1)\n    return rows\n\ndef enrich_plan(plan,contract,bootstrap=None,models=(),weekly_models=()):\n    p=copy.deepcopy(plan);canonical={x[\'id\']:x for x in contract[\'squad\']};teams={x[\'id\']:x[\'name\'] for x in (bootstrap or {}).get(\'teams\',[])};rows=[]\n    for w in p[\'gameweek_plans\'][1:]:\n        cap=next(x for x in p[\'squad\'] if x[\'id\']==w[\'captain_id\']);card=canonical.get(cap[\'id\'],{});cw=next((x for x in card.get(\'weeks\',[]) if x[\'gameweek\']==w[\'gameweek\']),{});fs=cw.get(\'fixtures\',[])\n        readable=\' and \'.join((teams.get(f.get(\'opponent_id\')) or str(f.get(\'opponent\',\'opponent\')))+(\' at home\' if f.get(\'home\') else \' away\') for f in fs)\n        short=\' / \'.join(str(f.get(\'opponent\',\'?\'))+(\'[H]\' if f.get(\'home\') else \'[A]\') for f in fs)\n        reason=\'Highest projected attacking option in this XI\'\n        if fs and all(f.get(\'home\') and num(f.get(\'fdr\')) is not None and float(f[\'fdr\'])<3 for f in fs):reason=\'Favourable home fixture; leads this XI projection\'\n        move=next((x for x in contract.get(\'transfer_plan\',[]) if x[\'gameweek\']==w[\'gameweek\']),{})\n        rows.append(dict(gameweek=w[\'gameweek\'],captain=cap,reason=reason,fixture=short,spoken_fixture=readable,transfer_action=\'ROLL TRANSFER\',transfer_narration=\'Roll the transfer if there is no injury, suspension or major fixture swing.\',trigger=move.get(\'trigger\',\'REASSESS IF INJURY, BAN OR MINUTES FALL\'),conditional=True))\n    p[\'action_plan\']=rows;p[\'transfer_options\']=copy.deepcopy(contract.get(\'transfer_plan\',[]));p[\'bench_boost_path\'][\'status\']=contract.get(\'bench_boost\',\'Unknown\');p[\'bench_boost_path\'][\'conditional\']=contract.get(\'bench_boost\')!=\'Available\'\n    p[\'narration_timeline\']=generate_wildcard_slide3_narration(p,p[\'action_plan\'],p[\'bench_boost_path\']);return p\n\n\ndef generate_wildcard_slide3_narration(wildcard_plan,transfer_plan=None,bench_boost_path=None):\n    p=wildcard_plan;actions=transfer_plan if transfer_plan is not None else p.get(\'action_plan\',[]);bb=bench_boost_path or p.get(\'bench_boost_path\',{});out=[]\n    def add(key,kind,text,ids=(),**kw):out.append(dict(segment_id=key,type=kind,narration_text=text,player_ids=list(ids),start=None,end=None,**kw))\n    add(\'intro\',\'title\',f"Let\'s look at the Wildcard plan from Gameweek {p[\'next_gw\']} to Gameweek {p[\'end_gw\']}. This is a no-immediate-Bench-Boost draft. The budget is concentrated in the starting eleven, with one reliable starting goalkeeper and a low-cost bench for cover. The opening formation is {p[\'formation\']}.")\n    captain=p[\'captain\'];order=sorted(p[\'starting_xi\'],key=lambda x:(0 if x[\'id\']==captain[\'id\'] else 1 if x[\'position\'] in (\'MID\',\'FWD\') else 2 if x[\'position\']==\'DEF\' else 3,-(x[\'projected_points\'] or 0),x[\'id\']))\n    for x in order:\n        reason=\'Opening captain; secure-minute estimate\' if x[\'id\']==captain[\'id\'] else \'Core attacking investment\' if x[\'position\'] in (\'MID\',\'FWD\') else \'Starting defender; budget balanced with cover\' if x[\'position\']==\'DEF\' else \'The fixed starting goalkeeper; no rotation budget\'\n        add(\'player_\'+str(x[\'id\']),\'players\',f"{spoken(x)} costs £{x[\'price\']:.1f} million. {reason}. His minutes estimate is {x[\'minutes\']:.0f} per appearance. Recheck team news before confirming.",[x[\'id\']],player_id=x[\'id\'],hero_reason=reason)\n    add(\'bench_intro\',\'phase\',\'The bench is deliberately lean. It provides low-cost cover rather than an immediate Bench Boost squad.\')\n    for x in p[\'bench\']:\n        reason=\'Cheapest valid backup goalkeeper. Playing time is optional.\' if x[\'position\']==\'GK\' else \'Affordable regular-minute estimate for defensive cover.\' if x[\'position\']==\'DEF\' else \'Low-cost outfield cover with a realistic minutes estimate.\'\n        add(\'bench_\'+str(x[\'id\']),\'bench\',f"{spoken(x)} costs £{x[\'price\']:.1f} million. {reason}",[x[\'id\']],player_id=x[\'id\'])\n    for a in actions:\n        add(\'action_\'+str(a[\'gameweek\']),\'gameweek_plan\',f"For Gameweek {a[\'gameweek\']}, captain {spoken(a[\'captain\'])}. "+(f"The fixture is {a[\'spoken_fixture\']}. " if a.get(\'spoken_fixture\') else \'\')+a[\'reason\'].replace(\'XI\',\'starting eleven\')+\'. \'+a[\'transfer_narration\']+\' Reassess if an injury, suspension or minutes concern changes the plan.\',[a[\'captain\'][\'id\']],gameweek=a[\'gameweek\'])\n    if bb.get(\'status\')==\'Used\':add(\'bb_intro\',\'bench_boost\',\'Bench Boost has already been used, so keep this low-cost cover structure.\')\n    else:\n        add(\'bb_intro\',\'bench_boost\',\'If Bench Boost is still available, do not force it now. Keep the strong starting eleven and upgrade bench depth gradually using free transfers. \'+(f"Gameweek {bb[\'candidate_gameweek\']} is a provisional candidate. Recheck every player\'s minutes and fixtures." if bb.get(\'candidate_gameweek\') else \'There is no confirmed Bench Boost window yet. Reassess when fixtures and team news provide a clearer opportunity.\'))\n        for i,x in enumerate(bb.get(\'targets\',[])):\n            add(\'bb_target_\'+str(x[\'id\']),\'future_target\',f"A future bench upgrade to monitor is {spoken(x)}, at £{x[\'price\']:.1f} million. {x[\'upgrade_reason\']}. This option needs £{x[\'extra_cost\']:.1f} million in additional funding. He is not part of the current fifteen.",[x[\'id\']],future_player_id=x[\'id\'],target_index=i)\n        add(\'bb_sequence\',\'bench_boost\',\'Upgrade the cheap non-playing goalkeeper only if required. Prioritise two playable outfield bench slots. Use free transfers rather than hits where possible. Play Bench Boost only when likely starters and fixtures align.\')\n    add(\'summary\',\'summary\',(\'The final draft uses the full budget.\' if p[\'bank\']==0 else f"The final draft leaves £{p[\'bank\']:.1f} million in the bank.")+ \' The focus stays on the strongest available starting eleven. Across the remaining Gameweeks, use transfers only when injuries, minutes risk or a genuine fixture swing creates a clear upgrade. Next, compare the captaincy options.\')\n    return out\n\ndef render_html(plan,timeline,media=None,badges=None,**unused):\n    esc=lambda x:html.escape(str(x),quote=True);media=media or {};badges=badges or {};p=plan\n    def picture(x,cls):\n        m=media.get(x[\'id\']) or media.get(str(x[\'id\'])) or {};src=uri(m.get(\'photo\')) if m.get(\'photo_kind\')==\'player\' else \'\'\n        if not src:\n            b=badges.get(x[\'club\']) or {};src=uri(b.get(\'path\'))\n        return f\'<img class="{cls}" src="{src}" alt="{esc(x["name"])}">\' if src else f\'<span class="{cls} s3-initials">{esc("".join(w[0] for w in x["name"].split())[:2])}</span>\'\n    nodes=[]\n    for pos,y in [(\'GK\',14.5),(\'DEF\',38.5),(\'MID\',62.5),(\'FWD\',86.5)]:\n        line=[x for x in p[\'starting_xi\'] if x[\'position\']==pos]\n        for i,x in enumerate(line):\n            arm=\'C\' if x[\'id\']==p[\'captain\'][\'id\'] else \'VC\' if x[\'id\']==p[\'vice_captain\'][\'id\'] else \'\'\n            nodes.append(f\'<div class="s3-pitch-player" data-s3-player="{x["id"]}" style="left:{(i+.5)*92/len(line)+4:.4f}%;top:{y}%">{picture(x,"s3-pitch-photo")}<b>{esc(x["name"])} <em>{arm}</em></b><span>{pos} · £{x["price"]:.1f}M</span></div>\')\n    bench=\'\'.join(f\'<article data-s3-bench="{x["id"]}">{picture(x,"s3-bench-photo")}<b>{esc(x["name"])}</b><span>{x["position"]} · £{x["price"]:.1f}M</span><small>{"COVER" if x["position"]=="DEF" else "BACKUP GK" if x["position"]=="GK" else "LOW-COST COVER"}</small></article>\' for x in p[\'bench\'])\n    action_cards=[]\n    for a in p[\'action_plan\']:\n        x=a[\'captain\'];action_cards.append(f\'<article class="s3-action" data-s3-gw="{a["gameweek"]}"><header><b>GW{a["gameweek"]}</b><span>{esc(a["fixture"])}</span><i class="s3-roll">↻</i></header><div class="s3-action-body">{picture(x,"s3-captain-photo")}<div><h3>CAPTAIN: {esc(x["name"])}</h3><p><strong>WHY:</strong> {esc(a["reason"])}</p><p><strong>TRANSFER:</strong> {esc(a["transfer_action"])}</p><small><strong>TRIGGER:</strong> {esc(a["trigger"])}</small></div></div></article>\')\n    heroes=\'\'.join(f\'<article class="s3-hero" data-s3-hero="{x["id"]}">{picture(x,"s3-hero-photo")}<div><h3>{esc(x["name"])}</h3><p>{x["position"]} · £{x["price"]:.1f}M · {x["minutes"]:.0f} EXPECTED MINS</p><small>{"OPENING CAPTAIN" if x["id"]==p["captain"]["id"] else "FIXED STARTING GOALKEEPER" if x["position"]=="GK" else "CORE STARTER · RECHECK TEAM NEWS"}</small></div></article>\' for x in p[\'starting_xi\'])\n    bb=p[\'bench_boost_path\'];targets=\'\'.join(f\'<article class="s3-future" data-s3-future="{x["id"]}">{picture(x,"s3-future-photo")}<div><small>FUTURE BENCH UPGRADE · NOT IN THIS 15</small><b>{esc(x["name"])} · {x["position"]} · £{x["price"]:.1f}M</b><p>{esc(x["upgrade_reason"])}</p><p>{esc(x["funding_note"])}</p></div></article>\' for x in bb.get(\'targets\',[]))\n    if bb[\'status\']==\'Used\':bbbody=\'<p>CHIP ALREADY USED · KEEP LOW-COST COVER</p>\'\n    else:bbbody=f\'<p>KEEP THE STRONG XI. BUILD BENCH DEPTH LATER.</p><b class="s3-bb-window">{esc(bb["window_text"])}</b><div class="s3-future-list">{targets or "<p>NO VERIFIED UPGRADE TARGETS YET</p>"}</div><ol>\'+\'\'.join(\'<li>\'+esc(step)+\'</li>\' for step in bb[\'steps\'])+\'</ol>\'\n    data=json.dumps(dict(timeline=timeline,plan=p),ensure_ascii=False).replace(\'<\',\'\\\\u003c\')\n    return f\'\'\'<style>{CSS}</style><section class="s3-action-layout"><main class="s3-left"><header class="s3-pitch-title"><div><h2>WILDCARD STARTING XI</h2><p>GW{p[\'next_gw\']}–GW{p[\'end_gw\']} | BUILT FOR SECURE MINUTES AND SUSTAINED FIXTURE VALUE</p></div><b>{p[\'formation\']}<small>£{p[\'bank\']:.1f}M BANK</small></b></header><div class="s3-pitch"><div class="s3-boundary"></div><div class="s3-halfway"></div><div class="s3-centre"></div><div class="s3-top-goal" data-goal="top"></div><div class="s3-top-penalty"></div><div class="s3-top-six"></div>{\'\'.join(nodes)}</div><section class="s3-bench"><h3>BENCH — LOW-COST COVER</h3><div>{bench}</div></section></main><aside class="s3-right"><header><h2>GW{p[\'next_gw\']+1}–GW{p[\'end_gw\']} WILDCARD PLAN</h2><p>TRANSFERS ONLY WHEN NEWS, MINUTES OR FIXTURES JUSTIFY A CHANGE</p></header><div class="s3-hero-zone">{heroes}<div class="s3-hero-default"><b>NO IMMEDIATE BENCH BOOST</b><p>£{p[\'total_cost\']:.1f}M SQUAD · £{p[\'bank\']:.1f}M BANK</p><small>ONE STARTING GOALKEEPER · TWO BENCH DEFENDERS</small></div></div><section class="s3-actions">{\'\'.join(action_cards)}<footer class="s3-page-summary"></footer></section><section class="s3-bb"><header><h2>BENCH BOOST PATHWAY</h2><span>{\'IF STILL AVAILABLE · STATUS UNVERIFIED\' if bb.get(\'conditional\') and bb[\'status\']!=\'Used\' else bb[\'status\']}</span></header>{bbbody}</section></aside><script type="application/json" class="s3-data">{data}</script></section>\'\'\'\n\nCSS=\'\'\'\n#content .s3-action-layout{height:100%;width:100%;display:grid;grid-template-columns:minmax(0,64fr) minmax(0,36fr);gap:26px;font-family:Arial,sans-serif;color:#f4f8ff;text-transform:uppercase;overflow:hidden}\n.s3-action-layout *{box-sizing:border-box}.s3-action-layout h2,.s3-action-layout h3,.s3-action-layout p{margin:0}.s3-left{min-width:0;display:grid;grid-template-rows:125px minmax(0,1fr) 285px;gap:18px;min-height:0}\n.s3-pitch-title{display:flex;align-items:center;justify-content:space-between;background:#071b2be8;border-left:6px solid #71d8ff;padding:16px 24px}.s3-pitch-title h2{font-size:48px;color:#fff}.s3-pitch-title p{font-size:29px;line-height:1.3;margin-top:10px;color:#aadef6}.s3-pitch-title>b{font-size:58px;color:#7ae2ff;padding-left:20px;white-space:nowrap}\n.s3-pitch{position:relative;min-height:0;overflow:hidden;border:3px solid #5b9e7c;border-radius:16px;background:repeating-linear-gradient(0deg,#11633f 0%,#11633f 14.28%,#16734b 14.28%,#16734b 28.56%)}\n.s3-boundary{position:absolute;inset:3% 3% 1%;border:5px solid #cceddb99;border-radius:22px}.s3-halfway{position:absolute;top:50%;left:3%;right:3%;border-top:4px solid #cceddb99}.s3-centre{position:absolute;top:50%;left:50%;width:30%;height:29%;transform:translate(-50%,-50%);border:4px solid #cceddb99;border-radius:50%}\n.s3-top-goal{position:absolute;top:.6%;left:44%;width:12%;height:3%;border:4px solid #d5ecdf;background:repeating-linear-gradient(0deg,transparent,transparent 9px,#bfd9cc88 10px),repeating-linear-gradient(90deg,transparent,transparent 14px,#bfd9cc88 15px)}\n.s3-top-penalty{position:absolute;top:3%;left:28%;width:44%;height:23%;border:4px solid #cceddb99;border-top:0}.s3-top-six{position:absolute;top:3%;left:40%;width:20%;height:9%;border:4px solid #cceddb99;border-top:0}\n.s3-pitch-player{position:absolute;transform:translate(-50%,-50%);width:350px;display:flex;align-items:center;flex-direction:column;opacity:.35;isolation:isolate}.s3-pitch-photo{width:205px;height:205px;object-fit:contain;filter:drop-shadow(0 9px 12px #0006)}.s3-pitch-player>b{display:block;width:100%;background:#051d29f5;padding:8px 6px 3px;text-align:center;font-size:40px;line-height:1.1;color:#fff;white-space:nowrap;border-radius:8px 8px 0 0}.s3-pitch-player>span{display:block;width:100%;background:#051d29f5;text-align:center;font-size:32px;padding:4px 0 9px;line-height:1.1;border-radius:0 0 8px 8px}.s3-pitch-player em{font-style:normal;color:#ffe17c;font-size:30px}.s3-pitch-player[data-active="true"] .s3-pitch-photo{filter:drop-shadow(0 0 18px #c6ff67)}.s3-pitch-player[data-active="true"]>b{color:#d3ff7d}\n.s3-bench{border:2px solid #65758c;background:#192332eb;border-radius:14px;padding:16px 20px;overflow:hidden}.s3-bench h3{color:#9ed3ef;font-size:34px}.s3-bench>div{display:flex;justify-content:space-around;margin-top:7px}.s3-bench article{text-align:center;min-width:0;flex:1;opacity:.5}.s3-bench-photo{height:95px;width:105px;object-fit:contain;display:block;margin:auto}.s3-bench b{display:block;color:#fff;font-size:34px;line-height:1.1;white-space:nowrap}.s3-bench span{display:block;color:#d8e1eb;font-size:29px;margin-top:5px}.s3-bench small{color:#f3d77c;font-size:25px;display:block;margin-top:5px}.s3-bench [data-active="true"]{opacity:1;filter:drop-shadow(0 0 9px #9cd8ff77)}\n.s3-right{min-width:0;min-height:0;background:#081329ef;border:2px solid #486078;border-radius:16px;padding:24px;display:grid;grid-template-rows:115px 165px 650px minmax(0,1fr);gap:16px;overflow:hidden}.s3-right>header h2{font-size:45px;color:#80dfff;line-height:1.05}.s3-right>header p{font-size:29px;line-height:1.2;margin-top:12px;color:#dce9fa}\n.s3-hero-zone{position:relative;border:1px solid #4a7089;border-radius:12px;background:#142d41;overflow:hidden}.s3-hero,.s3-hero-default{position:absolute;inset:0;padding:13px 18px}.s3-hero{display:flex;align-items:center;gap:20px;visibility:hidden;opacity:0}.s3-hero-photo{width:110px;height:135px;object-fit:contain}.s3-hero h3{font-size:43px;color:#fff;line-height:1.1}.s3-hero p{font-size:29px;color:#fff;margin-top:12px;line-height:1.2}.s3-hero small{font-size:27px;color:#a9e9ff;display:block;margin-top:8px}.s3-hero-default b{font-size:36px;color:#bfff79;display:block;margin-top:4px}.s3-hero-default p{font-size:34px;margin:12px 0}.s3-hero-default small{font-size:24px;color:#cddff3}\n.s3-actions{min-height:0;display:grid;grid-template-rows:repeat(3,196px) 38px;gap:8px}.s3-action{min-width:0;border:2px solid #415a76;border-radius:11px;padding:12px 15px;background:#102337;display:none;overflow:hidden}.s3-action>header{display:flex;align-items:center;gap:18px;margin-bottom:8px}.s3-action>header b{font-size:34px;color:#82dfff}.s3-action>header span{font-size:27px;color:#f0e5b4}.s3-roll{font-style:normal;font-size:35px;margin-left:auto;color:#bbf887}.s3-action-body{display:flex;gap:15px;align-items:center}.s3-captain-photo{width:78px;height:100px;object-fit:contain;flex-shrink:0}.s3-action h3{font-size:32px;line-height:1.08;color:#fff}.s3-action p{font-size:26px;line-height:1.14;margin-top:5px;color:#e4f1ff}.s3-action small{display:block;font-size:24px;line-height:1.12;margin-top:5px;color:#dee9f7}.s3-action strong{color:#89defa}.s3-action[data-active="true"]{border-color:#b8ff6b;box-shadow:inset 0 0 18px #a4f06a22,0 0 12px #a4f06a33}.s3-page-summary{font-size:26px;line-height:1.2;color:#bfdef5;padding:4px 6px}\n.s3-bb{min-height:0;border:2px solid #9e79c6;border-radius:12px;padding:16px;background:#251a36e8;overflow:hidden}.s3-bb h2{font-size:39px;color:#e8b3ff}.s3-bb header span{display:block;font-size:23px;color:#d9c2e9;margin-top:7px}.s3-bb>p{font-size:27px;color:#f5edff;line-height:1.15;margin:15px 0 11px}.s3-bb-window{font-size:27px;color:#ffd889;display:block;line-height:1.2}.s3-future-list{display:grid;gap:8px;margin:12px 0;overflow:hidden}.s3-future{display:flex;align-items:center;gap:14px;border:1px solid #8c6ea2;border-radius:8px;background:#17152d;padding:8px 12px;opacity:.65}.s3-future-photo{height:95px;width:70px;object-fit:contain;flex-shrink:0}.s3-future small{font-size:21px;display:block;color:#e4b4ff}.s3-future b{font-size:27px;display:block;margin-top:4px;color:#fff}.s3-future p{font-size:23px;line-height:1.12;margin-top:4px;color:#e4d7f3}.s3-bb ol{margin:12px 0 0;padding-left:28px}.s3-bb li{font-size:25px;line-height:1.18;color:#f2eaff;margin-top:5px}.s3-bb[data-active="true"]{border-color:#eec16f;box-shadow:0 0 15px #c68bff33}.s3-future[data-active="true"]{opacity:1;border-color:#ffd889}.s3-initials{display:flex!important;justify-content:center;align-items:center;color:#bfefff;background:#122f41;border-radius:50%;font-size:45px;font-weight:bold}\n\'\'\'\n\nSEEK_JS=\'\'\'<script>(function(){const previous=window.seekWildcard;window.seekWildcard=function(t){previous(t);const root=document.querySelector(\'.s3-action-layout\');if(!root)return;const d=JSON.parse(root.querySelector(\'.s3-data\').textContent),tl=d.timeline,plan=d.plan,s=tl.find(x=>t>=x.start&&t<x.end)||tl.at(-1),clamp=v=>Math.max(0,Math.min(1,v)),ease=v=>1-Math.pow(1-clamp(v),3),final=t>=tl.at(-1).start;\nfor(const el of root.querySelectorAll(\'[data-s3-player]\')){const id=+el.dataset.s3Player,cue=tl.find(x=>x.type===\'players\'&&x.player_ids.includes(id)),v=cue?ease((t-cue.start)/.35):0;el.style.opacity=String(.32+.68*v);el.style.scale=String(.96+.04*v);el.dataset.active=String(s.type===\'players\'&&s.player_ids.includes(id));}\nfor(const el of root.querySelectorAll(\'[data-s3-bench]\')){const id=+el.dataset.s3Bench,cue=tl.find(x=>x.type===\'bench\'&&x.player_ids.includes(id)),v=cue?ease((t-cue.start)/.35):0;el.style.opacity=String(.4+.6*v);el.dataset.active=String(s.type===\'bench\'&&s.player_ids.includes(id));}\nconst activePlayer=s.type===\'players\'?s.player_ids[0]:null;root.querySelector(\'.s3-hero-default\').style.visibility=activePlayer?\'hidden\':\'visible\';for(const el of root.querySelectorAll(\'[data-s3-hero]\')){const show=+el.dataset.s3Hero===activePlayer;el.style.visibility=show?\'visible\':\'hidden\';el.style.opacity=show?String(ease((t-s.start)/.25)):\'0\';el.style.translate=show?`${(1-ease((t-s.start)/.25))*25}px 0`:\'0 0\';}\nconst actions=plan.action_plan,index=actions.findIndex(a=>a.gameweek===s.gameweek),offset=index>=3?2:0;for(const [i,el] of [...root.querySelectorAll(\'[data-s3-gw]\')].entries()){el.style.display=i>=offset&&i<offset+3?\'block\':\'none\';el.dataset.active=String(+el.dataset.s3Gw===s.gameweek);el.querySelector(\'.s3-roll\').style.rotate=s.gameweek===+el.dataset.s3Gw?`${360*ease((t-s.start)/.6)}deg`:\'0deg\';}\nconst rest=offset===0?actions.slice(3):actions.slice(0,2);root.querySelector(\'.s3-page-summary\').textContent=rest.length?`GW${rest[0].gameweek}–GW${rest.at(-1).gameweek}: REASSESS WITH TEAM NEWS`:\'\';const bb=tl.find(x=>x.segment_id===\'bb_intro\'),active=s.type===\'bench_boost\'||s.type===\'future_target\';root.querySelector(\'.s3-bb\').style.opacity=bb&&t>=bb.start?\'1\':\'.55\';root.querySelector(\'.s3-bb\').dataset.active=String(active);for(const el of root.querySelectorAll(\'[data-s3-future]\')){const id=+el.dataset.s3Future,cue=tl.find(x=>x.future_player_id===id);const v=cue?ease((t-cue.start)/.35):0;el.style.opacity=String(v);el.style.visibility=v>0?\'visible\':\'hidden\';el.style.translate=`${(1-v)*20}px 0`;el.dataset.active=String(s.future_player_id===id);}\n};})();</script>\'\'\'\n\nclass WildcardSlide3Visual:\n    def __init__(self,scene,contract,width,height,renderer,metadata,media=None,badges=None):\n        self.manifest=scene;self.contract=contract;self.width=width;self.height=height;self.renderer=renderer;self.metadata=metadata;self.layout=[]\n        self.timeline=scene.get(\'narration_timeline\') or generate_wildcard_slide3_narration(scene[\'squad_plan\'])\n        if self.timeline[0].get(\'start\') is None:\n            from wildcard_plan import align_timeline\n            self.timeline=align_timeline(self.timeline,max(1,len(scene[\'narration_text\'].split())/2.4))\n        self.content_html=render_html(scene[\'squad_plan\'],self.timeline,media,badges)\n        self.complete=self.draw(self.timeline[-1][\'end\']+2,self.timeline[-1][\'end\']+2)\n    def draw(self,seconds,duration):return self.renderer.animation_image(self.manifest,self.metadata,self.width,self.height,self.content_html,seconds)\n    def frame(self,seconds,duration,previous=None):return np.asarray(self.draw(seconds,duration))\n\nCSS += \'\'\'\n.s3-right{grid-template-rows:130px 165px 565px minmax(0,1fr)}\n.s3-right>header p{font-size:32px}.s3-hero p{font-size:32px}.s3-hero small{font-size:30px}.s3-hero-default small{font-size:28px}\n.s3-actions{grid-template-rows:repeat(2,250px) 45px;gap:10px}.s3-action h3{font-size:40px}.s3-action>header b{font-size:44px}.s3-action>header span{font-size:34px}.s3-action p{font-size:33px;line-height:1.12}.s3-action small{font-size:30px;line-height:1.1}.s3-captain-photo{height:125px;width:85px}.s3-page-summary{font-size:30px;line-height:1.1}\n.s3-bb h2{font-size:41px}.s3-bb header span{font-size:28px}.s3-bb>p,.s3-bb-window{font-size:32px}.s3-future small{font-size:27px}.s3-future b{font-size:32px}.s3-future p{font-size:28px}.s3-future-photo{height:110px;width:72px}.s3-bb li{font-size:29px;line-height:1.10;margin-top:5px}\n.s3-action-layout[data-final="true"] .s3-right{grid-template-rows:130px 0px 650px minmax(0,1fr)}\n.s3-action-layout[data-final="true"] .s3-hero-zone{visibility:hidden}\n.s3-action-layout[data-final="true"] .s3-actions{grid-template-rows:repeat(3,194px) 45px}\n.s3-action-layout[data-final="true"] .s3-action p:first-of-type{display:none}\n.s3-action-layout[data-final="true"] .s3-action h3{font-size:39px}\n.s3-action-layout[data-final="true"] .s3-action>header{margin-bottom:4px}\n\'\'\'\nCSS += \'\'\'\n.s3-action-layout[data-final="true"] .s3-hero-zone{display:none}\n.s3-action-layout[data-final="true"] .s3-right{grid-template-rows:130px 675px minmax(0,1fr)}\n.s3-action-layout[data-final="true"] .s3-actions{grid-template-rows:repeat(3,200px) 45px}\n.s3-action-layout[data-final="true"] .s3-captain-photo{height:105px}\n.s3-action-layout[data-final="true"] .s3-action small{font-size:29px}\n\'\'\'\n# The live pager shows two full rows; the final summary shows three and names the remaining weeks.\nSEEK_JS=SEEK_JS.replace("const actions=plan.action_plan,index=actions.findIndex(a=>a.gameweek===s.gameweek),offset=index>=3?2:0;", "root.querySelector(\'.s3-pitch-title\').style.scale=String(.98+.02*ease(t/.3));root.dataset.final=String(final);const actions=plan.action_plan,index=actions.findIndex(a=>a.gameweek===s.gameweek),count=final?3:2,offset=final?0:Math.max(0,Math.min(3,index<0?0:index));")\nSEEK_JS=SEEK_JS.replace(\'i<offset+3\',\'i<offset+count\').replace("const rest=offset===0?actions.slice(3):actions.slice(0,2);", "const rest=final?actions.slice(3):[];")\n\ndef weekly_plan_rows(plan,squad_frame,weekly_scores):\n    import pandas as pd\n    rows=[]\n    for w in plan[\'gameweek_plans\']:\n        g=w[\'gameweek\'];meta=squad_frame[[\'player_id\',\'player_name\',\'team_id\',\'position\']].copy();points=weekly_scores[weekly_scores.gameweek.eq(g)].set_index(\'player_id\').gameweek_expected_points.to_dict();meta[\'score\']=meta.player_id.map(points)\n        xi_ids={p[\'id\'] for p in w[\'starting_xi\']};xi=meta[meta.player_id.isin(xi_ids)].copy();bench=meta[~meta.player_id.isin(xi_ids)].copy()\n        bench=pd.concat([bench[bench.position.ne(\'GK\')].sort_values(\'score\',ascending=False),bench[bench.position.eq(\'GK\')]])\n        if len(xi)!=11 or len(bench)!=4 or xi.score.isna().any():raise ValueError(\'Canonical squad projections incomplete\')\n        captain=next(p for p in plan[\'squad\'] if p[\'id\']==w[\'captain_id\']);vice=next(p for p in plan[\'squad\'] if p[\'id\']==w[\'vice_captain_id\'])\n        rows.append(dict(gameweek=g,xi=xi,bench=bench,xi_xpts=float(xi.score.sum()),bench_xpts=float(bench.score.sum()) if bench.score.notna().all() else None,formation=w[\'formation\'],captain_player_id=captain[\'id\'],vice_captain_player_id=vice[\'id\'],captain_name=captain[\'name\'],vice_captain_name=vice[\'name\'],captain_incremental_xpts=points[captain[\'id\']],xi_plus_captain_xpts=float(xi.score.sum())+points[captain[\'id\']]))\n    return rows\n\ndef attach_unprojected_goalkeepers(engine,pool):\n    """Keep cheapest selectable reserve keepers without fabricating an expected-points model."""\n    import pandas as pd\n    ids=set(map(int,pool.player_id));extra=[];weekly=[]\n    for p in engine.boot[\'elements\']:\n        if p[\'id\'] in ids or p.get(\'element_type\')!=1 or p.get(\'status\')!=\'a\' or p.get(\'removed\') or not p.get(\'can_select\',True):continue\n        team=engine.teams[int(p[\'team\'])]\n        extra.append(dict(p,player_id=p[\'id\'],player_name=p[\'web_name\'],team_id=p[\'team\'],team_name=team[\'name\'],team_short=team[\'short_name\'],position=\'GK\',price=p[\'now_cost\']/engine.boot[\'game_settings\'][\'ui_currency_multiplier\'],p_start=np.nan,expected_minutes=np.nan,news_monitor=False,availability=\'AVAILABLE\',xpts_wildcard_horizon=np.nan,xpts_freehit_gw=np.nan,value_score=np.nan,market_net=p.get(\'transfers_in_event\',0)-p.get(\'transfers_out_event\',0),wc_eligible=False,wc_core=False,wc_monitor_reason=\'Reserve goalkeeper; playing projection unavailable\',wc_routes=[],wc_price_tier=\'BUDGET\',wc_captaincy_weeks=0,projection_status=\'unavailable\'))\n        for g in engine.primary:weekly.append(dict(player_id=p[\'id\'],gameweek=g,gameweek_expected_points=np.nan,expected_minutes=np.nan,p_start=np.nan,p_play=np.nan,fixture_count=0,fixtures=\'Projection unavailable\',xg90=np.nan,xa90=np.nan))\n    if extra:\n        pool=pd.concat([pool,pd.DataFrame(extra)],ignore_index=True);engine.pool=pool.copy();engine.scores=pd.concat([engine.scores,pd.DataFrame(weekly)],ignore_index=True)\n    return pool\n\ndef unprojected_keeper_card(engine,pid):\n    p=engine.players[int(pid)];team=engine.teams[int(p[\'team\'])];weeks=[]\n    for g in engine.primary:\n        fs=[dict(f,xpts=None,expected_minutes=None,p_start=None,components={},missing_components=[\'Player playing-time projection unavailable\']) for f in engine.fdr.for_team(int(p[\'team\']),g,\'GK\')]\n        weeks.append(dict(gameweek=g,fixtures=fs,xpts=None,expected_minutes=None,blank=not fs))\n    return dict(id=int(pid),name=p[\'web_name\'],full_name=p.get(\'first_name\',\'\')+\' \'+p.get(\'second_name\',\'\'),club=team[\'short_name\'],club_name=team[\'name\'],team_id=p[\'team\'],position=\'GK\',price=p[\'now_cost\']/engine.boot[\'game_settings\'][\'ui_currency_multiplier\'],photo_code=p[\'code\'],projection_status=\'unavailable\',expected_points=None,horizon_expected_points=None,expected_minutes=None,base_expected_minutes=None,conditional_minutes=None,starting_probability=None,ownership_pct=num(p.get(\'selected_by_percent\')),top10k_ownership_pct=None,transfers_in=p.get(\'transfers_in_event\',0),transfers_out=p.get(\'transfers_out_event\',0),net_transfers=p.get(\'transfers_in_event\',0)-p.get(\'transfers_out_event\',0),market_direction=\'UNASSESSED\',price_change_event=num(p.get(\'cost_change_event\')),availability={\'code\':p[\'status\'],\'label\':\'AVAILABLE\',\'flagged\':False,\'chance_pct\':p.get(\'chance_of_playing_next_round\'),\'news\':p.get(\'news\',\'\'),\'source\':\'https://fantasy.premierleague.com/api/bootstrap-static/\'},injury_status=\'AVAILABLE\',rotation_risk=\'UNASSESSED\',penalties_order=None,corners_order=None,free_kicks_order=None,role_note=\'Low-cost backup; playing time not confirmed\',recent_minutes=[],history_sample_size=0,latest_match=None,latest_match_text=\'No verified recent appearance in the projection dataset\',latest_league_match=None,match_coverage={},club_news=[],verified_news=[],weeks=weeks,next_fixtures=[f[\'opponent\']+(\' (H)\' if f[\'home\'] else \' (A)\') for w in weeks for f in w[\'fixtures\']],fdr=[f[\'fdr\'] for w in weeks for f in w[\'fixtures\']],fdr_mean=None,fdr_source=engine.fdr.source,value_score=None,xg90=None,xa90=None,projection_limitations=[\'Playing-time and player-points projections unavailable; excluded from bench totals\'],selection_policy=VERSION,selection_eligible=True,selection_classification=\'LOW-COST BACKUP\',selection_reason=\'Cheapest available valid reserve goalkeeper; playing status optional\',routes_to_points=[],price_tier=\'BUDGET\',captaincy_gameweeks=0,alternative_ids=[],is_bench=True,is_captain=False,is_vice_captain=False,reason=\'Low-cost valid backup goalkeeper. No playing-time or points projection is asserted.\')\n\nCSS += """.s3-pitch-title>b small{display:block;font-size:32px;color:#d8ecfa;text-align:right;margin-top:6px}"""\n',encoding='utf-8')
sys.modules.pop('wildcard_slide3',None)
(_vx_module_dir/'wildcard_plan.py').write_text(_VX_SQUAD_PLAN_SOURCE,encoding='utf-8')
(_vx_module_dir/'wildcard_plan_view.py').write_text(_VX_SQUAD_PLAN_VIEW_SOURCE,encoding='utf-8')
sys.modules.pop('wildcard_plan',None)
sys.modules.pop('wildcard_plan_view',None)
(_vx_module_dir/'wildcard_plan_audio.py').write_text('"""Measured segment-level voice alignment using the notebook\'s existing speech utility."""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport re\nimport wave\nimport uuid\nfrom pathlib import Path\nfrom typing import Any, Callable, Mapping\nfrom wildcard_plan import align_timeline\n\n\ndef prepare_segment_audio(scene: dict[str,Any], script_segment: Mapping[str,Any], work: Path,\n                          synthesise: Callable[[dict[str,Any]],Path],\n                          decode_pcm: Callable[[Path,Path],None]) -> Path:\n    segments=scene[\'narration_timeline\']\n    canonical=\' \'.join(segment[\'narration_text\'] for segment in segments)\n    if \' \'.join(canonical.split())!=\' \'.join(scene[\'narration_text\'].split()):\n        raise ValueError(\'Update the structured Slide 3 timeline when editing its narration\')\n    def valid_pcm(path: Path) -> bool:\n        try:\n            with wave.open(str(path),\'rb\') as audio:\n                return audio.getnchannels()==2 and audio.getsampwidth()==2 and audio.getframerate()==48000 and audio.getnframes()>0 and len(audio.readframes(audio.getnframes()))==audio.getnframes()*4\n        except (OSError,EOFError,wave.Error):return False\n    tracks=[];durations=[]\n    for item in segments:\n        spoken=item[\'narration_text\']\n        voice=synthesise(dict(script_segment,id=scene[\'scene_id\']+\'_\'+item[\'segment_id\'],text=spoken,scene=scene,narration_cue=item))\n        if scene.get(\'visual_type\') == \'wc_blueprint\':\n            boundary_file = Path(voice).with_suffix(\'.words.json\')\n            if boundary_file.is_file():\n                try:\n                    words = json.loads(boundary_file.read_text(encoding=\'utf-8\'))\n                    if isinstance(words, list) and all(isinstance(row, dict) for row in words):\n                        item[\'word_boundaries\'] = [\n                            {\'text\': str(row.get(\'text\', \'\')), \'offset_ticks\': int(row.get(\'offset_ticks\', 0))}\n                            for row in words if str(row.get(\'text\', \'\')).strip()\n                        ]\n                except (OSError, ValueError, TypeError):\n                    item[\'word_boundaries\'] = []\n        digest=hashlib.sha256(Path(voice).read_bytes()).hexdigest()[:20]\n        pcm=work/f\'slide3_voice_{digest}.wav\'\n        if not valid_pcm(pcm):\n            temporary=work/(pcm.stem+\'.\'+uuid.uuid4().hex+\'.part.wav\')\n            try:\n                decode_pcm(Path(voice),temporary)\n                if not valid_pcm(temporary):raise ValueError(\'Speech segment did not decode to complete nonempty PCM\')\n                temporary.replace(pcm)\n            finally:\n                if temporary.exists():temporary.unlink()\n        with wave.open(str(pcm),\'rb\') as audio:\n            if audio.getnchannels()!=2 or audio.getsampwidth()!=2 or audio.getframerate()!=48000:\n                raise ValueError(\'Slide 3 voice segments require 48kHz stereo 16-bit PCM\')\n            durations.append(audio.getnframes()/48000)\n        tracks.append(pcm)\n    digest=hashlib.sha256(\'\'.join(str(track)+str(seconds) for track,seconds in zip(tracks,durations)).encode()).hexdigest()[:20]\n    combined=work/f\'slide3_measured_{digest}.wav\'\n    if not valid_pcm(combined):\n        temporary=work/(combined.stem+\'.\'+uuid.uuid4().hex+\'.part.wav\')\n        with wave.open(str(temporary),\'wb\') as output:\n            output.setnchannels(2);output.setsampwidth(2);output.setframerate(48000)\n            for track in tracks:\n                with wave.open(str(track),\'rb\') as audio:output.writeframes(audio.readframes(audio.getnframes()))\n        temporary.replace(combined)\n    scene[\'narration_timeline\']=align_timeline(segments,sum(durations),durations)\n    scene[\'timing\'][\'player_alignment\']=\'measured_segments\'\n    return combined\n',encoding='utf-8')
sys.modules.pop('wildcard_plan_audio',None)
(_vx_module_dir/'free_hit_plan.py').write_text('"""Independent next-Gameweek Free Hit selection. No old Free Hit selector is used."""\nfrom __future__ import annotations\nimport copy\nimport math\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom typing import Any, Mapping, Sequence\nfrom urllib.parse import urlparse\nimport numpy as np\nfrom scipy.optimize import Bounds, LinearConstraint, milp\nfrom scipy.sparse import lil_matrix\n\nPOSITIONS={1:\'GK\',2:\'DEF\',3:\'MID\',4:\'FWD\'}\nCOUNTS={\'GK\':2,\'DEF\':5,\'MID\':5,\'FWD\':3}\nSAMPLE_NARRATION=\'Let’s build the Free Hit plan for Gameweek [NEXT_GW]. [VERIFIED NEWS STATUS]. [CAPTAIN] leads the one-week projection. [PLAYER] offers [VERIFIED REASON]. The bench provides minimum cover. The final draft leaves £[BANK] million unspent. Check late team news before the deadline.\'\n\n\ndef number(value: Any) -> float | None:\n    try:\n        result=float(value)\n        return result if math.isfinite(result) else None\n    except (ValueError,TypeError):return None\n\n\ndef next_gameweek(events: Sequence[Mapping[str,Any]], now: datetime | None=None) -> int:\n    current=now or datetime.now(timezone.utc)\n    upcoming=[]\n    for event in events:\n        try:deadline=datetime.fromisoformat(str(event[\'deadline_time\']).replace(\'Z\',\'+00:00\'))\n        except (ValueError,KeyError):continue\n        if deadline>current:upcoming.append(int(event[\'id\']))\n    if not upcoming:raise ValueError(\'No upcoming Gameweek is available\')\n    return min(upcoming)\n\n\ndef get_next_gw_fixtures(fixtures: Sequence[Mapping[str,Any]], next_gw: int) -> dict[int,list[dict[str,Any]]]:\n    result={}\n    for fixture in fixtures:\n        if fixture.get(\'event\')!=next_gw or fixture.get(\'finished\'):continue\n        for home in (True,False):\n            team=int(fixture[\'team_h\'] if home else fixture[\'team_a\'])\n            result.setdefault(team,[]).append({\'gameweek\':next_gw,\'opponent_id\':int(fixture[\'team_a\'] if home else fixture[\'team_h\']),\n                \'home\':home,\'fdr\':number(fixture.get(\'team_h_difficulty\' if home else \'team_a_difficulty\')),\'kickoff_time\':fixture.get(\'kickoff_time\')})\n    return result\n\n\ndef validate_press_conference_data(data: Any, next_gw: int, now: datetime) -> list[dict[str,Any]]:\n    records=data.get(\'events\',[]) if isinstance(data,Mapping) else data or []\n    valid=[]\n    for record in records:\n        try:\n            stamp=datetime.fromisoformat(str(record[\'checked_at\']).replace(\'Z\',\'+00:00\'))\n            source=urlparse(str(record[\'source_url\']))\n            age=(now-stamp).total_seconds()\n        except (KeyError,ValueError,TypeError):continue\n        if not record.get(\'confirmed\') or not source.hostname or source.scheme!=\'https\' or not 0<=age<=48*3600:continue\n        if record.get(\'gameweek\') not in (None,next_gw):continue\n        valid.append(dict(record))\n    return valid\n\n\ndef calculate_free_hit_player_score(player: Mapping[str,Any], model: Mapping[str,Any], fixtures: list[dict[str,Any]], completed: int) -> dict[str,Any]:\n    count=len(fixtures);projection=number(model.get(\'gameweek_expected_points\'))\n    minutes=number(model.get(\'expected_minutes\'));probability=number(model.get(\'p_start\'))\n    source=\'one-week model\'\n    if minutes is None or probability is None:\n        minutes=min(90.,(number(player.get(\'minutes\')) or 0)/max(1,completed))*count\n        probability=min(1.,(number(player.get(\'starts\')) or 0)/max(1,completed));source=\'observed role proxy\'\n    per_match=minutes/max(1,count)\n    secure=probability>=.8 and per_match>=65 and not model.get(\'news_monitor\',False)\n    ratings=[f[\'fdr\'] for f in fixtures if f[\'fdr\'] is not None]\n    average=sum(ratings)/len(ratings) if ratings else None\n    form=number(player.get(\'form\'));position=POSITIONS[player[\'element_type\']]\n    underlying=number(player.get(\'clean_sheets_per_90\') if position in (\'GK\',\'DEF\') else player.get(\'expected_goal_involvements_per_90\'))\n    fixture_score=(6-average)/5 if average is not None else 0.\n    fallback=count*(.45*probability*min(1.,per_match/90)+.3*fixture_score+.15*min(1.,(form or 0)/8)+.1*min(1.,underlying or 0))\n    score=projection if projection is not None else fallback\n    return {\'score\':max(0.,score),\'projected_points\':projection,\'minutes\':minutes,\'starting_probability\':probability,\n        \'secure\':secure,\'role_source\':source,\'form\':form,\'fdr\':average,\n        \'score_dimensions\':{\'next_gw_xpts_score\':projection,\'next_gw_fixture_score\':fixture_score,\n            \'fixture_count\':count,\'expected_minutes_score\':per_match,\'recent_role_score\':probability,\n            \'attacking_or_clean_sheet_score\':underlying,\'rotation_risk_penalty\':1-probability}}\n\n\ndef build_free_hit_player_reason(player: Mapping[str,Any]) -> str:\n    if player[\'projected_points\'] is not None:return f"{player[\'projected_points\']:.1f} next-Gameweek xPts; {player[\'fixture_text\']}"\n    if player[\'fdr\'] is not None:return f"Average immediate FDR {player[\'fdr\']:.2f}; {player[\'fixture_text\']}"\n    return \'Available for selection; monitor the observed minutes outlook\'\n\n\ndef select_valid_formation(xi: Sequence[Mapping[str,Any]]) -> str:\n    counts=Counter(p[\'position\'] for p in xi)\n    if len(xi)!=11 or counts[\'GK\']!=1 or not 3<=counts[\'DEF\']<=5 or not 2<=counts[\'MID\']<=5 or not 1<=counts[\'FWD\']<=3:raise ValueError(\'Invalid Free Hit formation\')\n    return f"{counts[\'DEF\']}-{counts[\'MID\']}-{counts[\'FWD\']}"\n\n\ndef select_free_hit_starting_xi(pool: Sequence[Mapping[str,Any]], values: Any) -> list[dict[str,Any]]:\n    count=len(pool);return [dict(p) for i,p in enumerate(pool) if values[count+i]>.5]\n\n\ndef select_minimum_cover_bench(squad: Sequence[Mapping[str,Any]], xi: Sequence[Mapping[str,Any]]) -> list[dict[str,Any]]:\n    starting={p[\'id\'] for p in xi}\n    return sorted([dict(p) for p in squad if p[\'id\'] not in starting],key=lambda p:(p[\'position\']==\'GK\',not p[\'secure\'],-p[\'score\']))\n\n\ndef select_captain_and_vice_captain(xi: Sequence[Mapping[str,Any]], captain_id: int) -> tuple[dict[str,Any],dict[str,Any]]:\n    captain=next(dict(p) for p in xi if p[\'id\']==captain_id)\n    vice=max((dict(p) for p in xi if p[\'id\']!=captain_id),key=lambda p:p[\'score\'])\n    return captain,vice\n\n\ndef optimise_free_hit_squad(pool: list[dict[str,Any]], available_budget: float, multiplier: int=10, seconds: float=120) -> dict[str,Any]:\n    candidates=[p for p in pool if number(p.get(\'projected_points\')) is not None]\n    count=len(candidates)\n    if count<15 or not math.isfinite(available_budget) or available_budget<=0:\n        raise ValueError(\'Insufficient verified Free Hit projection or budget data\')\n    size=3*count;rows=[];lower=[];upper=[]\n    def add(items,low,high):\n        rows.append(items);lower.append(low);upper.append(high)\n    for position,slots in COUNTS.items():\n        add({i:1 for i,p in enumerate(candidates) if p[\'position\']==position},slots,slots)\n    for club in {p[\'team_id\'] for p in candidates}:\n        add({i:1 for i,p in enumerate(candidates) if p[\'team_id\']==club},0,3)\n    add({i:p[\'cost_units\'] for i,p in enumerate(candidates)},0,math.floor(available_budget*multiplier+1e-8))\n    add({count+i:1 for i in range(count)},11,11)\n    add({2*count+i:1 for i in range(count)},1,1)\n    for position,limits in {\'GK\':(1,1),\'DEF\':(3,5),\'MID\':(2,5),\'FWD\':(1,3)}.items():\n        add({count+i:1 for i,p in enumerate(candidates) if p[\'position\']==position},*limits)\n    bound_upper=np.ones(size)\n    for i,p in enumerate(candidates):\n        add({count+i:1,i:-1},-math.inf,0)\n        add({2*count+i:1,count+i:-1},-math.inf,0)\n        if not p[\'secure\']:\n            bound_upper[count+i]=0\n            bound_upper[2*count+i]=0\n    # One dependable emergency outfielder while keeping most funds on XI.\n    cover={i:1 for i,p in enumerate(candidates) if p[\'secure\'] and p[\'position\']!=\'GK\'}\n    cover.update({count+i:-1 for i,p in enumerate(candidates) if p[\'secure\'] and p[\'position\']!=\'GK\'})\n    add(cover,1,math.inf)\n    objective=np.zeros(size)\n    for i,p in enumerate(candidates):\n        points=float(p[\'projected_points\'])\n        objective[count+i]=-(1-.035)*points\n        objective[2*count+i]=-points\n        objective[i]=-.035*points\n    matrix=lil_matrix((len(rows),size))\n    for row_index,values in enumerate(rows):\n        for column,value in values.items():matrix[row_index,column]=value\n    solution=milp(objective,integrality=np.ones(size),bounds=Bounds(np.zeros(size),bound_upper),\n        constraints=LinearConstraint(matrix.tocsr(),lower,upper),\n        options={\'time_limit\':max(1.,float(seconds)),\'mip_rel_gap\':.001})\n    if solution.x is None:\n        raise ValueError(\'No legal projected Free Hit XI and affordable emergency bench under the active constraints\')\n    values=solution.x\n    squad=[dict(p) for i,p in enumerate(candidates) if values[i]>.5]\n    xi=select_free_hit_starting_xi(candidates,values)\n    captain_ids=[p[\'id\'] for i,p in enumerate(candidates) if values[2*count+i]>.5]\n    if len(squad)!=15 or len(xi)!=11 or len(captain_ids)!=1:\n        raise ValueError(\'Free Hit optimizer returned an invalid squad, XI or captain\')\n    captain,vice=select_captain_and_vice_captain(xi,captain_ids[0])\n    total_cost=sum(p[\'price\'] for p in squad)\n    return {\'squad\':squad,\'starting_xi\':xi,\'bench\':select_minimum_cover_bench(squad,xi),\n        \'captain\':captain,\'vice_captain\':vice,\'formation\':select_valid_formation(xi),\n        \'total_cost\':total_cost,\'bank\':round(available_budget-total_cost,8),\n        \'data_health\':{\'valid\':True,\'spend_optimality_proven\':False,\n            \'points_optimality_proven\':int(solution.status)==0,\'solver_gap\':getattr(solution,\'mip_gap\',None),\n            \'objective\':\'Single-Gameweek starting XI + incremental captaincy; small projected bench-cover contribution; no compulsory spending\',\n            \'warnings\':([\'Feasible Free Hit solution returned without formal proof of optimality\'] if solution.status else [])}}\n\ndef build_free_hit_squad(players: Sequence[Mapping[str,Any]], teams: Sequence[Mapping[str,Any]], fixtures: Sequence[Mapping[str,Any]], next_gw: int,\n                         available_budget: float, press_conference_data: Any=None, team_news_data: Any=None, projection_data: Any=None,\n                         target_bank: float=0., context: Mapping[str,Any] | None=None) -> dict[str,Any]:\n    if target_bank!=0.:raise ValueError(\'Free Hit does not reserve a future bank\')\n    context=dict(context or {});now=context.get(\'now\') or datetime.now(timezone.utc)\n    if context.get(\'news_processed\') is False:raise ValueError(\'Process the latest available team-news inputs before building the Free Hit\')\n    team_map={int(t[\'id\']):t for t in teams};fixtures_by_team=get_next_gw_fixtures(fixtures,next_gw)\n    if not fixtures_by_team:raise ValueError(\'No verified fixtures for the selected Gameweek\')\n    press=validate_press_conference_data(press_conference_data,next_gw,now)\n    press=[row for row in press if row.get(\'kind\')==\'press_conference\' or row.get(\'type\')==\'press_conference\']\n    news=validate_press_conference_data(team_news_data,next_gw,now);records=press+news\n    models={int(row[\'player_id\']):row for row in projection_data or [] if row.get(\'gameweek\')==next_gw}\n    multiplier=int(context.get(\'currency_multiplier\',10));pool=[]\n    for player in players:\n        player_id=int(player[\'id\']);team_id=int(player.get(\'team\',-1));cost=number(player.get(\'now_cost\'))\n        player_news=[r for r in records if r.get(\'player_id\')==player_id]\n        if player.get(\'status\')!=\'a\' or player.get(\'removed\') or not player.get(\'can_select\',True) or player.get(\'element_type\') not in POSITIONS:continue\n        if number(player.get(\'chance_of_playing_next_round\')) not in (None,100.):continue\n        if any(r.get(\'unavailable\') or r.get(\'suspended\') or r.get(\'monitor\') for r in player_news):continue\n        if cost is None or cost<=0 or cost!=int(cost) or team_id not in fixtures_by_team or team_id not in team_map:continue\n        completed=sum(bool(f.get(\'finished\')) and team_id in (f.get(\'team_h\'),f.get(\'team_a\')) for f in fixtures) or int(context.get(\'completed_gameweeks\',1))\n        metric=calculate_free_hit_player_score(player,models.get(player_id,{}),fixtures_by_team[team_id],completed)\n        fixtures_live=[dict(f,opponent=team_map[f[\'opponent_id\']][\'short_name\']) for f in fixtures_by_team[team_id]]\n        record=dict(metric,id=player_id,name=player[\'web_name\'],full_name=(str(player.get(\'first_name\',\'\'))+\' \'+str(player.get(\'second_name\',\'\'))).strip(),\n            team_id=team_id,club=team_map[team_id][\'short_name\'],club_name=team_map[team_id][\'name\'],position=POSITIONS[player[\'element_type\']],price=cost/multiplier,\n            cost_units=int(cost),photo_code=player.get(\'code\'),fixtures=fixtures_live,fixture_text=\', \'.join(f"{f[\'opponent\']} ({\'H\' if f[\'home\'] else \'A\'})" for f in fixtures_live),\n            latest_news_checked=bool(player_news),risk_note=None if metric[\'secure\'] else \'Minutes uncertain; minimum cover only\')\n        record[\'reason\']=build_free_hit_player_reason(record);pool.append(record)\n    if len({p[\'id\'] for p in pool})!=len(pool):raise ValueError(\'Duplicate player IDs\')\n    plan=optimise_free_hit_squad(pool,available_budget,multiplier,float(context.get(\'solver_seconds\',120)))\n    timestamp=max((r[\'checked_at\'] for r in records),default=None)\n    status={\'available\':bool(records),\'press_available\':bool(press),\'freshness_timestamp\':timestamp or context.get(\'source_snapshot_at\'),\n        \'label\':\'Latest press-conference/team news checked\' if records else \'Latest available team news used\'}\n    xi=plan[\'starting_xi\'];projections=[p[\'projected_points\'] for p in xi]\n    total=sum(projections) if all(value is not None for value in projections) else None\n    plan.update(chip=\'freehit\',next_gw=next_gw,analysis_gws=[next_gw],budget=available_budget,\n        next_gw_fixtures=fixtures_by_team,projected_points_total=total,\n        captain_adjusted_xpts=total+plan[\'captain\'][\'projected_points\'] if total is not None else None,\n        press_conference_status=status,selection_summary={\'secure_starters\':sum(p[\'secure\'] for p in xi),\'clubs\':len({p[\'team_id\'] for p in plan[\'squad\']})},\n        player_reasons={str(p[\'id\']):p[\'reason\'] for p in plan[\'squad\']},subtitle=\'One-week squad built after the latest available press-conference updates\' if press else \'One-week squad built using the latest available team news\')\n    if not records:plan[\'data_health\'][\'warnings\'].append(\'No fresh verified press-conference input; official FPL availability and latest supplied model used\')\n    from gameweek_plan import build_gameweek_plans\n    plan[\'gameweek_plans\']=build_gameweek_plans(plan,projection_data or [],fixtures,teams)\n    validate_free_hit_plan(plan)\n    return plan\n\n\ndef validate_free_hit_plan(plan: Mapping[str,Any]) -> None:\n    squad=plan[\'squad\'];xi=plan[\'starting_xi\'];bench=plan[\'bench\']\n    if len(squad)!=15 or len({p[\'id\'] for p in squad})!=15 or Counter(p[\'position\'] for p in squad)!=Counter(COUNTS):raise ValueError(\'Invalid 15-player squad\')\n    if max(Counter(p[\'team_id\'] for p in squad).values())>3 or len(bench)!=4:raise ValueError(\'Invalid club/bench count\')\n    if {p[\'id\'] for p in xi}&{p[\'id\'] for p in bench} or {p[\'id\'] for p in xi+bench}!={p[\'id\'] for p in squad}:raise ValueError(\'Invalid squad partition\')\n    select_valid_formation(xi)\n    if plan[\'bank\']<0 or abs(plan[\'bank\']+plan[\'total_cost\']-plan[\'budget\'])>1e-6:raise ValueError(\'Budget mismatch\')\n    if any(not p[\'fixtures\'] or any(f[\'gameweek\']!=plan[\'next_gw\'] for f in p[\'fixtures\']) for p in squad):raise ValueError(\'Future/blank fixture in Free Hit\')\n    if plan[\'captain\'][\'id\']==plan[\'vice_captain\'][\'id\'] or any(p[\'id\'] not in {r[\'id\'] for r in xi} for p in (plan[\'captain\'],plan[\'vice_captain\'])):raise ValueError(\'Invalid captaincy\')\n\n\ndef generate_free_hit_slide3_narration(plan):\n    from freehit_analyst import squad_segments\n    return squad_segments(plan)\n\n\ndef align_timeline(segments: Sequence[Mapping[str,Any]], duration: float, segment_durations: Sequence[float] | None=None) -> list[dict[str,Any]]:\n    result=copy.deepcopy(list(segments));weights=[len(s[\'narration_text\'].split()) for s in result]\n    durations=list(segment_durations) if segment_durations is not None else [duration*w/sum(weights) for w in weights]\n    if duration<=0 or len(result)!=len(durations) or any(v<=0 or not math.isfinite(v) for v in durations) or abs(sum(durations)-duration)>.05:raise ValueError(\'Invalid narration duration\')\n    cursor=0.\n    for segment,seconds in zip(result,durations):segment.update(start=cursor,end=cursor+seconds,alignment=\'measured_segments\' if segment_durations is not None else \'estimated_word_counts\',transition_overlap=.3);cursor+=seconds\n    return result\n',encoding='utf-8')
sys.modules.pop('free_hit_plan',None)
(_vx_module_dir/'gameweek_plan.py').write_text('"""Image-led per-Gameweek XI evidence from a fixed selected 15; no transfer planning."""\nfrom __future__ import annotations\nfrom itertools import combinations\nfrom typing import Any, Mapping, Sequence\nimport math\n\n\ndef finite(value: Any) -> float | None:\n    try:\n        result=float(value)\n        return result if math.isfinite(result) else None\n    except (ValueError,TypeError):return None\n\n\ndef build_gameweek_plans(plan: Mapping[str,Any], models: Sequence[Mapping[str,Any]], fixtures: Sequence[Mapping[str,Any]], teams: Sequence[Mapping[str,Any]]) -> list[dict[str,Any]]:\n    team_map={int(t[\'id\']):t for t in teams};result=[]\n    for gw in plan[\'analysis_gws\']:\n        lookup={int(row[\'player_id\']):row for row in models if row.get(\'gameweek\')==gw};players=[]\n        for player in plan[\'squad\']:\n            row=lookup.get(player[\'id\'],{});xp=finite(row.get(\'gameweek_expected_points\'))\n            if xp is None and plan[\'chip\']==\'freehit\':xp=player.get(\'projected_points\')\n            live=[]\n            for fixture in fixtures:\n                if fixture.get(\'event\')!=gw or fixture.get(\'finished\') or player[\'team_id\'] not in (fixture[\'team_h\'],fixture[\'team_a\']):continue\n                home=fixture[\'team_h\']==player[\'team_id\'];opponent=fixture[\'team_a\'] if home else fixture[\'team_h\']\n                live.append(team_map[opponent][\'short_name\']+(\' (H)\' if home else \' (A)\'))\n            record=dict(player,xpts=xp,gameweek=gw,fixture_text=\', \'.join(live) or \'NO SCHEDULED FIXTURE\')\n            probability=finite(row.get(\'p_start\'));minutes=finite(row.get(\'expected_minutes\'))\n            record[\'eligible\']=bool(live) and (probability is None or probability>=.6) and (minutes is None or minutes>0)\n            players.append(record)\n        original_ids={p[\'id\'] for p in plan[\'starting_xi\']}\n        xi=[p for p in players if p[\'id\'] in original_ids];complete=all(p[\'xpts\'] is not None for p in players)\n        if complete:\n            options=[]\n            # 15 choose 11 is small; the complete legal XI search avoids inherited selector rules.\n            for candidate in combinations([p for p in players if p[\'eligible\']],11):\n                counts={position:sum(p[\'position\']==position for p in candidate) for position in (\'GK\',\'DEF\',\'MID\',\'FWD\')}\n                if counts[\'GK\']==1 and 3<=counts[\'DEF\']<=5 and 2<=counts[\'MID\']<=5 and 1<=counts[\'FWD\']<=3:\n                    options.append((sum(p[\'xpts\'] for p in candidate)+max(p[\'xpts\'] for p in candidate),candidate))\n            if options:xi=list(max(options,key=lambda item:item[0])[1])\n        if gw==plan[\'next_gw\']:xi=[p for p in players if p[\'id\'] in original_ids]\n        cap=next(p for p in xi if p[\'id\']==plan[\'captain\'][\'id\']) if gw==plan[\'next_gw\'] else max(xi,key=lambda p:p[\'xpts\'] if p[\'xpts\'] is not None else -1)\n        vice=next(p for p in xi if p[\'id\']==plan[\'vice_captain\'][\'id\']) if gw==plan[\'next_gw\'] else max((p for p in xi if p[\'id\']!=cap[\'id\']),key=lambda p:p[\'xpts\'] if p[\'xpts\'] is not None else -1)\n        total=sum(p[\'xpts\'] for p in xi) if all(p[\'xpts\'] is not None for p in xi) else None\n        counts={position:sum(p[\'position\']==position for p in xi) for position in (\'DEF\',\'MID\',\'FWD\')}\n        result.append({\'gameweek\':gw,\'starting_xi\':xi,\'captain_id\':cap[\'id\'],\'vice_captain_id\':vice[\'id\'],\n            \'formation\':f"{counts[\'DEF\']}-{counts[\'MID\']}-{counts[\'FWD\']}",\'xi_xpts\':total,\n            \'captain_adjusted_xpts\':total+cap[\'xpts\'] if total is not None else None,\n            \'status\':\'MODEL PLAN · RECHECK TEAM NEWS\' if all(p[\'eligible\'] for p in xi) else \'PROVISIONAL XI · AVAILABILITY CHECK REQUIRED\'})\n    return result\n',encoding='utf-8')
sys.modules.pop('gameweek_plan',None)
# Slide 2 modules preserve the existing fixture table renderer and FDR scoring.
_VX_FIXTURE_SOURCE = '"""Slide 2 evidence, narration and explicit-time row focus; existing FDR renderer retained."""\nfrom __future__ import annotations\n\nimport copy\nimport math\nfrom collections.abc import Mapping, Sequence\nfrom typing import Any\n\nPOSITIONS = {1: \'GK\', 2: \'DEF\', 3: \'MID\', 4: \'FWD\'}\nVERSION = \'shared-fdr-six-gw-narrated-rows-v1\'\n\n\ndef number(value: Any) -> float | None:\n    try:\n        result = float(value)\n        return result if math.isfinite(result) else None\n    except (ValueError, TypeError):\n        return None\n\n\ndef average_fdr(row: Mapping[str, Any], gameweeks: Sequence[int]) -> float | None:\n    # Preserve the displayed calculation: every match counts; blanks/missing ratings do not.\n    values = [number(fixture.get(\'fdr\')) for week in row.get(\'weeks\', [])\n              if week.get(\'gameweek\') in gameweeks for fixture in week.get(\'fixtures\', [])]\n    valid = [value for value in values if value is not None and 1 <= value <= 5]\n    return sum(valid) / len(valid) if valid else None\n\n\ndef rank_teams(rows: Sequence[Mapping[str, Any]], gameweeks: Sequence[int]) -> list[dict[str, Any]]:\n    ranked = [dict(copy.deepcopy(row), average_fdr=average_fdr(row, gameweeks),\n                   row_id=f"team-row-{int(row[\'team_id\'])}") for row in rows]\n    if len({row[\'team_id\'] for row in ranked}) != len(ranked):\n        raise ValueError(\'Duplicate team IDs in fixture table\')\n    ranked.sort(key=lambda row: (row[\'average_fdr\'] if row[\'average_fdr\'] is not None else math.inf,\n                                 str(row[\'club\'])))\n    return ranked\n\n\ndef select_teams(rows: Sequence[Mapping[str, Any]], gameweeks: Sequence[int]) -> dict[str, Any]:\n    ranked = rank_teams(rows, gameweeks)\n    known = [row for row in ranked if row[\'average_fdr\'] is not None]\n    targets = known[:5]\n    target_ids = {row[\'team_id\'] for row in targets}\n    avoids = [row for row in reversed(known) if row[\'team_id\'] not in target_ids][:3]\n    return {\'rows\': ranked, \'targets\': targets, \'avoids\': avoids}\n\n\ndef select_target_player(team_id: int, fpl_players: Sequence[Mapping[str, Any]],\n                         fixtures: Sequence[Mapping[str, Any]], context: Mapping[str, Any]) -> dict[str, Any]:\n    projections = {int(row[\'player_id\']): row for row in context.get(\'player_models\', [])}\n    next_gw = context[\'gameweeks\'][0]\n    active = any(week[\'gameweek\'] == next_gw and week.get(\'fixtures\') for week in fixtures)\n    candidates = []\n    team_players = [player for player in fpl_players if int(player.get(\'team\', -1)) == team_id]\n    for player in team_players:\n        if player.get(\'status\') != \'a\' or not player.get(\'can_select\', True) or player.get(\'removed\') or (number(player.get(\'chance_of_playing_next_round\')) is not None and float(player[\'chance_of_playing_next_round\']) < 100):\n            continue\n        if context[\'chip\'] == \'freehit\' and not active:\n            continue\n        player_id = int(player[\'id\']); model = projections.get(player_id, {})\n        minutes = number(model.get(\'expected_minutes\')); start = number(model.get(\'p_start\'))\n        if minutes is None or start is None or minutes < 65 or start < .8 or model.get(\'news_monitor\'):\n            continue\n        price = number(player.get(\'now_cost\')); form = number(player.get(\'form\'))\n        if price is None or price <= 0:\n            continue\n        price /= float(context.get(\'currency_multiplier\') or 10)\n        position = POSITIONS.get(int(player.get(\'element_type\', 0)), \'UNKNOWN\')\n        projection = number(model.get(\'xpts_freehit_gw\' if context[\'chip\'] == \'freehit\' else \'xpts_wildcard_horizon\'))\n        form = form if form is not None else 0.\n        attacking = number(player.get(\'expected_goal_involvements_per_90\'))\n        defending = number(player.get(\'clean_sheets_per_90\'))\n        underlying = defending if position in (\'GK\', \'DEF\') else attacking\n        score = (projection / price if projection is not None else form / price) + .3 * form\n        score += .5 * (underlying or 0.) + start - max(0., 80. - minutes) / 90.\n        if projection is not None:\n            window = \'one-week\' if context[\'chip\'] == \'freehit\' else \'six-week\'\n            reason = f\'{projection:.1f} projected {window} points, {minutes:.0f} expected minutes\'\n        elif number(player.get(\'form\')) is not None:\n            reason = f\'FPL form {form:.1f} and {minutes:.0f} model expected minutes\'\n        else:\n            reason = f\'{start:.0%} model starting chance and {minutes:.0f} expected minutes\'\n        candidates.append((score, player_id, {\'player_id\': player_id, \'player_name\': player[\'web_name\'],\n            \'position\': position, \'price\': price, \'selection_reason\': reason,\n            \'risk_note\': \'Model estimates; confirm team news.\', \'is_fallback\': False}))\n    if candidates:\n        return max(candidates, key=lambda candidate: (candidate[0], -candidate[1]))[2]\n    unavailable_data = not team_players or not any(int(player[\'id\']) in projections for player in team_players)\n    return {\'player_id\': None, \'player_name\': None, \'position\': None, \'price\': None,\n            \'selection_reason\': \'Monitor the leading nailed starter from this team\' if unavailable_data\n                                else \'No available player meets the minutes and starting-confidence checks\',\n            \'risk_note\': \'Player-level evidence unavailable.\' if unavailable_data else \'Wait for stronger team news.\',\n            \'is_fallback\': True}\n\n\ndef fixture_reason(row: Mapping[str, Any], gameweeks: Sequence[int], category: str) -> str:\n    fixtures = [fixture for week in row[\'weeks\'] if week[\'gameweek\'] in gameweeks\n                for fixture in week.get(\'fixtures\', [])]\n    rated = [fixture for fixture in fixtures if number(fixture.get(\'fdr\')) is not None\n             and 1 <= float(fixture[\'fdr\']) <= 5]\n    favourable = sum(float(fixture[\'fdr\']) <= 2.5 for fixture in rated)\n    difficult = sum(float(fixture[\'fdr\']) >= 3.5 for fixture in rated)\n    if category == \'target\':\n        return f"FDR {row[\'average_fdr\']:.2f}, {favourable} favourable matches"\n    return f"FDR {row[\'average_fdr\']:.2f}, {difficult} difficult matches"\n\n\ndef immediate_reason(row: Mapping[str, Any], next_gw: int) -> str:\n    fixtures = next((week.get(\'fixtures\', []) for week in row[\'weeks\'] if week[\'gameweek\'] == next_gw), [])\n    if not fixtures:\n        return f\'No published fixture in Gameweek {next_gw}; skip one-week investment unless the schedule changes.\'\n    parts = []\n    for fixture in fixtures:\n        venue = \'at home\' if fixture.get(\'home\') else \'away\'\n        rating = number(fixture.get(\'fdr\'))\n        parts.append(f"{fixture.get(\'opponent\', \'an opponent not yet named\')} {venue}"\n                     + (f\', FDR {rating:g}\' if rating is not None else \', with FDR unavailable\'))\n    return \'; then \'.join(parts) + \'.\'\n\n\ndef generate_narration(case: Mapping[str, Any], chip: str) -> list[dict[str, Any]]:\n    if chip==\'wildcard\' and case.get(\'analyst_version\'):\n        from wildcard_fixture_analysis import generate_segments\n        return generate_segments(case)\n    if chip not in (\'wildcard\', \'freehit\'):\n        raise ValueError(\'Unknown chip\')\n    gameweeks = case[\'gameweeks\']; start = gameweeks[0]; end = gameweeks[-1]\n    intro = f\'Fixture difficulty from Gameweek {start} to {end}. \'\n    intro += (\'Target sustained runs for the Wildcard squad.\' if chip == \'wildcard\' else\n              f\'Free Hit covers Gameweek {start} only. These six-week rankings provide context; judge the immediate fixture separately.\')\n    segments = [{\'team_id\': None, \'team_name\': None, \'category\': \'intro\', \'order\': 0, \'narration_text\': intro}]\n    for category, teams in ((\'target\', case[\'targets\']), (\'avoid\', case[\'avoids\'])):\n        for rank, row in enumerate(teams, 1):\n            name = row.get(\'name\') or row[\'club\']\n            if category == \'target\':\n                text = f\'Target rank {rank}: {name}. \'\n                if chip == \'wildcard\':\n                    text += fixture_reason(row, gameweeks, category) + \'. \'\n                else:\n                    text += immediate_reason(row, start) + \' \'\n                player = row[\'target_player\']\n                if player[\'player_name\']:\n                    text += f"Consider {player[\'player_name\']}: {player[\'selection_reason\']}."\n                else:\n                    text += player[\'selection_reason\'] + \'.\'\n                if chip == \'freehit\' and row[\'immediate_average_fdr\'] is not None and row[\'immediate_average_fdr\'] >= 3.5:\n                    text += \' The immediate matchup warrants caution.\'\n            else:\n                text = f\'Avoid new investment rank {rank}: {name}; \' + fixture_reason(row, gameweeks, category) + \'.\'\n                if chip == \'freehit\':\n                    text = f\'Avoid new investment rank {rank}: {name}. \' + immediate_reason(row, start)\n                    if row[\'immediate_average_fdr\'] is not None and row[\'immediate_average_fdr\'] <= 2.5:\n                        text += \' The immediate fixture is better; six-week caution alone does not rule out a Free Hit pick.\'\n            segments.append({\'team_id\': int(row[\'team_id\']), \'team_name\': name, \'row_id\': row[\'row_id\'],\n                \'category\': category, \'rank\': rank, \'order\': len(segments), \'narration_text\': text})\n    outro = (\'These fixture runs shape the Wildcard squad plan.\' if chip == \'wildcard\' else\n             \'Next, build the Free Hit squad around immediate fixtures and secure minutes.\')\n    segments.append({\'team_id\': None, \'team_name\': None, \'category\': \'transition\',\n                     \'order\': len(segments), \'narration_text\': outro})\n    for segment in segments:\n        segment.update(highlight_start=None, highlight_end=None, audio_start=None, audio_end=None)\n    return segments\n\n\ndef build_fixture_slide(rows: Sequence[Mapping[str, Any]], gameweeks: Sequence[int],\n                        players: Sequence[Mapping[str, Any]], context: Mapping[str, Any]) -> dict[str, Any]:\n    weeks = list(map(int, gameweeks))\n    if not weeks or len(weeks) > 6 or len(set(weeks)) != len(weeks) or weeks != sorted(weeks):\n        raise ValueError(\'Slide 2 requires up to six ordered, available Gameweeks\')\n    selected = select_teams(rows, weeks)\n    for row in selected[\'rows\']:\n        row[\'immediate_average_fdr\'] = average_fdr(row, weeks[:1])\n    for row in selected[\'targets\']:\n        row[\'target_player\'] = select_target_player(int(row[\'team_id\']), players, row[\'weeks\'],\n                                                    dict(context, gameweeks=weeks))\n    label = f"Fixture Difficulty: Next {len(weeks)} Gameweeks"\n    case = dict(selected, gameweeks=weeks, chip=context[\'chip\'], label=label,\n                range_label=f\'GW{weeks[0]}–GW{weeks[-1]}\', version=VERSION,\n                diagnostics={\'requested_window\': 6, \'available_window\': len(weeks),\n                             \'ranking\': \'Existing displayed FDR mean; each match counts; club tie-break unchanged\',\n                             \'warnings\': [] if len(weeks) == 6 else [\'Season end: fewer than six future Gameweeks remain.\']})\n    if context[\'chip\']==\'wildcard\':\n        from wildcard_fixture_analysis import enrich_fixture_case\n        return enrich_fixture_case(case,players,dict(context,gameweeks=weeks))\n    case[\'narration_timeline\'] = generate_narration(case, context[\'chip\'])\n    case[\'narration_text\'] = \' \'.join(segment[\'narration_text\'] for segment in case[\'narration_timeline\'])\n    return case\n\n\ndef align_timeline(segments: Sequence[Mapping[str, Any]], speech_seconds: float,\n                   segment_durations: Sequence[float] | None = None,\n                   spoken_text: str | None = None) -> list[dict[str, Any]]:\n    duration = number(speech_seconds)\n    if duration is None or duration <= 0:\n        raise ValueError(\'Measured speech duration must be positive\')\n    canonical = \' \'.join(segment[\'narration_text\'] for segment in segments)\n    if spoken_text is not None and \' \'.join(spoken_text.split()) != \' \'.join(canonical.split()):\n        raise ValueError(\'Slide 2 prose changed outside its structured timeline. Regenerate its narration timeline before rendering.\')\n    if segment_durations is not None:\n        durations = [number(value) for value in segment_durations]\n        if len(durations) != len(segments) or any(value is None or value <= 0 for value in durations):\n            raise ValueError(\'Provide one positive duration for every segment, including intro and transition\')\n        if abs(sum(durations) - duration) > .05:\n            raise ValueError(\'Segment durations must describe the actual concatenated speech track\')\n        alignment = \'measured_segments\'\n    else:\n        words = [max(1, len(segment[\'narration_text\'].split())) for segment in segments]\n        durations = [duration * count / sum(words) for count in words]\n        alignment = \'estimated_word_counts_from_measured_audio\'\n    result = copy.deepcopy(list(segments)); cursor = 0.\n    team_indices = [index for index, segment in enumerate(result) if segment.get(\'team_id\') is not None]\n    final_team = team_indices[-1] if team_indices else None\n    for index, (segment, seconds) in enumerate(zip(result, durations)):\n        segment.update(start=cursor,end=cursor+seconds,audio_start=cursor, audio_end=cursor + seconds, alignment=alignment)\n        if segment.get(\'team_id\') is not None:\n            segment.update(highlight_start=cursor, highlight_end=cursor + seconds + (.75 if index == final_team else 0.),\n                           fade_seconds=.3, hold_seconds=.75 if index == final_team else 0.)\n        cursor += seconds\n    return result\n\n\ndef bind_scene_timeline(scene: dict[str, Any], speech_seconds: float,\n                        segment_durations: Sequence[float] | None = None) -> None:\n    scene[\'narration_timeline\'] = align_timeline(scene[\'fixture_case\'][\'narration_timeline\'], speech_seconds,\n                                              segment_durations, scene[\'narration_text\'])\n    scene[\'timing\'][\'row_highlight_alignment\'] = scene[\'narration_timeline\'][0][\'alignment\']\n\n\ndef row_states(segments: Sequence[Mapping[str, Any]], seconds: float) -> dict[int, float]:\n    states = {}\n    for segment in segments:\n        start = segment.get(\'highlight_start\'); end = segment.get(\'highlight_end\')\n        if start is None or end is None or not start <= seconds < end:\n            continue\n        fade = min(.3, (end - start) / 2)\n        opacity = min(1., (seconds - start) / fade, (end - seconds) / fade)\n        states[int(segment[\'team_id\'])] = max(0., opacity)\n    return states\n\n\ndef row_geometry(rows: Sequence[Mapping[str, Any]]) -> dict[int, dict[str, Any]]:\n    return {int(row[\'team_id\']): {\'row_id\': row[\'row_id\'], \'data-team-id\': int(row[\'team_id\']),\n            \'box\': (48 + (index // 10) * 926 + 3, 158 + (index % 10) * 75,\n                    48 + (index // 10) * 926 + 901, 158 + (index % 10) * 75 + 71)}\n            for index, row in enumerate(rows)}\n'
_VX_FIXTURE_VISUAL_SOURCE = '"""Retain the existing PIL fixture table; add only explicit-time row outlines."""\nfrom __future__ import annotations\n\nfrom typing import Any\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageOps\n\nfrom fixture_slide import align_timeline, row_geometry, row_states\n\n\nclass FixtureSlideVisual:\n    def __init__(self, scene: dict[str, Any], contract: dict[str, Any], width: int, height: int,\n                 original_visual_type: type, renderer: Any, metadata: dict[str, Any],\n                 assets: dict[str, Any], media: dict[str, Any]):\n        self.manifest, self.contract = scene, contract\n        self.width, self.height = width, height\n        self.layout: list[dict[str, Any]] = []\n        self.renderer, self._broadcast_contract = renderer, metadata\n        case = scene[\'fixture_case\']\n        self.rows = row_geometry(case[\'rows\'])\n        table_contract = dict(contract, fixture_table=case[\'rows\'], horizon_gameweeks=case[\'gameweeks\'])\n        # Bypass patched constructors so the table is composited exactly once.\n        original = original_visual_type.__new__(original_visual_type)\n        original.manifest = dict(scene, visual_type=\'wc_analysis\')\n        original.contract = table_contract\n        original.width, original.height = width, height\n        original.sx, original.sy = width / 1920, height / 1080\n        original.assets, original.media = assets, media\n        original.layout = []\n        original.byid = {player[\'id\']: player for player in contract.get(\'squad\', [])}\n        original.entities = {int(key): player for key, player in contract.get(\'media_entities\', {}).items()}\n        foreground = Image.new(\'RGB\', (width, height), \'black\')\n        original.fixture_table(foreground)\n        original.text(foreground,(960,972),case[\'label\']+\'  /  \'+case[\'range_label\'],26, \'#FFD75A\',True,\'mm\',max_width=1800,minimum=22,condensed=True)\n        self.foreground = foreground\n        self.layout = original.layout\n        duration = float(scene.get(\'audio_duration_seconds\') or len(scene[\'narration_text\'].split()) / 145 * 60)\n        self.timeline = scene.get(\'narration_timeline\')\n        if not self.timeline or self.timeline[0].get(\'audio_start\') is None:\n            self.timeline = align_timeline(case[\'narration_timeline\'], duration, spoken_text=scene[\'narration_text\'])\n        self.complete = self.draw(0., duration)\n\n    def draw(self, seconds: float, duration: float) -> Image.Image:\n        image = self.foreground.copy()\n        states = row_states(self.timeline, seconds)\n        if states:\n            overlay = Image.new(\'RGBA\', image.size, (0, 0, 0, 0))\n            draw = ImageDraw.Draw(overlay)\n            scale_x, scale_y = self.width / 1920, self.height / 1080\n            for team_id, row in self.rows.items():\n                left, top, right, bottom = row[\'box\']\n                box = tuple(round(value * (scale_x if index % 2 == 0 else scale_y))\n                            for index, value in enumerate((left, top, right, bottom)))\n                opacity = states.get(team_id, 0.)\n                if opacity > 0:\n                    accent = (104, 245, 158, round(255 * opacity))\n                    draw.rounded_rectangle(box, radius=round(9 * scale_y), outline=accent,\n                                           width=max(1, round(3 * scale_y)))\n                    draw.rectangle((box[0], box[1] + round(9 * scale_y), box[0] + max(2, round(4 * scale_x)),\n                                    box[3] - round(9 * scale_y)), fill=accent)\n                else:\n                    # Only de-emphasise the club badge/name: leave FDR cells\' RGB and contrast intact.\n                    draw.rectangle((box[0], box[1], round((left + 174) * scale_x), box[3]), fill=(0, 0, 0, 18))\n            image = Image.alpha_composite(image.convert(\'RGBA\'), overlay).convert(\'RGB\')\n        output = self.renderer.composite(self, image)\n        # Recover only the existing header strip cropped by the shared legacy compositor.\n        top = round(image.height * 140 / 1080)\n        bottom = round(image.height * 1070 / 1080)\n        safe_width = round(3320 * self.width / 3840)\n        safe_height = round(1360 * self.height / 2160)\n        fitted = ImageOps.contain(image.crop((0,top,image.width,bottom)),(safe_width,safe_height),Image.Resampling.LANCZOS)\n        ratio = fitted.width / image.width\n        header_top = round(image.height * 90 / 1080)\n        header = image.crop((0,header_top,image.width,top)).convert(\'RGBA\')\n        pixels = np.asarray(header).copy()\n        pixels[...,3] = np.where(np.max(pixels[...,:3],axis=2)==0,0,255)\n        header = Image.fromarray(pixels).resize((fitted.width,max(1,round(header.height*ratio))),Image.Resampling.LANCZOS)\n        left = round(260*self.width/3840)+(safe_width-fitted.width)//2\n        content_top = round(490*self.height/2160)+(safe_height-fitted.height)//2\n        output.paste(header,(left,content_top-header.height),header)\n        return output\n\n    def frame(self, seconds: float, duration: float, previous: Any = None) -> np.ndarray:\n        return np.asarray(self.draw(seconds, duration))\n'
(_vx_module_dir/'wildcard_fixture_analysis.py').write_text('"""Wildcard Slide 2: shared FDR rankings, evidence-bound player comparisons and cues."""\nfrom __future__ import annotations\nimport copy\nimport html\nimport json\nimport math\nVERSION=\'wildcard-fixture-analyst-v1\'\nPOSITIONS={1:\'GK\',2:\'DEF\',3:\'MID\',4:\'FWD\'}\ndef number(value):\n    try:\n        x=float(value);return x if math.isfinite(x) else None\n    except (TypeError,ValueError):return None\n\ndef spoken_name(p):\n    first=str(p.get(\'first_name\',\'\')).strip();last=str(p.get(\'second_name\',\'\')).strip()\n    return first.split()[0]+\' \'+last if first and last else str(p.get(\'web_name\',\'Player name unavailable\'))\n\ndef role(p):\n    if number(p.get(\'penalties_order\'))==1:return \'first-choice penalty taker\'\n    if number(p.get(\'corners_and_indirect_freekicks_order\'))==1 or number(p.get(\'direct_freekicks_order\'))==1:return \'first-choice set-piece involvement\'\n    return \'no first-choice set-piece duty confirmed\'\n\ndef player_candidates(team_id,players,context):\n    models={int(m[\'player_id\']):m for m in context.get(\'player_models\',[]) if m.get(\'player_id\') is not None};out=[]\n    for p in players:\n        if int(p.get(\'team\',-1))!=team_id or p.get(\'status\')!=\'a\' or p.get(\'removed\') or not p.get(\'can_select\',True):continue\n        chance=number(p.get(\'chance_of_playing_next_round\'))\n        if chance is not None and chance<100:continue\n        m=models.get(int(p[\'id\']),{});minutes=number(m.get(\'expected_minutes\'));start=number(m.get(\'p_start\'));points=number(m.get(\'xpts_wildcard_horizon\'))\n        price=number(p.get(\'now_cost\'));price=price/float(context.get(\'currency_multiplier\') or 10) if price is not None else None\n        if minutes is None or start is None or points is None or price is None or price<=0 or minutes<65 or start<.8 or m.get(\'news_monitor\'):continue\n        if \'wc_eligible\' in m and not bool(m[\'wc_eligible\']):continue\n        xg=number(p.get(\'expected_goals_per_90\'));xa=number(p.get(\'expected_assists_per_90\'));xgi=number(p.get(\'expected_goal_involvements_per_90\'))\n        if xgi is None and xg is not None and xa is not None:xgi=xg+xa\n        out.append(dict(player_id=int(p[\'id\']),player_name=spoken_name(p),web_name=p.get(\'web_name\'),position=POSITIONS.get(int(p.get(\'element_type\',0)),\'UNKNOWN\'),price=price,form=number(p.get(\'form\')),xg90=xg,xa90=xa,xgi90=xgi,expected_minutes=minutes,start_probability=start,projected_points=points,value=points/price,role=role(p),risk=\'Minutes and points are estimates; check deadline team news.\'))\n    out.sort(key=lambda p:(-p[\'value\'],-p[\'projected_points\'],-p[\'expected_minutes\'],p[\'player_id\']))\n    return out\n\ndef team_statistics(team_id,fixtures):\n    matches=[f for f in fixtures if f.get(\'finished\') and team_id in (f.get(\'team_h\'),f.get(\'team_a\')) and number(f.get(\'team_h_score\')) is not None and number(f.get(\'team_a_score\')) is not None]\n    matches.sort(key=lambda f:(str(f.get(\'kickoff_time\') or \'\'),int(f.get(\'id\') or 0)));matches=matches[-6:]\n    if not matches:return None\n    goals=[];against=[]\n    for f in matches:\n        home=f[\'team_h\']==team_id;goals.append(number(f[\'team_h_score\' if home else \'team_a_score\']));against.append(number(f[\'team_a_score\' if home else \'team_h_score\']))\n    n=len(matches)\n    return dict(matches=n,goals_for=sum(goals),goals_against=sum(against),goals_for_per_match=sum(goals)/n,goals_against_per_match=sum(against)/n,clean_sheets=sum(x==0 for x in against),source=\'Official FPL completed league fixtures\')\n\ndef fixture_words(f):\n    return str(f.get(\'opponent\',\'an unnamed opponent\'))+(\' at home\' if f.get(\'home\') else \' away\')\n\ndef evidence(row,weeks):\n    fixtures=[dict(f,gameweek=w[\'gameweek\']) for w in row.get(\'weeks\',[]) if w.get(\'gameweek\') in weeks for f in w.get(\'fixtures\',[])]\n    rated=[f for f in fixtures if number(f.get(\'fdr\')) is not None and 1<=float(f[\'fdr\'])<=5]\n    good=[f for f in rated if float(f[\'fdr\'])<=2.5];hard=[f for f in rated if float(f[\'fdr\'])>=3.5]\n    return dict(fixtures=fixtures,favourable=len(good),difficult=len(hard),home=sum(f.get(\'home\') is True for f in fixtures),away=sum(f.get(\'home\') is False for f in fixtures),blanks=sum(not w.get(\'fixtures\') for w in row.get(\'weeks\',[]) if w.get(\'gameweek\') in weeks),double_weeks=sum(len(w.get(\'fixtures\',[]))>1 for w in row.get(\'weeks\',[]) if w.get(\'gameweek\') in weeks),key_fixtures=sorted(good,key=lambda f:(float(f[\'fdr\']),f[\'gameweek\']))[:2],hard_fixtures=sorted(hard,key=lambda f:(-float(f[\'fdr\']),f[\'gameweek\']))[:2],rated_count=len(rated),missing_count=len(fixtures)-len(rated))\n\ndef player_narration(p):\n    text=f"{p[\'player_name\']} costs £{p[\'price\']:.1f} million and projects {p[\'projected_points\']:.1f} points across the window. The base minutes estimate is {p[\'expected_minutes\']:.0f} per appearance, with a {p[\'start_probability\']:.0%} starting estimate."\n    if p[\'form\'] is not None:text+=f" Official FPL form is {p[\'form\']:.1f}."\n    if p[\'xgi90\'] is not None:text+=f" Season underlying involvement is {p[\'xgi90\']:.2f} expected goal involvements per ninety minutes."\n    if p[\'xg90\'] is not None and p[\'xa90\'] is not None:text+=f" That includes {p[\'xg90\']:.2f} expected goals and {p[\'xa90\']:.2f} expected assists per ninety."\n    if p[\'position\'] in (\'GK\',\'DEF\'):text+=\' Defensive returns also depend on clean sheets, which the fixture rank alone cannot promise.\'\n    text+=\' His role includes \'+p[\'role\']+\'.\'\n    return text\n\ndef enrich_fixture_case(case,players,context):\n    result=copy.deepcopy(case);weeks=result[\'gameweeks\'];models=context.get(\'player_models\',[])\n    result[\'analyst_version\']=VERSION\n    result[\'analysis_title\']=f"FPL VORTEX — SIX-GAMEWEEK FIXTURE ANALYSIS | GW {weeks[0]}–{weeks[-1]}" if len(weeks)==6 else f"FPL VORTEX — {len(weeks)}-GAMEWEEK FIXTURE ANALYSIS | GW {weeks[0]}–{weeks[-1]}"\n    byid={int(r[\'team_id\']):r for r in result[\'rows\']}\n    eligible_by_team={tid:player_candidates(tid,players,context) for tid in byid}\n    for category in (\'targets\',\'avoids\'):\n        for row in result[category]:\n            tid=int(row[\'team_id\']);row[\'fixture_evidence\']=evidence(row,weeks);row[\'team_statistics\']=team_statistics(tid,context.get(\'official_fixtures\',[]))\n            row[\'players\']=eligible_by_team[tid][:2] if category==\'targets\' else []\n            if category==\'targets\':\n                pairs=row[\'players\'];preferred=pairs[0] if pairs else None\n                row[\'preferred_player_id\']=preferred[\'player_id\'] if preferred else None\n                if len(pairs)==2:\n                    a,b=pairs\n                    row[\'comparison\']=f"Of these two, {a[\'player_name\']} leads on projected points per million: {a[\'value\']:.2f} against {b[\'value\']:.2f}."\n                    if a[\'position\']!=b[\'position\']:row[\'comparison\']+=\' They fill different positions, so this does not establish a like-for-like replacement.\'\n                    row[\'comparison\']+=\' Treat that as a shortlist preference. Final inclusion depends on the squad budget and club limit, rather than automatically buying both.\'\n                elif preferred:row[\'comparison\']=f"Only {preferred[\'player_name\']} passes the available minutes and availability checks. The remaining options need stronger minutes and availability evidence."\n                else:row[\'comparison\']=\'The available candidates do not meet the projection and minutes checks. Monitor this club until the evidence improves.\'\n                if preferred:\n                    peers=[dict(p,team_name=byid[other].get(\'name\') or byid[other][\'club\']) for other,candidates in eligible_by_team.items() if other!=tid for p in candidates if p[\'position\']==preferred[\'position\'] and abs(p[\'price\']-preferred[\'price\'])<=.5]\n                    if peers:\n                        peer=max(peers,key=lambda p:(p[\'projected_points\'],p[\'value\'],-p[\'player_id\']))\n                        row[\'price_peer\']=peer\n                        row[\'price_comparison\']=f"At a similar price in the same position, {peer[\'player_name\']} from {peer[\'team_name\']} projects {peer[\'projected_points\']:.1f} points for £{peer[\'price\']:.1f} million. Compare that with {preferred[\'player_name\']} at {preferred[\'projected_points\']:.1f} points for £{preferred[\'price\']:.1f} million. A favourable club fixture rank does not automatically make its player the best purchase."\n                    else:row[\'price_comparison\']=\'No qualifying same-position alternative within half a million pounds is available for a fair price comparison.\'\n            byid[tid].update(copy.deepcopy(row))\n    # Narrative avoid order is third-worst → second-worst → worst. Ranking data stays unchanged.\n    result[\'narrative_avoids\']=list(reversed(result[\'avoids\']))\n    all_players=[p for row in result[\'targets\'] for p in row[\'players\']]\n    result[\'best_value_player\']=max(all_players,key=lambda p:(p[\'value\'],-p[\'player_id\'])) if all_players else None\n    result[\'highest_projected_player\']=max(all_players,key=lambda p:(p[\'projected_points\'],-p[\'player_id\'])) if all_players else None\n    result[\'narration_timeline\']=generate_segments(result)\n    result[\'narration_text\']=\' \'.join(s[\'narration_text\'] for s in result[\'narration_timeline\'])\n    return result\n\ndef generate_segments(case):\n    out=[];weeks=case[\'gameweeks\'];window=\'six\' if len(weeks)==6 else str(len(weeks))\n    def add(identity,kind,text,row=None,player=None,rank=None):\n        out.append(dict(segment_id=identity,type=kind,category=kind,narration_text=text,order=len(out),team_id=int(row[\'team_id\']) if row else None,team_name=(row.get(\'name\') or row[\'club\']) if row else None,row_id=row.get(\'row_id\') if row else None,player_ids=[player[\'player_id\']] if player else [],player_id=player[\'player_id\'] if player else None,rank=rank,start=None,end=None,audio_start=None,audio_end=None,highlight_start=None,highlight_end=None))\n    add(\'fixture_intro\',\'intro\',f"Now let\'s break down the fixture landscape for the next {window} Gameweeks. Getting these fixture runs right can shape the Wildcard strategy. We\'re looking at {len(case[\'targets\'])} teams to target and {len(case[\'avoids\'])} where new investment looks less attractive. The ranking uses the same fixture difficulty values shown on screen. Lower averages indicate easier rated matches. Actual FPL returns remain uncertain.")\n    add(\'targets_heading\',\'heading\',f"First, the top {len(case[\'targets\'])} teams to target.")\n    for rank,row in enumerate(case[\'targets\'],1):\n        name=row.get(\'name\') or row[\'club\'];e=row[\'fixture_evidence\'];fdr=row[\'average_fdr\']\n        text=f"Number {rank}: {name}. The fixture average is {fdr:.2f}, with {e[\'favourable\']} favourable rated matchups across the next {window} Gameweeks."\n        if e[\'key_fixtures\']:text+=\' The key opportunities include \'+\' and \'.join(fixture_words(f) for f in e[\'key_fixtures\'])+\'.\'\n        text+=f" The schedule has {e[\'home\']} home matches and {e[\'away\']} away matches."\n        if e[\'blanks\']:text+=f" There are {e[\'blanks\']} Gameweeks without a published fixture."\n        if e[\'double_weeks\']:text+=f" There are {e[\'double_weeks\']} Gameweeks with more than one scheduled match."\n        if e[\'hard_fixtures\']:text+=\' The main caution is \'+\' and \'.join(fixture_words(f) for f in e[\'hard_fixtures\'])+\'.\'\n        if e[\'missing_count\']:text+=f" Difficulty ratings are missing for {e[\'missing_count\']} matches, so the average describes only the rated fixtures."\n        add(f\'target_{row["team_id"]}_fixtures\',\'target\',text,row,rank=rank)\n        stats=row[\'team_statistics\'];positions={p[\'position\'] for p in row[\'players\']}\n        if stats:\n            opportunities=[]\n            if positions & {\'MID\',\'FWD\'}:opportunities.append(f"scoring {stats[\'goals_for_per_match\']:.2f} goals per match")\n            if positions & {\'GK\',\'DEF\'}:opportunities.append(f"conceding {stats[\'goals_against_per_match\']:.2f} goals per match with {stats[\'clean_sheets\']} clean sheets")\n            detail=\' and \'.join(opportunities) if opportunities else f"scoring {stats[\'goals_for_per_match\']:.2f} and conceding {stats[\'goals_against_per_match\']:.2f} goals per match"\n            add(f\'target_{row["team_id"]}_team\',\'team_evidence\',f"Across their latest {stats[\'matches\']} completed league matches, {name} have been {detail}. Those are observed results, separate from the forward fixture ranking.",row,rank=rank)\n        else:add(f\'target_{row["team_id"]}_team\',\'team_evidence\',\'Recent team-level results are unavailable in this snapshot. The fixture ranking alone cannot establish attacking strength or clean-sheet probability.\',row,rank=rank)\n        for i,p in enumerate(row[\'players\']):add(f\'target_{row["team_id"]}_player_{p["player_id"]}\',\'player\',(\'The first player to consider is \' if i==0 else \'The alternative is \')+player_narration(p),row,p,rank)\n        add(f\'target_{row["team_id"]}_comparison\',\'comparison\',row[\'comparison\'],row,rank=rank)\n        if row.get(\'price_comparison\'):add(f\'target_{row["team_id"]}_value\',\'value_comparison\',row[\'price_comparison\'],row,rank=rank)\n    add(\'avoids_heading\',\'heading\',"Now let\'s look at the other side of the fixture ticker. These clubs are less attractive for new Wildcard investment.")\n    for i,row in enumerate(case[\'narrative_avoids\']):\n        n=len(case[\'narrative_avoids\'])-i;label=\'Worst fixture run\' if n==1 else \'Second-worst\' if n==2 else \'Third-worst\';name=row.get(\'name\') or row[\'club\'];e=row[\'fixture_evidence\']\n        text=f"{label}: {name}. Their fixture average is {row[\'average_fdr\']:.2f}, with {e[\'difficult\']} difficult rated matches."\n        if e[\'hard_fixtures\']:text+=\' The challenging opponents include \'+\' and \'.join(fixture_words(f) for f in e[\'hard_fixtures\'])+\'.\'\n        text+=f" The run contains {e[\'home\']} home matches and {e[\'away\']} away matches. This raises the opportunity cost of new investment, but it does not prove that every player will underperform."\n        if n==1:text+=" We wouldn\'t necessarily sell every player from this team. An elite asset with strong underlying numbers can still deliver. Compare individual players before moving money away."\n        add(\'avoid_\'+str(row[\'team_id\']),\'avoid\',text,row,rank=n)\n    best=[r.get(\'name\') or r[\'club\'] for r in case[\'targets\'][:3]];worst=[r.get(\'name\') or r[\'club\'] for r in case[\'avoids\'][:2]]\n    def natural(items):return items[0] if len(items)==1 else \' and \'.join(items) if len(items)==2 else \', \'.join(items[:-1])+\' and \'+items[-1] if items else \'no team until coverage improves\'\n    text=\'The FPL VORTEX fixture verdict. Prioritise reviewing \'+natural(best)+\'.\'\n    value=case.get(\'best_value_player\');upside=case.get(\'highest_projected_player\')\n    if value:text+=f" {value[\'player_name\']} has the strongest projected points per million among the named candidates."\n    if upside:text+=f" {upside[\'player_name\']} has the highest projected total among those candidates. That is a model projection, not a measured ceiling."\n    if worst:text+=\' Be cautious about new investment in \'+natural(worst)+\'.\'\n    text+=\' Fixtures create opportunities. Expected minutes, role, price and underlying statistics determine how useful those opportunities are to the squad.\'\n    add(\'fixture_verdict\',\'verdict\',text)\n    add(\'fixture_transition\',\'transition\',"Next, let\'s see how these fixture rankings influence our Wildcard squad selection.")\n    return out\n\ndef estimated_timeline(segments):\n    result=copy.deepcopy(segments);cursor=0\n    for s in result:\n        seconds=max(1.,len(s[\'narration_text\'].split())/145*60);s.update(start=cursor,end=cursor+seconds,audio_start=cursor,audio_end=cursor+seconds,alignment=\'preview_estimate\');cursor+=seconds\n    return result\n\nclass WildcardFixtureVisual:\n    def __init__(self,scene,contract,width,height,renderer,metadata,media=None,badges=None):\n        import numpy as np\n        self.manifest,self.contract=scene,contract;self.width,self.height=width,height;self.layout=[];self.renderer=renderer;self.metadata=metadata\n        timeline=scene.get(\'narration_timeline\') or scene[\'fixture_case\'][\'narration_timeline\']\n        if any(s.get(\'start\') is None for s in timeline):timeline=estimated_timeline(timeline)\n        self.timeline=timeline;self.content_html=render_html(scene[\'fixture_case\'],timeline,media=media,badges=badges)\n        self.complete=self.draw(timeline[-1][\'end\']+.65,timeline[-1][\'end\']+.65)\n    def draw(self,seconds,duration):return self.renderer.animation_image(self.manifest,self.metadata,self.width,self.height,self.content_html,seconds)\n    def frame(self,seconds,duration,previous=None):\n        import numpy as np\n        return np.asarray(self.draw(seconds,duration))\n\ndef render_html(case,timeline,media=None,badges=None):\n    from wildcard_squad_analysis import asset_uri\n    media=media or {};badges=badges or {}\n    esc=lambda value:html.escape(str(value),quote=True)\n    def club_image(row,cls=\'fa-club-badge\'):\n        record=badges.get(row.get(\'club\')) or {};uri=asset_uri(record.get(\'path\'))\n        return f\'<img class="{cls}" src="{uri}" alt="{esc(row.get("name") or row.get("club",""))}">\' if uri else \'\'\n    def player_image(p,cls=\'fa-player-photo\'):\n        record=media.get(str(p[\'player_id\'])) or media.get(p[\'player_id\']) or {}\n        uri=asset_uri(record.get(\'photo\')) if record.get(\'photo_kind\')==\'player\' else \'\'\n        return f\'<img class="{cls}" src="{uri}" alt="{esc(p["player_name"])}">\' if uri else \'\'\n    def club_chips(rows):\n        return \'<div class="fa-club-chips">\'+\'\'.join(\'<div>\'+club_image(r)+\'<b>\'+esc(r.get(\'name\') or r[\'club\'])+\'</b></div>\' for r in rows)+\'</div>\'\n    targets={int(r[\'team_id\']):i+1 for i,r in enumerate(case[\'targets\'])};avoids={int(r[\'team_id\']):i+1 for i,r in enumerate(case[\'avoids\'])}\n    def cell(week):\n        if not week or not week.get(\'fixtures\'):return \'<span class="fa-blank">—</span>\'\n        parts=[]\n        for f in week[\'fixtures\']:\n            rating=number(f.get(\'fdr\'));bg,fg=(\'#233445\',\'#ffffff\') if rating is None else (\'#66D58A\',\'#000000\') if rating<2.5 else (\'#72BCF0\',\'#000000\') if rating<3.0 else (\'#BCC5CC\',\'#000000\') if rating<4.0 else (\'#CE3748\',\'#ffffff\')\n            label=str(f.get(\'opponent\',\'?\'))+(\'[H]\' if f.get(\'home\') else \'[A]\')+\'-\'+(format(rating,\'.1f\').removesuffix(\'.0\') if rating is not None else \'—\')\n            parts.append(f\'<span style="background:{bg};color:{fg}" title="{esc(label)} · FDR {esc(rating)}">{esc(label)}</span>\')\n        return \'<div class="fa-fixtures">\'+\'\'.join(parts)+\'</div>\'\n    rows=[]\n    for row in case[\'rows\']:\n        tid=int(row[\'team_id\']);weeks={w[\'gameweek\']:w for w in row[\'weeks\']};badge=f\'T{targets[tid]}\' if tid in targets else f\'A{avoids[tid]}\' if tid in avoids else \'\'\n        rows.append(f\'<tr data-fa-team="{tid}" data-fa-category="{("target" if tid in targets else "avoid" if tid in avoids else "other")}"><th><i>{badge}</i>{club_image(row)}<span class="fa-club-name">{esc(row.get("name") or row["club"])}</span></th>\'+\'\'.join(\'<td>\'+cell(weeks.get(gw))+\'</td>\' for gw in case[\'gameweeks\'])+f\'<td class="fa-avg">{row["average_fdr"]:.2f}</td></tr>\' if row.get(\'average_fdr\') is not None else f\'<tr data-fa-team="{tid}"><th>{club_image(row)}<span class="fa-club-name">{esc(row.get("name") or row["club"])}</span></th>\'+\'\'.join(\'<td>\'+cell(weeks.get(gw))+\'</td>\' for gw in case[\'gameweeks\'])+\'<td>—</td></tr>\')\n    panels=[]\n    for category in (\'targets\',\'avoids\'):\n        for row in case[category]:\n            e=row[\'fixture_evidence\'];stat=row[\'team_statistics\'];cards=[]\n            for p in row.get(\'players\',[]):\n                metrics=[(\'PRICE\',f\'£{p["price"]:.1f}m\'),(str(len(case[\'gameweeks\']))+\'-GW xPTS\',f\'{p["projected_points"]:.1f}\'),(\'BASE MINS\',f\'{p["expected_minutes"]:.0f}\'),(\'START\',f\'{p["start_probability"]:.0%}\'),(\'xGI/90\',f\'{p["xgi90"]:.2f}\' if p[\'xgi90\'] is not None else \'—\'),(\'FORM\',f\'{p["form"]:.1f}\' if p[\'form\'] is not None else \'—\')]\n                cards.append(f\'<article class="fa-player" data-fa-player="{p["player_id"]}"><div class="fa-player-heading">{player_image(p)}<div><h3>{esc(p["player_name"])}</h3><small>{esc(p["position"])} · {esc(p["role"])}</small></div></div><div class="fa-metrics">\'+\'\'.join(f\'<div><small>{label}</small><b>{esc(value)}</b></div>\' for label,value in metrics)+f\'</div><p>Season xG/90: {esc(f\'{p["xg90"]:.2f}\' if p["xg90"] is not None else "—")} · xA/90: {esc(f\'{p["xa90"]:.2f}\' if p["xa90"] is not None else "—")}</p></article>\')\n            details=(f\'Last {stat["matches"]} completed league games · GF/game {stat["goals_for_per_match"]:.2f} · GA/game {stat["goals_against_per_match"]:.2f} · {stat["clean_sheets"]} clean sheets\' if stat else \'Recent team statistics unavailable · fixture rank is not a return forecast\')\n            advice=row.get(\'comparison\',\'New investment needs a stronger player-level case. Difficult fixtures alone do not mean sell every asset.\')\n            peer=row.get(\'price_peer\')\n            peer_text=(f\'<aside class="fa-price-peer">Same position · similar price: {esc(peer["player_name"])} ({esc(peer["team_name"])}) · £{peer["price"]:.1f}m · {peer["projected_points"]:.1f} projected points</aside>\' if peer else \'\')\n            panels.append(f\'<section class="fa-detail" data-fa-detail="{row["team_id"]}"><header><small>{("TARGET" if category=="targets" else "AVOID NEW INVESTMENT")}</small><div class="fa-team-heading">{club_image(row)}<h2>{esc(row.get("name") or row["club"])}</h2></div><p>{e["favourable"]} favourable · {e["difficult"]} difficult · {e["home"]} home / {e["away"]} away</p></header><div class="fa-team-stat">{esc(details)}</div><div class="fa-player-pair">\'+(\'\'.join(cards) if cards else \'<p class="fa-empty">\'+esc(\'No player meets the evidence checks.\' if category==\'targets\' else \'Assess elite assets individually. Avoiding new investment is not an automatic sell instruction.\')+\'</p>\')+f\'</div><aside class="fa-comparison">{esc(advice)}</aside>{peer_text}</section>\')\n    payload=json.dumps(timeline,ensure_ascii=False).replace(\'<\',\'\\\\u003c\')\n    best=\', \'.join(r.get(\'name\') or r[\'club\'] for r in case[\'targets\'][:3]);worst=\', \'.join(r.get(\'name\') or r[\'club\'] for r in case[\'avoids\'][:2])\n    value=case.get(\'best_value_player\');upside=case.get(\'highest_projected_player\')\n    standouts=\'\'.join(\'<article>\'+player_image(p,\'fa-standout-photo\')+\'<div><small>\'+label+\'</small><h3>\'+esc(p[\'player_name\'])+\'</h3><p>\'+format(p[\'projected_points\'],\'.1f\')+\' PROJECTED PTS · £\'+format(p[\'price\'],\'.1f\')+\'M</p></div></article>\' for label,p in [(\'BEST VALUE\',value),(\'HIGHEST PROJECTED TOTAL\',upside)] if p)\n    verdict=f\'<section class="fa-verdict"><small>FPL VORTEX FIXTURE VERDICT</small><h2>PRIORITY CLUBS</h2>{club_chips(case["targets"][:3])}<div class="fa-standouts">{standouts}</div><h3>CAUTION ON NEW INVESTMENT</h3>{club_chips(case["avoids"][:2])}<footer>CHECK MINUTES, ROLE, PRICE AND DEADLINE TEAM NEWS.</footer></section>\'\n\n    return f\'<style>{CSS}{MEDIA_CSS}</style><section class="fixture-analyst"><header class="fa-heading"><small>FIXTURE LANDSCAPE → PLAYER VALUE → SQUAD IMPLICATIONS</small><h1>{esc(case["analysis_title"])}</h1></header><main class="fa-main"><section class="fa-table-wrap"><table class="fa-table"><thead><tr><th>TEAM</th>\'+\'\'.join(f\'<th>GW {gw}</th>\' for gw in case[\'gameweeks\'])+\'<th>AVG</th></tr></thead><tbody>\'+\'\'.join(rows)+\'</tbody></table><footer>Same displayed FDR mean · every rated match counts · blanks excluded · T = target / A = avoid new investment</footer></section><aside class="fa-focus"><section class="fa-overview"><small>THE SIX-GAMEWEEK PLAN</small><h2>\'+str(len(case[\'targets\']))+\' clubs to target</h2><p>\'+str(len(case[\'avoids\']))+\' clubs to approach cautiously</p><p>Fixture ranks guide the shortlist. Player evidence decides the squad.</p><footer>Lower fixture difficulty averages indicate easier rated fixtures.</footer></section>\'+\'\'.join(panels)+verdict+\'</aside></main><footer class="fa-next">NEXT → WILDCARD SQUAD PLAN</footer><script type="application/json" class="fa-timeline">\'+payload+\'</script></section>\'\n\nCSS=\'\'\'\n.fixture-analyst{width:100%;height:100%;display:grid;grid-template-rows:140px 1150px 40px;gap:15px;color:#f4f7fc}.fa-heading small{font-size:24px;letter-spacing:4px;color:var(--accent-secondary)}.fa-heading h1{font-size:46px;line-height:1.1;margin:20px 0}.fa-main{display:grid;grid-template-columns:minmax(0,58fr) minmax(0,42fr);gap:18px;min-height:0}.fa-table-wrap{background:#071321ee;border:1px solid #506279;border-radius:15px;padding:14px;overflow:hidden}.fa-table{width:100%;height:1065px;table-layout:fixed;border-collapse:separate;border-spacing:0 4px;font-size:24px}.fa-table th:first-child{width:27%;text-align:left;white-space:nowrap;overflow:hidden}.fa-table th:last-child{width:6%}.fa-table th,.fa-table td{padding:3px 4px}.fa-table thead th{font-size:22px;color:var(--accent-secondary);height:38px}.fa-table tbody tr{background:#102337}.fa-table tbody th{font-size:25px;font-weight:700}.fa-table th i{display:inline-block;min-width:39px;font-size:19px;font-style:normal;color:var(--accent-secondary)}.fa-table tbody tr[data-fa-category="avoid"] th i{color:#ff9e9e}.fa-fixtures{display:flex;flex-direction:column;gap:2px}.fa-fixtures span{display:block;padding:6px 3px;border-radius:4px;text-align:center;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;font-size:20px;font-weight:800;line-height:1.1}.fa-avg{text-align:center;font-weight:800;font-size:24px}.fa-blank{display:block;text-align:center;color:#9cafc2}.fa-table-wrap>footer{font-size:20px;color:#c2d1e2;margin:12px 5px 0;line-height:1.25}.fa-table tr[data-active="true"] th{color:var(--accent-secondary);box-shadow:inset 5px 0 var(--accent-primary)}.fa-table tr[data-active="true"] td.fa-avg{color:var(--accent-secondary);box-shadow:inset -4px 0 var(--accent-primary)}\n.fa-focus{position:relative;min-width:0;overflow:hidden;margin-right:18px}.fa-overview,.fa-detail,.fa-verdict{position:absolute;inset:0;padding:30px;border:2px solid #54718f;border-radius:17px;background:linear-gradient(140deg,#10273bf5,#060d1cf5);visibility:hidden;opacity:0}.fa-overview small,.fa-verdict>small,.fa-detail header small{font-size:25px;letter-spacing:3px;color:var(--accent-secondary);font-weight:800}.fa-overview h2,.fa-verdict h2{font-size:58px;margin:80px 0 35px}.fa-overview p,.fa-verdict p{font-size:36px;line-height:1.35}.fa-overview footer,.fa-verdict footer{position:absolute;left:30px;right:30px;bottom:40px;font-size:27px;color:#bbcee0}.fa-detail header h2{font-size:49px;margin:15px 0}.fa-detail header p{font-size:27px;line-height:1.2;margin:10px 0 20px;color:#d2e3f2}.fa-team-stat{background:#172e42;border-left:4px solid var(--accent-primary);font-size:24px;line-height:1.3;padding:17px;margin-bottom:18px;min-height:86px}.fa-player-pair{display:flex;flex-direction:column;gap:15px}.fa-player{border:1px solid #4b6681;border-radius:12px;padding:18px 22px;background:#0a192a;opacity:.8}.fa-player h3{font-size:32px;margin:0 0 7px;color:#fff}.fa-player>small{font-size:21px;color:#c5d8e8;display:block;line-height:1.2}.fa-player p{font-size:20px;margin:10px 0 0;color:#c3d2e1}.fa-metrics{display:grid;grid-template-columns:repeat(3,1fr);gap:9px;margin-top:14px}.fa-metrics>div{background:#172e45;padding:8px 10px;border-radius:6px}.fa-metrics small{display:block;font-size:17px;color:#b7cbdc}.fa-metrics b{font-size:27px;display:block;margin-top:5px;color:var(--accent-secondary)}.fa-player[data-active="true"]{border-color:var(--accent-primary);box-shadow:0 0 20px #4acb8c33;opacity:1}.fa-comparison{font-size:24px;line-height:1.28;margin-top:18px;padding:18px;background:#16332c;border:1px solid #496e5c;border-radius:10px}.fa-price-peer{font-size:23px;line-height:1.25;margin-top:12px;padding:14px;background:#1b293e;border-left:4px solid var(--metallic-trim)}.fa-empty{font-size:34px;line-height:1.5;margin-top:70px;color:#d2e0ec}.fa-verdict h3{font-size:30px;margin-top:50px;color:var(--accent-secondary)}.fa-verdict h2{font-size:48px;margin-top:60px}.fa-next{font-size:26px;font-weight:800;letter-spacing:2px;color:var(--accent-secondary);opacity:0}\n\'\'\'\nSEEK_JS=\'\'\'\n<script>(function(){const previous=window.seekWildcard;window.seekWildcard=function(seconds){previous(seconds);const root=document.querySelector(\'.fixture-analyst\');if(!root)return;const timeline=JSON.parse(root.querySelector(\'.fa-timeline\').textContent);const current=timeline.find(s=>seconds>=s.start&&seconds<s.end)||timeline[timeline.length-1];const active=current.team_id;const verdict=current.type===\'verdict\'||current.type===\'transition\';for(const row of root.querySelectorAll(\'[data-fa-team]\'))row.dataset.active=String(Number(row.dataset.faTeam)===active);for(const panel of root.querySelectorAll(\'.fa-detail,.fa-overview,.fa-verdict\')){const visible=panel.classList.contains(\'fa-verdict\')?verdict:panel.classList.contains(\'fa-overview\')?!verdict&&active==null:Number(panel.dataset.faDetail)===active;panel.style.visibility=visible?\'visible\':\'hidden\';const first=timeline.find(s=>s.team_id===active);const onset=active!=null&&first?first.start:current.start;const progress=Math.max(0,Math.min(1,(seconds-onset)/.3));const eased=1-Math.pow(1-progress,3);panel.style.opacity=visible?String(eased):\'0\';panel.style.translate=visible?`${(1-eased)*20}px 0px`:\'0px 0px\';}for(const p of root.querySelectorAll(\'[data-fa-player]\')){const activePlayer=Number(p.dataset.faPlayer)===current.player_id;p.dataset.active=String(activePlayer);const progress=Math.max(0,Math.min(1,(seconds-current.start)/.25));p.style.scale=String(activePlayer?.99+.01*(1-Math.pow(1-progress,3)):1);}for(const el of root.querySelectorAll(\'.fa-comparison\'))el.style.opacity=current.type===\'comparison\'?\'1\':\'.72\';for(const el of root.querySelectorAll(\'.fa-price-peer\'))el.style.opacity=current.type===\'value_comparison\'?\'1\':\'.65\';root.querySelector(\'.fa-next\').style.opacity=current.type===\'transition\'?\'1\':\'0\';};})();</script>\n\'\'\'\n\nMEDIA_CSS = """\n#content .fixture-analyst{text-transform:uppercase;color:#fff}\n#content .fa-table tbody th{font-size:30px;color:#fff}\n#content .fa-table th i{min-width:30px;font-size:21px}\n#content .fa-club-badge{width:48px;height:48px;object-fit:contain;vertical-align:middle;margin:0 12px 0 4px}\n#content .fa-club-name{vertical-align:middle}\n#content .fa-fixtures span{font-size:27px;font-weight:900;padding:11px 2px;letter-spacing:-.4px}\n#content .fa-table thead th{font-size:28px;color:#dfa7ff}\n#content .fa-avg{font-size:28px;color:#fff}\n#content .fa-team-heading{display:flex;align-items:center;gap:22px;margin:12px 0}\n#content .fa-team-heading .fa-club-badge{width:105px;height:105px;margin:0}\n#content .fa-team-heading h2{font-size:63px;margin:0;color:#fff}\n#content .fa-player-heading{display:flex;align-items:center;gap:22px;min-height:125px}\n#content .fa-player-photo{width:115px;height:130px;object-fit:contain;flex-shrink:0}\n#content .fa-player-heading h3{font-size:43px;color:#fff;margin-bottom:12px}\n#content .fa-player-heading small{font-size:25px;color:#dfedff;line-height:1.25}\n#content .fa-player{opacity:1;padding:20px 24px}\n#content .fa-player p{color:#e3efff;font-size:26px}\n#content .fa-metrics small{color:#d9e9ff;font-size:23px}\n#content .fa-metrics b{font-size:39px;color:#e8bcff}\n#content .fa-team-stat{color:#fff;font-size:31px}\n#content .fa-comparison{color:#fff;font-size:28px}\n#content .fa-detail header p{color:#e6f1ff;font-size:31px}\n#content .fa-verdict h2{font-size:62px;margin:30px 0}\n#content .fa-club-chips{display:flex;gap:20px;align-items:stretch;margin:20px 0 35px}\n#content .fa-club-chips>div{display:flex;flex:1;min-width:0;align-items:center;gap:16px;padding:22px 14px;background:#172d43;border-radius:12px}\n#content .fa-club-chips .fa-club-badge{width:92px;height:100px;flex-shrink:0;margin:0}\n#content .fa-club-chips b{font-size:33px;line-height:1.2;color:#fff}\n#content .fa-standouts{display:grid;gap:24px;margin:35px 0}\n#content .fa-standouts article{display:flex;align-items:center;gap:32px;background:#11283c;border:1px solid #56708c;border-radius:14px;padding:20px 28px}\n#content .fa-standout-photo{width:185px;height:205px;object-fit:contain}\n#content .fa-standouts small{font-size:28px;color:#dfa7ff;font-weight:800}\n#content .fa-standouts h3{font-size:49px;margin:17px 0;color:#fff;line-height:1.1}\n#content .fa-standouts p{font-size:31px;color:#e5efff;margin:0}\n#content .fa-verdict>h3{font-size:33px;color:#dfa7ff;margin:30px 0 10px}\n#content .fa-verdict footer{font-size:27px;color:#d6e9ff}\n"""\n\nMEDIA_CSS += """\n#content .fa-player-heading{min-height:105px}\n#content .fa-player-photo{width:100px;height:110px}\n#content .fa-player{padding:14px 22px}\n#content .fa-team-stat{min-height:80px;padding:14px;margin-bottom:14px}\n#content .fa-metrics{margin-top:10px;gap:7px}\n#content .fa-metrics>div{padding:6px 10px}\n#content .fa-comparison{margin-top:12px;padding:15px}\n#content .fa-price-peer{font-size:26px;padding:12px;margin-top:10px}\n"""\n',encoding='utf-8')
sys.modules.pop('wildcard_fixture_analysis',None)
(_vx_module_dir/'fixture_slide.py').write_text(_VX_FIXTURE_SOURCE,encoding='utf-8')
(_vx_module_dir/'fixture_visual.py').write_text(_VX_FIXTURE_VISUAL_SOURCE,encoding='utf-8')
(_vx_module_dir/'freehit_intro.py').write_text(_VX_FREEHIT_INTRO_SOURCE,encoding='utf-8')
(_vx_module_dir/'wildcard_intro.py').write_text(_VX_INTRO_SOURCE,encoding='utf-8')
(_vx_module_dir/'wildcard_final_slide.py').write_text('"""Wildcard final slide: squad-bound six-week captaincy cards and concise blueprint."""\nfrom __future__ import annotations\nimport base64,copy,html,json,math\nfrom pathlib import Path\nfrom collections import Counter\nVERSION=\'wildcard-captaincy-v8-requested-script\'\nVOICE_SETTINGS=dict(voice=\'en-GB-RyanNeural\',rate=\'+0%\',pitch=\'+0Hz\',volume=\'+0%\')\n\ndef finite(value):\n    try:\n        value=float(value)\n        return value if math.isfinite(value) else None\n    except (ValueError,TypeError):return None\n\ndef club_name(code,fallback=\'\'):\n    from tts_narration import CLUBS\n    return CLUBS.get(code,fallback or \'Club unavailable\')\n\ndef _availability(p,canonical):\n    a=canonical.get(\'availability\') or p.get(\'availability\') or {}\n    if isinstance(a,str):a={\'label\':a}\n    code=a.get(\'code\',\'\');label=str(a.get(\'label\',\'\')).upper()\n    blocked=code in (\'i\',\'s\',\'u\',\'n\') or label in (\'INJURED\',\'SUSPENDED\',\'UNAVAILABLE\')\n    watch=blocked or a.get(\'flagged\',False) or code==\'d\' or not (code==\'a\' or label==\'AVAILABLE\')\n    return blocked,watch,a.get(\'news\',\'\')\n\ndef _fixtures(player,canonical,gw):\n    week=next((w for w in canonical.get(\'weeks\',[]) if int(w[\'gameweek\'])==gw),{})\n    out=[];seen=set()\n    for source in week.get(\'fixtures\',[]):\n        fid=source.get(\'fixture_id\')\n        if fid is None or fid in seen:raise ValueError(\'Verified unique fixture identifiers are required\')\n        seen.add(fid)\n        f=copy.deepcopy(source);code=f.get(\'opponent\',\'\');home=f.get(\'home\')\n        if home is None:raise ValueError(\'Fixture home/away context is missing\')\n        f[\'opponent_code\']=code;f[\'opponent_name\']=club_name(code,f.get(\'opponent_name\',code))\n        own=player[\'club_name\'];f[\'display\']=f\'{own} vs {f["opponent_name"]}\' if home else f\'{f["opponent_name"]} vs {own}\'\n        f[\'venue\']=\'Home\' if home else \'Away\'\n        # Preserve the model\'s existing per-player continuous difficulty and palette.\n        style=copy.deepcopy(f.get(\'fdr_style\') or {})\n        value=finite(f.get(\'fdr\'));f[\'difficulty\']=value\n        f[\'difficulty_label\']=str(style.get(\'label\',f\'{value:.1f}\' if value is not None else \'Unavailable\'))\n        f[\'difficulty_colour\']=style.get(\'background\',\'#536477\');f[\'difficulty_text\']=style.get(\'text\',\'#ffffff\')\n        f[\'difficulty_source\']=f.get(\'source\',\'Source unavailable\');out.append(f)\n    return out,week\n\ndef _reason(p,fixtures,watch=False):\n    if watch:return \'Leading available projection; review late fitness and minutes\'\n    if len(fixtures)>1:return \'Multiple verified fixtures and leading points estimate\'\n    f=fixtures[0];v=f[\'difficulty\'];home=f[\'home\']\n    if v is not None and v<=2.5:return \'Favourable home fixture and leading projection\' if home else \'Favourable matchup and leading points estimate\'\n    if home:return \'Home advantage and leading points estimate\'\n    if p.get(\'analysis\',{}).get(\'role\')==\'First-choice penalty taker\':return \'Penalty responsibility and leading points estimate\'\n    return \'Leading points estimate with supported minutes estimate\'\n\ndef from_contract(contract):\n    if contract.get(\'chip\')!=\'wildcard\':raise ValueError(\'Wildcard captaincy renderer accepts Wildcard only\')\n    p=contract[\'slide3_plan\'];selected={int(x[\'id\']):copy.deepcopy(x) for x in p[\'squad\']};canonical={int(x[\'id\']):x for x in contract[\'squad\']};rows=[]\n    weeks=p[\'gameweek_plans\']\n    if len(weeks)!=6 or [int(w[\'gameweek\']) for w in weeks]!=list(range(p[\'next_gw\'],p[\'end_gw\']+1)):raise ValueError(\'Six consecutive planned Gameweeks required\')\n    for w in weeks:\n        gw=int(w[\'gameweek\']);eligible=[]\n        for starter in w[\'starting_xi\']:\n            pid=int(starter[\'id\'])\n            if pid not in selected:raise ValueError(\'Captaincy starter is outside Slide 3 squad\')\n            x=copy.deepcopy(selected[pid]);c=canonical.get(pid,{})\n            # Captain eligibility follows the legal Slide 3 starting XI, not a position shortlist.\n            x[\'club_name\']=club_name(x.get(\'club\'),x.get(\'club_name\'));x[\'spoken_name\']=c.get(\'web_name\') or x.get(\'web_name\') or x.get(\'name\') or x.get(\'spoken_name\') or x.get(\'full_name\')\n            fs,cw=_fixtures(x,c,gw);blocked,watch,news=_availability(x,c)\n            xp=finite(starter.get(\'xpts\',cw.get(\'xpts\')))\n            if blocked or not fs or xp is None:continue\n            minutes=finite(cw.get(\'expected_minutes\'));prob=finite(x.get(\'starting_probability\'))\n            if minutes is None and all(finite(f.get(\'expected_minutes\')) is not None for f in fs):minutes=sum(float(f[\'expected_minutes\']) for f in fs)\n            if minutes is None:minutes=finite(x.get(\'minutes\'))\n            if prob is None or prob<.8 or minutes is None or minutes/len(fs)<60:watch=True\n            x.update(projected_points=xp,fixtures=fs,minutes=minutes,review_team_news=bool(watch),news=news)\n            eligible.append(x)\n        eligible.sort(key=lambda x:(-x[\'projected_points\'],x[\'id\']))\n        if len(eligible)<2:raise ValueError(f\'Gameweek {gw}: two projected starters with verified fixtures required\')\n        # The final slide must display, not independently override, Slide 3 captaincy.\n        chosen={x[\'id\']:x for x in eligible};cid=int(w[\'captain_id\']);vid=int(w[\'vice_captain_id\'])\n        if cid==vid or cid not in chosen or vid not in chosen:\n            raise ValueError(f\'Gameweek {gw}: planned captain or vice is unavailable, blank or unprojected; regenerate Slide 3\')\n        cap=chosen[cid];vice=chosen[vid]\n        if cap[\'projected_points\']+1e-8<eligible[0][\'projected_points\']:\n            raise ValueError(f\'Gameweek {gw}: captain does not lead the weekly starting-XI projection\')\n        reason=_reason(cap,cap[\'fixtures\'],cap[\'review_team_news\'])\n        # Only players in this legal XI with a verified fixture and xPts can enter the comparison.\n        contenders=[cap]+[player for player in eligible if player[\'id\']!=cap[\'id\']][:2]\n        runner_up=next(x for x in eligible if x[\'id\']!=cap[\'id\'])\n        rows.append(dict(gw=gw,captain=cap,vice_captain=vice,candidates=contenders,projection_gap=cap[\'projected_points\']-runner_up[\'projected_points\'],reason=reason,double_gameweek=len(cap[\'fixtures\'])>1,review_team_news=cap[\'review_team_news\'],captain_bonus_xpts=cap[\'projected_points\']))\n    counts=Counter(r[\'captain\'][\'id\'] for r in rows)\n    # Opening GW captain is the headline. The fixed squad\'s later captain choices remain in the six-week map.\n    anchor=rows[0][\'captain\'];alternative=rows[0][\'vice_captain\']\n    switches=[r[\'gw\'] for prev,r in zip(rows,rows[1:]) if prev[\'captain\'][\'id\']!=r[\'captain\'][\'id\']]\n    switch=switches[0] if switches else None\n    return dict(version=VERSION,chip=\'wildcard\',next_gw=p[\'next_gw\'],end_gw=p[\'end_gw\'],captaincy_plan=rows,anchor=anchor,anchor_id=anchor[\'id\'],anchor_count=counts[anchor[\'id\']],anchor_reason=f\'Highest projected scorer in the GW{p["next_gw"]} starting XI; one additional copy of {anchor["projected_points"]:.1f} xPts\',alternative=alternative,alternative_label=f\'GW{p["next_gw"]} vice · {alternative["projected_points"]:.1f} xPts\',alternative_reason=\'Receives captain bonus only if the captain makes no appearance\',switch_gameweek=switch,switch_weeks=switches,no_required_switch=not switches,source_snapshot_at=contract.get(\'source_snapshot_at\'),data_health={\'valid\':True,\'weeks\':6,\'projection_display\':\'current Gameweek and provisional six-week map\'})\n\ndef generate_wildcard_final_captaincy_narration(captaincy_plan,wildcard_squad=None):\n    plan=captaincy_plan;rows=plan[\'captaincy_plan\'];out=[]\n    numbers={1:\'one\',2:\'two\',3:\'three\',4:\'four\',5:\'five\',6:\'six\'}\n    def number(value):return numbers.get(value,str(value))\n    def join(items):\n        return items[0] if len(items)==1 else \' and \'.join(items) if len(items)==2 else \', \'.join(items[:-1])+\', and \'+items[-1]\n    def add(identifier,kind,text,**kwargs):\n        out.append(dict(segment_id=identifier,type=kind,narration_text=text,start=None,end=None,player_ids=kwargs.pop(\'player_ids\',[]),**kwargs))\n    def fixture(player):\n        fixtures=player[\'fixtures\']\n        if not fixtures:raise ValueError(\'Captain narration requires a verified fixture\')\n        return join([f"{f[\'opponent_name\']} {\'at home\' if f[\'home\'] else \'away\'}" for f in fixtures])\n    gaps=[float(r[\'projection_gap\']) for r in rows]\n    add(\'captain_intro\',\'intro\',f"Now, let\'s talk captaincy, because over {number(len(rows))} Gameweeks, getting these decisions right could make a huge difference.")\n    add(\'captain_approach\',\'intro\',"I\'ve compared the main options each week, and here\'s how I\'m planning to use the armband.")\n    for index,row in enumerate(rows):\n        gw=row[\'gw\'];contenders=row[\'candidates\'];captain=row[\'captain\'];vice=row[\'vice_captain\']\n        if not contenders or contenders[0][\'id\']!=captain[\'id\']:raise ValueError(\'Captain must lead the displayed comparison\')\n        name=captain[\'spoken_name\'];backup=vice[\'spoken_name\'];points=captain[\'projected_points\'];gap=float(row[\'projection_gap\'])\n        previous=rows[index-1][\'captain\'] if index else None\n        repeated=any(r[\'captain\'][\'id\']==captain[\'id\'] for r in rows[:index])\n        closest=index==len(rows)-1 and len(rows)>1 and gap<min(gaps[:-1])\n        if index==0:transition=f\'Starting with Gameweek {gw}.\'\n        elif index==len(rows)-1:\n            transition=f\'Finally, Gameweek {gw}.\'\n            if closest:transition+=\' This is probably the closest captaincy decision of the entire run.\'\n        elif index==1:\n            transition=f\'Moving into Gameweek {gw}\'+(\', things change.\' if previous[\'id\']!=captain[\'id\'] else \', the same player stays on top.\')\n        elif index==2:\n            transition=f\'Gameweek {gw} brings another opportunity for {name}.\' if repeated else f\'Gameweek {gw} brings an opportunity for {name}.\'\n        elif index==3:transition=f\'Then we reach Gameweek {gw}.\'\n        else:transition=f\'Gameweek {gw} takes us back to {captain["club_name"]}.\' if repeated else f\'Next is Gameweek {gw}.\'\n        add(f\'captain_compare_gw_{gw}\',\'compare\',transition,gw=gw,player_ids=[p[\'id\'] for p in contenders])\n        for rank,player in enumerate(contenders):\n            pname=player[\'spoken_name\'];xp=player[\'projected_points\']\n            if rank==0:\n                matchup=f"{captain[\'club_name\']} face {fixture(captain)}"\n                if index==0:\n                    favourite=\'the obvious favourite\' if gap>1 else \'the leading option\' if gap>0 else \'joint top on points\'\n                    text=f"{matchup}, and {name} is {favourite}. He\'s projected at {points:.1f} points"\n                elif index==1:text=f\'{matchup}, and {name} moves to the top with {points:.1f} projected points.\' if previous[\'id\']!=captain[\'id\'] else f\'{matchup}, and {name} stays at the top with {points:.1f} projected points.\'\n                elif index==2:text=f\'{matchup}, and he\\\'s projected at {points:.1f} points.\'\n                elif index==3:text=f\'{matchup}, and {name} leads the projections at {points:.1f} points.\'\n                elif index==len(rows)-1:text=f\'{name} faces {fixture(captain)} and projects {points:.1f} points.\'\n                else:text=f\'{name} faces {fixture(captain)} and leads the numbers with {points:.1f} projected points.\'\n                if index==0:text+=\', ahead of\' if len(contenders)>1 else \'.\'\n            elif index==0:\n                text=f\'{pname} on {xp:.1f}\'+(\' and\' if rank<len(contenders)-1 else \'.\')\n            elif index==1:\n                relation=\'close behind\' if points-xp<=.5 else \'next\'\n                text=f\'{pname} is {relation} on {xp:.1f}\'+(\', while\' if rank<len(contenders)-1 else \'.\') if rank==1 else f\'{pname} sits at {xp:.1f}.\'\n            elif index==2:text=f\'{pname} follows on {xp:.1f}\'+(\', with\' if rank<len(contenders)-1 else \'.\') if rank==1 else f\'{pname} at {xp:.1f}.\'\n            elif index==3:text=f\'{pname} is \'+(\'close behind\' if points-xp<=.5 else \'next\')+f\' on {xp:.1f}\'+(\', with\' if rank<len(contenders)-1 else \'.\') if rank==1 else f\'{pname} at {xp:.1f}.\'\n            elif index==len(rows)-1:text=f\'{pname} is \'+(\'right behind him\' if points-xp<=.5 else \'next\')+f\' on {xp:.1f}\'+(\', while\' if rank<len(contenders)-1 else \'.\') if rank==1 else f\'{pname} also appears among the leading projections at {xp:.1f}.\'\n            else:text=f\'{pname} is next on {xp:.1f}\'+(\', followed by\' if rank<len(contenders)-1 else \'.\') if rank==1 else f\'{pname} on {xp:.1f}.\'\n            add(f\'captain_option_gw_{gw}_{player["id"]}\',\'candidate\',text,gw=gw,rank=rank,player_id=player[\'id\'],player_ids=[player[\'id\']])\n        if gap<=1e-8:\n            verdict=f"The top projection is tied, and {name} is the captain in my current plan, with {backup} as vice-captain."\n        elif index==0:verdict=f\'With that fixture and the strongest projection, {name} gets my captaincy, with {backup} as vice-captain.\'\n        elif closest and gap<=.2:verdict=f"There\'s barely anything separating the top two, but {name} just edges it for me. So he gets the armband, with {backup} as vice-captain."\n        elif index==1 and gap<=.5:verdict=f"It\'s a tighter decision, but {name} gets the armband, with {backup} as backup."\n        elif index==2:verdict=f\'For me, {name} is the strongest option here, and {backup} takes the vice-captaincy.\'\n        elif index==3:verdict=f"It\'s {\'another \' if repeated else \'a \'}good opportunity for {name}, so he\'s my captain, with {backup} as vice."\n        else:verdict=f"I\'m going with {name}{\' again\' if repeated else \'\'}, and {backup} gets the vice-captaincy."\n        add(f\'captain_gw_{gw}\',\'week\',verdict,gw=gw,player_id=captain[\'id\'],player_ids=[captain[\'id\'],vice[\'id\']])\n    counts=Counter(r[\'captain\'][\'id\'] for r in rows)\n    names={r[\'captain\'][\'id\']:r[\'captain\'][\'spoken_name\'] for r in rows}\n    allocation=join([f"{number(count).capitalize() if index==0 else number(count)} {\'week\' if count==1 else \'weeks\'} with {names[pid]}" for index,(pid,count) in enumerate(counts.items())])+\'.\'\n    add(\'captain_switch\',\'switch\',f"And that\'s my {number(len(rows))}-Gameweek captaincy plan. {allocation}",weeks=plan[\'switch_weeks\'])\n    add(\'captain_deadline\',\'deadline\',"But remember, these are early projections, not decisions set in stone. Injuries, expected minutes and team news could change everything.")\n    add(\'captain_outro\',\'outro\',"The important thing is staying flexible and giving yourself the best chance of getting those captaincy calls right.")\n    return out\n\ndef generate_wildcard_final_slide_narration(plan,*args):return generate_wildcard_final_captaincy_narration(plan,*args)\n\ndef clean_narration_for_tts(original,bootstrap=None,player_ids=()):\n    from tts_narration import prepare_tts\n    audit=prepare_tts(original,bootstrap,player_ids)\n    if not audit[\'validation_passed\']:raise ValueError(\'Unresolved spoken shorthand: \'+str(audit[\'unresolved_tokens\']))\n    return audit\n\ndef align_final_timeline(segments,duration,segment_durations=None):\n    result=copy.deepcopy(segments);words=[len(s[\'narration_text\'].split()) for s in result]\n    times=list(segment_durations) if segment_durations is not None else [duration*w/sum(words) for w in words]\n    if len(times)!=len(result) or any(not math.isfinite(t) or t<=0 for t in times) or abs(sum(times)-duration)>.05:raise ValueError(\'Invalid final-slide timing\')\n    cursor=0.\n    for item,seconds in zip(result,times):item.update(start=cursor,end=cursor+seconds,alignment=\'measured_segments\' if segment_durations is not None else \'estimated_word_counts\');cursor+=seconds\n    return result\n\ndef asset_uri(value):\n    if not value:return \'\'\n    p=Path(value)\n    if not p.is_file():return \'\'\n    mime=\'image/png\' if p.suffix.lower()==\'.png\' else \'image/webp\' if p.suffix.lower()==\'.webp\' else \'image/jpeg\'\n    return \'data:\'+mime+\';base64,\'+base64.b64encode(p.read_bytes()).decode()\n\ndef render_final_html(plan, timeline, media=None, badges=None, approved_heroes=None):\n    escape = lambda value: html.escape(str(value), quote=True)\n    media = media or {}\n    badges = badges or {}\n    approved_heroes = approved_heroes or {}\n    weeks = plan[\'captaincy_plan\']\n    if len(weeks) != 6:\n        raise ValueError(\'Captaincy slide needs exactly six planned Gameweeks\')\n\n    def team_logo(player):\n        item = badges.get(player[\'club\'], {})\n        source = asset_uri(item.get(\'path\')) if isinstance(item, dict) else \'\'\n        if not source:\n            raise FileNotFoundError(f\'Verified team crest missing for {player["club"]}\')\n        return f\'<img class="wcx6-crest" src="{source}" alt="{escape(player["club_name"])} badge">\'\n\n    def photo(player):\n        item = media.get(str(player[\'id\']), media.get(player[\'id\'], {}))\n        source = asset_uri(item.get(\'photo\')) if item.get(\'photo_kind\') == \'player\' else \'\'\n        if not source:\n            raise FileNotFoundError(f\'Official player portrait missing for {player["id"]}\')\n        return f\'<img class="wcx6-photo" src="{source}" alt="{escape(player["spoken_name"])}">\'\n\n    def hero(player):\n        raw = approved_heroes.get(int(player[\'id\'])) or approved_heroes.get(str(player[\'id\']))\n        if raw is None:\n            raise ValueError(f\'Verified original hero image missing for {player["id"]}\')\n        if isinstance(raw, (str, Path)):\n            source = asset_uri(raw)\n        else:\n            import io\n            from PIL import Image\n            if not isinstance(raw, Image.Image):\n                raise TypeError(\'Hero must be the verified original notebook card\')\n            output = io.BytesIO()\n            raw.save(output, format=\'PNG\')\n            source = \'data:image/png;base64,\' + base64.b64encode(output.getvalue()).decode(\'ascii\')\n        if not source:\n            raise FileNotFoundError(f\'Original Hero Card missing for {player["id"]}\')\n        return f\'<img class="wcx6-hero-art" src="{source}" alt="Original FPL Vortex hero card for {escape(player["spoken_name"])}">\'\n\n    def fixture_summary(player):\n        games = player.get(\'fixtures\') or []\n        if not games:\n            raise ValueError(f\'Missing verified fixtures for {player["id"]}\')\n        return \' + \'.join(escape(match[\'opponent_name\']) + (\' [H]\' if match[\'home\'] else \' [A]\') for match in games)\n\n    week_html = []\n    for week_index, row in enumerate(weeks):\n        captain = row[\'captain\']\n        vice = row[\'vice_captain\']\n        gameweek = int(row[\'gw\'])\n        captain_id = int(captain[\'id\'])\n        if not (2 <= len(row[\'candidates\']) <= 3):\n            raise ValueError(f\'GW {gameweek}: two or three verified candidates required\')\n        candidates = []\n        hero_panels = []\n        for rank, player in enumerate(row[\'candidates\']):\n            player_id = int(player[\'id\'])\n            if not math.isfinite(float(player[\'projected_points\'])):\n                raise ValueError(f\'GW {gameweek}: invalid projected points for {player_id}\')\n            minutes = f\'{player["minutes"]:.0f} MIN\' if player.get(\'minutes\') is not None else \'MINUTES CHECK\'\n            news = \'TEAM NEWS CHECK\' if player.get(\'review_team_news\') else \'VERIFIED STARTING XI\'\n            candidates.append(\n                f\'<article class="wcx6-option" data-wcx6-option="{player_id}" data-wcx6-rank="{rank}">\'\n                f\'<div class="wcx6-rank">{rank+1:02d}</div>\'\n                f\'{photo(player)}\'\n                f\'<div class="wcx6-option-copy"><div class="wcx6-player-line"><strong>{escape(player["spoken_name"])}</strong>{team_logo(player)}\'\n                f\'<span>{escape(player["club_name"])}</span></div>\'\n                f\'<div class="wcx6-fixture">{fixture_summary(player)}</div>\'\n                f\'<div class="wcx6-player-note">{news}</div></div>\'\n                f\'<div class="wcx6-minutes"><small>PROJECTED TIME</small><b>{minutes}</b></div>\'\n                f\'<div class="wcx6-points"><b>{player["projected_points"]:.1f}</b><small>xPTS</small></div></article>\'\n            )\n            hero_panels.append(\n                f\'<div class="wcx6-hero-option" data-wcx6-hero="{player_id}">{hero(player)}</div>\'\n            )\n        week_html.append(\n            f\'<section class="wcx6-week" data-wcx6-week="{gameweek}">\'\n            f\'<section class="wcx6-compare"><div class="wcx6-panel-title"><div><small>GAMEWEEK {gameweek} · {len(row["candidates"])}-PLAYER DECISION</small>\'\n            f\'<h3>WHO GETS THE ARMBAND?</h3></div><b>{week_index+1} / 6</b></div>\'\n            f\'<div class="wcx6-option-grid" style="grid-template-rows:repeat({len(row["candidates"])},minmax(0,1fr))">{"".join(candidates)}</div>\'\n            f\'<div class="wcx6-verdict" data-wcx6-verdict><div class="wcx6-analysis" data-wcx6-preview>\'\n            f\'<span>SCOUTING REPORT · BEFORE THE ARMBAND REVEAL</span>\'\n            f\'<strong data-wcx6-analysis>FIXTURES · PROJECTED POINTS · EXPECTED MINUTES</strong></div>\'\n            f\'<div class="wcx6-decision" data-wcx6-final><span>VORTEX PICK · GW {gameweek}</span>\'\n            f\'<strong>{escape(captain["spoken_name"])} <em>+{row["projection_gap"]:.1f} xPTS LEAD</em></strong>\'\n            f\'<p>{escape(row["reason"])} · VICE: {escape(vice["spoken_name"])}</p></div></div></section>\'\n            f\'<aside class="wcx6-hero-panel"><div class="wcx6-hero-title">OFFICIAL PLAYER HERO CARD</div>\'\n            f\'<div class="wcx6-hero-display">{"".join(hero_panels)}</div>\'\n            f\'<div class="wcx6-projection"><span>MY CALCULATION · BEFORE DOUBLING</span>\'\n            f\'<strong data-wcx6-points>{captain["projected_points"]:.1f} <i>xPTS</i></strong></div></aside></section>\'\n        )\n\n    summary_items = []\n    for row in weeks:\n        captain = row[\'captain\']\n        vice = row[\'vice_captain\']\n        summary_items.append(\n            f\'<article class="wcx6-summary-item"><div class="wcx6-summary-gw">GW {row["gw"]}</div>\'\n            f\'{photo(captain)}<div class="wcx6-summary-name"><b>{escape(captain["spoken_name"])}</b>\'\n            f\'<span>VICE · {escape(vice["spoken_name"])}</span></div>\'\n            f\'<strong>{captain["projected_points"]:.1f} <small>xPTS</small></strong></article>\'\n        )\n    anchor = weeks[0][\'captain\']\n    summary = (\n        f\'<section class="wcx6-summary" data-wcx6-summary><div class="wcx6-summary-list">\'\n        f\'<div class="wcx6-panel-title"><div><small>FPL VORTEX · WILDCARD</small><h3>YOUR SIX CAPTAIN PICKS</h3></div><b>6 / 6</b></div>\'\n        f\'<div class="wcx6-summary-grid">{"".join(summary_items)}</div>\'\n        f\'<div class="wcx6-summary-caution">CHECK LINEUPS, MINUTES & TEAM NEWS BEFORE EVERY DEADLINE</div></div>\'\n        f\'<aside class="wcx6-hero-panel"><div class="wcx6-hero-title">GAMEWEEK {weeks[0]["gw"]} · FIRST CAPTAIN</div>\'\n        f\'<div class="wcx6-hero-display wcx6-summary-art">{hero(anchor)}</div>\'\n        f\'<div class="wcx6-projection"><span>MY FIRST-WEEK CALCULATION</span><strong>{anchor["projected_points"]:.1f} <i>xPTS</i></strong></div>\'\n        f\'</aside></section>\'\n    )\n    payload = json.dumps({\n        \'timeline\': timeline,\n        \'weeks\': [int(row[\'gw\']) for row in weeks],\n        \'captains\': {str(row[\'gw\']): int(row[\'captain\'][\'id\']) for row in weeks},\n        \'projections\': {str(row[\'gw\']): {str(player[\'id\']): float(player[\'projected_points\']) for player in row[\'candidates\']} for row in weeks},\n        \'version\': VERSION,\n    }, ensure_ascii=False).replace(\'<\', \'\\\\u003c\')\n    return (\n        f\'<style>{CSS}</style><section class="wildcard-captaincy-slide wcx-slide wcx-v6">\'\n        f\'<header class="wcx6-banner"><div><span>FPL VORTEX · WILDCARD</span><h2>SIX-WEEK ARMBAND MAP</h2></div>\'\n        f\'<b>6 GAMEWEEKS · TOP CONTENDERS · 1 CAPTAIN</b></header>\'\n        f\'<div class="wcx6-content">{"".join(week_html)}{summary}</div>\'\n        f\'<footer class="wcx6-footer"><b>ARMBAND INTELLIGENCE · MY CALCULATIONS</b>\'\n        f\'<span>PROJECTIONS BEFORE CAPTAIN DOUBLING · SUBJECT TO TEAM NEWS</span></footer>\'\n        f\'<script type="application/json" class="wcx6-data">{payload}</script></section>\'\n    )\n\n\nCSS=\'#content .wcx-v6{height:100%;width:100%;display:grid;grid-template-rows:136px minmax(0,1fr) 66px;gap:16px;overflow:hidden;color:#fff;font-family:Arial,Helvetica,sans-serif;text-transform:uppercase;letter-spacing:.1px;isolation:isolate}\\n#content .wcx-v6 *{box-sizing:border-box;min-width:0}\\n#content .wcx-v6 h2,#content .wcx-v6 h3,#content .wcx-v6 p{margin:0;padding:0}\\n#content .wcx6-banner{display:flex;align-items:center;justify-content:space-between;padding:15px 35px;border:3px solid #70b5b2;border-left:12px solid #f9dc88;border-radius:20px;background:linear-gradient(100deg,#092f43f7,#063447f0 68%,#11343cf3);box-shadow:inset 0 2px 0 #81fff533,0 10px 28px #0007}\\n#content .wcx6-banner>div{display:flex;flex-direction:column;gap:4px}\\n#content .wcx6-banner span{font-size:32px;font-weight:800;color:#a8f2dd;letter-spacing:4px}\\n#content .wcx6-banner h2{font-size:76px;font-weight:900;line-height:1.02;color:#ffe6a3;font-style:italic;letter-spacing:0;text-shadow:0 5px 0 #001a2b,0 0 28px #ffda7766}\\n#content .wcx6-banner>b{font-size:35px;letter-spacing:1px;color:#f1f7f6;text-align:right}\\n#content .wcx6-content{position:relative;min-height:0;overflow:hidden}\\n#content .wcx6-week,#content .wcx6-summary{position:absolute;inset:0;display:grid;grid-template-columns:minmax(0,69fr) minmax(0,31fr);gap:22px;visibility:hidden;opacity:0}\\n#content .wcx6-compare,#content .wcx6-summary-list{height:100%;display:grid;grid-template-rows:117px minmax(0,1fr) 166px;gap:13px;padding:16px 19px;background:linear-gradient(140deg,#092438f8,#0e3545f5);border:3px solid #6da3a1;border-radius:23px;box-shadow:inset 0 0 42px #73d9be14;overflow:hidden}\\n#content .wcx6-panel-title{display:flex;justify-content:space-between;align-items:center;padding:8px 23px 14px;border-bottom:2px solid #5d9e9b}\\n#content .wcx6-panel-title small{display:block;font-size:30px;color:#c7ead7;font-weight:900;letter-spacing:3px}\\n#content .wcx6-panel-title h3{margin-top:6px;font-size:61px;line-height:1.02;color:#fff;font-weight:900}\\n#content .wcx6-panel-title>b{font-size:43px;color:#ffe6a7}\\n#content .wcx6-option-grid{display:grid;grid-template-rows:repeat(3,minmax(0,1fr));gap:14px;min-height:0}\\n#content .wcx6-option{display:grid;grid-template-columns:88px 132px minmax(0,1fr) 235px 186px;align-items:center;gap:15px;padding:9px 19px;border:3px solid #436d7b;background:linear-gradient(110deg,#102f42,#133c48);border-radius:19px;box-shadow:inset 0 0 30px #81ddc910;opacity:.76;overflow:hidden;will-change:transform,opacity}\\n#content .wcx6-option.wcx6-current{opacity:1!important;border-color:#ffe296;background:linear-gradient(110deg,#145248,#154c53 50%,#103447);box-shadow:inset 0 0 30px #ffd37324,0 0 26px #79e8d42b}\\n#content .wcx6-option.wcx6-picked{border-color:#c6f2b0;background:linear-gradient(110deg,#15564d,#113e42)}\\n#content .wcx6-rank{font-size:54px;font-weight:900;color:#ffe7a1;text-align:center}\\n#content .wcx6-photo{width:132px;height:160px;object-fit:contain;object-position:center bottom;filter:drop-shadow(0 7px 11px #0009)}\\n#content .wcx6-option-copy{display:flex;flex-direction:column;gap:14px;overflow:hidden}\\n#content .wcx6-player-line{display:flex;align-items:center;gap:16px;flex-wrap:wrap}\\n#content .wcx6-player-line strong{font-size:62px;line-height:1.02;color:#fff;letter-spacing:0}\\n#content .wcx6-crest{width:65px;height:65px;object-fit:contain}\\n#content .wcx6-player-line span{font-size:35px;color:#a9dfdf;font-weight:800}\\n#content .wcx6-fixture{font-size:45px;font-weight:800;line-height:1.13;color:#f3e7c3;text-transform:none}\\n#content .wcx6-player-note{font-size:30px;font-weight:900;letter-spacing:1px;color:#b0f5d0}\\n#content .wcx6-minutes{display:flex;flex-direction:column;gap:12px;border-left:2px solid #497a88;padding-left:20px}\\n#content .wcx6-minutes small{font-size:26px;color:#acd5e0;font-weight:900}\\n#content .wcx6-minutes b{font-size:38px;color:#fff;white-space:nowrap}\\n#content .wcx6-points{display:flex;flex-direction:column;align-items:center;gap:5px;border-left:2px solid #497a88;padding:0 8px}\\n#content .wcx6-points b{font-size:91px;line-height:1;color:#ffde80;text-shadow:0 0 24px #ffdd7360}\\n#content .wcx6-points small{font-size:29px;font-weight:900;color:#c5eeef}\\n#content .wcx6-verdict{display:grid;grid-template-columns:minmax(0,1fr);align-content:center;gap:8px;padding:13px 32px;border:3px solid #b8b079;border-radius:17px;background:linear-gradient(100deg,#153e40,#0a283a);overflow:hidden}\\n#content .wcx6-verdict>span{font-size:30px;letter-spacing:3px;color:#ffe498;font-weight:900}\\n#content .wcx6-verdict>strong{display:flex;align-items:baseline;gap:28px;font-size:56px;color:#fff;line-height:1.02}\\n#content .wcx6-verdict em{font-size:42px;color:#ffdf73;font-style:normal;white-space:nowrap}\\n#content .wcx6-verdict>p{font-size:29px;color:#bfe5dd;line-height:1.1;text-transform:none;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}\\n#content .wcx6-hero-panel{display:grid;grid-template-rows:58px minmax(0,1fr) 112px;gap:9px;padding:11px 10px;background:linear-gradient(160deg,#082b3af8,#0e3937f9 72%,#0a2738f8);border:3px solid #72bcb0;border-radius:22px;min-height:0;box-shadow:inset 0 0 38px #8af2e014}\\n#content .wcx6-hero-title{display:flex;align-items:center;justify-content:center;font-size:37px;color:#ffe7ab;font-weight:900;border-bottom:2px solid #527f7f;text-align:center;letter-spacing:1px}\\n#content .wcx6-hero-display{position:relative;min-height:0;display:flex;align-items:center;justify-content:center;overflow:hidden;border-radius:17px}\\n#content .wcx6-hero-option{position:absolute;inset:0;display:flex;align-items:center;justify-content:center;visibility:hidden;opacity:0;will-change:transform,opacity}\\n#content .wcx6-hero-art{height:100%;max-height:100%;width:100%;max-width:100%;object-fit:contain;object-position:center;filter:drop-shadow(0 10px 22px #000b)}\\n#content .wcx6-projection{display:flex;flex-direction:column;justify-content:center;align-items:center;gap:4px;border:3px solid #f0cb75;background:linear-gradient(110deg,#153e34,#092b2d);border-radius:15px;text-align:center}\\n#content .wcx6-projection span{font-size:24px;font-weight:900;color:#bce7e1;letter-spacing:1px}\\n#content .wcx6-projection strong{font-size:75px;line-height:1.02;color:#ffe15b;font-weight:900;text-shadow:0 0 28px #ffdc5b55}\\n#content .wcx6-projection i{font-size:43px;font-style:normal;color:#ffe88e}\\n#content .wcx6-footer{display:flex;align-items:center;justify-content:space-between;padding:8px 23px;background:#09283af5;border-top:3px solid #87c8af;border-radius:10px}\\n#content .wcx6-footer b{font-size:29px;color:#ffe2a0;letter-spacing:2px}\\n#content .wcx6-footer span{font-size:28px;color:#b7e4e3}\\n#content .wcx6-summary-list{grid-template-rows:117px minmax(0,1fr) 112px}\\n#content .wcx6-summary-grid{display:grid;grid-template-rows:repeat(6,minmax(0,1fr));gap:8px;min-height:0}\\n#content .wcx6-summary-item{display:grid;grid-template-columns:155px 115px minmax(0,1fr) 260px;align-items:center;gap:22px;padding:7px 20px;border:2px solid #427888;border-radius:14px;background:linear-gradient(100deg,#123d49,#0b2d40)}\\n#content .wcx6-summary-gw{font-size:52px;font-weight:900;color:#ffe3a2}\\n#content .wcx6-summary-item .wcx6-photo{width:102px;height:117px}\\n#content .wcx6-summary-name{display:flex;flex-direction:column;gap:10px}\\n#content .wcx6-summary-name b{font-size:50px;line-height:1;color:#fff}\\n#content .wcx6-summary-name span{font-size:30px;color:#bce6df}\\n#content .wcx6-summary-item>strong{font-size:57px;color:#ffe1a0;text-align:right;white-space:nowrap}\\n#content .wcx6-summary-item>strong small{font-size:29px}\\n#content .wcx6-summary-caution{display:flex;align-items:center;justify-content:center;background:#114844;border:2px solid #c1dda4;border-radius:13px;color:#f6e4a3;font-size:38px;font-weight:900;letter-spacing:1px}\\n#content .wcx6-summary-art>.wcx6-hero-art{height:100%;width:100%;object-fit:contain}\\n\\n#content .wcx6-analysis,#content .wcx6-decision{display:flex;flex-direction:column;justify-content:center;gap:9px;min-height:0;overflow:hidden}\\n#content .wcx6-analysis span,#content .wcx6-decision span{color:#ffe498;font-weight:900;font-size:30px;letter-spacing:2px}\\n#content .wcx6-analysis strong{font-size:43px;color:#fff;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}\\n#content .wcx6-decision strong{display:flex;align-items:baseline;gap:25px;font-size:55px;color:#fff}\\n#content .wcx6-decision em{font-size:42px;color:#ffe09c;font-style:normal}\\n#content .wcx6-decision p{font-size:29px;text-transform:none;color:#c2e8e0;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}\\n\'\n\nSEEK_JS=\'<script>(function(){\\n  const previous=window.seekWildcard;\\n  window.seekWildcard=function(seconds){\\n    if(typeof previous===\\\'function\\\')previous(seconds);\\n    const root=document.querySelector(\\\'.wcx-v6\\\');\\n    if(!root)return;\\n    const json=root.querySelector(\\\'.wcx6-data\\\');\\n    if(!json)return;\\n    const data=JSON.parse(json.textContent);\\n    const timeline=data.timeline;\\n    if(!Array.isArray(timeline)||!timeline.length)return;\\n    const first=timeline[0],last=timeline[timeline.length-1];\\n    const cue=timeline.find(item=>seconds>=item.start&&seconds<item.end)||(seconds<first.start?first:last);\\n    const clamp=value=>Math.min(1,Math.max(0,value));\\n    const ease=value=>1-Math.pow(1-clamp(value),3);\\n    const local=clamp((seconds-cue.start)/Math.max(.001,cue.end-cue.start));\\n    const final=[\\\'switch\\\',\\\'deadline\\\',\\\'outro\\\'].includes(cue.type)||seconds>=last.end;\\n    const currentWeek=String(cue.gw||data.weeks[0]);\\n    for(const week of root.querySelectorAll(\\\'[data-wcx6-week]\\\')){\\n      const active=!final&&week.dataset.wcx6Week===currentWeek;\\n      week.style.visibility=active?\\\'visible\\\':\\\'hidden\\\';\\n      week.style.opacity=active?String(Math.max(.2,ease(local/.15))):\\\'0\\\';\\n      if(!active)continue;\\n      const captId=data.captains[currentWeek];\\n      const rank=Number(cue.rank);\\n      const selected=cue.type===\\\'candidate\\\'?String(cue.player_id):\\n        cue.type===\\\'week\\\'?String(captId):\\n        week.querySelector(\\\'[data-wcx6-option]\\\')?.dataset.wcx6Option;\\n      for(const option of week.querySelectorAll(\\\'[data-wcx6-option]\\\')){\\n        const isCurrent=cue.type===\\\'candidate\\\'&&option.dataset.wcx6Option===selected;\\n        const isCaptain=cue.type===\\\'week\\\'&&option.dataset.wcx6Option===String(captId);\\n        option.classList.toggle(\\\'wcx6-current\\\',isCurrent);\\n        option.classList.toggle(\\\'wcx6-picked\\\',isCaptain);\\n        option.style.opacity=isCurrent||isCaptain?\\\'1\\\':\\\'.86\\\';\\n        option.style.transform=isCurrent||isCaptain?\\\'translateX(7px)\\\':\\\'translateX(0)\\\';\\n        option.style.boxShadow=isCurrent?\\\'0 0 29px #ffdb7566\\\':\\\'\\\';\\n      }\\n      for(const card of week.querySelectorAll(\\\'[data-wcx6-hero]\\\')){\\n        const activeCard=card.dataset.wcx6Hero===selected;\\n        const progress=Math.max(.25,ease(local/.22));\\n        card.style.visibility=activeCard?\\\'visible\\\':\\\'hidden\\\';\\n        card.style.opacity=activeCard?String(progress):\\\'0\\\';\\n        card.style.transform=activeCard?`translateY(${Math.round((1-progress)*21)}px) scale(${(.955+.045*progress).toFixed(3)})`:\\\'none\\\';\\n      }\\n      const points=week.querySelector(\\\'[data-wcx6-points]\\\');\\n      if(points){\\n        const expected=data.projections[currentWeek]?.[selected];\\n        if(Number.isFinite(expected))points.innerHTML=`${expected.toFixed(1)} <i>xPTS</i>`;\\n      }\\n      const verdict=week.querySelector(\\\'[data-wcx6-verdict]\\\');\\n      if(verdict){\\n        const verdictVisible=cue.type===\\\'week\\\';\\n        verdict.style.borderColor=verdictVisible?\\\'#ffe3a2\\\':\\\'#467b80\\\';\\n        verdict.querySelector(\\\'[data-wcx6-final]\\\').style.display=verdictVisible?\\\'flex\\\':\\\'none\\\';\\n        verdict.querySelector(\\\'[data-wcx6-preview]\\\').style.display=verdictVisible?\\\'none\\\':\\\'flex\\\';\\n        const row=week.querySelector(`[data-wcx6-option="${selected}"]`);\\n        const text=verdict.querySelector(\\\'[data-wcx6-analysis]\\\');\\n        if(row&&text){\\n          const player=row.querySelector(\\\'.wcx6-player-line strong\\\')?.textContent||\\\'\\\';\\n          const fixture=row.querySelector(\\\'.wcx6-fixture\\\')?.textContent||\\\'\\\';\\n          const model=data.projections[currentWeek]?.[selected];\\n          text.textContent=`${player} · ${fixture} · ${Number(model).toFixed(1)} xPTS`;\\n        }\\n        verdict.style.opacity=\\\'1\\\';\\n      }\\n    }\\n    const summary=root.querySelector(\\\'[data-wcx6-summary]\\\');\\n    if(summary){summary.style.visibility=final?\\\'visible\\\':\\\'hidden\\\';summary.style.opacity=final?String(Math.max(.2,ease(local/.23))):\\\'0\\\';}\\n  };\\n})();</script>\\n\'\n\nclass WildcardCaptaincyVisual:\n    """Deterministic frames using the measured narration segment clock."""\n    def __init__(self,scene,contract,width,height,renderer,metadata,media,badges,approved_heroes=None):\n        self.manifest,self.contract,self.width,self.height=scene,contract,width,height\n        self.renderer,self.metadata=renderer,metadata;self.layout=[]\n        self.timeline=scene[\'narration_timeline\']\n        if self.timeline[0].get(\'start\') is None:\n            self.timeline=align_final_timeline(self.timeline,float(scene.get(\'audio_duration_seconds\') or len(scene[\'narration_text\'].split())/145*60))\n        self.content_html=render_final_html(scene[\'captaincy_plan\'],self.timeline,media,badges,approved_heroes)\n        self.manifest=dict(scene,display_title=\'CAPTAINCY MASTERCLASS\',title=\'Captaincy Masterclass\',subheading=\'MY SIX-GAMEWEEK ARMBAND CALCULATIONS\')\n        self.complete=self.draw(self.timeline[-1][\'end\']+1.4,self.timeline[-1][\'end\']+2.5)\n    def draw(self,seconds,duration):\n        from PIL import ImageDraw,ImageFont\n        im=self.renderer.animation_image(self.manifest,self.metadata,self.width,self.height,self.content_html,seconds).convert(\'RGB\')\n        def verify_content():\n            return self.renderer._page.evaluate("""() => {\n                const root=document.querySelector(\'.wcx-v6\');\n                if(!root)throw new Error(\'Captaincy content was not mounted\');\n                const panels=[...root.querySelectorAll(\'[data-wcx6-week],[data-wcx6-summary]\')];\n                const active=panels.find(panel=>getComputedStyle(panel).visibility===\'visible\' && Number(getComputedStyle(panel).opacity)>0);\n                if(!active)throw new Error(\'Captaincy has no visible decision panel\');\n                const hero=[...active.querySelectorAll(\'.wcx6-hero-art\')].find(img=>{\n                    const panel=img.closest(\'.wcx6-hero-option\');\n                    return !panel || getComputedStyle(panel).visibility===\'visible\';\n                });\n                if(!hero || !hero.complete || hero.naturalWidth<100)throw new Error(\'Captaincy hero image is not ready\');\n                const portraits=[...active.querySelectorAll(\'.wcx6-photo\')];\n                if(!portraits.length || portraits.some(img=>!img.complete || img.naturalWidth<30))throw new Error(\'Captaincy official portraits are not ready\');\n                if(document.getElementById(\'heading\').textContent!==\'CAPTAINCY MASTERCLASS\')throw new Error(\'Captaincy heading is stale\');\n                return true;\n            }""")\n        self.renderer._executor.submit(verify_content).result()\n        return im\n    def frame(self,seconds,duration,previous=None):\n        import numpy as np\n        return np.asarray(self.draw(seconds,duration))\n\nCSS += \'#content .weekly-captain-xpts{margin-top:10px;font-size:27px;line-height:1.1;font-weight:800;color:#d8ff9d}#content .weekly-vice-xpts{font-size:24px;font-weight:800;color:#d7ecff;margin-top:8px}#content .alternative-player-card .captain-badge{font-size:24px}\'\n',encoding='utf-8')
sys.modules.pop('wildcard_final_slide',None)
(_vx_module_dir/'free_hit_final_slide.py').write_text('"""Free Hit final-slide adapter to the independent premium renderer."""\nfrom freehit_premium import captain_plan as from_contract,final_segments as generate_free_hit_final_slide_narration,align_timeline as align_final_timeline\nfrom freehit_premium import render_slide\ndef render_final_html(plan,timeline,media=None,badges=None):\n return render_slide(4,dict(gameweek=plan[\'next_gw\'],slide3_plan=plan[\'squad_plan\'],captain_pick=plan,freehit_case=plan[\'decision\']),timeline,media,badges)\n',encoding='utf-8')
sys.modules.pop('free_hit_final_slide',None)
(_vx_module_dir/'wildcard_analyst_intro.py').write_text('"""Data-backed Wildcard opening: one script, evidence cards and measured cues."""\nfrom __future__ import annotations\nimport copy\nimport html\nimport math\nimport json\nVERSION = \'wildcard-slide1-requested-script-v5\'\n\ndef num(value):\n    try:\n        result=float(value)\n        return result if math.isfinite(result) else None\n    except (ValueError,TypeError):return None\n\ndef validated_comparison(record, contract):\n    """Accept only complete calculated scenarios with matching scoring provenance."""\n    if not isinstance(record,dict):return None\n    a=record.get(\'current_with_free_transfers\',{}); b=record.get(\'wildcard\',{})\n    weeks=contract.get(\'horizon_gameweeks\',[])\n    if len(weeks)!=6:return None\n    keys=(\'model_id\',\'model_version\',\'projection_snapshot\',\'scoring_basis\',\'price_basis\',\'budget_basis\')\n    if any(not a.get(k) or a.get(k)!=b.get(k) for k in keys):return None\n    if a.get(\'scoring_basis\')!=\'starting_xi_plus_captain_net_hits\':return None\n    if not a.get(\'manager_squad_verified\') or not a.get(\'transfers_calculated\'):return None\n    if not a.get(\'legal_plan_verified\') or not b.get(\'legal_plan_verified\'):return None\n    if a.get(\'gameweeks\')!=weeks or b.get(\'gameweeks\')!=weeks:return None\n    if a.get(\'projection_snapshot\')!=contract.get(\'source_snapshot_at\'):return None\n    if num(a.get(\'budget\'))!=num(b.get(\'budget\')) or num(a.get(\'budget\'))!=num(contract.get(\'budget_total\')):return None\n    def total(scenario):\n        rows=scenario.get(\'weekly_results\',[])\n        if [r.get(\'gameweek\') for r in rows]!=weeks:return None\n        if any(num(r.get(\'xi_plus_captain_xpts\')) is None or num(r.get(\'hit_cost\')) is None or num(r.get(\'hit_cost\'))<0 for r in rows):return None\n        return sum(num(r[\'xi_plus_captain_xpts\'])-num(r[\'hit_cost\']) for r in rows)\n    current=total(a);wildcard=total(b)\n    if current is None or wildcard is None:return None\n    if not a.get(\'transfer_cost_analysis\'):return None\n    return dict(current=current,wildcard=wildcard,gain=wildcard-current,tradeoff=a[\'transfer_cost_analysis\'],source=record)\n\ndef enrich_case(case, contract, bootstrap=None, manager_squad=None):\n    """Use finalized squad evidence; never equate popular assets with the user\'s team."""\n    result=copy.deepcopy(case);bootstrap=bootstrap or {};weeks=result.get(\'analysis_gws\',[])\n    from wildcard_intro_context import refresh_context\n    result[\'intro_context\']=refresh_context(contract,bootstrap,manager_squad)\n    result[\'analyst_version\']=VERSION\n    result[\'analysis_title\']=(\'FPL VORTEX — WILDCARD ANALYSIS | GW \'+str(weeks[0])+\'–\'+str(weeks[-1])) if weeks else \'FPL VORTEX — WILDCARD ANALYSIS\'\n    comparison=validated_comparison(contract.get(\'wildcard_scenario_comparison\'),contract)\n    result[\'scenario_comparison\']=comparison\n    players=contract.get(\'squad\',[]);evidence=[]\n    # Select the strongest qualified club evidence, with positional FDR explicitly labelled.\n    usable=[p for p in players if p.get(\'position\') not in (\'GK\',\'DEF\') and num(p.get(\'horizon_expected_points\')) is not None]\n    usable.sort(key=lambda p:-(num(p.get(\'horizon_expected_points\')) or 0))\n    featured=next((p for p in usable if sum(1 for w in p.get(\'weeks\',[]) for f in w.get(\'fixtures\',[]) if num(f.get(\'fdr\')) is not None and num(f[\'fdr\'])<=2.5)>=3),None)\n    if featured:\n        fixtures=[f for w in featured.get(\'weeks\',[]) if w.get(\'gameweek\') in weeks for f in w.get(\'fixtures\',[])]\n        ratings=[num(f.get(\'fdr\')) for f in fixtures if num(f.get(\'fdr\')) is not None]\n        homes=sum(f.get(\'home\') is True for f in fixtures);aways=sum(f.get(\'home\') is False for f in fixtures)\n        detail=f"{len(fixtures)} scheduled fixtures, with {homes} at home and {aways} away"\n        text=f"The clearest fixture route comes through {featured[\'club_name\']}. They have {detail}."\n        if ratings:text+=f" Their average positional fixture difficulty rating is {sum(ratings)/len(ratings):.2f}."\n        text+=f" That supports considering {featured[\'name\']} in the Wildcard squad."\n        xg=num(featured.get(\'xg90\'));xa=num(featured.get(\'xa90\'));minutes=num(featured.get(\'base_expected_minutes\'))\n        if xg is not None and xa is not None:\n            text+=f" The model estimates {xg+xa:.2f} expected goal involvements per ninety minutes, combining {xg:.2f} expected goals and {xa:.2f} expected assists."\n        if minutes is not None:text+=f" His base minutes estimate is {minutes:.0f} per appearance, before fixture-specific adjustments."\n        if featured.get(\'role_note\'):text+=\' \'+featured[\'role_note\']+\'.\'\n        text+=\' This is an attacking opportunity, not a guarantee of returns.\'\n        evidence.append(dict(id=\'fixture\',label=\'Fixture opportunity\',metric=f\'{sum(ratings)/len(ratings):.2f} FDR\' if ratings else \'Fixture route\',detail=featured[\'club_name\']+\' · \'+detail,narration=text,player_ids=[featured[\'id\']],source=\'Finalized squad fixture model\',scope=\'selected Wildcard squad\',stats=[(\'MODEL xGI/90\',f\'{xg+xa:.2f}\') if xg is not None and xa is not None else (\'xGI/90\',\'Unavailable\'),(\'xG/90 · xA/90\',f\'{xg:.2f} · {xa:.2f}\') if xg is not None and xa is not None else (\'xG/90 · xA/90\',\'Unavailable\'),(\'BASE MINUTES\',f\'{minutes:.0f} / appearance\') if minutes is not None else (\'MINUTES\',\'Uncertain\')],role=featured.get(\'role_note\',\'Role evidence unavailable\')))\n    # Only authenticated manager picks can support "we currently carry N" claims.\n    official={int(p[\'id\']):p for p in bootstrap.get(\'elements\',[]) if p.get(\'id\') is not None}\n    picks=(manager_squad or {}).get(\'picks\',[])\n    if (manager_squad or {}).get(\'verified\') is True and len(picks)==15 and len({p.get(\'element\') for p in picks})==15:\n        concerns=[official.get(int(p[\'element\']),{}) for p in picks if official.get(int(p[\'element\']),{}).get(\'status\') in (\'i\',\'s\',\'u\',\'d\')]\n        if concerns:\n            evidence.append(dict(id=\'efficiency\',label=\'Current squad concerns\',metric=str(len(concerns))+\' flagged players\',detail=\'Official availability flags · current squad\',narration=f"Squad efficiency is the next question. Your verified current squad contains {len(concerns)} players with official availability flags. These include doubts as well as possible absences. A reset may improve cover, but first check whether free transfers can repair the affected positions.",player_ids=[p[\'id\'] for p in concerns],source=\'Authenticated current picks and official availability flags\',scope=\'current manager squad\'))\n    # Existing ranked triggers remain usable evidence, never neutral filler cards.\n    for card in result.get(\'reasons\',[]):\n        if len(evidence)>=2:break\n        if card.get(\'is_neutral\') or card.get(\'impact_level\')==\'Watchlist\':continue\n        if evidence and card.get(\'type\') in (\'fixture_swing\',\'optimisation_window\'):continue\n        evidence.append(dict(id=\'evidence_\'+str(card[\'type\']),label=card[\'label\'],metric=card[\'metric_value\'],detail=card[\'evidence_text\'],narration=card[\'evidence_text\']+\' \'+card[\'recommendation\'],player_ids=[p[\'id\'] for p in card.get(\'source_data_summary\',{}).get(\'players\',[]) if p.get(\'id\')],source=card.get(\'source_data_summary\',{}).get(\'source\',\'Verified live analysis\'),scope=\'wider player pool; not a count of your squad\'))\n    captain=next((p for p in players if p.get(\'id\')==contract.get(\'captain_picks\',{}).get(\'captain\',{}).get(\'id\')),None)\n    if captain and len(evidence)<3:\n        first=next((w for w in captain.get(\'weeks\',[]) if w.get(\'gameweek\')==contract.get(\'gameweek\')),{})\n        fixtures=first.get(\'fixtures\',[])\n        matches=\' and \'.join(str(f.get(\'opponent\',\'opponent\'))+(\' at home\' if f.get(\'home\') else \' away\') for f in fixtures[:2])\n        text=f"For captaincy, {captain[\'name\']} is the opening recommendation in the selected Wildcard squad."\n        if matches:text+=\' The opening fixture route is \'+matches+\'.\'\n        if num(captain.get(\'expected_points\')) is not None:text+=f" The model projects {captain[\'expected_points\']:.1f} points before captain doubling."\n        text+=\' This identifies a captaincy option. It does not establish an advantage over your current captain without comparing both squads.\'\n        evidence.append(dict(id=\'captaincy\',label=\'Captaincy route\',metric=captain[\'name\'],detail=matches or \'Selected squad opening captain\',narration=text,player_ids=[captain[\'id\']],source=\'Selected squad and shared projection model\',scope=\'selected Wildcard squad\',stats=[(\'OPENING xPTS\',f\'{captain["expected_points"]:.1f}\') ] if num(captain.get(\'expected_points\')) is not None else [],role=\'Before captain doubling · reassess late team news\'))\n    secure=sum((num(p.get(\'base_expected_minutes\')) or 0)>=75 for p in players)\n    flags=sum(bool(p.get(\'availability\',{}).get(\'flagged\')) for p in players)\n    structure=next((e for e in evidence if e[\'id\'] not in (\'fixture\',\'captaincy\')),None)\n    if structure:\n        structure.update(label=\'Squad structure\',metric=f\'{secure} of 15\',detail=\'Players with at least 75 base expected minutes\',narration=f\'The second factor is squad structure. In the selected Wildcard squad, {secure} of fifteen players have at least seventy-five base expected minutes. There are {flags} official availability flags. The table shows the leading six-week player projections before captaincy, alongside their prices. These are characteristics of the proposed squad, not a measured improvement over your current team. Minutes and availability can still change.\')\n    result[\'analyst_evidence\']=evidence[:3]\n    # No measured comparison means no chip verdict, even when the draft is strong.\n    decision=contract.get(\'wildcard_decision\') or {}\n    provenance=bool(comparison and contract.get(\'decision_input_hash\') and decision.get(\'source_data_hash\')==contract[\'decision_input_hash\'] and decision.get(\'evidence_based_verdict\'))\n    play=bool(provenance and decision.get(\'play_supported\') is True and decision.get(\'hold_supported\') is not True and comparison[\'gain\']>0)\n    hold=bool(provenance and decision.get(\'hold_supported\') is True and decision.get(\'play_supported\') is not True)\n    result[\'verdict\']=\'ACTIVATE\' if play else \'HOLD\' if hold else \'OPEN\'\n    result[\'verdict_label\']=\'Activation supported by verified comparison\' if play else \'Saving supported by verified comparison\' if hold else \'Exploratory draft — decision open\'\n    result[\'verdict_reason\']=str(decision[\'evidence_based_verdict\']) if play or hold else (\'A verified current-squad-plus-free-transfers comparison and a matching evidence-based model decision are missing.\' if not comparison else \'The comparison exists, but no validated chip-timing decision accounts for keeping the chip.\')\n    # Intro claims are derived from the finalized contract, never from a screenshot.\n    featured=featured or (usable[0] if usable else {})\n    captain=captain or {}\n    fixtures=[dict(f,gameweek=w[\'gameweek\']) for w in featured.get(\'weeks\',[]) if w.get(\'gameweek\') in weeks for f in w.get(\'fixtures\',[]) if num(f.get(\'fdr\')) is not None]\n    first=next((w for w in captain.get(\'weeks\',[]) if w.get(\'gameweek\')==contract.get(\'gameweek\')),{});opening=first.get(\'fixtures\',[])\n    clubs=contract.get(\'clubs\',[]);clubs=list(clubs.values()) if isinstance(clubs,dict) else clubs\n    club_names={c.get(\'id\',c.get(\'team_id\')):c.get(\'name\',c.get(\'club_name\')) for c in clubs}\n    f=opening[0] if len(opening)==1 else {}\n    opponent=club_names.get(f.get(\'opponent_id\')) or f.get(\'opponent\')\n    if opponent==\'LEE\':opponent=\'Leeds\'\n    featured=copy.deepcopy(featured)\n    captain=copy.deepcopy(captain)\n    for highlighted in (featured, captain):\n        if highlighted and highlighted.get(\'id\') in official:\n            highlighted[\'spoken_name\'] = str(official[highlighted[\'id\']].get(\'web_name\') or highlighted.get(\'name\') or \'\').strip()\n    result[\'intro_data\']=dict(fixture=featured,captain=captain,average_fdr=sum(float(f[\'fdr\']) for f in fixtures)/len(fixtures) if fixtures else None,ribbon=fixtures if len(fixtures)==6 and len({f[\'gameweek\'] for f in fixtures})==6 else [],secure=secure,total=len(players),opponent=opponent,match=captain.get(\'club_name\',\'\').upper()+\' vs \'+str(opponent or \'unconfirmed\'),venue=\'HOME\' if f.get(\'home\') is True else \'AWAY\' if f.get(\'home\') is False else \'VENUE UNCONFIRMED\',venue_spoken=\'at home\' if f.get(\'home\') is True else \'away\' if f.get(\'home\') is False else \'\')\n    result[\'narration_timeline\']=generate_segments(result)\n    result[\'narration_text\']=\' \'.join(s[\'narration_text\'] for s in result[\'narration_timeline\'])\n    return result\n\ndef generate_segments(case):\n    """Wildcard Slide 1 replacement; all changing facts come from its case data."""\n    weeks = list(case.get(\'analysis_gws\') or [])\n    if not weeks or weeks != sorted(set(weeks)) or not case.get(\'intro_context\'):\n        raise ValueError(\'Refresh the Wildcard introduction evidence and planning window\')\n    player = (case.get(\'intro_data\') or {}).get(\'captain\') or {}\n    player_name = str(player.get(\'spoken_name\') or player.get(\'name\') or \'\').strip()\n    horizon_points = num(player.get(\'horizon_expected_points\'))\n    opening_points = num(player.get(\'expected_points\'))\n    if not player.get(\'id\') or not player_name or horizon_points is None or opening_points is None:\n        raise ValueError(\'Slide 1 requires a verified featured player and both displayed projections\')\n    words = {1: \'one\', 2: \'two\', 3: \'three\', 4: \'four\', 5: \'five\', 6: \'six\', 7: \'seven\', 8: \'eight\'}\n    horizon = words.get(len(weeks), str(len(weeks)))\n    period = \'Gameweek\' if len(weeks) == 1 else \'Gameweeks\'\n    items = [\n        (\'question\', f"Thinking of playing your Wildcard in Gameweek {weeks[0]}? Well, before you hit that button, there\'s something worth considering."),\n        (\'context\', "Is your team genuinely in need of a rebuild, or are you just reacting to a couple of disappointing Gameweeks?"),\n        (\'discipline\', "Because if injuries are piling up and your squad needs serious changes, the Wildcard could be exactly what you need."),\n        (\'draft\', f"Now, I\'ve put together my own Wildcard draft, looking ahead from Gameweek {weeks[0]} all the way to Gameweek {weeks[-1]}."),\n        (\'plan\', "And we\'re not just chasing one good fixture here. We\'re looking at consistent minutes, favourable matchups, attacking potential and captaincy options."),\n        (\'captain_player\', f"{player_name} is a big part of the plan, with {horizon_points:.1f} projected points across the {horizon} {period} and {opening_points:.1f} points in the opening round."),\n        (\'decision_question\', "But does the overall squad justify using the chip?"),\n        (\'preview\', "That\'s what we\'re going to find out."),\n        (\'transition\', "First, let\'s take a look at the fixtures."),\n    ]\n    cues = [dict(segment_id=identity, type=identity, narration_text=text,\n        player_ids=[player[\'id\']] if identity == \'captain_player\' else [],\n        start=None, end=None, on_start=[\'activate_\' + identity], on_end=[])\n        for identity, text in items]\n    case[\'intro_indicators\'] = [\'SQUAD REVIEW\', \'REBUILD CHECK\']\n    case[\'intro_word_count\'] = sum(len(cue[\'narration_text\'].split()) for cue in cues)\n    return cues\n\ndef generate_narration(case):\n    return \' \'.join(s[\'narration_text\'] for s in generate_segments(case))\n\ndef render_html(case, timeline, contract=None, media=None, badges=None, **kwargs):\n    weeks=case[\'analysis_gws\']\n    window=f"GW {weeks[0]}–{weeks[-1]}"\n    return \'<section class="wc-blueprint personal-intro"><h1>WILDCARD: NOW OR WAIT?</h1><h2>MY WILDCARD DRAFT</h2><p>\'+html.escape(window)+\'</p><p>Minutes • Matchups • Attacking potential • Captaincy</p><footer>NEXT: FIXTURE ANALYSIS</footer></section>\'\n\nCSS=\'\'\nSEEK_JS=\'\'\n',encoding='utf-8')
(_vx_module_dir/'wildcard_intro_context.py').write_text('"""Fresh official context; missing research remains unknown, never a verdict."""\nfrom __future__ import annotations\nimport re\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom html.parser import HTMLParser\nfrom typing import Any\nimport requests\nAPI=\'https://fantasy.premierleague.com/api/\'\nCALENDAR=\'https://inside.fifa.com/tournament-organisation/international-match-calendars\'\nclass CalendarParser(HTMLParser):\n    def __init__(self)->None:\n        super().__init__();self.rows=[];self.row=None;self.cell=None;self.men=False\n    def handle_starttag(self,tag,attrs):\n        if tag==\'tr\' and self.men:self.row=[]\n        if tag in (\'td\',\'th\') and self.row is not None:self.cell=[]\n    def handle_data(self,data):\n        if re.search(r"(?:Men’s|Men\'s|Men’s) IMC",data):self.men=True\n        if re.search(r"(?:Women’s|Women\'s) IMC",data):self.men=False\n        if self.cell is not None:self.cell.append(data)\n    def handle_endtag(self,tag):\n        if tag in (\'td\',\'th\') and self.cell is not None:\n            if self.row is not None:self.row.append(\' \'.join(\' \'.join(self.cell).split()))\n            self.cell=None\n        if tag==\'tr\' and self.row is not None:self.rows.append(self.row);self.row=None\n\ndef refresh_context(contract:dict[str,Any],bootstrap:dict[str,Any],manager_squad:dict[str,Any]|None=None,*,horizon_size:int=6)->dict[str,Any]:\n    now=datetime.now(timezone.utc);snapshot=datetime.fromisoformat(str(contract[\'source_snapshot_at\']).replace(\'Z\',\'+00:00\'))\n    if not 0<=(now-snapshot).total_seconds()<=86400:raise ValueError(\'Regenerate the model: introduction snapshot is stale or future-dated\')\n    weeks=contract.get(\'horizon_gameweeks\',[])\n    if len(weeks)!=horizon_size or not weeks or weeks[0]!=contract[\'gameweek\']:raise ValueError(\'Introduction horizon does not match the selected chip\')\n    with requests.Session() as session:\n        session.headers.update({\'User-Agent\':\'FPL-VORTEX/intro-context\',\'Cache-Control\':\'no-cache\'})\n        response=session.get(API+\'bootstrap-static/\',timeout=25);response.raise_for_status();live=response.json()\n        response=session.get(API+\'fixtures/\',timeout=25);response.raise_for_status();fixtures=response.json()\n        if not live.get(\'elements\') or not isinstance(fixtures,list):raise ValueError(\'Invalid official context response\')\n        next_event=next((event for event in live[\'events\'] if event.get(\'is_next\')),None)\n        if not next_event or next_event[\'id\']!=contract[\'gameweek\']:raise ValueError(\'Target Gameweek changed: rerun the statistical model\')\n        players={player[\'id\']:player for player in live[\'elements\']};teams={team[\'id\']:team[\'name\'] for team in live[\'teams\']}\n        for player in contract.get(\'squad\',[]):\n            official=players.get(player[\'id\'])\n            if not official or official[\'web_name\']!=player[\'name\']:raise ValueError(\'Draft identity changed: regenerate the model\')\n        flags=[{key:player.get(key) for key in (\'id\',\'web_name\',\'status\',\'news\',\'news_added\',\'chance_of_playing_next_round\')} for player in players.values() if player.get(\'status\') in (\'i\',\'s\',\'u\',\'d\')]\n        counts=Counter((fixture.get(\'event\'),team) for fixture in fixtures for team in (fixture.get(\'team_h\'),fixture.get(\'team_a\')) if fixture.get(\'event\') in weeks)\n        # Unpublished rounds do not prove a blank; require a full published round.\n        complete={week:sum(counts[(week,team)] for team in teams)>=len(teams) for week in weeks}\n        doubles=[{\'gameweek\':week,\'team_id\':team,\'club\':teams[team]} for week in weeks for team in teams if counts[(week,team)]>1]\n        blanks=[{\'gameweek\':week,\'team_id\':team,\'club\':teams[team]} for week in weeks for team in teams if complete[week] and counts[(week,team)]==0]\n        international={\'verified\':False,\'recently_ended\':False,\'approaching\':False,\'source_url\':CALENDAR}\n        try:\n            response=session.get(CALENDAR,timeout=12);response.raise_for_status();parser=CalendarParser();parser.feed(response.text)\n            year=None;windows=[]\n            for row in parser.rows:\n                if row and re.fullmatch(r\'20\\d{2}\',row[0]):year=int(row[0]);row=row[1:]\n                if not year or not any(\'international match window\' in cell.lower() for cell in row):continue\n                for cell in row:\n                    match=re.fullmatch(r\'(\\d{1,2})\\s*([A-Za-z]+)?\\s*[–—-]\\s*(\\d{1,2})\\s*([A-Za-z]+)\',cell)\n                    if not match:continue\n                    first,month,last,end_month=match.groups()\n                    start=datetime.strptime(f\'{first} {month or end_month} {year}\',\'%d %B %Y\').replace(tzinfo=timezone.utc)\n                    end=datetime.strptime(f\'{last} {end_month} {year}\',\'%d %B %Y\').replace(tzinfo=timezone.utc)\n                    if start<=end:windows.append({\'start\':start.isoformat(),\'end\':end.isoformat()})\n            if not windows:raise ValueError(\'No unambiguous men’s calendar windows parsed\')\n            international.update(verified=True,windows=windows)\n            for window in windows:\n                if 0<=(now-datetime.fromisoformat(window[\'end\'])).total_seconds()<=7*86400:international.update(recently_ended=True,relevant_window=window)\n                if 0<(datetime.fromisoformat(window[\'start\'])-now).total_seconds()<=14*86400:international.update(approaching=True,relevant_window=window)\n        except (requests.RequestException,ValueError) as error:international[\'limitation\']=str(error)\n        manager=manager_squad or {};picks=manager.get(\'picks\',[]);ids={pick.get(\'element\') for pick in picks}\n        verified=manager.get(\'verified\') is True and len(picks)==15 and len(ids)==15 and ids.issubset(players)\n        reasons=contract.get(\'wildcard_case\',{}).get(\'reasons\',[])\n        qualified=[reason for reason in reasons if not reason.get(\'is_neutral\') and reason.get(\'impact_level\')!=\'Watchlist\']\n        return {\'checked_at\':now.isoformat(),\'projection_snapshot\':contract[\'source_snapshot_at\'],\'gameweek\':contract[\'gameweek\'],\'horizon\':weeks,\'sources\':[API+\'bootstrap-static/\',API+\'fixtures/\',CALENDAR],\n            \'availability\':{\'verified\':True,\'flags\':flags,\'injured\':[flag for flag in flags if flag[\'status\']==\'i\'],\'suspended\':[flag for flag in flags if flag[\'status\']==\'s\'],\'doubts\':[flag for flag in flags if flag[\'status\']==\'d\'],\'scope\':\'official pool; not the viewer squad\'},\'international\':international,\n            \'fixture_swing\':next((reason for reason in qualified if reason[\'type\']==\'fixture_swing\'),None),\'minutes_risk\':next((reason for reason in qualified if reason[\'type\']==\'minutes_risk\'),None),\n            \'doubles\':doubles,\'blanks\':blanks,\'schedule_complete\':complete,\'manager\':{\'verified\':verified,\'concerns\':[flag for flag in flags if flag[\'id\'] in ids] if verified else [],\'free_transfers\':manager.get(\'free_transfers\',contract.get(\'free_transfers\')) if verified else None,\'budget\':manager.get(\'budget\',contract.get(\'budget_total\')) if verified else None},\n            \'press_conferences\':contract.get(\'data_sources\',{}).get(\'news\',{}),\'limits\':[\'Press-conference evidence uses the existing refreshed source-linked news pipeline. Official doubts are not confirmed absences.\',\'Current squad, selling values and remaining free transfers need verified manager input.\']}\n')
sys.modules.pop('wildcard_intro_context',None)
sys.modules.pop('wildcard_analyst_intro',None)
(_vx_module_dir/'wildcard_squad_analysis.py').write_text('"""Wildcard Slide 3: the selected fifteen, individual reveals and six measured weekly plans."""\nfrom __future__ import annotations\nimport copy\nimport html\nimport json\nimport math\nimport base64\nfrom pathlib import Path\nVERSION=\'wildcard-squad-analyst-v1\'\nPOSITIONS=(\'GK\',\'DEF\',\'MID\',\'FWD\')\ndef number(v):\n    try:\n        x=float(v);return x if math.isfinite(x) else None\n    except (TypeError,ValueError):return None\n\ndef common_name(p,official):\n    first=str(official.get(\'first_name\',\'\')).strip();last=str(official.get(\'second_name\',\'\')).strip()\n    return first.split()[0]+\' \'+last if first and last else p.get(\'full_name\') or p[\'name\']\n\ndef fixture_text(f):return str(f.get(\'opponent\',\'an unnamed opponent\'))+(\' at home\' if f.get(\'home\') else \' away\')\n\ndef enrich_plan(plan,contract,bootstrap=None,models=(),weekly_models=()):\n    result=copy.deepcopy(plan);official={int(p[\'id\']):p for p in (bootstrap or {}).get(\'elements\',[])}\n    cards={int(p[\'id\']):p for p in contract.get(\'squad\',[])}\n    byid={int(p[\'id\']):p for p in result[\'squad\']};selected=set(byid)\n    if len(selected)!=15 or len(result[\'starting_xi\'])!=11 or len(result[\'bench\'])!=4:raise ValueError(\'Slide 3 requires the finalized fifteen-player squad\')\n    result[\'analyst_version\']=VERSION\n    result[\'transfer_options\']=copy.deepcopy(contract.get(\'transfer_plan\',[]))\n    for pid,p in byid.items():\n        o=official.get(pid,{});c=cards.get(pid,{});p[\'spoken_name\']=common_name(p,o)\n        fixtures=[dict(f,gameweek=w[\'gameweek\']) for w in c.get(\'weeks\',[]) if w.get(\'gameweek\') in result[\'analysis_gws\'] for f in w.get(\'fixtures\',[])]\n        xg=number(o.get(\'expected_goals_per_90\'));xa=number(o.get(\'expected_assists_per_90\'));xgi=number(o.get(\'expected_goal_involvements_per_90\'))\n        if xgi is None and xg is not None and xa is not None:xgi=xg+xa\n        clean=[number(f.get(\'clean_sheet_probability\')) for f in fixtures if number(f.get(\'clean_sheet_probability\')) is not None]\n        saves=[number(f.get(\'components\',{}).get(\'saves\')) for f in fixtures];bonus=[number(f.get(\'components\',{}).get(\'bonus\')) for f in fixtures]\n        p[\'analysis\']=dict(xg90=xg,xa90=xa,xgi90=xgi,fixtures=fixtures,clean_sheet_probability=sum(clean)/len(clean) if clean else None,save_points=sum(saves) if saves and all(x is not None for x in saves) else None,bonus_points=sum(bonus) if bonus and all(x is not None for x in bonus) else None,role=c.get(\'role_note\') or \'Set-piece priority is not confirmed\',form=number(o.get(\'form\')),minutes=number(p.get(\'minutes\')),start_probability=number(p.get(\'starting_probability\')),points=number(p.get(\'projected_points\')),value=number(p.get(\'projected_points\'))/p[\'price\'] if number(p.get(\'projected_points\')) is not None and p[\'price\']>0 else None,availability=c.get(\'availability\',{}),good_weeks=sorted({f[\'gameweek\'] for f in fixtures if number(f.get(\'fdr\')) is not None and number(f[\'fdr\'])<=2.5}))\n        # Compare like-position alternatives with the same six-week projection model.\n        peers=[m for m in models if int(m.get(\'player_id\',-1)) not in selected and m.get(\'position\')==p[\'position\'] and number(m.get(\'price\')) is not None and abs(float(m[\'price\'])-p[\'price\'])<=.5 and number(m.get(\'xpts_wildcard_horizon\')) is not None and number(m.get(\'expected_minutes\')) is not None and float(m[\'expected_minutes\'])>=65 and number(m.get(\'p_start\')) is not None and float(m[\'p_start\'])>=.8 and not m.get(\'news_monitor\') and m.get(\'wc_eligible\',True) and official.get(int(m.get(\'player_id\',-1)),{}).get(\'status\')==\'a\']\n        if peers:\n            peer=max(peers,key=lambda m:(float(m[\'xpts_wildcard_horizon\']),-int(m[\'player_id\'])))\n            po=official.get(int(peer[\'player_id\']),{});name=common_name(dict(name=peer.get(\'player_name\',\'Alternative\')),po)\n            p[\'analysis\'][\'alternative\']=dict(id=int(peer[\'player_id\']),name=name,price=float(peer[\'price\']),points=float(peer[\'xpts_wildcard_horizon\']))\n    weekly_lookup={(int(r[\'player_id\']),int(r[\'gameweek\'])):r for r in weekly_models if r.get(\'player_id\') is not None and r.get(\'gameweek\') is not None}\n    for week in result.get(\'gameweek_plans\',[]):\n        week[\'all_players\']=[dict(id=pid,xpts=number(weekly_lookup.get((pid,week[\'gameweek\']),{}).get(\'gameweek_expected_points\'))) for pid in byid]\n    # Rebind all references to the enriched copies without changing selection or order.\n    result[\'squad\']=[byid[int(p[\'id\'])] for p in result[\'squad\']]\n    for key in (\'starting_xi\',\'bench\'):result[key]=[byid[int(p[\'id\'])] for p in result[key]]\n    for key in (\'captain\',\'vice_captain\'):result[key]=byid[int(result[key][\'id\'])]\n    totals=[number(w.get(\'captain_adjusted_xpts\')) for w in result.get(\'gameweek_plans\',[])]\n    result[\'six_week_xpts\']=sum(totals) if len(totals)==len(result[\'analysis_gws\']) and [w[\'gameweek\'] for w in result.get(\'gameweek_plans\',[])]==result[\'analysis_gws\'] and all(v is not None for v in totals) else None\n    result[\'narration_timeline\']=generate_segments(result)\n    return result\n\ndef player_analysis(p,plan,index):\n    a=p[\'analysis\'];name=p[\'spoken_name\'];position=p[\'position\'];price=p[\'price\'];parts=[]\n    introductions={\'GK\':f"We\'re starting in goal with {name}.",\'DEF\':f"Our defender number {index+1} is {name}.",\'MID\':f"Our midfield selection number {index+1} is {name}.",\'FWD\':f"Our forward number {index+1} is {name}."}\n    parts.append(introductions[position]);parts.append(f"He costs £{price:.1f} million.")\n    if a[\'points\'] is not None:parts.append(f"The model projects {a[\'points\']:.1f} points across this {len(plan[\'analysis_gws\'])}-Gameweek window, before any captain doubling.")\n    fixtures=a[\'fixtures\'];good=sorted([f for f in fixtures if number(f.get(\'fdr\')) is not None and number(f[\'fdr\'])<=2.5],key=lambda f:(float(f[\'fdr\']),f[\'gameweek\']))[:2]\n    if good:parts.append(\'The favourable rated fixtures include \'+\' and \'.join(fixture_text(f) for f in good)+\'.\')\n    elif fixtures:parts.append(\'The opening matchup is \'+fixture_text(fixtures[0])+\'.\')\n    if position in (\'GK\',\'DEF\') and a[\'clean_sheet_probability\'] is not None:parts.append(f"The team clean-sheet probability averages {a[\'clean_sheet_probability\']:.0%} per scheduled fixture in the model. His minutes still determine whether a clean-sheet return is earned.")\n    if position==\'GK\':\n        if a[\'save_points\'] is not None:parts.append(f"Projected save points contribute {a[\'save_points\']:.1f} across the window.")\n        if a[\'bonus_points\'] is not None:parts.append(f"The bonus component contributes another {a[\'bonus_points\']:.1f} projected points.")\n    if position!=\'GK\' and a[\'xgi90\'] is not None:\n        parts.append(f"His observed season expected goal involvement is {a[\'xgi90\']:.2f} per ninety minutes.")\n        if a[\'xg90\'] is not None and a[\'xa90\'] is not None:parts.append(f"That combines {a[\'xg90\']:.2f} expected goals and {a[\'xa90\']:.2f} expected assists per ninety.")\n    if a[\'minutes\'] is not None:parts.append(f"The conservative minutes estimate is {a[\'minutes\']:.0f} per appearance.")\n    if a[\'start_probability\'] is not None:parts.append(f"The starting estimate is {a[\'start_probability\']:.0%}, rather than a confirmed team selection.")\n    if a[\'role\']!=\'Set-piece priority is not confirmed\':parts.append(a[\'role\']+\'.\')\n    alternative=a.get(\'alternative\')\n    if alternative and a[\'points\'] is not None:\n        if a[\'points\']>=alternative[\'points\']:\n            parts.append(f"At a similar price, {alternative[\'name\']} projects {alternative[\'points\']:.1f} points for £{alternative[\'price\']:.1f} million. This selection leads that available comparison by {a[\'points\']-alternative[\'points\']:.1f} model points.")\n        else:parts.append(f"An alternative at a similar price is {alternative[\'name\']}, projecting {alternative[\'points\']:.1f} points. Our selected player does not lead that individual comparison. Reassess the trade-off within the complete squad\'s budget and club limits.")\n    elif a[\'value\'] is not None:parts.append(f"That is {a[\'value\']:.2f} projected points per million pounds within the selected squad.")\n    if p[\'id\']==plan[\'captain\'][\'id\']:parts.append(\'He also takes the opening captaincy, which we will compare in detail on the final slide.\')\n    elif p[\'id\']==plan[\'vice_captain\'][\'id\']:parts.append(\'He is the opening vice-captain.\')\n    if not p.get(\'secure\'):parts.append(\'The role is less secure, so late team news matters before locking in this selection.\')\n    return \' \'.join(parts)\n\ndef generate_segments(plan):\n    out=[];start=plan[\'next_gw\'];end=plan[\'end_gw\'];byid={p[\'id\']:p for p in plan[\'squad\']}\n    def add(identity,kind,text,player=None,direction=\'left\',**extra):\n        ids=[player[\'id\']] if player else []\n        out.append(dict(segment_id=identity,type=kind,narration_text=text,player_ids=ids,hero_player_id=ids[0] if ids else None,pitch_node_ids=[(\'bench-player-\' if kind==\'bench\' else \'pitch-player-\')+str(i) for i in ids],direction=direction,start=None,end=None,on_start=[\'reveal_player_and_card\'] if player else [\'activate_\'+kind],on_end=[\'retain_pitch_player\',\'fade_card\'] if player else [],**extra))\n    add(\'intro\',\'title\',f"Here is our FPL VORTEX Wildcard squad for Gameweek {start}. We have assessed expected points and the available underlying numbers. Minutes security, clean-sheet potential and value also shape how these fifteen players work together across the next {len(plan[\'analysis_gws\'])} Gameweeks. We open in a {plan[\'formation\']} formation. The starting eleven can change as fixtures and projections shift. Let\'s build the team from the goalkeeper through to the forwards.")\n    for position in POSITIONS:\n        if position==\'DEF\':add(\'defence_intro\',\'phase\',\'Next, the defensive line. We are comparing clean-sheet opportunity, attacking involvement and minutes security.\',phase=\'DEF\')\n        if position==\'MID\':add(\'midfield_intro\',\'phase\',\'Now we get to midfield, where the key question is how underlying involvement and reliable minutes justify the attacking investment.\',phase=\'MID\')\n        if position==\'FWD\':add(\'forwards_intro\',\'phase\',"Let\'s move to our forwards. We are looking for expected goals, dependable playing time and value across the fixture window.",phase=\'FWD\')\n        line=[p for p in plan[\'starting_xi\'] if p[\'position\']==position]\n        for i,p in enumerate(line):\n            direction=(\'right\' if i%2==0 else \'left\') if position==\'DEF\' else (\'left\' if i%2==0 else \'right\')\n            add(\'player_\'+str(p[\'id\']),\'players\',player_analysis(p,plan,i),p,direction,phase=position)\n    add(\'bench_intro\',\'phase\',"The starting eleven is complete. The Wildcard still needs fifteen players, with useful cover that does not tie up too much budget.",phase=\'BENCH\')\n    outfields=[p for p in plan[\'bench\'] if p[\'position\']!=\'GK\'];reserves=[p for p in plan[\'bench\'] if p[\'position\']==\'GK\']+outfields\n    for index,p in enumerate(reserves):\n        a=p[\'analysis\'];rank=outfields.index(p)+1 if p[\'position\']!=\'GK\' else None\n        lead=\'Our backup goalkeeper is \' if rank is None else f"Outfield substitute number {rank} is "\n        text=lead+p[\'spoken_name\']+f", costing £{p[\'price\']:.1f} million. "\n        text+=(\'The model role indicators support playable cover, subject to team news. \' if p.get(\'secure\') else \'His minutes remain uncertain, so do not assume automatic bench cover. \')\n        if a[\'minutes\'] is not None:text+=f"The conservative minutes estimate is {a[\'minutes\']:.0f} per appearance. "\n        if a[\'good_weeks\']:text+=\'The more favourable rated fixtures fall in Gameweeks \'+\' and \'.join(map(str,a[\'good_weeks\'][:2]))+\'. \'\n        text+=\'His purpose is cover and budget balance. Check availability before relying on a substitution.\'\n        add(\'bench_\'+str(p[\'id\']),\'bench\',text,p,\'left\' if index%2==0 else \'right\',bench_rank=rank)\n    cap=plan[\'captain\'];vice=plan[\'vice_captain\']\n    cap_fixture=\' and \'.join(fixture_text(f) for f in cap[\'analysis\'][\'fixtures\'] if f[\'gameweek\']==start)\n    cap_reason=(f" The opening captain projects {cap[\'first_projection\']:.1f} points before doubling." if number(cap.get(\'first_projection\')) is not None else \'\')+(f" The opening fixture route is {cap_fixture}." if cap_fixture else \'\')\n    vice_reason=f" The vice-captain projects {vice[\'first_projection\']:.1f} points before doubling." if number(vice.get(\'first_projection\')) is not None else \'\'\n    add(\'captaincy_budget\',\'summary\',f"Here is the complete fifteen-player Wildcard squad. Our Gameweek {start} captain is {cap[\'spoken_name\']}, with {vice[\'spoken_name\']} as vice-captain.{cap_reason}{vice_reason} The squad costs £{plan[\'total_cost\']:.1f} million, leaving £{plan[\'bank\']:.1f} million in the bank. The opening captaincy follows the supplied projections and availability checks. Selecting the squad is only half the task. Let\'s examine how it can work over the coming Gameweeks.")\n    previous=None\n    for week in plan.get(\'gameweek_plans\',[]):\n        gw=week[\'gameweek\'];xi=week[\'starting_xi\'];ids={p[\'id\'] for p in xi};pmap={p[\'id\']:p for p in xi};caprow=pmap.get(week[\'captain_id\']);vice_row=pmap.get(week[\'vice_captain_id\'])\n        text=f"Gameweek {gw}. The model lineup uses a {week[\'formation\']} formation. "\n        if caprow:text+=f"The preferred captain is {byid[caprow[\'id\']][\'spoken_name\']}. "\n        if caprow:\n            cap_fixtures=[f for f in byid[caprow[\'id\']][\'analysis\'][\'fixtures\'] if f[\'gameweek\']==gw]\n            text+=\'The fixture route is \'+(\' and \'.join(fixture_text(f) for f in cap_fixtures) if cap_fixtures else \'not yet published\')+\'. \'\n        if vice_row:text+=f"The vice-captain is {byid[vice_row[\'id\']][\'spoken_name\']}. "\n        if number(week.get(\'xi_xpts\')) is not None:text+=f"The starting eleven projects {week[\'xi_xpts\']:.1f} points before captain doubling. "\n        if number(week.get(\'captain_adjusted_xpts\')) is not None:text+=f"Including the captain, that becomes {week[\'captain_adjusted_xpts\']:.1f} projected points. "\n        if previous is not None:\n            ins=ids-previous;outs=previous-ids\n            if ins and outs:\n                incoming=sorted(ins);outgoing=sorted(outs)\n                if len(incoming)==len(outgoing)==1:\n                    ip=byid[incoming[0]];op=byid[outgoing[0]];text+=f"The model brings {ip[\'spoken_name\']} into the starting eleven for {op[\'spoken_name\']}. "\n                    old=next((p for p in plan[\'squad\'] if p[\'id\']==op[\'id\']),None)\n                    # Weekly point differences only when both values exist in the same supplied week.\n                    old_points=next((p.get(\'xpts\') for p in week.get(\'all_players\',[]) if p[\'id\']==op[\'id\']),None)\n                    if number(old_points) is not None and number(pmap[ip[\'id\']].get(\'xpts\')) is not None:text+=f"The same-week projection difference is {pmap[ip[\'id\']][\'xpts\']-old_points:.1f} points before captaincy. "\n                else:text+=f"There are {len(incoming)} lineup changes as the fixture window shifts. "\n            else:text+=\'The starting eleven is unchanged from the previous model week. \'\n        else:text+=\'This is the opening setup shown on the pitch. \'\n        transfer=next((t for t in plan.get(\'transfer_options\',[]) if t.get(\'gameweek\')==gw),None)\n        if transfer and transfer.get(\'action\')==\'TRANSFER\' and transfer.get(\'out_player_id\') and transfer.get(\'in_player_id\'):\n            text+=f"A separate conditional transfer option is {transfer.get(\'out_name\',\'the outgoing player\')} to {transfer.get(\'in_name\',\'the incoming player\')}. "\n            if number(transfer.get(\'xi_effect_xpts\')) is not None:text+=f"Its calculated immediate lineup and captaincy effect is {transfer[\'xi_effect_xpts\']:.2f} points before any transfer hit. "\n            if number(transfer.get(\'hit_cost\')) is not None:text+=f"The model scenario includes a {transfer[\'hit_cost\']:.0f}-point transfer cost. "\n            text+=\'That option is not included in the fixed-squad weekly totals. Recheck price, availability and your actual free transfers before considering it. \'\n        elif transfer and transfer.get(\'action\')==\'ROLL\':text+=\'The separate transfer model suggests rolling. Its assumed free-transfer balance still needs checking against your account. \'\n        text+=\'Treat this as an early plan and review the latest team news near the deadline.\'\n        add(\'week_\'+str(gw),\'gameweek_plan\',text,gameweek=gw);previous=ids\n    totals=plan.get(\'six_week_xpts\')\n    if totals is not None:add(\'horizon_total\',\'horizon\',f"Across the full {len(plan[\'analysis_gws\'])}-Gameweek window, the selected fifteen project {totals:.1f} starting-eleven points including the weekly captains. This assumes we keep this squad and rotate the lineup. Conditional transfer options are excluded. Updated minutes, injuries and team news can change the outlook.")\n    strengths=sorted([p for p in plan[\'starting_xi\'] if number(p[\'analysis\'][\'points\']) is not None],key=lambda p:-p[\'analysis\'][\'points\'])[:2]\n    text="Here\'s the FPL VORTEX squad verdict. "\n    if strengths:text+=\'The strongest projected contributions in this starting group come from \'+\' and \'.join(p[\'spoken_name\'] for p in strengths)+\'. \'\n    text+=\'The bench keeps cover available while the weekly model adjusts the starting eleven. The main risk is that minutes and availability estimates change. The opening lineup is built for Gameweek \'+str(start)+\', but the goal is to make the full fixture window work together. \'\n    text+="That\'s our FPL VORTEX Wildcard blueprint. Next, we\'re putting the captaincy candidates head-to-head."\n    add(\'transition\',\'transition\',text)\n    return out\n\ndef asset_uri(value):\n    if not value:return \'\'\n    path=Path(value)\n    if not path.is_file():return \'\'\n    mime=\'image/jpeg\' if path.suffix.lower() in (\'.jpg\',\'.jpeg\') else \'image/png\'\n    return \'data:\'+mime+\';base64,\'+base64.b64encode(path.read_bytes()).decode()\n\ndef render_html(plan,timeline,media=None,badges=None,**kwargs):\n    esc=lambda value:html.escape(str(value),quote=True)\n    media=media or {};badges=badges or {};opening={p[\'id\'] for p in plan[\'starting_xi\']}\n    ordered_bench=[p for p in plan[\'bench\'] if p[\'position\']==\'GK\']+[p for p in plan[\'bench\'] if p[\'position\']!=\'GK\']\n    def picture(p):\n        record=media.get(p[\'id\']) or media.get(str(p[\'id\'])) or {};uri=asset_uri(record.get(\'photo\')) if record.get(\'photo_kind\')==\'player\' else \'\'\n        badge=badges.get(p[\'club\']) or badges.get(p[\'team_id\']) or badges.get(str(p[\'team_id\'])) or {}\n        uri=uri or asset_uri(badge.get(\'path\') if isinstance(badge,dict) else badge)\n        return \'<img src="\'+uri+\'" alt="\'+esc(p[\'name\'])+\'">\' if uri else \'<span class="wb-initials">\'+esc(\'\'.join(word[0] for word in p[\'name\'].split()[:2]))+\'</span>\'\n    players=[];reserves=[];panels=[]\n    for p in plan[\'squad\']:\n        a=p[\'analysis\'];pid=p[\'id\'];marker=\'C\' if pid==plan[\'captain\'][\'id\'] else \'VC\' if pid==plan[\'vice_captain\'][\'id\'] else \'\'\n        players.append(f\'<article class="wb-pitch-player" data-wb-player="{pid}" data-position="{p["position"]}"><div class="wb-photo">{picture(p)}</div><b>{esc(p["name"])}</b><small>{p["position"]} · £{p["price"]:.1f}m <em class="wb-armband">{marker}</em></small></article>\')\n        reserves.append(f\'<article class="wb-reserve" data-wb-reserve="{pid}"><div class="wb-photo">{picture(p)}</div><b>{esc(p["name"])}</b><small>{p["position"]} · £{p["price"]:.1f}m</small><span class="wb-bench-order"></span></article>\')\n        metrics=[(\'WINDOW xPTS\',f\'{a["points"]:.1f}\' if a[\'points\'] is not None else \'—\'),(\'BASE MINUTES\',f\'{a["minutes"]:.0f}\' if a[\'minutes\'] is not None else \'—\'),(\'START ESTIMATE\',f\'{a["start_probability"]:.0%}\' if a[\'start_probability\'] is not None else \'—\'),(\'SEASON xGI/90\',f\'{a["xgi90"]:.2f}\' if a[\'xgi90\'] is not None else \'—\'),(\'SEASON xG/90\',f\'{a["xg90"]:.2f}\' if a[\'xg90\'] is not None else \'—\'),(\'SEASON xA/90\',f\'{a["xa90"]:.2f}\' if a[\'xa90\'] is not None else \'—\')]\n        if p[\'position\'] in (\'GK\',\'DEF\'):metrics.append((\'TEAM CS ESTIMATE\',f\'{a["clean_sheet_probability"]:.0%}\' if a[\'clean_sheet_probability\'] is not None else \'—\'))\n        if p[\'position\']==\'GK\':metrics.extend([(\'WINDOW SAVE xPTS\',f\'{a["save_points"]:.1f}\' if a[\'save_points\'] is not None else \'—\'),(\'WINDOW BONUS xPTS\',f\'{a["bonus_points"]:.1f}\' if a[\'bonus_points\'] is not None else \'—\')])\n        if a[\'value\'] is not None:metrics.append((\'xPTS / £m\',f\'{a["value"]:.2f}\'))\n        fixtures=[]\n        for gw in plan[\'analysis_gws\']:\n            fx=[f for f in a[\'fixtures\'] if f[\'gameweek\']==gw]\n            label=\' · \'.join(fixture_text(f) for f in fx) or \'No scheduled fixture\'\n            fixtures.append(f\'<div><b>GW{gw}</b><span>{esc(label)}</span></div>\')\n        peer=a.get(\'alternative\');comparison=(f"Same-position alternative: {peer[\'name\']} · £{peer[\'price\']:.1f}m · {peer[\'points\']:.1f} projected points" if peer else \'No qualified same-position price comparison in the available candidate data.\')\n        panels.append(f\'<section class="wb-hero" data-wb-hero="{pid}"><header><div class="wb-hero-photo">{picture(p)}</div><div><small>{esc(p["club_name"])} · {p["position"]}</small><h2>{esc(p["spoken_name"])}</h2><strong>£{p["price"]:.1f}m · {marker or "SQUAD SELECTION"}</strong></div></header><div class="wb-metrics">\'+\'\'.join(f\'<div><small>{label}</small><b>{value}</b></div>\' for label,value in metrics)+f\'</div><p class="wb-role">{esc(a["role"])}</p><div class="wb-player-fixtures">\'+\'\'.join(fixtures)+f\'</div><aside class="wb-comparison">{esc(comparison)}</aside><footer>{esc(p.get("risk_note") or "Model estimates · check late team news")}</footer></section>\')\n    week_panels=[];weekly_data=[]\n    for week in plan.get(\'gameweek_plans\',[]):\n        gw=week[\'gameweek\'];xi=week[\'starting_xi\'];byid={p[\'id\']:p for p in plan[\'squad\']};captain=byid[week[\'captain_id\']];vice=byid[week[\'vice_captain_id\']]\n        total=f"{week[\'xi_xpts\']:.1f} XI xPts" if week.get(\'xi_xpts\') is not None else \'XI projection unavailable\'\n        cap_total=f"{week[\'captain_adjusted_xpts\']:.1f} with captain" if week.get(\'captain_adjusted_xpts\') is not None else \'Captain total unavailable\'\n        entries=\'\'.join(f\'<div><b>{esc(byid[p["id"]]["name"])}</b><span>{p["xpts"]:.1f}</span></div>\' if p.get(\'xpts\') is not None else f\'<div><b>{esc(byid[p["id"]]["name"])}</b><span>—</span></div>\' for p in xi)\n        transfer=next((t for t in plan.get(\'transfer_options\',[]) if t.get(\'gameweek\')==gw),None)\n        if transfer and transfer.get(\'action\')==\'TRANSFER\' and transfer.get(\'out_player_id\') and transfer.get(\'in_player_id\'):\n            transfer_text=f"Conditional option: {transfer.get(\'out_name\',\'Outgoing player\')} → {transfer.get(\'in_name\',\'Incoming player\')}. "\n            impact=number(transfer.get(\'xi_effect_xpts\'));cost=number(transfer.get(\'hit_cost\'))\n            transfer_text+=(f"Immediate XI/captain effect {impact:.2f} before hits. " if impact is not None else \'Immediate points effect unavailable. \')\n            transfer_text+=(f"Scenario transfer cost: {cost:g} points. " if cost is not None else \'Transfer cost needs checking. \')\n            transfer_text+=\'Not included in the fixed-squad totals above.\'\n        elif transfer and transfer.get(\'action\')==\'ROLL\':transfer_text=\'Transfer model: roll, subject to your actual free transfers.\'\n        else:transfer_text=\'No verified transfer action for this week.\'\n\n        week_panels.append(f\'<section class="wb-week" data-wb-week="{gw}"><small>SIX-GAMEWEEK STRATEGY</small><h2>Gameweek {gw} · {week["formation"]}</h2><div class="wb-week-total"><b>{total}</b><span>{cap_total}</span></div><p>Captain: {esc(captain["spoken_name"])}<br>Vice-captain: {esc(vice["spoken_name"])}</p><div class="wb-week-xi">{entries}</div><aside>{esc(transfer_text)}</aside><footer>Selected fifteen · weekly rotation model · recheck near the deadline</footer></section>\')\n        weekly_data.append(dict(gameweek=gw,xi_ids=[p[\'id\'] for p in xi],captain_id=week[\'captain_id\'],vice_captain_id=week[\'vice_captain_id\']))\n    payload=json.dumps(dict(timeline=timeline,players=[dict(id=p[\'id\'],position=p[\'position\']) for p in plan[\'squad\']],opening_ids=[p[\'id\'] for p in plan[\'starting_xi\']],bench_ids=[p[\'id\'] for p in ordered_bench],weeks=weekly_data,captain_id=plan[\'captain\'][\'id\'],vice_captain_id=plan[\'vice_captain\'][\'id\']),ensure_ascii=False).replace(\'<\',\'\\\\u003c\')\n    horizon=f\'{plan["six_week_xpts"]:.1f} with weekly captains\' if plan.get(\'six_week_xpts\') is not None else \'Window total unavailable\'\n    neutral=f\'<section class="wb-neutral"><small>FPL VORTEX WILDCARD BLUEPRINT</small><h2>The selected fifteen</h2><p class="wb-neutral-captains">Opening formation: {plan["formation"]}<br>Captain: {esc(plan["captain"]["spoken_name"])}<br>Vice-captain: {esc(plan["vice_captain"]["spoken_name"])}</p><div class="wb-window-total">{horizon}</div><p>Keep the selected squad and rotate the starting eleven. Conditional transfers are separate options.</p><footer>Player roles and minutes are estimates. Check deadline team news.</footer></section>\'\n    summary=f\'<div class="wb-budget"><span>OPENING <b>{plan["formation"]}</b></span><span>SQUAD <b>£{plan["total_cost"]:.1f}m</b></span><span>BANK <b>£{plan["bank"]:.1f}m</b></span></div>\'\n    return \'<style>\'+CSS+\'</style><section class="wildcard-squad-analyst"><div class="wb-pitch-side"><header><h2>FPL VORTEX — WILDCARD SQUAD BUILDER</h2><p>GW \'+str(plan[\'next_gw\'])+\'–\'+str(plan[\'end_gw\'])+\' · 15 PLAYERS · £\'+f\'{plan["total_cost"]:.1f}\'+\'m</p></header><div class="wb-field">\'+PITCH_SVG+\'\'.join(players)+\'</div><div class="wb-bench"><h3>OPENING BENCH · OUTFIELD ORDER PRESERVED</h3><div class="wb-bench-players">\'+\'\'.join(reserves)+\'</div></div></div><aside class="wb-support">\'+summary+\'<div class="wb-phase">BUILDING THE SQUAD</div><div class="wb-stage">\'+\'\'.join(panels)+\'\'.join(week_panels)+neutral+\'</div><div class="wb-week-track">\'+\'\'.join(f\'<b data-wb-tab="{gw}">GW{gw}</b>\' for gw in plan[\'analysis_gws\'])+\'</div></aside><script type="application/json" class="wb-timeline">\'+payload+\'</script></section>\'\n\nCSS=\'\'\'\n.wildcard-squad-analyst{display:grid;grid-template-columns:minmax(0,60fr) minmax(0,40fr);gap:24px;width:100%;height:100%;color:#f5f9ff}.wb-pitch-side{display:grid;grid-template-rows:104px minmax(0,1fr) 208px;gap:12px;min-width:0}.wb-pitch-side>header{background:#061626e8;border-left:6px solid var(--accent-primary);padding:10px 22px}.wb-pitch-side>header h2{font-size:37px;margin:0 0 12px;line-height:1.1}.wb-pitch-side>header p{font-size:26px;margin:0;color:var(--accent-secondary);letter-spacing:2px}.wb-field{position:relative;overflow:hidden;border-radius:16px;background:repeating-linear-gradient(0deg,#0b633ad9 0 90px,#0c7140d9 90px 180px);border:2px solid #6cae7d}.wb-field .sp-pitch-lines{position:absolute;inset:0;width:100%;height:100%;opacity:.7}.wb-pitch-player{position:absolute;translate:-50% -50%;width:230px;text-align:center;opacity:0;visibility:hidden;filter:drop-shadow(0 5px 5px #0008)}.wb-photo{width:110px;height:110px;display:flex;align-items:center;justify-content:center;margin:0 auto;overflow:hidden}.wb-photo img{height:100%;width:100%;object-fit:contain}.wb-initials{font-size:44px;background:#0a2034;border:2px solid #9bc7ce;border-radius:50%;padding:13px}.wb-pitch-player>b{display:block;font-size:29px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;background:#07182de8;border-radius:6px;padding:6px 4px;margin-top:5px}.wb-pitch-player>small{font-size:23px;background:#07182de8;display:block;padding:4px;border-radius:4px}.wb-armband{font-style:normal;color:#ffe38e;font-weight:900}.wb-pitch-player[data-active="true"]>b{color:var(--accent-secondary);box-shadow:0 0 17px #b4f78f88}.wb-bench{padding:12px 15px;background:#061525ee;border:1px solid #4b718a;border-radius:13px;overflow:hidden}.wb-bench h3{font-size:23px;margin:0 0 8px;color:var(--accent-secondary)}.wb-bench-players{position:relative;display:flex;gap:12px;height:140px;align-items:center;justify-content:space-around}.wb-reserve{position:absolute;top:0;width:24%;visibility:hidden;opacity:0;text-align:center}.wb-reserve .wb-photo{height:64px;width:64px}.wb-reserve>b{font-size:24px;display:block;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}.wb-reserve>small{font-size:20px;display:block;color:#c4d7e7}.wb-bench-order{display:block;font-size:17px;color:#ffe38e;margin-top:4px}.wb-reserve[data-active="true"]{outline:2px solid var(--accent-primary);border-radius:9px;background:#102c3b}\n.wb-support{display:grid;grid-template-rows:93px 42px minmax(0,1fr) 48px;gap:12px;min-width:0;overflow:hidden}.wb-budget{display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;padding:12px 18px;background:#0c2032;border:1px solid #52728d;border-radius:12px}.wb-budget span{font-size:20px;color:#c5d7e6}.wb-budget b{display:block;font-size:34px;color:var(--accent-secondary);margin-top:5px}.wb-phase{font-size:27px;font-weight:800;letter-spacing:3px;padding:6px 8px;color:var(--accent-secondary)}.wb-stage{position:relative;min-height:0;overflow:hidden}.wb-hero,.wb-week,.wb-neutral{position:absolute;inset:0;background:linear-gradient(135deg,#10283cf5,#071220f5);border:2px solid #506b85;border-radius:16px;padding:25px;visibility:hidden;opacity:0;overflow:hidden}.wb-hero>header{display:flex;align-items:center;gap:22px;margin-bottom:18px;min-height:170px}.wb-hero-photo{height:158px;width:160px;flex:none;display:flex;align-items:center;justify-content:center}.wb-hero-photo img{height:100%;width:100%;object-fit:contain}.wb-hero>header small{font-size:24px;color:#bad2e5}.wb-hero h2{font-size:47px;line-height:1.08;margin:10px 0 14px}.wb-hero>header strong{font-size:30px;color:var(--accent-secondary)}.wb-metrics{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px}.wb-metrics>div{padding:12px 15px;background:#19364c;border-radius:8px}.wb-metrics small{font-size:18px;display:block;color:#b9cedf}.wb-metrics b{display:block;font-size:31px;margin-top:6px;color:var(--accent-secondary)}.wb-role{font-size:25px;margin:16px 0;color:#d2e5f4}.wb-player-fixtures{display:grid;gap:6px;margin-top:15px}.wb-player-fixtures>div{display:grid;grid-template-columns:94px minmax(0,1fr);padding:7px 10px;background:#0c2033;border-left:3px solid #457088;gap:10px;font-size:23px;line-height:1.15}.wb-player-fixtures b{color:var(--accent-secondary)}.wb-comparison{font-size:24px;line-height:1.25;margin-top:17px;padding:15px;background:#18332e;border:1px solid #497269;border-radius:9px}.wb-hero>footer,.wb-neutral>footer,.wb-week>footer{position:absolute;bottom:19px;left:25px;right:25px;font-size:21px;color:#a9c3d7;line-height:1.2}.wb-week>small,.wb-neutral>small{font-size:22px;letter-spacing:3px;color:var(--accent-secondary);font-weight:800}.wb-week h2,.wb-neutral h2{font-size:46px;margin:35px 0 20px}.wb-week-total{background:#153b38;border-left:5px solid var(--accent-primary);padding:18px}.wb-week-total b{font-size:39px;display:block;color:var(--accent-secondary)}.wb-week-total span{font-size:29px;display:block;margin-top:10px}.wb-week>p{font-size:28px;line-height:1.3}.wb-week-xi{display:grid;grid-template-columns:1fr 1fr;gap:8px}.wb-week-xi>div{display:flex;justify-content:space-between;gap:8px;background:#172c40;padding:12px 14px;font-size:24px;min-width:0}.wb-week-xi b{overflow:hidden;text-overflow:ellipsis;white-space:nowrap}.wb-week-xi span{color:var(--accent-secondary);font-weight:800}.wb-week>aside{font-size:25px;line-height:1.3;padding:20px;background:#1b303f;margin-top:22px;border-radius:9px}.wb-neutral>p{font-size:32px;line-height:1.45}.wb-window-total{font-size:38px;color:var(--accent-secondary);padding:25px;background:#163a35;margin:35px 0}.wb-week-track{display:flex;gap:10px;align-items:center;justify-content:space-between;padding:5px 12px;background:#0e2135;border-radius:8px}.wb-week-track b{font-size:25px;color:#b9ccde}.wb-week-track b[data-active="true"]{color:var(--accent-secondary);border-bottom:3px solid var(--accent-primary)}\n\'\'\'\n\nSEEK_JS=\'\'\'\n<script>(function(){const previous=window.seekWildcard;window.seekWildcard=function(seconds){previous(seconds);const root=document.querySelector(\'.wildcard-squad-analyst\');if(!root)return;const data=JSON.parse(root.querySelector(\'.wb-timeline\').textContent);const current=data.timeline.find(s=>seconds>=s.start&&seconds<s.end)||data.timeline[data.timeline.length-1];const clamp=v=>Math.max(0,Math.min(1,v));const smooth=v=>1-Math.pow(1-clamp(v),3);const revealed=new Set(data.timeline.filter(s=>s.start<=seconds&&[\'players\',\'bench\'].includes(s.type)).flatMap(s=>s.player_ids));const weekly=current.type===\'gameweek_plan\'?data.weeks.find(w=>w.gameweek===current.gameweek):null;const endPlan=null;const selected=weekly||endPlan;const xiIds=selected?selected.xi_ids:data.opening_ids;const benchIds=selected?[...data.bench_ids,...data.opening_ids].filter(id=>!xiIds.includes(id)):data.bench_ids;const all=data.players;const cap=selected?selected.captain_id:data.captain_id;const vice=selected?selected.vice_captain_id:data.vice_captain_id;const active=current.player_ids?.[0];for(const pos of [\'GK\',\'DEF\',\'MID\',\'FWD\']){const line=all.filter(p=>p.position===pos&&xiIds.includes(p.id));line.forEach((p,i)=>{const el=root.querySelector(`[data-wb-player="${p.id}"]`);const enter=data.timeline.find(s=>s.player_ids.includes(p.id)&&[\'players\',\'bench\'].includes(s.type));const progress=enter?smooth((seconds-enter.start)/.35):0;el.style.left=`${(i+.5)/line.length*88+6}%`;el.style.top={GK:\'16%\',DEF:\'38%\',MID:\'63%\',FWD:\'86%\'}[pos];el.style.visibility=revealed.has(p.id)?\'visible\':\'hidden\';el.style.opacity=String(revealed.has(p.id)?progress:0);el.style.scale=String(.94+.06*progress);el.dataset.active=String(p.id===active);el.querySelector(\'.wb-armband\').textContent=p.id===cap?\'C\':p.id===vice?\'VC\':\'\';});}for(const el of root.querySelectorAll(\'[data-wb-player]\'))if(!xiIds.includes(Number(el.dataset.wbPlayer)))el.style.visibility=\'hidden\';const visibleBench=benchIds.filter((id,i,ids)=>ids.indexOf(id)===i).sort((a,b)=>{const agk=all.find(p=>p.id===a).position===\'GK\',bgk=all.find(p=>p.id===b).position===\'GK\';return agk===bgk?benchIds.indexOf(a)-benchIds.indexOf(b):agk?-1:1;});let rank=0;for(const el of root.querySelectorAll(\'[data-wb-reserve]\')){const id=Number(el.dataset.wbReserve);const slot=visibleBench.indexOf(id);const show=slot>=0&&revealed.has(id);el.style.visibility=show?\'visible\':\'hidden\';el.style.opacity=show?\'1\':\'0\';el.style.left=slot>=0?`${slot*25}%`:\'0%\';el.dataset.active=String(id===active);if(slot>=0){const gk=all.find(p=>p.id===id).position===\'GK\';const before=visibleBench.slice(0,slot).filter(pid=>all.find(p=>p.id===pid).position!==\'GK\').length;el.querySelector(\'.wb-bench-order\').textContent=gk?\'BACKUP GK\':`OUTFIELD ${before+1}`;}}root.querySelector(\'.wb-bench h3\').textContent=selected?`GW${selected.gameweek} MODEL RESERVES · ORDER PROVISIONAL`:\'OPENING BENCH · OUTFIELD ORDER PRESERVED\';for(const el of root.querySelectorAll(\'.wb-hero,.wb-week,.wb-neutral\')){const show=el.classList.contains(\'wb-hero\')?[\'players\',\'bench\'].includes(current.type)&&Number(el.dataset.wbHero)===active:el.classList.contains(\'wb-week\')?current.type===\'gameweek_plan\'&&Number(el.dataset.wbWeek)===current.gameweek:![\'players\',\'bench\',\'gameweek_plan\'].includes(current.type);const progress=smooth((seconds-current.start)/.4);const tail=current.end-seconds;const opacity=[\'players\',\'bench\'].includes(current.type)?Math.min(progress,clamp(tail/.25)):progress;el.style.visibility=show?\'visible\':\'hidden\';el.style.opacity=show?String(opacity):\'0\';el.style.translate=show?`${(1-progress)*(current.direction===\'right\'?65:-65)}px 0px`:\'0px 0px\';}for(const el of root.querySelectorAll(\'.wb-neutral-captains,.wb-window-total\'))el.style.visibility=revealed.size===15?\'inherit\':\'hidden\';root.querySelector(\'.wb-phase\').textContent=current.type===\'bench\'?\'BENCH COVER\':current.type===\'gameweek_plan\'?`GAMEWEEK ${current.gameweek} STRATEGY`:current.phase?{GK:\'STARTING GOALKEEPER\',DEF:\'DEFENSIVE LINE\',MID:\'MIDFIELD ENGINE\',FWD:\'ATTACKING LINE\',BENCH:\'BENCH COVER\'}[current.phase]||\'SQUAD ANALYSIS\':[\'summary\',\'horizon\',\'transition\'].includes(current.type)?\'CAPTAINCY & SQUAD VALUE\':\'THE WILDCARD BLUEPRINT\';for(const tab of root.querySelectorAll(\'[data-wb-tab]\'))tab.dataset.active=String(Number(tab.dataset.wbTab)===(selected?selected.gameweek:null));};})();</script>\n\'\'\'\n\nPITCH_SVG = \'<svg class="sp-pitch-lines" viewBox="0 0 1200 1360" preserveAspectRatio="none" aria-label="Top-down football pitch"><defs><pattern id="sp-net" width="12" height="12" patternUnits="userSpaceOnUse"><path d="M12 0H0V12" fill="none" stroke="#edf7ec" stroke-width="1"/></pattern></defs><g fill="none" stroke="#ecf8ed" stroke-width="4"><rect x="38" y="50" width="1124" height="1260" rx="12"/><path d="M38 680H1162"/><circle cx="600" cy="680" r="110"/><circle cx="600" cy="680" r="4" fill="#ecf8ed"/><rect x="290" y="50" width="620" height="220"/><rect x="445" y="50" width="310" height="95"/><circle cx="600" cy="205" r="4" fill="#ecf8ed"/><rect x="290" y="1090" width="620" height="220"/><rect x="445" y="1215" width="310" height="95"/><circle cx="600" cy="1155" r="4" fill="#ecf8ed"/><path d="M38 82A32 32 0 0 0 70 50M1130 50A32 32 0 0 0 1162 82M38 1278A32 32 0 0 1 70 1310M1130 1310A32 32 0 0 1 1162 1278"/><rect x="520" y="18" width="160" height="32" fill="url(#sp-net)"/><rect x="520" y="1310" width="160" height="32" fill="url(#sp-net)"/></g></svg>\'\n',encoding='utf-8')
sys.modules.pop('wildcard_squad_analysis',None)
(_vx_module_dir/'captaincy_showdown.py').write_text('"""Free Hit-only captaincy showdown. Wildcard uses wildcard_final_slide exclusively."""\nimport copy, html, json\nfrom collections import Counter\nfrom wildcard_squad_analysis import number, asset_uri, fixture_text\nVERSION = \'captaincy-showdown-v2\'\n\ndef build(contract):\n    if contract.get(\'chip\') != \'freehit\':\n        raise ValueError(\'Free Hit-only captaincy module\')\n    p = contract[\'slide3_plan\']\n    fh = True\n    canonical = {x[\'id\']: x for x in contract[\'squad\']}\n    selected = {x[\'id\']: x for x in p[\'squad\']}\n    rows = []\n    weeks = [dict(gameweek=p[\'next_gw\'], starting_xi=p[\'starting_xi\'], captain_id=p[\'captain\'][\'id\'], vice_captain_id=p[\'vice_captain\'][\'id\'])]\n    for w in weeks:\n        gw = w[\'gameweek\']\n        candidates = []\n        for x in w[\'starting_xi\']:\n            s = selected[x[\'id\']]\n            c = canonical.get(x[\'id\'], {})\n            a = s.get(\'analysis\', {})\n            cw = next((v for v in c.get(\'weeks\', []) if v[\'gameweek\'] == gw), {})\n            fs = c.get(\'current_fixtures\', s.get(\'fixtures\', []))\n            xp = number(s.get(\'projected_points\'))\n            minutes = number(s.get(\'minutes\'))\n            if minutes is None and fs and all((number(f.get(\'expected_minutes\')) is not None for f in fs)):\n                minutes = sum((float(f[\'expected_minutes\']) for f in fs))\n            starts = [number(f.get(\'p_start\', f.get(\'starting_probability\'))) for f in fs]\n            start = min(starts) if starts and all((v is not None for v in starts)) else number(s.get(\'starting_probability\'))\n            opponent_rows = {int(r[\'team_id\']): r for r in contract.get(\'fixture_case\', {}).get(\'rows\', []) if r.get(\'team_id\') is not None}\n            defensive = [dict(team=f.get(\'opponent\'), **opponent_rows.get(int(f.get(\'opponent_id\', -1)), {}).get(\'team_statistics\', {})) for f in fs if opponent_rows.get(int(f.get(\'opponent_id\', -1)), {}).get(\'team_statistics\')]\n            candidates.append(dict(s, opposition_defence=defensive, projected_points=xp, fixture_text=\' and \'.join((fixture_text(f) for f in fs)) or \'Fixture unavailable\', minutes=minutes, start_probability=start, xgi90=a.get(\'xgi90\'), xg90=a.get(\'xg90\'), role=a.get(\'role\', c.get(\'role_note\')), opponent_goals=[number(f.get(\'opponent_goals\', f.get(\'opponent_expected_goals\'))) for f in fs], fixture_count=len(fs)))\n        byid = {x[\'id\']: x for x in candidates}\n        cap = byid.get(w[\'captain_id\'])\n        vice = byid.get(w[\'vice_captain_id\'])\n        if cap is None or vice is None or cap[\'id\'] == vice[\'id\']:\n            raise ValueError(\'Captain and vice must be distinct planned starters\')\n        others = sorted((x for x in candidates if x[\'id\'] != cap[\'id\']), key=lambda x: (x[\'projected_points\'] is None, -(x[\'projected_points\'] or 0), x[\'id\']))[:2]\n        if len(others) != 2:\n            raise ValueError(\'Three starting captaincy candidates are required\')\n        rival = others[0]\n        gap = cap[\'projected_points\'] - rival[\'projected_points\'] if cap[\'projected_points\'] is not None and rival[\'projected_points\'] is not None else None\n        reason = f"{gap:.2f} extra captain points versus {rival.get(\'spoken_name\', rival[\'name\'])}" if gap is not None and gap >= 0 else \'Selected squad captain; an alternative has a higher raw projection\' if gap is not None else \'Projection comparison unavailable\'\n        if gap is not None and gap < 0:\n            reason += f\' ({abs(gap):.2f} points)\'\n        comparative = []\n        for key, label in [(\'minutes\', \'expected minutes\'), (\'xgi90\', \'season expected goal involvement per ninety\'), (\'xg90\', \'season expected goals per ninety\')]:\n            a = number(cap.get(key))\n            b = number(rival.get(key))\n            if a is not None and b is not None and (a != b):\n                comparative.append(f"{cap.get(\'spoken_name\', cap[\'name\'])} has {a:.2f} {label}, compared with {b:.2f} for {rival.get(\'spoken_name\', rival[\'name\'])}.")\n        rows.append(dict(comparative_text=\' \'.join(comparative), gw=gw, captain=cap, vice_captain=vice, candidates=[others[0], cap, others[1]], incremental_advantage=gap, reason=reason, confidence=\'Provisional\', risk_note=\'Projection uncertainty is not quantified; review minutes and team news.\', fixture_text=cap[\'fixture_text\']))\n    counts = Counter((r[\'captain\'][\'id\'] for r in rows))\n    anchor = max(counts, key=counts.get)\n    return dict(next_gw=p[\'next_gw\'], end_gw=p.get(\'end_gw\', p[\'next_gw\']), chip=contract[\'chip\'], fixture_targets=copy.deepcopy(contract.get(\'fixture_case\', {}).get(\'targets\', [])), captaincy_plan=rows, captain=rows[0][\'captain\'], vice_captain=rows[0][\'vice_captain\'], anchor_id=anchor, summary_recommendations=[f"{selected[anchor].get(\'spoken_name\', selected[anchor][\'name\'])} leads {counts[anchor]} of {len(rows)} planned captain selections.", \'Review expected minutes and availability before each deadline.\'], squad_plan=copy.deepcopy(p), decision=copy.deepcopy(contract.get(\'freehit_case\')), data_health={\'valid\': True, \'complete\': all((all((x[\'projected_points\'] is not None for x in r[\'candidates\'])) for r in rows)), \'version\': VERSION})\n\ndef detail(x):\n    name = x.get(\'spoken_name\', x[\'name\'])\n    club = x.get(\'club_name\') or x.get(\'club\')\n    out = name + (\' from \' + club if club else \'\') + f" faces {x[\'fixture_text\']}. "\n    if x[\'projected_points\'] is not None:\n        out += f"The model projects {x[\'projected_points\']:.1f} points before captain doubling. "\n    else:\n        out += \'His points projection is unavailable. \'\n    if x.get(\'minutes\') is not None:\n        out += f"Expected playing time is {x[\'minutes\']:.0f} minutes across {x[\'fixture_count\']} scheduled fixtures. "\n    if x.get(\'start_probability\') is not None:\n        out += f"The starting estimate is {x[\'start_probability\']:.0%}. "\n    metric = x.get(\'xg90\') if x[\'position\'] == \'FWD\' else x.get(\'xgi90\')\n    label = \'expected goals\' if x[\'position\'] == \'FWD\' else \'expected goal involvement\'\n    if metric is not None:\n        out += f\'Observed season {label} is {metric:.2f} per ninety minutes. \'\n    if x.get(\'role\'):\n        out += x[\'role\'] + \'. \'\n    for stat in x.get(\'opposition_defence\', []):\n        out += f"The opposition conceded {stat[\'goals_against\']:.0f} goals in their latest {stat[\'matches\']} completed league matches, with {stat[\'clean_sheets\']} clean sheets. "\n    opp = x.get(\'opponent_goals\', [])\n    if opp and all((v is not None for v in opp)):\n        out += f\'The fixture model estimates {sum(opp):.2f} goals for the opposition across the scheduled matches. That is a model estimate, not observed defensive performance. \'\n    return out\n\ndef segments(plan, *unused):\n    out = []\n\n    def add(identity, kind, text, **kw):\n        out.append(dict(segment_id=identity, type=kind, narration_text=text, start=None, end=None, player_ids=kw.pop(\'player_ids\', []), **kw))\n    add(\'captain_intro\', \'intro\', "We\'ve built the squad. Now we compare three planned starters for the armband, using the same weekly projections and available minutes evidence.")\n    for row in plan[\'captaincy_plan\']:\n        gw = row[\'gw\']\n        cap = row[\'captain\']\n        vice = row[\'vice_captain\']\n        name = cap.get(\'spoken_name\', cap[\'name\'])\n        add(f\'gw_{gw}_opening\', \'week\', f"Gameweek {gw}. Let\'s compare the three candidates.", gw=gw)\n        for slot, x in enumerate(row[\'candidates\']):\n            add(f"gw_{gw}_candidate_{x[\'id\']}", \'candidate\', (\'Our FPL VORTEX pick is \' if slot == 1 else \'The alternative is \') + detail(x), gw=gw, player_id=x[\'id\'], player_ids=[x[\'id\']], slot=slot)\n        text = f"Our planned captain is {name}. {row[\'reason\']}. "\n        if row[\'incremental_advantage\'] is not None:\n            text += \'Both players are already in the starting eleven. The captaincy difference is their single projection difference, not the difference between two doubled totals. \'\n        text += row.get(\'comparative_text\', \'\') + \' \'\n        text += f"The vice-captain is {vice.get(\'spoken_name\', vice[\'name\'])}. "\n        if vice.get(\'start_probability\') is not None:\n            text += f"His starting estimate is {vice[\'start_probability\']:.0%}. "\n        if row[\'incremental_advantage\'] is not None and abs(row[\'incremental_advantage\']) < 1:\n            text += \'The projected gap is small, so late minutes news could change the choice. \'\n        text += \'The vice-captain only takes over if the captain plays no minutes, under the standard scoring rules. \'\n        text += \'Check team news before confirming the armband.\'\n        add(f\'gw_{gw}_verdict\', \'week_verdict\', text, gw=gw, player_ids=[cap[\'id\'], vice[\'id\']])\n    p = plan[\'squad_plan\']\n    complete = [r for r in plan[\'captaincy_plan\'] if r[\'captain\'][\'projected_points\'] is not None]\n    text = \'The final FPL VORTEX recommendations. \'\n    if plan.get(\'fixture_targets\'):\n        top = plan[\'fixture_targets\'][0]\n        text += \'The highest-ranked target club is \' + str(top.get(\'name\') or top[\'club\']) + \'. \'\n        fs = [f for w in top.get(\'weeks\', []) for f in w.get(\'fixtures\', [])]\n        if plan[\'chip\'] == \'freehit\' and fs:\n            text += \'Their immediate fixture route is \' + \' and \'.join((fixture_text(f) for f in fs)) + \'. \'\n    if complete:\n        best = max(complete, key=lambda r: r[\'captain\'][\'projected_points\'])\n        text += f"The highest planned captain projection is {best[\'captain\'].get(\'spoken_name\', best[\'captain\'][\'name\'])} in Gameweek {best[\'gw\']}, with {best[\'captain\'][\'projected_points\']:.1f} points before doubling. "\n    text += \' \'.join(plan[\'summary_recommendations\']) + \' \'\n    leaders = sorted(p[\'starting_xi\'], key=lambda x: -(number(x.get(\'projected_points\')) or 0))[:2]\n    text += \'The key selected assets by model projection are \' + \' and \'.join((x.get(\'spoken_name\', x[\'name\']) for x in leaders)) + \'. \'\n    value = max(p[\'starting_xi\'], key=lambda x: (number(x.get(\'projected_points\')) or 0) / max(0.1, x[\'price\']))\n    text += f"{value.get(\'spoken_name\', value[\'name\'])} offers the strongest projected points per million within this starting eleven. "\n    risk = min(p[\'starting_xi\'], key=lambda x: number(x.get(\'starting_probability\')) if number(x.get(\'starting_probability\')) is not None else -1)\n    text += f"Review the minutes outlook for {risk.get(\'spoken_name\', risk[\'name\'])} before the deadline. "\n    xi = p.get(\'xi_xpts\')\n    total = p.get(\'captain_adjusted_xpts\')\n    if xi is not None and total is not None:\n        text += f\'The eleven project {xi:.1f} points before captaincy, or {total:.1f} including the extra captain projection. The original squad returns after this Gameweek. \'\n    decision = plan.get(\'decision\') or {}\n    text += \'The chip verdict remains \' + decision.get(\'verdict\', \'HOLD\').title() + \'. \' + decision.get(\'verdict_reason\', \'A verified current-squad comparison is still required before recommending activation.\') + \' \'\n    add(\'captain_final\', \'outro\', text + \'Refresh team news and projections before the deadline. Good luck.\')\n    return out\n\ndef render(plan, timeline, media=None, badges=None):\n    esc = lambda s: html.escape(str(s), quote=True)\n    media = media or {}\n    badges = badges or {}\n    panels = []\n    table = []\n    for row in plan[\'captaincy_plan\']:\n        cards = []\n        for slot, x in enumerate(row[\'candidates\']):\n            record = media.get(x[\'id\'], media.get(str(x[\'id\']), {}))\n            photo = asset_uri(record.get(\'photo\')) if record.get(\'photo_kind\') == \'player\' else \'\'\n            xp = x[\'projected_points\']\n            name = x.get(\'spoken_name\', x[\'name\'])\n            metrics = []\n            for label, v, fmt in [(\'Expected minutes\', x.get(\'minutes\'), \'.0f\'), (\'Season xGI/90\', x.get(\'xgi90\'), \'.2f\'), (\'Season xG/90\', x.get(\'xg90\'), \'.2f\')]:\n                metrics.append(f"<div>{label}<b>{(format(v, fmt) if v is not None else \'Unavailable\')}</b></div>")\n            cards.append(f\'\'\'<article data-cs-player="{x[\'id\']}" class="cs-card {(\'cs-pick\' if slot == 1 else \'\')}"><small>{(\'VORTEX PICK\' if slot == 1 else \'CONTENDER \' + str(2 if slot == 0 else 3))}</small>\'\'\' + (f\'<img src="{photo}" alt="{esc(name)}">\' if photo else \'<div class="cs-avatar">\' + esc(\'\'.join((v[0] for v in name.split()[:2]))) + \'</div>\') + f"""<h3>{esc(name)}</h3><strong>{(f\'{xp:.1f}\' if xp is not None else \'—\')}</strong><p>Projected points · before doubling</p><div class="cs-fixture">{esc(x[\'fixture_text\'])}</div><div class="cs-metrics">""" + \'\'.join(metrics) + f"</div><footer>{esc(x.get(\'role\') or \'Role unavailable\')}</footer></article>")\n        panels.append(f\'\'\'<section class="cs-week" data-cs-week="{row[\'gw\']}"><h2>Gameweek {row[\'gw\']} · Captaincy showdown</h2><div class="cs-three">\'\'\' + \'\'.join(cards) + f"""</div><p class="cs-reason">{esc(row[\'reason\'])}. Projection uncertainty is not quantified.</p></section>""")\n        cap = row[\'captain\']\n        vice = row[\'vice_captain\']\n        xp = cap[\'projected_points\']\n        table.append(f\'\'\'<tr data-cs-row="{row[\'gw\']}"><td>{row[\'gw\']}</td><td>{esc(cap.get(\'spoken_name\', cap[\'name\']))}</td><td>{esc(vice.get(\'spoken_name\', vice[\'name\']))}</td><td>{(f\'{xp:.1f}\' if xp is not None else \'—\')}</td><td>{esc(row[\'reason\'])}</td></tr>\'\'\')\n    payload = json.dumps(dict(timeline=timeline, weeks=[r[\'gw\'] for r in plan[\'captaincy_plan\']]), ensure_ascii=False).replace(\'<\', \'\\\\u003c\')\n    p = plan[\'squad_plan\']\n    leaders = sorted(p[\'starting_xi\'], key=lambda x: -(number(x.get(\'projected_points\')) or 0))[:2]\n    extra = [\'Key assets: \' + \' and \'.join((x.get(\'spoken_name\', x[\'name\']) for x in leaders)), \'Opening formation: \' + p[\'formation\'], \'Main risk: projected minutes and availability can change.\']\n    extra.append(f"XI {p[\'xi_xpts\']:.1f} + captain = {p[\'captain_adjusted_xpts\']:.1f} projected points")\n    decision = plan.get(\'decision\') or {}\n    final = \'<section class="cs-final"><h2>FPL VORTEX verdict · \' + esc(decision.get(\'verdict\', \'HOLD\')) + \'</h2><p>\' + esc(decision.get(\'verdict_reason\', \'Verified baseline comparison required.\')) + \'</p>\' + \'\'.join((\'<p>\' + esc(x) + \'</p>\' for x in plan[\'summary_recommendations\'] + extra)) + \'</section>\'\n    return \'<style>\' + CSS + \'</style><section class="captaincy-showdown"><main>\' + \'\'.join(panels) + final + \'</main><aside><h2>\' + \'One-Gameweek captaincy plan\' + \'</h2><table><thead><tr><th>GW</th><th>Captain</th><th>Vice-captain</th><th>xPts</th><th>Main reason</th></tr></thead><tbody>\' + \'\'.join(table) + \'</tbody></table><p>Points before doubling. Candidates are planned starters. Review each deadline.</p></aside><script class="cs-data" type="application/json">\' + payload + \'</script></section>\'\nCSS = \'.captaincy-showdown{display:grid;grid-template-columns:60fr 40fr;gap:28px;height:100%;color:#f5f9ff}.captaincy-showdown main{position:relative;min-width:0}.cs-week,.cs-final{position:absolute;inset:0;visibility:hidden}.cs-week h2,.cs-final h2,.captaincy-showdown aside h2{font-size:40px;margin:15px 0 35px}.cs-three{display:grid;grid-template-columns:1fr 1.13fr 1fr;gap:20px;align-items:center;height:900px}.cs-card{min-width:0;background:linear-gradient(140deg,#17344b,#071522);border:2px solid #55728a;border-radius:18px;padding:25px 20px;text-align:center;box-sizing:border-box;min-height:760px}.cs-card.cs-pick{border:3px solid var(--accent-primary);min-height:840px;background:linear-gradient(140deg,#214b45,#081e26)}.cs-card>small{font-size:23px;color:var(--accent-secondary);font-weight:900}.cs-card img,.cs-avatar{width:150px;height:170px;object-fit:contain;margin:25px auto;display:block}.cs-avatar{font-size:60px;line-height:170px;background:#19374c;border-radius:30px}.cs-card h3{font-size:35px;min-height:80px;margin:18px 0}.cs-card>strong{font-size:88px;color:var(--accent-secondary)}.cs-card>p{font-size:21px}.cs-fixture{font-size:27px;min-height:75px}.cs-metrics{display:grid;gap:10px;text-align:left}.cs-metrics>div{font-size:22px;border-top:1px solid #526779;padding:8px 0}.cs-metrics b{float:right;color:#ffdf85}.cs-card footer{font-size:22px;line-height:1.25;margin-top:20px}.cs-reason{font-size:32px;line-height:1.35;padding:25px;background:#102b3c;border-left:5px solid var(--accent-primary)}.captaincy-showdown aside{min-width:0;background:#081c2eee;padding:25px;border:1px solid #55728a;border-radius:16px}.captaincy-showdown table{width:100%;border-collapse:collapse;table-layout:fixed;font-size:24px}.captaincy-showdown th,.captaincy-showdown td{padding:20px 12px;text-align:left;border-bottom:1px solid #446078;overflow-wrap:anywhere}.captaincy-showdown th:first-child{width:70px;white-space:nowrap}.captaincy-showdown td:first-child{white-space:nowrap}.captaincy-showdown th:last-child{width:29%}.captaincy-showdown tr[data-active="true"]{background:#275445;box-shadow:inset 5px 0 var(--accent-primary)}.captaincy-showdown aside>p{font-size:26px;line-height:1.4}.cs-final{padding:45px;background:#102d3dee;box-sizing:border-box}.cs-final p{font-size:31px;line-height:1.3;margin:20px 0}.cs-card[data-active="true"]{box-shadow:0 0 35px #c5f38655}\'\nSEEK_JS = "<script>(function(){const prev=window.seekWildcard;window.seekWildcard=function(t){prev(t);const root=document.querySelector(\'.captaincy-showdown\');if(!root)return;const d=JSON.parse(root.querySelector(\'.cs-data\').textContent),s=d.timeline.find(x=>t>=x.start&&t<x.end)||d.timeline.at(-1);const gw=s.gw|| (s.type===\'intro\'?d.weeks[0]:d.weeks.at(-1));for(const el of root.querySelectorAll(\'[data-cs-week]\'))el.style.visibility=+el.dataset.csWeek===gw&&s.type!==\'outro\'?\'visible\':\'hidden\';for(const el of root.querySelectorAll(\'[data-cs-row]\'))el.dataset.active=String(+el.dataset.csRow===gw);root.querySelector(\'.cs-final\').style.visibility=s.type===\'outro\'?\'visible\':\'hidden\';for(const el of root.querySelectorAll(\'[data-cs-player]\')){const cue=d.timeline.find(x=>x.gw===gw&&x.player_id===+el.dataset.csPlayer);const v=cue?Math.max(0,Math.min(1,(t-cue.start)/.4)):0;el.style.opacity=String(v);el.style.translate=`0 ${(1-Math.pow(v,.5))*30}px`;el.dataset.active=String(s.player_id===+el.dataset.csPlayer);}};})();</script>"\n',encoding='utf-8')
sys.modules.pop('captaincy_showdown',None)
(_vx_module_dir/'freehit_premium.py').write_text('"""Four Free Hit broadcast slides. One-week facts, concise narration and deterministic cues."""\nfrom __future__ import annotations\nimport copy,html,json,math,re,base64\nfrom pathlib import Path\nfrom collections import Counter\nVERSION=\'freehit-premium-four-slides-hero-flight-v2\'\nVOICE=dict(voice=\'en-GB-RyanNeural\',rate=\'+0%\',pitch=\'+0Hz\',volume=\'+0%\')\ndef num(v):\n try:\n  v=float(v);return v if math.isfinite(v) else None\n except (ValueError,TypeError):return None\ndef full_club(code,fallback=\'\'):\n from tts_narration import CLUBS\n return CLUBS.get(code,fallback or code)\ndef name(p):return p.get(\'spoken_name\') or p.get(\'player_name\') or p.get(\'full_name\') or p[\'name\']\ndef seg(identity,kind,text,**kw):return dict(segment_id=identity,type=kind,narration_text=text,start=None,end=None,player_ids=kw.pop(\'player_ids\',[]),**kw)\ndef align_timeline(items,duration,segment_durations=None):\n out=copy.deepcopy(items);weights=[len(s[\'narration_text\'].split()) for s in out];times=list(segment_durations) if segment_durations is not None else [duration*w/sum(weights) for w in weights]\n if len(times)!=len(out) or any(not math.isfinite(t) or t<=0 for t in times) or abs(sum(times)-duration)>.05:raise ValueError(\'Invalid measured Free Hit timing\')\n cursor=0.\n for s,t in zip(out,times):s.update(start=cursor,end=cursor+t,alignment=\'measured_segments\' if segment_durations is not None else \'estimated_word_counts\');cursor+=t\n return out\n\ndef difficulty(value):\n from vortex_wildcard import fdr_display\n return fdr_display(num(value))\ndef fixtures(p):return p.get(\'analysis\',{}).get(\'fixtures\',p.get(\'current_fixtures\',p.get(\'fixtures\',[])))\ndef fixture_words(f):return full_club(f.get(\'opponent\',\'\'),f.get(\'opponent_name\',\'\'))+(\' at home\' if f.get(\'home\') else \' away\')\ndef fx_words(p):return \' and \'.join(fixture_words(f) for f in fixtures(p)) or \'no published fixture\'\ndef short_reason(p):\n a=p.get(\'analysis\',{});pos=p.get(\'position\');fs=fixtures(p)\n if a.get(\'availability\',{}).get(\'flagged\'):return \'Monitor availability and expected minutes\'\n if len(fs)>1:return \'Multiple verified fixtures this Gameweek\'\n if pos in (\'GK\',\'DEF\'):\n  cs=num(a.get(\'clean_sheet_probability\'))\n  if cs is not None:return f\'{cs:.0%} model clean-sheet chance\'\n  return \'Defensive returns and minutes estimate\'\n if a.get(\'role\') in (\'First-choice penalty taker\',\'first-choice penalty taker\',\'First penalty taker\'):return \'Penalty role and attacking involvement\'\n if num(a.get(\'xgi90\')) is not None:return f\'{a["xgi90"]:.2f} expected goal involvement per 90\'\n return \'One-week projection and budget fit\'\n\ndef intro_case(d,bootstrap=None):\n from freehit_personal_intro import build_case\n return build_case(d,bootstrap)\n\ndef fixture_segments(case, selected=()):\n    """Preparation-stage speech; the detailed slide uses its own live player cards."""\n    targets = case[\'targets\']\n    out = [seg(\'fh_fixture_open\', \'intro\',\n        f"For Gameweek {case[\'gameweeks\'][0]}, I\'m considering {len(targets)} fixture targets. "\n        "I\'ll look at the matchups first, then decide which players fit my one-week squad.")]\n    for rank, row in enumerate(targets, 1):\n        scheduled = [fixture for week in row[\'weeks\'] for fixture in week[\'fixtures\']]\n        club = full_club(row[\'club\'], row[\'name\'])\n        text = f"I\'m looking at {club}. "\n        text += \' \'.join(f"They face {fixture_words(fixture)}." for fixture in scheduled)\n        goals = [num(fixture.get(\'team_goals\')) for fixture in scheduled]\n        clean_sheets = [num(fixture.get(\'clean_sheet_probability\')) for fixture in scheduled]\n        if scheduled and all(value is not None for value in goals):\n            text += f" My calculations put them at {sum(goals):.2f} expected team goals."\n        if clean_sheets and all(value is not None for value in clean_sheets):\n            probability = 1 - math.prod(1 - value for value in clean_sheets)\n            outcome = \'at least one clean sheet\' if len(scheduled) > 1 else \'a clean sheet\'\n            text += f" The estimated chance of {outcome} is {probability:.0%}."\n        out.append(seg(\'fh_team_\' + str(row[\'team_id\']), \'team\', text, team_id=row[\'team_id\']))\n        players = row.get(\'players\', [])[:2]\n        for player in players:\n            text = f"{player[\'player_name\']} is an option at £{player[\'price\']:.1f} million. "\n            text += f"He projects {player[\'projected_points\']:.1f} points, with roughly {player[\'expected_minutes\']:.0f} minutes expected."\n            if player[\'position\'] in (\'MID\', \'FWD\') and num(player.get(\'xgi90\')) is not None:\n                text += f" His expected goal involvement is {player[\'xgi90\']:.2f} per ninety minutes."\n            else:\n                text += " I\'ll compare his defensive opportunity with the other options at this price."\n            out.append(seg(\'fh_target_\' + str(player[\'player_id\']), \'player\', text,\n                team_id=row[\'team_id\'], player_id=player[\'player_id\'], player_ids=[player[\'player_id\']]))\n        preferred = next((player for player in players if player[\'player_id\'] == row.get(\'preferred_player_id\')), None)\n        if preferred:\n            text = f"{preferred[\'player_name\']} offers the most projected points per million in this comparison. "\n            text += "I\'ll still check the position, minutes and squad fit before choosing."\n            out.append(seg(\'fh_compare_\' + str(row[\'team_id\']), \'comparison\', text,\n                team_id=row[\'team_id\'], player_ids=[preferred[\'player_id\']]))\n    for row in case[\'avoids\']:\n        scheduled = [fixture for week in row[\'weeks\'] for fixture in week[\'fixtures\']]\n        club = full_club(row[\'club\'], row[\'name\'])\n        picked = [player for player in selected if player.get(\'team_id\') == row[\'team_id\']]\n        text = f"I\'m more cautious about {club}. "\n        text += \' \'.join(f"They face {fixture_words(fixture)}." for fixture in scheduled) if scheduled else \'They have no scheduled fixture this Gameweek.\'\n        if picked:\n            text += f" {name(picked[0])} still fits my draft on his individual projection."\n            text += " A difficult team fixture doesn\'t rule out every player."\n        else:\n            text += " I\'d compare the stronger one-week matchups first."\n        out.append(seg(\'fh_avoid_\' + str(row[\'team_id\']), \'avoid\', text, team_id=row[\'team_id\']))\n    out.append(seg(\'fh_fixture_end\', \'summary\',\n        "Those are the fixtures I\'m considering. "\n        "Next, I\'ll use minutes, player roles and prices to build a squad that fits the budget."))\n    return out\n\ndef squad_segments(p):\n    """Use the creator\'s script structure with the exact selected slide data."""\n    roles = (\'GK\', \'DEF\', \'MID\', \'FWD\')\n    def role(player):\n        return \'GK\' if player[\'position\'] == \'GKP\' else player[\'position\']\n    starters = sorted(p[\'starting_xi\'], key=lambda player:\n        (roles.index(role(player)), -float(player[\'projected_points\']), int(player[\'id\'])))\n    bench = list(p[\'bench\'])\n    if len(starters) != 11 or len(bench) != 4 or len({player[\'id\'] for player in starters + bench}) != 15:\n        raise ValueError(\'Squad speech requires the same eleven starters and four distinct substitutes as the slide\')\n    groups = {position: [player for player in starters if role(player) == position] for position in roles}\n    formation = \'-\'.join(str(len(groups[position])) for position in (\'DEF\', \'MID\', \'FWD\'))\n    if len(groups[\'GK\']) != 1 or formation != p[\'formation\']:\n        raise ValueError(\'Spoken formation differs from the selected pitch lineup\')\n    expected_xi = sum(float(player[\'projected_points\']) for player in starters)\n    captain = next((player for player in starters if player[\'id\'] == p[\'captain\'][\'id\']), None)\n    if captain is None or abs(float(captain[\'projected_points\']) - float(p[\'captain\'][\'projected_points\'])) > 1e-6:\n        raise ValueError(\'Spoken captain bonus does not match a selected starter\')\n    if (abs(expected_xi - float(p[\'xi_xpts\'])) > 1e-6\n            or abs(expected_xi + float(captain[\'projected_points\']) - float(p[\'captain_adjusted_xpts\'])) > 1e-6):\n        raise ValueError(\'Spoken totals must equal the starting XI plus exactly one captain bonus\')\n    expected_cost = sum(float(player[\'price\']) for player in starters + bench)\n    if abs(expected_cost - float(p[\'total_cost\'])) > 1e-6:\n        raise ValueError(\'Spoken squad cost differs from the selected fifteen prices\')\n    out = [seg(\'fh_squad_open\', \'title\',\n        f"This is my Free Hit for Gameweek {p[\'next_gw\']}: a {p[\'formation\']} formation, with £{p[\'total_cost\']:.1f} million committed. "\n        "I\'ll build the team from the back, starting in goal, then work through the defence, midfield and attack.")]\n    openings = {\'GK\': \'In goal, I have\', \'DEF\': \'In defence, I have\', \'MID\': \'In midfield, I have\', \'FWD\': \'Up front, I have\'}\n    for index, player in enumerate(starters):\n        position = role(player)\n        group = groups[position]\n        position_index = next(i for i, candidate in enumerate(group) if candidate[\'id\'] == player[\'id\'])\n        player_name = name(player)\n        club = full_club(player[\'club\'], player[\'club_name\'])\n        if position_index == 0:\n            lead = f"{openings[position]} {player_name} from {club}."\n        elif position_index == len(group) - 1:\n            closing = \'Partnering him up front is\' if position == \'FWD\' else \'Completing the midfield is\' if position == \'MID\' else \'Completing the back line is\'\n            lead = f"{closing} {player_name} of {club}."\n        elif position_index % 2:\n            lead = f"Alongside him, {player_name} gives me another option from {club}."\n        else:\n            lead = f"I also have {player_name}, representing {club}."\n        points = num(player.get(\'projected_points\'))\n        minutes = num(player.get(\'minutes\'))\n        analysis = player.get(\'analysis\', {})\n        price = float(player[\'price\'])\n        fixture = fx_words(player)\n        if index % 3 == 0:\n            detail = f" He faces {fixture}. At £{price:.1f} million, the model gives him {points:.1f} points"\n            detail += f", with around {minutes:.0f} minutes expected." if minutes is not None else \'.\'\n        elif index % 3 == 1:\n            detail = f" The fixture is {fixture}. I\'m spending £{price:.1f} million for a {points:.1f}-point projection."\n            if minutes is not None:\n                detail += f" His playing-time estimate is {minutes:.0f} minutes."\n        else:\n            detail = f" For £{price:.1f} million, he projects {points:.1f} points against {fixture}."\n            if minutes is not None:\n                detail += f" The model expects roughly {minutes:.0f} minutes."\n        clean_sheet = num(analysis.get(\'clean_sheet_probability\'))\n        goal_involvement = num(analysis.get(\'xgi90\'))\n        if position in (\'GK\', \'DEF\') and clean_sheet is not None:\n            detail += f" The defensive opportunity is a {clean_sheet:.0%} model clean-sheet chance."\n        elif position in (\'MID\', \'FWD\') and goal_involvement is not None:\n            detail += f" His underlying expected goal involvement is {goal_involvement:.2f} per ninety."\n        if analysis.get(\'availability\', {}).get(\'flagged\'):\n            detail += \' I will check his availability before the deadline.\'\n        elif minutes is not None and minutes < 75:\n            detail += \' Playing time is the point I would watch here.\'\n        out.append(seg(\'fh_squad_\' + str(player[\'id\']), \'players\', lead + detail,\n            player_ids=[player[\'id\']], hero_player_id=player[\'id\'], position=position,\n            position_label={\'GK\': \'IN GOAL\', \'DEF\': \'IN DEFENCE\', \'MID\': \'IN MIDFIELD\', \'FWD\': \'UP FRONT\'}[position],\n            reveal_order=index, flight_after_speech=True))\n    bench_openings = (\'On the bench, I have\', \'For further cover, I have\', \'Another reserve is\', \'Completing the bench is\')\n    for index, player in enumerate(bench):\n        points = num(player.get(\'projected_points\'))\n        minutes = num(player.get(\'minutes\'))\n        text = f"{bench_openings[index]} {name(player)} from {full_club(player[\'club\'], player[\'club_name\'])}, at £{player[\'price\']:.1f} million. "\n        text += f"He projects {points:.1f} points" if points is not None else \'His projection needs review\'\n        text += f" with about {minutes:.0f} minutes expected." if minutes is not None else \'.\'\n        text += \' This is substitute cover; his points are separate from the starting-eleven total.\'\n        out.append(seg(\'fh_squad_\' + str(player[\'id\']), \'bench\', text,\n            player_ids=[player[\'id\']], hero_player_id=player[\'id\'], position=role(player),\n            position_label=\'BENCH COVER\', reveal_order=len(starters) + index))\n    out.append(seg(\'fh_squad_total\', \'summary\',\n        f"That completes the team. The eleven project {p[\'xi_xpts\']:.1f} points before captaincy. "\n        f"One additional captain score of {captain[\'projected_points\']:.1f} takes the total to {p[\'captain_adjusted_xpts\']:.1f}. "\n        "The bench stays outside that figure. Let\'s give the armband decision its own comparison."))\n    return out\n\ndef captain_plan(d):\n p=d[\'slide3_plan\'];cap=p[\'captain\'];vice=p[\'vice_captain\']\n eligible=sorted((x for x in p[\'starting_xi\'] if num(x.get(\'projected_points\')) is not None),key=lambda x:(-x[\'projected_points\'],x[\'id\']))\n if len(eligible)!=11 or cap[\'id\']!=eligible[0][\'id\']:\n  raise ValueError(\'Free Hit captain must lead the projected points of the selected legal XI\')\n if vice[\'id\']==cap[\'id\'] or vice[\'id\'] not in {x[\'id\'] for x in eligible}:\n  raise ValueError(\'Free Hit vice-captain must be a different player in the starting XI\')\n if num(cap.get(\'minutes\')) is None or num(vice.get(\'minutes\')) is None:\n  raise ValueError(\'Cannot verify captain minutes: regenerate the Free Hit squad\')\n if num(p.get(\'captain_adjusted_xpts\')) is None or abs(p[\'captain_adjusted_xpts\']-p[\'xi_xpts\']-cap[\'projected_points\'])>1e-6:\n  raise ValueError(\'Captain bonus must be counted exactly once\')\n # Include the actual chosen vice alongside the top non-captain alternative.\n alternatives=[x for x in eligible if x[\'id\'] not in (cap[\'id\'],vice[\'id\'])]\n if not alternatives:raise ValueError(\'Insufficient Free Hit captain alternatives\')\n return dict(version=VERSION,chip=\'freehit\',next_gw=d[\'gameweek\'],captain=cap,vice_captain=vice,candidates=[alternatives[0],cap,vice],squad_plan=p,decision=d[\'freehit_case\'])\n\ndef final_segments(plan):\n    """Creator\'s armband script, populated from the selected squad and card data."""\n    squad = plan[\'squad_plan\']\n    starters = squad[\'starting_xi\']\n    by_id = {player[\'id\']: player for player in starters}\n    if len(starters) != 11 or len(by_id) != 11:\n        raise ValueError(\'Captaincy narration requires the selected starting eleven\')\n    captain_id, vice_id = plan[\'captain\'][\'id\'], plan[\'vice_captain\'][\'id\']\n    if captain_id == vice_id or captain_id not in by_id or vice_id not in by_id:\n        raise ValueError(\'Captain and vice-captain must be distinct selected starters\')\n    if (squad[\'captain\'][\'id\'], squad[\'vice_captain\'][\'id\']) != (captain_id, vice_id):\n        raise ValueError(\'Spoken armbands differ from the slide armbands\')\n    captain, vice = by_id[captain_id], by_id[vice_id]\n    for player in starters:\n        if num(player.get(\'projected_points\')) is None or num(player.get(\'minutes\')) is None or num(player.get(\'price\')) is None or player[\'price\'] <= 0:\n            raise ValueError(\'Captaincy speech needs verified points, minutes and price data\')\n    if captain[\'projected_points\'] < max(player[\'projected_points\'] for player in starters):\n        raise ValueError(\'The spoken points leader must match the selected captain\')\n    xi_points = sum(player[\'projected_points\'] for player in starters)\n    if (num(squad.get(\'xi_xpts\')) is None or num(squad.get(\'captain_adjusted_xpts\')) is None\n            or abs(xi_points - squad[\'xi_xpts\']) > 1e-6\n            or abs(xi_points + captain[\'projected_points\'] - squad[\'captain_adjusted_xpts\']) > 1e-6):\n        raise ValueError(\'Captaincy total must equal the XI plus exactly one captain score\')\n    other_ids = [player[\'id\'] for player in plan[\'candidates\'] if player[\'id\'] not in (captain_id, vice_id)]\n    candidate_ids = other_ids + [vice_id, captain_id]\n    if len(candidate_ids) != len(set(candidate_ids)) or any(identity not in by_id for identity in candidate_ids):\n        raise ValueError(\'Every compared card must be a distinct selected starter\')\n    candidates = [by_id[identity] for identity in candidate_ids]\n    count_words = {1: \'one\', 2: \'two\', 3: \'three\', 4: \'four\', 5: \'five\'}\n    count = count_words.get(len(candidates), str(len(candidates)))\n    out = [seg(\'fh_captain_open\', \'intro\',\n        f"Let\'s settle the armband for Gameweek {plan[\'next_gw\']}. "\n        f"I\'ll compare {count} players from my starting eleven, then make the call.")]\n    for index, player in enumerate(candidates):\n        lead = \'The first option is\' if index == 0 else \'Completing the comparison is\' if index == len(candidates) - 1 else \'Another strong option is\'\n        text = f"{lead} {name(player)}, facing {fx_words(player)}. "\n        text += f"His projection is {player[\'projected_points\']:.1f} points, with around {player[\'minutes\']:.0f} minutes expected. "\n        text += f"Giving him the armband adds another {player[\'projected_points\']:.1f} projected points to the eleven. "\n        text += "That doubles his own score, rather than the team\'s total."\n        out.append(seg(\'fh_contender_\' + str(player[\'id\']), \'candidate\', text,\n            player_id=player[\'id\'], hero_player_id=player[\'id\'], player_ids=[player[\'id\']], slot=index))\n    out.append(seg(\'fh_captain_decision\', \'decision\',\n        f"My captain is {name(captain)}. He leads this eleven on the current projection. "\n        f"The captain bonus is {captain[\'projected_points\']:.1f}, counted once on top of the starting score.",\n        hero_player_id=captain_id, player_ids=[captain_id]))\n    out.append(seg(\'fh_vice\', \'vice\',\n        f"The vice-captain armband goes to {name(vice)}. "\n        "His extra score only comes into play if my captain makes no appearance. "\n        "I\'ll check both players\' availability before the deadline.", hero_player_id=vice_id, player_ids=[vice_id]))\n    value = max(starters, key=lambda player: player[\'projected_points\'] / player[\'price\'])\n    risk = min(starters, key=lambda player: (player[\'minutes\'], player.get(\'starting_probability\') or 0))\n    out.append(seg(\'fh_value\', \'takeaways\',\n        f"One final look at the squad. {name(value)} offers the strongest projected value for his price.",\n        hero_player_id=value[\'id\'], player_ids=[value[\'id\']]))\n    out.append(seg(\'fh_minutes_check\', \'takeaways\',\n        f"{name(risk)} has the lowest expected minutes, so he deserves an availability check.",\n        hero_player_id=risk[\'id\'], player_ids=[risk[\'id\']]))\n    out.append(seg(\'fh_captain_total\', \'takeaways\',\n        f"Including one captain bonus, the team projects {squad[\'captain_adjusted_xpts\']:.1f} points.",\n        hero_player_id=captain_id, player_ids=[captain_id]))\n    decision = plan[\'decision\']\n    text = decision[\'verdict_reason\'] if decision.get(\'scenario_comparison\') else (\n        \'The squad is ready. A verified current-team comparison is still needed before deciding whether the chip is worth using.\')\n    text += \' My original squad returns next Gameweek. The last step is checking the final team news.\'\n    out.append(seg(\'fh_final\', \'outro\', text, hero_player_id=captain_id, player_ids=[captain_id]))\n    return out\n\ndef prepare(d,bootstrap=None):\n d=copy.deepcopy(d);p=d[\'slide3_plan\'];official={int(x[\'id\']):x for x in (bootstrap or {}).get(\'elements\',[])}\n for group in [\'squad\',\'starting_xi\',\'bench\']:\n  for x in p[group]:\n   # Prefer concise common FPL names over long registered names.\n   e=official.get(int(x[\'id\']),{});first=str(e.get(\'first_name\',\'\')).split();web=x[\'name\']\n   x[\'spoken_name\']=web if len(web.split())>=2 else ((first[0]+\' \'+web) if first and first[0].casefold()!=web.casefold() else first[0]+\' \'+str(e.get(\'second_name\',\'\')).split()[-1] if first and str(e.get(\'second_name\',\'\')).strip() else x.get(\'spoken_name\',web))\n   x[\'club_name\']=full_club(x[\'club\'],x.get(\'club_name\',\'\'))\n byid={x[\'id\']:x for x in p[\'squad\']};p[\'captain\']=byid[p[\'captain\'][\'id\']];p[\'vice_captain\']=byid[p[\'vice_captain\'][\'id\']]\n p[\'narration_timeline\']=squad_segments(p);d[\'freehit_case\']=intro_case(d,bootstrap)\n d[\'fixture_case\'][\'narration_timeline\']=fixture_segments(d[\'fixture_case\'],p[\'starting_xi\']);d[\'fixture_case\'][\'narration_text\']=\' \'.join(x[\'narration_text\'] for x in d[\'fixture_case\'][\'narration_timeline\'])\n d[\'captain_pick\']=captain_plan(d);return d\n\ndef uri(path):\n p=Path(path) if path else None\n return \'data:image/png;base64,\'+base64.b64encode(p.read_bytes()).decode() if p and p.is_file() else \'\'\ndef render_slide(kind,d,timeline,media=None,badges=None,approved_cards=None):\n esc=lambda x:html.escape(str(x),quote=True);media=media or {};badges=badges or {};approved_cards=approved_cards or {};p=d[\'slide3_plan\'];gw=d[\'gameweek\']\n if kind==1:\n  case=d[\'freehit_case\'];labels=\' • \'.join(case.get(\'intro_indicators\',[]))\n  return \'<style>.fh-personal{color:#f2f6ff;padding:80px;font:42px Arial}.fh-personal h1{font-size:86px}.fh-personal h2{font-size:68px;color:#55e9ad}</style><section class="fh-personal"><h1>FREE HIT: NOW OR WAIT?</h1><p>\'+esc(labels)+\'</p><h2>MY FREE HIT DRAFT</h2><p>GAMEWEEK \'+str(gw)+\' ONLY</p><p>Your original squad returns next Gameweek</p><footer>NEXT: THIS WEEK’S FIXTURES</footer></section>\'\n\n def logo(code,cls=\'fp-logo\'):\n  src=uri(badges.get(code,{}).get(\'path\'))\n  return f\'<img class="{cls}" src="{src}" alt="{esc(full_club(code))} badge">\' if src else f\'<span class="{cls} fp-logo-fallback">{esc(code)}</span>\'\n def photo(x,cls=\'fp-photo\'):\n  pid=x.get(\'id\',x.get(\'player_id\'));record=media.get(str(pid),media.get(pid,{}));src=uri(record.get(\'photo\')) if record.get(\'photo_kind\')==\'player\' else \'\'\n  return f\'<img class="{cls}" src="{src}" alt="{esc(name(x))}">\' if src else f\'<div class="{cls} fp-avatar">{logo(x.get("club",""))}<b>{esc("".join(v[0] for v in name(x).split()[:2]))}</b></div>\'\n def fdr(f):\n  style=f.get(\'fdr_style\') or difficulty(f.get(\'fdr\'));return f\'<span class="fp-fdr" style="background:{style["background"]};color:{style["text"]}">{esc(style["label"])}</span>\'\n def fixture(x):return \' / \'.join(esc(f.get(\'opponent\',\'\'))+(\' [H]\' if f.get(\'home\') else \' [A]\')+\' \'+fdr(f) for f in fixtures(x)) or \'FIXTURE UNAVAILABLE\'\n def minis(xs):return \'<div class="fp-intro-minis">\'+\'\'.join(\'<article>\'+photo(x)+\'<b>\'+esc(x[\'name\'])+\'</b></article>\' for x in xs)+\'</div>\'\n def metric(label,value):return f\'<div class="fp-metric"><small>{label}</small><b>{esc(value)}</b></div>\'\n def approved_card(player):\n  pid=int(player.get(\'id\',player.get(\'player_id\')))\n  path=approved_cards.get(pid) or approved_cards.get(str(pid))\n  if not path or not Path(path).is_file():raise FileNotFoundError(f\'Original notebook Hero Card is missing for player {pid}\')\n  return f\'<img class="fp-approved-card" src="{uri(path)}" alt="Original notebook hero card for {esc(name(player))}">\'\n def shortmetrics(x, compact_labels=False):\n  a=x.get(\'analysis\',{});val=a.get(\'clean_sheet_probability\') if x.get(\'position\') in (\'GK\',\'DEF\') else a.get(\'xgi90\');label=\'Clean-sheet chance\' if x.get(\'position\') in (\'GK\',\'DEF\') else \'Expected goal involvement / 90\';fmt=(f\'{val:.0%}\' if x.get(\'position\') in (\'GK\',\'DEF\') else f\'{val:.2f}\') if val is not None else \'UNAVAILABLE\'\n  if compact_labels:\n   label=\'CLEAN SHEET %\' if x.get(\'position\') in (\'GK\',\'DEF\') else \'xGI / 90\'\n  return metric(\'PROJECTED PTS\' if compact_labels else \'Projected points\',f\'{x["projected_points"]:.1f}\' if num(x.get(\'projected_points\')) is not None else \'UNAVAILABLE\')+metric(\'EXPECTED MIN\' if compact_labels else \'Expected minutes\',f\'{x["minutes"]:.0f}\' if num(x.get(\'minutes\')) is not None else \'UNAVAILABLE\')+metric(label,fmt)\n if kind==1:\n  c=d[\'freehit_case\'];r=c[\'focus_team\'];fs=[f for w in r[\'weeks\'] for f in w[\'fixtures\']];goals=sum(f[\'team_goals\'] for f in fs) if fs and all(num(f.get(\'team_goals\')) is not None for f in fs) else None;cs=fs[0].get(\'clean_sheet_probability\') if len(fs)==1 else None;cap=p[\'captain\'];comp=c[\'scenario_comparison\']\n  comparison=f\'CURRENT {comp["current"]:.1f} · FREE HIT {comp["freehit"]:.1f} · GAIN {comp["gain"]:+.1f}\' if comp else \'VERIFIED CURRENT-TEAM BASELINE REQUIRED\'\n  content=f\'<header class="fp-section-header"><h2>ONE GAMEWEEK. ONE CHIP DECISION.</h2><p>Build the opportunity, then assess the benefit</p></header><div class="fp-points" data-fp-focus="comparison">{metric("Starting eleven",(""+format(p["xi_xpts"],".1f")+""))}{metric("Captain bonus",("+"+format(cap["projected_points"],".1f")+""))}{metric("Total projected points",(""+format(p["captain_adjusted_xpts"],".1f")+""))}<div><b>{comparison}</b><p>Bench excluded. Captain bonus counted once.</p></div></div><div class="fp-intro-grid"><section class="fp-panel" data-fp-focus="fixture"><h2>FIXTURE OPPORTUNITY</h2><div class="fp-team-title">{logo(r["club"])}<b>{esc(full_club(r["club"],r["name"]))}</b></div><div class="fp-intro-fixtures">\'+\'\'.join(f\'<div>{logo(f["opponent"])}<b>{esc(fixture_words(f))}</b>{fdr(f)}</div>\' for f in fs)+f\'</div><div class="fp-metrics">{metric("Projected team goals",f"{goals:.2f}" if goals is not None else "UNAVAILABLE")}{metric("Clean-sheet chance",f"{cs:.0%}" if cs is not None else "UNAVAILABLE")}</div><p class="fp-note">Model estimates for Gameweek {gw}</p>{minis([x for x in p["starting_xi"] if x["team_id"]==r["team_id"]])}</section><section class="fp-panel" data-fp-focus="structure"><h2>SQUAD STRUCTURE</h2><strong class="fp-big">{p["formation"]}</strong><div class="fp-metrics">{metric("Squad cost",("\\u00a3"+format(p["total_cost"],".1f")+"M"))}{metric("In the bank",("\\u00a3"+format(p["bank"],".1f")+"M"))}</div><ul><li>Eleven starters</li><li>Low-cost bench cover</li><li>Original team returns next week</li></ul>{minis(p["bench"])}</section><section class="fp-panel fp-intro-captain" data-fp-focus="captain"><h2>CAPTAINCY OPPORTUNITY</h2>{photo(cap)}<h3>{esc(name(cap))}</h3><div class="fp-club">{logo(cap["club"])}{esc(cap["club_name"])}</div><div class="fp-intro-fixtures">{fixture(cap)}</div><div class="fp-metrics">{metric("Projected points",(""+format(cap["projected_points"],".1f")+""))}{metric("Expected minutes",(""+format(cap["minutes"],".0f")+""))}</div></section></div><footer class="fp-decision" data-fp-focus="decision"><b>{c["decision_label"]}</b><p>\'+(\'Compare available free transfers before spending the chip.\' if not comp else esc(c[\'verdict_reason\']))+\'</p></footer>\'\n elif kind==2:\n  case=d[\'fixture_case\'];targets={r[\'team_id\']:i+1 for i,r in enumerate(case[\'targets\'])};avoids={r[\'team_id\'] for r in case[\'avoids\']};rows=[];panels=[]\n  for row in case[\'rows\']:\n   fs=[f for w in row[\'weeks\'] for f in w[\'fixtures\']];goals=sum(f[\'team_goals\'] for f in fs) if fs and all(num(f.get(\'team_goals\')) is not None for f in fs) else None;cs=fs[0].get(\'clean_sheet_probability\') if len(fs)==1 else None\n   cell=\' / \'.join(esc(f[\'opponent\'])+(\'[H]-\' if f[\'home\'] else \'[A]-\')+str(difficulty(f[\'fdr\'])[\'label\']) for f in fs) or \'BLANK\';style=difficulty(row.get(\'average_fdr\'));tag=f\'T{targets[row["team_id"]]}\' if row[\'team_id\'] in targets else \'AVOID\' if row[\'team_id\'] in avoids else \'\'\n   rows.append(f\'<tr data-fp-team="{row["team_id"]}"><td><span class="fp-rank">{tag}</span>{logo(row["club"])}<b>{esc(full_club(row["club"],row["name"]))}</b></td><td><span class="fp-fixture-cell" style="background:{style["background"]};color:{style["text"]}">{cell}</span></td><td>{f"{goals:.2f}" if goals is not None else "—"}</td><td>{f"{cs:.0%}" if cs is not None else "—"}</td></tr>\')\n  for row in case[\'targets\']:\n   players=[]\n   for x in row[\'players\'][:2]:\n    players.append(f\'<article class="fp-shortlist" data-fp-player="{x["player_id"]}">{photo(x)}<div><h3>{esc(x["player_name"])}</h3><b>£{x["price"]:.1f}M</b><div class="fp-metrics">{metric("Projected points",(""+format(x["projected_points"],".1f")+""))}{metric("Expected minutes",(""+format(x["expected_minutes"],".0f")+""))}</div><p>\'+(\'Expected goal involvement / 90: \'+f\'{x["xgi90"]:.2f}\' if num(x.get(\'xgi90\')) is not None else \'Underlying involvement unavailable\')+\'</p></div></article>\')\n   preferred=next((x for x in row[\'players\'][:2] if x[\'player_id\']==row.get(\'preferred_player_id\')),None)\n   preference=(\'<b>PREFERRED VALUE: \'+esc(preferred[\'player_name\'])+\'</b><span>\'+f"{preferred[\'value\']:.2f}"+\' PROJECTED POINTS PER MILLION</span>\') if preferred else \'\'\n   panels.append(f\'<section class="fp-team-detail" data-fp-detail="{row["team_id"]}"><header>{logo(row["club"])}<div><small>TARGET {targets[row["team_id"]]}</small><h2>{esc(full_club(row["club"],row["name"]))}</h2></div></header>\'+\'\'.join(players)+f\'<div class="fp-detail-note">{preference}<p>Shortlist candidates. Check minutes and full squad budget.</p></div></section>\')\n  summary=\'<section class="fp-fixture-summary"><h2>FIVE TEAMS TO TARGET</h2>\'+\'\'.join(f\'<div>{logo(r["club"])}<b>{esc(full_club(r["club"],r["name"]))}</b></div>\' for r in case[\'targets\'])+\'<h2>THREE CAUTIOUS MATCHUPS</h2>\'+\'\'.join(f\'<div>{logo(r["club"])}<b>{esc(full_club(r["club"],r["name"]))}</b></div>\' for r in case[\'avoids\'])+\'<p>Fixture rank guides the shortlist. Individual returns decide the final picks.</p></section>\'\n  content=f\'<header class="fp-section-header"><h2>GAMEWEEK {gw} FIXTURE LANDSCAPE</h2><p>Five targets. Three cautious matchups.</p></header><div class="fp-fixture-layout"><main class="fp-panel fp-ticker"><table><thead><tr><th>TEAM</th><th>FIXTURE / DIFFICULTY</th><th>PROJECTED<br>GOALS</th><th>CLEAN<br>SHEET</th></tr></thead><tbody>\'+\'\'.join(rows)+\'</tbody></table><p>One-week team model. Colours show fixture difficulty.</p></main><aside class="fp-panel fp-details">\'+\'\'.join(panels)+summary+\'</aside></div>\'\n elif kind==3:\n  starters=p[\'starting_xi\'];bypos={pos:[x for x in starters if x[\'position\']==pos] for pos in (\'GK\',\'DEF\',\'MID\',\'FWD\')};nodes=[];heroes=[];bench=[]\n  for pos,ys in [(\'GK\',14),(\'DEF\',38),(\'MID\',62),(\'FWD\',85)]:\n   for i,x in enumerate(bypos[pos]):\n    left=(i+1)*100/(len(bypos[pos])+1);label=\'C\' if x[\'id\']==p[\'captain\'][\'id\'] else \'VC\' if x[\'id\']==p[\'vice_captain\'][\'id\'] else \'\'\n    nodes.append(f\'<div class="fp-pitch-player" data-fp-pitch="{x["id"]}" style="left:{left}%;top:{ys}%">{photo(x)}<b>{esc(x["name"])} <em>{label}</em></b><span>{pos} · £{x["price"]:.1f}M</span></div>\')\n  for x in p[\'bench\']:bench.append(f\'<article data-fp-bench="{x["id"]}">{photo(x)}<b>{esc(x["name"])}</b><span>{x["position"]} · £{x["price"]:.1f}M</span></article>\')\n  for x in sorted(p[\'starting_xi\'],key=lambda x:(\'GK\',\'DEF\',\'MID\',\'FWD\').index(x[\'position\']))+sorted(p[\'bench\'],key=lambda x:x[\'position\']!=\'GK\'):\n   heroes.append(f\'<article class="fp-player-hero" data-fp-hero="{x["id"]}">{approved_card(x)}<div class="fp-card-meta"><b>{esc(name(x))}</b><span>{fixture(x)} · {x["projected_points"]:.1f} xPts</span></div><div class="fp-why-popup"><strong>WHY THIS FREE HIT PICK</strong><p>{esc(short_reason(x))}</p><small>{x["position"]} · £{x["price"]:.1f}M · {x["projected_points"]:.1f} xPts</small></div></article>\')\n  # Present only the selected 11 in the primary visualization; bench cover stays secondary.\n  best_projection=max(max(float(x[\'projected_points\']),0.0) for x in starters) or 1.0\n  top_ids={x[\'id\'] for x in sorted(starters,key=lambda x:(-float(x[\'projected_points\']),x[\'id\']))[:3]}\n  xi_cards=[]\n  for x in sorted(starters,key=lambda x:((\'GK\',\'DEF\',\'MID\',\'FWD\').index(x[\'position\']),-float(x[\'projected_points\']),x[\'id\'])):\n   percentage=max(0.0,min(100.0,100.0*float(x[\'projected_points\'])/best_projection))\n   highlight=\' fp-xpts-leader\' if x[\'id\'] in top_ids else \'\'\n   badge=\'C\' if x[\'id\']==p[\'captain\'][\'id\'] else \'VC\' if x[\'id\']==p[\'vice_captain\'][\'id\'] else \'\'\n   xi_cards.append(f\'<article class="fp-xpts-player{highlight}" data-fp-squad-row="{x["id"]}"><div class="fp-xpts-player-head"><span class="fp-xpts-role">{x["position"]}</span><b title="{esc(x["name"])}">{esc(x["name"])}</b><em>{badge}</em><strong>{x["projected_points"]:.1f}<small> xPts</small></strong></div><div class="fp-xpts-track"><div style="width:{percentage:.2f}%"></div></div></article>\')\n  bench_cards=[]\n  for index,x in enumerate(p[\'bench\'],1):\n   order=\'GK\' if x[\'position\']==\'GK\' else f\'SUB {index - (1 if any(b["position"]=="GK" for b in p["bench"][:index]) else 0)}\'\n   minutes_watch=\' · MINUTES WATCH\' if not x.get(\'secure\',True) else \'\'\n   bench_cards.append(f\'<article class="fp-xpts-sub" data-fp-squad-row="{x["id"]}"><small>{esc(order)} · {x["position"]}{minutes_watch}</small><b title="{esc(x["name"])}">{esc(x["name"])}</b><strong>{x["projected_points"]:.1f}<small> xPts</small></strong></article>\')\n  flights=[]\n  for x in starters:\n   flights.append(f\'<article class="fp-card-flight" data-fp-flight="{x["id"]}">{approved_card(x)}<b>{esc(x["name"])}</b><small>{x["projected_points"]:.1f} xPTS · {esc(x["club"])}</small></article>\')\n  starter_cards_html=\'\'.join(xi_cards)\n  bench_cards_html=\'\'.join(bench_cards)\n  breakdown=f\'<section class="fp-panel fp-xpts-dashboard" aria-label="Starting XI xPts breakdown"><header class="fp-xpts-dashboard-header"><h2>STARTING XI / xPTS BREAKDOWN</h2><span>11 STARTERS · {esc(p["formation"])} · GW {gw}</span></header><div class="fp-xpts-grid">{starter_cards_html}</div><section class="fp-xpts-bench"><h3>BENCH COVER <span>SEPARATE FROM XI</span></h3><div>{bench_cards_html}</div></section><footer class="fp-xpts-total"><b>XI {p["xi_xpts"]:.1f} <span>+</span> CAPTAIN {p["captain"]["projected_points"]:.1f} <span>=</span> {p["captain_adjusted_xpts"]:.1f}</b><small>BENCH EXCLUDED · CAPTAIN BONUS ONCE</small></footer></section>\'\n  content=f\'<div class="fp-squad-layout"><main class="fp-pitch-side"><header class="fp-section-header"><h2>FREE HIT STARTING XI</h2><b>{p["formation"]}</b></header><div class="fp-pitch"><div class="fp-boundary"></div><div class="fp-halfway"></div><div class="fp-circle"></div><div class="fp-top-goal"></div><div class="fp-top-box"></div>\'+\'\'.join(nodes)+\'</div><section class="fp-bench"><h3>BENCH — MINIMUM COVER</h3><div>\'+\'\'.join(bench)+f\'</div></section></main><aside class="fp-squad-side"><header class="fp-section-header"><h2>ONE-WEEK PLAN</h2><b>£{p["bank"]:.1f}M BANK</b></header><section class="fp-hero-stage">\'+\'\'.join(heroes)+f\'<article class="fp-squad-overview"><h2>THE SELECTED FIFTEEN</h2><div class="fp-metrics">{metric("Squad cost",("\\u00a3"+format(p["total_cost"],".1f")+"M"))}{metric("XI projection",(""+format(p["xi_xpts"],".1f")+""))}{metric("With captain",(""+format(p["captain_adjusted_xpts"],".1f")+""))}</div><p>Captain: {esc(name(p["captain"]))}</p><p>Vice-captain: {esc(name(p["vice_captain"]))}</p><p>One Gameweek only. Original squad returns afterwards.</p></article></section>{breakdown}</aside></div>\'+\'\'.join(flights)\n else:\n  plan=d[\'captain_pick\'];cards=[]\n  for i,x in enumerate(plan[\'candidates\']):\n   cards.append(f\'<article class="fp-captain-card {"fp-vortex-pick" if i==1 else ""}" data-fp-contender="{x["id"]}"><small>{"CAPTAIN · BEST GAMEWEEK xPts" if i==1 else "VICE-CAPTAIN" if x["id"]==plan["vice_captain"]["id"] else "ALTERNATIVE"}</small>{photo(x)}<h2>{esc(name(x))}</h2><div class="fp-club">{logo(x["club"])}{esc(x["club_name"])}</div><div class="fp-captain-fixture">{fixture(x)}</div><div class="fp-metrics">{shortmetrics(x, compact_labels=True)}</div><p>{esc(short_reason(x))}</p></article>\')\n  value=max(p[\'starting_xi\'],key=lambda x:x[\'projected_points\']/x[\'price\']);risk=min(p[\'starting_xi\'],key=lambda x:(x.get(\'starting_probability\') or 0,x.get(\'minutes\') or 0));c=d[\'freehit_case\'];v=p[\'vice_captain\']\n  visible_hero_ids={x[\'id\']:x for x in plan[\'candidates\']}\n  visible_hero_ids[v[\'id\']]=v\n  right_heroes=\'\'.join(f\'<figure class="fp-approved-captain" data-fp-captain-hero="{x["id"]}">{approved_card(x)}</figure>\' for x in visible_hero_ids.values())\n  content=f\'<header class="fp-section-header"><h2>GAMEWEEK {gw} · CAPTAINCY CALL</h2><p>One-week xPts · +{p["captain"]["projected_points"]:.1f} captain bonus, counted once</p></header><div class="fp-final-layout"><main class="fp-captain-three">\'+\'\'.join(cards)+f\'</main><aside class="fp-final-takeaways"><div class="fp-approved-hero-stage">\'+right_heroes+\'</div><section class="fp-panel fp-vice" data-fp-focus="vice"><h2>VICE-CAPTAIN</h2><header>{photo(v)}<div><h3>{esc(name(v))}</h3><div class="fp-club">{logo(v["club"])}{esc(v["club_name"])}</div></div></header><div class="fp-vice-projection"><strong>{v["projected_points"]:.1f} xPts</strong><span>{v["minutes"]:.0f} expected minutes</span></div><p>Recheck fitness and expected minutes</p></section><section class="fp-panel" data-fp-focus="takeaways"><h2>FINAL TAKEAWAYS</h2><div class="fp-takeaway"><small>BEST VALUE · xPts / £m</small><b>{esc(name(value))}</b><span>{value["projected_points"]/value["price"]:.2f} xPts / £m · £{value["price"]:.1f}M</span></div><div class="fp-takeaway"><small>MINUTES TO REVIEW</small><b>{esc(name(risk))}</b><span>{risk["minutes"]:.0f} estimated minutes · Review team news</span></div><div class="fp-metrics">{metric("XI with captain",(""+format(p["captain_adjusted_xpts"],".1f")+""))}</div></section></aside></div><footer class="fp-decision" data-fp-focus="outro"><b>{c["decision_label"]}</b><p>Original squad returns next week. Check team news before the deadline.</p></footer>\'\n payload=json.dumps(dict(kind=kind,timeline=timeline,captain_id=p[\'captain\'][\'id\']),ensure_ascii=False).replace(\'<\',\'\\\\u003c\')\n return \'<style>\'+CSS+\'</style><section class="fp-slide fp-slide-\'+str(kind)+\'">\'+content+\'<script class="fp-data" type="application/json">\'+payload+\'</script></section>\'\n\nCSS=\'\'\'\n#content .fp-slide{height:100%;width:100%;font-family:Arial,sans-serif;color:#f5f9ff;text-transform:uppercase;overflow:hidden;display:grid;gap:22px}.fp-slide *{box-sizing:border-box;min-width:0}.fp-slide h2,.fp-slide h3,.fp-slide p{margin:0;padding:0;min-height:0;height:auto}.fp-section-header{display:flex;align-items:center;justify-content:space-between;gap:24px;padding:20px 28px;border-left:6px solid #95deff;background:#071a2df2}.fp-section-header h2{font-size:48px;line-height:1.1;color:#fff}.fp-section-header p,.fp-section-header>b{font-size:35px;line-height:1.15;color:#b7e6fa}.fp-panel{border:2px solid #4c708b;border-radius:17px;background:#081b2ef0;padding:30px;overflow:hidden;min-height:0}.fp-panel h2{font-size:42px;color:#8ddfff;line-height:1.1;margin-bottom:25px}.fp-panel h3{font-size:45px;line-height:1.1}.fp-logo{width:65px;height:65px;object-fit:contain;flex-shrink:0}.fp-photo{width:210px;height:250px;object-fit:contain;flex-shrink:0}.fp-club{display:flex;align-items:center;gap:16px;font-size:32px;color:#cbe8f6;line-height:1.1}.fp-club .fp-logo{width:48px;height:48px}.fp-metrics{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:18px}.fp-metric{background:#173349;border:1px solid #3f637b;border-radius:10px;padding:20px 22px;min-height:0}.fp-metric>small{display:block;font-size:30px;line-height:1.12;color:#beddeb;font-weight:700}.fp-metric>b{display:block;font-size:66px;line-height:1.1;color:#fff;margin-top:12px}.fp-fdr{display:inline-flex;align-items:center;justify-content:center;padding:6px 14px;min-width:65px;border-radius:6px;font-size:33px;font-weight:900;vertical-align:middle}.fp-note{font-size:31px;color:#badbea;margin-top:22px!important}.fp-decision{display:flex;align-items:center;justify-content:space-between;gap:28px;border:2px solid #88bf9f;border-radius:15px;padding:24px 30px;background:#103b37;min-height:0}.fp-decision>b{font-size:43px;color:#d4ff91;line-height:1.1}.fp-decision>p{font-size:34px;line-height:1.15;color:#fff;max-width:65%}.fp-avatar{display:flex;flex-direction:column;align-items:center;justify-content:center;background:#234151;border-radius:16px}.fp-avatar b{font-size:55px}.fp-logo-fallback{display:grid;place-items:center;background:#234151;border-radius:9px;font-size:23px;font-weight:900}.fp-active{border-color:#d4ee94!important;box-shadow:inset 0 0 20px #c0ff7518,0 0 13px #d9ff8c33}\n.fp-slide-1{grid-template-rows:110px 210px minmax(0,1fr) 150px}.fp-points{display:grid;grid-template-columns:1fr 1fr 1.2fr 2.2fr;gap:22px;min-height:0}.fp-points>.fp-metric{padding:24px 28px}.fp-points>.fp-metric>b{font-size:91px;color:#dbff9d;margin-top:15px}.fp-points>div:last-child{display:flex;flex-direction:column;justify-content:center;background:#10283ced;padding:24px;border-radius:13px;border:2px solid #547486}.fp-points>div:last-child>b{font-size:36px;color:#ffe49a;line-height:1.2}.fp-points>div:last-child>p{font-size:30px;line-height:1.2;margin-top:16px;color:#c9e6f5}.fp-intro-grid{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:24px;min-height:0}.fp-team-title{display:flex;align-items:center;gap:28px;margin:48px 0 38px}.fp-team-title>.fp-logo{width:145px;height:145px}.fp-team-title>b{font-size:63px;color:#fff;line-height:1.1}.fp-intro-fixtures{font-size:39px;line-height:1.3;margin:20px 0 40px}.fp-intro-fixtures>div{display:flex;align-items:center;gap:20px;padding:22px;background:#123047;border-radius:12px}.fp-intro-fixtures .fp-fdr{margin-left:auto}.fp-big{display:block;font-size:155px;line-height:1.1;color:#d5ff94;margin:45px 0 55px}.fp-panel ul{padding-left:45px;margin:45px 0 0;display:grid;gap:33px}.fp-panel li{font-size:43px;line-height:1.2}.fp-intro-captain{position:relative;display:flex;flex-direction:column}.fp-intro-captain>.fp-photo{width:330px;height:320px;margin:0 auto 14px}.fp-intro-captain h3{font-size:57px;text-align:center;color:#fff;margin:12px 0 20px}.fp-intro-captain>.fp-club{justify-content:center}.fp-intro-captain>.fp-intro-fixtures{margin:25px 0;font-size:39px;text-align:center}.fp-intro-captain>.fp-metrics{margin-top:auto}\n.fp-slide-2{grid-template-rows:110px minmax(0,1fr)}.fp-fixture-layout{display:grid;grid-template-columns:59fr 41fr;gap:26px;min-height:0}.fp-ticker{padding:20px}.fp-ticker table{width:100%;height:calc(100% - 65px);table-layout:fixed;border-collapse:collapse}.fp-ticker th{font-size:27px;text-align:left;color:#9be0ff;line-height:1.1;padding:12px 10px}.fp-ticker th:first-child{width:37%}.fp-ticker th:nth-child(2){width:33%}.fp-ticker th:nth-child(3){width:16%}.fp-ticker td{font-size:32px;padding:6px 10px;border-bottom:2px solid #253f54;font-weight:700}.fp-ticker td:first-child{display:flex;align-items:center;gap:12px;height:100%}.fp-ticker td:first-child>b{font-size:30px;line-height:1.05}.fp-ticker .fp-logo{height:47px;width:47px}.fp-rank{font-size:21px;width:60px;color:#cffb98;flex-shrink:0}.fp-fixture-cell{display:flex;justify-content:center;align-items:center;min-height:44px;border-radius:6px;padding:6px 9px;font-size:29px;line-height:1.1}.fp-ticker>p{font-size:28px;margin-top:18px;color:#b9d7e8;line-height:1.15}.fp-details{position:relative;padding:28px}.fp-team-detail,.fp-fixture-summary{position:absolute;inset:28px;visibility:hidden;overflow:hidden}.fp-team-detail>header{display:flex;gap:24px;align-items:center;height:145px;margin-bottom:25px}.fp-team-detail>header>.fp-logo{height:125px;width:125px}.fp-team-detail header h2{font-size:61px;margin:12px 0 0}.fp-team-detail header small{font-size:32px;color:#d8ff96;font-weight:900}.fp-shortlist{display:flex;align-items:center;gap:24px;border:2px solid #4a718c;border-radius:15px;padding:25px;margin-bottom:25px;background:#102b40;min-height:490px}.fp-shortlist>.fp-photo{height:300px;width:230px}.fp-shortlist>div{flex:1}.fp-shortlist h3{font-size:48px;line-height:1.1;margin-bottom:14px}.fp-shortlist>div>b{font-size:45px;color:#e1ff9f}.fp-shortlist .fp-metrics{margin:22px 0}.fp-shortlist .fp-metric{padding:16px}.fp-shortlist .fp-metric small{font-size:28px}.fp-shortlist .fp-metric b{font-size:50px}.fp-shortlist p,.fp-detail-note{font-size:32px;line-height:1.22;color:#d1e7f5}.fp-fixture-summary>h2{font-size:43px;margin:16px 0 22px;color:#a6e5ff}.fp-fixture-summary>div{display:flex;align-items:center;gap:24px;padding:14px 18px;margin-bottom:13px;background:#133047;border-radius:10px;font-size:40px;line-height:1.1}.fp-fixture-summary .fp-logo{width:62px;height:62px}.fp-fixture-summary>p{font-size:33px;line-height:1.2;margin-top:24px;color:#d6e7f2}\n.fp-slide-3{display:block}.fp-squad-layout{height:100%;display:grid;grid-template-columns:60fr 40fr;gap:26px}.fp-pitch-side{min-height:0;display:grid;grid-template-rows:115px minmax(0,1fr) 270px;gap:18px}.fp-pitch-side>.fp-section-header>b{font-size:60px;color:#d3ff9d}.fp-pitch{position:relative;min-height:0;overflow:hidden;border:3px solid #6eaa8d;border-radius:16px;background:repeating-linear-gradient(0deg,#126342 0%,#126342 14.28%,#18724d 14.28%,#18724d 28.56%)}.fp-boundary{position:absolute;inset:3% 3% 1%;border:4px solid #cee9db99;border-radius:18px}.fp-halfway{position:absolute;left:3%;right:3%;top:50%;border-top:4px solid #cee9db99}.fp-circle{position:absolute;left:35%;top:35%;width:30%;height:30%;border:4px solid #cee9db99;border-radius:50%}.fp-top-goal{position:absolute;left:44%;top:.6%;height:3%;width:12%;border:4px solid #d8ecdf;background:repeating-linear-gradient(90deg,transparent,transparent 15px,#c2dfcf88 16px)}.fp-top-box{position:absolute;top:3%;left:28%;width:44%;height:22%;border:4px solid #cee9db99;border-top:0}.fp-pitch-player{position:absolute;transform:translate(-50%,-50%);width:330px;display:flex;flex-direction:column;align-items:center;opacity:.32}.fp-pitch-player>.fp-photo{width:200px;height:205px}.fp-pitch-player>b{width:100%;font-size:39px;text-align:center;line-height:1.1;background:#061e2af5;padding:8px 4px 4px;border-radius:7px 7px 0 0;white-space:nowrap}.fp-pitch-player>span{width:100%;text-align:center;background:#061e2af5;font-size:31px;padding:4px 0 9px;border-radius:0 0 7px 7px}.fp-pitch-player em{font-size:28px;font-style:normal;color:#ffe49a}.fp-pitch-player.fp-active{border:0;box-shadow:none}.fp-pitch-player.fp-active>.fp-photo{filter:drop-shadow(0 0 13px #d5ff8588)}.fp-pitch-player.fp-active>b{color:#e3ff9d}.fp-bench{padding:18px 22px;border:2px solid #5e788c;border-radius:14px;background:#142536ee;overflow:hidden}.fp-bench h3{font-size:33px;color:#a7ddf3}.fp-bench>div{display:flex;justify-content:space-around;margin-top:8px}.fp-bench article{text-align:center;flex:1;opacity:.5}.fp-bench .fp-photo{width:110px;height:100px;display:block;margin:auto}.fp-bench b{display:block;font-size:33px;line-height:1.1}.fp-bench span{display:block;font-size:28px;color:#d1e4f2;margin-top:6px}.fp-squad-side{display:grid;grid-template-rows:115px 550px minmax(0,1fr);gap:18px;min-height:0}.fp-squad-side>.fp-section-header h2{font-size:42px}.fp-squad-side>.fp-section-header>b{font-size:34px}.fp-hero-stage{position:relative;min-height:0;border:2px solid #567d98;border-radius:15px;background:#0d263cf5;overflow:hidden}.fp-player-hero,.fp-squad-overview{position:absolute;inset:26px;visibility:hidden;overflow:hidden}.fp-player-hero>header{display:flex;align-items:center;gap:26px;height:225px}.fp-player-hero>header>.fp-photo{width:180px;height:210px}.fp-player-hero h2{font-size:51px;line-height:1.07;margin-bottom:18px}.fp-player-hero header>div>b{display:block;margin-top:15px;font-size:43px;color:#e1ff9f}.fp-player-hero>.fp-metrics{grid-template-columns:repeat(3,minmax(0,1fr));gap:14px}.fp-player-hero .fp-metric{padding:13px 16px}.fp-player-hero .fp-metric small{font-size:26px}.fp-player-hero .fp-metric b{font-size:45px;margin-top:9px}.fp-hero-fixture{font-size:35px;margin-top:20px}.fp-player-hero>p{font-size:31px;line-height:1.15;margin-top:18px}.fp-squad-overview h2{font-size:51px;color:#fff;margin:10px 0 30px}.fp-squad-overview>.fp-metrics{grid-template-columns:repeat(3,minmax(0,1fr));gap:14px}.fp-squad-overview .fp-metric{padding:17px 15px}.fp-squad-overview .fp-metric small{font-size:27px}.fp-squad-overview .fp-metric b{font-size:53px}.fp-squad-overview>p{font-size:37px;line-height:1.2;margin-top:24px;color:#d4e9f4}.fp-squad-table{padding:17px 20px;display:flex;flex-direction:column}.fp-squad-table table{width:100%;table-layout:fixed;border-collapse:collapse;flex:1}.fp-squad-table th{font-size:23px;line-height:1.1;color:#9bddfa;text-align:left;padding:8px}.fp-squad-table th:first-child{width:110px}.fp-squad-table th:nth-child(2){width:30%}.fp-squad-table th:last-child{width:110px}.fp-squad-table td{font-size:27px;font-weight:700;padding:8px 7px;line-height:1.05;border-bottom:1px solid #35536c}.fp-squad-table td:first-child{font-size:22px;color:#bcdcec}.fp-squad-table .fp-fdr{font-size:24px;padding:3px 7px;min-width:45px}.fp-squad-table footer{display:flex;flex-direction:column;gap:12px;padding:20px 4px 4px}.fp-squad-table footer>b{font-size:30px;color:#d9ff96}.fp-squad-table footer>span{font-size:26px;color:#c4ddea}\n.fp-slide-4{grid-template-rows:110px minmax(0,1fr) 145px}.fp-final-layout{display:grid;grid-template-columns:70fr 30fr;gap:26px;min-height:0}.fp-captain-three{display:grid;grid-template-columns:1fr 1.08fr 1fr;align-items:center;gap:24px;min-height:0;padding:10px}.fp-captain-card{background:#0b243aee;border:2px solid #56768e;border-radius:18px;padding:30px 26px;text-align:center;min-height:0;height:95%;display:flex;flex-direction:column;align-items:center;overflow:hidden}.fp-vortex-pick{height:100%;border:3px solid #a9df9d;background:#123c3af0}.fp-captain-card>small{font-size:35px;color:#bde9fa;font-weight:900;line-height:1.1}.fp-vortex-pick>small{color:#e0ff9d}.fp-captain-card>.fp-photo{width:290px;height:330px;margin:20px 0}.fp-captain-card h2{font-size:53px;line-height:1.05;color:#fff}.fp-captain-card>.fp-club{font-size:29px;justify-content:center;margin-top:22px}.fp-captain-fixture{font-size:35px;margin:30px 0;line-height:1.2}.fp-captain-card>.fp-metrics{grid-template-columns:1fr;width:100%;gap:14px;text-align:left}.fp-captain-card .fp-metric{display:flex;align-items:center;justify-content:space-between;gap:18px;min-height:115px;padding:18px}.fp-captain-card .fp-metric small{font-size:29px;max-width:65%}.fp-captain-card .fp-metric b{font-size:60px;margin:0}.fp-captain-card>p{font-size:32px;line-height:1.2;color:#e0edf5;margin-top:26px}.fp-final-takeaways{display:grid;grid-template-rows:470px minmax(0,1fr);gap:22px;min-height:0}.fp-vice>header{display:flex;gap:22px;align-items:center;margin-top:20px}.fp-vice .fp-photo{width:170px;height:210px}.fp-vice h3{font-size:44px}.fp-vice .fp-club{font-size:26px;margin-top:20px}.fp-vice>p{font-size:32px;margin-top:24px;line-height:1.2;color:#d3e5f2}.fp-takeaway{padding:24px 0;border-bottom:2px solid #385c73}.fp-takeaway small{font-size:28px;color:#acd9ee;display:block;line-height:1.1}.fp-takeaway b{font-size:45px;color:#fff;display:block;margin-top:16px;line-height:1.1}.fp-takeaway span{font-size:30px;color:#d0e5f2;display:block;margin-top:14px;line-height:1.2}.fp-final-takeaways>.fp-panel>.fp-metrics{margin-top:28px;grid-template-columns:1fr}\n\'\'\'\n# Typography is scoped above the broadcast shell\'s defaults.\nCSS += \'\\n\\n#content .fp-slide-3 .fp-pitch-side{grid-template-rows:102px minmax(0,1fr) 212px;gap:12px}\\n#content .fp-slide-3 .fp-pitch-player{width:min(325px,22%)}\\n#content .fp-slide-3 .fp-pitch-player>.fp-photo{width:210px;height:218px}\\n#content .fp-slide-3 .fp-pitch-player>b{font-size:clamp(27px,1.05vw,39px);overflow:hidden;white-space:nowrap;text-overflow:ellipsis}\\n#content .fp-slide-3 .fp-bench{padding:9px 15px}\\n#content .fp-slide-3 .fp-bench .fp-photo{height:80px}\\n#content .fp-slide-3 .fp-squad-side{gap:11px}\\n#content .fp-slide-3 .fp-section-header h2{font-size:43px}\\n\'\n\nCSS += \'\\n#content .fp-slide-3 .fp-squad-layout{grid-template-columns:60fr 40fr;gap:24px}\\n#content .fp-slide-3 .fp-squad-side{grid-template-rows:102px minmax(0,370px) minmax(0,1fr);gap:12px}\\n#content .fp-slide-3 .fp-hero-stage{border-radius:16px}\\n#content .fp-slide-3 .fp-squad-overview,#content .fp-slide-3 .fp-player-hero{inset:18px 22px}\\n#content .fp-slide-3 .fp-squad-overview h2{font-size:39px;margin:5px 0 17px;line-height:1.05}\\n#content .fp-slide-3 .fp-squad-overview>.fp-metrics{gap:11px}\\n#content .fp-slide-3 .fp-squad-overview .fp-metric{padding:12px 15px;border-radius:9px}\\n#content .fp-slide-3 .fp-squad-overview .fp-metric small{font-size:24px;line-height:1.05}\\n#content .fp-slide-3 .fp-squad-overview .fp-metric b{font-size:47px;line-height:1.05}\\n#content .fp-slide-3 .fp-squad-overview .fp-metric:nth-child(3){border:2px solid #d5b96e;background:#243c40}\\n#content .fp-slide-3 .fp-squad-overview .fp-metric:nth-child(3) b{color:#f5d988}\\n#content .fp-slide-3 .fp-squad-overview>p{font-size:26px;line-height:1.1;margin-top:8px}\\n#content .fp-slide-3 .fp-player-hero>header{height:135px;gap:16px}\\n#content .fp-slide-3 .fp-player-hero>header>.fp-photo{width:104px;height:130px}\\n#content .fp-slide-3 .fp-player-hero h2{font-size:36px;margin-bottom:8px}\\n#content .fp-slide-3 .fp-player-hero header>div>b{margin-top:5px;font-size:27px}\\n#content .fp-slide-3 .fp-player-hero .fp-metric{padding:8px 11px}\\n#content .fp-slide-3 .fp-player-hero .fp-metric small{font-size:20px}\\n#content .fp-slide-3 .fp-player-hero .fp-metric b{font-size:34px;margin-top:5px}\\n#content .fp-slide-3 .fp-hero-fixture{font-size:24px;margin-top:10px}\\n#content .fp-slide-3 .fp-player-hero>p{font-size:23px;margin-top:9px}\\n#content .fp-slide-3 .fp-xpts-dashboard{display:grid;grid-template-rows:62px minmax(0,1fr) 152px 51px;gap:10px;min-height:0;padding:15px 17px 12px;background:#0c2033f4;border:2px solid #43667c;border-radius:17px;overflow:hidden}\\n#content .fp-slide-3 .fp-xpts-dashboard-header{display:flex;align-items:center;justify-content:space-between;gap:16px;border-bottom:1px solid #3f6175;padding-bottom:10px}\\n#content .fp-slide-3 .fp-xpts-dashboard-header h2{font-size:31px;line-height:1.04;color:#f3f8ff;white-space:nowrap}\\n#content .fp-slide-3 .fp-xpts-dashboard-header span{font-size:21px;color:#9ddcf0;white-space:nowrap;font-weight:800}\\n#content .fp-slide-3 .fp-xpts-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));grid-template-rows:repeat(6,minmax(0,1fr));grid-auto-flow:column;gap:9px 14px;min-height:0}\\n#content .fp-slide-3 .fp-xpts-player{background:linear-gradient(110deg,#17364e,#0c2b42);border:1px solid #54778a;border-radius:11px;padding:10px 13px;min-height:0;display:flex;flex-direction:column;justify-content:center;gap:10px;overflow:hidden}\\n#content .fp-slide-3 .fp-xpts-player.fp-xpts-leader{border-color:#b59b5f;background:linear-gradient(110deg,#213a4c,#193447)}\\n#content .fp-slide-3 .fp-xpts-player-head{display:flex;align-items:center;gap:10px;min-width:0}\\n#content .fp-slide-3 .fp-xpts-role{background:#123e4d;color:#8de2ed;border:1px solid #367887;border-radius:5px;font-size:21px;font-weight:900;padding:6px 7px;min-width:58px;text-align:center}\\n#content .fp-slide-3 .fp-xpts-player-head>b{font-size:29px;line-height:1.05;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;min-width:0;flex:1;color:#f0f8ff}\\n#content .fp-slide-3 .fp-xpts-player-head em{font-style:normal;font-size:20px;color:#f3d88a;font-weight:900}\\n#content .fp-slide-3 .fp-xpts-player-head strong{font-size:35px;color:#dff79f;white-space:nowrap;line-height:1}\\n#content .fp-slide-3 .fp-xpts-player-head strong small{font-size:18px;color:#afcfde}\\n#content .fp-slide-3 .fp-xpts-track{height:9px;background:#244457;border-radius:20px;overflow:hidden}\\n#content .fp-slide-3 .fp-xpts-track>div{height:100%;background:linear-gradient(90deg,#29a1b4,#7fe2c9);border-radius:20px}\\n#content .fp-slide-3 .fp-xpts-leader .fp-xpts-track>div{background:linear-gradient(90deg,#77c6ce,#e5d98e)}\\n#content .fp-slide-3 .fp-xpts-bench{min-height:0;border-top:1px solid #45677b;padding-top:10px;display:grid;grid-template-rows:30px minmax(0,1fr);gap:8px}\\n#content .fp-slide-3 .fp-xpts-bench h3{font-size:26px;color:#acd8ea;line-height:1.05}\\n#content .fp-slide-3 .fp-xpts-bench h3 span{font-size:17px;color:#799fb5;padding-left:14px}\\n#content .fp-slide-3 .fp-xpts-bench>div{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:9px}\\n#content .fp-slide-3 .fp-xpts-sub{display:flex;flex-direction:column;gap:6px;min-width:0;overflow:hidden;padding:10px 9px;background:#142a3c;border:1px solid #3e5a6e;border-radius:8px}\\n#content .fp-slide-3 .fp-xpts-sub>small{font-size:17px;line-height:1.08;color:#8db4c7;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}\\n#content .fp-slide-3 .fp-xpts-sub>b{font-size:23px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;color:#d5e5ef}\\n#content .fp-slide-3 .fp-xpts-sub>strong{font-size:26px;color:#d1eeb7;margin-top:auto;white-space:nowrap}\\n#content .fp-slide-3 .fp-xpts-sub>strong small{font-size:17px;color:#aecbd8}\\n#content .fp-slide-3 .fp-xpts-total{display:flex;justify-content:space-between;align-items:center;gap:10px;border-top:1px solid #426376;padding:9px 1px 0;min-width:0}\\n#content .fp-slide-3 .fp-xpts-total>b{font-size:27px;line-height:1.05;color:#e1f7b1;white-space:nowrap}\\n#content .fp-slide-3 .fp-xpts-total>b span{color:#9dc9db;padding:0 7px}\\n#content .fp-slide-3 .fp-xpts-total>small{font-size:17px;white-space:nowrap;color:#a6bfd0;font-weight:700}\\n#content .fp-slide-3 .fp-xpts-player.fp-active{border-color:#cbdba7;box-shadow:inset 0 0 0 2px #88ddbbaa}\\n\'\n\nCSS += \'\\n#content .fp-slide-3{height:calc(100% - 90px)}\\n#content .fp-slide-3 .fp-bench article{opacity:1}\\n#content .fp-slide-3 .fp-bench b{font-size:27px;line-height:1.06;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}\\n#content .fp-slide-3 .fp-bench span{font-size:23px;margin-top:4px;line-height:1.1}\\n#content .fp-slide-3 .fp-bench .fp-photo{height:76px}\\n#content .fp-slide-3 .fp-xpts-total>b{font-size:24px}\\n#content .fp-slide-3 .fp-xpts-total>small{font-size:15px}\\n#content .fp-slide-3 .fp-xpts-sub{padding:8px 9px}\\n\'\n\nCSS += \'\\n#content .fp-slide-3 .fp-pitch-player>b{font-size:29px;padding:6px 2px 4px}\\n#content .fp-slide-3 .fp-pitch-player>span{font-size:24px;padding:2px 0 6px}\\n\'\n\nCSS=re.sub(r\'(^|})([^{}]+){\',lambda m:m[1]+\',\'.join(s.strip() if s.strip().startswith(\'#content\') else \'#content \'+s.strip() for s in m[2].split(\',\'))+\'{\',CSS)\n\n# Free Hit Slide 04 only: title plaque, clear footer, and readable captaincy cards.\nCSS += \'\'\'\n#content:has(.fp-slide-4){height:1360px!important}\n#slide:has(.fp-slide-4) .title-group{isolation:isolate}\n#slide:has(.fp-slide-4) .title-group::before{content:"";position:absolute;top:4px;left:50%;transform:translateX(-50%);width:1050px;height:137px;z-index:0;background:#071523;border:2px solid #bda064b5;border-radius:17px;box-shadow:0 8px 25px #000b,inset 0 0 18px #203b5066;pointer-events:none}\n#slide:has(.fp-slide-4) .title-group .heading{position:relative;z-index:1}\n#content .fp-slide-4{grid-template-rows:108px minmax(0,1fr) 122px;gap:20px}\n#content .fp-slide-4>.fp-section-header{padding:14px 24px;background:#081b2ef7}\n#content .fp-slide-4>.fp-section-header h2{font-size:45px}\n#content .fp-slide-4>.fp-section-header p{font-size:31px}\n#content .fp-slide-4 .fp-final-layout{gap:22px;grid-template-columns:minmax(0,70fr) minmax(0,30fr)}\n#content .fp-slide-4 .fp-captain-three{align-items:stretch;gap:20px;padding:0 4px}\n#content .fp-slide-4 .fp-captain-card{height:100%;padding:22px 19px 19px;gap:10px;justify-content:space-around;background:#092138f7;border-color:#577c96}\n#content .fp-slide-4 .fp-vortex-pick{background:#123e38fa;border-color:#d1db92;box-shadow:inset 0 0 21px #75d4ab15,0 0 19px #9fcf8244}\n#content .fp-slide-4 .fp-captain-card>small{font-size:33px;line-height:1.1}\n#content .fp-slide-4 .fp-captain-card>.fp-photo{width:260px;height:275px;margin:5px auto 2px}\n#content .fp-slide-4 .fp-captain-card h2{font-size:54px;line-height:1.08;overflow-wrap:anywhere}\n#content .fp-slide-4 .fp-captain-card>.fp-club{font-size:32px;margin-top:2px;line-height:1.1}\n#content .fp-slide-4 .fp-captain-fixture{font-size:35px;line-height:1.15;margin:10px 0}\n#content .fp-slide-4 .fp-captain-card>.fp-metrics{gap:12px}\n#content .fp-slide-4 .fp-captain-card .fp-metric{min-height:105px;padding:14px 16px;gap:12px;background:#15364cf5}\n#content .fp-slide-4 .fp-captain-card .fp-metric small{font-size:32px;max-width:70%;line-height:1.15}\n#content .fp-slide-4 .fp-captain-card .fp-metric b{font-size:59px}\n#content .fp-slide-4 .fp-captain-card>p{font-size:33px;line-height:1.19;margin:10px 0 2px;color:#ecf5fb}\n#content .fp-slide-4 .fp-final-takeaways{grid-template-rows:440px minmax(0,1fr);gap:18px}\n#content .fp-slide-4 .fp-final-takeaways>.fp-panel{background:#081b30f8;padding:24px 25px}\n#content .fp-slide-4 .fp-final-takeaways>.fp-panel h2{font-size:43px;margin-bottom:19px}\n#content .fp-slide-4 .fp-vice>header{gap:20px;margin-top:10px}\n#content .fp-slide-4 .fp-vice .fp-photo{width:176px;height:185px}\n#content .fp-slide-4 .fp-vice h3{font-size:43px;line-height:1.1}\n#content .fp-slide-4 .fp-vice .fp-club{font-size:29px;margin-top:12px}\n#content .fp-slide-4 .fp-vice-projection{display:flex;justify-content:space-between;align-items:center;background:#12334b;border:1px solid #466e82;border-radius:9px;padding:11px 15px;margin-top:15px;gap:10px}\n#content .fp-slide-4 .fp-vice-projection strong{font-size:40px;color:#dff5a4;white-space:nowrap}\n#content .fp-slide-4 .fp-vice-projection span{font-size:27px;color:#cde5f0;white-space:nowrap}\n#content .fp-slide-4 .fp-vice>p{font-size:29px;margin-top:14px;line-height:1.16}\n#content .fp-slide-4 .fp-takeaway{padding:20px 0}\n#content .fp-slide-4 .fp-takeaway small{font-size:31px;color:#aee3f5;line-height:1.18}\n#content .fp-slide-4 .fp-takeaway b{font-size:45px;margin-top:13px;line-height:1.12}\n#content .fp-slide-4 .fp-takeaway span{font-size:31px;margin-top:10px;line-height:1.17}\n#content .fp-slide-4 .fp-final-takeaways>.fp-panel:last-child{display:flex;flex-direction:column;justify-content:space-evenly}\n#content .fp-slide-4 .fp-final-takeaways>.fp-panel>.fp-metrics{margin-top:20px}\n#content .fp-slide-4 .fp-final-takeaways .fp-metric{padding:17px 20px;border-color:#658e7c}\n#content .fp-slide-4 .fp-final-takeaways .fp-metric small{font-size:32px}\n#content .fp-slide-4 .fp-final-takeaways .fp-metric b{font-size:72px;margin-top:8px;color:#dff7a2}\n#content .fp-slide-4>.fp-decision{padding:16px 26px;gap:25px;background:#0d3834fb}\n#content .fp-slide-4>.fp-decision b{font-size:43px;white-space:nowrap}\n#content .fp-slide-4>.fp-decision p{font-size:34px;line-height:1.16;max-width:68%;text-align:right}\n\'\'\'\n\nSEEK_JS=\'\'\'<script>(function(){const prev=window.seekWildcard;window.seekWildcard=function(t){if(prev)prev(t);const root=document.querySelector(\'.fp-slide\');if(!root)return;const d=JSON.parse(root.querySelector(\'.fp-data\').textContent),tl=d.timeline,s=tl.find(x=>t>=x.start&&t<x.end)||(t<tl[0].start?tl[0]:tl.at(-1));const ease=x=>{x=Math.max(0,Math.min(1,x));return 1-Math.pow(1-x,3)};const phase=ease((t-s.start)/.4);root.dataset.phase=s.type;for(const el of root.querySelectorAll(\'[data-fp-focus]\'))el.classList.toggle(\'fp-active\',el.dataset.fpFocus===s.type);if(d.kind===2){for(const el of root.querySelectorAll(\'[data-fp-team]\'))el.classList.toggle(\'fp-active\',+el.dataset.fpTeam===s.team_id);const detail=[\'team\',\'player\',\'comparison\'].includes(s.type);for(const el of root.querySelectorAll(\'[data-fp-detail]\')){const show=detail&&+el.dataset.fpDetail===s.team_id;el.style.visibility=show?\'visible\':\'hidden\';el.style.opacity=String(show?.7+.3*phase:0);}root.querySelector(\'.fp-fixture-summary\').style.visibility=detail?\'hidden\':\'visible\';for(const el of root.querySelectorAll(\'[data-fp-player]\'))el.classList.toggle(\'fp-active\',+el.dataset.fpPlayer===s.player_id);}if(d.kind===3){const revealed=new Set(tl.filter(x=>x.start<=t).flatMap(x=>x.player_ids||[]));for(const el of root.querySelectorAll(\'[data-fp-pitch],[data-fp-bench]\')){const id=+(el.dataset.fpPitch||el.dataset.fpBench),cue=tl.find(x=>x.hero_player_id===id),v=cue?ease((t-cue.start)/.4):0;el.style.opacity=String(revealed.has(id)?.72+.28*v:el.dataset.fpBench?.72:.72);el.classList.toggle(\'fp-active\',id===s.hero_player_id);}for(const el of root.querySelectorAll(\'[data-fp-hero]\')){const show=+el.dataset.fpHero===s.hero_player_id;el.style.visibility=show?\'visible\':\'hidden\';el.style.opacity=String(phase);el.style.translate=`${(1-phase)*(s.direction===\'left\'?-25:25)}px 0`;}root.querySelector(\'.fp-squad-overview\').style.visibility=s.hero_player_id?\'hidden\':\'visible\';for(const el of root.querySelectorAll(\'[data-fp-squad-row]\')){el.style.opacity=revealed.has(+el.dataset.fpSquadRow)||s.type===\'summary\'?\'1\':\'.45\';el.classList.toggle(\'fp-active\',+el.dataset.fpSquadRow===s.hero_player_id);}}if(d.kind===4){for(const el of root.querySelectorAll(\'[data-fp-contender]\')){const active=+el.dataset.fpContender===s.player_id;el.classList.toggle(\'fp-active\',active);el.style.opacity=active||[\'takeaways\',\'outro\',\'vice\'].includes(s.type)?\'1\':\'.8\';el.style.scale=active?String(1+.006*phase):\'1\';}}};})();</script>\'\'\'\n\nclass FreeHitPremiumVisual:\n def __init__(self,scene,contract,width,height,renderer,metadata,media,badges,approved_cards=None):\n  self.manifest,self.contract,self.width,self.height=scene,contract,width,height;self.renderer,self.metadata=renderer,metadata;self.layout=[]\n  kind={\'fh_blueprint\':1,\'fixture_heatmap\':2,\'fh_squad_plan\':3,\'fh_captain_final\':4}[scene[\'visual_type\']];d=copy.deepcopy(contract)\n  self.timeline=scene[\'narration_timeline\']\n  if self.timeline[0].get(\'start\') is None:self.timeline=align_timeline(self.timeline,float(scene.get(\'audio_duration_seconds\') or len(scene[\'narration_text\'].split())/145*60))\n  d[\'captain_pick\']=captain_plan(d);self.content_html=render_slide(kind,d,self.timeline,media,badges,approved_cards);self.complete=self.draw(self.timeline[-1][\'end\']+2.5,0)\n def draw(self,seconds,duration):return self.renderer.animation_image(self.manifest,self.metadata,self.width,self.height,self.content_html,seconds)\n def frame(self,seconds,duration,previous=None):\n  import numpy as np\n  return np.asarray(self.draw(seconds,duration))\n\nCSS+="""#content .fp-intro-grid>.fp-panel{display:flex;flex-direction:column}#content .fp-intro-minis{margin-top:auto;padding-top:24px;display:flex;justify-content:space-around;gap:12px}#content .fp-intro-minis article{flex:1;text-align:center}#content .fp-intro-minis .fp-photo{width:145px;height:170px;display:block;margin:auto}#content .fp-intro-minis b{font-size:28px;line-height:1.1;display:block;margin-top:10px}#content .fp-intro-grid>.fp-panel:nth-child(2) .fp-intro-minis .fp-photo{width:105px;height:140px}#content .fp-intro-grid>.fp-panel:nth-child(2) .fp-intro-minis b{font-size:24px}"""\n\nCSS+="""#content .fp-detail-note>b{display:block;color:#dfff9c;font-size:34px;line-height:1.15}#content .fp-detail-note>span{display:block;margin-top:12px;font-size:29px;color:#fff}#content .fp-detail-note>p{margin-top:18px;font-size:30px}"""\n\n# VX_FH_SLIDE3_PREMIUM_V1\nCSS += \'\\n/* VX_FH_SLIDE3_PREMIUM_V1: isolated heading treatment, original masthead/footer preserved. */\\n#slide:has(.fp-slide-3) .title-group{isolation:isolate}\\n#slide:has(.fp-slide-3) .title-group::after{display:none!important}\\n#slide:has(.fp-slide-3) .title-group::before{\\n content:"";position:absolute;left:50%;top:7px;translate:-50% 0;width:1040px;height:136px;\\n background:linear-gradient(112deg,#071523 0%,#0c2738 55%,#071523 100%);\\n border:1px solid #927c4b;border-radius:16px;z-index:0;pointer-events:none;\\n box-shadow:0 12px 30px #0009,inset 0 1px 0 #d6ba7780;\\n}\\n#slide:has(.fp-slide-3) .title-group .heading{position:relative;z-index:1}\\n#content .fp-slide-3{position:relative;isolation:isolate}\\n#content .fp-slide-3 .fp-card-flight{\\n position:absolute;top:0;left:0;z-index:80;pointer-events:none;visibility:hidden;opacity:0;\\n display:flex;flex-direction:column;align-items:center;justify-content:center;\\n width:296px;min-height:328px;padding:13px;border-radius:17px;\\n border:2px solid #b8dfc9;background:linear-gradient(160deg,#123c4a,#081727 88%);\\n box-shadow:0 20px 46px #000c,0 0 28px #65e9cd75,inset 0 0 20px #8df5da20;\\n transform-origin:center center;will-change:transform,opacity;\\n}\\n#content .fp-slide-3 .fp-card-flight .fp-photo{width:190px;height:202px;object-fit:contain;filter:drop-shadow(0 8px 10px #0009)}\\n#content .fp-slide-3 .fp-card-flight b{width:100%;margin-top:8px;overflow:hidden;text-align:center;\\n font-size:31px;white-space:nowrap;text-overflow:ellipsis;color:#f7fffe}\\n#content .fp-slide-3 .fp-card-flight small{font-size:23px;font-weight:800;color:#d7ff9c;margin-top:6px}\\n#content .fp-slide-3 .fp-hero-stage{isolation:isolate}\\n#content .fp-slide-3 .fp-player-hero.fp-flying{filter:drop-shadow(0 0 10px #96ffdf40)}\\n#content .fp-slide-3 .fp-pitch-player.fp-landed>.fp-photo{filter:drop-shadow(0 0 16px #deffb966)}\\n#content .fp-slide-3 .fp-xpts-player.fp-active{box-shadow:inset 0 0 0 2px #b5f4ceaa,0 0 12px #66e6d858}\\n\'\n\n# Scoped Free Hit 3/4 premium presentation; underlying approved player-card art stays untouched.\nCSS += """\n#content .fp-slide-3 .fp-player-hero{display:flex;flex-direction:column;align-items:center;justify-content:center;inset:12px;padding:0}\n#content .fp-slide-3 .fp-approved-card{display:block;max-width:98%;height:calc(100% - 82px);object-fit:contain;filter:drop-shadow(0 6px 18px #0007)}\n#content .fp-slide-3 .fp-card-meta{width:98%;display:flex;flex-direction:column;align-items:center;gap:5px;text-align:center;padding:9px;border-top:2px solid #c4a569;background:#0b2837}\n#content .fp-slide-3 .fp-card-meta b{font-size:32px;color:white}\n#content .fp-slide-3 .fp-card-meta span{font-size:25px;color:#9de5ec}\n#content .fp-slide-4 .fp-final-takeaways{position:relative}\n#content .fp-slide-4 .fp-approved-hero-stage{position:absolute;inset:0;z-index:5;pointer-events:none;background:linear-gradient(150deg,#0b2137,#082437);border:2px solid #e0c789;border-radius:19px;overflow:hidden}\n#content .fp-slide-4 .fp-approved-captain{position:absolute;inset:0;margin:0;padding:18px;display:flex;align-items:center;justify-content:center;visibility:hidden;opacity:0}\n#content .fp-slide-4 .fp-approved-captain .fp-approved-card{width:auto;height:100%;max-width:100%;object-fit:contain;filter:drop-shadow(0 9px 24px #0009)}\n#content .fp-slide-4 .fp-captain-card{visibility:hidden;opacity:0;transition:none}\n#content .fp-slide-4 .fp-captain-card .fp-photo{width:208px;height:228px;margin:12px 0}\n#content .fp-slide-4 .fp-captain-card h2{font-size:40px}\n#content .fp-slide-4 .fp-captain-card .fp-metric{padding:8px;min-height:82px}\n#content .fp-slide-4 .fp-captain-card .fp-metric small{font-size:22px}\n#content .fp-slide-4 .fp-captain-card .fp-metric b{font-size:42px}\n#content .fp-slide-4 .fp-captain-card>p{font-size:24px;margin-top:10px}\n"""\n\n\n# VX_FH_FULL_HERO_FLIGHT_V2: only affects Free Hit Slide 3.\nCSS += """\n#content .fp-slide-3 .fp-card-flight {width:296px;height:360px;min-height:0;padding:7px;overflow:hidden}\n#content .fp-slide-3 .fp-card-flight .fp-approved-card {display:block;width:100%;height:276px;max-width:100%;object-fit:contain;filter:drop-shadow(0 8px 16px #000c)}\n#content .fp-slide-3 .fp-card-flight>b {margin-top:3px;font-size:23px;line-height:1.05}\n#content .fp-slide-3 .fp-card-flight>small {font-size:19px;margin-top:3px}\n#content .fp-slide-3 .fp-player-hero {position:absolute;overflow:hidden;gap:5px;justify-content:flex-start}\n#content .fp-slide-3 .fp-player-hero>.fp-approved-card {height:calc(100% - 206px);max-height:calc(100% - 206px);min-height:0;flex:1 1 auto}\n#content .fp-slide-3 .fp-card-meta {flex:0 0 auto;min-height:59px}\n#content .fp-slide-3 .fp-why-popup {flex:0 0 129px;width:98%;max-width:100%;padding:12px 17px 10px;background:#0a2736f7;border:2px solid #bfe8a8;border-radius:12px;box-shadow:0 12px 35px #000a;opacity:0;transform:translateX(36px);visibility:hidden}\n#content .fp-slide-3 .fp-why-popup strong {display:block;font-size:29px;line-height:1;color:#deffa2}\n#content .fp-slide-3 .fp-why-popup p {margin:8px 0 5px!important;font-size:25px;line-height:1.12;color:white;text-wrap:balance}\n#content .fp-slide-3 .fp-why-popup small {display:block;font-size:19px;color:#b5ecfa}\n"""\n\n# Stage height is large enough to show the original artwork and a legible explanation.\nCSS += """\n#content .fp-slide-3 .fp-squad-side{grid-template-rows:102px minmax(0,520px) minmax(0,1fr)}\n#content .fp-slide-3 .fp-player-hero>.fp-approved-card{height:calc(100% - 205px);max-height:calc(100% - 205px)}\n"""\n',encoding='utf-8')
(_vx_module_dir/'freehit_personal_intro.py').write_text('"""Personal Free Hit opening; one-Gameweek evidence, no automatic chip verdict."""\nfrom __future__ import annotations\nimport copy\nimport re\nfrom typing import Any\nfrom wildcard_intro_context import refresh_context\n\n\nVERSION=\'freehit-dynamic-intro-v5-six-cues\'\n\ndef select_showcase(contract:dict[str,Any])->dict[str,dict[str,Any]]:\n    players=contract[\'squad\']\n    captain_ids={contract[\'slide3_plan\'][\'captain\'][\'id\'],contract[\'slide3_plan\'][\'vice_captain\'][\'id\']}\n    starting_players=[player for player in players if player.get(\'is_starting_xi\')]\n    if not starting_players:raise ValueError(\'Free Hit showcase requires a validated starting XI\')\n    attackers=[player for player in starting_players if player[\'position\'] in (\'MID\',\'FWD\') and player[\'id\'] not in captain_ids]\n    defenders=[player for player in starting_players if player[\'position\']==\'DEF\']\n    if not attackers or not defenders:raise ValueError(\'Free Hit showcase requires an attacker and defender from the starting XI\')\n    # Feature the draft beyond its captain and vice, without altering their analytical selection.\n    return {\'fixture\':max(attackers,key=lambda player:player[\'expected_points\']), \'captain\':max(defenders,key=lambda player:player[\'expected_points\'])}\n\ndef build_case(contract:dict[str,Any],bootstrap:dict[str,Any]|None=None)->dict[str,Any]:\n    from freehit_analyst import comparison\n    if contract.get(\'chip\')!=\'freehit\' or contract.get(\'horizon_gameweeks\')!=[contract[\'gameweek\']]:raise ValueError(\'Free Hit introduction accepts one Gameweek only\')\n    context=refresh_context(contract,bootstrap or {},horizon_size=1)\n    compared=comparison(contract);evidence=contract.get(\'freehit_decision_evidence\') or {}\n    verified=bool(compared and contract.get(\'decision_input_hash\') and evidence.get(\'source_data_hash\')==contract[\'decision_input_hash\'] and evidence.get(\'evidence_based_verdict\'))\n    activate=bool(verified and compared[\'gain\']>0 and evidence.get(\'play_supported\') is True and evidence.get(\'hold_supported\') is not True)\n    hold=bool(verified and evidence.get(\'hold_supported\') is True and evidence.get(\'play_supported\') is not True)\n    verdict=\'ACTIVATE\' if activate else \'HOLD\' if hold else \'OPEN\'\n    reason=str(evidence[\'evidence_based_verdict\']) if activate or hold else \'A verified one-Gameweek comparison against the current squad plus realistic free transfers, and a matching chip-value decision, are needed.\'\n    if context[\'blanks\']:\n        indicators=[\'BLANK GAMEWEEK\',\'ONE-WEEK COVER\']\n    elif context[\'doubles\']:\n        indicators=[\'DOUBLE FIXTURES\',\'MINUTES MATTER\']\n    else:\n        indicators=[\'THIS WEEK ONLY\',\'CHIP VALUE\']\n    items = [\n        (\'question\', f"Thinking about using your Free Hit in Gameweek {contract[\'gameweek\']}? Here\'s how I look at it."),\n        (\'context\', "For me, the question is whether a temporary reset can improve this week\'s team enough to justify spending the chip."),\n        (\'temporary\', "A Free Hit solves a one-week problem. If my squad needs lasting changes, I\'d look at a different approach."),\n        (\'draft\', "I\'ve built my Free Hit draft for this Gameweek, focusing on fixtures, expected minutes, value and captaincy."),\n        (\'return\', "I\'ll explain my picks and whether they justify spending the chip. My original squad returns next Gameweek."),\n        (\'transition\', "First, let\'s look at this week\'s fixtures."),\n    ]\n    timeline=[dict(segment_id=identity,type=identity,narration_text=spoken,player_ids=[],start=None,end=None,on_start=[\'activate_\'+identity],on_end=[]) for identity,spoken in items]\n    narration=\' \'.join(item[\'narration_text\'] for item in timeline)\n    if re.search(r"\\b(?:we|our|we\'ve|we\'re|we\'d)\\b",narration,re.I):raise ValueError(\'Free Hit opening must use first-person narration\')\n    showcase=select_showcase(contract)\n    result=copy.deepcopy(contract.get(\'freehit_case\',{}))\n    plan=contract[\'slide3_plan\']\n    result.update(starting_xi_points=plan[\'xi_xpts\'],captain_points=plan[\'captain\'][\'projected_points\'],total_points=plan[\'captain_adjusted_xpts\'],formation=plan[\'formation\'],total_cost=plan[\'total_cost\'],bank=plan[\'bank\'])\n    result.update(next_gw=contract[\'gameweek\'],analysis_gws=[contract[\'gameweek\']],analyst_version=VERSION,analysis_title=\'FREE HIT: NOW OR WAIT?\',verdict=verdict,decision_label=\'DECISION OPEN\' if verdict==\'OPEN\' else verdict,verdict_reason=reason,scenario_comparison=compared,intro_context=context,intro_indicators=indicators,intro_data=showcase,intro_showcase_roles={\'fixture\':\'ATTACKING PICK\',\'captain\':\'DEFENSIVE PICK\'},narration_timeline=timeline,narration_text=narration,intro_word_count=len(narration.split()))\n    return result\n')
sys.modules.pop('freehit_personal_intro',None)
sys.modules.pop('freehit_premium',None)
(_vx_module_dir/'freehit_analyst.py').write_text('"""Free Hit data enrichment. Presentation and narration live in freehit_premium."""\nimport copy\nfrom wildcard_squad_analysis import number,enrich_plan as enrich_squad\nfrom wildcard_fixture_analysis import enrich_fixture_case\nfrom freehit_premium import VERSION,fixture_segments,squad_segments,intro_case\ndef comparison(d):\n r=d.get(\'freehit_scenario_comparison\') or {};a=r.get(\'current_with_free_transfers\',{});b=r.get(\'freehit\',{});weeks=[d[\'gameweek\']]\n keys=(\'model_id\',\'model_version\',\'projection_snapshot\',\'scoring_basis\',\'price_basis\',\'budget_basis\')\n if any(not a.get(k) or a.get(k)!=b.get(k) for k in keys):return None\n if a.get(\'scoring_basis\')!=\'starting_xi_plus_captain_net_hits\' or a.get(\'projection_snapshot\')!=d[\'source_snapshot_at\']:return None\n if not a.get(\'manager_squad_verified\') or not a.get(\'transfers_calculated\') or not a.get(\'legal_plan_verified\') or not b.get(\'legal_plan_verified\'):return None\n if a.get(\'gameweeks\')!=weeks or b.get(\'gameweeks\')!=weeks or number(a.get(\'budget\'))!=number(d.get(\'budget_total\')) or number(a.get(\'budget\'))!=number(b.get(\'budget\')):return None\n totals=[]\n for s in (a,b):\n  rows=s.get(\'weekly_results\',[])\n  if len(rows)!=1 or rows[0].get(\'gameweek\')!=weeks[0]:return None\n  x=number(rows[0].get(\'xi_plus_captain_xpts\'));hit=number(rows[0].get(\'hit_cost\'))\n  if x is None or hit is None or hit<0:return None\n  totals.append(x-hit)\n if not a.get(\'transfer_cost_analysis\'):return None\n if abs(number(b[\'weekly_results\'][0][\'xi_plus_captain_xpts\'])-d[\'slide3_plan\'][\'captain_adjusted_xpts\'])>.01:return None\n return dict(current=totals[0],freehit=totals[1],gain=totals[1]-totals[0],tradeoff=a[\'transfer_cost_analysis\'])\n\ndef fixture_case(case,players,context):\n models=[dict(m,xpts_wildcard_horizon=m.get(\'xpts_freehit_gw\'),wc_eligible=True) for m in context[\'player_models\']]\n result=enrich_fixture_case(case,players,dict(context,player_models=models));result[\'analysis_title\']=f"FPL VORTEX — FREE HIT FIXTURE ANALYSIS | GW {case[\'gameweeks\'][0]}";result[\'label\']=\'Fixture Difficulty: Next Gameweek\';result[\'range_label\']=\'GW\'+str(case[\'gameweeks\'][0])+\' only\';result[\'analyst_version\']=VERSION\n # A verified blank provides no one-week scoring opportunity, even though its FDR is undefined.\n from wildcard_fixture_analysis import evidence,team_statistics\n target_ids={r[\'team_id\'] for r in result[\'targets\']}\n blanks=[r for r in result[\'rows\'] if r[\'team_id\'] not in target_ids and not any(w.get(\'fixtures\') for w in r[\'weeks\'])]\n if blanks:\n  existing=result[\'avoids\'];chosen=[]\n  for r in blanks+existing:\n   if r[\'team_id\'] in {x[\'team_id\'] for x in chosen}:continue\n   item=copy.deepcopy(r);item.update(fixture_evidence=evidence(item,result[\'gameweeks\']),team_statistics=team_statistics(item[\'team_id\'],context.get(\'official_fixtures\',[])),players=[]);chosen.append(item)\n   if len(chosen)==3:break\n  result[\'avoids\']=chosen;result[\'narrative_avoids\']=list(reversed(chosen))\n if len(result[\'targets\'])!=5 or len(result[\'avoids\'])!=3:raise ValueError(\'Free Hit fixture analysis needs five target clubs and three avoids\')\n # Fixture rankings remain independent of player availability. Missing candidates are disclosed, never fabricated.\n result[\'player_coverage\']=[{\'team_id\':int(row[\'team_id\']), \'club\':row[\'club\'],\n     \'verified_candidates\':len(row.get(\'players\',[])), \'comparison_available\':len(row.get(\'players\',[]))==2}\n     for row in result[\'targets\']]\n result[\'coverage_warnings\']=[f"{row[\'club\']}: {row[\'verified_candidates\']} verified player candidates"\n     for row in result[\'player_coverage\'] if row[\'verified_candidates\']<2]\n result[\'narration_timeline\']=fixture_segments(result,context.get(\'selected_squad\',[]))\n result[\'narration_text\']=\' \'.join(s[\'narration_text\'] for s in result[\'narration_timeline\'])\n result[\'analyst_version\']=VERSION\n return result\n\ndef enrich(d,bootstrap,models,manager_squad=None,official_fixtures=()):\n p=copy.deepcopy(d[\'slide3_plan\']);p[\'end_gw\']=p[\'next_gw\'];p[\'gameweek_plans\']=[]\n cards=[]\n for c in d[\'squad\']:\n  cards.append(dict(c,weeks=[dict(gameweek=p[\'next_gw\'],fixtures=c.get(\'current_fixtures\',[]))]))\n p=enrich_squad(p,dict(squad=cards),bootstrap)\n byid={x[\'id\']:x for x in p[\'squad\']};vals=[number(x.get(\'projected_points\')) for x in p[\'starting_xi\']]\n p[\'xi_xpts\']=sum(vals) if all(v is not None for v in vals) else None\n cap=number(p[\'captain\'].get(\'projected_points\'));p[\'captain_adjusted_xpts\']=p[\'xi_xpts\']+cap if cap is not None and p[\'xi_xpts\'] is not None else None\n if p[\'xi_xpts\'] is None:raise ValueError(\'Free Hit starting projections incomplete\')\n p[\'analyst_version\']=VERSION;p[\'narration_timeline\']=squad_segments(p);d[\'slide3_plan\']=p\n d[\'summary_stats\'][\'starting_xi_xpts\']=p[\'xi_xpts\'];d[\'summary_stats\'][\'captain_adjusted_xpts\']=p[\'captain_adjusted_xpts\']\n from freehit_premium import prepare\n return prepare(d,bootstrap)\n\n',encoding='utf-8')
sys.modules.pop('freehit_analyst',None)
(_vx_module_dir/'vortex_wildcard.py').write_text(_VX_MODULE_SOURCE,encoding='utf-8')
if str(_vx_module_dir) not in sys.path:sys.path.insert(0,str(_vx_module_dir))
import importlib
importlib.invalidate_caches()
# Changing a .py file in Colab does not replace objects already imported in this kernel.
from pathlib import Path
for _vx_solver_module in ('wildcard_slide3', 'wildcard_plan'):
    sys.modules.pop(_vx_solver_module, None)
    Path(importlib.util.cache_from_source(str(_vx_module_dir / (_vx_solver_module + '.py')))).unlink(missing_ok=True)
Path(importlib.util.cache_from_source(str(_vx_module_dir / 'wildcard_intro.py'))).unlink(missing_ok=True)
import wildcard_intro
importlib.reload(wildcard_intro)
Path(importlib.util.cache_from_source(str(_vx_module_dir / 'freehit_intro.py'))).unlink(missing_ok=True)
import freehit_intro
importlib.reload(freehit_intro)
Path(importlib.util.cache_from_source(str(_vx_module_dir / 'vortex_wildcard.py'))).unlink(missing_ok=True)
import vortex_wildcard as VXWC
importlib.reload(VXWC)
print("✅ CELL 2 COMPLETE")
(_vx_module_dir/'tts_narration.py').write_text('"""Spoken-only FPL normalization. No network calls or display-text mutations."""\nimport re\nVERSION = \'spoken-fpl-v7-audio-exact-web-name\'\nVOICE = \'en-GB-RyanNeural\'\nPROFILE = {\'rate\': \'+7%\', \'pitch\': \'+0Hz\', \'volume\': \'+0%\'}\nCLUBS = dict(zip(\'ARS BHA MCI MUN LIV CHE NEW TOT AVL WHU FUL EVE BRE CRY BOU NFO WOL LEI IPS SOU SUN LEE BUR SHU\'.split(), [\'Arsenal\',\'Brighton\',\'Manchester City\',\'Manchester United\',\'Liverpool\',\'Chelsea\',\'Newcastle United\',\'Tottenham Hotspur\',\'Aston Villa\',\'West Ham United\',\'Fulham\',\'Everton\',\'Brentford\',\'Crystal Palace\',\'Bournemouth\',\'Nottingham Forest\',\'Wolverhampton Wanderers\',\'Leicester City\',\'Ipswich Town\',\'Southampton\',\'Sunderland\',\'Leeds United\',\'Burnley\',\'Sheffield United\']))\nTERMS = {\'xGI\':\'expected goal involvement\',\'xG\':\'expected goals\',\'xA\':\'expected assists\',\'xPts\':\'expected points\',\'xP\':\'expected points\',\'FDR\':\'fixture difficulty rating\',\'CS\':\'clean-sheet potential\',\'PPM\':\'points per million\',\'DGW\':\'Double Gameweek\',\'BGW\':\'Blank Gameweek\',\'GW\':\'Gameweek\',\'BB\':\'Bench Boost\',\'FH\':\'Free Hit\',\'WC\':\'Wildcard\',\'VC\':\'vice-captain\',\'mins\':\'minutes\',\'apps\':\'appearances\',\'app\':\'appearance\',\'xMins\':\'expected minutes\',\'xMINS\':\'expected minutes\',\'DEFCON\':\'defensive contribution\',\'GKP\':\'goalkeeper\',\'GK\':\'goalkeeper\',\'DEF\':\'defender\',\'MID\':\'midfielder\',\'FWD\':\'forward\',\'FPL\':\'Fantasy Premier League\'}\nONES = \'zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen\'.split()\nTENS = \'zero ten twenty thirty forty fifty sixty seventy eighty ninety\'.split()\ndef number(value):\n    value = str(value).replace(\',\', \'\')\n    if \'.\' in value:\n        a,b=value.split(\'.\',1)\n        return number(a)+\' point \'+\' \'.join(ONES[int(c)] for c in b)\n    n=int(value)\n    if n<0:return \'minus \'+number(-n)\n    if n<20:return ONES[n]\n    if n<100:return TENS[n//10]+(\' \'+number(n%10) if n%10 else \'\')\n    for scale,label in ((1000000000,\'billion\'),(1000000,\'million\'),(1000,\'thousand\'),(100,\'hundred\')):\n        if n>=scale:return number(n//scale)+\' \'+label+(\' \'+(\'and \' if scale==100 else \'\')+number(n%scale) if n%scale else \'\')\ndef prepare_tts(original, bootstrap=None, player_ids=()):\n    original=str(original); text=original; changes=[]; unresolved=[]\n    def sub(pattern, replacement, flags=0):\n        nonlocal text\n        def apply(m):\n            new=replacement(m) if callable(replacement) else m.expand(replacement)\n            if new!=m.group():changes.append({\'original\':m.group(),\'replacement\':new})\n            return new\n        text=re.sub(pattern,apply,text,flags=flags)\n    bootstrap=bootstrap or {}; clubs=dict(CLUBS)\n    for team in bootstrap.get(\'teams\',[]):\n        if team.get(\'short_name\') and team.get(\'name\'):\n            code=team[\'short_name\']; clubs[code]=CLUBS.get(code,team[\'name\'])\n    scope={int(player_id) for player_id in player_ids if str(player_id).isdigit()}\n    names_by_alias={}\n    for player in bootstrap.get(\'elements\',[]):\n        player_id=player.get(\'id\')\n        first=str(player.get(\'first_name\') or \'\').strip()\n        surname=str(player.get(\'second_name\') or \'\').strip()\n        web=str(player.get(\'web_name\') or \'\').strip()\n        if not (first and surname and web):\n            continue\n        # Preserve the precise official FPL web_name, including meaningful initials.\n        preferred=web\n        first_token=first.split()[0]\n        last_token=surname.split()[-1]\n        aliases={\n            first+\' \'+surname,\n            first_token+\' \'+surname,\n            first_token+\' \'+last_token,\n            first_token+\' \'+web,\n            first_token[0]+\'. \'+surname,\n            first_token[0]+\'. \'+web,\n            first_token[0]+\'.\'+web,\n            first_token[0]+\'. \'+last_token,\n            first_token[0]+\'.\'+last_token,\n            web,\n        }\n        for alias in aliases:\n            alias=alias.strip()\n            if alias and alias!=preferred:\n                names_by_alias.setdefault(alias,[]).append((player_id,preferred))\n    spoken_aliases={}\n    for alias,candidates in names_by_alias.items():\n        # Prefer the current scene when an alias represents multiple players.\n        scoped=[(player_id,name) for player_id,name in candidates if player_id in scope]\n        # Cues carry player IDs. Only use them to resolve duplicate aliases;\n        # full names outside that scope can still appear in comparative sentences.\n        applicable=scoped if scoped else candidates\n        replacements={preferred for _,preferred in applicable}\n        if len(replacements)==1:\n            spoken_aliases[alias]=next(iter(replacements))\n        elif re.search(r\'(?<!\\w)\'+re.escape(alias)+r\'(?!\\w)\',text):\n            unresolved.append(\'Ambiguous player: \'+alias)\n    if spoken_aliases:\n        alternatives=sorted(spoken_aliases,key=len,reverse=True)\n        sub(r\'(?<!\\w)(?:\'+\'|\'.join(map(re.escape,alternatives))+r\')(?!\\w)\',lambda m:spoken_aliases[m[0]])\n    # Do not let ordinary speech abbreviations re-expand a player name later.\n    for player in bootstrap.get(\'elements\',[]):\n        player_id=player.get(\'id\')\n        if scope and player_id not in scope:\n            continue\n        web=str(player.get(\'web_name\') or \'\').strip()\n        first=str(player.get(\'first_name\') or \'\').strip()\n        surname=str(player.get(\'second_name\') or \'\').strip()\n        if not (web and first and surname):\n            continue\n        longer={first+\' \'+surname, first.split()[0]+\' \'+surname,\n                first.split()[0]+\' \'+surname.split()[-1], first.split()[0]+\' \'+web}\n        for name in longer:\n            if name!=web and re.search(r\'(?<!\\w)\'+re.escape(name)+r\'(?!\\w)\',text):\n                unresolved.append(\'Unshortened player: \'+name)\n    sub(r\'\\bFPL\\s+VORTEX\\b\',\'F P L Vortex\')\n    sub(r\'\\bVORTEX\\b\',\'Vortex\')\n    sub(r\'\\b(HOLD|PLAY|WAIT)\\b\',lambda m:m[0].lower())\n    sub(r\'\\b(xGI|xG|xA|xPts|xP)\\s*/\\s*90\\b\',lambda m:TERMS[m[1]]+\' per ninety minutes\')\n    codes=\'(?:\'+\'|\'.join(map(re.escape,sorted(clubs,key=len,reverse=True)))+\')\'\n    sub(r\'\\b(\'+codes+r\')\\s*(?:v(?:s)?\\.?\\s*|(?=\\s)\\s+)(\'+codes+r\')\\b\',lambda m:clubs[m[1]]+\' at home to \'+clubs[m[2]])\n    sub(r\'\\b(\'+codes+r\')\\s*[\\[(]\\s*([HA])\\s*[\\])]\',lambda m:clubs[m[1]]+(\' at home\' if m[2]==\'H\' else \' away\'))\n    sub(r\'\\b(\'+codes+r\')\\s+([HA])\\b\',lambda m:clubs[m[1]]+(\' at home\' if m[2]==\'H\' else \' away\'))\n    sub(r\'\\b(?:DGW|BGW|GW)\\s*(\\d+)\\b\',lambda m:TERMS[re.match(\'[A-Z]+\',m[0])[0]]+\' \'+number(m[1]))\n    sub(r\'£\\s*(\\d+(?:\\.\\d+)?)\\s*(?:[mM]\\b|million\\b)(?:\\s+(?:in the bank|ITB))?\',lambda m:(\'zero million pounds in the bank\' if float(m[1])==0 else number(m[1])+\' million pounds\'))\n    sub(r\'\\b([1-5])-([1-5])-([1-5])\\b(?:\\s+formation)?\',lambda m:\' \'.join(number(m[i]) for i in (1,2,3))+\' formation\')\n    sub(r\'\\b(\\d+(?:\\.\\d+)?)%\',lambda m:number(m[1])+\' percent\')\n    for token,spoken in sorted({**TERMS,**clubs}.items(),key=lambda pair:-len(pair[0])):\n        sub(r\'(?<!\\w)\'+re.escape(token)+r\'(?!\\w)\',spoken)\n    sub(r\'(?<!\\w)C(?=\\s*[:=]|\\s*\\))\',\'captain\')\n    sub(r\'(?m)^C(?=\\s+[A-Z])\',\'captain\')\n    sub(r\'\\(C\\)\', \'(captain)\')\n    sub(r\'\\b\\d+\\.\\d+\\b\',lambda m:number(m[0]))\n    sub(r\'(?i)\\b(Gameweek)\\s+(\\d+)\\b\',lambda m:m[1]+\' \'+number(m[2]))\n    sub(r\'\\b\\d+\\b\',lambda m:number(m[0]))\n    sub(r\'<[^>]+>\',\' \')\n    sub(r\'\\.{2,}|…\',\'.\')\n    sub(r\'[;\\n]+\',\'. \')\n    sub(r\'\\s+\',\' \')\n    text=text.strip()\n    # Break comma lists into at most three items per sentence.\n    sentences=re.split(r\'(?<=[.!?])\\s+\',text)\n    normalized=[]\n    for sentence in sentences:\n        pieces=sentence.split(\', \')\n        if len(pieces)>3:\n            sentence=\'. \'.join(\', \'.join(pieces[i:i+3]) for i in range(0,len(pieces),3))\n        if len(sentence.split())>40:\n            sentence=re.sub(r\',?\\s+(and|but|while|because)\\s+\',lambda m:\'. \'+m[1].capitalize()+\' \',sentence,count=1)\n        normalized.append(sentence)\n    cleaned=\' \'.join(normalized)\n    if cleaned!=text:changes.append({\'original\':text,\'replacement\':cleaned})\n    text=cleaned\n    for token in {**TERMS,**clubs}:\n        if re.search(r\'(?<!\\w)\'+re.escape(token)+r\'(?!\\w)\',text):unresolved.append(token)\n    # Initialled official web names are valid player names, not unresolved codes.\n    official_web_names={str(p.get(\'web_name\') or \'\').strip() for p in bootstrap.get(\'elements\',[])}\n    unresolved += [candidate for candidate in re.findall(r\'\\b(?:[A-Z]\\.\\s*)+[A-Z][\\w’\\\'-]+\',text)\n                   if candidate not in official_web_names]\n    unresolved+=re.findall(r\'\\b[A-Z]{2,5}\\s*(?:\\([HA]\\)|\\bv\\b\\s*[A-Z]{2,5})\',text)\n    # Unknown uppercase codes/acronyms fail closed instead of being guessed.\n    unresolved+=re.findall(r\'\\b[A-Z]{2,}\\b\',text)\n    unresolved=list(dict.fromkeys(unresolved))\n    return {\'original_narration\':original,\'tts_cleaned_narration\':text,\'replacements_applied\':changes,\'unresolved_tokens\':unresolved,\'validation_passed\':not unresolved}\n')
import sys
sys.modules.pop('tts_narration',None)
(VX_MEDIA_WORK / 'wildcard_engine' / 'wildcard_intro_motion.py').write_text('"""Wildcard Slide 1: narration-synchronised player heroes with the approved broadcast shell."""\nfrom __future__ import annotations\n\nfrom functools import lru_cache\nfrom pathlib import Path\nimport math\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\nVERSION = \'wildcard-slide1-ryan-cue-v7\'\nGREEN = \'#75edb8\'\nCYAN = \'#8bdff4\'\nGOLD = \'#f1d286\'\nWHITE = \'#f0f7ff\'\nMUTED = \'#bad3e1\'\nDARK = \'#081c2d\'\n\n\ndef ease(value: float) -> float:\n    progress = min(1.0, max(0.0, float(value)))\n    return 1.0 - (1.0 - progress) ** 3\n\n\n@lru_cache(maxsize=160)\ndef font(size: int, bold: bool = True):\n    from vortex_wildcard_visuals import font as shared_font\n    return shared_font(max(14, int(size)), bold)\n\n\nclass WildcardIntroMotion:\n    requires_motion = True\n\n    def __init__(self, scene, contract, width, height, media, badges, brand_root, background=None):\n        self.manifest = scene\n        self.contract = contract\n        self.width, self.height = int(width), int(height)\n        self.layout = []\n        self.timeline = scene[\'narration_timeline\']\n        self.case = scene[\'wildcard_case\']\n        self.data = self.case.get(\'intro_data\', {})\n        self.media = media or {}\n        self.badges = badges or {}\n        self.assets = {}\n        self.sprite_cache = {}\n        self.brand_root = Path(brand_root)\n        if not self.timeline or any(cue.get(\'start\') is None or cue.get(\'end\') is None for cue in self.timeline):\n            raise ValueError(\'Wildcard Slide 1 requires measured narration cues\')\n        if any(float(cue[\'end\']) <= float(cue[\'start\']) for cue in self.timeline):\n            raise ValueError(\'Wildcard Slide 1 has an invalid narration timeline\')\n        self.background = (background.convert(\'RGB\') if background is not None else Image.new(\'RGB\', (1920,1080), \'#071726\'))\n        self.background = self.background.resize((1920, 1080), Image.Resampling.LANCZOS)\n        for label in (\'fixture\',\'captain\'):\n            player = self.data.get(label) or {}\n            ident = player.get(\'id\')\n            record = self.media.get(ident) or self.media.get(str(ident)) or {}\n            portrait = record.get(\'photo\')\n            if record.get(\'photo_kind\') == \'player\' and portrait and Path(portrait).is_file():\n                self.load(label, portrait)\n            badge_path = (self.badges.get(player.get(\'club\')) or {}).get(\'path\')\n            if badge_path and Path(badge_path).is_file():\n                self.load(label + \'_crest\', badge_path)\n        self.complete = self.draw(max(0, float(self.timeline[-1][\'end\']) - .05), float(self.timeline[-1][\'end\']))\n\n    def load(self, key, path):\n        with Image.open(path) as source:\n            self.assets[key] = source.convert(\'RGBA\')\n\n    def asset(self, target, key, rectangle, progress=1.0):\n        source = self.assets.get(key)\n        if source is None or progress <= 0:\n            return False\n        x,y,width,height = rectangle\n        q=ease(progress)\n        factor=min(width/source.width,height/source.height)*(.86+.14*q)\n        dimensions=(max(1,int(source.width*factor)),max(1,int(source.height*factor)))\n        cache_key=(key,dimensions)\n        if cache_key not in self.sprite_cache:\n            if len(self.sprite_cache)>100:self.sprite_cache.clear()\n            self.sprite_cache[cache_key]=source.resize(dimensions,Image.Resampling.LANCZOS)\n        sprite=self.sprite_cache[cache_key]\n        if q < 1:\n            sprite=sprite.copy()\n            sprite.putalpha(sprite.getchannel(\'A\').point(lambda value: int(value*q)))\n        target.alpha_composite(sprite,(int(x+(width-sprite.width)/2),int(y+height-sprite.height)))\n        return True\n\n    def label(self, target, message, x, y, size, color=WHITE, reveal=1.0, bold=True, max_width=None):\n        q=ease(reveal)\n        if q<=0:return\n        text=str(message)\n        painter=ImageDraw.Draw(target)\n        size=int(size)\n        face=font(size,bold)\n        if max_width:\n            while size>19 and painter.textlength(text,font=face)>max_width:\n                size-=2\n                face=font(size,bold)\n        width=max(1,int(painter.textlength(text,font=face))+15)\n        layer=Image.new(\'RGBA\',(width,size+24),(0,0,0,0))\n        ImageDraw.Draw(layer).text((0,-3),text,font=face,fill=color)\n        layer=layer.crop((0,0,max(1,int(layer.width*q)),layer.height))\n        target.alpha_composite(layer,(int(x),int(y+(1-q)*19)))\n\n    def rounded(self, target, box, fill, outline=None, radius=19, width=2):\n        ImageDraw.Draw(target).rounded_rectangle(tuple(round(v) for v in box),radius=radius,fill=fill,outline=outline,width=width)\n\n    @staticmethod\n    def _cue_length(cue):\n        return max(.10, float(cue[\'end\']) - float(cue[\'start\']))\n\n    def _beat(self, cue, elapsed, fraction):\n        duration = self._cue_length(cue)\n        onset = duration * max(0.0, min(.95, float(fraction)))\n        # Frame-driven entrance; retain readable dwell time within the voice cue.\n        entrance = max(.12, min(.38, duration * .065))\n        return ease((float(elapsed) - onset) / entrance)\n\n    def _word_start(self, cue, keywords, fallback):\n        """Align metric entrances with Ryan\'s spoken word when a boundary exists.\n\n        Edge-TTS offsets are 100ns ticks from the start of the current speech cue.\n        An older cached voice lacking boundaries uses a measured-cue approximation.\n        """\n        duration = self._cue_length(cue)\n        words = cue.get(\'word_boundaries\') or []\n        for needle in keywords:\n            target = str(needle).casefold().strip()\n            if not target:\n                continue\n            for word in words:\n                token = \'\'.join(character for character in str(word.get(\'text\', \'\')).casefold() if character.isalnum())\n                if token == target:\n                    try:\n                        offset = float(word.get(\'offset_ticks\', 0)) / 10_000_000.0\n                    except (ValueError, TypeError):\n                        continue\n                    if math.isfinite(offset):\n                        return max(0.02, min(.88, offset / duration))\n        spoken = str(cue.get(\'narration_text\', \'\')).casefold()\n        matches = [spoken.find(str(needle).casefold()) for needle in keywords]\n        matches = [match for match in matches if match >= 0]\n        if matches:\n            words_before = len(spoken[:min(matches)].split())\n            total_words = max(1, len(spoken.split()))\n            return max(.02, min(.85, words_before / total_words))\n        return fallback\n\n    def chapter(self, clock):\n        for cue in self.timeline:\n            if clock < cue[\'end\']:\n                return cue,max(0.,clock-cue[\'start\'])\n        cue=self.timeline[-1]\n        return cue,max(0.,clock-cue[\'start\'])\n\n    def _stat(self, target, x, y, title, value, accent, reveal):\n        q=ease(reveal)\n        if q<=0:return\n        xx=x+(1-q)*28\n        self.rounded(target,(xx,y,xx+304,y+103),\'#183c50\',\'#4c7788\',18)\n        self.rounded(target,(xx+15,y+12,xx+22,y+91),accent,radius=3)\n        self.label(target,title,xx+34,y+11,22,MUTED,q,max_width=247)\n        self.label(target,value,xx+34,y+43,40,WHITE,q,max_width=256)\n\n    def _generic(self, target, cue, u):\n        kind = cue[\'type\']\n        weeks = self.contract.get(\'horizon_gameweeks\') or []\n        window = f\'GW {weeks[0]}–{weeks[-1]}\' if weeks else \'PLANNING WINDOW\'\n        titles = {\n            \'question\': (\'THINKING OF A WILDCARD?\', \'Consider the rebuild before pressing the button\'),\n            \'context\': (\'DOES YOUR TEAM NEED A REBUILD?\', \'Squad needs or disappointing Gameweeks?\'),\n            \'discipline\': (\'WHEN A REBUILD MAKES SENSE\', \'If injuries pile up and serious changes are needed\'),\n            \'draft\': (\'MY WILDCARD DRAFT\', window),\n            \'plan\': (\'BUILD BEYOND ONE FIXTURE\', \'Minutes, matchups, attacking potential and captaincy\'),\n            \'decision_question\': (\'IS THE CHIP WORTH USING?\', \'Does the overall squad justify the rebuild?\'),\n            \'preview\': ("LET\'S LOOK THROUGH THE PLAN", \'The draft and the chip decision\'),\n            \'transition\': (\'UP NEXT: THE FIXTURES\', \'Start with the matchups across the planning window\')}\n        title, description = titles.get(kind, (\'WILDCARD PLAN\', \'Verified model evidence\'))\n        # One voice-led reveal at a time; the shell remains unobstructed.\n        self.label(target, title, 995, 284, 37, WHITE, self._beat(cue, u, .06), max_width=770)\n        self.label(target, description.upper(), 998, 349, 25, MUTED, self._beat(cue, u, .23), max_width=747)\n        items = (\n            (1000, 468, \'EVIDENCE FIRST\', f\'{len(weeks)} GAMEWEEKS\', GREEN, .35),\n            (1350, 468, \'PLAYER MINUTES\', \'ROLE CHECK\', CYAN, .49),\n            (1000, 608, \'CHIP DECISION\', \'TRANSFER COST\', GOLD, .64),\n            (1350, 608, \'SQUAD RULE\', \'LEGAL 15\', GREEN, .78),\n        )\n        for x, y, label, value, color, fraction in items:\n            self._stat(target, x, y, label, value, color, self._beat(cue, u, fraction))\n        self.label(target, \'THE PLAYER CARDS APPEAR WHEN RYAN NAMES THE PICK.\',\n                   1002, 801, 25, CYAN, self._beat(cue, u, .89), max_width=750)\n\n    def _hero(self, target, cue, u):\n        kind = cue[\'type\']\n        role = \'fixture\' if kind == \'fixture_player\' else \'captain\'\n        player = self.data.get(role) or {}\n        if not player or not player.get(\'id\'):\n            self._generic(target, dict(cue, type=\'draft\'), u)\n            return\n        q = self._beat(cue, u, .035)\n        if q <= 0:\n            return\n        x, y = 996, 265 + (1 - q) * 42\n        self.rounded(target, (x, y, x+782, y+612), \'#0c2c40\',\n                     \'#94d6ba\' if role == \'fixture\' else \'#dfc580\', 29, 3)\n        self.rounded(target, (x+18, y+18, x+765, y+71),\n                     \'#175044\' if role == \'fixture\' else \'#55462c\', radius=14)\n        self.label(target, \'FIXTURE-DRIVEN HERO\' if role == \'fixture\' else \'A BIG PART OF THE PLAN\',\n                   x+38, y+28, 28, GREEN if role == \'fixture\' else GOLD,\n                   self._beat(cue, u, .08))\n        photo_x, photo_y = x+36, y+110\n        self.rounded(target, (photo_x, photo_y, photo_x+286, photo_y+284), \'#194557\', \'#4e8b93\', 25)\n        photo_entrance = self._beat(cue, u, .11)\n        if not self.asset(target, role, (photo_x+7, photo_y+8, 272, 272), photo_entrance):\n            self.label(target, str(player.get(\'spoken_name\') or player.get(\'name\') or \'PLAYER\').upper(),\n                       photo_x+20, photo_y+115, 36, WHITE, photo_entrance, max_width=250)\n        self.asset(target, role+\'_crest\', (x+344, y+106, 80, 80), self._beat(cue, u, .18))\n        display = str(player.get(\'spoken_name\') or player.get(\'name\') or \'\').upper()\n        self.label(target, display, x+351, y+194, 52, WHITE,\n                   self._beat(cue, u, .15), max_width=385)\n        self.label(target, str(player.get(\'club_name\') or player.get(\'club\') or \'\').upper(),\n                   x+351, y+264, 27, CYAN, self._beat(cue, u, .29), max_width=380)\n        weeks = self.contract.get(\'horizon_gameweeks\') or []\n        short_horizon = f\'GW {weeks[0]} - {weeks[-1]}\' if len(weeks)>1 else \'PLANNING WINDOW\'\n        self.label(target, short_horizon, x+351, y+313, 29, MUTED,\n                   self._beat(cue, u, .37), max_width=360)\n        points = player.get(\'horizon_expected_points\')\n        primary_anchor = self._word_start(cue, (\'projected\', \'projection\'), .45)\n        points_reveal = self._beat(cue, u, primary_anchor)\n        try:\n            points = float(points)\n            point_value = f\'{points * points_reveal:.1f}\' if math.isfinite(points) else \'UNAVAILABLE\'\n        except (ValueError, TypeError):\n            point_value = \'UNAVAILABLE\'\n        self._stat(target, x+32, y+419, f\'{len(weeks)}-GW xPTS\', point_value, GOLD, points_reveal)\n        if role == \'captain\':\n            opening_anchor = self._word_start(cue, (\'opening\',), .75)\n            opening_reveal = self._beat(cue, u, opening_anchor)\n            try:\n                opening = float(player.get(\'expected_points\'))\n                opening_value = f\'{opening * opening_reveal:.1f}\' if math.isfinite(opening) else \'UNAVAILABLE\'\n            except (ValueError, TypeError):\n                opening_value = \'UNAVAILABLE\'\n            self._stat(target, x+367, y+419, \'OPENING xPTS\', opening_value, GREEN, opening_reveal)\n        else:\n            try:\n                xgi = float(player.get(\'xg90\')) + float(player.get(\'xa90\'))\n                value = f\'{xgi * self._beat(cue, u, .78):.2f}\' if math.isfinite(xgi) else \'UNAVAILABLE\'\n            except (ValueError, TypeError):\n                value = \'UNAVAILABLE\'\n            self._stat(target, x+367, y+419, \'xGI PER 90\', value, GREEN, self._beat(cue, u, .75))\n        self.label(target, \'MODEL PROJECTION • NOT A GUARANTEE OF RETURNS\',\n                   x+47, y+554, 24, MUTED, self._beat(cue, u, .88), max_width=677)\n        duration = self._cue_length(cue)\n        ratio = min(1, max(0, u / duration))\n        self.rounded(target, (x+34, y+592, x+746, y+599), \'#305c65\', radius=3)\n        if ratio > 0:\n            self.rounded(target, (x+34, y+592, x+34+max(4,int(712*ratio)), y+599),\n                         GREEN if role == \'fixture\' else GOLD, radius=3)\n\n    def scene_image(self, cue, u, t):\n        kind=cue[\'type\']\n        surface=self.background.convert(\'RGBA\')\n        overlay=Image.new(\'RGBA\',surface.size,(0,0,0,0))\n        self.rounded(overlay,(82,196,1835,967),(4,16,29,237),(53,103,122,255),27,2)\n        self.rounded(overlay,(107,216,917,941),(8,33,47,249),(44,93,108,255),21,2)\n        self.rounded(overlay,(946,216,1813,941),(7,26,44,239),(44,89,107,255),21,2)\n        surface=Image.alpha_composite(surface,overlay)\n        # The left-side composition is deliberately complete at 00:00.\n        weeks=self.contract.get(\'horizon_gameweeks\') or []\n        self.label(surface,f\'WILDCARD  /  {len(weeks)}-GAMEWEEK PLAN\',146,253,28,CYAN)\n        self.label(surface,\'WILDCARD:\',145,332,80,WHITE)\n        self.label(surface,\'NOW OR WAIT?\',145,436,77,GREEN)\n        self.label(surface,\'A DATA-LED DECISION, NOT A DEADLINE GAMBLE.\',146,550,29,MUTED,max_width=744)\n        weeks=self.contract.get(\'horizon_gameweeks\') or []\n        horizon=f\'GW {weeks[0]}–{weeks[-1]}\' if len(weeks)>1 else f\'GW {self.contract["gameweek"]}\'\n        self.rounded(surface,(143,642,492,759),\'#133248\',\'#467287\',17)\n        self.rounded(surface,(516,642,869,759),\'#123e37\',\'#609b82\',17)\n        self.label(surface,\'ACTIVE GAMEWEEK\',161,655,23,MUTED)\n        self.label(surface,f\'GW {self.contract["gameweek"]}\',160,694,43,WHITE)\n        self.label(surface,\'PLANNING WINDOW\',534,655,23,MUTED)\n        self.label(surface,horizon,533,694,42,GREEN)\n        self.rounded(surface,(143,806,869,891),\'#103144\',\'#3e6376\',14)\n        # Left editorial panel is final and stable from exactly 00:00.\n        self.label(surface, f\'THE WILDCARD DECISION • {len(weeks)}-GW EVIDENCE\', 164, 830, 29, GOLD, max_width=680)\n        self.rounded(surface, (144,910,869,920), \'#315466\', radius=4)\n        self.rounded(surface, (144,910,869,920), GREEN, radius=4)\n        if kind in (\'fixture_player\',\'captain_player\'):\n            self._hero(surface,cue,u)\n        else:\n            self._generic(surface,cue,u)\n        return surface.convert(\'RGB\')\n\n    def draw(self, seconds, duration):\n        t=max(0.,min(float(seconds),float(self.timeline[-1][\'end\'])))\n        cue,u=self.chapter(t)\n        image=self.scene_image(cue,u,t)\n        if image.size!=(self.width,self.height):\n            image=image.resize((self.width,self.height),Image.Resampling.LANCZOS)\n        return image\n\n    def frame(self, seconds, duration, previous=None):\n        return np.asarray(self.draw(seconds,duration))\n', encoding='utf-8')

✅ CELL 2 COMPLETE


15757

In [4]:
#@title 🧪 CELL 2A - WILDCARD OPTIMISER REGRESSION CHECKS {display-mode: "form"}
from collections import Counter
from wildcard_slide3 import build_wildcard_squad, attach_unprojected_goalkeepers, prepare_pool

def _vx_test_wildcard_cover_eligibility():
    # A secure XI with two dependable-but-not-core bench defenders must be feasible.
    prices = [42, 45, 48, 48, 48, 52, 52, 62, 62, 70, 72, 72, 72, 70, 75]
    types = [1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4]
    players=[];models=[];weekly=[]
    for index,(cost,position) in enumerate(zip(prices,types),1):
        role_is_cover=(position==2 and index in (6,7))
        p_start=.77 if role_is_cover else (.95 if position==1 else .90)
        minutes=61 if role_is_cover else (87 if position==1 else 78)
        players.append(dict(id=index,first_name='Test',second_name=f'Player{index}',web_name=f'P{index}',team=index,element_type=position,now_cost=cost,status='a',removed=False,can_select=True,chance_of_playing_next_round=None))
        models.append(dict(player_id=index,p_start=p_start,expected_minutes=minutes,news_monitor=False,xpts_wildcard_horizon=float(index+20)))
        weekly.extend(dict(player_id=index,gameweek=week,fixture_count=1,gameweek_expected_points=float(index+20)/6,expected_minutes=minutes,p_start=p_start) for week in range(7,13))
    teams=[dict(id=index,name=f'Team{index}',short_name=f'T{index}') for index in range(1,16)]
    context=dict(models=models,weekly_models=weekly,currency_multiplier=10,available_gameweeks=list(range(7,13)),solver_seconds=15)
    pool=prepare_pool(players,teams,context,list(range(7,13)))
    assert sum(p['secure'] for p in pool if p['position']=='DEF')==3
    assert sum(p['cover'] for p in pool if p['position']=='DEF')==5
    total_price=sum(p['now_cost'] for p in players)/10
    plan=build_wildcard_squad(players,teams,[],7,budget=total_price,context=context)
    assert len(plan['squad'])==15 and len(plan['starting_xi'])==11
    assert Counter(p['position'] for p in plan['bench'])==Counter({'GK':1,'DEF':2,'MID':1})
    assert all(p['cover'] for p in plan['bench'] if p['position']=='DEF')
    assert all(p['secure'] for p in plan['starting_xi'])
    # A manager can legally leave cash unused; the old £0.3m hard floor made
    # this otherwise-valid £89m selection infeasible when budget was £100m.
    flexible_bank_plan=build_wildcard_squad(players,teams,[],7,budget=100.,context=context)
    assert len(flexible_bank_plan['squad'])==15
    assert flexible_bank_plan['bank']>=10.9
    assert flexible_bank_plan['selection_summary']['selection_policy']=='bank-preference-relaxed'
    assert all(p['secure'] for p in flexible_bank_plan['starting_xi'])
    assert all(p['cover'] for p in flexible_bank_plan['bench'] if p['position']=='DEF')
    assert len(flexible_bank_plan['gameweek_plans'])==6
    # An 85%-start, 72-minute goalkeeper is backed by real projections; rejecting
    # them solely for missing an arbitrary 90%/80-minute gate was the Cell 11 bug.
    model_by_id={row['player_id']:row for row in models}
    model_by_id[2]['p_start']=.85
    model_by_id[2]['expected_minutes']=72
    for row in weekly:
        if row['player_id']==2:
            row['p_start']=.85
            row['expected_minutes']=72
    less_certain_pool=prepare_pool(players,teams,context,list(range(7,13)))
    assert sum(bool(player['nailed']) for player in less_certain_pool if player['position']=='GK')>=1
    less_certain_plan=build_wildcard_squad(players,teams,[],7,budget=total_price,context=context)
    starting_keeper=next(player for player in less_certain_plan['starting_xi'] if player['position']=='GK')
    assert starting_keeper['id']==2 and starting_keeper['starting_probability']==.85
    assert starting_keeper['minutes']==72
    assert len(less_certain_plan['gameweek_plans'])==6
    # Truly insufficient playing-time data remains a hard failure, never fake estimates.
    for row in models:
        if row['player_id'] in (1,2):
            row['p_start']=.40
            row['expected_minutes']=40
    try:
        build_wildcard_squad(players,teams,[],7,budget=total_price,context=context)
    except ValueError as failure:
        assert 'verified starting goalkeepers=0' in str(failure)
        assert 'Best goalkeeper evidence' in str(failure)
    else:
        raise AssertionError('Missing goalkeeper evidence must not be accepted')
    return True

assert _vx_test_wildcard_cover_eligibility()
print('✅ Wildcard solver regression: verified defensive cover works without weakening starting XI requirements')

✅ Wildcard solver regression: verified defensive cover works without weakening starting XI requirements


In [5]:
#@title 🔴 CELL 3 - OFFICIAL IMAGES {display-mode: "form"}
# 🔴 CELL 3 - OFFICIAL IMAGES
# One identity rule everywhere: FPL player ID -> official FPL code/photo_code -> official PL photo.
# No FPL-Core name rescue, no FotMob name search, and no AI/synthetic player portraits.
import importlib.util, subprocess, sys
for module, requirement in [('requests','requests'),('PIL','pillow')]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement], check=True)

_VX_MODULE_SOURCE = '"""FPL VORTEX player images: official FPL code -> Premier League photo chain -> VORTEX fallback.\n\nIdentity is never guessed from a player name and no AI/synthetic player portrait is generated.\n"""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom io import BytesIO\nfrom PIL import Image\nimport json\nimport tempfile\nimport requests\n\nPHOTO_RESOLVER_VERSION = "official-fpl-code-four-endpoints-v4"\nPL_PHOTO_BASE_CURRENT = "https://resources.premierleague.com/premierleague25/photos/players"\nPL_PHOTO_BASE_LEGACY = "https://resources.premierleague.com/premierleague/photos/players"\n\n_session = requests.Session()\n_session.headers.update({\n    "User-Agent": "Mozilla/5.0 FPL-VORTEX/4.0",\n    "Accept": "image/avif,image/webp,image/png,image/*,*/*;q=0.8",\n})\n\n\ndef _row(player):\n    if hasattr(player, "to_dict"):\n        player = player.to_dict()\n    return dict(player or {})\n\n\ndef _player_code(player):\n    row = _row(player)\n    value = row.get("photo_code") or row.get("code") or row.get("player_code")\n    if value in (None, ""):\n        return None\n    text = str(value).strip()\n    if text.lower().startswith("p"):\n        text = text[1:]\n    if ".png" in text.lower():\n        text = text.split(".", 1)[0]\n    try:\n        code = int(float(text))\n    except (TypeError, ValueError):\n        return None\n    return code if code > 0 else None\n\n\ndef official_photo_urls(fpl_player, season=None):\n    """Exact shared FPL VORTEX photo chain, ordered from preferred to legacy."""\n    code = _player_code(fpl_player)\n    if code is None:\n        return []\n    return [\n        f"{PL_PHOTO_BASE_CURRENT}/500x500/{code}.png",\n        f"{PL_PHOTO_BASE_CURRENT}/250x250/{code}.png",\n        f"{PL_PHOTO_BASE_LEGACY}/250x250/p{code}.png",\n        f"{PL_PHOTO_BASE_LEGACY}/110x140/p{code}.png",\n    ]\n\n\ndef _valid_image(payload):\n    if not isinstance(payload, (bytes, bytearray)) or len(payload) < 300:\n        return False\n    try:\n        with Image.open(BytesIO(payload)) as image:\n            image.verify()\n        with Image.open(BytesIO(payload)) as image:\n            image.load()\n        return True\n    except Exception:\n        return False\n\n\ndef _download_image(url):\n    try:\n        response = _session.get(url, timeout=(4, 10))\n        if response.status_code != 200:\n            return None\n        content_type = str(response.headers.get("content-type", "")).lower()\n        if content_type and not content_type.startswith("image/"):\n            return None\n        payload = bytes(response.content)\n        return payload if _valid_image(payload) else None\n    except requests.RequestException:\n        return None\n\n\ndef _save_png(payload, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(dir=path.parent,prefix=path.name+\'.\',suffix=\'.part\',delete=False) as handle:\n        temporary=Path(handle.name)\n    try:\n        with Image.open(BytesIO(payload)) as image:\n            image.convert("RGBA").save(temporary,format="PNG")\n        temporary.replace(path)\n    finally:temporary.unlink(missing_ok=True)\n    return path\n\n\ndef _official_fpl_shirt_urls(team_code):\n    return [\n        f"https://fantasy.premierleague.com/dist/img/shirts/standard/shirt_{team_code}-220.png",\n        f"https://fantasy.premierleague.com/dist/img/shirts/standard/shirt_{team_code}-110.png",\n    ]\n\n\ndef resolve_media(player_ids, work, bootstrap, core_root=None):\n    """Resolve current players strictly by official FPL id/code metadata."""\n    work = Path(work)\n    directory = work / "wildcard_player_media"\n    directory.mkdir(parents=True, exist_ok=True)\n\n    players = {int(player["id"]): player for player in bootstrap["elements"]}\n    teams = {int(team["id"]): team for team in bootstrap["teams"]}\n\n    deadlines = [\n        str(event.get("deadline_time") or "")\n        for event in bootstrap.get("events", [])\n        if event.get("deadline_time")\n    ]\n    year = min(int(value[:4]) for value in deadlines) if deadlines else datetime.now(timezone.utc).year\n    season = f"{year}-{year + 1}"\n\n    result = {}\n    for player_id in sorted(set(map(int, player_ids))):\n        if player_id not in players:\n            raise KeyError(f"Official FPL player id {player_id} is not present in bootstrap-static.")\n\n        player = players[player_id]\n        photo_code = _player_code(player)\n        team_id = int(player.get("team") or 0)\n        team = teams.get(team_id, {})\n        team_code = int(team.get("code") or 0)\n        team_short = str(team.get("short_name") or team.get("name") or "")\n        if photo_code is None:\n            raise RuntimeError(\n                f"Official FPL player {player_id} ({player.get(\'web_name\')}) has no usable player code."\n            )\n\n        path = directory / f"player_{player_id}_{photo_code}_{team_code}_official-v4.png"\n        stamp = path.with_suffix(".source.json")\n\n        old = {}\n        if stamp.exists():\n            try:\n                old = json.loads(stamp.read_text(encoding="utf-8"))\n            except (ValueError, OSError):\n                old = {}\n\n        try:\n            checked = datetime.fromisoformat(str(old.get("checked_at") or ""))\n        except (TypeError, ValueError):\n            checked = None\n\n        fresh = (\n            checked is not None\n            and checked.tzinfo is not None\n            and (datetime.now(timezone.utc) - checked).total_seconds() < 86400\n        )\n        valid_cache = (\n            path.is_file()\n            and fresh\n            and old.get("resolver_version") == PHOTO_RESOLVER_VERSION\n            and int(old.get("player_id") or -1) == player_id\n            and int(old.get("photo_code") or -1) == photo_code\n            and int(old.get("team_code") or -1) == team_code\n        )\n        if valid_cache:\n            try:\n                with Image.open(path) as image:\n                    image.verify()\n                with Image.open(path) as image:\n                    image.load()\n            except Exception:\n                valid_cache = False\n\n        if valid_cache:\n            item = dict(old)\n            item.update(\n                photo=str(path.resolve()),\n                player_id=player_id,\n                name=str(player.get("web_name") or ""),\n                photo_code=photo_code,\n                team_code=team_code,\n                club=team_short,\n                badge=None,\n            )\n            result[player_id] = item\n            continue\n\n        payload = None\n        source_url = None\n        photo_kind = "missing"\n\n        for candidate in official_photo_urls({"photo_code": photo_code}):\n            payload = _download_image(candidate)\n            if payload is not None:\n                source_url = candidate\n                photo_kind = "player"\n                break\n\n        # VORTEX fallback: official current FPL club shirt, never another person\'s face.\n        if payload is None and team_code > 0:\n            for candidate in _official_fpl_shirt_urls(team_code):\n                payload = _download_image(candidate)\n                if payload is not None:\n                    source_url = candidate\n                    photo_kind = "shirt"\n                    break\n\n        if payload is not None:\n            _save_png(payload, path)\n            photo_path = str(path.resolve())\n        else:\n            photo_path = None\n\n        item = {\n            "photo": photo_path,\n            "photo_kind": photo_kind,\n            "source_url": source_url,\n            "player_id": player_id,\n            "name": str(player.get("web_name") or ""),\n            "photo_code": photo_code,\n            "team_code": team_code,\n            "club": team_short,\n            "badge": None,\n            "resolver": (\n                "official FPL player_code -> Premier League 500x500 -> Premier League 250x250 "\n                "-> legacy Premier League 250x250 -> legacy Premier League 110x140 -> VORTEX shirt fallback"\n            ),\n            "resolver_version": PHOTO_RESOLVER_VERSION,\n            "season": season,\n            "identity_rule": "Official FPL player id -> official FPL code/photo_code only; no name matching",\n            "checked_at": datetime.now(timezone.utc).isoformat(),\n            "image_normalization": "official-original-aspect-v4",\n        }\n\n        if payload is not None:\n            stamp.write_text(json.dumps(item, indent=2), encoding="utf-8")\n        result[player_id] = item\n\n    (work / "wildcard_player_media_audit.json").write_text(\n        json.dumps(result, indent=2),\n        encoding="utf-8",\n    )\n    return result\n'
_vx_module_dir = VX_MEDIA_WORK / 'wildcard_engine'
_vx_module_dir.mkdir(parents=True, exist_ok=True)
(_vx_module_dir / 'vortex_player_images.py').write_text(_VX_MODULE_SOURCE, encoding='utf-8')
if str(_vx_module_dir) not in sys.path:
    sys.path.insert(0, str(_vx_module_dir))
import importlib
importlib.invalidate_caches()
Path(importlib.util.cache_from_source(str(_vx_module_dir / 'vortex_player_images.py'))).unlink(missing_ok=True)
print("✅ CELL 3 COMPLETE")

✅ CELL 3 COMPLETE


In [6]:
#@title 🔴 CELL 4 - WILDCARD VISUALS {display-mode: "form"}
# 🔴 CELL 4 - WILDCARD VISUALS
# This readable embedded module is materialised automatically; no extra download.
import importlib.util,subprocess,sys
for module,requirement in [('numpy','numpy'),('pandas','pandas'),('scipy','scipy'),('requests','requests'),('PIL','pillow')]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',requirement],check=True)
_VX_HERO_SOURCE = '"""Position-aware hero cards; presentation never invents unavailable statistics."""\nfrom __future__ import annotations\nimport html\nimport math\nfrom pathlib import Path\nfrom typing import Any, Mapping\n\nFIELDS = {\n \'GKP\': [(\'points\',\'Points\'),(\'appearances\',\'Apps (Starts)\'),(\'minutes\',\'Minutes\'),(\'cleanSheets\',\'CS\'),(\'saves\',\'Saves\'),(\'savePct\',\'Save %\'),(\'savePoints\',\'Save Pts\'),(\'goalsConceded\',\'GC\'),(\'penaltiesSaved\',\'Pen Saves\'),(\'xgc\',\'xGC\'),(\'goalsPrevented\',\'Goals Prev\'),(\'bonus\',\'Bonus\')],\n \'DEF\': [(\'points\',\'Points\'),(\'appearances\',\'Apps (Starts)\'),(\'minutes\',\'Minutes\'),(\'goals\',\'Goals\'),(\'assists\',\'Assists\'),(\'cleanSheets\',\'CS\'),(\'goalsConceded\',\'GC\'),(\'xgc\',\'xGC\'),(\'defcon\',\'DefCons\'),(\'defcon90\',\'DC/90\'),(\'xgi\',\'xGI\'),(\'bonus\',\'Bonus\')],\n \'MID\': [(\'points\',\'Points\'),(\'appearances\',\'Apps (Starts)\'),(\'minutes\',\'Minutes\'),(\'goals\',\'Goals\'),(\'assists\',\'Assists\'),(\'xg\',\'xG\'),(\'xa\',\'xA\'),(\'xgi\',\'xGI\'),(\'shotsOnTarget\',\'SoT\'),(\'chancesCreated\',\'Chances\'),(\'defcon\',\'DefCons\'),(\'bonus\',\'Bonus\')],\n \'FWD\': [(\'points\',\'Points\'),(\'appearances\',\'Apps (Starts)\'),(\'minutes\',\'Minutes\'),(\'goals\',\'Goals\'),(\'assists\',\'Assists\'),(\'xg\',\'xG\'),(\'xa\',\'xA\'),(\'xgi\',\'xGI\'),(\'totalShots\',\'Shots\'),(\'shotsOnTarget\',\'SoT\'),(\'bigChances\',\'Big Chances\'),(\'bonus\',\'Bonus\')],\n}\nFALLBACKS = {\n \'GKP\':[(\'saves90\',\'Saves/90\'),(\'xgc90\',\'xGC/90\'),(\'bps\',\'BPS\'),(\'points90\',\'Pts/90\'),(\'form\',\'Form\'),(\'influence\',\'Influence\'),(\'creativity\',\'Creativity\')],\n \'DEF\':[(\'xgc90\',\'xGC/90\'),(\'xg90\',\'xG/90\'),(\'xa90\',\'xA/90\'),(\'bps\',\'BPS\'),(\'points90\',\'Pts/90\'),(\'form\',\'Form\'),(\'influence\',\'Influence\')],\n \'MID\':[(\'xg90\',\'xG/90\'),(\'xa90\',\'xA/90\'),(\'xgi90\',\'xGI/90\'),(\'defcon90\',\'DC/90\'),(\'creativity\',\'Creativity\'),(\'bps\',\'BPS\'),(\'points90\',\'Pts/90\'),(\'form\',\'Form\'),(\'influence\',\'Influence\'),(\'threat\',\'Threat\')],\n \'FWD\':[(\'xg90\',\'xG/90\'),(\'xa90\',\'xA/90\'),(\'xgi90\',\'xGI/90\'),(\'threat\',\'Threat\'),(\'bps\',\'BPS\'),(\'points90\',\'Pts/90\'),(\'form\',\'Form\'),(\'creativity\',\'Creativity\'),(\'influence\',\'Influence\')],\n}\nLABELS = {\'cleanSheets\':\'Clean Sheets\',\'goalsConceded\':\'Goals Conceded\',\'defcon90\':\'Defensive Contributions per 90\',\'savePoints\':\'Save Points\',\'penaltiesSaved\':\'Penalty Saves\',\'goalsPrevented\':\'Goals Prevented\',\'shotsOnTarget\':\'Shots on Target\',\'chancesCreated\':\'Chances Created\',\'totalShots\':\'Total Shots\',\'bigChances\':\'Big Chances\'}\nCOLORS = [\'#008d46\',\'#9509e9\',\'#ff7900\',\'#ed0049\',\'#08a42e\',\'#0074ff\',\'#d3ac00\',\'#8805ed\',\'#009b83\',\'#e00054\',\'#007bff\',\'#bc9800\']\nINKS = [\'#00613a\',\'#200054\',\'#071656\',\'#9b0033\',\'#006038\',\'#071656\',\'#805b00\',\'#24005b\',\'#005950\',\'#950033\',\'#071656\',\'#745600\']\nICONS = [\'<circle cx="12" cy="12" r="10"/><path d="m12 6 5 4-2 6H9l-2-6zM12 2v4M2 12l5-2M22 12l-5-2M6 21l3-5m9 5-3-5"/>\', \'<path d="m8 3-6 4 3 5 3-2v12h8V10l3 2 3-5-6-4c0 5-8 5-8 0z"/>\', \'<circle cx="12" cy="14" r="8"/><path d="M9 2h6m-3 0v4m0 4v5l3 2M5 5 3 7m16-2 2 2"/>\', \'<path d="M3 21V4h18v17H3m0-12h18M3 15h18M9 4v17m6-17v17"/>\', \'<path d="M4 3v12l-2 4h20v-4l-8-3-3-7v7M2 22h20m-8-9 1-4m2 5 1-3"/>\', \'<path d="M8 20c-5-5-6-9-4-10l3 3V3c0-2 3-2 3 0v7-8c0-2 3-2 3 0v8-7c0-2 3-2 3 0v8-5c0-2 3-2 3 0v10l-4 6H8z"/>\', \'<path d="M2 17V3h18v10M2 8h18m-18 5h14M8 3v14m6-14v9m-1 10 5-9 5 9zM18 17v2m0 2v.1"/>\', \'<path d="M3 21V14h4v7m3 0V9h4v12m3 0V3h4v18M2 10l7-7 5 2 7-4"/>\', \'<path d="m12 2 9 4v7c0 6-9 10-9 10S3 19 3 13V6z"/>\', \'<path d="M3 22V15h4v7m3 0V9h4v13m3 0V3h4v19"/>\', \'<circle cx="12" cy="12" r="9"/><circle cx="12" cy="12" r="2"/><path d="M12 0v6m0 12v6M0 12h6m12 0h6"/>\', \'<path d="m12 1 3.5 7.2 8 .8-6 5.4 1.8 8L12 18l-7.3 4.4 1.8-8L.5 9l8-.8z"/>\']\n\ndef number(value: Any) -> float | None:\n try:\n  result=float(value)\n  return result if math.isfinite(result) else None\n except (TypeError,ValueError):return None\n\ndef extend_stats(view: dict[str, Any], player: Mapping[str, Any], history: list[dict[str, Any]] | None,\n                 advanced: Mapping[str, Any] | None = None) -> dict[str, Any]:\n stats=view[\'stats\']\n for key,raw in [(\'saves\',\'saves\'),(\'savePct\',\'save_percentage\'),(\'goalsPrevented\',\'goals_prevented\'),(\'shotsOnTarget\',\'shots_on_target\'),(\'chancesCreated\',\'chances_created\'),(\'totalShots\',\'total_shots\'),(\'bigChances\',\'big_chances\')]:\n  stats[key]=number(player.get(raw))\n for key,raw in [(\'saves90\',\'saves_per_90\'),(\'xgc90\',\'expected_goals_conceded_per_90\'),(\'xg90\',\'expected_goals_per_90\'),(\'xa90\',\'expected_assists_per_90\'),(\'xgi90\',\'expected_goal_involvements_per_90\'),(\'bps\',\'bps\'),(\'form\',\'form\'),(\'influence\',\'influence\'),(\'creativity\',\'creativity\'),(\'threat\',\'threat\')]:\n  stats[key]=number(player.get(raw))\n actual_xgi=number(player.get(\'expected_goal_involvements\'))\n if actual_xgi is not None:stats[\'xgi\']=actual_xgi\n # Save points accrue per match, so dividing the season total would be incorrect.\n if history is not None and all(number(match.get(\'saves\')) is not None for match in history):\n  stats[\'savePoints\']=sum(math.floor(float(match[\'saves\'])/3) for match in history)\n else:stats[\'savePoints\']=None\n view[\'metricSources\']={key:\'Official FPL\' for key,value in stats.items() if value is not None}\n if stats.get(\'savePoints\') is not None:view[\'metricSources\'][\'savePoints\']=\'Derived from official per-match saves\'\n if advanced:\n  for key in (\'savePct\',\'goalsPrevented\',\'shotsOnTarget\',\'chancesCreated\',\'totalShots\',\'bigChances\'):\n   record=advanced.get(key)\n   if isinstance(record,Mapping) and record.get(\'source_url\') and record.get(\'checked_at\'):\n    value=number(record.get(\'value\'))\n    if value is not None:stats[key]=value;view[\'metricSources\'][key]=str(record[\'source_url\'])\n position=\'GKP\' if view[\'position\']==\'GK\' else view[\'position\']\n if position not in FIELDS:raise ValueError(\'Unknown FPL position\')\n selected=[];used=set();substitutions=[]\n for key,label in FIELDS[position]:\n  available=number(stats.get(key)) is not None and (key!=\'appearances\' or number(stats.get(\'starts\')) is not None)\n  if available and key not in used:selected.append((key,label));used.add(key)\n  else:selected.append(None)\n for index,field in enumerate(selected):\n  if field is not None:continue\n  replacement=next(((key,label) for key,label in FALLBACKS[position] if key not in used and number(stats.get(key)) is not None),None)\n  if replacement is None:\n   selected[index]=FIELDS[position][index]\n   substitutions.append({\'requested\':FIELDS[position][index][0],\'displayed\':FIELDS[position][index][0],\'status\':\'unavailable\'})\n   continue\n  selected[index]=replacement;used.add(replacement[0]);substitutions.append({\'requested\':FIELDS[position][index][0],\'displayed\':replacement[0]})\n view[\'fields\']=selected\n view[\'statSubstitutions\']=substitutions\n return view\n\ndef esc(value: Any) -> str:return html.escape(str(value),quote=True)\n\ndef asset_uri(path: str | Path | None) -> str | None:\n if not path:return None\n path=Path(path)\n return path.resolve().as_uri() if path.is_file() else None\n\ndef display(value: Any, decimals: int = 0) -> str:\n parsed=number(value)\n return \'—\' if parsed is None else f\'{parsed:,.{decimals}f}\'\n\ndef card_html(view: Mapping[str, Any], assets: Mapping[str, Any], season: str | None = None,\n              horizon: int | None = None) -> str:\n position=str(view[\'position\']);position=\'GKP\' if position==\'GK\' else position\n if position not in FIELDS:raise ValueError(\'Unknown FPL player position\')\n logo=asset_uri(assets.get(\'logo\'))\n if not logo:raise FileNotFoundError(\'Original logo.png unavailable; no replacement channel logo is used\')\n photo=asset_uri(assets.get(\'photo\'));club_badge=asset_uri(assets.get(\'club_badge\'))\n badges=assets.get(\'badges\',{})\n fixtures=[]\n for index in range(5):\n  fixture=view.get(\'fixtures\',[])[index] if index<len(view.get(\'fixtures\',[])) else None\n  if fixture is None:\n   fixtures.append(\'<div class="hero-fixture" style="--fixture-bg:#25394c;--fixture-fg:white"><span class="hero-fixture-name">—</span><span class="hero-badge-unavailable">TBC</span><span class="hero-fixture-info">Unavailable</span></div>\');continue\n  rating=number(fixture.get(\'fdr\'))\n  from vortex_wildcard import fdr_display\n  fdr_theme=fdr_display(rating)\n  color=(fdr_theme[\'gradient\'] if rating is not None else fdr_theme[\'background\'],fdr_theme[\'text\'])\n  badge=asset_uri(badges.get(fixture.get(\'opponentId\')) or badges.get(str(fixture.get(\'opponentId\'))))\n  image=f\'<img src="{esc(badge)}" alt="{esc(fixture.get("opponentName","Opponent"))}">\' if badge else \'<span class="hero-badge-unavailable">No logo</span>\'\n  venue=\'H\' if fixture.get(\'home\') is True else \'A\' if fixture.get(\'home\') is False else \'—\'\n  fixtures.append(f\'<div class="hero-fixture" style="--fixture-bg:{color[0]};--fixture-fg:{color[1]}"><span class="hero-fixture-name">{esc(fixture.get("opponent","—"))}</span>{image}<span class="hero-fixture-info">{venue} · FDR {display(rating,1).removesuffix(".0")}</span></div>\')\n stats=view[\'stats\'];tiles=[]\n for index,(key,label) in enumerate(view.get(\'fields\',FIELDS[position])):\n  decimals=2 if key in (\'xg\',\'xa\',\'xgi\',\'xgc\',\'defcon90\',\'goalsPrevented\',\'saves90\',\'xgc90\',\'xg90\',\'xa90\',\'xgi90\',\'points90\') else 1 if key in (\'savePct\',\'form\',\'influence\',\'creativity\',\'threat\') else 0\n  value=display(stats.get(key),decimals)\n  if key==\'appearances\':value=f\'{display(stats.get(key))} ({display(stats.get("starts"))})\'\n  if key==\'savePct\' and value!=\'—\':value+=\'%\'\n  source=view.get(\'metricSources\',{}).get(key,\'Unavailable from current verified data\')\n  label_title=LABELS.get(key,label)\n  tiles.append(f\'<div class="hero-stat" style="--stat-color:{COLORS[index]};--stat-ink:{INKS[index]}"><div class="hero-stat-icon"><svg viewBox="0 0 24 24" aria-hidden="true">{ICONS[index]}</svg></div><div class="hero-stat-copy"><div class="hero-stat-label" title="{esc(label_title)}">{esc(label)}</div><div class="hero-stat-value" title="{esc(source)}">{esc(value)}</div></div></div>\')\n title=f\'{season} • {horizon}-GW PLAN\' if season and horizon else f\'{season} • PLAYER PROFILE\' if season else \'PLAYER PROFILE\'\n photo_html=f\'<img class="hero-photo" src="{esc(photo)}" alt="{esc(view["name"])} official player photo">\' if photo else \'<div class="hero-photo-unavailable">Photo unavailable</div>\'\n club_html=f\'<img class="hero-club-badge" src="{esc(club_badge)}" alt="{esc(view.get("clubName","Club"))}">\' if club_badge else \'\'\n owned=display(view.get(\'ownership\'),1);owned=owned+\'% owned\' if owned!=\'—\' else \'Ownership —\'\n price=display(view.get(\'price\'),1);price=\'£\'+price+\'M\' if price!=\'—\' else \'Price —\'\n source_label=\'FPL • verified stats\' if view.get(\'statSubstitutions\') else \'Official FPL • season totals\'\n return f\'<article class="hero-card" data-player-id="{int(view["id"])}"><div class="hero-series">{esc(title)}</div><div class="hero-shell"><div class="hero-lines"></div>{photo_html}<div class="hero-identity"><h2 class="hero-name">{esc(view["name"])}</h2><div class="hero-rule"></div><div class="hero-clubline">{club_html}<div class="hero-club">{esc(view.get("clubName",view.get("club","—")))}</div></div><div class="hero-meta"><span class="hero-pos">{"GK" if position=="GKP" else position}</span><span class="hero-price">{esc(price)}</span></div><div class="hero-owned">{esc(owned)}</div></div><section class="hero-fixtures"><h3 class="hero-section-title">NEXT FIXTURES</h3><div class="hero-fixture-row">{"".join(fixtures)}</div></section><section class="hero-grid">{"".join(tiles)}</section><img class="hero-brand" src="{esc(logo)}" alt="FPL VORTEX original channel logo"><div class="hero-source">{source_label}</div></div></article>\'\n'
_VX_MODULE_SOURCE = '"""Wildcard-only broadcast renderer. Consumes the verified canonical contract.\n\nThe same functions render previews and MP4 frames. Geometry uses a 1920x1080\ndesign canvas; player photographs retain their aspect ratio. No facts are\nrecalculated in the presentation layer.\n"""\nfrom hero_card import FIELDS as HERO_FIELDS, extend_stats as extend_hero_stats\nfrom functools import lru_cache\nfrom pathlib import Path\nimport base64\nimport io\nimport json\nimport math\nimport concurrent.futures as futures\n\nimport numpy as np\nimport requests\nfrom PIL import Image, ImageDraw, ImageFont, ImageOps\n\nfrom vortex_wildcard import fdr_display, validate_contract, clean_json, write_json\n\nNAVY = "#091721"\nCARD = "#112530"\nBORDER = "#355260"\nWHITE = "#F1F6F8"\nMUTED = "#AFC5CF"\nGREEN = "#7CEA9B"\nGOLD = "#E6BD58"\nYELLOW = "#F6D759"\n\n\nPITCH_DESIGN = {\'x\':42, \'y\':96, \'width\':1314, \'height\':916, \'field_height\':735,\n                \'sidebar_x\':1372, \'sidebar_width\':500, \'card_width\':190, \'card_height\':172}\nPITCH_COLOURS = {\'gold\':\'#FFD85A\', \'white\':\'#F5F7FA\', \'positive\':\'#79F28E\',\n                 \'ink\':\'#07111F\', \'easy\':\'#21E885\', \'neutral\':\'#F1F4F7\', \'hard\':\'#F51E42\'}\n\ndef pitch_fdr_palette(value):\n    style = fdr_display(value)\n    return style["background"], style["text"]\n\n\n@lru_cache(maxsize=128)\ndef font(size,bold=False):\n    names=[\'DejaVuSans-Bold.ttf\',\'LiberationSans-Bold.ttf\',\'Arial Bold.ttf\'] if bold else [\'DejaVuSans.ttf\',\'LiberationSans-Regular.ttf\',\'Arial.ttf\']\n    for root in (\'/usr/share/fonts\',\'/usr/local/share/fonts\',\'/System/Library/Fonts/Supplemental\'):\n        directory=Path(root)\n        if directory.is_dir():\n            for name in names:\n                found=sorted(directory.rglob(name))\n                if found:return ImageFont.truetype(str(found[0]),max(1,int(size)))\n    return ImageFont.load_default(size=max(1,int(size)))\n\n\n@lru_cache(maxsize=128)\ndef fixture_font(size, bold=True):\n    """Condensed broadcast typography, with portable font fallbacks."""\n    names = [\'DejaVuSansCondensed-Bold.ttf\', \'LiberationSansNarrow-Bold.ttf\', \'Arial Narrow Bold.ttf\'] if bold else [\'DejaVuSansCondensed.ttf\', \'LiberationSansNarrow-Regular.ttf\', \'Arial Narrow.ttf\']\n    for root in (\'/usr/share/fonts\', \'/usr/local/share/fonts\', \'/System/Library/Fonts/Supplemental\'):\n        directory = Path(root)\n        if directory.is_dir():\n            for name in names:\n                found = sorted(directory.rglob(name))\n                if found:\n                    return ImageFont.truetype(str(found[0]), max(1, int(size)))\n    return font(size, bold)\n\n\ndef fixture_palette(value):\n    style = fdr_display(value)\n    return style["background"], style["text"]\n\n\ndef safe_player_media(pool, ids, work, bootstrap):\n    """Reuse the inspected Day 3 image resolver and source audit."""\n    from vortex_player_images import resolve_media\n    return resolve_media(ids,work,bootstrap)\n\n\nclass ScaledGeometryDraw:\n    def __init__(self,image,sx,sy):self.draw=ImageDraw.Draw(image);self.sx,self.sy=sx,sy\n    def geometry(self,method,box,**kwargs):\n        coords=tuple(round(x*(self.sx if i%2==0 else self.sy)) for i,x in enumerate(box))\n        if \'width\' in kwargs:kwargs[\'width\']=max(1,round(kwargs[\'width\']*self.sy))\n        return getattr(self.draw,method)(coords,**kwargs)\n    def rectangle(self,box,**kwargs):return self.geometry(\'rectangle\',box,**kwargs)\n    def ellipse(self,box,**kwargs):return self.geometry(\'ellipse\',box,**kwargs)\n    def line(self,box,**kwargs):return self.geometry(\'line\',box,**kwargs)\n\nclass WildcardVisual:\n    def __init__(self, scene, contract, width, height, assets, media):\n        self.manifest,self.contract=scene,contract\n        self.width,self.height=width,height\n        self.sx,self.sy=width/1920,height/1080\n        self.assets,self.media=assets,media\n        self.byid={p[\'id\']:p for p in contract[\'squad\']}\n        self.entities={int(k):p for k,p in contract[\'media_entities\'].items()}\n        self.layout=[]\n        self.complete=self.draw(100,100)\n\n    def xy(self,*values):\n        return tuple(round(value*(self.sx if i%2==0 else self.sy)) for i,value in enumerate(values))\n\n    def rect(self,im,box,fill=CARD,outline=BORDER,radius=18):\n        ImageDraw.Draw(im).rounded_rectangle(self.xy(*box),radius=round(radius*self.sy),fill=fill,outline=outline,width=max(1,round(2*self.sy)))\n\n    def text(self,im,xy,text,size=28,fill=WHITE,bold=False,anchor=\'la\',max_width=None,minimum=22,condensed=False,stroke_width=0,stroke_fill=\'#04090D\'):\n        # Font sizes already include card and frame scaling; a fixed floor changes the layout.\n        draw=ImageDraw.Draw(im);pixels=max(1,round(size*self.sy));choose=fixture_font if condensed else font;f=choose(pixels,bold)\n        stroke=max(0,round(stroke_width*self.sy))\n        while max_width and draw.textbbox((0,0),str(text),font=f,stroke_width=stroke)[2]>max_width*self.sx and pixels>round(minimum*self.sy):\n            pixels-=1;f=choose(pixels,bold)\n        native=draw.textbbox(self.xy(*xy),str(text),font=f,anchor=anchor,stroke_width=stroke)\n        box=[native[i]/(self.sx if i%2==0 else self.sy) for i in range(4)]\n        if max_width and box[2]-box[0]>max_width+2:raise ValueError(\'Text does not fit: \'+str(text))\n        if box[0]<0 or box[1]<0 or box[2]>1921 or box[3]>1081:raise ValueError(\'Text outside slide: \'+str(text))\n        self.layout.append({\'kind\':\'text\',\'text\':str(text),\'box\':box,\'font_size\':f.size/self.sy,\'native_font_pixels\':f.size})\n        draw.text(self.xy(*xy),str(text),font=f,fill=fill,anchor=anchor,stroke_width=stroke,stroke_fill=stroke_fill);return box\n\n    def paragraph(self,im,box,text,size=27,fill=WHITE,bold=False):\n        draw=ImageDraw.Draw(im);x,y,x2,y2=box\n        while True:\n            f=font(size,bold);words=str(text).split();lines=[];line=\'\'\n            for word in words:\n                test=(line+\' \'+word).strip()\n                if draw.textbbox((0,0),test,font=f)[2]<=x2-x:line=test\n                else:\n                    if line:lines.append(line)\n                    line=word\n            if line:lines.append(line)\n            step=round(size*1.34)\n            if len(lines)*step<=y2-y:break\n            if size<=22:raise ValueError(f"Paragraph does not fit: {text[:90]}")\n            size-=1\n        for i,line in enumerate(lines):self.text(im,(x,y+i*step),line,size,fill,bold)\n\n    def photo(self,im,p,box,show_badge=True):\n        x,y,x2,y2=map(int,box);m=self.media.get(int(p[\'id\']),{});path=m.get(\'photo\')\n        if path and Path(path).exists():\n            pic=Image.open(path).convert(\'RGBA\');left,top,right,bottom=self.xy(x,y,x2,y2)\n            fitted=ImageOps.contain(pic,(right-left,bottom-top),Image.Resampling.LANCZOS)\n            im.paste(fitted,(left+(right-left-fitted.width)//2,bottom-fitted.height),fitted)\n            if show_badge:self.badge(im,p[\'club\'],x2-min(48,x2-x),y,min(48,x2-x))\n            self.layout.append({\'kind\':\'photo\',\'player_id\':p[\'id\'],\'box\':[x,y,x2,y2],\'source_aspect\':pic.width/pic.height,\'rendered_aspect\':fitted.width/fitted.height,\'photo_kind\':m.get(\'photo_kind\')})\n        else:\n            self.rect(im,(x+12,y+12,x2-12,y2-12),\'#1A3846\',BORDER,22)\n            if y2-y>130:self.text(im,((x+x2)//2,(y+y2)//2),\'PHOTO UNAVAILABLE\',22,MUTED,False,\'mm\',max_width=x2-x-24)\n\n    def badge(self,im,club,x,y,size=64):\n        record=globals().get(\'CLUB_MEDIA\',{}).get(str(club),{});path=record.get(\'path\')\n        if not path or not Path(path).is_file():return False\n        pic=ImageOps.contain(Image.open(path).convert(\'RGBA\'),self.xy(size,size),Image.Resampling.LANCZOS)\n        im.paste(pic,self.xy(x,y),pic)\n        self.layout.append({\'kind\':\'club_logo\',\'club\':club,\'box\':[x,y,x+size,y+size]})\n        return True\n\n    def status(self,im,p,x,y,w=220):\n        if not p[\'availability\'][\'flagged\']:return\n        self.rect(im,(x,y,x+w,y+36),YELLOW,YELLOW,6)\n        self.text(im,(x+w/2,y+18),p[\'injury_status\'],24,\'#0E151B\',True,\'mm\',max_width=w-14)\n\n    def base(self):\n        return Image.new(\'RGB\',(self.width,self.height),\'black\')\n\n    def fdr_cell(self,im,box,fx,compact=False):\n        x,y,x2,y2=box\n        if not fx:\n            self.rect(im,box,\'#233445\',\'#35546A\',9)\n            self.text(im,((x+x2)/2,(y+y2)/2),\'BLANK\',25,\'#FFFFFF\',True,\'mm\',max_width=x2-x-10,minimum=18,condensed=True)\n            return\n        width=(x2-x)/len(fx);height=y2-y\n        for i,f in enumerate(fx):\n            left=x+i*width\n            bg,fg=fixture_palette(f.get(\'fdr\'))\n            self.rect(im,(left,y,left+width-2,y2),bg,bg,9)\n            center=left+(width-2)/2\n            label=f.get(\'fdr_style\',{}).get(\'label\',\'—\')\n            home=f.get(\'home\')\n            detail=((\'H/\' if home is True else \'A/\' if home is False else \'\')+label)\n            opponent_size=25 if len(fx)==1 and not compact else 24\n            detail_size=24 if len(fx)==1 and not compact else 23\n            opponent_min=16 if len(fx)==1 else 8\n            detail_min=15 if len(fx)==1 else 8\n            self.text(im,(center,y+height*.27),f.get(\'opponent\',\'—\'),opponent_size,fg,True,\'mm\',\n                      max_width=width-8,minimum=opponent_min,condensed=True)\n            self.text(im,(center,y+height*.74),detail,detail_size,fg,True,\'mm\',\n                      max_width=width-8,minimum=detail_min,condensed=True)\n\n    def fixture_table(self,im):\n        weeks=self.contract[\'horizon_gameweeks\']\n        def average_fdr(row):\n            values=[]\n            for week in row.get(\'weeks\',[]):\n                if week.get(\'gameweek\') not in weeks:continue\n                for fixture in week.get(\'fixtures\',[]):\n                    value=fixture.get(\'fdr\')\n                    if value is None:continue\n                    try:value=float(value)\n                    except (TypeError,ValueError):continue\n                    if math.isfinite(value) and 1 <= value <= 5:values.append(value)\n            # DGWs count each match; blanks and unavailable ratings are excluded.\n            return sum(values)/len(values) if values else None\n        ranked=[(row,average_fdr(row)) for row in self.contract[\'fixture_table\']]\n        ranked.sort(key=lambda pair:(pair[1] if pair[1] is not None else math.inf,str(pair[0][\'club\'])))\n        for side in (0,1):\n            x=48+side*926;w=904;name_w=180;avg_w=88;cell_w=(w-name_w-avg_w)/len(weeks)\n            avg_x=x+w-avg_w/2\n            self.rect(im,(x,90,x+w,936),\'#04111D\',\'#287BA0\',16)\n            self.text(im,(x+22,128),\'CLUB\',34,\'#68F59E\',True,\'lm\',max_width=name_w-30,condensed=True)\n            for j,gw in enumerate(weeks):\n                self.text(im,(x+name_w+(j+.5)*cell_w,128),f\'GW {gw}\',30,\'#FFD75A\',True,\'mm\',max_width=cell_w-8,minimum=22,condensed=True)\n            self.text(im,(avg_x,116),\'AVG\',23,\'#FFD75A\',True,\'mm\',max_width=avg_w-8,minimum=18,condensed=True)\n            self.text(im,(avg_x,140),\'FDR\',23,\'#FFD75A\',True,\'mm\',max_width=avg_w-8,minimum=18,condensed=True)\n            for i,(row,average) in enumerate(ranked[side*10:(side+1)*10]):\n                top=158+i*75\n                self.badge(im,row[\'club\'],x+12,top+1,70)\n                self.text(im,(x+91,top+36),row[\'club\'],37,\'#FFFFFF\',True,\'lm\',max_width=82,minimum=29,condensed=True)\n                week_lookup={week[\'gameweek\']:week for week in row[\'weeks\']}\n                for j,gw in enumerate(weeks):\n                    fixtures=week_lookup.get(gw,{}).get(\'fixtures\',[])\n                    self.fdr_cell(im,(x+name_w+j*cell_w+3,top,x+name_w+(j+1)*cell_w-3,top+71),fixtures,True)\n                bg,fg=fixture_palette(average)\n                self.rect(im,(x+w-avg_w+3,top,x+w-6,top+71),bg,bg,9)\n                label=\'—\' if average is None else f\'{average:.1f}\'\n                self.text(im,(avg_x-1,top+36),label,33,fg,True,\'mm\',max_width=avg_w-14,minimum=24,condensed=True)\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n    def draw(self,t,duration):\n        self.layout=[]\n        if self.manifest[\'visual_type\']!=\'wc_analysis\':raise ValueError(\'Only the shared fixture table uses this renderer\')\n        image=self.base();self.fixture_table(image);return image\n\n    def frame(self,t,duration,previous=None):\n        return np.asarray(self.complete.convert(\'RGB\'))\n\n\ndef verify_visuals(contract,scenes,assets,media,output,width=1920,height=1080,preview_limit=None):\n    validate_contract(contract);output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    report=[]\n    selected=scenes if preview_limit is None else [scenes[i] for i in sorted({0,len(scenes)//2,len(scenes)-1})][:preview_limit]\n    for scene in selected:\n        if scene[\'visual_type\'] in (\'wc_blueprint\',\'wc_analysis\',\'wc_squad_plan\',\'wc_captaincy_final\'):\n            factory=globals().get(\'SHARED_VISUAL_FACTORY\') or globals().get(\'BLUEPRINT_VISUAL_FACTORY\')\n            if factory is None:raise RuntimeError(\'Blueprint renderer has not been installed; run the broadcast cell\')\n            v=factory(scene,contract,width,height)\n        else:\n            raise ValueError(\'Unknown slide outside the four-slide sequence\')\n        for t in ((0,13.5,26) if preview_limit is None else (26,)):v.draw(t,27)\n        v.complete.save(output/(scene[\'scene_id\']+\'.png\'))\n        for obj in v.layout:\n            if obj[\'kind\']==\'photo\':\n                assert abs(obj[\'source_aspect\']-obj[\'rendered_aspect\'])<.025,\'Player image distorted\'\n            if obj[\'kind\']==\'pitch\':\n                assert abs(obj[\'aspect\']-obj[\'reference_aspect\'])<.003,\'Pitch aspect wrong\'\n        texts=[o for o in v.layout if o[\'kind\']==\'text\']\n        for i,a in enumerate(texts):\n            for b in texts[i+1:]:\n                ax,ay,ax2,ay2=a[\'box\'];bx,by,bx2,by2=b[\'box\']\n                assert not (min(ax2,bx2)-max(ax,bx)>2 and min(ay2,by2)-max(ay,by)>2), \'Overlapping text: \'+a[\'text\']+\' / \'+b[\'text\']\n        report.append({\'scene_id\':scene[\'scene_id\'],\'visual_type\':scene[\'visual_type\'],\'verified\':True,\n                       \'layout\':v.layout,\'output\':str(output/(scene[\'scene_id\']+\'.png\'))})\n    write_json(output/\'visual_validation.json\',report)\n    return report\n\n\ndef preview_html(contract,scenes,assets,media,path,width=1920,height=1080,preview_limit=None):\n    data=[]\n    selected=scenes if preview_limit is None else [scenes[i] for i in sorted({0,len(scenes)//2,len(scenes)-1})][:preview_limit]\n    for scene in selected:\n        if scene[\'visual_type\'] in (\'wc_blueprint\',\'wc_analysis\',\'wc_squad_plan\',\'wc_captaincy_final\'):\n            factory=globals().get(\'SHARED_VISUAL_FACTORY\') or globals().get(\'BLUEPRINT_VISUAL_FACTORY\')\n            if factory is None:raise RuntimeError(\'Blueprint renderer has not been installed; run the broadcast cell\')\n            v=factory(scene,contract,width,height)\n        else:\n            raise ValueError(\'Unknown slide outside the four-slide sequence\')\n        b=io.BytesIO();v.complete.save(b,format=\'JPEG\',quality=90)\n        data.append({\'title\':scene[\'title\'],\n                     \'image\':\'data:image/jpeg;base64,\'+base64.b64encode(b.getvalue()).decode()})\n    title=f"FPL Vortex • Wild Card • Gameweek {contract[\'gameweek\']}"\n    html="""<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>__TITLE__</title><style>*{box-sizing:border-box}body{margin:0;background:#081720;color:#eff6f7;font:16px system-ui}main{max-width:1920px;margin:auto;padding:16px}header{display:flex;gap:12px;align-items:center;justify-content:space-between;flex-wrap:wrap}select,button{font:inherit;padding:12px 18px;border:1px solid #46646f;border-radius:12px;background:#17313d;color:white}button{cursor:pointer}img{width:100%;height:auto;aspect-ratio:16/9;object-fit:contain;border-radius:12px;margin-top:16px}.note{color:#b5cad4;line-height:1.5;padding:12px 0}a{color:#89efac}@media(max-width:700px){main{padding:6px}select{max-width:100%}}</style>\n<main><header><strong>__TITLE__</strong><div><button id="prev">←</button> <select id="scenes" aria-label="Choose scene"></select> <button id="next">→</button></div></header>\n<p class="note">Verified snapshot: __DATE__. __FDR__. xPts and expected minutes are estimates. Missing international coverage and account state are disclosed in the data report. Keyboard arrows change slides.</p>\n<img id="frame" alt="Verified Wildcard slide"><p class="note" id="label"></p></main>\n<script>const data=__DATA__;let index=0;const menu=document.getElementById(\'scenes\');data.forEach((s,i)=>{let o=document.createElement(\'option\');o.value=i;o.textContent=(i+1)+\'. \'+s.title;menu.appendChild(o)});function show(i){index=(i+data.length)%data.length;menu.value=index;document.getElementById(\'frame\').src=data[index].image;document.getElementById(\'label\').textContent=data[index].title}menu.onchange=()=>show(+menu.value);document.getElementById(\'prev\').onclick=()=>show(index-1);document.getElementById(\'next\').onclick=()=>show(index+1);document.onkeydown=e=>{if(e.key===\'ArrowRight\')show(index+1);if(e.key===\'ArrowLeft\')show(index-1)};show(0)</script></html>"""\n    html=html.replace(\'__TITLE__\',title).replace(\'__DATE__\',contract[\'generated_at\']).replace(\'__FDR__\',contract[\'fdr_source\']).replace(\'__DATA__\',json.dumps(data,ensure_ascii=False).replace(\'</\',\'<\\\\/\'))\n    Path(path).parent.mkdir(parents=True,exist_ok=True);Path(path).write_text(html,encoding=\'utf-8\')\n    return Path(path)\n\n\ndef reference_pitch_positions(contract):\n    """Position every legal formation from IDs, never from a pictured example XI."""\n    byid={p[\'id\']:p for p in contract[\'squad\']};out={};spec=PITCH_DESIGN\n    for pos,cy in [(\'GK\',201),(\'DEF\',382),(\'MID\',563),(\'FWD\',744)]:\n        group=sorted((byid[i] for i in contract[\'starting_xi_ids\'] if byid[i][\'position\']==pos),key=lambda p:p[\'id\'])\n        for j,p in enumerate(group):out[p[\'id\']]=(spec[\'x\']+spec[\'width\']*(j+1)/(len(group)+1),cy)\n    # Goalkeeper first visually; preserve the optimiser\'s outfield substitute order.\n    bench=sorted(contract[\'bench_ids\'],key=lambda pid:byid[pid][\'position\']!=\'GK\')\n    for j,pid in enumerate(bench):out[pid]=(spec[\'x\']+spec[\'width\']*(j+.5)/len(bench),921)\n    return out\n# Single normalized view model used by interactive cards and movie player panels.\nCARD_FIELDS=HERO_FIELDS\nCARD_RESULT_COLUMNS=[(\'gw\',\'GW\'),(\'opponent\',\'Opp\'),(\'result\',\'Result\'),(\'points\',\'Pts\'),(\'minutes\',\'Min\'),(\'goals\',\'G\'),(\'assists\',\'A\'),(\'xg\',\'xG\'),(\'xa\',\'xA\'),(\'xgi\',\'xGI\'),(\'cleanSheets\',\'CS\'),(\'goalsConceded\',\'GC\'),(\'xgc\',\'xGC\'),(\'defcon\',\'DC\'),(\'yellow\',\'YC\'),(\'red\',\'RC\'),(\'saves\',\'S\'),(\'bonus\',\'BP\'),(\'bps\',\'BPS\')]\n\ndef card_number(value):\n    try:\n        value=float(value)\n        return value if math.isfinite(value) else None\n    except (TypeError,ValueError):return None\n\ndef card_sum(*values):\n    return sum(values) if all(v is not None for v in values) else None\n\ndef card_ratio(numerator,denominator,multiplier=1):\n    return numerator/denominator*multiplier if numerator is not None and denominator is not None and denominator>0 else None\n\ndef card_display(key,value,starts=None):\n    if value is None:return \'—\'\n    if key==\'appearances\':return f"{int(value)} ({\'—\' if starts is None else int(starts)})"\n    if key==\'minsPerGame\':return f\'{value:.0f}′\'\n    if key in (\'xg\',\'xa\',\'xgi\',\'xgc\',\'points90\',\'gi90\',\'defcon90\'):return f\'{value:.2f}\'\n    return f\'{value:g}\' if isinstance(value,(int,float)) else str(value)\n\ndef normalize_player_card(player,history,fixtures,teams,fdr=None,snapshots=None,history_error=None):\n    pid=int(player[\'id\']);tid=int(player[\'team\']);position={1:\'GKP\',2:\'DEF\',3:\'MID\',4:\'FWD\'}.get(player.get(\'element_type\'),\'UNKNOWN\');team=teams.get(tid,{})\n    def n(key):return card_number(player.get(key))\n    stats={k:n(raw) for k,raw in [(\'points\',\'total_points\'),(\'starts\',\'starts\'),(\'minutes\',\'minutes\'),(\'goals\',\'goals_scored\'),(\'assists\',\'assists\'),(\'cleanSheets\',\'clean_sheets\'),(\'goalsConceded\',\'goals_conceded\'),(\'xgc\',\'expected_goals_conceded\'),(\'xg\',\'expected_goals\'),(\'xa\',\'expected_assists\'),(\'defcon\',\'defensive_contribution\'),(\'penaltiesSaved\',\'penalties_saved\'),(\'bonus\',\'bonus\')]}\n    stats[\'appearances\']=sum(card_number(h.get(\'minutes\')) is not None and card_number(h[\'minutes\'])>0 for h in history) if history is not None else (0 if stats[\'minutes\']==0 else None)\n    stats[\'minsPerGame\']=card_ratio(stats[\'minutes\'],stats[\'appearances\']);stats[\'gi\']=card_sum(stats[\'goals\'],stats[\'assists\']);stats[\'xgi\']=card_sum(stats[\'xg\'],stats[\'xa\'])\n    for key,num in [(\'points90\',\'points\'),(\'gi90\',\'gi\'),(\'defcon90\',\'defcon\')]:stats[key]=card_ratio(stats[num],stats[\'minutes\'],90)\n    raw_by_id={int(f[\'id\']):f for f in fixtures};upcoming=[]\n    model_records=getattr(fdr,\'records\',{}) if fdr else {}\n    for f in fixtures:\n        if f.get(\'finished\') or tid not in (f[\'team_h\'],f[\'team_a\']):continue\n        home=tid==f[\'team_h\'];opponent=int(f[\'team_a\'] if home else f[\'team_h\']);record=model_records.get((int(f[\'id\']),tid),{});channel=\'defence\' if position in (\'GKP\',\'DEF\') else \'attack\'\n        vortex=card_number(record.get(channel+\'_fdr\'))\n        if vortex is None:\n            raise ValueError(f\'DAY-3 canonical FDR missing for hero-card fixture {f["id"]}, team {tid}\')\n        upcoming.append({\'id\':int(f[\'id\']),\'gw\':f.get(\'event\'),\'opponentId\':opponent,\'opponent\':teams.get(opponent,{}).get(\'short_name\',\'—\'),\'opponentName\':teams.get(opponent,{}).get(\'name\',\'—\'),\'home\':home,\'kickoff\':f.get(\'kickoff_time\'),\'officialFdr\':None,\'vortexFdr\':vortex,\'fdr\':vortex,\'teamGoals\':card_number(record.get(\'team_goals\')),\'opponentGoals\':card_number(record.get(\'opponent_goals\')),\'cleanSheetProbability\':card_number(record.get(\'clean_sheet_probability\'))})\n    upcoming.sort(key=lambda f:(f[\'kickoff\'] is None,f[\'kickoff\'] or \'\',f[\'gw\'] if f[\'gw\'] is not None else 1000,f[\'id\']));upcoming=upcoming[:5]\n    results=[]\n    for h in history or []:\n        opponent=int(h[\'opponent_team\']);home=h.get(\'was_home\');hs=card_number(h.get(\'team_h_score\'));aws=card_number(h.get(\'team_a_score\'));result=None\n        if hs is not None and aws is not None and home is not None:\n            own,against=(hs,aws) if home else (aws,hs);result=(\'W\' if own>against else \'L\' if own<against else \'D\')+f\' {own:g}–{against:g}\'\n        row={\'id\':h.get(\'fixture\'),\'gw\':h.get(\'round\'),\'opponentId\':opponent,\'opponent\':teams.get(opponent,{}).get(\'short_name\',\'—\')+(\' (H)\' if home is True else \' (A)\' if home is False else \'\'),\'opponentClub\':teams.get(opponent,{}).get(\'short_name\'),\'result\':result,\'kickoff\':h.get(\'kickoff_time\')}\n        for key,raw in [(\'points\',\'total_points\'),(\'minutes\',\'minutes\'),(\'goals\',\'goals_scored\'),(\'assists\',\'assists\'),(\'xg\',\'expected_goals\'),(\'xa\',\'expected_assists\'),(\'cleanSheets\',\'clean_sheets\'),(\'goalsConceded\',\'goals_conceded\'),(\'xgc\',\'expected_goals_conceded\'),(\'defcon\',\'defensive_contribution\'),(\'yellow\',\'yellow_cards\'),(\'red\',\'red_cards\'),(\'saves\',\'saves\'),(\'bonus\',\'bonus\'),(\'bps\',\'bps\')]:row[key]=card_number(h.get(raw))\n        row[\'xgi\']=card_sum(row[\'xg\'],row[\'xa\']);results.append(row)\n    results.sort(key=lambda r:(r[\'kickoff\'] or \'\',r[\'id\'] or 0),reverse=True)\n    series=snapshots or [];ownership=n(\'selected_by_percent\');previous=series[-2][\'ownership\'] if len(series)>1 else None\n    ownership_view={\'current\':ownership,\'previous\':previous,\'change\':ownership-previous if ownership is not None and previous is not None else None,\'transfersIn\':n(\'transfers_in_event\'),\'transfersOut\':n(\'transfers_out_event\'),\'netTransfers\':card_sum(n(\'transfers_in_event\'),-n(\'transfers_out_event\') if n(\'transfers_out_event\') is not None else None),\'price\':n(\'now_cost\')/10 if n(\'now_cost\') is not None else None,\'priceMovement\':n(\'cost_change_event\')/10 if n(\'cost_change_event\') is not None else None,\'series\':series,\'managerCounts\':[{\'at\':h.get(\'kickoff_time\'),\'gw\':h.get(\'round\'),\'value\':card_number(h.get(\'selected\'))} for h in history or [] if card_number(h.get(\'selected\')) is not None]}\n    view={\'id\':pid,\'name\':player.get(\'web_name\') or \'—\',\'fullName\':((player.get(\'first_name\') or \'\')+\' \'+(player.get(\'second_name\') or \'\')).strip(),\'club\':team.get(\'short_name\',\'—\'),\'clubName\':team.get(\'name\',\'—\'),\'teamId\':tid,\'position\':position,\'price\':ownership_view[\'price\'],\'ownership\':ownership,\'photoCode\':player.get(\'code\'),\'stats\':stats,\'fields\':CARD_FIELDS.get(position,[]),\'results\':results,\'fixtures\':upcoming,\'ownershipData\':ownership_view,\'historyError\':history_error,\'source\':\'Official FPL player evidence and fixture IDs; DAY-3 GitHub model-only FDR\'}\n    return extend_hero_stats(view,player,history,globals().get(\'HERO_ADVANCED_STATS\',{}).get(pid))\n\ndef build_player_card_views(bootstrap,fdr,cache_root):\n    from vortex_wildcard import DataClient,API\n    import hashlib\n    client=DataClient(cache_root);fixtures=client.get(API+\'fixtures/\',required=True);teams={int(t[\'id\']):t for t in bootstrap[\'teams\']}\n    snapshot_path=Path(cache_root).parent/\'player_card_ownership_snapshots.json\';digest=hashlib.sha256(json.dumps(bootstrap,sort_keys=True).encode()).hexdigest()\n    try:journal=json.loads(snapshot_path.read_text())\n    except (OSError,ValueError):journal={\'hash\':None,\'players\':{}}\n    source_cache=Path(cache_root)/(hashlib.sha256((API+\'bootstrap-static/\').encode()).hexdigest()[:24]+\'.json\')\n    stamp=json.loads(source_cache.read_text()).get(\'fetched_at\') if source_cache.exists() else None\n    if journal[\'hash\']!=digest:\n        for p in bootstrap[\'elements\']:\n            row=journal[\'players\'].setdefault(str(p[\'id\']),[]);row.append({\'at\':stamp,\'ownership\':card_number(p.get(\'selected_by_percent\')),\'price\':card_number(p.get(\'now_cost\'))/10 if card_number(p.get(\'now_cost\')) is not None else None});del row[:-128]\n        journal[\'hash\']=digest;write_json(snapshot_path,journal)\n    def one(p):\n        data=client.get(API+f"element-summary/{p[\'id\']}/",timeout=6)\n        history=data.get(\'history\') if isinstance(data,dict) else None\n        return int(p[\'id\']),normalize_player_card(p,history,fixtures,teams,fdr,journal[\'players\'].get(str(p[\'id\']),[]),None if history is not None else \'Match history is unavailable from the current source. Refresh the data to retry.\')\n    out={}\n    with futures.ThreadPoolExecutor(max_workers=6) as executor:\n        for i,(pid,view) in enumerate(executor.map(one,bootstrap[\'elements\']),1):\n            out[pid]=view\n            if i%100==0:print(f\'Player cards: {i}/{len(bootstrap["elements"])} normalized\',flush=True)\n    return out\n\ndef card_fdr_palette(value):\n    style = fdr_display(value)\n    return style["background"], style["text"]\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'
_vx_module_dir=VX_MEDIA_WORK/'wildcard_engine'
_vx_module_dir.mkdir(parents=True,exist_ok=True)
(_vx_module_dir/'vortex_wildcard_visuals.py').write_text(_VX_MODULE_SOURCE,encoding='utf-8')
(_vx_module_dir/'hero_card.py').write_text(_VX_HERO_SOURCE,encoding='utf-8')
if str(_vx_module_dir) not in sys.path:sys.path.insert(0,str(_vx_module_dir))
import importlib
importlib.invalidate_caches()
Path(importlib.util.cache_from_source(str(_vx_module_dir / 'vortex_wildcard_visuals.py'))).unlink(missing_ok=True)
import vortex_wildcard_visuals as VXWV
importlib.reload(VXWV)
print("✅ CELL 4 COMPLETE")

✅ CELL 4 COMPLETE


In [7]:
#@title 🔴 CELL 5 - SHARED INTERFACES {display-mode: "form"}
# 🔴 CELL 5 - SHARED INTERFACES
import importlib as _fplv_importlib
from pathlib import Path as _FplvPath
_FPLV_SHARED_SOURCES = {'__init__.py': '"""Shared, opt-in FPL VORTEX presentation interfaces.\n\nThe existing Wildcard and Free Hit call sites are intentionally unchanged.\nThese modules delegate to the notebook\'s original engine/render code.\n"""\nfrom .contracts import PlayerCardRef, normalized_card, validate_player_card\nfrom .assets import ClubBadgeRef, OfficialPhotoRef, club_badge, official_photo, resolve_official_media\nfrom .fixtures import FixtureCell, fixture_strip, canonical_fdr_display\nfrom .captaincy import CaptainBadge, captain_badge, verify_captaincy\nfrom .scenes import NarrationBlock, scene_envelope, verify_narration_block\nfrom .theme import ThemeRef, wildcard_theme\nfrom .exports import export_json\nfrom .validation import validate_existing_squad, validate_starting_xi\n\n__all__ = [\n    "PlayerCardRef", "normalized_card", "validate_player_card",\n    "ClubBadgeRef", "OfficialPhotoRef", "club_badge", "official_photo", "resolve_official_media",\n    "FixtureCell", "fixture_strip", "canonical_fdr_display",\n    "CaptainBadge", "captain_badge", "verify_captaincy",\n    "NarrationBlock", "scene_envelope", "verify_narration_block",\n    "ThemeRef", "wildcard_theme", "export_json",\n    "validate_existing_squad", "validate_starting_xi",\n]\n', 'assets.py': '"""Read-only official FPL identity, photo and club badge adapters."""\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Mapping\n\n\n@dataclass(frozen=True)\nclass OfficialPhotoRef:\n    player_id: int\n    photo_code: int\n    urls: tuple[str, ...]\n\n\n@dataclass(frozen=True)\nclass ClubBadgeRef:\n    team_id: int\n    team_code: int\n    short_name: str\n    path: str | None\n\n\ndef _official_player(player_id: int, bootstrap: Mapping[str, Any]) -> Mapping[str, Any]:\n    try:\n        player_id = int(player_id)\n    except (ValueError, TypeError) as exc:\n        raise ValueError("Official FPL player id is required") from exc\n    matches = [player for player in bootstrap["elements"] if int(player["id"]) == player_id]\n    if len(matches) != 1:\n        raise ValueError(f"Player id {player_id} is not unique in official bootstrap")\n    return matches[0]\n\n\ndef official_photo(player_id: int, bootstrap: Mapping[str, Any]) -> OfficialPhotoRef:\n    from vortex_player_images import _player_code, official_photo_urls\n    player = _official_player(player_id, bootstrap)\n    photo_code = _player_code(player)\n    if photo_code is None:\n        raise ValueError(f"Official player id {player_id} lacks an FPL photo code")\n    urls = tuple(official_photo_urls(player))\n    if not urls or not all(str(photo_code) in url for url in urls):\n        raise ValueError("Official photo URL does not match the player photo code")\n    return OfficialPhotoRef(int(player_id), photo_code, urls)\n\n\ndef resolve_official_media(player_ids: list[int], work: str | Path, bootstrap: Mapping[str, Any]) -> dict[int, Any]:\n    from vortex_player_images import resolve_media\n    for player_id in player_ids:\n        official_photo(player_id, bootstrap)\n    return resolve_media(player_ids, work, bootstrap)\n\n\ndef club_badge(team_id: int, bootstrap: Mapping[str, Any],\n               club_media: Mapping[str, Mapping[str, Any]] | None = None) -> ClubBadgeRef:\n    try:\n        team_id = int(team_id)\n    except (ValueError, TypeError) as exc:\n        raise ValueError("Official FPL team ID is required") from exc\n    matches = [team for team in bootstrap["teams"] if int(team["id"]) == team_id]\n    if len(matches) != 1:\n        raise ValueError(f"Team id {team_id} does not exist uniquely in bootstrap")\n    team = matches[0]\n    team_code = int(team.get("code") or 0)\n    if team_code <= 0:\n        raise ValueError("Official FPL team code missing")\n    short_name = str(team.get("short_name") or "").strip()\n    if not short_name:\n        raise ValueError("Official club short name missing")\n    media = (club_media or {}).get(short_name) or {}\n    declared_code = media.get("team_code")\n    if declared_code is not None and int(declared_code) != team_code:\n        raise ValueError("Club badge asset does not match official team code")\n    declared_team_id = media.get("team_id")\n    if declared_team_id is not None and int(declared_team_id) != team_id:\n        raise ValueError("Club badge asset does not match official team id")\n    raw_path = media.get("path")\n    path = str(raw_path) if raw_path and Path(raw_path).is_file() else None\n    return ClubBadgeRef(team_id, team_code, short_name, path)\n', 'captaincy.py': '"""Badge view data only; this module never selects a captain."""\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import Iterable\n\n\n@dataclass(frozen=True)\nclass CaptainBadge:\n    role: str\n    display_marker: str\n    border_color: str\n\n\ndef captain_badge(role: str) -> CaptainBadge:\n    from vortex_wildcard_visuals import PITCH_COLOURS\n    if role not in ("captain", "vice_captain"):\n        raise ValueError("Badge role must be captain or vice_captain")\n    # This reflects the existing Wildcard renderer\'s on-pitch \'C\'/\'V\' gold markers.\n    return CaptainBadge(role, "C" if role == "captain" else "V", PITCH_COLOURS["gold"])\n\n\ndef verify_captaincy(starting_xi_ids: Iterable[int], captain_id: int, vice_id: int) -> None:\n    starters = [int(value) for value in starting_xi_ids]\n    if len(starters) != 11 or len(set(starters)) != 11:\n        raise ValueError("Starting XI must have eleven unique player ids")\n    if int(captain_id) == int(vice_id) or int(captain_id) not in starters or int(vice_id) not in starters:\n        raise ValueError("Captain and vice must be distinct starting-XI players")\n', 'contracts.py': '"""An adapter around the canonical WildcardVisual player-card data view.\n\nNo player metrics or projections are computed in this module.\n"""\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import Any, Mapping, Sequence\n\n_POSITIONS = {"GKP", "GK", "DEF", "MID", "FWD"}\n\n@dataclass(frozen=True)\nclass PlayerCardRef:\n    player_id: int\n    name: str\n    team_id: int\n    position: str\n    source: Mapping[str, Any]\n\n\ndef validate_player_card(view: Mapping[str, Any]) -> PlayerCardRef:\n    if not isinstance(view, Mapping):\n        raise TypeError("Card data must be a mapping")\n    try:\n        player_id = int(view["id"])\n        team_id = int(view["teamId"])\n        name = str(view["name"]).strip()\n        position = str(view["position"]).upper()\n    except (KeyError, ValueError, TypeError) as exc:\n        raise ValueError("Card needs verified id, teamId, name and position") from exc\n    if player_id <= 0 or team_id <= 0 or not name or position not in _POSITIONS:\n        raise ValueError("Invalid canonical player identity or position")\n    if not isinstance(view.get("stats"), Mapping):\n        raise ValueError("Canonical player stats must be a mapping")\n    if not isinstance(view.get("fixtures"), Sequence) or isinstance(view.get("fixtures"), (str, bytes)):\n        raise ValueError("Canonical fixture list is required")\n    return PlayerCardRef(player_id, name, team_id, position, view)\n\n\ndef normalized_card(player: Mapping[str, Any], history: Sequence[Mapping[str, Any]] | None,\n                    fixtures: Sequence[Mapping[str, Any]], teams: Mapping[int, Mapping[str, Any]],\n                    fdr: Any = None) -> PlayerCardRef:\n    from vortex_wildcard_visuals import normalize_player_card\n    view = normalize_player_card(player, history, fixtures, teams, fdr)\n    return validate_player_card(view)\n', 'exports.py': '"""Export API backed by the existing Wildcard atomic JSON writer."""\nfrom __future__ import annotations\nfrom pathlib import Path\nfrom typing import Any\n\n\ndef export_json(path: str | Path, value: Any) -> Path:\n    from vortex_wildcard import write_json\n    target = Path(path)\n    write_json(target, value)\n    return target\n', 'fixtures.py': '"""Fixture strip view records using the engine\'s unchanged FDR display policy."""\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import Any, Mapping, Sequence\n\n\n@dataclass(frozen=True)\nclass FixtureCell:\n    gameweek: int\n    opponent: str | None\n    is_home: bool | None\n    fdr: float | None\n    background: str\n    foreground: str\n    label: str\n    is_blank: bool\n\n\ndef canonical_fdr_display(value: Any) -> dict[str, Any]:\n    from vortex_wildcard import fdr_display\n    return fdr_display(value)\n\n\ndef fixture_strip(fixtures: Sequence[Mapping[str, Any]], gameweeks: Sequence[int]) -> tuple[FixtureCell, ...]:\n    if not isinstance(fixtures, Sequence) or isinstance(fixtures, (str, bytes)):\n        raise TypeError("Fixtures must be a sequence")\n    weeks = [int(gameweek) for gameweek in gameweeks]\n    if not weeks or len(weeks) > 6 or len(set(weeks)) != len(weeks) or min(weeks) <= 0:\n        raise ValueError("Select between one and six unique upcoming gameweeks")\n    out: list[FixtureCell] = []\n    for week in weeks:\n        found = [fixture for fixture in fixtures if fixture.get("gw", fixture.get("gameweek")) is not None and int(fixture.get("gw", fixture.get("gameweek"))) == week]\n        if not found:\n            style = canonical_fdr_display(None)\n            out.append(FixtureCell(week, None, None, None, style["background"], style["text"], "BLANK", True))\n            continue\n        for fixture in found:\n            raw = fixture.get("fdr")\n            if raw is None:\n                raw = fixture.get("officialFdr")\n            style = canonical_fdr_display(raw)\n            venue = fixture.get("home", fixture.get("was_home"))\n            if venue not in (True, False, None):\n                raise ValueError(f"Fixture venue must be H/A/unknown in Gameweek {week}")\n            opponent = str(fixture.get("opponent") or fixture.get("opponent_short_name") or "").strip()\n            out.append(FixtureCell(week, opponent or None, venue, style["value"],\n                                   style["background"], style["text"], style["label"], False))\n    return tuple(out)\n', 'scenes.py': '"""Chip-neutral scene and narration envelopes; no strategic inference."""\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import Any, Mapping\n\n\n@dataclass(frozen=True)\nclass NarrationBlock:\n    scene_id: str\n    text: str\n    source_hash: str\n\n\ndef verify_narration_block(block: NarrationBlock) -> None:\n    if not all((block.scene_id.strip(), block.text.strip(), block.source_hash.strip())):\n        raise ValueError("Narration text needs scene identity and source hash")\n\n\ndef scene_envelope(scene: Mapping[str, Any], chip: str) -> dict[str, Any]:\n    from vortex_wildcard import clean_json\n    if chip not in ("wildcard", "freehit"):\n        raise ValueError("Unknown chip scene type")\n    if not isinstance(scene, Mapping):\n        raise TypeError("Scene must be a mapping")\n    actual_chip = scene.get("chip")\n    if actual_chip is not None and actual_chip != chip:\n        raise ValueError("Scene/chip mismatch")\n    # Reuse the existing encoder; chip-specific validation remains at the original call sites.\n    return clean_json(dict(scene))\n', 'smoke.py': '"""Offline Phase 1 smoke suite; never requests network or invokes optimisation."""\nfrom __future__ import annotations\nimport json\nimport math\nimport tempfile\nfrom pathlib import Path\nfrom types import SimpleNamespace\nfrom PIL import Image\n\n\ndef run_smoke_suite() -> dict[str, str]:\n    from .contracts import normalized_card\n    from .assets import club_badge, official_photo\n    from .fixtures import fixture_strip, canonical_fdr_display\n    from .captaincy import captain_badge, verify_captaincy\n    from .scenes import NarrationBlock, scene_envelope, verify_narration_block\n    from .theme import wildcard_theme\n    from .exports import export_json\n    from .validation import validate_starting_xi\n    from vortex_player_images import official_photo_urls\n    from vortex_wildcard_visuals import normalize_player_card, pitch_fdr_palette, card_fdr_palette\n    from vortex_wildcard import fdr_display\n    from vortex_upstream_fdr import vx_phase4_attack_fdr, vx_phase4_defence_fdr\n\n    bootstrap = {\n        "elements": [{"id": 101, "code": 9912, "team": 1, "element_type": 3, "web_name": "Test Athlete", "minutes": 90, "total_points": 2, "now_cost": 50}],\n        "teams": [{"id": 1, "code": 12, "short_name": "AAA", "name": "Test Club"},\n                  {"id": 2, "code": 23, "short_name": "BBB", "name": "Opponent"}],\n    }\n    results: dict[str, str] = {}\n    photo = official_photo(101, bootstrap)\n    assert photo.photo_code == 9912 and photo.urls == tuple(official_photo_urls(bootstrap["elements"][0]))\n    try:\n        official_photo(555, bootstrap)\n    except ValueError:\n        pass\n    else:\n        raise AssertionError("Unknown player accepted")\n    results["official_photo_identity"] = "PASS (offline URL mapping)"\n\n    with tempfile.TemporaryDirectory() as directory:\n        badge_file = Path(directory) / "official.png"\n        Image.new("RGBA", (27, 44), (13, 55, 66, 255)).save(badge_file)\n        badge = club_badge(1, bootstrap, {"AAA": {"path": str(badge_file), "team_id": 1, "team_code": 12}})\n        assert badge.team_id == 1 and badge.team_code == 12 and badge.path == str(badge_file)\n        try:\n            club_badge(1, bootstrap, {"AAA": {"path": str(badge_file), "team_code": 99}})\n        except ValueError:\n            pass\n        else:\n            raise AssertionError("Wrong logo code accepted")\n        results["club_logo_identity"] = "PASS (offline verified ID/code)"\n        path = export_json(Path(directory) / "scene.json", {"values": [4, 5]})\n        assert json.loads(path.read_text()) == {"values": [4, 5]}\n        results["scene_export"] = "PASS (original atomic writer)"\n\n    fixtures = [\n        {"id": 50, "event": 7, "team_h": 1, "team_a": 2, "team_h_difficulty": 2, "team_a_difficulty": 4, "finished": False, "kickoff_time": "2026-10-10T15:00:00Z"},\n    ]\n    # Cards require canonical model records; official FPL difficulty is not a substitute.\n    team_goals, opponent_goals = 1.8, 0.9\n    clean_sheet_probability = math.exp(-opponent_goals)\n    model = SimpleNamespace(records={(50, 1): {\n        "attack_fdr": vx_phase4_attack_fdr(team_goals),\n        "defence_fdr": vx_phase4_defence_fdr(clean_sheet_probability),\n        "team_goals": team_goals,\n        "opponent_goals": opponent_goals,\n        "clean_sheet_probability": clean_sheet_probability,\n    }})\n    teams = {team["id"]: team for team in bootstrap["teams"]}\n    player = bootstrap["elements"][0]\n    view = normalized_card(player, [], fixtures, teams, model)\n    base = normalize_player_card(player, [], fixtures, teams, model)\n    assert dict(view.source) == base and view.position == "MID" and len(view.source["fixtures"]) == 1\n    fixture = view.source["fixtures"][0]\n    assert fixture["fdr"] == model.records[(50, 1)]["attack_fdr"]\n    assert fixture["officialFdr"] is None\n    for position in (1, 2):\n        defensive_view = normalized_card({**player, "element_type": position}, [], fixtures, teams, model)\n        assert defensive_view.source["fixtures"][0]["fdr"] == model.records[(50, 1)]["defence_fdr"]\n    for missing_model in (None, SimpleNamespace(records={})):\n        try:\n            normalized_card(player, [], fixtures, teams, missing_model)\n        except ValueError as exc:\n            assert "DAY-3 canonical FDR missing" in str(exc)\n        else:\n            raise AssertionError("Missing canonical FDR was accepted")\n    results["canonical_player_card"] = "PASS (exact existing normalizer)"\n\n    cells = fixture_strip([\n        {"gw": 7, "opponent": "BBB", "home": True, "fdr": 2.3},\n        {"gw": 7, "opponent": "CCC", "home": False, "fdr": 3.9},\n        {"gw": 9, "opponent": "EEE", "home": False, "fdr": None},\n    ], [7, 8, 9])\n    assert len(cells) == 4 and cells[0].opponent == "BBB" and cells[1].opponent == "CCC"\n    assert cells[2].is_blank and cells[2].label == "BLANK" and cells[3].label == "—"\n    assert canonical_fdr_display(2.3) == fdr_display(2.3)\n    assert pitch_fdr_palette(3.0) == pitch_fdr_palette(3.0)\n    assert card_fdr_palette(4.0) == card_fdr_palette(4.0)\n    try:\n        fixture_strip([{"gw": 7, "opponent": "BBB", "fdr": 9}], [7])\n    except ValueError:\n        pass\n    else:\n        raise AssertionError("Invalid FDR was accepted")\n    assert len(fixture_strip([{"gw": 7, "opponent": "BBB", "home": True, "fdr": 3}], [7])) == 1\n    results["fixture_fdr_blank_double"] = "PASS (one GW, doubles, blanks, invalid FDR)"\n\n    squad_ids = list(range(1, 16))\n    verify_captaincy(squad_ids[:11], 1, 2)\n    validate_starting_xi(squad_ids, squad_ids[:11], squad_ids[11:])\n    assert captain_badge("captain").display_marker == "C"\n    assert captain_badge("vice_captain").display_marker == "V"\n    try:\n        verify_captaincy(squad_ids[:11], 1, 1)\n    except ValueError:\n        pass\n    else:\n        raise AssertionError("Equal captain/vice ids were accepted")\n    results["captain_vice_legality"] = "PASS (same Wildcard badges)"\n\n    narration = NarrationBlock("scene_one", "Modelled estimates, not promises.", "hash001")\n    verify_narration_block(narration)\n    assert scene_envelope({"chip": "wildcard", "when": None}, "wildcard") == {"chip": "wildcard", "when": None}\n    assert wildcard_theme().gold == "#FFD85A"\n    results["shared_scene_narration_theme"] = "PASS (no scene selection)"\n    return results\n', 'theme.py': '"""Read-only pointers to the existing Wildcard palettes; no restyling."""\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import Any\n\n\n@dataclass(frozen=True)\nclass ThemeRef:\n    navy: str\n    white: str\n    gold: str\n\n\ndef wildcard_theme() -> ThemeRef:\n    from vortex_wildcard_visuals import NAVY, WHITE, PITCH_COLOURS\n    return ThemeRef(NAVY, WHITE, PITCH_COLOURS["gold"])\n\n\ndef pitch_fixture_colors(fdr: Any) -> tuple[str, str]:\n    from vortex_wildcard_visuals import pitch_fdr_palette\n    return pitch_fdr_palette(fdr)\n\n\ndef card_fixture_colors(fdr: Any) -> tuple[str, str]:\n    from vortex_wildcard_visuals import card_fdr_palette\n    return card_fdr_palette(fdr)\n', 'validation.py': '"""Shared legality adapters. Existing Wildcard validation remains authoritative."""\nfrom __future__ import annotations\nfrom typing import Any, Iterable\n\n\ndef validate_existing_squad(squad: Any, budget: float, teams_limit: int = 3) -> None:\n    from vortex_wildcard import validate_squad\n    validate_squad(squad, budget, teams_limit)\n\n\ndef validate_starting_xi(squad_ids: Iterable[int], starting_xi_ids: Iterable[int],\n                         bench_ids: Iterable[int]) -> None:\n    squad = [int(value) for value in squad_ids]\n    starters = [int(value) for value in starting_xi_ids]\n    bench = [int(value) for value in bench_ids]\n    if len(squad) != 15 or len(set(squad)) != 15:\n        raise ValueError("Squad must contain 15 unique ids")\n    if len(starters) != 11 or len(set(starters)) != 11 or len(bench) != 4 or len(set(bench)) != 4:\n        raise ValueError("Expected 11 starters and 4 bench players")\n    if set(starters) & set(bench) or set(starters) | set(bench) != set(squad):\n        raise ValueError("Starting XI and bench must partition the squad")\n'}
_fplv_shared_dir = _FplvPath(_vx_module_dir) / "fplv_shared"
_fplv_shared_dir.mkdir(parents=True, exist_ok=True)
for _fplv_name, _fplv_source in _FPLV_SHARED_SOURCES.items():
    _fplv_target = _fplv_shared_dir / _fplv_name
    if not _fplv_target.is_file() or _fplv_target.read_text(encoding="utf-8") != _fplv_source:
        _fplv_target.write_text(_fplv_source, encoding="utf-8")
    _FplvPath(_fplv_importlib.util.cache_from_source(str(_fplv_target))).unlink(missing_ok=True)
# Rerunning this cell must load the updated embedded interfaces and tests.
import sys as _fplv_sys
for _fplv_cached_name in list(_fplv_sys.modules):
    if _fplv_cached_name == "fplv_shared" or _fplv_cached_name.startswith("fplv_shared."):
        _fplv_sys.modules.pop(_fplv_cached_name, None)
_fplv_importlib.invalidate_caches()
import fplv_shared as _fplv_shared
print("✅ LOG: Cell 4A completed — optional shared interfaces available")
print("✅ CELL 5 COMPLETE")

✅ LOG: Cell 4A completed — optional shared interfaces available
✅ CELL 5 COMPLETE


In [8]:
#@title 🔴 CELL 6 - SHARED TESTS {display-mode: "form"}
# 🔴 CELL 6 - SHARED TESTS
# No live FPL data, asset downloads, video rendering or solver runs.
from fplv_shared.smoke import run_smoke_suite as _fplv_run_smoke_suite
_fplv_smoke_results = _fplv_run_smoke_suite()
for _fplv_test, _fplv_result in _fplv_smoke_results.items():
    print(f"{_fplv_test}: {_fplv_result}")
assert len(_fplv_smoke_results) == 7
print("✅ LOG: Cell 4B completed — 7 offline interface checks passed")
print("✅ CELL 6 COMPLETE")

official_photo_identity: PASS (offline URL mapping)
club_logo_identity: PASS (offline verified ID/code)
scene_export: PASS (original atomic writer)
canonical_player_card: PASS (exact existing normalizer)
fixture_fdr_blank_double: PASS (one GW, doubles, blanks, invalid FDR)
captain_vice_legality: PASS (same Wildcard badges)
shared_scene_narration_theme: PASS (no scene selection)
✅ LOG: Cell 4B completed — 7 offline interface checks passed
✅ CELL 6 COMPLETE


In [9]:
#@title 🔴 CELL 7 - APPROVED ASSETS {display-mode: "form"}
# 🔴 CELL 7 - APPROVED ASSETS
import importlib.metadata, subprocess, sys
try:
    _vx_edge_version = tuple(int(p) for p in importlib.metadata.version('edge-tts').split('.')[:2])
except importlib.metadata.PackageNotFoundError:
    _vx_edge_version = (0,0)
if _vx_edge_version < (7,2):
    subprocess.run([sys.executable,'-m','pip','install','-q','edge-tts>=7.2'],check=True)
import base64, hashlib
VX_WC_EMBEDDED_ASSET_DIR=VX_MEDIA_WORK/'wildcard_visual_assets'
VX_WC_EMBEDDED_ASSET_DIR.mkdir(parents=True,exist_ok=True)
_vx_asset_payloads = {'WC_clean.png': '...', 'grass_pitch.png': '...'}
for name,payload in _vx_asset_payloads.items():
    target=VX_WC_EMBEDDED_ASSET_DIR/name
    stamp=target.with_suffix(target.suffix+'.payload.sha256')
    payload_sha=hashlib.sha256(payload.encode('ascii')).hexdigest()
    if not (target.is_file() and stamp.is_file() and stamp.read_text().strip()==payload_sha):
        temporary=target.with_suffix(target.suffix+'.part')
        temporary.write_bytes(base64.b64decode(payload))
        temporary.replace(target)
        stamp.write_text(payload_sha)
# Observed match/news records carry their actual source and checked-at timestamp.
# They expire automatically; no permanent player, injury, season or GW override.
VX_WC_VERIFIED_EVIDENCE_JSON = '{\n  "checked_at": "2026-10-05T15:29:59.494533+00:00",\n  "events": [\n    {\n      "player_id": 411,\n      "team_id": 15,\n      "checked_at": "2026-10-05T15:29:59.494533+00:00",\n      "confirmed": true,\n      "source_url": "https://www.mancity.com/news/mens/international-roundup-4-october-2026-63926730",\n      "summary": "Norway appearance: substituted after 67 minutes; workload/fitness monitoring is needed. No club absence is confirmed.",\n      "monitor": true,\n      "match": {\n        "date": "2026-10-04T18:45:00Z",\n        "competition": "UEFA Nations League",\n        "opponent": "Portugal",\n        "minutes": 67,\n        "goals": 0,\n        "assists": null,\n        "rating": null,\n        "result": "Norway lost 2-1",\n        "date_precision": "kickoff"\n      }\n    },\n    {\n      "player_id": 12,\n      "team_id": 1,\n      "checked_at": "2026-10-05T15:29:59.494533+00:00",\n      "confirmed": true,\n      "source_url": "https://www.englandfootball.com/articles/2026/Oct/03/croatia-england-uefa-nations-league-harry-kane-bukayo-saka-reaction-20260310",\n      "summary": "Two goals for England against Croatia. Match minutes are unavailable from this report.",\n      "monitor": false,\n      "match": {\n        "date": "2026-10-03T00:00:00Z",\n        "date_precision": "day",\n        "competition": "UEFA Nations League",\n        "opponent": "Croatia",\n        "minutes": null,\n        "goals": 2,\n        "assists": null,\n        "rating": null,\n        "result": "England won 7-0"\n      }\n    },\n    {\n      "team_id": 15,\n      "checked_at": "2026-10-05T15:29:59.494533+00:00",\n      "confirmed": true,\n      "source_url": "https://www.premierleague.com/en/news/4729207",\n      "summary": "Manchester City appealed the commission decision. This statement supplies no confirmed fixture or player-eligibility change; no numerical penalty applied.",\n      "monitor": false,\n      "numeric_effect": null\n    }\n  ]\n}'
print("✅ CELL 7 COMPLETE")

✅ CELL 7 COMPLETE


In [10]:
#@title 🔴 CELL 8 - LIVE DATA & PROJECTIONS {display-mode: "form"}
# 🔴 CELL 8 - LIVE DATA & PROJECTIONS
import importlib.util, subprocess, sys
from uuid import uuid4
VX_LIVE_DATA_REVISION = uuid4().hex
VX_CELL1_COMPLETE = False
VX_CELL2_COMPLETE = False
for package in ('numpy', 'pandas', 'requests', 'scipy', 'ipywidgets'):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)
"""Live FPL projections and joint squad/XI/captain optimisation.

Player identities, prices, clubs, fixtures, gameweeks and squad constraints
come from the API. Statistical smoothing parameters are model assumptions,
not observed facts. No external team-news or shot-location data is invented.
"""
import os
import math
import json
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from scipy.optimize import milp, Bounds, LinearConstraint
from scipy.sparse import lil_matrix

# Legacy split live-model removed. Wildcard and Free Hit now share VXWC.WildcardEngine as the sole live data/FDR/evidence source.


# Shared live data/FDR/evidence foundation; strategy remains chip-specific.
VX_WC_ENGINE = None
if VX_GENERATE_WILDCARD or VX_GENERATE_FREEHIT:
    _shared_cfg = VXWC.Config(
        horizon=6, root=VX_ANALYSIS_ROOT, entry_id=VX_FPL_ENTRY_ID,
        budget=VX_WC_BUDGET if VX_WC_BUDGET>0 else None,
        bench_boost=VX_WC_BENCH_BOOST,
        bench_boost_gameweek=VX_WC_BB_GAMEWEEK if VX_WC_BB_GAMEWEEK>0 else None,
        free_transfers=VX_WC_FREE_TRANSFERS if VX_WC_FREE_TRANSFERS>=0 else None,
        fdr_policy=VX_WC_FDR_POLICY, enable_news=VX_WC_NEWS_ENABLED,
        enable_espn=VX_WC_INTERNATIONAL_ENABLED, solver_seconds=VX_WC_SOLVER_SECONDS,
        core_root=VX_ANALYSIS_ROOT/'model_inputs'/'fpl_core_live',
        model_inputs=VX_ANALYSIS_ROOT/'model_inputs',
        primary_news_urls=tuple(u.strip() for u in VX_WC_PRIMARY_NEWS_URLS.split(',') if u.strip())
    )
    _shared_cfg.model_inputs.mkdir(parents=True,exist_ok=True)
    _shared_seed = _shared_cfg.model_inputs/'verified_player_news.json'
    if not _shared_seed.exists():_shared_seed.write_text(VX_WC_VERIFIED_EVIDENCE_JSON,encoding='utf-8')
    VX_WC_ENGINE=VXWC.WildcardEngine(_shared_cfg).load()

_bootstrap=VX_WC_ENGINE.boot; VX_POOL=VX_WC_ENGINE.pool.copy()
_VX_GW_SCORES=VX_WC_ENGINE.scores.copy(); VX_FUTURE_GWS=VX_WC_ENGINE.primary
VX_BUDGET=VX_WC_ENGINE.budget
_VX_TYPES={e['id']:dict(e,position=e['singular_name_short'].replace('GKP','GK')) for e in _bootstrap['element_types']}
VX_MODEL_AUDIT=dict(VX_WC_ENGINE.audit,budget_source=VX_WC_ENGINE.budget_source,
    unavailable_inputs=VX_WC_ENGINE.audit['warnings'])
_VX_FIXTURES=VX_WC_ENGINE.fixtures
# Compatibility rankings now use the same centralized current-GW projections.
_current_score_map=_VX_GW_SCORES[_VX_GW_SCORES.gameweek.eq(VX_FUTURE_GWS[0])].set_index('player_id').gameweek_expected_points
VX_POOL['xpts_freehit_gw']=VX_POOL.player_id.map(_current_score_map).fillna(0.)
VX_POOL['freehit_fdr']=VX_POOL.apply(lambda r: (float(np.mean([f['fdr'] for f in VX_WC_ENGINE.fdr.for_team(int(r.team_id),VX_FUTURE_GWS[0],str(r.position))])) if VX_WC_ENGINE.fdr.for_team(int(r.team_id),VX_FUTURE_GWS[0],str(r.position)) else 5.0),axis=1)

VX_CURRENT_GW = VX_FUTURE_GWS[0]
VX_GW_WEIGHTS = [1.0] * len(VX_FUTURE_GWS)
VX_INPUT_ROOT = VX_ANALYSIS_ROOT
VX_INPUT_ROOT.mkdir(parents=True, exist_ok=True)
_VX_GW_SCORES.to_csv(VX_INPUT_ROOT / 'player_projections.csv', index=False)
(VX_INPUT_ROOT / 'model_audit.json').write_text(json.dumps(VX_MODEL_AUDIT, indent=2))
print(f'Live model: GW{VX_CURRENT_GW}, horizon {VX_FUTURE_GWS}, budget £{VX_BUDGET:.1f}m')
print('Budget source:', VX_MODEL_AUDIT['budget_source'])
print('Unavailable inputs:', ', '.join(VX_MODEL_AUDIT['unavailable_inputs']))
import difflib


def _vx_normalize(text):
    return str(text).strip().lower().replace(".", "").replace("-", " ")


def _vx_build_name_index(pool):
    idx = {}
    for _, r in pool.iterrows():
        pid = int(r["player_id"])
        web = r.get("player_name")
        if web:
            idx.setdefault(_vx_normalize(web), []).append(pid)
        full = " ".join(str(x) for x in (r.get("first_name"), r.get("second_name")) if x).strip()
        if full:
            idx.setdefault(_vx_normalize(full), []).append(pid)
    return idx


_VX_NAME_INDEX = _vx_build_name_index(VX_POOL)
_VX_ALL_NAMES = list(_VX_NAME_INDEX.keys())


def vx_resolve_player_name(name, pool=None):
    pool = VX_POOL if pool is None else pool
    raw = str(name).strip()
    if not raw:
        raise ValueError("Empty player name.")

    # Accept stable IDs directly and ID-bearing manual dropdown labels.
    id_match = re.search(r"(?:^| — )ID\s+(\d+)(?: — |$)", raw)
    if id_match:
        as_id = int(id_match.group(1))
        if as_id in set(pool["player_id"].astype(int)):
            return as_id
        raise ValueError(f"Unknown player_id {as_id} in {name!r}.")
    try:
        as_id = int(raw)
        if as_id in set(pool["player_id"].astype(int)):
            return as_id
    except ValueError:
        pass

    # Old dropdown labels carried name/price/xPts but no ID. Preserve the
    # display name here; Cell 9 uses the additional fields to disambiguate.
    raw_name = raw.split(" — ")[0].strip()
    key = _vx_normalize(raw_name)
    if key in _VX_NAME_INDEX:
        hits = _VX_NAME_INDEX[key]
        if len(hits) == 1:
            return hits[0]
        raise ValueError(f"'{name}' matches {len(hits)} players — be more specific: {hits}")
    substr_hits = sorted({pid for k, ids in _VX_NAME_INDEX.items() if key in k for pid in ids})
    if len(substr_hits) == 1:
        return substr_hits[0]
    if len(substr_hits) > 1:
        names = pool.loc[pool["player_id"].isin(substr_hits), "player_name"].tolist()
        raise ValueError(f"'{name}' is ambiguous — matches: {names}. Use full name or player_id.")
    close = difflib.get_close_matches(key, _VX_ALL_NAMES, n=5, cutoff=0.5)
    close_ids = sorted({pid for c in close for pid in _VX_NAME_INDEX[c]})
    close_names = pool.loc[pool["player_id"].isin(close_ids), "player_name"].tolist()
    raise ValueError(f"Could not resolve '{name}'. Closest matches: {close_names or 'none found'}")


def _vx_build_freehit_manual_squad(names_list, captain_name, vice_name, label, pool=None):
    """Resolves 15 typed/selected names into a validated squad frame.
    Returns None if the block was left entirely blank."""
    pool = VX_POOL if pool is None else pool
    filled = [n for n in names_list if str(n).strip()]
    if len(filled) == 0:
        return None
    if len(filled) != 15:
        raise ValueError(
            f"{label}: {len(filled)}/15 names entered — "
            "fill in all 15, or leave all 15 blank to use the model squad."
        )
    resolved_ids, errors = [], []
    for n in filled:
        try:
            resolved_ids.append(vx_resolve_player_name(n, pool))
        except ValueError as exc:
            errors.append(str(exc))
    if errors:
        raise ValueError(f"{label}: " + " | ".join(errors))
    if len(set(resolved_ids)) != 15:
        raise ValueError(f"{label}: resolved names contain duplicates: {resolved_ids}")

    meta = pool[["player_id", "player_name", "team_id", "position", "price"]].copy()
    frame = pd.DataFrame({"player_id": resolved_ids}).merge(meta, on="player_id", how="left", validate="one_to_one")
    frame["position"] = frame["position"].astype(str).str.upper().replace({"GKP": "GK"})

    required = {"GK": 2, "DEF": 5, "MID": 5, "FWD": 3}
    counts = frame["position"].value_counts().to_dict()
    bad = {p: int(counts.get(p, 0)) for p in required if int(counts.get(p, 0)) != required[p]}
    if bad:
        raise ValueError(f"{label}: illegal position counts (need 2 GK/5 DEF/5 MID/3 FWD): {counts}")
    club_counts = frame.groupby("team_id").size()
    if club_counts.max() > 3:
        raise ValueError(f"{label}: more than 3 players from one club: {club_counts.loc[club_counts > 3].to_dict()}")

    total_cost = float(frame["price"].sum())
    budget_source = str(globals().get("VX_MODEL_AUDIT", {}).get("budget_source", ""))
    exact_personal_budget = "authenticated selling prices plus bank" in budget_source.lower()
    if total_cost > VX_BUDGET + 1e-9:
        if exact_personal_budget:
            raise ValueError(
                f"{label}: squad costs £{total_cost:.1f}m, over your exact available budget £{VX_BUDGET:.1f}m"
            )
        # The public FPL API exposes the £100m starting budget, not this manager's
        # exact live Wildcard/Free-Hit selling budget. Manual mode must not reject
        # a deliberately selected squad solely because team value may have risen.
        print(
            f"⚠️ {label}: selected squad costs £{total_cost:.1f}m versus the "
            f"£{VX_BUDGET:.1f}m API starting-budget reference. No authenticated "
            "personal selling budget is available, so manual mode will continue "
            f"using £{total_cost:.1f}m as its working budget."
        )
        frame.attrs["working_budget"] = total_cost
        frame.attrs["budget_validation"] = "manual_assumed_from_selected_squad"
    else:
        frame.attrs["working_budget"] = float(VX_BUDGET)
        frame.attrs["budget_validation"] = (
            "authenticated_exact" if exact_personal_budget else "within_api_reference"
        )

    frame["is_captain"] = False
    frame["is_vice_captain"] = False
    if str(captain_name).strip():
        cap_id = vx_resolve_player_name(captain_name, pool)
        if cap_id not in resolved_ids:
            raise ValueError(f"{label}: captain '{captain_name}' is not in the 15-player squad.")
        frame.loc[frame["player_id"] == cap_id, "is_captain"] = True
    if str(vice_name).strip():
        vice_id = vx_resolve_player_name(vice_name, pool)
        if vice_id not in resolved_ids:
            raise ValueError(f"{label}: vice-captain '{vice_name}' is not in the 15-player squad.")
        if frame.loc[frame["player_id"] == vice_id, "is_captain"].any():
            raise ValueError(f"{label}: captain and vice-captain must be different players.")
        frame.loc[frame["player_id"] == vice_id, "is_vice_captain"] = True

    return frame
# Strict Wildcard manual validation; Free Hit continues through its original builder.
def vx_build_manual_squad(names_list,captain_name,vice_name,label,pool=None):
    if not label.startswith('Wildcard'):
        return _vx_build_freehit_manual_squad(names_list,captain_name,vice_name,label,pool)
    pool=VX_WC_ENGINE.pool
    ids=[vx_resolve_player_name(n,pool) for n in names_list if str(n).strip()]
    if not ids:return None
    if len(ids)!=15 or len(set(ids))!=15:raise ValueError('Wildcard requires 15 different resolved player IDs')
    squad=pool.set_index('player_id',drop=False).loc[ids].copy().reset_index(drop=True)
    VXWC.validate_squad(squad,VX_WC_ENGINE.budget)
    squad['is_captain']=False;squad['is_vice_captain']=False
    cap=vx_resolve_player_name(captain_name,pool) if str(captain_name).strip() else None
    vice=vx_resolve_player_name(vice_name,pool) if str(vice_name).strip() else None
    if cap and vice and cap==vice:raise ValueError('Captain and vice-captain must differ')
    for pid,col in [(cap,'is_captain'),(vice,'is_vice_captain')]:
        if pid:
            if pid not in ids:raise ValueError('Captain/vice must be in the squad')
            squad.loc[squad.player_id.eq(pid),col]=True
    squad.attrs['working_budget']=VX_WC_ENGINE.budget
    squad.attrs['budget_validation']=VX_WC_ENGINE.budget_source
    VX_WC_ENGINE.score_plan(squad)  # captain and vice must fit a legal XI
    return squad
# ── DEFENDER TWO-WAY INTELLIGENCE: LIVE EVIDENCE, EACH RUN ──
import importlib
import hashlib
if importlib.util.find_spec('matplotlib') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib'], check=True)
_VX_DEFENDER_TWO_WAY_SOURCE = '"""Official-FPL observed defender contributions + existing VORTEX model projections.\n\nRecomputed on every notebook execution from the current engine snapshot.\nDefensive contributions are match-stat counts, not awarded FPL points.\n"""\nfrom __future__ import annotations\n\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Mapping\nimport math\nimport numpy as np\nimport pandas as pd\n\nSOURCE = \'https://fantasy.premierleague.com/api/element-summary/{player_id}/\'\nVERSION = \'observed-defender-dual-route-v1\'\n\n\ndef _number(value: Any) -> float | None:\n    if value is None or value == \'\':\n        return None\n    try:\n        numeric = float(value)\n    except (TypeError, ValueError, OverflowError):\n        return None\n    return numeric if math.isfinite(numeric) else None\n\n\ndef _sample(player: Mapping[str, Any], history: list[dict[str, Any]], checked_events: set[int]) -> dict[str, Any] | None:\n    """Reject missing per-match fields rather than treating absent stats as zero."""\n    if not checked_events:\n        return None\n    selected = []\n    for entry in history:\n        try:\n            gameweek = int(entry[\'round\'])\n        except (KeyError, TypeError, ValueError):\n            continue\n        if gameweek not in checked_events:\n            continue\n        minutes = _number(entry.get(\'minutes\'))\n        defensive = _number(entry.get(\'defensive_contribution\'))\n        expected_goals = _number(entry.get(\'expected_goals\'))\n        expected_assists = _number(entry.get(\'expected_assists\'))\n        if minutes is None or defensive is None or expected_goals is None or expected_assists is None:\n            raise ValueError(f"Official defender {player[\'id\']} GW{gameweek} is missing DC, xG, xA or minutes")\n        if minutes < 0 or minutes > 130 or min(defensive, expected_goals, expected_assists) < 0:\n            raise ValueError(f"Official defender {player[\'id\']} GW{gameweek} has invalid match statistics")\n        selected.append((minutes, defensive, expected_goals, expected_assists, gameweek))\n    if not selected:\n        return None\n    minutes = sum(row[0] for row in selected)\n    if minutes <= 0:\n        return None\n    defensive_total = sum(row[1] for row in selected)\n    xgi_total = sum(row[2] + row[3] for row in selected)\n    ownership = _number(player.get(\'selected_by_percent\'))\n    if ownership is None or not 0 <= ownership <= 100:\n        raise ValueError(f"Official defender {player[\'id\']} has invalid ownership")\n    price_scale = 10.0\n    price = _number(player.get(\'now_cost\'))\n    if price is None:\n        raise ValueError(f"Official defender {player[\'id\']} lacks price")\n    return {\n        \'player_id\': int(player[\'id\']), \'player_name\': str(player[\'web_name\']),\n        \'team_id\': int(player[\'team\']), \'minutes\': round(minutes, 0),\n        \'games\': len(selected), \'defensive_contribution\': round(defensive_total, 2),\n        \'xgi\': round(xgi_total, 4), \'dc90\': round(90.0 * defensive_total / minutes, 3),\n        \'xgi90\': round(90.0 * xgi_total / minutes, 4),\n        \'ownership_pct\': round(ownership, 2), \'price\': price / price_scale,\n        \'availability\': str(player.get(\'status\', \'u\')),\n        \'sample_gws\': sorted({row[4] for row in selected}),\n        \'data_source\': SOURCE.format(player_id=int(player[\'id\'])),\n    }\n\n\ndef build_report(engine: Any, *, minimum_minutes: int = 360, minimum_ownership: float = 5.0) -> dict[str, Any]:\n    boot = engine.boot\n    events = boot.get(\'events\', [])\n    verified_gws = sorted({int(event[\'id\']) for event in events\n        if event.get(\'finished\') is True and event.get(\'data_checked\') is True\n        and int(event[\'id\']) < int(engine.target)})\n    if not verified_gws:\n        return {\'version\':VERSION,\'generated_at_utc\':datetime.now(timezone.utc).isoformat(),\n                \'gameweek\':int(engine.target),\'sample_gameweeks\':[],\n                \'sample_note\':\'Official FPL has no completed, data-checked Gameweeks yet.\',\n                \'source\':\'Official FPL bootstrap-static + element-summary player history\',\n                \'minimum_minutes\':minimum_minutes,\'minimum_ownership_pct\':minimum_ownership,\n                \'median_dc90\':None,\'median_xgi90\':None,\'cohort_size\':0,\n                \'all_defenders\':[],\'cohort_ids\':[],\'recommended_player_ids\':{},\n                \'leading_defenders\':[],\'status\':\'awaiting_verified_matches\'}\n    checked_events = set(verified_gws)\n    team_codes = {int(team[\'id\']):str(team.get(\'short_name\') or team[\'name\']) for team in boot[\'teams\']}\n    score_lookup = engine.pool.set_index(\'player_id\')\n    all_players = []\n    missing_history = []\n    for player in boot[\'elements\']:\n        if int(player[\'element_type\']) != 2 or player.get(\'removed\'):\n            continue\n        player_id = int(player[\'id\'])\n        history = engine.histories.get(player_id)\n        if history is None:\n            if (_number(player.get(\'minutes\')) or 0) >= minimum_minutes:\n                missing_history.append(player_id)\n            continue\n        evidence = _sample(player, history, checked_events)\n        if evidence is None:\n            continue\n        if player_id not in score_lookup.index:\n            continue\n        model = score_lookup.loc[player_id]\n        evidence.update(club=team_codes[int(player[\'team\'])],\n            expected_six_gw_points=round(float(model[\'xpts_wildcard_horizon\']), 2),\n            expected_next_gw_points=round(float(model[\'xpts_freehit_gw\']), 2),\n            model_start_probability=round(float(model[\'p_start\']), 4),\n            model_expected_minutes=round(float(model[\'expected_minutes\']), 1),\n            news_monitor=bool(model.get(\'news_monitor\',False)))\n        all_players.append(evidence)\n    cohort = [item for item in all_players\n        if item[\'minutes\'] >= minimum_minutes and item[\'ownership_pct\'] >= minimum_ownership]\n    sufficient_sample=len(cohort)>=4\n    dc_median = float(np.median([row[\'dc90\'] for row in cohort])) if sufficient_sample else None\n    xgi_median = float(np.median([row[\'xgi90\'] for row in cohort])) if sufficient_sample else None\n    for player in all_players:\n        if (not sufficient_sample or player[\'minutes\'] < minimum_minutes\n                or player[\'ownership_pct\'] < minimum_ownership):\n            player[\'quadrant\'] = \'OUTSIDE COHORT\'\n        elif player[\'dc90\'] >= dc_median:\n            player[\'quadrant\'] = \'DOUBLE ROUTE\' if player[\'xgi90\'] >= xgi_median else \'BALL WINNER\'\n        else:\n            player[\'quadrant\'] = \'ATTACK-FIRST\' if player[\'xgi90\'] >= xgi_median else \'LOWER CEILING\'\n    qualified = [player for player in all_players\n        if player[\'quadrant\'] != \'OUTSIDE COHORT\'\n        and player[\'availability\'] == \'a\' and not player[\'news_monitor\']\n        and player[\'model_start_probability\'] >= 0.75]\n    qualified.sort(key=lambda item: (-item[\'expected_six_gw_points\'],\n        -item[\'xgi90\'], -item[\'dc90\'], item[\'player_id\']))\n    recommendations = {}\n    for label in (\'DOUBLE ROUTE\', \'BALL WINNER\', \'ATTACK-FIRST\'):\n        recommendation = next((row for row in qualified if row[\'quadrant\'] == label), None)\n        if recommendation:\n            recommendations[label] = recommendation[\'player_id\']\n    top_eligible = qualified[:5]\n    output = {\n        \'version\': VERSION,\n        \'generated_at_utc\': datetime.now(timezone.utc).isoformat(),\n        \'gameweek\': int(engine.target),\n        \'sample_gameweeks\': verified_gws,\n        \'sample_note\': \'Only official FPL completed, data-checked Gameweeks; observed statistics, not future predictions.\',\n        \'source\': \'Official FPL bootstrap-static + element-summary player history\',\n        \'minimum_minutes\': minimum_minutes, \'minimum_ownership_pct\': minimum_ownership,\n        \'median_dc90\': round(dc_median, 4) if dc_median is not None else None,\n        \'median_xgi90\': round(xgi_median, 4) if xgi_median is not None else None,\n        \'cohort_size\': len(cohort),\n        \'all_defenders\': all_players,\n        \'cohort_ids\': [row[\'player_id\'] for row in cohort],\n        \'recommended_player_ids\': recommendations,\n        \'leading_defenders\': [row[\'player_id\'] for row in top_eligible],\n        \'missing_history_ids\': missing_history,\n        \'status\': \'ready\' if sufficient_sample else \'insufficient_verified_sample\',\n    }\n    return output\n\n\ndef enrich_pool(pool: pd.DataFrame, report: Mapping[str, Any]) -> pd.DataFrame:\n    result = pool.copy()\n    lookup = {int(player[\'player_id\']):player for player in report[\'all_defenders\']}\n    result[\'defender_dc90_observed\'] = result.player_id.map(lambda pid: lookup.get(int(pid), {}).get(\'dc90\'))\n    result[\'defender_xgi90_observed\'] = result.player_id.map(lambda pid: lookup.get(int(pid), {}).get(\'xgi90\'))\n    result[\'defender_two_way_quadrant\'] = result.player_id.map(lambda pid: lookup.get(int(pid), {}).get(\'quadrant\'))\n    return result\n\n\ndef render_chart(report: Mapping[str, Any], destination: str | Path) -> Path:\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    cohort_ids = set(report[\'cohort_ids\'])\n    rows = [player for player in report[\'all_defenders\'] if player[\'player_id\'] in cohort_ids]\n    if report.get(\'status\') != \'ready\' or len(rows)<4:\n        fig=plt.figure(figsize=(10, 11.4),facecolor=\'#020d19\')\n        fig.text(.07,.89,\'FPL VORTEX\',color=\'#eff7ff\',fontsize=30,fontweight=\'bold\')\n        fig.text(.07,.84,\'DEFENDER TWO-WAY INTELLIGENCE\',color=\'#43d6ff\',fontsize=20,fontweight=\'bold\')\n        fig.text(.07,.65,\'Awaiting enough verified defender data\',color=\'#f2f7fc\',fontsize=18)\n        fig.text(.07,.59,\'No invented or stale statistics will be displayed.\',color=\'#a5bfd0\',fontsize=14)\n        fig.text(.07,.07,f"GW{report[\'gameweek\']}  •  {report[\'sample_note\']}",color=\'#f1d47b\',fontsize=11)\n        fig.savefig(destination, facecolor=fig.get_facecolor(),dpi=175)\n        plt.close(fig)\n        return destination\n    colors = {\'DOUBLE ROUTE\':\'#2ff5a5\', \'BALL WINNER\':\'#a489ff\',\n              \'ATTACK-FIRST\':\'#4fc8ff\', \'LOWER CEILING\':\'#8493a6\'}\n    fig = plt.figure(figsize=(10, 11.4), facecolor=\'#020d19\')\n    axes = fig.add_axes([.10, .25, .82, .52], facecolor=\'#061827\')\n    max_x = max(.45, max(row[\'xgi90\'] for row in rows) * 1.17)\n    max_y = max(12, max(row[\'dc90\'] for row in rows) * 1.14)\n    axes.set(xlim=(0,max_x), ylim=(0,max_y))\n    axes.axhspan(report[\'median_dc90\'], max_y, alpha=.085, color=\'#ae84f1\')\n    axes.axvspan(report[\'median_xgi90\'], max_x, alpha=.065, color=\'#36dfa6\')\n    axes.axvline(report[\'median_xgi90\'], ls=\'--\', lw=1.3, color=\'#bda1fc\', alpha=.85)\n    axes.axhline(report[\'median_dc90\'], ls=\'--\', lw=1.3, color=\'#36dfa6\', alpha=.85)\n    axes.grid(alpha=.12, color=\'#87bbdb\')\n    for _,group in pd.DataFrame(rows).groupby(\'quadrant\'):\n        color = colors[str(group.iloc[0][\'quadrant\'])]\n        axes.scatter(group[\'xgi90\'],group[\'dc90\'],\n            s=45+group[\'ownership_pct\'].clip(upper=65)*2.6,\n            color=color, alpha=.92, edgecolors=\'#02101b\', linewidths=1.0, zorder=5)\n    axes.tick_params(colors=\'#a1bed0\', labelsize=11)\n    for spine in axes.spines.values():\n        spine.set_color(\'#315269\')\n    axes.set_xlabel(\'EXPECTED GOAL INVOLVEMENT / 90  →\',color=\'#e7f2f9\', fontsize=13, fontweight=\'bold\', labelpad=15)\n    axes.set_ylabel(\'DEFENSIVE CONTRIBUTIONS / 90  →\',color=\'#e7f2f9\', fontsize=13, fontweight=\'bold\', labelpad=15)\n    axes.text(.02,.975,\'BALL WINNERS\',transform=axes.transAxes,va=\'top\',color=\'#d0afff\',fontsize=13,fontweight=\'bold\')\n    axes.text(.98,.975,\'DOUBLE ROUTE\',transform=axes.transAxes,ha=\'right\',va=\'top\',color=\'#2ff5a5\',fontsize=13,fontweight=\'bold\')\n    axes.text(.02,.03,\'LOWER CEILING\',transform=axes.transAxes,va=\'bottom\',color=\'#a4b0bf\',fontsize=12,fontweight=\'bold\')\n    axes.text(.98,.03,\'ATTACK-FIRST\',transform=axes.transAxes,ha=\'right\',va=\'bottom\',color=\'#55d0fa\',fontsize=12,fontweight=\'bold\')\n    axes.text(.49,.96,f"xGI/90 median {report[\'median_xgi90\']:.2f}",transform=axes.transAxes,color=\'#d1beff\',fontsize=10,ha=\'right\',va=\'top\')\n    axes.text(.985,report[\'median_dc90\']/max_y+.012,f"DC/90 median {report[\'median_dc90\']:.2f}",\n        transform=axes.transAxes,color=\'#56f9bf\',fontsize=10,ha=\'right\',va=\'bottom\')\n    by_id={row[\'player_id\']:row for row in rows}\n    labels=[by_id[identifier] for identifier in report[\'recommended_player_ids\'].values() if identifier in by_id]\n    chosen={item[\'player_id\'] for item in labels}\n    for item in sorted(rows,key=lambda row:row[\'expected_six_gw_points\'],reverse=True):\n        if len(labels)>=4:break\n        if item[\'player_id\'] not in chosen:\n            labels.append(item);chosen.add(item[\'player_id\'])\n    for index, player in enumerate(labels):\n        x_position,y_position=player[\'xgi90\'],player[\'dc90\']\n        xoffset=10 if x_position<max_x*.72 else -10\n        yoffset=13 if index%2==0 else -14\n        axes.annotate(f"{player[\'player_name\']} · {player[\'club\']}", xy=(x_position,y_position),\n            xytext=(xoffset,yoffset),textcoords=\'offset points\',color=\'#f1f5fb\',fontsize=9,\n            ha=\'left\' if xoffset>0 else \'right\',va=\'center\',\n            arrowprops={\'arrowstyle\':\'-\',\'alpha\':.45,\'color\':\'#a9bace\',\'lw\':.7},zorder=9)\n    fig.text(.06,.963,\'FPL VORTEX\',color=\'#eff7ff\',fontsize=26,fontweight=\'bold\')\n    fig.text(.06,.927,\'DEFENDER TWO-WAY INTELLIGENCE\',color=\'#43d6ff\',fontsize=18,fontweight=\'bold\')\n    fig.text(.06,.894,\'Who earns points through defending AND attacking?\',color=\'#bec8dc\',fontsize=14)\n    fig.text(.94,.967,f"GW{report[\'gameweek\']} · LIVE FPL",ha=\'right\',color=\'#2df4aa\',fontsize=12,fontweight=\'bold\')\n    fig.text(.06,.851,f"Verified GW {\', \'.join(map(str, report[\'sample_gameweeks\']))}  •  {report[\'cohort_size\']} defenders  •  360+ min  •  5%+ owned",\n        color=\'#f0e6a1\',fontsize=12)\n    fig.text(.08,.176,\'DEFENDER RECOMMENDATIONS · SIX-WEEK MODEL\',color=\'#55d0fa\',fontsize=14,fontweight=\'bold\')\n    by_id = {item[\'player_id\']:item for item in rows}\n    recommendations=report[\'recommended_player_ids\']\n    for index, label in enumerate((\'DOUBLE ROUTE\',\'BALL WINNER\',\'ATTACK-FIRST\')):\n        y=.135-index*.041\n        if label not in recommendations:\n            continue\n        player = by_id[recommendations[label]]\n        fig.text(.08,y,label,color=colors[label],fontsize=11,fontweight=\'bold\')\n        fig.text(.36,y,f"{player[\'player_name\']} · {player[\'club\']}  |  DC {player[\'dc90\']:.1f}/90  |  xGI {player[\'xgi90\']:.2f}/90  |  {player[\'expected_six_gw_points\']:.1f} xPts",\n            color=\'#f4f7fc\',fontsize=10)\n    fig.text(.07,.018,\'SOURCE  Official Fantasy Premier League API  •  observed season statistics  •  model xPts are projections\',\n        color=\'#94afc4\',fontsize=9)\n    fig.text(.93,.018,report[\'generated_at_utc\'][:16].replace(\'T\',\' \')+\' UTC\',ha=\'right\',color=\'#94afc4\',fontsize=8)\n    fig.savefig(destination, facecolor=fig.get_facecolor(),dpi=175)\n    plt.close(fig)\n    if not destination.is_file() or destination.stat().st_size<10_000:\n        raise RuntimeError(\'Defender two-way visual was not written\')\n    return destination\n'
_vx_defender_module_path = Path(VX_MEDIA_WORK) / 'wildcard_engine' / 'defender_two_way_intelligence.py'
_vx_defender_module_path.parent.mkdir(parents=True, exist_ok=True)
_vx_defender_module_path.write_text(_VX_DEFENDER_TWO_WAY_SOURCE, encoding='utf-8')
sys.modules.pop('defender_two_way_intelligence', None)
importlib.invalidate_caches()
from defender_two_way_intelligence import build_report as _vx_build_defender_report
from defender_two_way_intelligence import enrich_pool as _vx_enrich_defender_pool
from defender_two_way_intelligence import render_chart as _vx_render_defender_chart
VX_DEFENDER_REPORT = _vx_build_defender_report(VX_WC_ENGINE)
VX_WC_ENGINE.defender_two_way = VX_DEFENDER_REPORT
VX_POOL = _vx_enrich_defender_pool(VX_POOL, VX_DEFENDER_REPORT)
VX_WC_ENGINE.pool = _vx_enrich_defender_pool(VX_WC_ENGINE.pool, VX_DEFENDER_REPORT)
_VX_DEFENDER_CHART = _vx_render_defender_chart(
    VX_DEFENDER_REPORT, Path(VX_ANALYSIS_ROOT) / f'defender_two_way_GW{VX_CURRENT_GW}.png')
Path(VX_ANALYSIS_ROOT, f'defender_two_way_GW{VX_CURRENT_GW}.json').write_text(
    json.dumps(VX_DEFENDER_REPORT, indent=2, ensure_ascii=False), encoding='utf-8')
pd.DataFrame(VX_DEFENDER_REPORT['all_defenders']).to_csv(
    Path(VX_ANALYSIS_ROOT) / f'defender_two_way_GW{VX_CURRENT_GW}.csv', index=False)
VX_MODEL_AUDIT['defender_two_way'] = {
    'source': VX_DEFENDER_REPORT['source'],
    'verified_gws': VX_DEFENDER_REPORT['sample_gameweeks'],
    'cohort_size': VX_DEFENDER_REPORT['cohort_size'],
    'observation': 'DC/90 and xGI/90 observed; selection xPts separately forecast by existing model',
    'chart_path': str(_VX_DEFENDER_CHART),
}
VX_WC_ENGINE.audit['defender_two_way'] = VX_MODEL_AUDIT['defender_two_way']
Path(VX_ANALYSIS_ROOT, 'model_audit.json').write_text(
    json.dumps(VX_MODEL_AUDIT, indent=2, ensure_ascii=False), encoding='utf-8')
import shutil as _vx_defender_shutil
Path(VX_4K_PNG_OUT_DIR).mkdir(parents=True, exist_ok=True)
_vx_defender_shutil.copy2(_VX_DEFENDER_CHART,
    Path(VX_4K_PNG_OUT_DIR) / _VX_DEFENDER_CHART.name)
print(f"Defenders: {VX_DEFENDER_REPORT['cohort_size']} qualified; "
      f"GWs {VX_DEFENDER_REPORT['sample_gameweeks']}; "
      f"chart {_VX_DEFENDER_CHART}", flush=True)
# ── END DEFENDER INTELLIGENCE ──

print("✅ CELL 8 COMPLETE")

FPL Core refresh: 60 completed fixtures verified; 9 red-card timings verified
✅ Phase 1 identity: 2025-26 matches use teams.csv `code` keys; 20 previous-season PL clubs resolved.
ℹ️ Phase 1: omitted 40 red-card-distorted match(es) whose exact send-off timing could not be proven.
Live model: GW7, horizon [7, 8, 9, 10, 11, 12], budget £100.0m
Budget source: official starting-budget reference; personal selling budget unknown
Unavailable inputs: Bench Boost availability unknown; conservative XI-focused strategy used
Defenders: 23 qualified; GWs [1, 2, 3, 4, 5, 6]; chart /content/drive/MyDrive/FPL_VORTEX/Analysis/wc_fh/defender_two_way_GW7.png
✅ CELL 8 COMPLETE


In [11]:
#@title 🔴 CELL 9 - MANUAL PLAYER OPTIONS {display-mode: "form"}
# 🔴 CELL 9 - MANUAL PLAYER OPTIONS
"""
CELL 7 — MANUAL SQUAD ENTRY VIA DROPDOWNS (Wildcard & Free Hit) — STANDALONE

Runs after Cell 6. Builds dropdown option lists straight from VX_POOL's own
xpts_wildcard_horizon / xpts_freehit_gw columns (no external score table
needed) — sorted descending by expected points within each position.

Each dropdown lists every player at that position, formatted as
"Name — ID <FPL id> — £price m — x.xx xPts". You can either click a dropdown option, or
type a few letters of a first/last name and Colab's widget UI filters live.

Only builds manual choices for a chip whose visible selection mode in Cell 0
is set to Manual. Auto mode needs no player input and is optimized by the model.
"""

_VX_REQUIRED_HERE = ("VX_POOL", "_VX_NAME_INDEX", "vx_resolve_player_name", "vx_build_manual_squad")
_missing_0b = [n for n in _VX_REQUIRED_HERE if n not in globals()]
if _missing_0b:
    raise RuntimeError(
        "Cell 7 must run AFTER Cell 6 (missing: " + ", ".join(_missing_0b) + "). "
        "Run Cells 0 through 6 first, then re-run Cell 7."
    )

_DROPDOWN_NONE = "— none —"


def _vx_build_dropdown_options(position, xpts_col):
    """Sorted (desc xPts) display-label list for one position, plus a
    label->player_id map for resolving the selection back to an id."""
    pool=VX_WC_ENGINE.pool if xpts_col=='xpts_wildcard_horizon' and VX_WC_ENGINE is not None else VX_POOL
    pos_pool = pool.loc[pool["position"].astype(str).str.upper() == position].copy()
    pos_pool = pos_pool.sort_values([xpts_col, 'player_name'],ascending=[False,True],kind='mergesort')
    # Recommendations come from the notebook's actual horizon/single-GW projections,
    # not hand-picked lists. Full verified position pool remains selectable below.
    candidates = pos_pool.loc[pos_pool.get('status', 'a').astype(str).eq('a')] if 'status' in pos_pool else pos_pool
    leaders = set(candidates.head(max(3, {'GK':2,'DEF':5,'MID':5,'FWD':3}[position]))['player_id'].astype(int))
    ordered = pos_pool[pos_pool.player_id.astype(int).isin(leaders)]
    ordered = __import__('pandas').concat([ordered,pos_pool[~pos_pool.player_id.astype(int).isin(leaders)]],ignore_index=True)
    labels = [_DROPDOWN_NONE]
    for _, r in ordered.iterrows():
        pid=int(r['player_id']); is_rec=pid in leaders
        # The official selected_by_percent is overall ownership, NOT effective ownership.
        # Include validated EO only when an explicit source column exists.
        eo=next((float(r[k]) for k in ('effective_ownership_pct','effective_ownership','eo_pct')
                 if k in ordered.columns and __import__('pandas').notna(r[k])),None)
        eo_text=f'{eo:.1f}%' if eo is not None else 'N/A'
        label=(f"{position} | {'★ RECOMMENDED' if is_rec else 'ALL PLAYERS'} | "
               f"{r['player_name']} — ID {pid} — £{float(r['price']):.1f}m — "
               f"EO {eo_text} — {float(r[xpts_col]):.2f} xPts")
        labels.append(label)
    return labels


VX_WC_GK_OPTS = _vx_build_dropdown_options("GK", "xpts_wildcard_horizon")
VX_WC_DEF_OPTS = _vx_build_dropdown_options("DEF", "xpts_wildcard_horizon")
VX_WC_MID_OPTS = _vx_build_dropdown_options("MID", "xpts_wildcard_horizon")
VX_WC_FWD_OPTS = _vx_build_dropdown_options("FWD", "xpts_wildcard_horizon")

VX_FH_GK_OPTS = _vx_build_dropdown_options("GK", "xpts_freehit_gw")
VX_FH_DEF_OPTS = _vx_build_dropdown_options("DEF", "xpts_freehit_gw")
VX_FH_MID_OPTS = _vx_build_dropdown_options("MID", "xpts_freehit_gw")
VX_FH_FWD_OPTS = _vx_build_dropdown_options("FWD", "xpts_freehit_gw")

print(f"✅ Dropdown pools built — Wildcard horizon xPts ({VX_FUTURE_GWS}), Free Hit GW{VX_CURRENT_GW} xPts")
print("   Scroll down, expand the form fields below, and pick players (or type to filter).")
print("✅ CELL 9 COMPLETE")

✅ Dropdown pools built — Wildcard horizon xPts ([7, 8, 9, 10, 11, 12]), Free Hit GW7 xPts
   Scroll down, expand the form fields below, and pick players (or type to filter).
✅ CELL 9 COMPLETE


In [12]:
#@title 🔴 CELL 10 - MANUAL 15-PLAYER SELECTION {display-mode: "form"}
# 🔴 CELL 10 - MANUAL 15-PLAYER SELECTION
"""
CELL 8 — MANUAL SQUAD DROPDOWN UI

- Search players by any typed name fragment; IN dropdowns update instantly.
- Existing picks persist while search results change; Clear restores all options.
- Shows only the chip(s) selected by VX_GENERATE_CHIPS in Cell 0.
- A green SAVE MANUAL SELECTIONS button snapshots the picks.
- Cell 9 refuses to run until every enabled manual squad has been saved.
"""

try:
    import ipywidgets as widgets
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"], check=False)
    import ipywidgets as widgets
from IPython.display import display, HTML

if "VX_GENERATE_WILDCARD" not in globals() or "VX_GENERATE_FREEHIT" not in globals():
    raise RuntimeError("Run Cell 0 first so the Generate selector is available.")
if "VX_POOL" not in globals():
    raise RuntimeError("Run Cells 6 and 7 before Cell 8.")

_VX_POS_LABELS = [("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3)]
VX_MANUAL_WIDGETS = {}
VX_MANUAL_SAVED_VALUES = {}
VX_MANUAL_SELECTION_SAVED = False
VX_CELL0C_COMPLETE = False
VX_MANUAL_BUDGET_WIDGETS = {}
VX_MANUAL_BUDGET_SIDEBARS = {}
VX_MANUAL_SEARCH_WIDGETS = {}
VX_MANUAL_SEARCH_CLEAR_BUTTONS = {}
VX_MANUAL_SEARCH_FEEDBACK = {}
VX_MANUAL_FULL_OPTIONS = {}
VX_MANUAL_SELECTED_BUDGETS = {}
VX_MANUAL_OUT_WIDGETS = {}
VX_MANUAL_TRANSFERS = {}


def _vx_current_squad_out_options():
    """Read the user's last published public squad; never invent outgoing players."""
    import requests

    entry = str(globals().get('VX_FPL_ENTRY_ID', '')).strip()
    if not entry.isdigit():
        return {}, 'No official FPL entry ID configured; showing IN selectors only.'
    # The current game's picks may be unpublished before the GW deadline.
    gameweeks = [gw for gw in (int(VX_CURRENT_GW)-1, int(VX_CURRENT_GW), int(VX_CURRENT_GW)-2) if 1 <= gw <= 38]
    errors = []
    for gw in dict.fromkeys(gameweeks):
        try:
            response = requests.get(
                f'https://fantasy.premierleague.com/api/entry/{entry}/event/{gw}/picks/',
                timeout=12,
                headers={'User-Agent':'FPL-Vortex/2.0 (analytics)'},
            )
            response.raise_for_status()
            players = response.json().get('picks', [])
            ids = [int(player['element']) for player in players]
            if len(ids) != 15 or len(set(ids)) != 15:
                raise ValueError(f'GW{gw} public picks did not contain 15 unique players')
            pool = VX_POOL.set_index('player_id', drop=False)
            if any(pid not in pool.index for pid in ids):
                raise ValueError('Published picks contain an ID missing from current official player pool')
            choices = {}
            for position, count in _VX_POS_LABELS:
                current = pool.loc[[pid for pid in ids if str(pool.loc[pid,'position']).upper() == position]]
                if len(current) != count:
                    raise ValueError(f'Published team has an invalid {position} count')
                choices[position] = [
                    f"{row['player_name']} — ID {int(row['player_id'])} — £{float(row['price']):.1f}m"
                    for _,row in current.iterrows()
                ]
            return choices, f'OUT list loaded from FPL entry {entry}, official GW{gw} published team.'
        except (requests.RequestException,ValueError,KeyError,TypeError) as exc:
            errors.append(f'GW{gw}: {type(exc).__name__}')
    return {}, ('OUT selectors unavailable (' + ', '.join(errors) + '). '
                'Choose the 15 IN players; no unverifiable outgoing player is shown.')


_VX_AVAILABLE_OUT_BY_POSITION, VX_MANUAL_OUT_STATUS = _vx_current_squad_out_options() if (
    USE_MANUAL_WILDCARD or USE_MANUAL_FREEHIT
) else ({}, 'Auto mode: no manual player choices needed.')


def _vx_make_position_dropdowns(position, count, options, key_prefix):
    """True click-open dropdowns. Every option carries the unique FPL ID."""
    box_list, widget_map = [], {}
    for i in range(1, count + 1):
        dd = widgets.Dropdown(
            options=options,
            value=_DROPDOWN_NONE,
            description=f"IN {position}{i}:",
            style={"description_width": "80px"},
            layout=widgets.Layout(width="680px"),
        )
        widget_map[f"{key_prefix}_{position}{i}"] = dd
        old_choices = _VX_AVAILABLE_OUT_BY_POSITION.get(position, [])
        if len(old_choices) >= i:
            outgoing = widgets.Dropdown(
                options=old_choices,
                value=old_choices[i - 1],
                description=f"OUT {position}{i}:",
                style={"description_width": "82px"},
                layout=widgets.Layout(width="270px"),
            )
            VX_MANUAL_OUT_WIDGETS[f"{key_prefix}_OUT_{position}{i}"] = outgoing
            dd.layout = widgets.Layout(width="540px")
            box_list.append(widgets.HBox([outgoing, dd]))
        else:
            box_list.append(dd)
    return box_list, widget_map



def _vx_price_from_label(label):
    """Extract the displayed FPL price from a dropdown option."""
    m = __import__("re").search(r"£\s*([0-9]+(?:\.[0-9]+)?)m", str(label))
    return float(m.group(1)) if m else 0.0


def _vx_budget_context():
    """Use the same live budget and provenance as the squad optimizer."""
    engine = globals().get("VX_WC_ENGINE")
    raw_budget = getattr(engine, "budget", None) if engine is not None else globals().get("VX_BUDGET")
    if raw_budget is None:
        raise RuntimeError("Manual budget is unavailable; run Cell 8 before Cell 10.")
    budget = float(raw_budget)
    import math
    if not math.isfinite(budget) or budget <= 0:
        raise ValueError(f"Manual budget must be positive and finite, got {raw_budget!r}")

    source = str(getattr(engine, "budget_source", "") if engine is not None
                 else globals().get("VX_MODEL_AUDIT", {}).get("budget_source", ""))
    provenance = source.casefold()
    authenticated = "authenticated selling prices plus bank" in provenance
    configured = "explicit manager budget" in provenance
    if authenticated:
        label = "Authenticated available budget"
    elif configured:
        label = "Configured available budget"
    else:
        label = "Public FPL budget reference"
    return budget, (authenticated or configured), label


def _vx_render_budget_box(prefix):
    """Update the top banner and pinned sidebar from the same live selections."""
    budget_widget = VX_MANUAL_BUDGET_WIDGETS.get(prefix)
    sidebar_widget = VX_MANUAL_BUDGET_SIDEBARS.get(prefix)
    if budget_widget is None and sidebar_widget is None:
        return

    keys = [
        f"{prefix}_{position}{index}"
        for position, count in _VX_POS_LABELS
        for index in range(1, count + 1)
    ]
    values = []
    for key in keys:
        dropdown = VX_MANUAL_WIDGETS.get(key)
        value = str(dropdown.value or "").strip() if dropdown is not None else ""
        if value and value != _DROPDOWN_NONE:
            values.append(value)

    # Dropdown option labels contain the current official FPL purchase price.
    selected_cost = round(sum(_vx_price_from_label(value) for value in values), 1)
    selected_count = len(values)
    player_ids = [_vx_id_from_label(value) for value in values]
    valid_ids = [player_id for player_id in player_ids if player_id is not None]
    duplicate_count = len(valid_ids) - len(set(valid_ids))
    budget, confirmed, budget_label = _vx_budget_context()
    remaining = round(budget - selected_cost, 1)
    complete = selected_count == sum(count for _, count in _VX_POS_LABELS)
    chip_label = "Wildcard" if prefix == "WC" else "Free Hit"

    if remaining < -0.000001:
        accent, background = "#ff8585", "#4a1515"
        condition = "OVER BUDGET" if confirmed else "ABOVE REFERENCE"
        balance = f"£{abs(remaining):.1f}m {condition}"
    elif complete and confirmed:
        accent, background = "#3fe184", "#0d3520"
        balance = f"WITHIN BUDGET · £{remaining:.1f}m left"
    elif complete:
        accent, background = "#ffc857", "#3d3010"
        balance = f"£{remaining:.1f}m below reference — personal budget unverified"
    else:
        accent, background = "#58d8f3", "#113448"
        balance = f"£{remaining:.1f}m available for remaining picks" if remaining >= 0 else f"£{abs(remaining):.1f}m above reference"

    duplicate_note = (
        f"<div style='color:#ffb6b6;margin-top:7px;font-weight:700'>"
        f"⚠️ {duplicate_count} duplicate selection{'s' if duplicate_count != 1 else ''}</div>"
        if duplicate_count else ""
    )
    provenance_note = (
        "" if confirmed else
        "<div style='font-size:12px;line-height:1.35;margin-top:7px;color:#e8ddad'>"
        "Your personal selling-value budget is not verified. "
        "This compares against the public FPL starting-budget reference only.</div>"
    )
    common = (
        f"<div style='padding:13px;border-radius:12px;border:2px solid {accent};"
        f"background:{background};color:white;font-family:Arial,sans-serif;"
        "font-variant-numeric:tabular-nums;box-sizing:border-box'>"
        f"<div style='font-weight:800;font-size:15px;margin-bottom:8px'>💰 {chip_label} LIVE COST</div>"
        f"<div style='font-size:13px;margin-bottom:5px'>{selected_count}/15 players chosen</div>"
        f"<div style='font-size:23px;font-weight:800'>£{selected_cost:.1f}m</div>"
        f"<div style='font-size:12px;margin:7px 0'>{budget_label}: £{budget:.1f}m</div>"
        f"<div style='font-size:14px;font-weight:800;color:{accent};line-height:1.5'>{balance}</div>"
        f"{duplicate_note}{provenance_note}</div>"
    )
    if budget_widget is not None:
        budget_widget.value = common
    if sidebar_widget is not None:
        sidebar_widget.value = common
    VX_MANUAL_SELECTED_BUDGETS[prefix] = selected_cost


def _vx_attach_budget_observers(prefix):
    """Make the budget box react immediately to every player-dropdown change."""
    for pos, count in _VX_POS_LABELS:
        for i in range(1, count + 1):
            key = f"{prefix}_{pos}{i}"
            w = VX_MANUAL_WIDGETS.get(key)
            if w is not None:
                w.observe(lambda change, p=prefix: _vx_render_budget_box(p), names="value")
    _vx_render_budget_box(prefix)


def _vx_normalize_player_search(value):
    """Match what users type without case or accent sensitivity."""
    import unicodedata
    normalized = unicodedata.normalize("NFKD", str(value or ""))
    return "".join(char for char in normalized if not unicodedata.combining(char)).casefold().strip()


def _vx_option_player_name(option):
    """Only search player names, never the RECOMMENDED, team or xPts metadata."""
    text = str(option)
    return text.split(" — ID ", 1)[0].rsplit(" | ", 1)[-1].strip()


def _vx_filter_manual_players(prefix, query):
    """Filter all 15 IN selectors in this chip while preserving selected values."""
    pools = VX_MANUAL_FULL_OPTIONS.get(prefix)
    if pools is None:
        return
    term = _vx_normalize_player_search(query)
    total_matches = 0
    for position, count in _VX_POS_LABELS:
        options = pools[position]
        matches = [
            option for option in options
            if option != _DROPDOWN_NONE
            and (not term or term in _vx_normalize_player_search(_vx_option_player_name(option)))
        ]
        total_matches += len(matches)
        for index in range(1, count + 1):
            dropdown = VX_MANUAL_WIDGETS.get(f"{prefix}_{position}{index}")
            if dropdown is None:
                continue
            current = dropdown.value
            visible = [_DROPDOWN_NONE, *matches]
            # Keeping an existing pick among the options prevents ipywidgets from
            # clearing the 15-player squad when the search text changes.
            if current not in (None, _DROPDOWN_NONE) and current not in visible:
                visible.append(current)
            dropdown.options = visible
            if current in visible and dropdown.value != current:
                dropdown.value = current

    feedback = VX_MANUAL_SEARCH_FEEDBACK.get(prefix)
    if feedback is not None:
        if not term:
            message = f"Showing all {total_matches} players. Search narrows the IN dropdowns."
        elif total_matches:
            message = f"{total_matches} matching player{'s' if total_matches != 1 else ''} — choose the appropriate position slot."
        else:
            message = "No matching player names. Your existing picks are preserved."
        feedback.value = (
            "<div style='font-size:12px;color:#d0e9f4;margin:5px 0 12px;"
            "line-height:1.35'>" + message + "</div>"
        )


def _vx_build_chip_ui(chip_label, key_prefix, opt_map):
    all_widgets = {}
    top_budget = widgets.HTML()
    side_budget = widgets.HTML(layout=widgets.Layout(width="285px"))
    VX_MANUAL_BUDGET_WIDGETS[key_prefix] = top_budget
    VX_MANUAL_BUDGET_SIDEBARS[key_prefix] = side_budget
    VX_MANUAL_FULL_OPTIONS[key_prefix] = {
        position: tuple(opt_map[position]) for position, _ in _VX_POS_LABELS
    }
    search = widgets.Text(
        value="",
        placeholder="Name, e.g. Saka",
        continuous_update=True,
        layout=widgets.Layout(width="204px"),
    )
    clear_search = widgets.Button(
        description="Clear",
        tooltip="Show every player again",
        layout=widgets.Layout(width="67px"),
    )
    search_feedback = widgets.HTML()
    VX_MANUAL_SEARCH_WIDGETS[key_prefix] = search
    VX_MANUAL_SEARCH_CLEAR_BUTTONS[key_prefix] = clear_search
    VX_MANUAL_SEARCH_FEEDBACK[key_prefix] = search_feedback
    search.observe(
        lambda change, prefix=key_prefix: _vx_filter_manual_players(prefix, change["new"]),
        names="value",
    )
    clear_search.on_click(lambda button, field=search: setattr(field, "value", ""))
    rows = [
        widgets.HTML(
            f"<h4 style='margin:8px 0'>{chip_label} — choose the final 15</h4>"
            "<div style='color:#cfe9f7;margin-bottom:8px'>"
            "OUT = existing squad player; IN = the player to keep or bring in. "
            "Pick a complete legal 15-player IN squad (2 GK, 5 DEF, 5 MID, 3 FWD). "
            "Unchanged OUT → IN pairs stay in the squad; captain and vice are automatic."
            "</div>"
        ),
        top_budget,
        widgets.HTML(f"<p style='font-size:13px;color:#ccdbeb'>{VX_MANUAL_OUT_STATUS}</p>"),
    ]
    for position, count in _VX_POS_LABELS:
        boxes, widget_map = _vx_make_position_dropdowns(position, count, opt_map[position], key_prefix)
        all_widgets.update(widget_map)
        rows.append(widgets.VBox(boxes))

    # Hidden captain/vice controls preserve the existing saved-selection contract.
    captain = widgets.Dropdown(options=[_DROPDOWN_NONE], value=_DROPDOWN_NONE)
    vice = widgets.Dropdown(options=[_DROPDOWN_NONE], value=_DROPDOWN_NONE)
    all_widgets[f"{key_prefix}_CAPTAIN"] = captain
    all_widgets[f"{key_prefix}_VICE"] = vice

    # Keep a persistent side budget alongside the long player list.
    selector_column = widgets.VBox(rows, layout=widgets.Layout(flex="1 1 760px", min_width="550px"))
    sidebar = widgets.VBox([
        side_budget,
        widgets.HTML("<div style='margin:12px 0 5px;font-weight:700;color:#d1f1ff'>🔎 SEARCH PLAYER NAME</div>"),
        widgets.HBox([search, clear_search]),
        search_feedback,
    ], layout=widgets.Layout(
        width="295px", flex="0 0 295px", align_self="flex-start"))
    sidebar.add_class("vx-manual-budget-sticky")
    display(widgets.HBox([selector_column, sidebar], layout=widgets.Layout(
        width="100%", align_items="flex-start", overflow="visible")))
    return all_widgets


# Stick the cost summary to the upper right while the manual dropdowns scroll.
# The duplicate top summary remains visible if a narrow/mobile output cannot show the sidebar.
display(HTML("""<style>
.vx-manual-budget-sticky {
  position: sticky !important;
  top: 12px !important;
  z-index: 20 !important;
  align-self: flex-start !important;
}
</style>"""))

_enabled_manual_prefixes = []

if VX_GENERATE_WILDCARD:
    if USE_MANUAL_WILDCARD:
        display(HTML(
            "<hr><h3>🟣 Manual Wildcard</h3>"
            "<p>Choose exactly 2 GK / 5 DEF / 5 MID / 3 FWD. "
            f"Players are ordered by Wildcard-horizon xPts for {VX_FUTURE_GWS}.</p>"
        ))
        _wc_opts = {
            "GK": VX_WC_GK_OPTS, "DEF": VX_WC_DEF_OPTS,
            "MID": VX_WC_MID_OPTS, "FWD": VX_WC_FWD_OPTS,
        }
        VX_MANUAL_WIDGETS.update(_vx_build_chip_ui("Wildcard", "WC", _wc_opts))
        _enabled_manual_prefixes.append("WC")
        _vx_attach_budget_observers("WC")
        _vx_filter_manual_players("WC", "")
    else:
        display(HTML(
            "<hr><div style='padding:10px;border:1px solid #7ab8d3;border-radius:8px'>"
            "✅ <b>Wildcard = Auto.</b> No manual player save is required for Wildcard."
            "</div>"
        ))
else:
    display(HTML(
        "<hr><div style='padding:10px;opacity:.8'>⏭️ Wildcard generation is OFF in Cell 0.</div>"
    ))

if VX_GENERATE_FREEHIT:
    if USE_MANUAL_FREEHIT:
        display(HTML(
            "<hr><h3>🟢 Manual Free Hit</h3>"
            "<p>Choose exactly 2 GK / 5 DEF / 5 MID / 3 FWD. "
            f"Players are ordered by one-week xPts for GW{VX_CURRENT_GW}.</p>"
        ))
        _fh_opts = {
            "GK": VX_FH_GK_OPTS, "DEF": VX_FH_DEF_OPTS,
            "MID": VX_FH_MID_OPTS, "FWD": VX_FH_FWD_OPTS,
        }
        VX_MANUAL_WIDGETS.update(_vx_build_chip_ui("Free Hit", "FH", _fh_opts))
        _enabled_manual_prefixes.append("FH")
        _vx_attach_budget_observers("FH")
        _vx_filter_manual_players("FH", "")
    else:
        display(HTML(
            "<hr><div style='padding:10px;border:1px solid #7ab8d3;border-radius:8px'>"
            "✅ <b>Free Hit = Auto.</b> No manual player save is required for Free Hit."
            "</div>"
        ))
else:
    display(HTML(
        "<hr><div style='padding:10px;opacity:.8'>⏭️ Free Hit generation is OFF in Cell 0.</div>"
    ))


def _vx_selected_squad_values(prefix):
    keys = [
        f"{prefix}_{pos}{i}"
        for pos, count in _VX_POS_LABELS
        for i in range(1, count + 1)
    ]
    return keys, [str(VX_MANUAL_WIDGETS[k].value or "").strip() for k in keys]


def _vx_id_from_label(label):
    m = __import__("re").search(r"(?:^| — )ID\s+(\d+)(?: — |$)", str(label))
    return int(m.group(1)) if m else None


_status = widgets.HTML()
_save_btn = widgets.Button(
    description="SAVE MANUAL SELECTIONS",
    button_style="success",
    icon="check",
    layout=widgets.Layout(width="360px", height="46px"),
    tooltip="Validate and lock the manual team(s) before Cell 9.",
)


def _vx_save_manual_selections(_):
    global VX_MANUAL_SELECTION_SAVED, VX_MANUAL_SAVED_VALUES, VX_CELL0C_COMPLETE

    errors = []
    snapshot = {}
    saved_costs = {}
    outgoing_changes = {}

    for prefix in _enabled_manual_prefixes:
        keys, values = _vx_selected_squad_values(prefix)
        missing = [k for k, v in zip(keys, values) if not v or v == _DROPDOWN_NONE]
        if missing:
            errors.append(f"{prefix}: choose all 15 players ({len(missing)} still empty).")
            continue

        ids = [_vx_id_from_label(v) for v in values]
        if any(pid is None for pid in ids):
            errors.append(f"{prefix}: one or more selections do not contain a valid FPL player ID.")
            continue
        if len(set(ids)) != 15:
            errors.append(f"{prefix}: the same player is selected more than once.")
            continue

        saved_costs[prefix] = round(sum(_vx_price_from_label(v) for v in values), 1)
        if _VX_AVAILABLE_OUT_BY_POSITION:
            out_keys = [f"{prefix}_OUT_{pos}{i}" for pos,count in _VX_POS_LABELS for i in range(1,count+1)]
            out_labels = [str(VX_MANUAL_OUT_WIDGETS[k].value) for k in out_keys]
            out_ids = [_vx_id_from_label(label) for label in out_labels]
            expected_out_ids = [_vx_id_from_label(label) for pos,_ in _VX_POS_LABELS for label in _VX_AVAILABLE_OUT_BY_POSITION[pos]]
            if any(pid is None for pid in out_ids) or len(set(out_ids)) != 15 or set(out_ids) != set(expected_out_ids):
                errors.append(f'{prefix}: OUT dropdowns must include all 15 unique players from your published squad.')
                continue
            outgoing_changes[prefix] = [
                {'out_id':old, 'in_id':new, 'out_label':old_label, 'in_label':new_label}
                for old,new,old_label,new_label in zip(out_ids, ids, out_labels,values) if old != new
            ]
            for key, value in zip(out_keys, out_labels):
                snapshot[key] = value
        if prefix=='WC':
            try:
                chosen=VX_WC_ENGINE.pool.set_index('player_id',drop=False).loc[ids].copy().reset_index(drop=True)
                VXWC.validate_squad(chosen,VX_WC_ENGINE.budget)
            except (AssertionError,ValueError,KeyError) as exc:
                errors.append('WC: illegal squad or over budget: '+str(exc))
                continue

        captain = str(VX_MANUAL_WIDGETS[f"{prefix}_CAPTAIN"].value or "").strip()
        vice = str(VX_MANUAL_WIDGETS[f"{prefix}_VICE"].value or "").strip()
        cap_id = _vx_id_from_label(captain) if captain and captain != _DROPDOWN_NONE else None
        vice_id = _vx_id_from_label(vice) if vice and vice != _DROPDOWN_NONE else None

        if cap_id is not None and cap_id not in ids:
            errors.append(f"{prefix}: captain must be one of your selected 15.")
        if vice_id is not None and vice_id not in ids:
            errors.append(f"{prefix}: vice-captain must be one of your selected 15.")
        if cap_id is not None and vice_id is not None and cap_id == vice_id:
            errors.append(f"{prefix}: captain and vice-captain must be different.")

        for key, value in zip(keys, values):
            snapshot[key] = value
        snapshot[f"{prefix}_CAPTAIN"] = captain
        snapshot[f"{prefix}_VICE"] = vice

    if errors:
        VX_MANUAL_SELECTION_SAVED = False
        VX_CELL0C_COMPLETE = False
        _status.value = (
            "<div style='margin-top:10px;padding:12px;border-radius:8px;"
            "background:#4b1212;color:#ffd5d5;border:1px solid #ff6f6f'>"
            "❌ <b>NOT SAVED</b><br>" + "<br>".join(errors) +
            "<br><b>Cell 9 remains locked.</b></div>"
        )
        return

    for prefix, cost in saved_costs.items():
        snapshot[f"{prefix}_SELECTED_COST"] = float(cost)
        VX_MANUAL_SELECTED_BUDGETS[prefix] = float(cost)

    VX_MANUAL_SAVED_VALUES = dict(snapshot)
    VX_MANUAL_TRANSFERS.clear()
    VX_MANUAL_TRANSFERS.update(outgoing_changes)
    for prefix, changes in outgoing_changes.items():
        output = VX_NARRATION_OUT_DIR / ('Wildcard' if prefix == 'WC' else 'Free Hit')
        output.mkdir(parents=True, exist_ok=True)
        target = output / 'Manual OUT IN Player Changes.txt'
        details = [f"OUT: {item['out_label']}  ->  IN: {item['in_label']}" for item in changes]
        target.write_text('\n'.join(details or ['No changes from published squad.']) + '\n', encoding='utf-8')
    VX_MANUAL_SELECTION_SAVED = True
    VX_CELL0C_COMPLETE = True

    for w in (list(VX_MANUAL_WIDGETS.values())
              + list(VX_MANUAL_OUT_WIDGETS.values())
              + list(VX_MANUAL_SEARCH_WIDGETS.values())
              + list(VX_MANUAL_SEARCH_CLEAR_BUTTONS.values())):
        w.disabled = True
    _save_btn.disabled = True
    _save_btn.description = "✅ SAVED — CELL 8 COMPLETE"
    _save_btn.icon = "check"
    _saved_budget_bits = []
    if "WC" in saved_costs:
        _saved_budget_bits.append(f"Wildcard £{saved_costs['WC']:.1f}m")
    if "FH" in saved_costs:
        _saved_budget_bits.append(f"Free Hit £{saved_costs['FH']:.1f}m")
    _saved_budget_text = " | ".join(_saved_budget_bits)
    _transfer_text = ' | '.join(f'{prefix}: {len(changes)} changed OUT → IN pairs' for prefix,changes in outgoing_changes.items())
    _status.value = (
        "<div style='margin-top:10px;padding:14px;border-radius:8px;"
        "background:#0b5d2a;color:#eaffef;border:2px solid #36d56d;font-size:16px'>"
        "✅ <b>CELL 8 SAVED SUCCESSFULLY.</b> Manual selections are locked. "
        + (f"<br><b>Saved budget:</b> {_saved_budget_text}" if _saved_budget_text else "") +
        (f"<br><b>OUT → IN comparison:</b> {_transfer_text}" if _transfer_text else "") +
        "<br>You can now run Cell 9.</div>"
    )


_save_btn.on_click(_vx_save_manual_selections)

if _enabled_manual_prefixes:
    display(HTML(
        "<hr><h3>Save before continuing</h3>"
        "<p>Cell 9 is deliberately locked until this button succeeds.</p>"
    ))
    display(_save_btn, _status)
else:
    VX_MANUAL_SELECTION_SAVED = True
    VX_CELL0C_COMPLETE = True
    display(HTML(
        "<hr><div style='padding:14px;border-radius:8px;background:#0b5d2a;"
        "color:#eaffef;border:2px solid #36d56d'>"
        "✅ <b>CELL 8 COMPLETE.</b> No manual selections are required. "
        "You can run Cell 9.</div>"
    ))

print("✅ Manual OUT → IN selectors ready. Pick the IN squad and click SAVE before running Cell 11.")
print("✅ CELL 10 COMPLETE")

✅ Manual OUT → IN selectors ready. Pick the IN squad and click SAVE before running Cell 11.
✅ CELL 10 COMPLETE


In [13]:
#@title 🔴 CELL 11 - SQUAD OPTIMISATION {display-mode: "form"}
# 🔴 CELL 11 - SQUAD OPTIMISATION
"""
CELL 9 — SCORE AUTO + MANUAL SQUADS (WILDCARD & FREE HIT) — STANDALONE

Optimizes only the chip target(s) selected in Cell 0 from VX_POOL (built from the official
FPL API) and scores them with an exact optimal-XI + captain search per
gameweek. No Day 1 notebook or shared runtime required.
"""

from itertools import combinations
import re
import numpy as np
import pandas as pd

# Ensure Cell 11 executes the *saved* canonical optimiser even when Cell 2 was
# not rerun after the notebook was edited in Colab. A reload alone is not enough
# when the source file on disk is stale.
def _vx_sync_saved_wildcard_optimizer():
    import base64
    import gzip
    import hashlib
    import importlib
    import sys
    from pathlib import Path

    expected_sha256 = '10969b65f5ca8aa91a384b508430db047b5df0c757182961287c66afeed9622f'
    compressed_b64 = (
        '...'
        'xLrDAAA='
    )
    canonical_source = gzip.decompress(base64.b64decode(compressed_b64))
    if hashlib.sha256(canonical_source).hexdigest() != expected_sha256:
        raise RuntimeError('Embedded Wildcard optimiser checksum invalid')

    module_root = globals().get('_vx_module_dir')
    if module_root is None:
        media_root = globals().get('VX_MEDIA_WORK')
        if media_root is None:
            raise RuntimeError('Run Cell 0 before Cell 11 to configure the module directory')
        module_root = Path(media_root) / 'wildcard_engine'
    module_root = Path(module_root).resolve()
    module_root.mkdir(parents=True, exist_ok=True)
    module_file = module_root / 'wildcard_slide3.py'
    stale = not module_file.is_file() or hashlib.sha256(module_file.read_bytes()).hexdigest() != expected_sha256
    if stale:
        pending = module_root / '.wildcard_slide3.vx_pending'
        try:
            pending.write_bytes(canonical_source)
            pending.replace(module_file)
        finally:
            pending.unlink(missing_ok=True)
    if str(module_root) not in sys.path:
        sys.path.insert(0,str(module_root))
    for name in ('wildcard_slide3','wildcard_plan'):
        sys.modules.pop(name,None)
        py_file = module_root / (name + '.py')
        Path(importlib.util.cache_from_source(str(py_file))).unlink(missing_ok=True)
    importlib.invalidate_caches()
    loaded_module = importlib.import_module('wildcard_slide3')
    if Path(loaded_module.__file__).resolve() != module_file:
        raise RuntimeError('Optimizer import path does not match canonical module')
    if hashlib.sha256(module_file.read_bytes()).hexdigest() != expected_sha256:
        raise RuntimeError('Optimizer file changed during reload')
    print('✅ Cell 11 synchronized: evidence-aligned goalkeeper eligibility (>=80% start; >=65 expected minutes)' + ('; outdated source replaced' if stale else ''))


_vx_sync_saved_wildcard_optimizer()

VX_CELL1_COMPLETE = False
VX_CELL2_COMPLETE = False
VX_SELECTION_ENGINE_REVISION = None
VX_SELECTION_SIGNATURE = None
if (globals().get("USE_MANUAL_WILDCARD", False) or globals().get("USE_MANUAL_FREEHIT", False)):
    if not globals().get("VX_MANUAL_SELECTION_SAVED", False):
        raise RuntimeError(
            "🔒 Cell 9 is locked. Go back to Cell 8, choose all 15 player dropdowns "
            "for every enabled manual chip, then press the green SAVE MANUAL SELECTIONS button."
        )
    if not globals().get("VX_CELL0C_COMPLETE", False):
        raise RuntimeError("🔒 Cell 8 has not been completed and saved.")


def _vx_best_lineup_local(frame, score_col):
    """Exact optimal-XI search, mirrors Day 1's _vx_phase11_best_lineup."""
    if not isinstance(frame, pd.DataFrame) or frame.empty or score_col not in frame.columns:
        return pd.DataFrame(), pd.DataFrame()
    f = frame.copy()
    f[score_col] = pd.to_numeric(f[score_col], errors="coerce")
    f = f.dropna(subset=["player_id", "position", score_col]).copy()
    groups = {p: f.loc[f["position"].astype(str).str.upper().eq(p)].copy() for p in ("GK", "DEF", "MID", "FWD")}
    if len(groups["GK"]) < 1 or len(groups["DEF"]) < 3 or len(groups["MID"]) < 2 or len(groups["FWD"]) < 1:
        return pd.DataFrame(), pd.DataFrame()
    score_map = f.set_index("player_id")[score_col].to_dict()
    def ids(pos): return groups[pos]["player_id"].astype(int).tolist()
    best_ids, best_score = None, -np.inf
    for gk in ids("GK"):
        for nd in range(3, 6):
            for nm in range(2, 6):
                nf = 10 - nd - nm
                if nf < 1 or nf > 3:
                    continue
                if nd > len(ids("DEF")) or nm > len(ids("MID")) or nf > len(ids("FWD")):
                    continue
                for dc in combinations(ids("DEF"), nd):
                    for mc in combinations(ids("MID"), nm):
                        for fc in combinations(ids("FWD"), nf):
                            chosen = (gk,) + dc + mc + fc
                            sc = float(sum(float(score_map[x]) for x in chosen) + max(float(score_map[x]) for x in chosen))
                            if sc > best_score:
                                best_score, best_ids = sc, set(chosen)
    if not best_ids:
        return pd.DataFrame(), pd.DataFrame()
    xi = f.loc[f["player_id"].isin(best_ids)].copy()
    bench = f.loc[~f["player_id"].isin(best_ids)].copy()
    return xi, bench


_vx_best_lineup_fn = _vx_best_lineup_local


def vx_static_plan_score(squad, gameweeks, gw_scores_source):
    """Per-GW best XI + captain for a static 15-man squad. Mirrors Day 1's
    _vx_phase11_static_plan_score exactly (same output columns)."""
    rows = []
    if not isinstance(squad, pd.DataFrame) or squad.empty:
        return pd.DataFrame()
    ids = set(squad["player_id"].astype(int))
    meta_cols = [c for c in ("player_id", "player_name", "position", "team_id") if c in squad.columns]
    meta = squad[meta_cols].drop_duplicates("player_id")
    for gw in gameweeks:
        scores = gw_scores_source.loc[
            pd.to_numeric(gw_scores_source["gameweek"], errors="coerce").eq(gw),
            ["player_id", "gameweek_expected_points"],
        ].copy()
        tmp = meta.merge(scores, on="player_id", how="left")
        tmp = tmp.loc[tmp["player_id"].isin(ids)].rename(columns={"gameweek_expected_points": "score"})
        if len(gameweeks)==1 and squad.attrs.get("freehit_lineup_ids"):
            lineup=set(squad.attrs["freehit_lineup_ids"])
            xi=tmp[tmp.player_id.isin(lineup)].copy();bench=tmp[~tmp.player_id.isin(lineup)].copy()
        else:
            xi, bench = _vx_best_lineup_fn(tmp, "score")
        if xi.empty:
            continue
        ranked = xi.sort_values('score', ascending=False)
        cap, vice = ranked.iloc[0], ranked.iloc[1]
        bench = pd.concat([bench[bench.position.ne('GK')].sort_values('score',ascending=False), bench[bench.position.eq('GK')]])
        rows.append({
            "gameweek": int(gw),
            "xi": xi.sort_values("score", ascending=False).reset_index(drop=True),
            "bench": bench.reset_index(drop=True) if not bench.empty else bench,
            "xi_xpts": float(pd.to_numeric(xi["score"], errors="coerce").sum()),
            "captain_player_id": int(cap["player_id"]),
            "captain_name": cap.get("player_name"),
            "vice_captain_name": vice.get("player_name"),
            "vice_captain_player_id": int(vice["player_id"]),
            "formation": "-".join(str(int(xi.position.eq(p).sum())) for p in ("DEF","MID","FWD")),
            "captain_incremental_xpts": float(cap["score"]),
            "xi_plus_captain_xpts": float(pd.to_numeric(xi["score"], errors="coerce").sum() + float(cap["score"])),
            "bench_xpts": float(pd.to_numeric(bench["score"], errors="coerce").sum()) if not bench.empty else 0.0,
        })
    return pd.DataFrame(rows)


# _VX_GW_SCORES contains live fixture-by-fixture model projections.

# ------------------------------------------------------------------
# Translate Cell 8's dropdown picks (ipywidgets) into the same
# VX_MANUAL_*_NAMES / *_CAPTAIN / *_VICE_CAPTAIN inputs the resolver
# below already expects — so vx_build_manual_squad() needs no changes.
# A dropdown left on "— none —" becomes an empty string, same as an
# empty text field did before.
# ------------------------------------------------------------------
_VX_WIDGETS = globals().get("VX_MANUAL_WIDGETS", {})
_VX_SAVED_VALUES = globals().get("VX_MANUAL_SAVED_VALUES", {})


def _vx_widget_value_to_player_ref(key):
    """Return a unique FPL player_id string from either the new ID-bearing
    dropdown label or an older saved label like
    'Palmer — £9.7m — 18.84 xPts'.

    The legacy path resolves with name + price + the chip-specific xPts column,
    so duplicate display names never collapse to an ambiguous surname.
    """
    # Manual values are read from the green SAVE snapshot, not live widgets.
    if key in _VX_SAVED_VALUES:
        val = str(_VX_SAVED_VALUES.get(key) or "").strip()
    else:
        dd = _VX_WIDGETS.get(key)
        if dd is None:
            return ""
        val = str(dd.value or "").strip()
    if not val or val == _DROPDOWN_NONE:
        return ""

    # New dropdown format: "Name — ID 123 — £x.xm — y.yy xPts"
    id_match = re.search(r"(?:^| — )ID\s+(\d+)(?: — |$)", val)
    if id_match:
        player_id = int(id_match.group(1))
        if player_id in set(VX_POOL["player_id"].astype(int)):
            return str(player_id)
        raise ValueError(f"{key}: dropdown contains unknown player ID {player_id}.")

    # Backward compatibility with selections already made before this fix.
    parts = [part.strip() for part in val.split(" — ")]
    display_name = parts[0]
    price_match = re.search(r"£\s*([0-9]+(?:\.[0-9]+)?)m", val)
    xpts_match = re.search(r"([0-9]+(?:\.[0-9]+)?)\s*xPts", val, re.IGNORECASE)
    score_col = "xpts_wildcard_horizon" if key.startswith("WC_") else "xpts_freehit_gw"

    candidates = VX_POOL.loc[
        VX_POOL["player_name"].astype(str).str.strip().eq(display_name)
    ].copy()

    if price_match and not candidates.empty:
        wanted_price = float(price_match.group(1))
        candidates = candidates.loc[
            np.isclose(
                pd.to_numeric(candidates["price"], errors="coerce"),
                wanted_price,
                atol=0.001,
                rtol=0.0,
            )
        ]

    if xpts_match and score_col in candidates.columns and not candidates.empty:
        wanted_xpts = float(xpts_match.group(1))
        candidates = candidates.loc[
            np.isclose(
                pd.to_numeric(candidates[score_col], errors="coerce"),
                wanted_xpts,
                atol=0.011,  # dropdown is rounded to 2 decimals
                rtol=0.0,
            )
        ]

    if len(candidates) == 1:
        return str(int(candidates.iloc[0]["player_id"]))

    # Full-name index fallback, only after using every discriminator carried
    # by the old dropdown label.
    try:
        return str(vx_resolve_player_name(display_name))
    except ValueError as exc:
        candidate_ids = (
            candidates["player_id"].astype(int).tolist()
            if not candidates.empty else []
        )
        raise ValueError(
            f"{key}: could not uniquely resolve saved selection {val!r}. "
            f"Filtered candidate IDs: {candidate_ids}. Original error: {exc}"
        ) from exc


VX_MANUAL_WILDCARD_NAMES = [
    _vx_widget_value_to_player_ref(f"WC_{p}{i}")
    for p, n in (("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3))
    for i in range(1, n + 1)
] if globals().get("USE_MANUAL_WILDCARD", False) else []
WC_CAPTAIN = _vx_widget_value_to_player_ref("WC_CAPTAIN") if USE_MANUAL_WILDCARD else ""
WC_VICE_CAPTAIN = _vx_widget_value_to_player_ref("WC_VICE") if USE_MANUAL_WILDCARD else ""

VX_MANUAL_FREEHIT_NAMES = [
    _vx_widget_value_to_player_ref(f"FH_{p}{i}")
    for p, n in (("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3))
    for i in range(1, n + 1)
] if globals().get("USE_MANUAL_FREEHIT", False) else []
FH_CAPTAIN = _vx_widget_value_to_player_ref("FH_CAPTAIN") if USE_MANUAL_FREEHIT else ""
FH_VICE_CAPTAIN = _vx_widget_value_to_player_ref("FH_VICE") if USE_MANUAL_FREEHIT else ""

_wc_filled = sum(1 for n in VX_MANUAL_WILDCARD_NAMES if n.strip())
_fh_filled = sum(1 for n in VX_MANUAL_FREEHIT_NAMES if n.strip())
print(f"Wildcard manual squad: {_wc_filled}/15 selected" if USE_MANUAL_WILDCARD else "Wildcard mode: Auto")
print(f"Free Hit manual squad: {_fh_filled}/15 selected" if USE_MANUAL_FREEHIT else "Free Hit mode: Auto")

if USE_MANUAL_WILDCARD and _wc_filled != 15:
    raise ValueError(f"Wildcard mode is Manual, but only {_wc_filled}/15 players are selected. Go back to Cell 8, choose all 15, then rerun Cell 9.")
if USE_MANUAL_FREEHIT and _fh_filled != 15:
    raise ValueError(f"Free Hit mode is Manual, but only {_fh_filled}/15 players are selected. Go back to Cell 8, choose all 15, then rerun Cell 9.")

# ------------------------------------------------------------------
# Structure-first auto-squad dispatch (dependable six-week MILP for Wildcard;
# activation-week joint optimization for Free Hit). Replaces the old Day-1
# lookup, since this notebook now runs on its own.
# ------------------------------------------------------------------

def _vx_optimize_squad(pool, score_col, budget, max_per_club=None):
    if score_col=='xpts_wildcard_horizon':
        from wildcard_slide3 import build_wildcard_squad,attach_unprojected_goalkeepers
        from wildcard_plan import next_six_gameweeks
        pool=attach_unprojected_goalkeepers(VX_WC_ENGINE,pool)
        weeks=next_six_gameweeks(_bootstrap['events'],VX_WC_ENGINE.now)
        _eligible_ids=set(map(int,pool.player_id))
        recommendation=build_wildcard_squad(_bootstrap['elements'],_bootstrap['teams'],VX_WC_ENGINE.fixtures,
            weeks[0],budget=VX_WC_ENGINE.budget,context={'models':pool.to_dict('records'),'weekly_models':_VX_GW_SCORES.to_dict('records'),
                'currency_multiplier':_bootstrap['game_settings']['ui_currency_multiplier'],
                'completed_gameweeks':sum(bool(event.get('finished')) for event in _bootstrap['events']),
                'available_gameweeks':weeks,'solver_seconds':VX_WC_ENGINE.cfg.solver_seconds,'bench_boost':VX_WC_ENGINE.bb})
        VX_WC_ENGINE.slide3_plan=recommendation
        selected_ids={player['id'] for player in recommendation['squad']}
        return pool[pool.player_id.isin(selected_ids)].copy(),recommendation['total_cost']
    from free_hit_plan import build_free_hit_squad,next_gameweek
    global VX_FREEHIT_SQUAD_PLAN
    _next=next_gameweek(_bootstrap['events'],VX_WC_ENGINE.now)
    _eligible_ids=set(map(int,pool.player_id))
    _news=getattr(VX_WC_ENGINE.news,'events',[])
    _source_stamps=[row['fetched_at'] for row in VX_WC_ENGINE.client.audit if row.get('fetched_at') and row.get('url')==VXWC.API+'bootstrap-static/']
    VX_FREEHIT_SQUAD_PLAN=build_free_hit_squad(
        [player for player in _bootstrap['elements'] if int(player['id']) in _eligible_ids],
        _bootstrap['teams'],VX_WC_ENGINE.fixtures,_next,budget,
        press_conference_data=_news,team_news_data=_news,
        projection_data=_VX_GW_SCORES[_VX_GW_SCORES.gameweek.eq(_next)].to_dict('records'),
        context={'now':VX_WC_ENGINE.now,'news_processed':True,
            'source_snapshot_at':max(_source_stamps,key=VXWC.utc) if _source_stamps else None,
            'currency_multiplier':_bootstrap['game_settings']['ui_currency_multiplier'],
            'completed_gameweeks':sum(bool(event.get('finished')) for event in _bootstrap['events']),
            'solver_seconds':VX_WC_ENGINE.cfg.solver_seconds})
    selected_ids={player['id'] for player in VX_FREEHIT_SQUAD_PLAN['squad']}
    return pool[pool.player_id.isin(selected_ids)].copy(),VX_FREEHIT_SQUAD_PLAN['total_cost']

# ------------------------------------------------------------------
# Selected chip target(s) only. For each selected chip, generate exactly the
# mode chosen in Cell 0: Auto OR Manual, never both unless the user explicitly
# changes the mode and reruns.
# ------------------------------------------------------------------
VX_WILDCARD_AUTO_SQUAD = None
VX_WILDCARD_AUTO_PLAN = pd.DataFrame()
VX_WILDCARD_MANUAL_SQUAD = None
VX_WILDCARD_MANUAL_PLAN = pd.DataFrame()
VX_FREEHIT_AUTO_SQUAD = None
VX_FREEHIT_AUTO_PLAN = pd.DataFrame()
VX_FREEHIT_MANUAL_SQUAD = None
VX_FREEHIT_MANUAL_PLAN = pd.DataFrame()
# Chip-specific working budgets. Auto uses the live model budget. Manual may
# legitimately exceed the £100m starting reference when the manager's team
# value has risen but no authenticated selling-price budget is available.
VX_WILDCARD_BUDGET = float(VX_WC_ENGINE.budget) if VX_GENERATE_WILDCARD else float(VX_BUDGET)
VX_FREEHIT_BUDGET = float(VX_BUDGET)

if VX_GENERATE_WILDCARD:
    if USE_MANUAL_WILDCARD:
        print("🟣 Building saved Manual Wildcard squad...")
        VX_WILDCARD_MANUAL_SQUAD = vx_build_manual_squad(
            VX_MANUAL_WILDCARD_NAMES, WC_CAPTAIN, WC_VICE_CAPTAIN,
            "Wildcard manual squad"
        )
        VX_WILDCARD_BUDGET = float(
            VX_WILDCARD_MANUAL_SQUAD.attrs.get("working_budget", VX_BUDGET)
        )
        VX_WILDCARD_MANUAL_PLAN = VX_WC_ENGINE.score_plan(VX_WILDCARD_MANUAL_SQUAD)
        print(
            f"✅ Wildcard Manual: 15 players | {len(VX_WILDCARD_MANUAL_PLAN)} GWs scored | "
            f"working budget £{VX_WILDCARD_BUDGET:.1f}m"
        )
    else:
        print("⚙️  Optimizing Auto Wildcard squad (horizon xPts)...")
        VX_WILDCARD_AUTO_SQUAD, _wc_cost = _vx_optimize_squad(
            VX_POOL, "xpts_wildcard_horizon", VX_BUDGET
        )
        from wildcard_slide3 import weekly_plan_rows
        VX_WILDCARD_AUTO_PLAN = pd.DataFrame(weekly_plan_rows(
            VX_WC_ENGINE.slide3_plan, VX_WILDCARD_AUTO_SQUAD, VX_WC_ENGINE.scores))
        print(
            f"✅ Wildcard Auto: {len(VX_WILDCARD_AUTO_SQUAD)}/15 players, "
            f"£{_wc_cost:.1f}m of £{VX_BUDGET:.1f}m | "
            f"{len(VX_WILDCARD_AUTO_PLAN)} GWs scored"
        )
else:
    print("⏭️ Wildcard skipped — Cell 0 Generate selector does not include Wildcard.")

if VX_GENERATE_FREEHIT:
    if USE_MANUAL_FREEHIT:
        print("🟢 Building saved Manual Free Hit squad...")
        VX_FREEHIT_MANUAL_SQUAD = vx_build_manual_squad(
            VX_MANUAL_FREEHIT_NAMES, FH_CAPTAIN, FH_VICE_CAPTAIN,
            "Free Hit manual squad"
        )
        VX_FREEHIT_BUDGET = float(
            VX_FREEHIT_MANUAL_SQUAD.attrs.get("working_budget", VX_BUDGET)
        )
        VX_FREEHIT_MANUAL_PLAN = vx_static_plan_score(
            VX_FREEHIT_MANUAL_SQUAD, [VX_CURRENT_GW], _VX_GW_SCORES
        )
        print(
            f"✅ Free Hit Manual: 15 players | {len(VX_FREEHIT_MANUAL_PLAN)} GW scored | "
            f"working budget £{VX_FREEHIT_BUDGET:.1f}m"
        )
    else:
        print("⚙️  Optimizing Auto Free Hit squad (single-GW xPts)...")
        VX_FREEHIT_AUTO_SQUAD, _fh_cost = _vx_optimize_squad(
            VX_POOL, "xpts_freehit_gw", VX_BUDGET
        )
        from wildcard_slide3 import weekly_plan_rows
        VX_FREEHIT_AUTO_PLAN = pd.DataFrame(weekly_plan_rows(
            VX_FREEHIT_SQUAD_PLAN, VX_FREEHIT_AUTO_SQUAD, _VX_GW_SCORES))
        print(
            f"✅ Free Hit Auto: {len(VX_FREEHIT_AUTO_SQUAD)}/15 players, "
            f"£{_fh_cost:.1f}m of £{VX_BUDGET:.1f}m | "
            f"{len(VX_FREEHIT_AUTO_PLAN)} GW scored"
        )
else:
    print("⏭️ Free Hit skipped — Cell 0 Generate selector does not include Free Hit.")

# Detailed, data-driven selection report and closest affordable replacements.
VX_REPORT_LINES = ['Automatic Wildcard and Free Hit report', json.dumps(VX_MODEL_AUDIT, indent=2)]
VX_SOLVER_AUDIT = {}
_VX_REPORT_TARGETS = []
if VX_GENERATE_WILDCARD:
    if USE_MANUAL_WILDCARD:
        _VX_REPORT_TARGETS.append(('Wildcard Manual', VX_WILDCARD_MANUAL_SQUAD, VX_WILDCARD_MANUAL_PLAN, VX_WILDCARD_BUDGET))
    else:
        _VX_REPORT_TARGETS.append(('Wildcard Auto', VX_WILDCARD_AUTO_SQUAD, VX_WILDCARD_AUTO_PLAN, VX_WILDCARD_BUDGET))
if VX_GENERATE_FREEHIT:
    if USE_MANUAL_FREEHIT:
        _VX_REPORT_TARGETS.append(('Free Hit Manual', VX_FREEHIT_MANUAL_SQUAD, VX_FREEHIT_MANUAL_PLAN, VX_FREEHIT_BUDGET))
    else:
        _VX_REPORT_TARGETS.append(('Free Hit Auto', VX_FREEHIT_AUTO_SQUAD, VX_FREEHIT_AUTO_PLAN, VX_FREEHIT_BUDGET))
for label, squad, plan, working_budget in _VX_REPORT_TARGETS:
    if label.startswith('Wildcard'):
        VX_SOLVER_AUDIT[label]=squad.attrs.get('solver',{})
        squad.to_csv(VX_INPUT_ROOT/(label.lower().replace(' ','_')+'_squad.csv'),index=False)
        VX_REPORT_LINES.append(f"{label}: £{squad.price.sum():.1f}m / £{working_budget:.1f}m; {VX_WC_ENGINE.budget_source}")
        VX_REPORT_LINES.append('FDR: '+VX_WC_ENGINE.fdr.source)
        for pid in squad.player_id.astype(int):
            card=VX_WC_ENGINE.player_card(pid)
            VX_REPORT_LINES.append(card['name']+': '+card['reason']+' Latest verified: '+card['latest_match_text'])
        continue
    VX_SOLVER_AUDIT[label] = squad.attrs.get('solver', {})
    squad.to_csv(VX_INPUT_ROOT / (label.lower().replace(' ','_')+'_squad.csv'), index=False)
    VX_REPORT_LINES.append(f'\n{label}: £{squad.price.sum():.1f}m / £{working_budget:.1f}m working budget; solver {squad.attrs.get("solver")}')
    for _, pr in plan.iterrows():
        gw = int(pr.gameweek)
        VX_REPORT_LINES.append(f'GW{gw}: {pr.formation}; XI {pr.xi_xpts:.2f}; with captain {pr.xi_plus_captain_xpts:.2f}; captain {pr.captain_name}; vice {pr.vice_captain_name}')
        xi_ids=set(pr.xi.player_id.astype(int))
        for pid in squad.player_id.astype(int):
            card=VX_WC_ENGINE.player_card(pid); role='XI' if pid in xi_ids else 'bench'; current=card['weeks'][0]
            fixture_text=', '.join(f["opponent"]+(' (H)' if f['home'] else ' (A)') for f in current['fixtures']) or 'BLANK'
            fdr_text='/'.join(f"{float(f['fdr']):.1f}" for f in current['fixtures']) or '—'
            VX_REPORT_LINES.append(
                f"{role} {card['position']} {card['name']} £{card['price']:.1f}: {card['expected_points']:.2f} xPts; "
                f"{fixture_text}; FDR {fdr_text}; xMins {card['expected_minutes']:.1f}; start {card['starting_probability']:.0%}; "
                f"availability {card['injury_status']}; latest verified: {card['latest_match_text']}"
            )
        VX_REPORT_LINES.append('Bench order: '+', '.join(pr.bench.player_name))
    score_col = 'xpts_wildcard_horizon' if label.startswith('Wildcard') else 'xpts_freehit_gw'
    for _, r in squad.iterrows():
        rest = squad[squad.player_id.ne(r.player_id)]
        candidates = VX_POOL[~VX_POOL.player_id.isin(squad.player_id) & VX_POOL.position.eq(r.position)].copy()
        candidates = candidates[candidates.price.le(working_budget-rest.price.sum()+1e-8)]
        counts = rest.team_id.value_counts()
        candidates = candidates[candidates.team_id.map(counts).fillna(0).lt(_bootstrap['game_settings']['squad_team_limit'])]
        if not candidates.empty:
            alt = candidates.sort_values(score_col, ascending=False).iloc[0]
            # Manual squads intentionally contain only squad metadata, captain flags,
            # and player IDs; the horizon/single-GW ranking columns live in VX_POOL.
            # Always look up the selected player's comparison score by unique FPL ID
            # instead of assuming the squad frame carries score_col.
            _selected_score_rows = VX_POOL.loc[VX_POOL.player_id.eq(int(r.player_id)), score_col]
            if _selected_score_rows.empty:
                _selected_score = float("nan")
            else:
                _selected_score = float(_selected_score_rows.iloc[0])
            _alt_score = float(alt[score_col])
            _delta_score = _alt_score - _selected_score if np.isfinite(_selected_score) else float("nan")
            _delta_text = f"{_delta_score:+.2f}" if np.isfinite(_delta_score) else "n/a"
            VX_REPORT_LINES.append(
                f'Alternative {r.player_name} → {alt.player_name}: '
                f'Δprice £{alt.price-r.price:+.1f}, Δplayer xPts {_delta_text}; '
                'joint XI/captain objective determines selection.'
            )
VX_REPORT_PATH = VX_INPUT_ROOT / 'selection_report.txt'
VX_REPORT_PATH.write_text('\n'.join(VX_REPORT_LINES))
(VX_INPUT_ROOT / 'solver_audit.json').write_text(json.dumps(VX_SOLVER_AUDIT,indent=2))
print(f'✅ Selection report saved: {VX_REPORT_PATH} ({len(VX_REPORT_LINES)} lines)')
VX_SELECTION_ENGINE_REVISION = globals().get('VX_LIVE_DATA_REVISION')
VX_SELECTION_SIGNATURE = (bool(VX_GENERATE_WILDCARD), bool(VX_GENERATE_FREEHIT), bool(USE_MANUAL_WILDCARD), bool(USE_MANUAL_FREEHIT))
VX_CELL1_COMPLETE = True
print('✅ Cell 9 complete — selected chip target(s) scored successfully.')
print("✅ CELL 11 COMPLETE")

✅ Cell 11 synchronized: evidence-aligned goalkeeper eligibility (>=80% start; >=65 expected minutes); outdated source replaced
Wildcard mode: Auto
Free Hit mode: Auto
⚙️  Optimizing Auto Wildcard squad (horizon xPts)...


/content/vx_wc_fh_work/final_v2/wildcard_engine/wildcard_slide3.py:417: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pool=pd.concat([pool,additions],ignore_index=True)
/content/vx_wc_fh_work/final_v2/wildcard_engine/wildcard_slide3.py:421: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  engine.pool=pd.concat([engine.pool,new_players],ignore_index=True)


✅ Wildcard Auto: 15/15 players, £99.5m of £100.0m | 6 GWs scored
⚙️  Optimizing Auto Free Hit squad (single-GW xPts)...
✅ Free Hit Auto: 15/15 players, £99.5m of £100.0m | 1 GW scored
✅ Selection report saved: /content/drive/MyDrive/FPL_VORTEX/Analysis/wc_fh/selection_report.txt (52 lines)
✅ Cell 9 complete — selected chip target(s) scored successfully.
✅ CELL 11 COMPLETE


In [14]:
#@title 🔴 CELL 12 - SCENES & RYAN NARRATION {display-mode: "form"}
# 🔴 CELL 12 - SCENES & RYAN NARRATION
"""CELL 10 — Canonical Free Hit one-Gameweek contract plus finalized Wildcard Scene-JSON narration path.

Shared facts come from VX_WC_ENGINE. Strategy remains separate:
- Wildcard: six-Gameweek sustainable squad and transfer path.
- Free Hit: activation Gameweek only, maximum realistic XI/captain score, cheap playable bench, no future-transfer plan.
"""
import re, json, math, hashlib, unicodedata
import numpy as np
import pandas as pd

# Restore the saved Free Hit source even when Cell 2 was not rerun in this runtime.
def _vx_sync_saved_freehit_modules():
    import base64
    import gzip
    import hashlib
    import importlib
    import sys
    from pathlib import Path

    module_root = globals().get('_vx_module_dir')
    if module_root is None:
        raise RuntimeError('Cell 12 requires Cell 2 to configure the verified module directory.')
    module_root = Path(module_root).resolve()
    if not module_root.is_dir():
        raise RuntimeError(f'Free Hit module directory is missing: {module_root}. Run Cell 2.')

    sources = {'freehit_analyst.py': ('a5a23d2aa44d9310aaafee5d204acd5a82063c62f662ccd0c0bebe730f0ac1d2', 'H4sIAAAAAAAC/41Y3Y7cthW+n6dgpxeSYHlgx+jNTFTAQHYdA6kTrF0nxUAQKImapVeiVJKan24GyEP0CfskPYc/Gkk7u8jNLimef57zncNZLpe3kjHyI9ekpJoSJiQv7hsm9Ir8IpmCBdW8FYSKkggqpd3VfM8IF6QC5nuus06yhvfNarlcLnjTtVKTou1Oi0q2DTnwuiyoLDP1756WGRW0PimuiCMUfZMzGVvNWVdTUKacIZZlJqbiR91L9kSQY/HHBVXMcs6s9PRfb+4+f/z5U+wZFNuh4yq2dg5bLrRsrbiSVeBY01HJVSvCMloviEzK1Y7pMPBqVMEgUhxYBsogIq0kj+cNTaQlLnopQXp24BpMBs5MSypUxaQK4sdztMk9pRNrvx4Ye1DJttwGO9ow3AVpuiAP7KSSMGjaktUZL4PYLfcgDu4L9p1sv7ECLy9TgnbqvgWJgSpaycUuyynE0VDxgg27vC/BAreNFoRXkAanULSaUGPbg3HLr/+S5P5rBZ8fMEHQsGgtGQRYkE+tYFaK9WyqHfgDpanU+OnIIRN6BVHvNOUiE2AHBEEFF4VXXQIhEBvV9hL88F8zqoP0iREXN4IGUmnHpEtQCBqvOCvtpY3IhhsCs+qir6m+QlSzHa1NGl8RlL9A81yU/EWbCJkFSstfOrQlFToB9hqDCCjcQTk+yHSraQ3Hf4Yznx1MjTaSID0hIzEDFGZASOMcy4TI9qASZfnR0vqUAcD0NVxqvE0hvdDnmokQCUHjW7QH19s36dTZwVc4mRpAjt7QCeM8m44dKI2iDWTUVXqs4qJV6KA160gAZlAF2gSns+33b2Zm2ECsaNcxUYbH10ATzZPOZ5PRNIDZtTzIVejDv52HLgWjt9cdTKPXWAs1L9k7k2xAHHgKWn7rFeSvJ/376s3bqWa3KXmhQ4dWifULVMYOlfyXt2m8A7GX7esLKThasraqErp9zmm4foOtY+wO8U8Mdp+g4uKiFZodNWaSgTYEQbSsidGBbGgO9wAp/2lF0ribx0MPzLtDEMWHImM13/G8ZskX2TMLVg2mqtOxDazSzCoyAGvDnVxpMVMzbbSsnHgiJrH/IOusrG3gvc801zUL0qRa3v7yE/n6892Xm9/I//74L7m9u7khP378Qm4//vbln3c35P2n9z/96/PHz+R38uFX8oiqtyMMwGQ4LwcFNc1ZDXKDW2sv+YFXFQfg0qc1+QQWkg++ogYeuJ8dywbOD78Gr5SW4TVN0StAY1GfgplHemg7aeJ67IL8lbwnHutIDsn4QAC/95CbCooC5LDXKJm4nkDaDnt0L7g+xYTtmSD6vu139wSaALn94Q4rsBeQM1ywcgV48+dGBFQoChZrRptM4WyjNC8UQBeViIW8VMmjhDzFc2ikqckOidnhnbSUEILzwjoCqSifkiGgADvU70QcAgBQXdTZyQphAVrrwbV8az6AgZF7MHJt6WOtpAYWrG5EVnZEL8QuGe5h34LkIN0U9y2McAaQLxZaxleeCyU8MRPIHo/zMBxNkRiR5zUmORc9M9yaNQkOfKuSsQ4XoYw2+HXVdzBXstBfiL+AxC9CpIq94aMEi+Z3lMz2hnNkoocIG8IWM51Dlx1iiU0m8oUKIYlcdDxKo7jIxQLbkD2NkuTdOpeMPmALm8U3sTRD/rsBeQ+J5ylqMDaUDCuClV6m7QWm182TClrb37CrjA+9MDh7t5aUK0a+0rpnN1K2MgyGCd65SoasF4xBglU4rFsFpKj73Kacvkc2JzrCAvUoARDwAHmhwN0GMB0uvWQYIWgApK2IjSBwUg4wwWuo0BX5B1cKy7YA0RxvHJSAqJKrogaXyxhsgRiQiuaAoDg4rRZDOB1OFi1QwBwGUds+Dve6hvEbu/PopqOYBOhIsDbfzTKN8eoICTzIZBdTgrUbK9zcaFPAJQSIGsb0zDkFaPwCS5J8d7bKKjuhXEWH9OKfdyw7UCkwsuhhtXwcWX9eE7u9Zn56vkCni/7ldPmsKU/Caqr8WSXffzcyeXjqQXNqoF8K059mDyWXoNPCU6yGoZy5156N2FW5wIEdhgSrby0XoXp6eBkiXzJrJP2F/uOmGUtphw3bzMMyzttWQ5OjXWx7dDx5DyQ4DsVP8CQJIxxGuhnuzSeuaNNtAygeHD7SBNYCXDMbPPB4Z4hNWsAdYAe7DNEF+o9STThTRGtD4GHLThyxfxjizkudqIsHw4vpC3QKkOa2umT8/A47O9bYaBjl0SVmQJ6feJlgu8DiXB8vvaIbzD5v9uZl4ObY4+QFB8nSQQqYmXzKfHkOmoLqzKRrJ9ZE9U2IQiMzI9d1uMeZADupmcxRzh7lWBq4VuYGWxiB/dTfDQNxkD5r0qZ7dmxOxha9AiK0xfwbWYJoO6abHI7sAtY5GR68hPg+QOTyFsZ6QUSrmWaI690LZYGeXa306S8gYRdt5nmddAuTlX3TUHkybRmnwskL/kqMNtd4XgjuM0du3nvmhx3YAqBfnjBuP650eG/8H/GXadH8EgAA'), 'freehit_premium.py': ('...'), 'freehit_personal_intro.py': ('52312b263d963a41ed1392d858a8daeb534e40d40651e2a16d3890f16fbd64e5', 'H4sIAAAAAAAC/6VY3XPiOBJ/56/QcQ+COkPd1dW9ZMs7xSZk4poEpiZMZq9YyiVsAZoYyyPJEDaT//269WE+FmZr6/JgbKnVX+r+dXfa7fZHrrQsWUFuFefkThgiK16KcvkTkSXvvWdrvuX8mfCNyHmZ8YiUkrDayDUzIiPZSlRkw1UuMtNvt9uthZJrkqaL2tSKpykR60oqQ1hZSgMnZKlbfimT1S68K+4Oml0FosOhQblzy1tR5BlTeSpKo2SaydLwF0OawwvF9Sost1qtp+Gnx2Q8iils8JUwvXxXsrXIevZ8b/OfnhYvvazmmrZaOV8QzQuemVSv5DZjmneQlWKZuUK7ptqoCHSZdXs/N99HG7OrFoG/qmA7cGccTk+p/laznM7sbsYqw0SZilzHrwckBTj23ymcLelsSj0Vvgo4GV0k3IiMpyfUb1aQNkwZcGMa9Jm6F7KQyutIRBm0JWLhX/tLbjpU6LRh8CJo1ykPRHCBf2B9pZjQnDyxouZDpaTq0CaOgjPhfr7VAm6IMLJhYAQzPG84kV8T2rUimDEse76s76nsveJTWkktMLjoDCk79CG5oRG9/XJDuxB6eUOHTrKGANXBfTgbIRB4mf+/CsQxvRne0iO3NaYRYIgLjai/6MCyYWXNCmyIS54VP+PWv5NbzjAZ7X6u2MKQOd9JOC6MDl6w7DCkIkg2s5I1KF0YrpAVnBMojxU7SHmACpctYG3fSlAc2JfklS7ECwqiV2v20mlsjp75Li7Yep4z766r4DX+UgEjnqeVhMzUdNaNSJMBlkvjqL/E5c1l9bwG3Eh/kNDRXEoDH6w6Xv8+AuyL8XGY8rjjEt2622NLah2jGzTK5LpiSgCohggIsl1+IWLS7t8abKIYE8ckK6nE77JMlx58NdJP90gQ1ikAzw/ix4JdXtubIizLeAX3DTaRBtRlWex8mHjwjE/AtHHc3lWo7+tbFJTU4nce/yswQdt5Hu+d0DDo/hRKSHxsbfBjzjOhQdU00EHuWlGWNdQYsRDAGvQoOkGQDdtjdg0bUVa1SVdMrzwIBL6OTstaAYICGjFPFB8A9zkuszNcwlc6hyjLU18JaddDGvh+A3DnlA4meKWdBXibCOA///MMd4zwVNcVBhbPwQyhyUTV/AzpSkKon5Ai0iC5UwYpzijy52wuSDyn3LFE742YDq4nydNgMqSYDsEphBcQufRufH9j11G0Xxt/HI6oxxYGQRRD4HWC9Oklp8+6R+whdg5YDpoIOu5r9pFKGN4EJDLiZFYrxUvAYCzgADi1RlUKobHpwYglECilXiAyWd8wAg0R5DbgJWZ4b4MZSUIYAQ0AcMl5zvM+PQQGyLEpnUNNhyz36GJ3SzCJGYnFiP5yPxh9IO8HD8Mvw+EHKGzj0bCHr+R6DK2OrzW8OGSZy3pe8Ms8b8aff7kfktvk18nnT8NHYPqQjD5Pho/kYTCZHDDV/AKHyV3ySKwW49H9f4HB9V3ykTwN7j8PQ/UzfK1JTKYNgw79Bk2XLZQRWbQn4K9ndBmbY8WpNb7vIDHJAYjt4er1HAS+vSN3HIqOhtvekoQUUj5DlQTh/XY3OpDsXQOC27cQG2sodXjTQSEM3+2KwxLUOgKaQ1wztYNr1xxQHUovALySG6yjSIrCNdCxNeGlrJcrYiT5WgOvxY5o6KFzXzttPJzo0rBHbQZ7a7UsNrZRwhi1NoNIuMd1nyQLst75eMRA0qRg2rh4Y+WSQxwmNG/MZyQXCwhPDGJWAReWrU6UsL0AKpBQsAqLpUERjTKuV8AeyFocrgEuTmbuqsBrvuaD9FCGyVqUtcEVlwMW7VxNz3YnKrjWwelQFMiiwGYEtKhE9qzt2XAp8Nj9wMHkYQcpL5YChxnnJscdUAmHhaD+6U1gGgsfke1bobSJSMEN3G1w5eF9B3MbLi7SjVhD+kHPMEUk6mi+XIPjobWMEamMMLsIRhu+/yqZUnYcSm3V1ZV85mXkOho7IUxnke3lbBcSgbHuBQuuXZ7SAHQp/UdgO8N9pJ0CFuLFNfKcAMwmm5RO60aJmBLa/wrdUwd3p/RYOyh8lhdsIYNgazfAmOJ9zZnKVh3V/m3eeXe15d8hh79vOcQVPpV95t3f5u294REcS7o/6l/8IErWcOeADRycD7fTq+zEulfeNzChW44vzXJdX1B0XZgYh89+znmFL53zDQmeptHrmy/mOHjFF8axA9b9usIJp3MwRPnGNEbaKYXvlwrb1ChMIEfbByMdZO3X4842MjBFF2cPpCzH5ABizx3ubO1u19E13w2bTGrjN/cLsDuHauTX8RUK6xn7MKnS5TY+h8mR64lhlFxu9YXO1dMYLN9YImM/tO/PQuAWPKa3n4ZDcpdMrsho/IWMP5Evg2Tyjkahu/C/UdOyFWzOC5zCrhNkSGw3gYEaTsSuwXCdQTjuf1PfcbifSGccAk3g/xxCnxCH1i06+m9E7H/96kGp3L/6Pew54xCffi18pkpC2Y4PhikK9Xhw/SEZvScfk2ss/82EhJPmcPSYPA3d1tshrgRICi+nmLPPRKfBVioYlmRdmrjgZafZ7uuqEKbT7XYPpz0XDK3/AQOPwH9CEgAA')}
    changed = []
    for basename, (expected_hash, compressed) in sources.items():
        source = gzip.decompress(base64.b64decode(compressed))
        if hashlib.sha256(source).hexdigest() != expected_hash:
            raise RuntimeError(f'Embedded Free Hit module checksum mismatch: {basename}')
        target = module_root / basename
        if not target.exists() or hashlib.sha256(target.read_bytes()).hexdigest() != expected_hash:
            target.write_bytes(source)
            changed.append(basename)
        cache_path = importlib.util.cache_from_source(str(target))
        Path(cache_path).unlink(missing_ok=True)
    if str(module_root) in sys.path:
        sys.path.remove(str(module_root))
    sys.path.insert(0, str(module_root))
    for name in ('freehit_analyst', 'freehit_premium', 'freehit_personal_intro'):
        sys.modules.pop(name, None)
    importlib.invalidate_caches()
    for basename, (expected_hash, _) in sources.items():
        module_name = basename.removesuffix('.py')
        module = importlib.import_module(module_name)
        if Path(module.__file__).resolve() != module_root / basename:
            raise RuntimeError(f'Wrong {module_name} import directory: {module.__file__}')
        if hashlib.sha256((module_root / basename).read_bytes()).hexdigest() != expected_hash:
            raise RuntimeError(f'Free Hit source changed unexpectedly: {basename}')
    print('✅ Cell 12 Free Hit modules synchronized with saved source' +
          (' (outdated copies replaced: ' + ', '.join(changed) + ')' if changed else ''))

if VX_GENERATE_FREEHIT:
    _vx_sync_saved_freehit_modules()


# Reject missing/stale selections before scene generation, narration or partial JSON writes.
VX_CELL2_COMPLETE = False

def vx_require_selected_squads():
    import pandas as pd

    expected_modes = (
        bool(VX_GENERATE_WILDCARD), bool(VX_GENERATE_FREEHIT),
        bool(USE_MANUAL_WILDCARD), bool(USE_MANUAL_FREEHIT),
    )
    if not globals().get('VX_CELL1_COMPLETE', False):
        raise RuntimeError(
            'Cell 12 cannot build scenes: Cell 11 has not successfully finished '
            'the squad optimisation. Run Cell 11 first and resolve any errors there, '
            'then rerun Cell 12. Do not reuse a previous squad.'
        )
    source_revision = globals().get('VX_LIVE_DATA_REVISION')
    if not source_revision or globals().get('VX_SELECTION_ENGINE_REVISION') != source_revision:
        raise RuntimeError(
            'Cell 12 cannot use an old squad after Cell 8 refreshed the player model. '
            'Run Cell 11 again after Cell 8, then rerun Cell 12.'
        )
    if globals().get('VX_SELECTION_SIGNATURE') != expected_modes:
        raise RuntimeError(
            'Chip or manual/auto mode changed since Cell 11. '
            'Rerun Cell 11 with the current Cell 0 settings, then Cell 12.'
        )
    if not any(expected_modes[:2]):
        raise RuntimeError('No chip is selected for generation in Cell 0.')
    if globals().get('VX_WC_ENGINE') is None:
        raise RuntimeError('Cell 8 has no loaded Wildcard engine; rerun Cell 8, 11 and 12.')

    for label, squad_name, plan_name in (
        ('Wildcard', 'VX_WILDCARD_MANUAL_SQUAD' if USE_MANUAL_WILDCARD else 'VX_WILDCARD_AUTO_SQUAD',
         'VX_WILDCARD_MANUAL_PLAN' if USE_MANUAL_WILDCARD else 'VX_WILDCARD_AUTO_PLAN'),
        ('Free Hit', 'VX_FREEHIT_MANUAL_SQUAD' if USE_MANUAL_FREEHIT else 'VX_FREEHIT_AUTO_SQUAD',
         'VX_FREEHIT_MANUAL_PLAN' if USE_MANUAL_FREEHIT else 'VX_FREEHIT_AUTO_PLAN'),
    ):
        enabled = VX_GENERATE_WILDCARD if label == 'Wildcard' else VX_GENERATE_FREEHIT
        if not enabled:
            continue
        squad = globals().get(squad_name)
        plan = globals().get(plan_name)
        if not isinstance(squad, pd.DataFrame) or squad.empty:
            raise RuntimeError(
                f'{label} selected squad ({squad_name}) is missing. '
                'Cell 11 must finish successfully before Cell 12. Rerun Cell 11.'
            )
        required_columns = {'player_id', 'position', 'team_id', 'price'}
        missing_columns = required_columns.difference(squad.columns)
        if missing_columns:
            raise RuntimeError(f'{label} squad has missing fields: {sorted(missing_columns)}. Rerun Cells 8 and 11.')
        player_ids = pd.to_numeric(squad['player_id'], errors='coerce')
        if len(squad) != 15 or player_ids.isna().any() or player_ids.nunique() != 15:
            raise RuntimeError(
                f'{label} squad must contain exactly 15 unique player IDs; '
                f'got {len(squad)} rows / {player_ids.nunique()} unique IDs. '
                'Inspect the Cell 11 solver before generating scenes.'
            )
        positions = squad['position'].astype(str).str.upper().value_counts().to_dict()
        required_positions = {'GK': 2, 'DEF': 5, 'MID': 5, 'FWD': 3}
        if positions != required_positions:
            raise RuntimeError(f'{label} illegal squad position counts: {positions}; expected {required_positions}.')
        if not isinstance(plan, pd.DataFrame) or plan.empty or 'gameweek' not in plan.columns:
            raise RuntimeError(f'{label} plan ({plan_name}) is incomplete. Rerun Cell 11.')
        weeks = pd.to_numeric(plan['gameweek'], errors='coerce')
        if weeks.isna().any() or weeks.nunique() != len(plan):
            raise RuntimeError(f'{label} plan has invalid or repeated Gameweeks. Rerun Cell 11.')
        if label == 'Wildcard' and len(plan) != len(VX_WC_ENGINE.primary):
            raise RuntimeError(
                f'Wildcard has {len(plan)} Gameweek plans but the current model has '
                f'{len(VX_WC_ENGINE.primary)}; rerun Cell 11 after Cell 8.'
            )
        if label == 'Free Hit' and len(plan) != 1:
            raise RuntimeError('Free Hit must contain exactly one Gameweek plan. Rerun Cell 11.')
    return True

vx_require_selected_squads()


VX_STORY_VERSION='contract-human-analyst-v5-2026-10-05'
VX_FREEHIT_STORY_VERSION='freehit-premium-four-slides-v1'
VX_PRESENTATION_VERSION='premium-blueprint-dense-uppercase-v4-2026-10-08'
VX_DIFFERENTIAL_THRESHOLD=None  # differential status is relative to viable positional peers, never ownership-only


def vx_say_number(value):
    number=round(float(value),1);whole=int(number)
    small=['zero','one','two','three','four','five','six','seven','eight','nine','ten','eleven','twelve','thirteen','fourteen','fifteen','sixteen','seventeen','eighteen','nineteen']
    tens=['','','twenty','thirty','forty','fifty','sixty','seventy','eighty','ninety']
    def say(n):
        if n<0:return 'minus '+say(-n)
        if n<20:return small[n]
        if n<100:return tens[n//10]+(' '+small[n%10] if n%10 else '')
        if n<1000:return small[n//100]+' hundred'+(' '+say(n%100) if n%100 else '')
        return str(n)
    decimal=round((number-whole)*10)
    return say(whole)+(' point '+small[abs(decimal)] if decimal else '')


def vx_club_name(code):
    return next((str(t['name']) for t in _bootstrap['teams'] if str(t['short_name']).casefold()==str(code).casefold()),str(code))


def vx_say_fixture(text):
    def expand(m):
        return vx_club_name(m.group(1))+(' at home' if m.group(2).upper()=='H' else ' away')
    return re.sub(r'\b([A-Za-z]{2,4})\s*\(([HAha])\)',expand,str(text)).replace('BLANK','no scheduled fixture')


def vx_player_focus(scene,text=None):
    """Estimate player-card focus from first narrated name mention."""
    text=scene['narration_text'] if text is None else text
    words=list(re.finditer(r'\S+',text));hits=[]
    for p in scene.get('players',[]):
        found=re.search(r'(?<!\w)'+re.escape(p['name'])+r'(?!\w)',text,re.I)
        if found:
            index=next((i for i,w in enumerate(words) if w.end()>found.start()),0)
            hits.append((index,p['id']))
    hits.sort();count=len(scene.get('players',[]))
    if len(hits)!=count or len({pid for _,pid in hits})!=count:
        return [{'player_id':p['id'],'start_ratio':i/max(1,count),'method':'equal fallback: name mention unavailable'} for i,p in enumerate(scene.get('players',[]))]
    return [{'player_id':pid,'start_ratio':0. if i==0 else index/max(1,len(words)),'method':'first verified name mention'} for i,(index,pid) in enumerate(hits)]


def _fh_base_validate(d):
    assert d['chip']=='freehit'
    assert len(d['squad'])==15 and len({p['id'] for p in d['squad']})==15,'Free Hit needs 15 unique players'
    counts={pos:sum(p['position']==pos for p in d['squad']) for pos in ['GK','DEF','MID','FWD']}
    assert counts=={'GK':2,'DEF':5,'MID':5,'FWD':3},f'Illegal Free Hit position counts: {counts}'
    assert max(sum(p['club']==club for p in d['squad']) for club in {p['club'] for p in d['squad']})<=3,'More than three players from a club'
    assert d['money_spent']<=d['budget_total']+.001,'Free Hit exceeds working budget'
    assert abs(d['money_remaining']-(d['budget_total']-d['money_spent']))<.011,'Free Hit bank mismatch'
    xi=[p for p in d['squad'] if not p['is_bench']]
    assert len(xi)==11,'Starting XI must contain 11 players'
    formation={p:sum(q['position']==p for q in xi) for p in ['GK','DEF','MID','FWD']}
    assert formation['GK']==1 and 3<=formation['DEF']<=5 and 2<=formation['MID']<=5 and 1<=formation['FWD']<=3,'Illegal Free Hit formation'
    assert sum(p['is_captain'] for p in xi)==1 and sum(p['is_vice_captain'] for p in xi)==1,'Captain and vice must be in XI'
    assert len(d['bench_ids'])==4 and len(set(d['bench_ids']))==4,'Bench order must contain four unique players'
    bench_gk=[p for p in d['squad'] if p['id'] in d['bench_ids'] and p['position']=='GK']
    assert len(bench_gk)==1 and bench_gk[0]['is_bench'],'Bench goalkeeper mismatch'
    assert d['horizon_gameweeks']==[d['gameweek']],'Free Hit horizon must be activation Gameweek only'
    assert 'transfer_plan' not in d and 'future_transfer_plan' not in d,'Free Hit must not contain a future transfer plan'
    return True


def _fh_fixture_rows(gw,selected_counts):
    rows=[]
    teams=VX_WC_ENGINE.teams
    for tid,team in teams.items():
        attacks=VX_WC_ENGINE.fdr.for_team(int(tid),gw,'MID')
        defences=VX_WC_ENGINE.fdr.for_team(int(tid),gw,'DEF')
        defence_by_id={int(f['fixture_id']):f for f in defences}
        for attack in attacks:
            defence=defence_by_id.get(int(attack['fixture_id']),attack)
            opp=int(attack['opponent_id']);opp_stats=VX_WC_ENGINE.team_stats.get(opp,{})
            rows.append({
                'team_id':int(tid),'club':str(team['short_name']),'club_name':str(team['name']),
                'fixture_id':int(attack['fixture_id']),'opponent_id':opp,'opponent':str(attack['opponent']),
                'home':bool(attack['home']),'fixture':str(attack['opponent'])+(' (H)' if attack['home'] else ' (A)'),
                'attack_fdr':float(attack['fdr']),'defence_fdr':float(defence['fdr']),
                'expected_team_goals':None if attack.get('team_goals') is None else float(attack['team_goals']),
                'clean_sheet_probability':None if defence.get('clean_sheet_probability') is None else float(defence['clean_sheet_probability']),
                'opponent_attack_strength':None if not opp_stats else float(opp_stats['attack']),
                'opponent_defensive_concede_strength':None if not opp_stats else float(opp_stats['concede']),
                'selected_count':int(selected_counts.get(int(tid),0)),
                'fdr_source':VX_WC_ENGINE.fdr.source,
            })
    return sorted(rows,key=lambda r:(min(r['attack_fdr'],r['defence_fdr']),r['club'],r['fixture_id']))


def _fh_status_available(card):
    label=str(card.get('injury_status','')).upper()
    return not any(token in label for token in ('INJURED','SUSPENDED','UNAVAILABLE'))


def _fh_player_object(pid,selected_ids,xi_ids,bench_ids,cap,vice):
    card=VX_WC_ENGINE.player_card(int(pid));current=card['weeks'][0]
    fixtures=[]
    for f in current['fixtures']:
        opp=int(f['opponent_id']);opp_stats=VX_WC_ENGINE.team_stats.get(opp,{})
        fixtures.append({
            'fixture_id':int(f['fixture_id']),'opponent_id':opp,'opponent':str(f['opponent']),'home':bool(f['home']),
            'label':str(f['opponent'])+(' (H)' if f['home'] else ' (A)'),'fdr':float(f['fdr']),
            'xpts':float(f['xpts']),'expected_minutes':float(f['expected_minutes']),'starting_probability':float(f['p_start']),
            'team_expected_goals':float(f['team_goals']),'opponent_expected_goals':float(f['opponent_goals']),
            'clean_sheet_probability':float(f['clean_sheet_probability']),'expected_goals':float(f['expected_goals']),
            'expected_assists':float(f['expected_assists']),'opponent_attack_strength':float(opp_stats.get('attack',0.)),
            'opponent_defensive_concede_strength':float(opp_stats.get('concede',0.)),'fdr_source':VX_WC_ENGINE.fdr.source,
        })
    fixture_count=max(1,len(fixtures));xgi=sum(f['expected_goals']+f['expected_assists'] for f in fixtures)
    cs_any=1-math.prod(1-f['clean_sheet_probability'] for f in fixtures) if fixtures else 0.
    team_goals=sum(f['team_expected_goals'] for f in fixtures)
    fdrs=[f['fdr'] for f in fixtures]
    start=float(card['starting_probability']);mins=float(card['expected_minutes'])
    per_fixture_mins=mins/fixture_count
    security='high' if start>=.85 and per_fixture_mins>=75 else ('medium' if start>=.65 and per_fixture_mins>=55 else 'low')
    rotation='LOW' if security=='high' else ('MEDIUM' if security=='medium' else 'HIGH')
    p={
        'id':int(pid),'code':int(card['photo_code']),'name':card['name'],'short_name':card['name'],'full_name':card['full_name'],
        'position':card['position'],'club':card['club'],'club_full_name':card['club_name'],'team_id':int(card['team_id']),
        'price':float(card['price']),'current_gameweek':int(VX_CURRENT_GW),'current_fixtures':fixtures,
        'next_fixture':', '.join(f['label'] for f in fixtures) or 'BLANK','next_fixtures':[f['label'] for f in fixtures],
        'current_fixture_fdrs':fdrs,'next_fixture_difficulty':float(fdrs[0]) if len(fdrs)==1 else None,
        'fdr_mean':float(np.mean(fdrs)) if fdrs else None,'expected_points_next_gw':float(card['expected_points']),
        'expected_points':float(card['expected_points']),'current_gw_xpts':float(card['expected_points']),
        'expected_minutes':mins,'starting_probability':start,'minutes_security':security,'rotation_risk':rotation,
        'ownership_pct':card['ownership_pct'],'top10k_ownership_pct':card['top10k_ownership_pct'],
        'transfers_in':card['transfers_in'],'transfers_out':card['transfers_out'],'net_transfers':card['net_transfers'],
        'market_direction':card['market_direction'],'availability':card['availability'],'injury_status':card['injury_status'],
        'status_marker':card['injury_status'] if card['availability'].get('flagged') else '',
        'recent_minutes':card['recent_minutes'],'latest_match':card['latest_match'],'latest_match_text':card['latest_match_text'],
        'verified_news':card['verified_news'],'club_news':card['club_news'],'penalties_order':card['penalties_order'],
        'corners_order':card['corners_order'],'free_kicks_order':card['free_kicks_order'],'role_note':card['role_note'],
        'current_expected_goals':sum(f['expected_goals'] for f in fixtures),'current_expected_assists':sum(f['expected_assists'] for f in fixtures),
        'current_expected_goal_involvement':xgi,'clean_sheet_probability_any':cs_any,'team_expected_goals':team_goals,
        'team_attacking_strength':float(VX_WC_ENGINE.team_stats[int(card['team_id'])]['attack']),
        'team_defensive_concede_strength':float(VX_WC_ENGINE.team_stats[int(card['team_id'])]['concede']),
        'value_score_1gw':float(card['expected_points'])/max(.1,float(card['price'])),
        'is_selected':int(pid) in selected_ids,'is_bench':int(pid) in bench_ids,'is_captain':int(pid)==cap,'is_vice_captain':int(pid)==vice,
        'is_starting_xi':int(pid) in xi_ids,'alternative_ids':[],'alternatives':[],'alternative_prices':[],
        'projection_limitations':card.get('projection_limitations',[]),'match_coverage':card.get('match_coverage',{}),
    }
    if p['is_bench']:
        reason=f"Bench cover is kept efficient: £{p['price']:.1f}m with {start:.0%} starting probability and {mins:.0f} expected minutes if needed."
    elif p['is_captain']:
        role=' Verified first-choice penalties strengthen the upside.' if p['penalties_order']==1 else ''
        reason=f"The captaincy route drives this pick: {p['current_gw_xpts']:.1f} xPts, {mins:.0f} expected minutes and {team_goals:.2f} modeled team goals this Gameweek.{role}"
    elif p['is_vice_captain']:
        reason=f"He provides the strongest vice-captain cover in the XI: {p['current_gw_xpts']:.1f} xPts with {start:.0%} starting probability."
    elif p['position']=='GK':
        reason=f"This is a one-week goalkeeper play: {cs_any:.0%} modeled clean-sheet probability across the activation fixtures and {mins:.0f} expected minutes."
    elif p['position']=='DEF':
        if xgi>=.15:
            reason=f"He has two routes to points this week: {cs_any:.0%} clean-sheet probability plus {xgi:.2f} modeled attacking involvement."
        elif p['corners_order']==1 or p['free_kicks_order']==1:
            reason=f"The defensive projection is supported by verified set-piece involvement and {mins:.0f} expected minutes."
        else:
            reason=f"The pick is built on one-week defensive value: {cs_any:.0%} clean-sheet probability, {start:.0%} start probability and {p['current_gw_xpts']:.1f} xPts."
    else:
        if p['penalties_order']==1:
            reason=f"Verified first-choice penalties add to {xgi:.2f} modeled goal involvement and a {p['current_gw_xpts']:.1f}-point projection."
        elif xgi>=.35:
            reason=f"The attacking case is direct: {xgi:.2f} modeled goal involvement in a team projected for {team_goals:.2f} goals this Gameweek."
        else:
            reason=f"The model prefers his one-week balance of {p['current_gw_xpts']:.1f} xPts, {mins:.0f} expected minutes and {team_goals:.2f} team goals."
    if p['availability'].get('flagged'):
        reason += ' Availability is conditional and must be rechecked before the deadline.'
    p['why_freehit_pick']=reason;p['pick_reason']=reason;p['reason']=reason
    p['pick_type']='risk' if p['availability'].get('flagged') or security=='low' else ('premium' if p['price']>=8 else ('ceiling' if xgi>=.35 else 'value'))
    return p


def _fh_viable_pool(gw):
    scores=_VX_GW_SCORES[_VX_GW_SCORES.gameweek.eq(gw)][['player_id','gameweek_expected_points','expected_minutes','p_start','fixture_count']]
    frame=VX_POOL.merge(scores,on='player_id',how='left',suffixes=('','_gw'))
    frame['selected_by_percent']=pd.to_numeric(frame['selected_by_percent'],errors='coerce').fillna(0.)
    frame['gameweek_expected_points']=pd.to_numeric(frame['gameweek_expected_points'],errors='coerce').fillna(0.)
    frame['expected_minutes_gw']=pd.to_numeric(frame['expected_minutes_gw'],errors='coerce').fillna(0.)
    frame['p_start_gw']=pd.to_numeric(frame['p_start_gw'],errors='coerce').fillna(0.)
    return frame


def _fh_add_alternatives(player,all_cards,selected_ids,budget,squad_cost,club_counts):
    frame=_fh_viable_pool(VX_CURRENT_GW)
    frame=frame[(frame.position.eq(player['position'])) & (~frame.player_id.isin(selected_ids))]
    frame=frame[~frame.status.astype(str).isin(['i','s','u']) & frame.fixture_count.gt(0) & frame.expected_minutes_gw.ge(45) & frame.p_start_gw.ge(.60)]
    max_price=float(player['price'])+float(budget)-float(squad_cost)
    frame=frame[frame.price.le(max_price+.001)]
    frame=frame.loc[frame.team_id.map(lambda tid:club_counts.get(int(tid),0)+(0 if int(tid)==player['team_id'] else 1)<=3).astype(bool)]
    frame=frame.sort_values(['gameweek_expected_points','p_start_gw'],ascending=False).head(3)
    player['alternative_ids']=list(map(int,frame.player_id));player['alternatives']=list(map(str,frame.player_name));player['alternative_prices']=list(map(float,frame.price))


def _fh_differentials(cards,candidate_cards):
    universe=cards+candidate_cards
    bypos={}
    for pos in ['GK','DEF','MID','FWD']:
        ps=[p for p in universe if p['position']==pos and p['starting_probability']>=.70 and _fh_status_available(p)]
        if not ps:continue
        own=np.array([float(p['ownership_pct'] or 0) for p in ps]);xp=np.array([p['current_gw_xpts'] for p in ps])
        bypos[pos]=(float(np.nanmedian(own)),float(np.quantile(xp,.60)))
    result=[]
    for p in universe:
        if p['position'] not in bypos:continue
        own_med,xp_cut=bypos[p['position']]
        if p['starting_probability']>=.70 and float(p['ownership_pct'] or 0)<own_med and p['current_gw_xpts']>=xp_cut and _fh_status_available(p):
            q=dict(p);q['differential_reason']=(f"Ownership is below the viable {p['position']} peer median, while {p['current_gw_xpts']:.1f} xPts and {p['starting_probability']:.0%} start probability support the one-week upside.")
            q['_diff_score']=p['current_gw_xpts']+.5*p['current_expected_goal_involvement']-.01*float(p['ownership_pct'] or 0)
            result.append(q)
    result=sorted({p['id']:p for p in result}.values(),key=lambda p:p['_diff_score'],reverse=True)[:5]
    for p in result:p.pop('_diff_score',None)
    return result


def _fh_avoid_monitor(selected_ids):
    frame=_fh_viable_pool(VX_CURRENT_GW);frame=frame[~frame.player_id.isin(selected_ids)]
    if frame.empty:return []
    popular=float(np.quantile(frame.selected_by_percent,.65))
    pos_median=frame.groupby('position').gameweek_expected_points.median().to_dict()
    candidates=frame[frame.selected_by_percent.ge(popular)].sort_values('selected_by_percent',ascending=False)
    out=[]
    for _,r in candidates.iterrows():
        card=_fh_player_object(int(r.player_id),selected_ids,set(),set(),-1,-1)
        fdr=card['fdr_mean'] if card['fdr_mean'] is not None else 5.
        if card['availability'].get('flagged'):
            classification='Fitness-dependent option';reason='Current availability is not secure enough for a one-week chip without a deadline recheck.'
        elif card['starting_probability']<.65 or card['expected_minutes']<55:
            classification='Minutes-risk option';reason=f"The model has only {card['starting_probability']:.0%} starting probability and {card['expected_minutes']:.0f} expected minutes."
        elif fdr>4.0:
            classification='Fixture-dependent option';reason=f"The position-specific centralized FDR averages {fdr:.1f}, weakening the one-week case."
        elif card['current_gw_xpts']<float(pos_median.get(card['position'],0)):
            classification='Good player, weaker Free Hit fit';reason=f"The {card['current_gw_xpts']:.1f}-point projection is below the viable {card['position']} median for this Gameweek."
        else:
            continue
        card['classification']=classification;card['avoid_reason']=reason;out.append(card)
        if len(out)>=5:break
    return out


def vx_freehit_contract(key,squad,plan,budget):
    if not key.startswith('free_hit'):raise RuntimeError('Free Hit contract requested with non-Free-Hit key')
    if plan is None or plan.empty:raise ValueError('Free Hit plan is empty')
    row=plan.iloc[0];gw=int(row.gameweek);selected_ids=set(map(int,squad.player_id));xi_ids=set(map(int,row.xi.player_id));bench_ids=list(map(int,row.bench.player_id));bench_set=set(bench_ids)
    cap=int(row.captain_player_id);vice=int(row.vice_captain_player_id);club_counts=squad.team_id.value_counts().astype(int).to_dict()
    cards=[_fh_player_object(int(pid),selected_ids,xi_ids,bench_set,cap,vice) for pid in squad.player_id]
    byid={p['id']:p for p in cards};squad_cost=float(squad.price.sum())
    for p in cards:_fh_add_alternatives(p,None,selected_ids,budget,squad_cost,club_counts)
    frame=_fh_viable_pool(gw)
    viable=frame[(~frame.player_id.isin(selected_ids)) & (~frame.status.astype(str).isin(['i','s','u'])) & frame.fixture_count.gt(0) & frame.expected_minutes_gw.ge(55) & frame.p_start_gw.ge(.65)].sort_values(['gameweek_expected_points','p_start_gw'],ascending=False).head(12)
    candidate_cards=[_fh_player_object(int(pid),selected_ids,set(),set(),-1,-1) for pid in viable.player_id]
    alternatives=[]
    for p in candidate_cards[:6]:
        q=dict(p);q['alternative_reason']=(f"Alternative Free Hit route if team news changes: {p['current_gw_xpts']:.1f} xPts, {p['starting_probability']:.0%} start probability and {p['next_fixture']}.")
        alternatives.append(q)
    differentials=_fh_differentials(cards,alternatives)
    avoid=_fh_avoid_monitor(selected_ids)
    fixtures=_fh_fixture_rows(gw,club_counts)
    cap_card=byid[cap];vice_card=byid[vice]
    def captain_reason(p,role):
        extras=[]
        if p['penalties_order']==1:extras.append('verified first-choice penalties')
        if p['current_expected_goal_involvement']>0:extras.append(f"{p['current_expected_goal_involvement']:.2f} modeled goal involvement")
        extras.append(f"{p['expected_minutes']:.0f} expected minutes")
        return f"{role}: {p['current_gw_xpts']:.1f} xPts with "+', '.join(extras)+'.'
    starting=[byid[i] for i in xi_ids];bench=[byid[i] for i in bench_ids]
    risk=sorted(starting,key=lambda p:(p['availability'].get('flagged',False)==False,p['starting_probability'],p['expected_minutes']))[0] if starting else None
    best_diff=differentials[0]['name'] if differentials else 'No evidence-supported differential required'
    clubs={str(t['short_name']):{'id':int(t['id']),'code':int(t['code']),'name':str(t['name']),'short_name':str(t['short_name'])} for t in _bootstrap['teams']}
    ids=set(selected_ids)
    for group in [alternatives,differentials,avoid]:ids.update(int(p['id']) for p in group)
    media_entities={str(int(r.player_id)):{'id':int(r.player_id),'name':str(r.player_name),'club':str(r.team_short),'position':str(r.position)} for _,r in VX_POOL[VX_POOL.player_id.isin(ids)].iterrows()}
    d={
        'schema_version':'freehit-2.0','gameweek':gw,'next_gameweek':gw,'chip':'freehit','mode':key.split('_')[-1],
        'horizon_gameweeks':[gw],'budget_total':float(budget),'money_spent':squad_cost,'budget_used':squad_cost,'money_remaining':round(float(budget)-squad_cost,1),
        'formation':str(row.formation),'squad':cards,'starting_xi_ids':list(map(int,row.xi.player_id)),'bench_ids':bench_ids,
        'bench_order':[byid[i]['name'] for i in bench_ids],'bench_gk':next((p['name'] for p in bench if p['position']=='GK'),None),
        'fixtures':fixtures,'alternative_free_hit_picks':alternatives,'watchlist':alternatives,'differentials':differentials,'avoid_list':avoid,
        'captain_picks':{'captain':{'id':cap,'name':cap_card['name'],'reason':captain_reason(cap_card,'Captain')},'vice_captain':{'id':vice,'name':vice_card['name'],'reason':captain_reason(vice_card,'Vice-captain')}},
        'summary_stats':{'starting_xi_xpts':float(row.xi_xpts),'captain_adjusted_xpts':float(row.xi_plus_captain_xpts),'bench_xpts':float(row.bench_xpts),
            'total_squad_xpts':float(row.xi_xpts)+float(row.bench_xpts),'money_spent':squad_cost,'money_remaining':round(float(budget)-squad_cost,1),
            'highest_upside_differential':best_diff,'lineup_risk_player':risk['name'] if risk else None,'lineup_risk_start_probability':risk['starting_probability'] if risk else None},
        'club_stack':{'clubs':[{'club':vx_club_name(VX_WC_ENGINE.teams[int(tid)]['short_name']),'short_name':VX_WC_ENGINE.teams[int(tid)]['short_name'],'count':int(count),'player_ids':[p['id'] for p in cards if p['team_id']==int(tid)]} for tid,count in sorted(club_counts.items(),key=lambda x:-x[1])]},
        'reversion_note':f'Free Hit applies to Gameweek {gw} only; the previous squad returns after the chip Gameweek under official FPL rules.',
        'clubs':clubs,'media_entities':media_entities,'presentation_version':VX_PRESENTATION_VERSION,
        'data_sources':{'players':'official FPL bootstrap-static plus centralized VORTEX player evidence','fixtures':VX_WC_ENGINE.fdr.source,
            'projections':'shared centralized activation-Gameweek xPts/minutes model','latest_match':'verified centralized evidence layer','top10k':'optional model input when available'},
        'selection_audit':{'objective':'maximize activation-Gameweek legal Starting XI xPts plus captain xPts; no future fixtures','future_fixtures_used':False,
            'bench_policy':'secondary optimization minimizes reserve cost while requiring credible outfield cover','solver':squad.attrs.get('solver',{})}
    }
    _new_plan=globals().get('VX_FREEHIT_SQUAD_PLAN')
    if _new_plan is None or {player['id'] for player in _new_plan['squad']}!=selected_ids:
        raise ValueError('Free Hit Slide 3 is an automated new recommendation; run the Auto Free Hit selector first.')
    d['slide3_plan']=_new_plan
    d['selection_audit']={'objective':_new_plan['data_health']['objective'],'future_fixtures_used':False,
        'bank_optimality_proven':_new_plan['data_health']['spend_optimality_proven'],
        'bench_policy':'One playable outfield reserve, minimum other cover; available players only',
        'solver':_new_plan['data_health']}
    d['generated_at']=VX_WC_ENGINE.now.isoformat()
    _source_stamps=[r['fetched_at'] for r in VX_WC_ENGINE.client.audit if r.get('fetched_at') and r.get('url') in (VXWC.API+'bootstrap-static/',VXWC.API+'fixtures/')]
    d['source_snapshot_at']=min(_source_stamps,key=VXWC.utc) if _source_stamps else d['generated_at']
    # Only next-GW projection rows enter Free Hit's intro; horizon totals are never read.
    from freehit_intro import build_freehit_case
    one_week_rows=_VX_GW_SCORES[_VX_GW_SCORES.gameweek.eq(gw)].to_dict('records')
    d['freehit_case']=build_freehit_case(_bootstrap,VX_WC_ENGINE.fixtures,
        projections=one_week_rows,refreshed_at=d['source_snapshot_at'],now=VX_WC_ENGINE.now,
        fdr_records=VX_WC_ENGINE.fdr.records)
    if d['freehit_case']['next_gw']!=gw:
        raise ValueError('Free Hit intro must use the selected next available Gameweek')
    from fixture_slide import build_fixture_slide
    six_week_rows=[]
    for team_id,team in VX_WC_ENGINE.teams.items():
        six_week_rows.append({'team_id':team_id,'club':team['short_name'],'name':team['name'],
            'weeks':[{'gameweek':week,'fixtures':VX_WC_ENGINE.fdr.for_team(team_id,week,'MID')}
                     for week in [gw]]})
    _immediate_models={int(row['player_id']):row for row in one_week_rows}
    _fixture_player_models=[]
    for row in VX_WC_ENGINE.pool.to_dict('records'):
        immediate=_immediate_models.get(int(row['player_id']))
        if immediate is None:continue
        _fixture_player_models.append(dict(row,p_start=immediate['p_start'],
            expected_minutes=min(float(row['expected_minutes']),float(immediate['expected_minutes'])),
            xpts_freehit_gw=immediate['gameweek_expected_points']))
    d['fixture_case']=build_fixture_slide(six_week_rows,[gw],_bootstrap['elements'],
        {'chip':'freehit','player_models':_fixture_player_models,
         'currency_multiplier':_bootstrap['game_settings'].get('ui_currency_multiplier',10)})
    from freehit_analyst import fixture_case,enrich
    d['fixture_case']=fixture_case(d['fixture_case'],_bootstrap['elements'],{'player_models':_fixture_player_models,'selected_squad':d['squad'],'official_fixtures':VX_WC_ENGINE.fixtures,'currency_multiplier':_bootstrap['game_settings'].get('ui_currency_multiplier',10)})
    d=enrich(d,_bootstrap,one_week_rows,getattr(VX_WC_ENGINE,'verified_manager_squad',None),VX_WC_ENGINE.fixtures)
    d['data_hash']=hashlib.sha256(json.dumps(d,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
    vx_validate_freehit(d)
    return d


def vx_validate_freehit(d):
    _fh_base_validate(d);gw=int(d['gameweek']);score_index=_VX_GW_SCORES[_VX_GW_SCORES.gameweek.eq(gw)].set_index('player_id')
    for p in d['squad']:
        if int(p['id']) not in score_index.index:raise ValueError(f"Free Hit player {p['id']} has no canonical GW{gw} projection")
        row=score_index.loc[int(p['id'])]
        if abs(float(p['current_gw_xpts'])-float(row.gameweek_expected_points))>.001:raise ValueError(f"{p['name']}: xPts differs from shared model")
        if abs(float(p['expected_minutes'])-float(row.expected_minutes))>.001:raise ValueError(f"{p['name']}: expected minutes differ from shared model")
        if abs(float(p['starting_probability'])-float(row.p_start))>.001:raise ValueError(f"{p['name']}: start probability differs from shared model")
        for f in p['current_fixtures']:
            if not 1<=float(f['fdr'])<=5:raise ValueError(f"{p['name']}: invalid centralized FDR {f['fdr']}")
            if f['fdr_source']!=VX_WC_ENGINE.fdr.source:raise ValueError(f"{p['name']}: non-centralized FDR source")
        if any(w.get('gameweek') not in (None,gw) for w in p.get('current_fixtures',[])):raise ValueError(f"{p['name']}: future fixture leaked into Free Hit")
    cap=next(p for p in d['squad'] if p['is_captain']);vice=next(p for p in d['squad'] if p['is_vice_captain'])
    if cap['is_bench'] or vice['is_bench']:raise ValueError('Captain or vice is on the Free Hit bench')
    if abs(d['summary_stats']['starting_xi_xpts']-sum(p['current_gw_xpts'] for p in d['squad'] if not p['is_bench']))>.01:raise ValueError('Starting XI xPts summary mismatch')
    return True












def vx_freehit_manifest(d):
    scenes=[];gw=d['gameweek'];ps=d['squad'];byid={p['id']:p for p in ps};cap=byid[d['captain_picks']['captain']['id']];vice=byid[d['captain_picks']['vice_captain']['id']]
    def add(title,kind,text,players=None,tables=None,source=None):
        scenes.append({'scene_id':'fh_scene_'+str(len(scenes)+1),'title':title,'gameweek':gw,'duration_seconds':None,'narration_text':text,'narration_voice':'ryan','narration_speed':1.,
            'pronunciation_overrides':[],'visual_type':kind,'players':players or [],'tables':tables or [],
            'theme':{'primary':'#FF6B00','positive':'#00FF87','captain':'#FFD700','vice':'#C0C0C0','chip_badge':'FH.png'},
            'source_data_fields':source or ['squad','fixtures','summary_stats'],'animation':{'card_entry_seconds':.45,'stat_stagger_seconds':.12,'pitch_reveal_seconds':6,'crossfade_seconds':.5},
            'timing':{'audio_padding_seconds':1.5,'target_words_per_minute':145,'status':'awaiting_tts'},'story_version':VX_FREEHIT_STORY_VERSION,'presentation_version':VX_PRESENTATION_VERSION})
    from freehit_analyst import intro_case
    freehit_narration=lambda case:case['narration_text']
    case=d['freehit_case']
    scenes.append({'scene_id':'fh_scene_1','title':'When Should You Play Your Free Hit?',
        'gameweek':gw,'visual_type':'fh_blueprint','players':[],'tables':[],
        'freehit_case':case,'narration_text':freehit_narration(case),'narration_timeline':case['narration_timeline'],'duration_seconds':None,
        'pronunciation_overrides':[],'source_data_fields':['freehit_case.reasons','freehit_case.next_gw'],
        'timing':{'status':'awaiting measured narration','audio_padding_seconds':.65},
        'animation':{'clock':'measured video seconds','crossfade_seconds':.5},
        'story_version':VX_FREEHIT_STORY_VERSION,'presentation_version':VX_PRESENTATION_VERSION})
    case=d['fixture_case']
    add(case['label'],'fixture_heatmap',case['narration_text'],source=['fixture_case.rows','fixture_case.gameweeks','fixture_case.targets','fixture_case.avoids'])
    scenes[-1].update(fixture_case=case,narration_timeline=case['narration_timeline'],subheading=case['range_label'],display_title='FIXTURE DIFFICULTY')
    summary=d['summary_stats']
    from free_hit_plan import generate_free_hit_slide3_narration
    squad_plan=d['slide3_plan'];timeline=generate_free_hit_slide3_narration(squad_plan)
    add(f"Free Hit Plan: GW{gw}",'fh_squad_plan',' '.join(item['narration_text'] for item in timeline),source=['slide3_plan'])
    scenes[-1].update(squad_plan=squad_plan,narration_timeline=timeline,subheading=squad_plan['subtitle'])
    from free_hit_final_slide import from_contract,generate_free_hit_final_slide_narration
    captain_pick=from_contract(d)
    final_timeline=generate_free_hit_final_slide_narration(captain_pick)
    add(f"Captain Pick: GW{gw}",'fh_captain_final',' '.join(item['narration_text'] for item in final_timeline),
        source=['slide3_plan.starting_xi','slide3_plan.press_conference_status','squad.availability'])
    scenes[-1].update(captain_pick=captain_pick,narration_timeline=final_timeline,
        display_title='CAPTAIN PICK',subheading='Final Free Hit decision for one Gameweek only')
    expected_types=['fh_blueprint','fixture_heatmap','fh_squad_plan','fh_captain_final']
    if [scene['visual_type'] for scene in scenes]!=expected_types:
        raise ValueError('Free Hit requires exactly four ordered slides')
    return scenes


def vx_contract(key,squad,plan,budget):
    if key.startswith('wildcard'):return VX_WC_ENGINE.contract(squad,plan)
    return vx_freehit_contract(key,squad,plan,budget)


def vx_build_manifest(d):
    scenes = VXWC.build_manifest(d) if d['chip']=='wildcard' else vx_freehit_manifest(d)
    summary = d.get('summary_stats', {})
    squad = d.get('squad', [])
    evidence = {
        'model_version':'rolling-xg-shrinkage-bps-2026-v1',
        'scope':'six-Gameweek Wildcard' if d['chip']=='wildcard' else 'single-Gameweek Free Hit',
        'gameweek':d['gameweek'],
        'xi_xpts':summary.get('starting_xi_xpts'),
        'squad':[
            {'player_id':p.get('id'),'name':p.get('name'),
             'expected_points':p.get('expected_points'),
             'expected_minutes':p.get('expected_minutes'),
             'starting_probability':p.get('starting_probability')}
            for p in squad
        ],
        'data_hash':d.get('data_hash'),
    }
    for scene in scenes:
        scene['projection_evidence'] = evidence
    return scenes


def vx_validate_contract(d):
    return VXWC.validate_contract(d) if d['chip']=='wildcard' else vx_validate_freehit(d)


VX_DATA_CONTRACTS={};VX_SCENE_MANIFESTS={};VX_STORY_GENERATED={};_VX_NARRATION_PLANS={}
VX_WILDCARD_SCENE_JSONS={};VX_NARRATION_OBJECTS={}
_targets=[]
if VX_GENERATE_WILDCARD:_targets.append(('wildcard_manual' if USE_MANUAL_WILDCARD else 'wildcard_auto',VX_WILDCARD_MANUAL_SQUAD if USE_MANUAL_WILDCARD else VX_WILDCARD_AUTO_SQUAD,VX_WILDCARD_MANUAL_PLAN if USE_MANUAL_WILDCARD else VX_WILDCARD_AUTO_PLAN,VX_WILDCARD_BUDGET))
if VX_GENERATE_FREEHIT:_targets.append(('free_hit_manual' if USE_MANUAL_FREEHIT else 'free_hit_auto',VX_FREEHIT_MANUAL_SQUAD if USE_MANUAL_FREEHIT else VX_FREEHIT_AUTO_SQUAD,VX_FREEHIT_MANUAL_PLAN if USE_MANUAL_FREEHIT else VX_FREEHIT_AUTO_PLAN,VX_FREEHIT_BUDGET))
for key,squad,plan,budget in _targets:
    contract=vx_contract(key,squad,plan,budget)
    path=VX_ANALYSIS_ROOT/('wildcard_data.json' if key.startswith('wildcard') else 'freehit_data.json');path.write_text(json.dumps(contract,indent=2,ensure_ascii=False))
    VX_DATA_CONTRACTS[key]=json.loads(path.read_text());manifest=vx_build_manifest(VX_DATA_CONTRACTS[key]);VX_SCENE_MANIFESTS[key]=manifest
    _vx_model_dir=VX_NARRATION_OUT_DIR/key/VX_VIDEO_MODE.lower()
    _vx_model_dir.mkdir(parents=True,exist_ok=True)
    (_vx_model_dir/'Projection Evidence.json').write_text(
        json.dumps({'chip':key,'projection_evidence':manifest[0]['projection_evidence'],
                    'model_audit':VX_MODEL_AUDIT},indent=2,ensure_ascii=False,default=str),encoding='utf-8')
    if key.startswith('wildcard'):
        _VX_NARRATION_PLANS[key]=plan;VX_WILDCARD_FUTURE_PLAN=contract['transfer_plan']
        scene_json=VXWC.build_scene_json(VX_DATA_CONTRACTS[key],manifest);report=VXWC.validate_scene_json(scene_json)
        report_path=VX_ANALYSIS_ROOT/(key+'_scene_validation.json');report_path.write_text(json.dumps(report,indent=2,ensure_ascii=False))
        if not report['passed']:
            raise RuntimeError('Narration blocked: Wildcard Scene JSON validation failed. Resolve the listed data inconsistencies before generating video assets.\n'+'\n'.join(report['errors']))
        scene_json_path=VX_ANALYSIS_ROOT/(key+'_final_scene.json');scene_json_path.write_text(json.dumps(scene_json,indent=2,ensure_ascii=False))
        narration_json_path=VX_ANALYSIS_ROOT/(key+'_narration.json');script_path=VX_NARRATION_OUT_DIR/f"{key}_GW{contract['gameweek']}.txt"
        narration_object=VXWC.attach_verified_narration(scene_json,script_path,encode_script,decode_script,validation_report_path=str(report_path),narration_json_path=narration_json_path,script_saver=save_verified_script,policy=VX_NARRATION_POLICY)
        VX_WILDCARD_SCENE_JSONS[key]=scene_json;VX_NARRATION_OBJECTS[key]=narration_object
        print(f"Wildcard narration complete: finalized Scene JSON validated, all spoken facts synchronized with rendered Wildcard data, conditional risk blocks applied, and premium humanized narration generated for Gameweek {scene_json['gameweek']}.")
        generated=VXWC.narration_segments(narration_object);VX_STORY_GENERATED[key]=generated
        for scene,section in zip(manifest,narration_object['sections']):
            scene['narration_text']=section['narration'];scene['timing'].update(status='scene-json narration validated',estimated_duration_seconds=section['estimated_duration_seconds'],pause_before_seconds=section.get('pause_before_seconds',0),pause_after_seconds=section.get('pause_after_seconds',0))
    else:
        vx_validate_freehit(contract)
        VX_STORY_GENERATED[key]=[{'id':s['scene_id'],'gameweek':s['gameweek'],'text':s['narration_text'],'mood':'analysis','visual':s['visual_type'],'story_version':VX_FREEHIT_STORY_VERSION} for s in manifest]
        _VX_NARRATION_PLANS[key]=plan
    (VX_ANALYSIS_ROOT/(key+'_scene_manifest.json')).write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
VX_STORY_META={key:{s['id']:{'visual':s['visual']} for s in segs} for key,segs in VX_STORY_GENERATED.items()}


def _vx_story_text(generated,data_hash=None):
    return '# STORY_VERSION: '+generated[0].get('story_version',VX_STORY_VERSION)+'\n'+('# DATA_HASH: '+data_hash+'\n' if data_hash else '')+encode_script(generated)


VX_NARRATION={};VX_NARRATION_PATHS={}
for key,generated in VX_STORY_GENERATED.items():
    if not generated:continue
    weeks=_VX_NARRATION_PLANS[key].gameweek.astype(int).tolist();path=VX_NARRATION_OUT_DIR/f'{key}_GW{weeks[0]}.txt';VX_NARRATION_PATHS[key]=path
    if key.startswith('wildcard'):
        VX_NARRATION[key]=decode_script(path.read_text(encoding='utf-8'))
        # Slide 1 has measured player-name cues; 'Keep edits' must not replay an older intro without those spoken names.
        _vx_wc_intro_scene=next(scene for scene in VX_SCENE_MANIFESTS[key] if scene['visual_type']=='wc_blueprint')
        _vx_wc_intro_text=' '.join(cue['narration_text'] for cue in _vx_wc_intro_scene['narration_timeline'])
        _vx_wc_intro_script=next(segment for segment in VX_NARRATION[key] if segment['id']==_vx_wc_intro_scene['scene_id'])
        if ' '.join(_vx_wc_intro_script['text'].split()) != ' '.join(_vx_wc_intro_text.split()):
            _vx_wc_intro_script['text']=_vx_wc_intro_text
            _vx_wc_preserved_header='\n'.join(line for line in path.read_text(encoding='utf-8').splitlines() if line.startswith('#'))
            path.write_text((_vx_wc_preserved_header+'\n' if _vx_wc_preserved_header else '')+encode_script(VX_NARRATION[key]),encoding='utf-8')
            print('Wildcard Slide 1 narration synchronized with official spoken player names; other slides preserved.',flush=True)
        _vx_wc_intro_scene['narration_text']=_vx_wc_intro_text
        _vx_wc_intro_section=next(section for section in VX_NARRATION_OBJECTS[key]['sections'] if section['slide_id']==_vx_wc_intro_scene['scene_id'])
        _vx_wc_intro_section['narration']=_vx_wc_intro_text
        _vx_wc_intro_section['estimated_duration_seconds']=round(len(_vx_wc_intro_text.split())/145*60+_vx_wc_intro_section.get('pause_before_seconds',0)+_vx_wc_intro_section.get('pause_after_seconds',0),2)
        # Final Slide captain comparisons have their own measured name cues; stale Keep-edits scripts must not override them.
        _vx_wc_final_scene=next(scene for scene in VX_SCENE_MANIFESTS[key] if scene['visual_type']=='wc_captaincy_final')
        _vx_wc_final_text=' '.join(part['narration_text'] for part in _vx_wc_final_scene['narration_timeline'])
        _vx_wc_final_script=next(segment for segment in VX_NARRATION[key] if segment['id']==_vx_wc_final_scene['scene_id'])
        if ' '.join(_vx_wc_final_script['text'].split()) != ' '.join(_vx_wc_final_text.split()):
            _vx_wc_final_script['text']=_vx_wc_final_text
            _vx_wc_final_header='\n'.join(line for line in path.read_text(encoding='utf-8').splitlines() if line.startswith('#'))
            path.write_text((_vx_wc_final_header+'\n' if _vx_wc_final_header else '')+encode_script(VX_NARRATION[key]),encoding='utf-8')
            print('Wildcard Slide 4 captaincy comparison synced with spoken names; other slides preserved.',flush=True)
        _vx_wc_final_scene['narration_text']=_vx_wc_final_text
        _vx_wc_final_section=next(section for section in VX_NARRATION_OBJECTS[key]['sections'] if section['slide_id']==_vx_wc_final_scene['scene_id'])
        _vx_wc_final_section['narration']=_vx_wc_final_text
        _vx_wc_final_section['estimated_duration_seconds']=round(len(_vx_wc_final_text.split())/145*60+_vx_wc_final_section.get('pause_before_seconds',0)+_vx_wc_final_section.get('pause_after_seconds',0),2)
        VX_NARRATION_OBJECTS[key]['total_estimated_duration_seconds']=sum(section['estimated_duration_seconds'] for section in VX_NARRATION_OBJECTS[key]['sections'])
        (VX_ANALYSIS_ROOT/(key+'_narration.json')).write_text(json.dumps(VX_NARRATION_OBJECTS[key],indent=2,ensure_ascii=False),encoding='utf-8')
    else:
        # Free Hit facts are regenerated from the validated one-Gameweek contract every run; stale manual facts cannot survive.
        VX_NARRATION[key]=save_verified_script(path,_vx_story_text(generated,VX_DATA_CONTRACTS[key]['data_hash']),VX_DATA_CONTRACTS[key]['data_hash'],VX_NARRATION_POLICY)
        # The current data-derived intro supersedes an older Keep-edits script.
        _fh_intro = next(s for s in manifest if s['visual_type'] == 'fh_blueprint')
        _fh_script = next(s for s in VX_NARRATION[key] if s['id'] == _fh_intro['scene_id'])
        _fh_approved = ' '.join(c['narration_text'] for c in _fh_intro['narration_timeline'])
        if ' '.join(_fh_script['text'].split()) != ' '.join(_fh_approved.split()):
            _fh_script['text'] = _fh_approved
            _fh_headers = '\n'.join(line for line in path.read_text(encoding='utf-8').splitlines() if line.startswith('#'))
            path.write_text((_fh_headers + '\n' if _fh_headers else '') + encode_script(VX_NARRATION[key]), encoding='utf-8')
        assert len(_fh_intro['narration_timeline']) == 6
        assert _fh_script['text'].split() == _fh_approved.split()
        print('Free Hit Slide 1: current Gameweek speech and six slide cues synchronized.')
    words=sum(len(seg['text'].split()) for seg in VX_NARRATION[key]);estimate=words/145
    print(f'✅ {key}: {len(VX_NARRATION[key])} scenes, {words} words, roughly {estimate:.1f} min narration at 145 WPM')
    print('   Narration:',path)

# TXT-only per-slide narration exports; the validated chip-level TXT remains authoritative.
for _vx_key,_vx_segments in VX_NARRATION.items():
    for _vx_seg in _vx_segments:
        _vx_txt = VX_NARRATION_OUT_DIR/(f"{_vx_key}_{_vx_seg['id']}.txt")
        _vx_txt.write_text(_vx_seg['text'].strip()+"\n",encoding='utf-8')

VX_CELL2_COMPLETE = True
print('✅ Wildcard remains finalized; Free Hit now uses the shared verified data/FDR/evidence foundation and activation-Gameweek-only strategy.')
print("✅ CELL 12 COMPLETE")

✅ Cell 12 Free Hit modules synchronized with saved source
Wildcard narration complete: finalized Scene JSON validated, all spoken facts synchronized with rendered Wildcard data, conditional risk blocks applied, and premium humanized narration generated for Gameweek 7.
✅ wildcard_auto: 4 scenes, 3136 words, roughly 21.6 min narration at 145 WPM
   Narration: /content/drive/MyDrive/FPL_VORTEX/Wildcard_FreeHit/data/wildcard_auto_GW7.txt
Free Hit Slide 1: current Gameweek speech and six slide cues synchronized.
✅ free_hit_auto: 4 scenes, 1632 words, roughly 11.3 min narration at 145 WPM
   Narration: /content/drive/MyDrive/FPL_VORTEX/Wildcard_FreeHit/data/free_hit_auto_GW7.txt
✅ Wildcard remains finalized; Free Hit now uses the shared verified data/FDR/evidence foundation and activation-Gameweek-only strategy.
✅ CELL 12 COMPLETE


In [ ]:
#@title 🔴 CELL 13 - WILDCARD FDR SLIDE 2 {display-mode: "form"}

def vx_require_scene_contracts(cell_name):
    contracts = globals().get('VX_DATA_CONTRACTS')
    scenes = globals().get('VX_SCENE_MANIFESTS')
    narration = globals().get('VX_NARRATION_PATHS')
    if (not globals().get('VX_CELL2_COMPLETE', False)
            or not isinstance(contracts, dict) or not contracts
            or not isinstance(scenes, dict) or set(contracts) != set(scenes)
            or not isinstance(narration, dict) or set(contracts) != set(narration)):
        raise RuntimeError(
            f'Cell 13 requires a successful Cell 12 scene-and-narration run. '
            'Finish Cells 11 and 12 in order before rendering or exporting.'
        )
    return True

vx_require_scene_contracts("Cell 13")

# Only Wildcard Slide 02 presentation/narration changes here.
# Disable this preview flag to return to the existing selected intro/full-video flow.
VX_WILDCARD_SLIDE2_PREVIEW_ONLY = True #@param {type:"boolean"}
import importlib,sys,json,copy,re,hashlib
from pathlib import Path
_VX_ELITE_FDR_SOURCE = '"""Wildcard Slide 02 only. Existing six-GW FDR, projections and exact Hero Card."""\nfrom __future__ import annotations\nimport copy,hashlib,html,io,json,math,re\nfrom pathlib import Path\nfrom PIL import Image,ImageDraw,ImageFont,ImageOps\nimport numpy as np\nfrom wildcard_fixture_analysis import player_candidates,evidence\nfrom wildcard_intro_motion import ease,font,GREEN,CYAN,GOLD,WHITE\nfrom tts_narration import CLUBS\nVERSION=\'wildcard-slide2-live-value-pairs-v5\'\n\ndef format_fixture(fixture):\n    """One shared display contract for Wildcard and Free Hit Slide 2."""\n    opponent = str(fixture[\'opponent\']).upper()\n    if len(opponent) != 3 or not opponent.isalpha():\n        raise ValueError(\'Fixture must use the official three-letter opponent abbreviation\')\n    rating = float(fixture[\'fdr\'])\n    if not math.isfinite(rating) or not 1 <= rating <= 5:\n        raise ValueError(\'Fixture difficulty must be a finite rating in [1,5]\')\n    rating_text = str(max(1,min(5,math.floor(rating+0.5))))\n    return f"{opponent}[{\'H\' if fixture[\'home\'] else \'A\'}]-{rating_text}"\n\nCRIMSON=\'#ed6575\';NAVY=\'#071521\';MUTED=\'#a7bfcd\'\n\ndef build_wildcard_fixture_narration(weeks, targets, avoids, club, fixture_words):\n    """Creator\'s Slide 2 structure, driven by the ranked rows and displayed cards."""\n    out = []\n    words = {1: \'one\', 2: \'two\', 3: \'three\', 4: \'four\', 5: \'five\', 6: \'six\', 7: \'seven\', 8: \'eight\', 9: \'nine\', 10: \'ten\'}\n    def count(number): return words.get(number, str(number))\n    def money(value): return f\'{value:.1f}\'.rstrip(\'0\').rstrip(\'.\')\n    def join_names(names):\n        return names[0] if len(names) == 1 else \', \'.join(names[:-1]) + \' and \' + names[-1] if names else \'\'\n    def add(identity, kind, text, row=None, player=None, fixtures=()):\n        out.append(dict(segment_id=identity, type=kind, narration_text=text,\n            team_id=row[\'team_id\'] if row else None, player_id=player[\'player_id\'] if player else None,\n            player_ids=[player[\'player_id\']] if player else [],\n            fixture_ids=[fixture[\'fixture_id\'] for fixture in fixtures], start=None, end=None))\n    def role_suffix(player):\n        role = str(player.get(\'role\') or \'\').casefold()\n        if role == \'first-choice penalty taker\': return \', with penalty duties adding to his appeal\'\n        if role == \'first-choice set-piece involvement\': return \', with set-piece involvement\'\n        return \'\'\n    add(\'opening\', \'intro\',\n        "Now, if you\'re playing your Wildcard, getting the fixtures right is absolutely crucial. "\n        f"We\'re planning for {count(len(weeks))} Gameweeks, not just one, and I\'ve identified {count(len(targets))} teams worth considering.")\n    for index, row in enumerate(targets):\n        team = club(row)\n        players = row.get(\'players\', [])\n        fixtures = row[\'fixture_evidence\'][\'fixtures\']\n        easiest = min(fixtures, key=lambda fixture: (fixture[\'fdr\'], fixture[\'gameweek\'])) if fixtures else None\n        opening = [fixture for fixture in fixtures if fixture[\'gameweek\'] == weeks[0]]\n        first = opening[0] if len(opening) == 1 and opening[0][\'fdr\'] <= row[\'average_fdr\'] else easiest\n        hardest = max(fixtures, key=lambda fixture: (fixture[\'fdr\'], -fixture[\'gameweek\'])) if fixtures else None\n        average = row[\'average_fdr\']\n        if index == 0:\n            text = f\'Starting with {team}. They actually have the best fixture run in my rankings, averaging {average:.2f} in difficulty.\'\n        elif index == 1:\n            text = f\'Next up, {team}. Their average fixture difficulty is {average:.2f},\'\n        elif index == 2:\n            text = f\'Moving on to {team}. Their fixture average is {average:.2f},\'\n        elif index == 3:\n            text = f\'{team} are another team on my radar. Their fixtures average {average:.2f},\'\n        elif index == len(targets) - 1:\n            text = f\'Finally, {team}. Their fixture difficulty averages {average:.2f},\'\n        else:\n            text = f\'Another team worth considering is {team}, averaging {average:.2f} in fixture difficulty.\'\n        # Avoid leaving a hanging clause when no verified player/fixture is available.\n        if text.endswith(\',\') and ((index in (1, 2) and first is None) or (index >= 3 and not players)):\n            text = text[:-1] + \'.\'\n        add(f\'team_{index + 1}\', \'team\', text, row)\n        if index == 0 and first:\n            opening_phrase = first[\'gameweek\'] == weeks[0]\n            text = (f"{fixture_words(first)} is the standout opening fixture" if opening_phrase else\n                f"{fixture_words(first)} in Gameweek {first[\'gameweek\']} is one of the stronger opportunities")\n            has_test = hardest and hardest[\'fixture_id\'] != first[\'fixture_id\'] and hardest[\'fdr\'] > first[\'fdr\']\n            add(\'fixture_1_best\', \'fixture\', text + (\',\' if has_test else \'.\'), row, fixtures=[first])\n            if has_test:\n                add(\'fixture_1_test\', \'fixture\',\n                    f"although {fixture_words(hardest)} in Gameweek {hardest[\'gameweek\']} is a tougher test.", row, fixtures=[hardest])\n        elif index == 1 and first:\n            if first[\'gameweek\'] == weeks[0]:\n                location = \'home game\' if first[\'home\'] else \'away game\'\n                text = f"and that opening {location} against {fixture_words(first, venue=False)} looks very attractive."\n            else:\n                text = f"and {fixture_words(first)} in Gameweek {first[\'gameweek\']} looks particularly promising."\n            add(\'fixture_2_best\', \'fixture\', text, row, fixtures=[first])\n        elif index == 2 and easiest:\n            add(\'fixture_3_best\', \'fixture\',\n                f"with {fixture_words(easiest)} in Gameweek {easiest[\'gameweek\']} looking particularly promising.", row, fixtures=[easiest])\n        for player_index, player in enumerate(players):\n            name = player[\'player_name\'];price = money(player[\'price\']);points = player[\'projected_points\']\n            suffix = role_suffix(player)\n            player[\'role_detail\'] = player.get(\'role\') if suffix else None\n            if index == 0 and player_index == 0:\n                text = f\'{name} looks particularly interesting at £{price} million, with {points:.1f} projected points across the {count(len(weeks))} weeks.\'\n            elif index == 0:\n                text = f\'{name} offers another route into {team}. He costs £{price} million and projects {points:.1f} points{suffix}.\'\n            elif index == 1 and player_index == 0:\n                text = f\'{name} is projected at {points:.1f} points for £{price} million\' + (\',\' if len(players) > 1 else \'.\')\n            elif index == 1:\n                cheaper = player[\'price\'] < players[0][\'price\']\n                text = f"while {name} offers {\'a cheaper alternative\' if cheaper else \'another alternative\'} at £{price} million, with {points:.1f} projected points."\n            elif index == 2 and player_index == 0:\n                text = f\'{name} comes in at £{price} million, with {points:.1f} projected points\' + (\',\' if len(players) > 1 else \'.\')\n            elif index == 2:\n                cheaper = player[\'price\'] < players[0][\'price\']\n                defensive = player[\'position\'] in (\'GK\', \'GKP\', \'DEF\')\n                description = (\'a cheaper defensive option\' if defensive else \'a cheaper option\') if cheaper else (\'another defensive option\' if defensive else \'another option\')\n                text = f\'while {name} offers {description} at £{price} million.\'\n            elif index == 3 and player_index == 0:\n                text = f\'and {name} stands out at just £{price} million, with {points:.1f} projected points.\'\n            elif index == 3:\n                involvement = \' and set-piece involvement\' if str(player.get(\'role\') or \'\').casefold() == \'first-choice set-piece involvement\' else \' and penalty duties\' if str(player.get(\'role\') or \'\').casefold() == \'first-choice penalty taker\' else \'\'\n                text = f\'{name} is also worth considering, with {points:.1f} projected points{involvement}.\'\n            elif index == len(targets) - 1 and player_index == 0:\n                text = f"and {name} is the player who catches my attention. At £{price} million, he\'s projected to return {points:.1f} points across the {count(len(weeks))} Gameweeks."\n            else:\n                lower = player[\'projected_points\'] < players[0][\'projected_points\']\n                text = f\'{name} is another option at £{price} million\'\n                text += f\', although his projection is lower at {points:.1f}.\' if lower else f\', with {points:.1f} projected points.\'\n            add(f\'player_{player["player_id"]}\', \'player\', text, row, player)\n        if not players:\n            add(f\'team_{index + 1}_no_pick\', \'availability\',\n                f"I don\'t have a player at {team} who currently passes the availability and playing-time checks. I won\'t force a pick.", row)\n        if index == 1 and hardest and first and hardest[\'fixture_id\'] != first[\'fixture_id\'] and hardest[\'fdr\'] > first[\'fdr\']:\n            add(\'fixture_2_test\', \'fixture\',\n                f"Of course, {fixture_words(hardest)} later in the run is something to keep in mind.", row, fixtures=[hardest])\n    if avoids:\n        add(\'caution_heading\', \'caution_intro\',\n            f"Now, there are also {count(len(avoids))} teams I\'d be cautious about investing in. "\n            f"{join_names([club(row) for row in avoids])} have some particularly difficult fixture runs.")\n        add(\'caution_context\', \'caution_intro\',\n            "That doesn\'t mean selling every player from those clubs, but I\'d certainly think twice before bringing in new ones.")\n    other_targets = [row for row in targets[1:] if row.get(\'players\')]\n    strongest = sorted(other_targets, key=lambda row: -max(player[\'projected_points\'] for player in row[\'players\']))[:3]\n    selected_ids = {row[\'team_id\'] for row in strongest}\n    alternatives = [club(row) for row in other_targets if row[\'team_id\'] in selected_ids]\n    text = f\'So, {club(targets[0])} lead the fixture rankings\' if targets else \'Those are the current fixture rankings\'\n    text += f\', while several players from {join_names(alternatives)} offer interesting alternatives.\' if alternatives else \'.\'\n    text += " The important thing is balancing fixtures with playing time, attacking potential and price. "\n    text += "Now, let\'s see how that translates into my actual Wildcard squad."\n    add(\'final_verdict\', \'summary\', text)\n    return out\n\ndef build_case(contract,bootstrap,models):\n    if contract.get(\'chip\')!=\'wildcard\':raise ValueError(\'Wildcard Slide 2 only\')\n    weeks=contract[\'horizon_gameweeks\']\n    if len(weeks)!=6 or weeks!=list(range(weeks[0],weeks[0]+6)):raise ValueError(\'Six consecutive Gameweeks required\')\n    case=copy.deepcopy(contract[\'fixture_case\']);rows=case[\'rows\']\n    if len(rows)!=20 or len({r[\'team_id\'] for r in rows})!=20:raise ValueError(\'All 20 clubs required\')\n    byid={r[\'team_id\']:r for r in rows};teams={t[\'id\']:t for t in bootstrap[\'teams\']};official={p[\'id\']:p for p in bootstrap[\'elements\']}\n    for row in rows:\n        if row[\'team_id\'] not in teams or teams[row[\'team_id\']][\'short_name\']!=row[\'club\']:raise ValueError(\'Club mismatch\')\n        if [w[\'gameweek\'] for w in row[\'weeks\']]!=weeks:raise ValueError(\'Fixture window mismatch\')\n        fs=[f for w in row[\'weeks\'] for f in w[\'fixtures\']]\n        rated=[float(f[\'fdr\']) for f in fs if f.get(\'fdr\') is not None]\n        if not rated or abs(sum(rated)/len(rated)-row[\'average_fdr\'])>1e-7:raise ValueError(\'Canonical FDR mean mismatch\')\n        for f in fs:\n            if f[\'team_id\']!=row[\'team_id\'] or teams[f[\'opponent_id\']][\'short_name\']!=f[\'opponent\']:raise ValueError(\'Fixture identity mismatch\')\n        for week in row[\'weeks\']:\n            for fixture in week[\'fixtures\']:\n                fixture[\'display\'] = format_fixture(fixture)\n        row[\'fixture_evidence\']=evidence(row,weeks)\n    # Preserve the existing model ranking; never round before ranking.\n    ranked=rows\n    if any(a[\'average_fdr\']>b[\'average_fdr\']+1e-9 for a,b in zip(rows,rows[1:])):raise ValueError(\'Existing FDR ranking is not ascending\')\n    targets=[copy.deepcopy(r) for r in ranked[:5]];avoids=[copy.deepcopy(r) for r in ranked[-3:]]\n    context={\'player_models\':models,\'currency_multiplier\':bootstrap.get(\'game_settings\',{}).get(\'ui_currency_multiplier\',10)}\n    players=[]\n    candidates_by_team={}\n    for row in targets:\n        candidates=player_candidates(row[\'team_id\'],bootstrap[\'elements\'],context)\n        candidates.sort(key=lambda p:(-p[\'value\'],-p[\'projected_points\'],-p[\'expected_minutes\'],p[\'player_id\']))\n        candidates_by_team[row[\'team_id\']]=candidates\n    # Each narrated pair comes from the existing eligible value-ranked player pool.\n    quotas={row[\'team_id\']:min(2,len(candidates_by_team[row[\'team_id\']])) for row in targets}\n    counts=[quotas[row[\'team_id\']] for row in targets]\n    for rank,row in enumerate(targets,1):\n        row[\'players\']=candidates_by_team[row[\'team_id\']][:quotas[row[\'team_id\']]]\n        row[\'rank\']=rank\n        row[\'candidate_count\']=len(candidates_by_team[row[\'team_id\']])\n        for p in row[\'players\']:\n            raw=official[p[\'player_id\']]\n            if raw[\'team\']!=row[\'team_id\'] or raw[\'status\']!=\'a\' or abs(raw[\'now_cost\']/context[\'currency_multiplier\']-p[\'price\'])>1e-8:raise ValueError(\'Player club, availability or price mismatch\')\n            p[\'player_name\']=str(raw.get(\'first_name\',\'\')).split()[0]+\' \'+re.sub(r\'^[A-Z]\\.\', \'\',str(raw.get(\'web_name\',\'\')))\n            p.update(team_id=row[\'team_id\'],club=row[\'club\'],club_name=teams[row[\'team_id\']][\'name\'])\n            players.append(p)\n    for row in avoids:row[\'players\']=[]\n    def club(row):return CLUBS.get(row[\'club\'],teams[row[\'team_id\']][\'name\'])\n    def fixture_words(fixture, venue=True):\n        opponent=CLUBS.get(teams[fixture[\'opponent_id\']][\'short_name\'],teams[fixture[\'opponent_id\']][\'name\'])\n        return opponent+(\' at home\' if fixture[\'home\'] else \' away\') if venue else opponent\n    out=build_wildcard_fixture_narration(weeks,targets,avoids,club,fixture_words)\n    case.update(season=str(contract.get(\'season\') or \'\').replace(\'-\', \' / \'),analyst_version=VERSION,targets=targets,avoids=avoids,elite_players=players,narration_timeline=out,narration_text=\' \'.join(c[\'narration_text\'] for c in out),gameweeks=weeks)\n    case[\'verification\']={\'teams\':20,\'targets\':len(targets),\'avoids\':len(avoids),\'featured_counts\':counts,\'featured_players\':len(players),\'coverage_by_target\':{row[\'club\']:dict(eligible=row[\'candidate_count\'],featured=len(row[\'players\'])) for row in targets},\'selection_order\':\'Up to two players per target; existing horizon xPts per million, then xPts/minutes/ID; availability and minutes checks unchanged\',\'projection_basis\':\'Existing notebook engine.pool xpts_wildcard_horizon; no recomputation\',\'source_hash\':contract[\'data_hash\'],\'word_count\':len(case[\'narration_text\'].split())}\n    return case\n\n\ndef render_hero_assets(case,views,media,badges,brand_root,work):\n    """Screenshot the exact existing card_html and broadcast CSS, without redesign."""\n    if not case[\'elite_players\']:\n        return {},[]\n    from hero_card import card_html\n    from playwright.sync_api import sync_playwright\n    brand_root=Path(brand_root);work=Path(work);work.mkdir(parents=True,exist_ok=True)\n    css=\'\\n\'.join(re.findall(r\'<style[^>]*>(.*?)</style>\',(brand_root/\'broadcast.html\').read_text(),re.S))\n    badge_paths={}\n    for row in case[\'rows\']:\n        badge_paths[int(row[\'team_id\'])]=(badges.get(row[\'club\']) or {}).get(\'path\')\n    output={};proof=[]\n    with sync_playwright() as pw:\n        browser=pw.chromium.launch(headless=True)\n        page=browser.new_page(viewport={\'width\':485,\'height\':693},device_scale_factor=2)\n        for p in case[\'elite_players\']:\n            pid=p[\'player_id\'];view=views.get(pid) or views.get(str(pid));m=media.get(pid) or media.get(str(pid))\n            if not view or view[\'teamId\']!=p[\'team_id\'] or abs(view[\'price\']-p[\'price\'])>.001:raise ValueError(\'Hero Card data does not match model snapshot\')\n            if not m or m.get(\'photo_kind\')!=\'player\':raise FileNotFoundError(\'Official player photograph required: \'+p[\'player_name\'])\n            view=copy.deepcopy(view)\n            canonical=[f for row in case[\'rows\'] if row[\'team_id\']==p[\'team_id\'] for week in row[\'weeks\'] for f in week[\'fixtures\']]\n            for fixture in view.get(\'fixtures\',[]):\n                match=next((f for f in canonical if f[\'opponent\']==fixture.get(\'opponent\') and f.get(\'home\')==fixture.get(\'home\')),None)\n                if match:fixture[\'fdr\']=match[\'fdr\']\n            raw=card_html(view,dict(logo=brand_root/\'assets/logo.png\',photo=m[\'photo\'],club_badge=(badges.get(p[\'club\']) or {}).get(\'path\'),badges=badge_paths),season=case.get(\'season\') or None,horizon=6)\n            digest=hashlib.sha256((raw+css).encode()).hexdigest()[:16];path=work/f\'hero_{pid}_{digest}.png\'\n            html_path=work/f\'hero_{pid}_{digest}.html\'\n            if not path.exists():\n                html_path.write_text(\'<html><head><meta charset="utf-8"><style>\'+css+\'\\nhtml,body{margin:0;width:485px;height:693px;background:transparent;overflow:hidden}.hero-card{width:485px;height:693px}</style></head><body>\'+raw+\'</body></html>\')\n                page.goto(html_path.as_uri());page.wait_for_function(\'Array.from(document.images).every(i=>i.complete)\');page.evaluate(\'document.fonts.ready\')\n                page.evaluate("() => {const el=document.querySelector(\'.hero-name\');let size=parseFloat(getComputedStyle(el).fontSize);while(el.scrollWidth>el.clientWidth && size>18){el.style.fontSize=(--size)+\'px\';}}")\n                if not page.evaluate(\'Array.from(document.images).every(i=>i.naturalWidth>0)\'):raise ValueError(\'Hero Card has a missing image\')\n                page.screenshot(path=str(path),omit_background=True)\n            output[pid]=Image.open(path).convert(\'RGBA\').resize((485,693),Image.Resampling.LANCZOS)\n            proof.append(dict(player_id=pid,path=str(path),component=\'Existing hero_card.card_html plus existing broadcast.html CSS\',price=p[\'price\'],projected_points=p[\'projected_points\']))\n        browser.close()\n    return output,proof\n\nclass EliteFDRVisual:\n    requires_motion=True\n    graphics_version="wildcard-slide2-readable-fdr-v1"\n    def __init__(self,scene,contract,width,height,background,hero_images,badges):\n        self.manifest=scene;self.contract=contract;self.width=width;self.height=height;self.case=scene[\'elite_fixture_case\'];self.timeline=scene[\'narration_timeline\'];self.heroes=hero_images;self.layout=[]\n        if any(c.get(\'alignment\')!=\'measured_segments\' or c.get(\'start\') is None for c in self.timeline):raise ValueError(\'Measured narration cues required; estimated timings refused\')\n        self.manifest[\'animation_version\']=self.graphics_version\n        self.background=background.convert(\'RGB\').resize((1920,1080),Image.Resampling.LANCZOS)\n        self.badges={}\n        for code,b in badges.items():\n            path=b.get(\'path\') if isinstance(b,dict) else None\n            if path and Path(path).exists():self.badges[code]=Image.open(path).convert(\'RGBA\')\n        self.rows=self.case[\'rows\'];self.byid={r[\'team_id\']:r for r in self.rows};self.players={p[\'player_id\']:p for p in self.case[\'elite_players\']};self.table_cache={};self.tiles={};self.sprite_cache={}\n        self.targets=list(self.case[\'targets\']);self.avoids=list(self.case[\'avoids\'])\n        self.table_bounds=(55,182,1321,1010)\n        self.stage_bounds=(1336,182,1865,1010)\n        self.blocks=[(69,205,688,966),(697,205,1310,966)]\n        self.header_box=(477,114,1290,171)\n        self.rank_w=24;self.badge_w=28;self.team_w=54;self.fixture_w=68;self.avg_w=56;self.col_gap=2\n        self.header_h=49;self.row_height=66;self.row_gap=5\n        self.row_geometry={}\n        for index,row in enumerate(self.rows):\n            block=self.blocks[0] if index<10 else self.blocks[1]\n            local=index if index<10 else index-10\n            bx1,by1,bx2,by2=block\n            row_y=by1+self.header_h+5+local*(self.row_height+self.row_gap)\n            x=bx1+10\n            rank_box=(x,row_y+8,x+self.rank_w,row_y+self.row_height-8);x+=self.rank_w+4\n            badge_box=(x,row_y+19,x+self.badge_w,row_y+19+self.badge_w);x+=self.badge_w+4\n            team_box=(x,row_y+8,x+self.team_w,row_y+self.row_height-8);x+=self.team_w+4\n            fixture_boxes=[]\n            for _ in range(6):\n                fixture_boxes.append((x,row_y+5,x+self.fixture_w,row_y+self.row_height-5));x+=self.fixture_w+self.col_gap\n            avg_box=(x,row_y+7,x+self.avg_w,row_y+self.row_height-7)\n            self.row_geometry[row[\'team_id\']]={\'row_box\':(bx1+6,row_y,bx2-6,row_y+self.row_height),\'rank_box\':rank_box,\'badge_box\':badge_box,\'team_box\':team_box,\'fixture_boxes\':fixture_boxes,\'avg_box\':avg_box,\'block\':0 if index<10 else 1,\'order\':index+1}\n        # All 20 rows must stay inside the same safe-area panel without footer or block overlap.\n        assert len(self.row_geometry)==20 and len({r[\'team_id\'] for r in self.rows})==20\n        for geo in self.row_geometry.values():\n            block=self.blocks[geo[\'block\']]\n            assert geo[\'row_box\'][1]>=block[1]+self.header_h and geo[\'row_box\'][3]<=block[3], \'Row overflow\'\n            assert geo[\'avg_box\'][2]<=block[2]-7, \'AVG column overflow\'\n            for fx in geo[\'fixture_boxes\']:\n                assert fx[0]>=block[0] and fx[2]<=block[2]-7, \'Fixture cell outside block\'\n        self.complete=self.draw(max(0,self.timeline[-1][\'end\']-.01),self.timeline[-1][\'end\'])\n    def text(self,im,text,x,y,size=32,color=WHITE,align=\'left\',max_width=None):\n        d=ImageDraw.Draw(im);size=int(size)\n        text=str(text)\n        f=self._face(size)\n        if max_width:\n            while size>10 and d.textlength(text,font=f)>max_width:\n                size-=1;f=self._face(size)\n        w=d.textlength(text,font=f)\n        if align==\'center\':x-=w/2\n        if align==\'right\':x-=w\n        d.text((int(x),int(y)),text,font=f,fill=color,stroke_width=0)\n    def badge(self,im,club,x,y,size):\n        a=self.badges.get(club)\n        if a is None:raise FileNotFoundError(\'Official club crest required: \'+club)\n        key=(club,size)\n        if key not in self.sprite_cache:self.sprite_cache[key]=ImageOps.contain(a,(size,size),Image.Resampling.LANCZOS)\n        a=self.sprite_cache[key];im.paste(a,(int(x+(size-a.width)/2),int(y+(size-a.height)/2)),a)\n    def _face(self,size,italic=False):\n        key=(\'font\',int(size),italic)\n        if key in self.sprite_cache:return self.sprite_cache[key]\n        variant=(\'DejaVuSansCondensed-BoldOblique.ttf\' if italic else \'DejaVuSansCondensed-Bold.ttf\')\n        fallback=(\'DejaVuSans-BoldOblique.ttf\' if italic else \'DejaVuSans-Bold.ttf\')\n        candidates=[\n            Path(\'/usr/share/fonts/truetype/dejavu\')/variant,\n            Path(\'/usr/local/lib/python3.11/site-packages/matplotlib/mpl-data/fonts/ttf\')/(\'DejaVuSans-BoldOblique.ttf\' if italic else \'DejaVuSans-Bold.ttf\'),\n            Path(\'/usr/share/fonts/truetype/dejavu\')/fallback,\n        ]\n        result=None\n        for candidate in candidates:\n            if candidate.is_file():\n                try:result=ImageFont.truetype(str(candidate),int(size));break\n                except OSError:continue\n        if result is None:\n            try:result=ImageFont.truetype(variant,int(size))\n            except OSError:result=font(max(14,size))\n        self.sprite_cache[key]=result\n        return result\n    def _hex(self,value):\n        value=value.lstrip(\'#\')\n        return tuple(int(value[i:i+2],16) for i in (0,2,4))\n    def _gradient_box(self,size,left,right,radius=8):\n        key=(\'grad\',size,left,right,radius)\n        if key in self.sprite_cache:return self.sprite_cache[key]\n        w,h=size\n        base=Image.new(\'RGBA\',(w,h),(0,0,0,0))\n        ld=self._hex(left); rd=self._hex(right)\n        draw=ImageDraw.Draw(base)\n        for xx in range(w):\n            t=0 if w<=1 else xx/(w-1)\n            color=tuple(int(ld[i]*(1-t)+rd[i]*t) for i in range(3))+(255,)\n            draw.line((xx,0,xx,h),fill=color)\n        mask=Image.new(\'L\',(w,h),0)\n        ImageDraw.Draw(mask).rounded_rectangle((0,0,w-1,h-1),radius,fill=255)\n        out=Image.new(\'RGBA\',(w,h),(0,0,0,0));out.paste(base,(0,0),mask)\n        self.sprite_cache[key]=out\n        return out\n    def _fdr_palette(self,value):\n        rating=float(value)\n        if rating<=2.5:return (\'#25d66f\',\'#8bffb8\',\'#062614\')\n        if rating<3.0:return (\'#58b9ff\',\'#a9e0ff\',\'#0a2238\')\n        if rating<3.5:return (\'#c7ced6\',\'#eef2f5\',\'#1f2d38\')\n        if rating<4.3:return (\'#e2b24e\',\'#ffe08f\',\'#241502\')\n        return (\'#ff6876\',\'#ffb0b6\',\'#35060d\')\n    def _draw_cell(self,im,box,left,right,text,color,radius=7,outline=None):\n        x1,y1,x2,y2=[int(v) for v in box]\n        sprite=self._gradient_box((max(2,x2-x1),max(2,y2-y1)),left,right,radius)\n        im.paste(sprite,(x1,y1),sprite)\n        d=ImageDraw.Draw(im)\n        d.rounded_rectangle((x1,y1,x2,y2),radius,outline=outline or (255,255,255,38),width=1)\n        available_width=x2-x1-6\n        available_height=y2-y1\n        font_size=min(24,max(12,int(available_height*.46)))\n        face=self._face(font_size)\n        while font_size>9 and ImageDraw.Draw(im).textlength(str(text),font=face)>available_width:\n            font_size-=1;face=self._face(font_size)\n        self.text(im,text,(x1+x2)/2,y1+max(1,(available_height-font_size)/2-2),font_size,color,align=\'center\',max_width=available_width)\n    def _draw_fixture_cell(self,im,box,fixture,radius=6):\n        # Rendering only: keep the canonical fixture and its FDR unchanged.\n        left,right,color=self._fdr_palette(fixture[\'fdr\'])\n        self._draw_cell(im,box,left,right,\'\',color,radius)\n        opponent,rating=format_fixture(fixture).rsplit(\'-\',1)\n        x1,y1,x2,y2=box; height=y2-y1; width=x2-x1-6\n        top_size=min(21,max(10,int(height*.38)))\n        rating_size=min(26,max(11,int(height*.46)))\n        draw=ImageDraw.Draw(im)\n        top_face=self._face(top_size)\n        while top_size>10 and draw.textlength(opponent,font=top_face)>width:\n            top_size-=1;top_face=self._face(top_size)\n        rating_face=self._face(rating_size)\n        top_box=draw.textbbox((0,0),opponent,font=top_face)\n        rating_box=draw.textbbox((0,0),rating,font=rating_face)\n        top_height=top_box[3]-top_box[1];rating_height=rating_box[3]-rating_box[1]\n        gap=3 if height>=40 else 1\n        top_y=y1+(height-top_height-rating_height-gap)/2\n        center=(x1+x2)/2\n        draw.text((int(center-draw.textlength(opponent,font=top_face)/2),int(top_y-top_box[1])),opponent,font=top_face,fill=color)\n        draw.text((int(center-draw.textlength(rating,font=rating_face)/2),int(top_y+top_height+gap-rating_box[1])),rating,font=rating_face,fill=color)\n\n    def _draw_heading(self,im):\n        x1,y1,x2,y2=self.header_box\n        sprite=self._gradient_box((x2-x1,y2-y1),\'#ff566b\',\'#fffdfd\',22)\n        im.paste(sprite,(x1,y1),sprite)\n        d=ImageDraw.Draw(im)\n        d.rounded_rectangle((x1,y1,x2,y2),22,outline=\'#ffe9ee\',width=3)\n        title=\'WILDCARD FIXTURE INTELLIGENCE\'\n        title_face=self._face(34,italic=True)\n        while d.textlength(title,font=title_face)>x2-x1-70:\n            title_face=self._face(title_face.size-1,italic=True)\n        bbox=d.textbbox((0,0),title,font=title_face,stroke_width=0)\n        tx=int((x1+x2-d.textlength(title,font=title_face))/2)\n        ty=int(y1+(y2-y1-(bbox[3]-bbox[1]))/2-bbox[1])\n        d.text((tx+1,ty+1),title,font=title_face,fill=\'#ffffff\',stroke_width=0)\n        d.text((tx,ty),title,font=title_face,fill=\'#111111\',stroke_width=0)\n    def _base_background(self):\n        bg=self.background.convert(\'RGBA\')\n        tint=Image.new(\'RGBA\',bg.size,(0,0,0,0))\n        td=ImageDraw.Draw(tint)\n        # Recolor only the slide canvas: shell branding above and footer below stay unchanged.\n        for yy in range(172,1020):\n            t=(yy-172)/(1020-172)\n            color=(int(12+26*t),int(28-6*t),int(47+21*t),76)\n            td.line((0,yy,bg.width,yy),fill=color)\n        bg=Image.alpha_composite(bg,tint)\n        return bg.convert(\'RGB\')\n    def table(self,active,category):\n        key=(active,category)\n        if key in self.table_cache:return self.table_cache[key].copy()\n        im=self._base_background();d=ImageDraw.Draw(im)\n        self._draw_heading(im)\n        d.rounded_rectangle(self.table_bounds,24,fill=\'#071826\',outline=\'#37536c\',width=2)\n        for block_index,block in enumerate(self.blocks):\n            bx1,by1,bx2,by2=block\n            d.rounded_rectangle(block,18,fill=\'#0b2335\',outline=\'#2d4f64\',width=2)\n            label=\'01–10\' if block_index==0 else \'11–20\'\n            self.text(im,\'#\',bx1+10+self.rank_w/2,by1+20,20,MUTED,align=\'center\',max_width=self.rank_w)\n            x=bx1+10+self.rank_w+4+self.badge_w+4+self.team_w+4\n            team_center=bx1+10+self.rank_w+4+self.badge_w+4+self.team_w/2\n            self.text(im,\'TEAM\',team_center,by1+20,20,CYAN,align=\'center\',max_width=self.team_w)\n            for gw in self.case[\'gameweeks\']:\n                self.text(im,\'GW\'+str(gw),x+self.fixture_w/2,by1+20,20,CYAN,align=\'center\',max_width=self.fixture_w-2)\n                x+=self.fixture_w+self.col_gap\n            self.text(im,\'AVG\',x+self.avg_w/2,by1+20,20,GOLD,align=\'center\',max_width=self.avg_w-2)\n        for idx,row in enumerate(self.rows):\n            geo=self.row_geometry[row[\'team_id\']]\n            x1,y1,x2,y2=geo[\'row_box\']\n            selected=row[\'team_id\']==active\n            is_warning=category.startswith(\'avoid\') and selected\n            accent=CRIMSON if is_warning else GREEN\n            fill=\'#103248\' if idx%2==0 else \'#0e2d42\'\n            if selected:fill=\'#173f4d\' if not is_warning else \'#53313a\'\n            d.rounded_rectangle((x1,y1,x2,y2),10,fill=fill,outline=accent if selected else \'#27485c\',width=3 if selected else 1)\n            d.rounded_rectangle((x1+3,y1+9,x1+8,y2-10),3,fill=accent if selected else \'#35586b\')\n            order_text=f"{geo[\'order\']:02d}"\n            self.text(im,order_text,(geo[\'rank_box\'][0]+geo[\'rank_box\'][2])/2,y1+21,20,accent if selected else MUTED,align=\'center\')\n            bb=geo[\'badge_box\'];self.badge(im,row[\'club\'],bb[0],bb[1],bb[2]-bb[0])\n            tb=geo[\'team_box\'];self.text(im,row[\'club\'].upper(),(tb[0]+tb[2])/2,y1+19,23,WHITE if selected else \'#dcebf5\',align=\'center\',max_width=tb[2]-tb[0]-4)\n            for j,week in enumerate(row[\'weeks\']):\n                box=geo[\'fixture_boxes\'][j]\n                fixtures=week[\'fixtures\']\n                if not fixtures:\n                    self._draw_cell(im,box,\'#173446\',\'#24485d\',\'BGW\' if week.get(\'blank_confirmed\',True) else \'TBC\',MUTED,6,\'#3b5668\')\n                    continue\n                if len(fixtures)==1:\n                    f=fixtures[0];self._draw_fixture_cell(im,box,f,6)\n                else:\n                    sub_h=(box[3]-box[1]-2)/len(fixtures)\n                    for k,f in enumerate(fixtures):\n                        sub=(box[0],box[1]+k*sub_h,box[2],box[1]+(k+1)*sub_h-2)\n                        self._draw_fixture_cell(im,sub,f,5)\n            avg=row[\'average_fdr\'];a,b,txtc=self._fdr_palette(avg)\n            self._draw_cell(im,geo[\'avg_box\'],a,b,f"{avg:.2f}",txtc,7,\'#ffffff20\')\n        d.rounded_rectangle((75,977,1300,1001),7,fill=\'#152f40\')\n        self.text(im,\'ALL 20 CLUBS  •  LOWER FDR = EASIER  •  RANKED BY SIX-GAMEWEEK AVERAGE\',91,979,17,MUTED,max_width=1180)\n        self.table_cache[key]=im.copy()\n        return im\n    def render_state(self,t,duration):\n        idx=next((i for i,c in enumerate(self.timeline) if t<c[\'end\']),len(self.timeline)-1);cue=self.timeline[idx];kind=cue[\'type\'];row=self.byid.get(cue.get(\'team_id\'));u=max(0,t-cue[\'start\']);cue_length=max(.08,cue[\'end\']-cue[\'start\']);e=ease(u/max(.20,min(.7,cue_length*.40)))\n        im=self.table(cue.get(\'team_id\'),kind);d=ImageDraw.Draw(im)\n        x0,y0,x1,y1=self.stage_bounds;stage_w=x1-x0;stage_c=(x0+x1)/2\n        d.rounded_rectangle((x0,y0,x1,y1),22,fill=\'#081d2b\',outline=CRIMSON if kind.startswith(\'avoid\') else \'#365a64\',width=2)\n        def txt(text,x,y,size=40,color=WHITE,delay=0,align=\'left\',mw=None):\n            pr=ease((u-min(delay,cue_length*.15))/max(.20,min(.7,cue_length*.45)))\n            self.text(im,text,x+26*(1-pr),y+16*(1-pr),size,color,align,mw)\n        if row:\n            geometry=self.row_geometry.get(row[\'team_id\'])\n            if geometry:\n                yb=geometry[\'row_box\'];color=CRIMSON if kind.startswith(\'avoid\') else GREEN\n                d.rounded_rectangle((yb[0]-1,yb[1]-1,yb[2]+1,yb[3]+1),10,outline=color,width=2+int(2*e))\n                d.rounded_rectangle((yb[0]+16,yb[3]-6,yb[0]+16+int((yb[2]-yb[0]-40)*e),yb[3]-3),2,fill=color)\n                for fid in cue.get(\'fixture_ids\',[]):\n                    for j,week in enumerate(row[\'weeks\']):\n                        for fixture in week[\'fixtures\']:\n                            if fixture[\'fixture_id\']==fid:\n                                box=geometry[\'fixture_boxes\'][j]\n                                highlight=GREEN if fixture[\'fdr\']<=2.5 else CRIMSON if kind.startswith(\'avoid\') else GOLD\n                                d.rounded_rectangle(box,7,outline=highlight,width=2+int(2*e))\n        if kind in (\'player_name\',\'player\'):\n            p=self.players[cue[\'player_id\']];source=self.heroes[p[\'player_id\']];lift=ease(u/.8 if kind==\'player_name\' else u/1.1)\n            target_width=int(430+18*lift);target_height=int(source.height*target_width/source.width)\n            key=(p[\'player_id\'],(target_width,target_height))\n            if key not in self.sprite_cache:self.sprite_cache[key]=source.resize((target_width,target_height),Image.Resampling.LANCZOS)\n            card=self.sprite_cache[key]\n            px=int(stage_c-target_width/2);py=int(y0+67+12*(1-lift))\n            halo=Image.new(\'RGBA\',im.size,(0,0,0,0));shine=ImageDraw.Draw(halo)\n            shine.rounded_rectangle((px-8,py-8,px+target_width+8,py+target_height+8),20,fill=(31,78,86,int(120*lift)),outline=(124,233,203,int(180*lift)),width=3)\n            im.paste(halo,(0,0),halo)\n            im.paste(card,(px,py),card)\n            txt(\'VERIFIED PLAYER / SIX-GW PROJECTION\',stage_c,y0+20,22,CYAN,align=\'center\',mw=stage_w-44)\n            txt(f"{p[\'projected_points\']:.1f} xPts",stage_c,y0+729,52,\'#ffff00\',.25,\'center\',stage_w-48)\n        elif kind in (\'team\',\'verdict\',\'fixture\',\'availability\',\'avoid\',\'avoid_fixture\'):\n            color=CRIMSON if kind.startswith(\'avoid\') else GREEN\n            self.badge(im,row[\'club\'],stage_c-90,275,180)\n            txt(row[\'name\'].upper(),stage_c,480,44,WHITE,.1,\'center\',stage_w-56)\n            txt(f"{row[\'average_fdr\']*ease(u/max(.25,min(1.1,cue_length*.55))):.2f}",stage_c,556,78,GOLD if not kind.startswith(\'avoid\') else CRIMSON,.1,\'center\')\n            txt(\'AVERAGE SIX-GW FDR\',stage_c,648,25,CYAN,.25,\'center\',stage_w-70)\n            if kind in (\'fixture\',\'avoid\',\'avoid_fixture\') and cue.get(\'fixture_ids\'):\n                f=next(f for week in row[\'weeks\'] for f in week[\'fixtures\'] if f[\'fixture_id\']==cue[\'fixture_ids\'][0])\n                opp=next(r for r in self.rows if r[\'team_id\']==f[\'opponent_id\']);self.badge(im,opp[\'club\'],x0+36,748,80)\n                txt(opp[\'name\']+(\' (H)\' if f[\'home\'] else \' (A)\'),x0+130,753,30,WHITE,.25,mw=stage_w-160)\n                txt(f"GW{f[\'gameweek\']} • FDR {f[\'fdr\']:.1f}",x0+130,804,26,color,.45,mw=stage_w-160)\n            txt(\'AVOID NEW INVESTMENT\' if kind.startswith(\'avoid\') else \'TARGET • FIXTURE OPPORTUNITY\',stage_c,920,25,color,.35,\'center\',stage_w-60)\n        elif kind==\'comparison\':\n            txt(\'SAME-POSITION OPTIONS\',stage_c,270,36,GOLD,align=\'center\',mw=stage_w-60)\n            ps=[self.players[i] for i in cue[\'comparison_ids\']];maximum=max(p[\'projected_points\'] for p in ps)\n            for j,p in enumerate(ps):\n                yy=400+j*220;txt(p[\'web_name\'],x0+28,yy,40,mw=stage_w-56);txt(f"£{p[\'price\']:.1f}m • {p[\'expected_minutes\']:.0f} xMins",x0+28,yy+58,26,CYAN)\n                progress=ease((u-.2-j*.15)/1.2);d.rounded_rectangle((x0+28,yy+112,x0+28+max(2,int((stage_w-105)*p[\'projected_points\']/maximum*progress)),yy+156),10,fill=GREEN if j==0 else CYAN)\n                txt(f"{p[\'projected_points\']*progress:.1f} xPts",x1-34,yy+112,30,NAVY,align=\'right\')\n            txt(\'SIX-GAMEWEEK RETURNS • PRICE TRADE-OFF\',stage_c,925,22,MUTED,align=\'center\',mw=stage_w-40)\n        elif kind==\'caution_intro\':\n            txt(\'FIXTURE CAUTION\',stage_c,328,52,CRIMSON,align=\'center\',mw=stage_w-50)\n            txt(f"{len(self.case[\'avoids\'])} TOUGH RUNS",stage_c,425,38,WHITE,.2,\'center\')\n            for j,r in enumerate(self.case[\'avoids\']):self.badge(im,r[\'club\'],x0+62+j*156,590+int(26*(1-e)),96)\n            txt(\'REASSESS NEW INVESTMENT\',stage_c,845,29,CRIMSON,.4,\'center\',stage_w-70)\n        elif kind==\'intro\':\n            txt(f"{len(self.case[\'targets\'])} TARGETS • FULL {len(self.rows)}-TEAM BOARD",stage_c,255,34,GOLD,align=\'center\',mw=stage_w-56)\n            txt(\'ROW HIGHLIGHTS FOLLOW THE NARRATION\',stage_c,305,20,CYAN,align=\'center\',mw=stage_w-56)\n            for j,team in enumerate(self.case[\'targets\'][:3]):\n                start=.08+j*.16;progress=ease((u-start)/.55)\n                box_x=x0+34;box_y=392+j*150+int(26*(1-progress))\n                d.rounded_rectangle((box_x,box_y,box_x+stage_w-68,box_y+118),14,fill=\'#102938\',outline=\'#325361\',width=2)\n                self.badge(im,team[\'club\'],box_x+16,box_y+14,88)\n                txt(f"#{j+1}  {team[\'name\']}",box_x+120,box_y+24,28,WHITE,mw=stage_w-210)\n                txt(f"{team[\'average_fdr\']:.2f} AVG FDR",box_x+120,box_y+64,22,GREEN,mw=stage_w-210)\n            txt(\'NEXT: \'+\', \'.join(r[\'club\'] for r in self.case[\'targets\'][3:]),stage_c,920,22,MUTED,align=\'center\',mw=stage_w-46)\n        else:\n            summary=kind==\'summary\';txt(\'WILDCARD INTELLIGENCE\' if not summary else \'THE INVESTMENT VERDICT\',stage_c,280,36,GOLD,align=\'center\',mw=stage_w-56)\n            for j,r in enumerate(self.case[\'targets\']):\n                yy=355+j*(60 if summary else 82);self.badge(im,r[\'club\'],x0+26,yy,52);txt(f"{j+1}. {r[\'name\']}",x0+95,yy+8,30,WHITE,.1*j,mw=stage_w-170)\n                txt(f"{r[\'average_fdr\']:.2f}",x1-28,yy+8,30,GREEN,.1*j,\'right\')\n            if summary:\n                best=max(self.case[\'elite_players\'],key=lambda p:p[\'projected_points\']) if self.case[\'elite_players\'] else None\n                txt(f"{best[\'player_name\']} • {best[\'projected_points\']:.1f} xPts" if best else \'NO VERIFIED PLAYER SHORTLIST\',stage_c,710,30,CYAN,.4,\'center\',mw=stage_w-56)\n                txt(\'CAUTION: \'+\', \'.join(r[\'club\'] for r in self.case[\'avoids\']),stage_c,790,26,CRIMSON,.5,\'center\',mw=stage_w-56)\n            else:\n                txt(\'FIXTURES + MINUTES + VALUE\',stage_c,860,28,CYAN,.4,\'center\')\n            txt(\'NEXT: SQUAD STRUCTURE\' if summary else \'GW\'+str(self.case[\'gameweeks\'][0])+\'–\'+str(self.case[\'gameweeks\'][-1])+\' • SIX GAMEWEEKS\',stage_c,926,24,GOLD,.5,\'center\',mw=stage_w-52)\n        if len(self.sprite_cache)>260:self.sprite_cache={k:v for k,v in self.sprite_cache.items() if isinstance(k,tuple)}\n        if im.size!=(self.width,self.height):im=im.resize((self.width,self.height),Image.Resampling.LANCZOS)\n        return im\n    def draw(self,t,duration):\n        im=self.render_state(t,duration)\n        idx=next((i for i,c in enumerate(self.timeline) if t<c[\'end\']),len(self.timeline)-1)\n        cue=self.timeline[idx];u=max(0,t-cue[\'start\'])\n        transition_duration=min(.30,max(.12,(cue[\'end\']-cue[\'start\'])*.25))\n        if idx and u<transition_duration:\n            previous=self.timeline[idx-1]\n            continuation=(previous.get(\'player_id\') is not None and previous.get(\'player_id\')==cue.get(\'player_id\'))\n            if not continuation:\n                old=self.render_state(max(previous[\'start\'],previous[\'end\']-.01),duration)\n                sx=self.width/1920;sy=self.height/1080;box=tuple(int(v*(sx if i%2==0 else sy)) for i,v in enumerate(self.stage_bounds))\n                old_stage=old.crop(box);new_stage=im.crop(box);e=ease(u/transition_duration)\n                im.paste(Image.blend(old_stage,new_stage,e),box[:2])\n        return im\n    def frame(self,t,duration,previous=None):return np.asarray(self.draw(t,duration))\n\nfrom PIL import ImageOps\n\n\n# VX_PLAYWRIGHT_JUPYTER_WORKER_V1\nfrom concurrent.futures import ThreadPoolExecutor as _VxBrowserThreadPool\n_vx_original_render_hero_assets = render_hero_assets\n\ndef render_hero_assets(case, views, media, badges, brand_root, work):\n    # Playwright\'s Sync API must not run on Colab\'s active asyncio thread.\n    with _VxBrowserThreadPool(max_workers=1, thread_name_prefix=\'vortex-playwright\') as executor:\n        return executor.submit(_vx_original_render_hero_assets, case, views, media, badges, brand_root, work).result()\n'
_vx_elite_path=VX_MEDIA_WORK/'wildcard_engine'/'wildcard_fdr_elite.py'
_vx_elite_path.write_text(_VX_ELITE_FDR_SOURCE,encoding='utf-8')
Path(importlib.util.cache_from_source(str(_vx_elite_path))).unlink(missing_ok=True)
if str(_vx_elite_path.parent) not in sys.path:sys.path.insert(0,str(_vx_elite_path.parent))
import wildcard_fdr_elite as VX_ELITE_FDR
VX_ELITE_FDR=importlib.reload(VX_ELITE_FDR)
if VX_GENERATE_WILDCARD:
    def _vx_elite_slide2_scene(scene,contract):
        if scene['visual_type']!='wc_analysis':return scene
        case=VX_ELITE_FDR.build_case(contract,_bootstrap,VX_WC_ENGINE.pool.to_dict('records'))
        scene.update(elite_fixture_case=case,fixture_case=case,narration_timeline=case['narration_timeline'],narration_text=case['narration_text'],display_title='WILDCARD FIXTURE INTELLIGENCE',title=case['analysis_title'],subheading=f"GW{case['gameweeks'][0]}–GW{case['gameweeks'][-1]} • Six-Gameweek investment window")
        scene['timing'].update(status='awaiting measured Ryan phrases',player_alignment='awaiting_tts',audio_padding_seconds=0)
        return scene
    if not getattr(vx_build_manifest,'_elite_slide2',False):
        _vx_before_elite_manifest=vx_build_manifest
        def vx_build_manifest(contract):
            scenes=_vx_before_elite_manifest(contract)
            if contract.get('chip')=='wildcard':
                for scene in scenes:_vx_elite_slide2_scene(scene,contract)
            return scenes
        vx_build_manifest._elite_slide2=True
    for _vx_key,_vx_contract in VX_DATA_CONTRACTS.items():
        if not _vx_key.startswith('wildcard'):continue
        _vx_scenes=VX_SCENE_MANIFESTS[_vx_key]
        _vx_s2=next(s for s in _vx_scenes if s['visual_type']=='wc_analysis')
        _vx_elite_slide2_scene(_vx_s2,_vx_contract)
        _vx_final=VX_WILDCARD_SCENE_JSONS[_vx_key]
        _vx_final['scenes']=[copy.deepcopy(_vx_s2) if s['visual_type']=='wc_analysis' else s for s in _vx_final['scenes']]
        _vx_narration=VX_NARRATION_OBJECTS[_vx_key]
        _vx_section=next(s for s in _vx_narration['sections'] if s['visual_type']=='wc_analysis')
        _vx_section.update(narration=_vx_s2['narration_text'],source_fields=['scenes.fixture_case.rows','scenes.fixture_case.elite_players','scenes.fixture_case.gameweeks'],estimated_duration_seconds=round(len(_vx_s2['narration_text'].split())/145*60,2),pause_before_seconds=0,pause_after_seconds=0)
        _vx_narration['total_estimated_duration_seconds']=sum(s['estimated_duration_seconds'] for s in _vx_narration['sections'])
        if not VXWC.validate_scene_json(_vx_final)['passed']:raise ValueError('Wildcard Scene JSON failed after Slide 2 update')
        VXWC.validate_narration_object(_vx_final,_vx_narration)
        _vx_script=VX_NARRATION_PATHS[_vx_key]
        _vx_text=_vx_script.read_text(encoding='utf-8')
        _vx_stamp=VX_ANALYSIS_ROOT/'narration_provenance'/(_vx_script.name+'.provenance.json')
        _vx_provenance=json.loads(_vx_stamp.read_text()) if _vx_stamp.exists() else {}
        _vx_generated_before=_vx_provenance.get('generated_sha256')==hashlib.sha256(_vx_text.encode()).hexdigest()
        _vx_pattern='(\\[\\[SEGMENT {id} \\|[^\\n]*\\]\\]\\s*\\n)(.*?)(?=\\n\\[\\[SEGMENT |\\Z)'.replace('{id}',re.escape(_vx_s2['scene_id']))
        _vx_text,_vx_count=re.subn(_vx_pattern,lambda m:m.group(1)+_vx_s2['narration_text']+'\n',_vx_text,flags=re.S)
        if _vx_count!=1:raise ValueError('Could not locate exactly one Wildcard Slide 2 narration section')
        _vx_script.write_text(_vx_text,encoding='utf-8')
        if _vx_generated_before:
            _vx_stamp.write_text(json.dumps(dict(data_hash=_vx_contract['data_hash'],generated_sha256=hashlib.sha256(_vx_text.encode()).hexdigest()),indent=2))
        VXWC.validate_render_script(_vx_narration,decode_script(_vx_text))
        for _vx_name,_vx_value in [( _vx_key+'_final_scene.json',_vx_final),(_vx_key+'_narration.json',_vx_narration),(_vx_key+'_scene_manifest.json',_vx_scenes),(_vx_key+'_slide2_elite_case.json',_vx_s2['elite_fixture_case'])]:
            (VX_ANALYSIS_ROOT/_vx_name).write_text(json.dumps(_vx_value,indent=2,ensure_ascii=False),encoding='utf-8')
        (VX_ANALYSIS_ROOT/'wildcard_slide2_narration.txt').write_text(_vx_s2['narration_text'],encoding='utf-8')
        print('Wildcard Slide 2: requested script, live fixture rankings and eligible player pairs synchronized. Player cards:', len(_vx_s2['elite_fixture_case']['elite_players']), 'script words:', len(_vx_s2['narration_text'].split()))
else:
    VX_WILDCARD_SLIDE2_PREVIEW_ONLY=False
    print('Wildcard Slide 2 skipped: Free Hit workflow is unchanged.')
print('✅ CELL 13 COMPLETE')


# FREE HIT SLIDE 2 ONLY — MATCHDAY CONTROL ROOM / REMOTION
VX_FREEHIT_SLIDE2_PREVIEW_ONLY = True #@param {type:"boolean"}
import base64,io,zipfile,importlib,copy,re,hashlib
_VX_FREEHIT_MATCHDAY_SOURCE = '"""Free Hit Slide 2: one-GW evidence, measured cues and Remotion bridge."""\nfrom __future__ import annotations\nimport copy,hashlib,json,math,os,shutil,subprocess\nfrom pathlib import Path\nfrom PIL import Image\nfrom wildcard_fixture_analysis import team_statistics\nfrom wildcard_fdr_elite import format_fixture\nfrom tts_narration import CLUBS\nVERSION=\'freehit-matchday-natural-live-cues-v3\'\n\ndef build_fixture_narration(gameweek, targets, avoids, featured, captains, club_name, fixture_name):\n    """Write speech from the same rows and player cards supplied to the slide."""\n    cues = []\n    def add(identity, kind, text, row=None, player=None):\n        cues.append(dict(segment_id=identity, type=kind, narration_text=text,\n            team_id=row[\'team_id\'] if row else None,\n            player_id=player[\'player_id\'] if player else None,\n            player_ids=[player[\'player_id\']] if player else [],\n            fixture_ids=[f[\'fixture_id\'] for f in row[\'fixtures\']] if row else [],\n            start=None, end=None))\n    def count(value):\n        words = (\'no\', \'one\', \'two\', \'three\', \'four\', \'five\', \'six\', \'seven\', \'eight\', \'nine\', \'ten\')\n        return words[value] if 0 <= value < len(words) else str(value)\n    target_count = len(targets)\n    introduction = f"For Gameweek {gameweek}, I\'m looking at {count(target_count)} fixture target{\'s\' if target_count != 1 else \'\'}. "\n    introduction += "I\'ll explain the matchups, the players worth considering, and how they affect my captaincy thinking."\n    add(\'matchday_open\', \'opening\', introduction)\n    for index, row in enumerate(targets):\n        team = club_name(row)\n        lead = "I\'m starting with" if index == 0 else "Next, I\'m looking at" if index == 1 else "Another fixture I want to consider is"\n        text = f"{lead} {team}. "\n        text += \' \'.join(f"They face {fixture_name(fixture)}." for fixture in row[\'fixtures\'])\n        if row.get(\'average_fdr\') is not None:\n            label = \'average fixture difficulty\' if len(row[\'fixtures\']) > 1 else \'fixture difficulty\'\n            text += f" The {label} on the slide is {row[\'average_fdr\']:.1f}."\n        add(f\'target_{index + 1}\', \'matchup\', text, row)\n        evidence = []\n        if row.get(\'team_goals\') is not None:\n            evidence.append(f"My calculations put {team} at {row[\'team_goals\']:.2f} expected team goals.")\n        if row.get(\'cs_probability\') is not None:\n            outcome = \'at least one clean sheet\' if len(row[\'fixtures\']) > 1 else \'a clean sheet\'\n            evidence.append(f"The estimated chance of {outcome} is {row[\'cs_probability\']:.0%}.")\n        # Opponent history is linked to its actual fixture, including double Gameweeks.\n        if index < 2:\n            for fixture, recent in zip(row[\'fixtures\'], row.get(\'opponent_recent\', [])):\n                if recent and recent.get(\'matches\', 0) > 0 and recent.get(\'goals_against\') is not None:\n                    opponent = fixture_name(fixture, venue=False)\n                    evidence.append(f"{opponent} conceded {recent[\'goals_against\']:.0f} goals across their last {recent[\'matches\']} completed league matches.")\n        if evidence:\n            add(f\'evidence_{index + 1}\', \'evidence\', \' \'.join(evidence), row)\n        players = row.get(\'players\', [])\n        for player_index, player in enumerate(players):\n            name = player[\'spoken_name\']\n            if player_index == 0:\n                text = f"{name} is the first player I\'m considering here. "\n                text += f"He projects {player[\'xpts\']:.1f} points, with around {player[\'expected_minutes\']:.0f} minutes expected."\n            elif player_index == 1:\n                text = f"I also have {name} on my shortlist. "\n                text += f"His projection is {player[\'xpts\']:.1f} points over an estimated {player[\'expected_minutes\']:.0f} minutes."\n            else:\n                text = f"{name} is another option, at {player[\'xpts\']:.1f} expected points. "\n                text += f"I\'m working with a playing-time estimate of {player[\'expected_minutes\']:.0f} minutes."\n            if player.get(\'penalties\'):\n                text += " Being first in line for penalties gives him another way to score."\n            elif player[\'position\'] in (\'GK\', \'GKP\', \'DEF\'):\n                text += f"For his defensive returns, the estimated clean-sheet chance is {player[\'cs_probability\']:.0%}."\n            elif player.get(\'set_pieces\'):\n                text += " His first-choice set-piece role also gives him an assist route."\n            else:\n                text += f"His expected goals and assists add up to {player[\'xgi\']:.2f} for this Gameweek."\n            text = text.replace(\'.For\', \'. For\').replace(\'.His\', \'. His\')\n            add(f\'player_{index + 1}_{player_index + 1}\', \'strike\', text, row, player)\n        if index < 2 and len(players) > 1:\n            points_leader = max(players, key=lambda player: player[\'xpts\'])\n            value_leader = max(players, key=lambda player: player[\'value\'])\n            text = f"Of these options, {points_leader[\'spoken_name\']} has the highest points projection."\n            if points_leader[\'player_id\'] == value_leader[\'player_id\']:\n                text += " He also offers the most projected points per million."\n                text += " I\'ll weigh that against my budget and club limits when I build the squad."\n                add(f\'comparison_{index + 1}\', \'trio\', text, row, points_leader)\n            else:\n                add(f\'comparison_{index + 1}\', \'trio\', text, row, points_leader)\n                text = f"For projected points per million, {value_leader[\'spoken_name\']} comes out ahead. "\n                text += "I\'ll weigh that against my budget and club limits when I build the squad."\n                add(f\'value_{index + 1}\', \'trio\', text, row, value_leader)\n        elif not players:\n            add(f\'no_pick_{index + 1}\', \'evidence\',\n                "I don\'t have a player here who passes the availability and minutes checks. "\n                "I\'ll keep the fixture in mind without forcing a pick.", row)\n    for index, row in enumerate(reversed(avoids)):\n        team = club_name(row)\n        if not row[\'fixtures\']:\n            text = f"I\'m leaving {team} out of my one-week shortlist because they have no scheduled fixture."\n        else:\n            text = f"I\'m more cautious about {team}. "\n            text += \' \'.join(f"They face {fixture_name(fixture)}." for fixture in row[\'fixtures\'])\n            if row.get(\'average_fdr\') is not None:\n                text += f" Their fixture difficulty is {row[\'average_fdr\']:.1f}."\n            if index == 0 and row.get(\'team_goals\') is not None:\n                text += f" The team-goal estimate is {row[\'team_goals\']:.2f}."\n            elif index == 1 and row.get(\'cs_probability\') is not None:\n                outcome = \'at least one clean sheet\' if len(row[\'fixtures\']) > 1 else \'a clean sheet\'\n                text += f" The estimated chance of {outcome} is {row[\'cs_probability\']:.0%}."\n            text += " I\'d compare the stronger matchups before committing a place."\n        add(f\'threat_{index + 1}\', \'redzone\', text, row)\n    for index, player in enumerate(captains):\n        if index == 0:\n            text = f"For captaincy, {player[\'spoken_name\']} leads the eligible one-week pool at {player[\'xpts\']:.1f} expected points."\n        else:\n            text = f"{player[\'spoken_name\']} is another captaincy option, with {player[\'xpts\']:.1f} expected points."\n        text += " He passes the current starting-probability and expected-minutes checks."\n        add(f\'captain_option_{index + 1}\', \'captains\', text, player=player)\n    if len(captains) >= 2:\n        first, second = captains[:2]\n        close = abs(first[\'xpts\'] - second[\'xpts\']) < .6\n        text = "The projections are close, so this is a narrow preference. " if close else "That points me towards the highest projection. "\n        text += f"{first[\'spoken_name\']} is my leading captaincy option from this pool. "\n        text += "I\'ll confirm the armband against the final starting eleven."\n        add(\'captain_lock\', \'captain_lock\', text, player=first)\n        add(\'captain_alternative\', \'captains\',\n            f"I\'d keep {second[\'spoken_name\']} as the next option. "\n            "The vice-captain choice still needs to fit the final squad.", player=second)\n    elif captains:\n        add(\'captain_lock\', \'captain_lock\',\n            "Only one player passes the current checks. "\n            "I\'d review the minutes evidence before settling on a captain and vice-captain.", player=captains[0])\n    else:\n        add(\'captain_board\', \'captains\',\n            "I don\'t have a captaincy candidate who passes the current checks. "\n            "I\'ll review availability and expected minutes before choosing the armband.")\n    text = f"That gives me {count(target_count)} fixture target{\'s\' if target_count != 1 else \'\'} and {count(len(featured))} player option{\'s\' if len(featured) != 1 else \'\'}. "\n    text += "Next, I\'ll put the shortlist into a Free Hit squad and check what fits the budget."\n    add(\'deployment\', \'deployment\', text)\n    return cues\n\ndef build_case(contract,bootstrap,weekly,models,player_card,official_fixtures):\n    if contract[\'chip\']==\'wildcard\':raise ValueError(\'Free Hit only\')\n    gw=int(contract[\'gameweek\'])\n    if contract[\'horizon_gameweeks\']!=[gw]:raise ValueError(\'Free Hit requires one Gameweek\')\n    source=copy.deepcopy(contract[\'fixture_case\']);rows=source[\'rows\']\n    if len(rows)!=20 or len({r[\'team_id\'] for r in rows})!=20:raise ValueError(\'20 clubs required\')\n    teams={int(t[\'id\']):t for t in bootstrap[\'teams\']};official={int(p[\'id\']):p for p in bootstrap[\'elements\']}\n    weeks={int(p[\'player_id\']):p for p in weekly if int(p[\'gameweek\'])==gw}\n    pool={int(p[\'player_id\']):p for p in models};eligible=[]\n    def club(row):return CLUBS.get(row[\'club\'],teams[row[\'team_id\']][\'name\'])\n    for r in rows:\n        if len(r[\'weeks\'])!=1 or r[\'weeks\'][0][\'gameweek\']!=gw:raise ValueError(\'Future fixtures entered matchday board\')\n        fs=r[\'weeks\'][0][\'fixtures\'];r[\'fixtures\']=fs\n        if any(f[\'gameweek\']!=gw or f[\'team_id\']!=r[\'team_id\'] or teams[f[\'opponent_id\']][\'short_name\']!=f[\'opponent\'] for f in fs):raise ValueError(\'Fixture identity mismatch\')\n        for fixture in fs:\n            fixture[\'display\'] = format_fixture(fixture)\n            # Visual only: reuse the precomputed Vortex difficulty swatch when supplied.\n            if \'fdr_style\' not in fixture:\n                from vortex_wildcard import fdr_display\n                fixture[\'fdr_style\'] = fdr_display(fixture[\'fdr\'])\n        r[\'average_fdr\']=sum(f[\'fdr\'] for f in fs)/len(fs) if fs else None\n        r[\'classification\']=\'NO FIXTURE\' if not fs else \'HIGH RISK\' if round(r[\'average_fdr\'],1)>=3.5 else \'NEUTRAL\'\n        r[\'team_goals\']=sum(f[\'team_goals\'] for f in fs) if fs and all(f.get(\'team_goals\') is not None for f in fs) else None\n        r[\'cs_probability\']=1-math.prod(1-f[\'clean_sheet_probability\'] for f in fs) if fs and all(f.get(\'clean_sheet_probability\') is not None for f in fs) else None\n        r[\'recent\']=team_statistics(r[\'team_id\'],official_fixtures)\n        r[\'opponent_recent\']=[team_statistics(f[\'opponent_id\'],official_fixtures) for f in fs]\n    ranked=sorted([r for r in rows if r[\'fixtures\']],key=lambda r:(r[\'average_fdr\'],r[\'club\']))\n    targets=copy.deepcopy(ranked[:5]);blanks=[r for r in rows if not r[\'fixtures\']]\n    avoids=copy.deepcopy((blanks+list(reversed(ranked)))[:3])\n    for p in bootstrap[\'elements\']:\n        pid=int(p[\'id\']);w=weeks.get(pid);m=pool.get(pid,{})\n        if not w or p.get(\'status\')!=\'a\' or p.get(\'removed\') or not p.get(\'can_select\',True):continue\n        if p.get(\'chance_of_playing_next_round\') not in (None,100) or m.get(\'news_monitor\'):continue\n        n=int(w.get(\'fixture_count\',0))\n        if n<1 or float(w[\'expected_minutes\'])/n<65 or float(w[\'p_start\'])<.8:continue\n        points=float(w[\'gameweek_expected_points\'])\n        if not math.isfinite(points):continue\n        eligible.append(dict(player_id=pid,team_id=int(p[\'team\']),xpts=points,expected_minutes=float(w[\'expected_minutes\']),start_probability=float(w[\'p_start\']),fixture_count=n))\n    eligible.sort(key=lambda p:(-p[\'xpts\'],-p[\'expected_minutes\'],p[\'player_id\']))\n    # The board can report limited captain evidence without inventing options.\n    # Only the actual optimised XI determines the eventual captain.\n    if not eligible:\n        print(\'Free Hit matchday: no verified player shortlist; showing fixture-only evidence.\')\n    cards={}\n    def profile(p):\n        pid=p[\'player_id\']\n        if pid in cards:return cards[pid]\n        card=player_card(pid);raw=official[pid];week=next(w for w in card[\'weeks\'] if int(w[\'gameweek\'])==gw);fs=week[\'fixtures\']\n        if card[\'team_id\']!=raw[\'team\'] or abs(float(card[\'expected_points\'])-p[\'xpts\'])>1e-7 or abs(sum(f[\'xpts\'] for f in fs)-p[\'xpts\'])>1e-7:raise ValueError(\'One-GW card projection mismatch\')\n        first=str(raw.get(\'first_name\',\'\')).split()[0];last=str(raw[\'web_name\']).removeprefix(first[:1]+\'.\');name=last if first==last else first+\' \'+last\n        price=float(raw[\'now_cost\'])/float(bootstrap.get(\'game_settings\',{}).get(\'ui_currency_multiplier\',10))\n        xg=sum(f[\'expected_goals\'] for f in fs);xa=sum(f[\'expected_assists\'] for f in fs)\n        out=dict(p,name=name,spoken_name=name,club=teams[raw[\'team\']][\'short_name\'],club_name=teams[raw[\'team\']][\'name\'],position=card[\'position\'],price=price,\n            fixtures=[dict(f) for f in fs],xg=xg,xa=xa,xgi=xg+xa,cs_probability=1-math.prod(1-f[\'clean_sheet_probability\'] for f in fs),\n            penalties=raw.get(\'penalties_order\')==1,set_pieces=raw.get(\'corners_and_indirect_freekicks_order\')==1 or raw.get(\'direct_freekicks_order\')==1,\n            ownership=float(raw.get(\'selected_by_percent\') or 0),value=p[\'xpts\']/price,availability=\'AVAILABLE\',risk=\'Modelled minutes; check deadline team news\')\n        out[\'verdict\']=\'PENALTY DUTY\' if out[\'penalties\'] else \'SET-PIECE INVOLVEMENT\' if out[\'set_pieces\'] else \'CLEAN-SHEET ROUTE\' if out[\'position\'] in (\'GKP\',\'GK\',\'DEF\') else \'PROJECTED ATTACKING RETURN\'\n        out[\'designation\']=\'DIFFERENTIAL • <10% OWNED\' if out[\'ownership\']<10 else \'HIGH PROJECTION\'\n        cards[pid]=out;return out\n    featured=[];counts=[3,3,1,1,1]\n    for rank,(r,count) in enumerate(zip(targets,counts),1):\n        candidates=[p for p in eligible if p[\'team_id\']==r[\'team_id\']]\n        r[\'players\']=[profile(p) for p in candidates[:count]];r[\'rank\']=rank;r[\'classification\']=\'ATTACK\'\n        r[\'eligible_player_count\']=len(candidates)\n        featured.extend(r[\'players\'])\n        next(row for row in rows if row[\'team_id\']==r[\'team_id\'])[\'classification\']=\'ATTACK\'\n    captains=[profile(p) for p in eligible[:3]]\n    def fixture_name(fixture, venue=True):\n        opponent = CLUBS.get(fixture[\'opponent\'], teams[fixture[\'opponent_id\']][\'name\'])\n        return opponent + (\' at home\' if fixture[\'home\'] else \' away\') if venue else opponent\n    out = build_fixture_narration(gw, targets, avoids, featured, captains, club, fixture_name)\n    text=\' \'.join(c[\'narration_text\'] for c in out)\n    event=next((event for event in bootstrap.get(\'events\',[]) if int(event[\'id\'])==gw),None)\n    return dict(version=VERSION,gameweek=gw,gameweeks=[gw],season=contract.get(\'season\') or contract.get(\'freehit_case\',{}).get(\'season\'),deadline=event.get(\'deadline_time\') if event else None,updated_at=contract.get(\'source_snapshot_at\') or contract.get(\'generated_at\'),rows=rows,targets=targets,avoids=avoids,featured_players=featured,captains=captains,players=list(cards.values()),narration_text=text,narration_timeline=out,\n        diagnostics=source.get(\'diagnostics\',{}),verification=dict(clubs=20,gameweeks=[gw],featured_counts=[len(r[\'players\']) for r in targets],featured_players=len(featured),captain_basis=\'Entire eligible next-GW pool sorted by one-GW xPts\',projection_basis=\'Existing engine one-GW scores and fixture_scores; no six-week totals\',fdr_basis=\'Existing VORTEX attacking FDR; defensive CS model displayed separately\',tracking_data_used=False,final_xi_fabricated=False,word_count=len(text.split())))\n\ndef prepare_project(scene,project,media,badges,brand_root,audio=None,width=854,height=480,fps=24,duration=None,views=None):\n    project=Path(project);public=project/\'public\';public.mkdir(parents=True,exist_ok=True)\n    data=copy.deepcopy(scene[\'matchday_case\']);data[\'timeline\']=copy.deepcopy(scene[\'narration_timeline\']);data.update(width=width,height=height,fps=fps,audio=\'\',duration=duration or (data[\'timeline\'][-1].get(\'end\') or 210))\n    for row in data[\'rows\']:\n        src=Path((badges.get(row[\'club\']) or {}).get(\'path\',\'\'))\n        if not src.is_file():raise FileNotFoundError(\'Missing club crest: \'+row[\'club\'])\n        name=\'club_\'+str(row[\'team_id\'])+\'.png\';shutil.copy2(src,public/name);row[\'badge\']=name\n    badge_map={r[\'team_id\']:r[\'badge\'] for r in data[\'rows\']}\n    for row in data[\'targets\']+data[\'avoids\']:row[\'badge\']=badge_map[row[\'team_id\']]\n    hero_paths, hero_proof = render_original_heroes(data, views or CARD_VIEWS, media, badges, brand_root, project/\'hero_cache\')\n    for p in data[\'players\']:\n        m=media.get(p[\'player_id\']) or media.get(str(p[\'player_id\']))\n        if not m or m.get(\'photo_kind\')!=\'player\':raise FileNotFoundError(\'Official player cutout required: \'+p[\'name\'])\n        image=Image.open(m[\'photo\']).convert(\'RGBA\');image.thumbnail((700,700),Image.Resampling.LANCZOS)\n        alpha=image.getchannel(\'A\').getbbox()\n        if alpha:image=image.crop(alpha)\n        name=\'player_\'+str(p[\'player_id\'])+\'.png\';image.save(public/name);p[\'photo\']=name;p[\'badge\']=badge_map[p[\'team_id\']]\n        hero_name=\'hero_\'+str(p[\'player_id\'])+\'.png\';shutil.copy2(hero_paths[p[\'player_id\']],public/hero_name);p[\'hero\']=hero_name\n    for name in (\'logo1.png\',\'logo.png\',\'pl.png\'):\n        shutil.copy2(Path(brand_root)/\'assets\'/name,public/name)\n    if audio:shutil.copy2(audio,public/\'narration.wav\');data[\'audio\']=\'narration.wav\'\n    data[\'hero_card_asset_proof\']=hero_proof\n    scene[\'hero_card_asset_proof\']=hero_proof\n    if not (public/\'intro.json\').is_file():\n        (public/\'intro.json\').write_text(json.dumps(dict(width=width,height=height,fps=fps,duration=1,gameweek=data[\'gameweek\'],timeline=[dict(start=0,end=1,type=\'question\')],shortlist=[],total_points=None,audio=\'\')))\n    path=public/\'matchday.json\';path.write_text(json.dumps(data,ensure_ascii=False,indent=2));return data\n\nclass RemotionMatchdayVisual:\n    requires_motion=True\n    def __init__(self,scene,contract,width,height,project,media,badges,brand_root):\n        self.preserve_frame_edges=True\n        self.scene=scene;self.project=Path(project);self.width=width;self.height=height;self.media=media;self.badges=badges;self.brand_root=brand_root\n    def render_video(self,audio,duration,target,profile):\n        if not (math.isfinite(duration) and 45 <= duration <= 600):\n            raise ValueError(f\'Invalid measured Free Hit Slide 2 audio length: {duration!r}s\')\n        profile=dict(profile)\n        if profile[\'height\']<=480:profile.update(width=854,height=480)\n        prepare_project(self.scene,self.project,self.media,self.badges,self.brand_root,audio,profile[\'width\'],profile[\'height\'],profile[\'fps\'],duration)\n        if not (self.project/\'node_modules\'/\'remotion\').exists():subprocess.run([\'npm\',\'install\',\'--no-audit\',\'--no-fund\'],cwd=self.project,check=True)\n        command=[\'npx\',\'remotion\',\'render\',\'src/index.ts\',\'FreeHitMatchday\',str(target),\'--codec=h264\',\'--crf=\'+str(profile.get(\'crf\',28)),\'--concurrency=2\',\'--log=error\',\'--overwrite\']\n        browser=Path(\'/Applications/Google Chrome.app/Contents/MacOS/Google Chrome\')\n        if browser.exists():command.append(\'--browser-executable=\'+str(browser))\n        subprocess.run(command,cwd=self.project,check=True)\n        return Path(target)\n\n\n# VX_FREEHIT_REMOTION_SAFE_V3\n\ndef _vx_freehit_safe_render_video(self, audio, duration, target, profile):\n    import math\n    from wildcard_masterplan import _vx_remotion_cli, _vx_remotion_browser_arguments, _vx_remotion_command_output\n    if not math.isfinite(duration) or duration <= 0:\n        raise ValueError(f\'Invalid Free Hit Slide 2 narration duration: {duration!r}\')\n    profile = dict(profile)\n    if profile.get(\'height\', 0) <= 480:\n        profile.update(width=854, height=480)\n    prepare_project(self.scene, self.project, self.media, self.badges, self.brand_root, audio,\n        profile[\'width\'], profile[\'height\'], profile[\'fps\'], duration)\n    cli = _vx_remotion_cli(self.project)\n    browser_args = _vx_remotion_browser_arguments(self.project)\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    _vx_remotion_command_output(self.project, \'freehit_video\',\n        [cli, \'render\', \'src/index.ts\', \'FreeHitMatchday\', str(target),\n         \'--codec=h264\', \'--crf=\'+str(profile.get(\'crf\',28)),\n         \'--concurrency=1\', \'--timeout=60000\', \'--log=info\', \'--overwrite\', *browser_args])\n    if not target.is_file() or target.stat().st_size <= 0:\n        raise RuntimeError(\'Free Hit Remotion did not produce a video: \'+str(target))\n    return target\n\nRemotionMatchdayVisual.render_video = _vx_freehit_safe_render_video\n\nCARD_VIEWS = {}\ndef render_original_heroes(data, views, media, badges, brand_root, work):\n    """Use the unchanged notebook Hero Card HTML/CSS and snapshot-normalized views."""\n    from concurrent.futures import ThreadPoolExecutor\n    with ThreadPoolExecutor(max_workers=1, thread_name_prefix=\'freehit-original-hero\') as pool:\n        return pool.submit(_render_original_heroes, data, views, media, badges, brand_root, work).result()\n\ndef _render_original_heroes(data, views, media, badges, brand_root, work):\n    import re\n    from hero_card import card_html\n    from playwright.sync_api import sync_playwright\n    brand_root=Path(brand_root);work=Path(work);work.mkdir(parents=True,exist_ok=True)\n    css=\'\\n\'.join(re.findall(r\'<style[^>]*>(.*?)</style>\',(brand_root/\'broadcast.html\').read_text(),re.S))\n    crest_paths={r[\'team_id\']:(badges.get(r[\'club\']) or {}).get(\'path\') for r in data[\'rows\']}\n    output={};proof=[]\n    with sync_playwright() as pw:\n        options={}\n        if os.environ.get(\'VORTEX_CHROMIUM_EXECUTABLE\'): options[\'executable_path\']=os.environ[\'VORTEX_CHROMIUM_EXECUTABLE\']\n        browser=pw.chromium.launch(headless=True,**options)\n        page=browser.new_page(viewport={\'width\':485,\'height\':693},device_scale_factor=2)\n        for p in data[\'players\']:\n            pid=p[\'player_id\'];v=views.get(pid) or views.get(str(pid));m=media.get(pid) or media.get(str(pid))\n            if not v or v[\'teamId\']!=p[\'team_id\'] or abs(v[\'price\']-p[\'price\'])>.001:\n                raise ValueError(\'Original Hero Card view does not match the Free Hit snapshot: \'+str(pid))\n            if not m or m.get(\'photo_kind\')!=\'player\':raise FileNotFoundError(\'Official player photograph required: \'+p[\'name\'])\n            view=copy.deepcopy(v)\n            canonical=[f for r in data[\'rows\'] if r[\'team_id\']==p[\'team_id\'] for f in r[\'fixtures\']]\n            for fixture in view.get(\'fixtures\',[]):\n                match=next((f for f in canonical if f[\'opponent\']==fixture.get(\'opponent\') and f.get(\'home\')==fixture.get(\'home\')),None)\n                if match:fixture[\'fdr\']=match[\'fdr\']\n            raw=card_html(view,dict(logo=brand_root/\'assets/logo.png\',photo=m[\'photo\'],club_badge=(badges.get(p[\'club\']) or {}).get(\'path\'),badges=crest_paths),season=data.get(\'season\'),horizon=1)\n            digest=hashlib.sha256((raw+css).encode()).hexdigest()[:16];path=work/f\'hero_{pid}_{digest}.png\'\n            if not path.is_file():\n                source=work/f\'hero_{pid}_{digest}.html\'\n                source.write_text(\'<html><head><meta charset="utf-8"><style>\'+css+\'\\nhtml,body{margin:0;width:485px;height:693px;background:transparent;overflow:hidden}.hero-card{width:485px;height:693px}</style></head><body>\'+raw+\'</body></html>\')\n                page.goto(source.resolve().as_uri());page.wait_for_function(\'Array.from(document.images).every(i=>i.complete)\');page.evaluate(\'document.fonts.ready\')\n                page.evaluate("() => {const el=document.querySelector(\'.hero-name\');let size=parseFloat(getComputedStyle(el).fontSize);while(el.scrollWidth>el.clientWidth && size>18){el.style.fontSize=(--size)+\'px\';}}")\n                if not page.evaluate(\'Array.from(document.images).every(i=>i.naturalWidth>0)\'):raise ValueError(\'Hero Card has a missing image\')\n                page.screenshot(path=str(path),omit_background=True)\n            output[pid]=path\n            proof.append(dict(player_id=pid,path=str(path),component=\'Existing hero_card.card_html + existing broadcast.html CSS\',html_css_sha256=digest,price=p[\'price\'],projected_points=p[\'xpts\']))\n        browser.close()\n    return output,proof\n\ndef prepare_intro(scene,contract,project,media,background,audio=None,width=854,height=480,fps=5,duration=None):\n    project=Path(project);public=project/\'public\';public.mkdir(parents=True,exist_ok=True)\n    shortlist=[]\n    for player in contract[\'squad\']:\n        if not player.get(\'is_starting_xi\'):continue\n        points=float(player[\'expected_points\'])\n        if not math.isfinite(points) or points<0:continue\n        shortlist.append(dict(id=int(player[\'id\']),name=str(player.get(\'name\') or player.get(\'web_name\') or player[\'id\']),club=str(player.get(\'club\') or player.get(\'team_short\') or \'\'),role=str(player.get(\'position\') or \'\'),points=points))\n    shortlist=sorted(shortlist,key=lambda p:(-p[\'points\'],p[\'id\']))[:4]\n    for player in shortlist:\n        record=media.get(player[\'id\']) or media.get(str(player[\'id\'])) or {}\n        if record.get(\'photo_kind\')!=\'player\':raise FileNotFoundError(\'Official intro player photo missing: \'+player[\'name\'])\n        name=\'intro_player_\'+str(player[\'id\'])+\'.png\';shutil.copy2(record[\'photo\'],public/name);player[\'photo\']=name\n    background.convert(\'RGB\').resize((1920,1080),Image.Resampling.LANCZOS).save(public/\'intro_shell.png\')\n    data=dict(gameweek=contract[\'gameweek\'],timeline=scene[\'narration_timeline\'],shortlist=shortlist,total_points=scene[\'freehit_case\'].get(\'total_points\'),width=width,height=height,fps=fps,duration=duration or scene[\'narration_timeline\'][-1][\'end\'],audio=\'\')\n    if audio:shutil.copy2(audio,public/\'intro_narration.wav\');data[\'audio\']=\'intro_narration.wav\'\n    (public/\'intro.json\').write_text(json.dumps(data,indent=2))\n    if not (public/\'matchday.json\').is_file():\n        (public/\'matchday.json\').write_text(json.dumps(dict(width=width,height=height,fps=fps,duration=data[\'duration\'],gameweek=contract[\'gameweek\'],rows=[],players=[],timeline=[dict(start=0,end=data[\'duration\'],type=\'opening\',team_id=None,player_ids=[])],targets=[],avoids=[],featured_players=[],captains=[],audio=\'\',deadline=None,updated_at=None)))\n    return data\n\nclass RemotionFreeHitIntroVisual:\n    requires_motion=True\n    preserve_frame_edges=True\n    def __init__(self,fallback,scene,contract,project,media,background):\n        self.fallback=fallback;self.scene=scene;self.manifest=scene;self.contract=contract;self.project=Path(project);self.media=media;self.background=background;self.width=fallback.width;self.height=fallback.height;self.complete=fallback.complete\n    def draw(self,seconds,duration):return self.fallback.draw(seconds,duration)\n    def frame(self,seconds,duration,previous=None):return self.fallback.frame(seconds,duration,previous)\n    def render_video(self,audio,duration,target,profile):\n        from wildcard_masterplan import _vx_remotion_cli,_vx_remotion_browser_arguments,_vx_remotion_command_output\n        prepare_intro(self.scene,self.contract,self.project,self.media,self.background,audio,profile[\'width\'],profile[\'height\'],profile[\'fps\'],duration)\n        cli=_vx_remotion_cli(self.project);args=_vx_remotion_browser_arguments(self.project);target=Path(target);target.parent.mkdir(parents=True,exist_ok=True)\n        _vx_remotion_command_output(self.project,\'freehit_intro_video\',[cli,\'render\',\'src/index.ts\',\'FreeHitIntro\',str(target),\'--codec=h264\',\'--crf=\'+str(profile.get(\'crf\',28)),\'--concurrency=1\',\'--timeout=60000\',\'--log=info\',\'--overwrite\',*args])\n        if not target.is_file() or target.stat().st_size<=0:raise RuntimeError(\'Free Hit intro video missing\')\n        return target\n'
_VX_FREEHIT_REMOTION_BUNDLE = '...'
_vx_fh_module=VX_MEDIA_WORK/'wildcard_engine'/'freehit_matchday.py'
_vx_fh_module.write_text(_VX_FREEHIT_MATCHDAY_SOURCE)
Path(importlib.util.cache_from_source(str(_vx_fh_module))).unlink(missing_ok=True)
import freehit_matchday as VX_FREEHIT_MATCHDAY
VX_FREEHIT_MATCHDAY=importlib.reload(VX_FREEHIT_MATCHDAY)
VX_FREEHIT_MATCHDAY_PROJECT=VX_MEDIA_WORK/'freehit_matchday_remotion'
VX_FREEHIT_MATCHDAY_PROJECT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_VX_FREEHIT_REMOTION_BUNDLE))) as _vx_fh_zip:
    _vx_fh_zip.extractall(VX_FREEHIT_MATCHDAY_PROJECT)
if VX_GENERATE_FREEHIT:
    def _vx_matchday_scene(scene,contract):
        if scene['visual_type']!='fixture_heatmap' or contract.get('chip')=='wildcard':return scene
        case=VX_FREEHIT_MATCHDAY.build_case(contract,_bootstrap,_VX_GW_SCORES.to_dict('records'),VX_WC_ENGINE.pool.to_dict('records'),VX_WC_ENGINE.player_card,VX_WC_ENGINE.fixtures)
        scene.update(matchday_case=case,fixture_case=case,narration_text=case['narration_text'],narration_timeline=case['narration_timeline'],title='Free Hit Matchday Intelligence',display_title='MATCHDAY CONTROL ROOM',subheading='GW'+str(case['gameweek'])+' only')
        scene['matchday_render_source_hash']=hashlib.sha256((_VX_FREEHIT_MATCHDAY_SOURCE+_VX_FREEHIT_REMOTION_BUNDLE).encode()).hexdigest()
        scene['timing'].update(status='awaiting measured Ryan phrases',player_alignment='awaiting_tts',audio_padding_seconds=0)
        return scene
    if not getattr(vx_build_manifest,'_freehit_matchday',False):
        _vx_manifest_before_matchday=vx_build_manifest
        def vx_build_manifest(contract,_build=_vx_manifest_before_matchday):
            scenes=_build(contract)
            if contract.get('chip')!='wildcard':
                for scene in scenes:_vx_matchday_scene(scene,contract)
            return scenes
        vx_build_manifest._freehit_matchday=True
    for _vx_fh_key,_vx_fh_contract in VX_DATA_CONTRACTS.items():
        if not _vx_fh_key.startswith('free_hit'):continue
        _vx_fh_scenes=VX_SCENE_MANIFESTS[_vx_fh_key]
        _vx_fh_s2=next(s for s in _vx_fh_scenes if s['visual_type']=='fixture_heatmap')
        _vx_matchday_scene(_vx_fh_s2,_vx_fh_contract)
        for _vx_fh_segment in VX_STORY_GENERATED[_vx_fh_key]:
            if _vx_fh_segment['id']==_vx_fh_s2['scene_id']:_vx_fh_segment['text']=_vx_fh_s2['narration_text']
        _vx_fh_script=VX_NARRATION_PATHS[_vx_fh_key];_vx_fh_text=_vx_fh_script.read_text()
        _vx_fh_stamp=VX_ANALYSIS_ROOT/'narration_provenance'/(_vx_fh_script.name+'.provenance.json')
        _vx_fh_provenance=json.loads(_vx_fh_stamp.read_text()) if _vx_fh_stamp.exists() else {}
        _vx_fh_generated=_vx_fh_provenance.get('generated_sha256')==hashlib.sha256(_vx_fh_text.encode()).hexdigest()
        _vx_fh_pattern='(\\[\\[SEGMENT {id} \\|[^\\n]*\\]\\]\\s*\\n)(.*?)(?=\\n\\[\\[SEGMENT |\\Z)'.replace('{id}',re.escape(_vx_fh_s2['scene_id']))
        _vx_fh_text,_vx_fh_count=re.subn(_vx_fh_pattern,lambda m:m.group(1)+_vx_fh_s2['narration_text']+'\n',_vx_fh_text,flags=re.S)
        if _vx_fh_count!=1:raise ValueError('Expected exactly one Free Hit Slide 2 narration header')
        _vx_fh_script.write_text(_vx_fh_text)
        if _vx_fh_generated:_vx_fh_stamp.write_text(json.dumps(dict(data_hash=_vx_fh_contract['data_hash'],generated_sha256=hashlib.sha256(_vx_fh_text.encode()).hexdigest()),indent=2))
        VX_NARRATION[_vx_fh_key]=decode_script(_vx_fh_text)
        (VX_NARRATION_OUT_DIR/(_vx_fh_key+'_'+_vx_fh_s2['scene_id']+'.txt')).write_text(_vx_fh_s2['narration_text']+'\n')
        for _vx_fh_name,_vx_fh_value in [(_vx_fh_key+'_scene_manifest.json',_vx_fh_scenes),(_vx_fh_key+'_slide2_matchday_case.json',_vx_fh_s2['matchday_case'])]:
            (VX_ANALYSIS_ROOT/_vx_fh_name).write_text(json.dumps(_vx_fh_value,ensure_ascii=False,indent=2))
        print('Free Hit Slide 2: live fixture narration and measured player cues synchronized; player options:', len(_vx_fh_s2['matchday_case']['featured_players']), 'words:', len(_vx_fh_s2['narration_text'].split()), flush=True)
else:
    VX_FREEHIT_SLIDE2_PREVIEW_ONLY=False


_VX_MASTERPLAN_SOURCE = '"""Wildcard Slide 3: contract-driven squad, reconciled plan and continuous Hero flights."""\nfrom __future__ import annotations\nimport copy,json,math,re,shutil,subprocess,hashlib\nfrom collections import Counter\nfrom pathlib import Path\nfrom PIL import Image\nfrom tts_narration import CLUBS\nVERSION=\'wildcard-masterplan-v7-requested-squad-script\'\n\ndef build_squad_narration(gws, opening, bench, players, weeks, plan, contract):\n    """The requested Wildcard team story, with one live card per player cue."""\n    out = []\n    def money(value): return f\'{float(value):.1f}\'.rstrip(\'0\').rstrip(\'.\')\n    def count(value): return {1:\'one\',2:\'two\',3:\'three\',4:\'four\',5:\'five\',6:\'six\',7:\'seven\',8:\'eight\'}.get(value,str(value))\n    def add(identity, kind, text, pid=None, week=None):\n        ids = [pid] if pid is not None else []\n        if week:\n            ids = list(dict.fromkeys([week[\'captain_id\'], week[\'vice_id\']] + [identity for move in week[\'moves\'] for identity in (move[\'out_id\'],move[\'in_id\'])]))\n        out.append(dict(segment_id=identity,type=kind,narration_text=text,player_id=pid,player_ids=ids,\n            gameweek=week[\'gameweek\'] if week else None,start=None,end=None))\n    groups = {position:[pid for pid in opening if players[pid][\'position\']==position] for position in (\'GK\',\'DEF\',\'MID\',\'FWD\')}\n    if len(opening)!=11 or len(bench)!=4 or len(set(opening+bench))!=15:\n        raise ValueError(\'Slide 3 speech must cover exactly the selected eleven and four substitutes\')\n    formation = \'-\'.join(str(len(groups[position])) for position in (\'DEF\',\'MID\',\'FWD\'))\n    if formation != weeks[0][\'formation\']:\n        raise ValueError(\'Spoken formation differs from the opening pitch\')\n    for pid in opening+bench:\n        player=players[pid]\n        if player.get(\'horizon_xpts\') is None or not math.isfinite(float(player[\'horizon_xpts\'])):\n            raise ValueError(\'Missing displayed full-window player projection\')\n    cover = [players[pid] for pid in bench if (players[pid].get(\'p_start\') or 0)>=.65 and (players[pid].get(\'minutes\') or 0)>0]\n    cover_text = \'keeping enough cover on the bench\' if len(cover)>=2 else \'keeping reserve options on the bench\'\n    add(\'opening\',\'opening\',f"Right, here\'s the Wildcard team I\'ve put together for Gameweek {gws[0]}. "\n        f"I\'m going with a {formation} formation, focusing heavily on the starting eleven while {cover_text}.")\n    for index,pid in enumerate(groups[\'GK\']):\n        player=players[pid]\n        minutes=player.get(\'minutes\')\n        text=f"Let\'s start in goal. I\'ve gone with {player[\'spoken_name\']} at £{money(player[\'price\'])} million. "\n        text+=(f"He\'s projected for around {minutes:.0f} minutes in the opening Gameweek, " if minutes is not None else "I\'ll check his opening-Gameweek minutes, ")\n        text+="and I\'m looking for steady returns through saves and clean sheets."\n        add(\'player_\'+str(pid),\'players\',text,pid)\n    defenders=groups[\'DEF\']\n    for index,pid in enumerate(defenders):\n        player=players[pid];name=player[\'spoken_name\'];price=money(player[\'price\']);points=player[\'horizon_xpts\']\n        if index==0:\n            text=f"Moving into defence, {name} comes in at £{price} million. He\'s one of my favourite value options, "\n            text+=(f"with strong projected minutes and a {count(len(gws))}-Gameweek projection of {points:.1f} points." if (player.get(\'minutes\') or 0)>=75 else\n                f"with a {count(len(gws))}-Gameweek projection of {points:.1f} points. His playing time needs a check.")\n        else:\n            prefix="Alongside him, I\'ve got " if index==1 else \'and \' if index==len(defenders)-1 else \'\'\n            ending=\'.\' if index==len(defenders)-1 else \',\' if index<len(defenders)-2 else \' and\'\n            # Each name/price is its own measured cue; punctuation joins the list.\n            if index==len(defenders)-1 and index>1 and out[-1][\'narration_text\'].endswith(\' and\'):\n                prefix=\'\'\n            text=f"{prefix}{name} at £{price} million{ending}"\n            if index==len(defenders)-1:\n                remaining=[players[identity] for identity in defenders[1:]]\n                attacking=all((candidate.get(\'xgi90\') or (candidate.get(\'defender_evidence\') or {}).get(\'xgi90\') or 0)>0 for candidate in remaining)\n                if attacking:\n                    text+=f" {\'Both\' if len(remaining)==2 else \'They\'} offer interesting attacking potential from defence, with {player[\'name\']} projecting {points:.1f} points across the {count(len(gws))} weeks."\n                else:\n                    text+=f" That completes the back line, with {player[\'name\']} projecting {points:.1f} points across the {count(len(gws))} weeks."\n        add(\'player_\'+str(pid),\'players\',text,pid)\n    midfield=groups[\'MID\']\n    midfield_cost=sum(players[pid][\'price\'] for pid in midfield)\n    heavy=midfield_cost>=max(sum(players[pid][\'price\'] for pid in groups[position]) for position in (\'DEF\',\'FWD\'))\n    for index,pid in enumerate(midfield):\n        player=players[pid];name=player[\'spoken_name\'];price=money(player[\'price\']);points=player[\'horizon_xpts\']\n        if index==0:\n            text="Now, midfield is where I\'ve invested quite heavily. " if heavy else "Now, let\'s look at the midfield. "\n            text+=f"{name} is my premium pick at £{price} million. He\'s projected for {points:.1f} points across the {count(len(gws))} Gameweeks, "\n            text+=("and his attacking involvement makes him a key part of this team." if (player.get(\'xgi90\') or 0)>0 or player.get(\'penalties\') or player.get(\'set_pieces\') else\n                "and his projected minutes and fixtures make him a key part of this team.")\n        elif index==1:\n            text=f\'Next is {name} at £{price} million, with {points:.1f} projected points.\'\n        else:\n            prefix="I\'ve also included " if index==2 else \'and \' if index==len(midfield)-1 else \'\'\n            ending=\'.\' if index==len(midfield)-1 else \',\'\n            text=f\'{prefix}{name} at £{price} million{ending}\'\n        if index==len(midfield)-1:\n            attacking=all((players[identity].get(\'xgi90\') or 0)>0 or players[identity].get(\'penalties\') or players[identity].get(\'set_pieces\') for identity in midfield)\n            text+=f" That gives me {count(len(midfield))} midfielders"+(\' with attacking potential\' if attacking else \' across the midfield\')\n            text+=\', rather than relying on just one or two players to deliver.\' if len(midfield)>2 else \'.\'\n        add(\'player_\'+str(pid),\'players\',text,pid)\n    forwards=groups[\'FWD\']\n    for index,pid in enumerate(forwards):\n        player=players[pid];name=player[\'spoken_name\'];price=money(player[\'price\']);points=player[\'horizon_xpts\']\n        if index==0:\n            text=f"Up front, there\'s one name I simply don\'t want to leave out. {name}. "\n            expensive=player[\'price\']==max(players[identity][\'price\'] for identity in opening)\n            text+=(f"Yes, he\'s expensive at £{price} million, but with " if expensive else f"At £{price} million, with ")\n            text+=f"{points:.1f} projected points across the {count(len(gws))} weeks, he\'s my main attacking investment"\n            text+=\' and opening captain.\' if pid==weeks[0][\'captain_id\'] else \'.\'\n        else:\n            text=f"{\'Partnering him is\' if index==1 else \'Completing the attack is\'} {name} at £{price} million. "\n            text+=f"He\'s projected for {points:.1f} points, and I really like the balance he brings alongside {players[forwards[0]][\'name\']}."\n        add(\'player_\'+str(pid),\'players\',text,pid)\n    for index,pid in enumerate(bench):\n        player=players[pid]\n        prefix="Now, let\'s talk about the bench. I\'ve gone with " if index==0 else \'and \' if index==len(bench)-1 else \'\'\n        ending=\'.\' if index==len(bench)-1 else \',\'\n        add(\'bench_\'+str(pid),\'bench\',f"{prefix}{player[\'spoken_name\']} at £{money(player[\'price\'])} million{ending}",pid)\n    status=contract.get(\'bench_boost\',\'Unknown\')\n    planned=contract.get(\'bench_boost_planned_gameweek\')\n    if planned is None:planned=contract.get(\'model_audit\',{}).get(\'bench_boost_planned_gameweek\')\n    immediate=status==\'Available\' and planned==gws[0]\n    text=("This draft includes an immediate Bench Boost plan. I\'ll check the minutes of all fifteen players before committing." if immediate else\n        "This isn\'t a squad built for an immediate Bench Boost. The priority is getting the strongest possible starting eleven while keeping some cover available.")\n    text+=f" And importantly, the draft leaves £{money(plan[\'bank\'])} million in the bank."\n    add(\'bench_strategy\',\'overview\',text)\n    add(\'captaincy_preview\',\'overview\',\'Looking ahead, the captaincy plan is fairly straightforward.\')\n    seen=set()\n    for index,week in enumerate(weeks):\n        captain=players[week[\'captain_id\']];name=captain[\'name\'];opponents=week[\'captain_opponents\'];gw=week[\'gameweek\']\n        previous=weeks[index-1][\'captain_id\'] if index else None\n        if index==0:text=f\'{name} gets the armband in Gameweek {gw} against {opponents}.\'\n        elif index==1:text=f"{name} {\'takes over\' if week[\'captain_id\']!=previous else \'keeps the armband\'} in Gameweek {gw} against {opponents}."\n        elif index==2 and week[\'captain_id\'] in seen and week[\'captain_id\']!=previous:text=f\'We return to {name} for {opponents} in Gameweek {gw},\'\n        elif index==3 and week[\'captain_id\']!=previous:text=f\'before {name} gets {opponents} in Gameweek {gw}.\'\n        elif index==4:text=f"Then it\'s {name} against {opponents} in Gameweek {gw},"\n        elif index==5:text=f\'followed by {name} against {opponents} in Gameweek {gw}.\'\n        else:text=f"For Gameweek {gw}, {name} {\'keeps\' if week[\'captain_id\']==previous else \'gets\'} the armband against {opponents}."\n        if index==len(weeks)-1 or week[\'moves\']:text=text.rstrip(\',\')+\'.\' if text.endswith(\',\') else text\n        if week[\'moves\']:\n            text+=\' \'+ \' \'.join(f"The current plan also lists {players[move[\'out_id\']][\'spoken_name\']} to {players[move[\'in_id\']][\'spoken_name\']} in Gameweek {gw}." for move in week[\'moves\'])\n            if week[\'hit_cost\']:text+=f" The planned hit cost is {week[\'hit_cost\']:.0f} points."\n        add(\'week_\'+str(gw),\'week\',text,week=week);seen.add(week[\'captain_id\'])\n    add(\'captaincy_review\',\'overview\',\'Of course, those decisions can change depending on injuries and team news.\')\n    text="As for transfers, I\'m not planning to make changes just for the sake of it. "\n    if any(week[\'moves\'] for week in weeks):\n        text+=\'The listed moves are provisional. I would review the actual opportunity and team news before making them.\'\n    else:\n        text+="If the squad stays healthy, I\'d rather save those transfers and use them when a genuine opportunity appears."\n    add(\'transfer_policy\',\'overview\',text)\n    if status==\'Used\':text="Bench Boost has already been used, so I\'m keeping the focus on the starting eleven and reliable cover."\n    elif immediate:text="The immediate Bench Boost remains conditional on reliable minutes and the final team news."\n    else:text="And I\'m not forcing the Bench Boost either. We can strengthen the bench gradually and wait for a better opportunity."\n    add(\'bench_boost_policy\',\'overview\',text)\n    add(\'verdict\',\'verdict\',"So that\'s my Wildcard structure. Now, let\'s take a closer look at those captaincy decisions.")\n    return out\n\ndef build_case(contract,bootstrap,player_card):\n    if contract[\'chip\']!=\'wildcard\':raise ValueError(\'Wildcard only\')\n    plan=copy.deepcopy(contract[\'slide3_plan\']);gws=list(contract[\'horizon_gameweeks\'])\n    if len(gws)!=6 or gws!=list(range(gws[0],gws[0]+6)):raise ValueError(\'Six consecutive rounds required\')\n    raw={int(p[\'id\']):p for p in bootstrap[\'elements\']};teams={int(t[\'id\']):t for t in bootstrap[\'teams\']}\n    rows={int(r[\'team_id\']):r for r in contract[\'fixture_table\']};players={}\n    entities={int(k):p for k,p in contract[\'media_entities\'].items()}\n    entities.update({int(p[\'id\']):p for p in contract[\'squad\']})\n    defender_evidence={int(pid):item for pid,item in contract.get(\'defender_two_way\',{}).get(\'players\',{}).items()}\n    planplayers={int(p[\'id\']):p for p in plan[\'squad\']}\n    def profile(pid):\n        pid=int(pid)\n        if pid in players:return players[pid]\n        source=entities.get(pid) or player_card(pid);p=raw[pid];entry=planplayers.get(pid,{})\n        if source[\'team_id\']!=p[\'team\']:raise ValueError(\'Player club mismatch\')\n        price=float(p[\'now_cost\'])/bootstrap.get(\'game_settings\',{}).get(\'ui_currency_multiplier\',10)\n        if abs(price-source[\'price\'])>1e-8:raise ValueError(\'Player price mismatch\')\n        fullname=entry.get(\'spoken_name\') or (str(p[\'first_name\']).split()[0]+\' \'+str(p[\'web_name\']))\n        weekly={int(w[\'gameweek\']):copy.deepcopy(w) for w in source[\'weeks\']}\n        total=source.get(\'horizon_expected_points\')\n        if total is not None and abs(total-sum(float(weekly[g][\'xpts\']) for g in gws))>1e-7:raise ValueError(\'Player six-week projection mismatch\')\n        minutes=source.get(\'expected_minutes\');prob=source.get(\'starting_probability\')\n        analysis=entry.get(\'analysis\',{});xgi=analysis.get(\'xgi90\')\n        duty=p.get(\'penalties_order\')==1;setpiece=p.get(\'corners_and_indirect_freekicks_order\')==1 or p.get(\'direct_freekicks_order\')==1\n        pos=source[\'position\'];pos=\'GK\' if pos==\'GKP\' else pos\n        advantage=\'DEFENSIVE RETURNS\' if pos in (\'GK\',\'DEF\') else \'PENALTY DUTY\' if duty else \'SET-PIECE ROUTE\' if setpiece else \'ATTACKING INVOLVEMENT\'\n        q=dict(id=pid,player_id=pid,name=p[\'web_name\'],spoken_name=fullname,full_name=source.get(\'full_name\',fullname),position=pos,team_id=p[\'team\'],club=teams[p[\'team\']][\'short_name\'],club_name=CLUBS.get(teams[p[\'team\']][\'short_name\'],teams[p[\'team\']][\'name\']),price=price,horizon_xpts=total,minutes=minutes,p_start=prob,xgi90=xgi,penalties=duty,set_pieces=setpiece,advantage=advantage,weeks=weekly,availability=source.get(\'injury_status\') or p.get(\'status\'))\n        q[\'value\']=total/price if total is not None else None\n        q[\'defender_evidence\']=defender_evidence.get(pid) if pos==\'DEF\' else None\n        if q[\'defender_evidence\'] and q[\'defender_evidence\'][\'quadrant\']==\'DOUBLE ROUTE\':\n            q[\'advantage\']=\'TWO-WAY DEFENDER\'\n        players[pid]=q;return q\n    opening=[int(p[\'id\']) for p in plan[\'starting_xi\']]\n    if set(opening)!=set(contract[\'starting_xi_ids\']):raise ValueError(\'Opening XI differs from latest contract\')\n    midfield_ids=[pid for pid in opening if profile(pid)[\'position\']==\'MID\']\n    premium_mid=max(midfield_ids,key=lambda pid:(profile(pid)[\'price\'],profile(pid)[\'horizon_xpts\'] or 0)) if midfield_ids else None\n    opening_captain=int(plan[\'gameweek_plans\'][0][\'captain_id\'])\n    def spoken_order(pid):\n        player=profile(pid);position=player[\'position\'];points=player[\'horizon_xpts\'] or 0\n        role_rank={\'GK\':0,\'DEF\':1,\'MID\':2,\'FWD\':3}[position]\n        if position==\'DEF\':return (role_rank,0,-(player[\'value\'] or 0),-points,pid)\n        if position==\'MID\':return (role_rank,0 if pid==premium_mid else 1,-points,0,pid)\n        if position==\'FWD\':return (role_rank,0 if pid==opening_captain else 1,-points,0,pid)\n        return (role_rank,0,-points,0,pid)\n    opening=sorted(opening,key=spoken_order)\n    bench=[int(p[\'id\']) for p in plan[\'bench\']]\n    if set(bench)!=set(contract[\'bench_ids\']) or len(bench)!=4:raise ValueError(\'Bench differs from contract\')\n    for pid in opening+bench:profile(pid)\n    transfers={int(a[\'gameweek\']):copy.deepcopy(a) for a in contract.get(\'transfer_plan\',[])}\n    state=set(opening+bench);weeks=[];running=0\n    for wp in plan[\'gameweek_plans\']:\n        gw=int(wp[\'gameweek\']);action=transfers.get(gw);moves=[]\n        if gw!=gws[0] and action is None:raise ValueError(\'Missing explicit weekly transfer decision\')\n        if action and action[\'action\']==\'TRANSFER\':\n            outid=int(action[\'out_player_id\']);inid=int(action[\'in_player_id\'])\n            if outid not in state or inid in state:raise ValueError(\'Illegal transfer identity\')\n            outp=profile(outid);inp=profile(inid)\n            if outp[\'position\']!=inp[\'position\']:raise ValueError(\'Transfers must preserve squad quotas\')\n            state.remove(outid);state.add(inid)\n            if set(action[\'squad_after\'])!=state:raise ValueError(\'Transfer state mismatch\')\n            moves=[dict(out_id=outid,in_id=inid,price_delta=inp[\'price\']-outp[\'price\'],reason=action.get(\'reason\',\'Projection-led squad change\'),impact=action.get(\'xi_effect_xpts\'))]\n        elif action:\n            if action[\'action\']!=\'ROLL\' or action[\'transfers_required\']!=0 or set(action[\'squad_after\'])!=state:raise ValueError(\'Roll altered squad\')\n        xi=[int(p[\'id\']) for p in wp[\'starting_xi\']]\n        if len(xi)!=11 or len(set(xi))!=11 or not set(xi)<=state or len(state)!=15:raise ValueError(\'Invalid weekly XI after transfers\')\n        for pid in state:profile(pid)\n        counts=Counter(profile(pid)[\'position\'] for pid in xi)\n        if counts[\'GK\']!=1 or not 3<=counts[\'DEF\']<=5 or not 2<=counts[\'MID\']<=5 or not 1<=counts[\'FWD\']<=3:raise ValueError(\'Illegal weekly formation\')\n        if Counter(profile(pid)[\'position\'] for pid in state)!=dict(GK=2,DEF=5,MID=5,FWD=3):raise ValueError(\'Illegal squad quotas\')\n        if max(Counter(profile(pid)[\'team_id\'] for pid in state).values())>3:raise ValueError(\'Club limit exceeded\')\n        cap=int(wp[\'captain_id\']);vice=int(wp[\'vice_captain_id\'])\n        if cap==vice or cap not in xi or vice not in xi:raise ValueError(\'Captain or vice not in XI\')\n        sx=0\n        for pp in wp[\'starting_xi\']:\n            v=profile(pp[\'id\'])[\'weeks\'][gw][\'xpts\']\n            if v is None or not math.isfinite(v):raise ValueError(\'Missing weekly player projection\')\n            if abs(float(pp[\'xpts\'])-float(v))>1e-7:raise ValueError(\'Plan/model weekly projection mismatch\')\n            sx+=float(v)\n        bonus=float(profile(cap)[\'weeks\'][gw][\'xpts\']);gross=sx+bonus;hit=float(action.get(\'hit_cost\',0)) if action else 0\n        if abs(sx-wp[\'xi_xpts\'])>1e-7 or abs(gross-wp[\'captain_adjusted_xpts\'])>1e-7:raise ValueError(\'Weekly captaincy total mismatch\')\n        net=gross-hit;running+=net\n        # Reference transfer counts are expressly assumed in the engine: do not publish them.\n        ft=None\n        if action and contract.get(\'free_transfers\') is not None and \'unknown\' not in str(action.get(\'free_transfer_source\',\'\')).lower() and \'reference\' not in str(action.get(\'free_transfer_source\',\'\')).lower():ft=action.get(\'free_transfers_after\')\n        formation=f"{counts[\'DEF\']}-{counts[\'MID\']}-{counts[\'FWD\']}"\n        captain=profile(cap);fixtures=captain[\'weeks\'][gw][\'fixtures\']\n        fixture=\' and \'.join(CLUBS.get(f[\'opponent\'],teams[f[\'opponent_id\']][\'name\'])+(\' at home\' if f[\'home\'] else \' away\') for f in fixtures) or \'a blank Gameweek\'\n        opponents=\' and \'.join(CLUBS.get(f[\'opponent\'],teams[f[\'opponent_id\']][\'name\']) for f in fixtures) or \'a blank Gameweek\'\n        remaining=sorted((pid for pid in state if pid not in xi),key=lambda pid:(profile(pid)[\'position\']==\'GK\',-(profile(pid)[\'weeks\'][gw][\'xpts\'] or 0),pid))\n        if len(remaining)!=4 or set(remaining)&set(xi) or set(remaining)|set(xi)!=state:raise ValueError(\'Slide 3 weekly XI/bench must partition exactly one squad\')\n        weeks.append(dict(gameweek=gw,xi_ids=xi,bench_ids=remaining,captain_id=cap,vice_id=vice,formation=formation,xi_xpts=sx,captain_bonus=bonus,gross_xpts=gross,hit_cost=hit,net_xpts=net,cumulative_xpts=running,action=\'WILDCARD ACTIVE\' if gw==gws[0] else \'TRANSFER\' if moves else \'ROLL\',moves=moves,free_transfers=ft,captain_fixture=fixture,captain_opponents=opponents,squad_ids=sorted(state)))\n    if [w[\'gameweek\'] for w in weeks]!=gws:raise ValueError(\'Roadmap window mismatch\')\n    formation=weeks[0][\'formation\']\n    out=build_squad_narration(gws,opening,bench,players,weeks,plan,contract)\n    text=\' \'.join(c[\'narration_text\'] for c in out)\n    return dict(version=VERSION,season=contract[\'season\'],gameweeks=gws,opening_ids=opening,bench_ids=bench,formation=formation,players=list(players.values()),weeks=weeks,total_xpts=running,bank=plan[\'bank\'],budget=plan[\'budget\'],total_cost=plan[\'total_cost\'],timeline=out,narration_text=text,narration_timeline=out,verification=dict(source_hash=contract[\'data_hash\'],opening_xi_matches=True,weekly_xi_legal=True,transfer_states_verified=True,weekly_captaincy_verified=True,projection_basis=\'Projected weekly XI fixture xPts plus one captain bonus minus applicable transfer hits\',total_xpts=running,weekly_net_xpts=[w[\'net_xpts\'] for w in weeks],banked_transfers_published=any(w[\'free_transfers\'] is not None for w in weeks),word_count=len(text.split())))\n\ndef hero_assets(case,views,media,badges,brand_root,public):\n    from hero_card import card_html\n    from playwright.sync_api import sync_playwright\n    css=\'\\n\'.join(re.findall(r\'<style[^>]*>(.*?)</style>\',(Path(brand_root)/\'broadcast.html\').read_text(),re.S))\n    badges_by_id={p[\'team_id\']:(badges.get(p[\'club\']) or {}).get(\'path\') for p in case[\'players\']}\n    # Include all opponent crests, including clubs outside the selected squad.\n    for view in views.values():\n        for f in view.get(\'fixtures\',[]):\n            code=f.get(\'opponent\');m=badges.get(code) or {}\n            if f.get(\'opponentId\') and m.get(\'path\'):badges_by_id[f[\'opponentId\']]=m[\'path\']\n    with sync_playwright() as pw:\n        browser=pw.chromium.launch(headless=True);page=browser.new_page(viewport=dict(width=485,height=693),device_scale_factor=2)\n        for p in case[\'players\']:\n            pid=p[\'id\'];m=media.get(pid) or media.get(str(pid));view=views.get(pid) or views.get(str(pid))\n            if not m or m.get(\'photo_kind\')!=\'player\':raise FileNotFoundError(\'Official player image required: \'+p[\'name\'])\n            if pid not in set(case[\'opening_ids\'])|set(case[\'bench_ids\']):continue\n            if not view and pid in case[\'bench_ids\']:\n                from PIL import ImageDraw,ImageFont\n                base=Image.new(\'RGBA\',(485,693),(5,27,22,242));draw=ImageDraw.Draw(base)\n                draw.rounded_rectangle((5,5,480,688),radius=26,fill=(8,41,32,250),outline=(103,220,172,240),width=3)\n                draw.rounded_rectangle((25,23,460,86),radius=14,fill=(28,83,67,255))\n                def fallback_font(size):\n                    try:return ImageFont.truetype(\'/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf\',size)\n                    except OSError:return ImageFont.load_default()\n                draw.text((44,35),\'FPL VORTEX / BENCH COVER\',font=fallback_font(21),fill=(222,252,227,255))\n                draw.rounded_rectangle((71,134,414,460),radius=24,fill=(24,72,58,255),outline=(100,179,143,190),width=2)\n                draw.rounded_rectangle((24,510,462,591),radius=13,fill=(22,79,59,255))\n                draw.text((38,604),f"{p[\'position\']}  ·  £{p[\'price\']:.1f}M",font=fallback_font(28),fill=(244,215,120,255))\n                draw.text((37,650),\'MODELLED MINUTES / RESERVE\',font=fallback_font(17),fill=(190,221,204,255))\n                p[\'hero\']=\'hero_\'+str(pid)+\'.png\'\n                p[\'hero_geometry\']=dict(photo=dict(x=108,y=130,width=269,height=328),name=dict(x=60,y=530,width=365,height=55,fontSize=34,fontFamily=\'Arial\',color=\'#f2faf6\'))\n                base.save(public/p[\'hero\'])\n                base.save(public/f"hero_full_{pid}.png")\n                continue\n            if not view or view[\'teamId\']!=p[\'team_id\'] or abs(view[\'price\']-p[\'price\'])>.001:raise ValueError(\'Existing Hero Card identity mismatch\')\n            view=copy.deepcopy(view)\n            for f in view.get(\'fixtures\',[]):\n                canonical=next((a for w in p[\'weeks\'].values() for a in w[\'fixtures\'] if a[\'opponent\']==f.get(\'opponent\') and a[\'home\']==f.get(\'home\')),None)\n                if canonical:f[\'fdr\']=canonical[\'fdr\']\n            html=card_html(view,dict(logo=Path(brand_root)/\'assets/logo.png\',photo=m[\'photo\'],club_badge=(badges.get(p[\'club\']) or {}).get(\'path\'),badges=badges_by_id),season=case[\'season\'],horizon=len(case[\'gameweeks\']))\n            doc=\'<html><head><meta charset="utf-8"><style>\'+css+\'\\nhtml,body{margin:0;width:485px;height:693px;background:transparent;overflow:hidden}.hero-card{width:485px;height:693px}</style></head><body>\'+html+\'</body></html>\'\n            # Reuse exact hero renders when narration changes but the design, player and asset bytes do not.\n            photo_digest=hashlib.sha256(Path(m[\'photo\']).read_bytes()).hexdigest()\n            badge_path=(badges.get(p[\'club\']) or {}).get(\'path\')\n            badge_digest=hashlib.sha256(Path(badge_path).read_bytes()).hexdigest() if badge_path and Path(badge_path).is_file() else \'\'\n            logo_path=Path(brand_root)/\'assets/logo.png\'\n            logo_digest=hashlib.sha256(logo_path.read_bytes()).hexdigest()\n            digest=hashlib.sha256((doc+photo_digest+badge_digest+logo_digest).encode()).hexdigest()[:20]\n            cutout=public/f\'hero_{pid}.png\';full=public/f\'hero_full_{pid}.png\';cached=public/f\'hero_{pid}.cache.json\'\n            if cutout.is_file() and full.is_file() and cached.is_file():\n                try:\n                    metadata=json.loads(cached.read_text(encoding=\'utf-8\'))\n                    if metadata.get(\'digest\')==digest and isinstance(metadata.get(\'geometry\'),dict):\n                        p[\'hero_geometry\']=metadata[\'geometry\'];p[\'hero\']=cutout.name\n                        continue\n                except (ValueError,OSError,TypeError):\n                    pass\n            htmlpath=public/f\'hero_{pid}_{digest}.html\';htmlpath.write_text(doc,encoding=\'utf-8\')\n            page.goto(htmlpath.as_uri());page.wait_for_function(\'Array.from(document.images).every(i=>i.complete)\');page.evaluate(\'document.fonts.ready\')\n            page.evaluate("()=>{let el=document.querySelector(\'.hero-name\');let s=parseFloat(getComputedStyle(el).fontSize);while(el.scrollWidth>el.clientWidth&&s>18)el.style.fontSize=(--s)+\'px\';}")\n            if not page.evaluate(\'Array.from(document.images).every(i=>i.naturalWidth>0)\'):raise ValueError(\'Missing Hero image\')\n            geometry=page.evaluate("()=>Object.fromEntries([\'photo\',\'name\'].map(k=>{let e=document.querySelector(\'.hero-\'+k),r=e.getBoundingClientRect(),s=getComputedStyle(e);return [k,{x:r.x,y:r.y,width:r.width,height:r.height,fontSize:parseFloat(s.fontSize),fontFamily:s.fontFamily,color:s.color,text:e.textContent,objectFit:s.objectFit,objectPosition:s.objectPosition}]}))")\n            p[\'hero_geometry\']=geometry;p[\'hero\']=\'hero_\'+str(pid)+\'.png\'\n            page.screenshot(path=str(full),omit_background=True)\n            page.add_style_tag(content=\'.hero-photo,.hero-name{visibility:hidden!important}\')\n            page.screenshot(path=str(cutout),omit_background=True)\n            cached.write_text(json.dumps({\'digest\':digest,\'geometry\':geometry},ensure_ascii=False),encoding=\'utf-8\')\n        browser.close()\n\ndef _write_soft_placement_sound(target):\n    """Generate a low-volume stereo landing chime without network assets or dependencies."""\n    import struct,wave\n    rate=48000;duration=.34;frames=[]\n    for index in range(int(duration*rate)):\n        t=index/rate;attack=min(1.,t/.018);tail=max(0.,1.-t/duration)**2.7\n        tone=(math.sin(2*math.pi*690*t)+.33*math.sin(2*math.pi*1040*t))*.070*attack*tail\n        sample=int(max(-1.,min(1.,tone))*32767)\n        frames.append(struct.pack(\'<hh\',sample,sample))\n    with wave.open(str(target),\'wb\') as output:\n        output.setnchannels(2);output.setsampwidth(2);output.setframerate(rate)\n        output.writeframes(b\'\'.join(frames))\n\ndef prepare_project(scene,contract,project,media,badges,brand_root,views,background,audio=None,width=854,height=480,fps=24,duration=None):\n    project=Path(project);public=project/\'public\';public.mkdir(parents=True,exist_ok=True)\n    data=copy.deepcopy(scene[\'masterplan_case\']);data[\'timeline\']=copy.deepcopy(scene[\'narration_timeline\']);data.update(width=width,height=height,fps=fps,duration=duration or data[\'timeline\'][-1][\'end\'],audio=\'\')\n    background.save(public/\'background.png\');hero_assets(data,views,media,badges,brand_root,public)\n    initial_ids=set(data[\'opening_ids\'])|set(data[\'bench_ids\'])\n    for p in data[\'players\']:\n        if p[\'id\'] not in initial_ids:\n            continue\n        full_card=public/f"hero_full_{p[\'id\']}.png"\n        if not full_card.is_file() or full_card.stat().st_size == 0:\n            raise FileNotFoundError(\'Original complete hero card missing: \'+str(full_card))\n        p[\'hero_full\']=full_card.name\n    for p in data[\'players\']:\n        m=media.get(p[\'id\']) or media.get(str(p[\'id\']));photo=Image.open(m[\'photo\']).convert(\'RGBA\');photo.save(public/f"player_{p[\'id\']}.png");p[\'photo\']=f"player_{p[\'id\']}.png"\n        badge=(badges.get(p[\'club\']) or {}).get(\'path\')\n        if not badge:raise FileNotFoundError(\'Missing club badge\')\n        p[\'badge\']=f"club_{p[\'team_id\']}.png";shutil.copy2(badge,public/p[\'badge\'])\n    _write_soft_placement_sound(public/\'squad_placement.wav\')\n    if audio:shutil.copy2(audio,public/\'narration.wav\');data[\'audio\']=\'narration.wav\'\n    (public/\'masterplan.json\').write_text(json.dumps(data,ensure_ascii=False,indent=2));return data\n\nclass RemotionMasterplanVisual:\n    requires_motion=True\n    def __init__(self,scene,contract,width,height,project,media,badges,brand_root,views,background):\n        self.scene=scene;self.contract=contract;self.width=width;self.height=height;self.project=Path(project);self.media=media;self.badges=badges;self.brand_root=brand_root;self.views=views;self.background=background\n    def prepare(self,audio,duration,profile):return prepare_project(self.scene,self.contract,self.project,self.media,self.badges,self.brand_root,self.views,self.background,audio,profile[\'width\'],profile[\'height\'],profile[\'fps\'],duration)\n    def frame(self,seconds):\n        import numpy as np\n        duration=self.scene[\'narration_timeline\'][-1].get(\'end\')\n        if duration is None:raise ValueError(\'Measure narration before a Slide 3 still\')\n        self.prepare(None,duration,dict(width=self.width,height=self.height,fps=24))\n        digest=hashlib.sha256((json.dumps(self.scene,sort_keys=True,default=str)+str(seconds)+str(self.width)).encode()).hexdigest()[:20]\n        target=self.project/(\'still_\'+digest+\'.png\')\n        if not target.exists():\n            command=[\'npx\',\'remotion\',\'still\',\'src/index.ts\',\'WildcardMasterplan\',str(target),\'--frame=\'+str(min(math.ceil(duration*24)-1,round(seconds*24))),\'--props={"mute":true}\',\'--log=error\',\'--overwrite\']\n            browser=Path(\'/Applications/Google Chrome.app/Contents/MacOS/Google Chrome\')\n            if browser.exists():command.append(\'--browser-executable=\'+str(browser))\n            subprocess.run(command,cwd=self.project,check=True)\n        return np.asarray(Image.open(target).convert(\'RGB\'))\n    @property\n    def complete(self):\n        t=self.scene[\'narration_timeline\'][-1].get(\'end\')\n        if t is None:raise ValueError(\'Measure narration before exporting Slide 3\')\n        return Image.fromarray(self.frame(max(0,t-.05)))\n    def render_video(self,audio,duration,target,profile):\n        if not 345<=duration<=420:raise ValueError(\'Wildcard Slide 3 must be 5:45–7:00. Edit words; preserve speech settings.\')\n        self.prepare(audio,duration,profile)\n        if not (self.project/\'node_modules/remotion\').exists():subprocess.run([\'npm\',\'install\',\'--no-audit\',\'--no-fund\'],cwd=self.project,check=True)\n        command=[\'npx\',\'remotion\',\'render\',\'src/index.ts\',\'WildcardMasterplan\',str(target),\'--codec=h264\',\'--crf=\'+str(profile.get(\'crf\',28)),\'--concurrency=3\',\'--log=info\',\'--overwrite\']\n        browser=Path(\'/Applications/Google Chrome.app/Contents/MacOS/Google Chrome\')\n        if browser.exists():command.append(\'--browser-executable=\'+str(browser))\n        subprocess.run(command,cwd=self.project,check=True);return Path(target)\n\ndef prepare_audio(scene,script,work,synthesise,decode_pcm):\n    import wave\n    work=Path(work);tracks=[];cues=copy.deepcopy(scene[\'narration_timeline\']);clock=0\n    if \' \'.join(c[\'narration_text\'] for c in cues).split()!=scene[\'narration_text\'].split():raise ValueError(\'Narration and cue text disagree\')\n    for cue in cues:\n        print(\'Measuring Wildcard Slide 3:\',cue[\'segment_id\'],flush=True)\n        source=Path(synthesise(dict(script,id=scene[\'scene_id\']+\'_\'+cue[\'segment_id\'],text=cue[\'narration_text\'],scene=scene,narration_cue=cue)))\n        digest=hashlib.sha256(source.read_bytes()).hexdigest()[:20];pcm=work/f\'masterplan_{digest}.wav\'\n        if not pcm.exists():decode_pcm(source,pcm)\n        with wave.open(str(pcm),\'rb\') as f:\n            if (f.getnchannels(),f.getsampwidth(),f.getframerate())!=(2,2,48000):raise ValueError(\'Invalid narration PCM\')\n            seconds=f.getnframes()/48000;frames=f.readframes(f.getnframes())\n        cue[\'start\']=clock;cue[\'speech_end\']=clock+seconds;gap=1.1 if cue[\'type\'] in (\'players\',\'bench\') else 0\n        cue[\'end\']=cue[\'speech_end\']+gap;clock=cue[\'end\'];tracks.append((frames,round(gap*48000)))\n    digest=hashlib.sha256(json.dumps(cues,sort_keys=True).encode()).hexdigest()[:20];target=work/f\'masterplan_measured_{digest}.wav\'\n    if not target.exists():\n        with wave.open(str(target),\'wb\') as f:\n            f.setnchannels(2);f.setsampwidth(2);f.setframerate(48000)\n            for frames,gap in tracks:f.writeframes(frames);f.writeframes(b\'\\0\'*(gap*4))\n    scene[\'narration_timeline\']=cues;scene[\'timing\'][\'player_alignment\']=\'measured_segments\';return target\n\n\n# VX_PLAYWRIGHT_JUPYTER_WORKER_V1\nfrom concurrent.futures import ThreadPoolExecutor as _VxBrowserThreadPool\n_vx_original_hero_assets = hero_assets\n\ndef hero_assets(case, views, media, badges, brand_root, public):\n    # Playwright\'s Sync API must not run on Colab\'s active asyncio thread.\n    with _VxBrowserThreadPool(max_workers=1, thread_name_prefix=\'vortex-playwright\') as executor:\n        return executor.submit(_vx_original_hero_assets, case, views, media, badges, brand_root, public).result()\n\n\n# VX_REMOTION_CLI_SAFE_V3\n# Keep Remotion\'s diagnostics and use a browser already installed for Colab.\n\ndef _vx_remotion_command_output(project, stage, command):\n    import subprocess\n    project = Path(project)\n    log_path = project / (\'vortex_remotion_\' + stage + \'.log\')\n    try:\n        with log_path.open(\'wb\') as log:\n            completed = subprocess.run(command, cwd=project, stdout=log, stderr=subprocess.STDOUT, check=False)\n    except OSError as exc:\n        raise RuntimeError(\'Remotion \'+stage+\' could not start: \'+str(exc)) from exc\n    if completed.returncode:\n        lines = log_path.read_text(encoding=\'utf-8\', errors=\'replace\').splitlines()\n        detail = \'\\n\'.join(lines[-110:]) or \'(Remotion produced no stdout/stderr)\'\n        raise RuntimeError(\n            f\'Remotion {stage} failed (exit {completed.returncode}).\\n\'\n            f\'Full log: {log_path}\\n\'\n            + detail\n        )\n    return log_path\n\n\ndef _vx_remotion_cli(project):\n    project = Path(project)\n    binary = project / \'node_modules\' / \'.bin\' / \'remotion\'\n    if not binary.is_file():\n        print(\'Installing pinned Remotion CLI dependencies...\', flush=True)\n        _vx_remotion_command_output(project, \'npm_install\', [\'npm\', \'ci\', \'--no-audit\', \'--no-fund\'])\n    if not binary.is_file():\n        raise RuntimeError(\'Remotion CLI is still missing after npm ci: \'+str(binary))\n    return str(binary)\n\n\ndef _vx_remotion_browser_arguments(project):\n    import os\n    import shutil\n    from concurrent.futures import ThreadPoolExecutor\n    project = Path(project)\n    pinned = project / \'node_modules\' / \'.remotion\' / \'chrome-headless-shell\'\n    if any(p.is_file() for p in pinned.glob(\'**/chrome-headless-shell\')):\n        return []\n    explicit = os.getenv(\'VX_REMOTION_BROWSER_EXECUTABLE\')\n    candidates = ([explicit] if explicit else [])\n    candidates.extend(filter(None, [shutil.which(\'google-chrome-stable\'), shutil.which(\'google-chrome\'),\n        shutil.which(\'chromium\'), shutil.which(\'chromium-browser\')]))\n    candidates.append(\'/Applications/Google Chrome.app/Contents/MacOS/Google Chrome\')\n    def playwright_browser():\n        from playwright.sync_api import sync_playwright\n        with sync_playwright() as runtime:\n            return runtime.chromium.executable_path\n    try:\n        with ThreadPoolExecutor(max_workers=1, thread_name_prefix=\'vortex-browser-detection\') as pool:\n            candidates.insert(0, pool.submit(playwright_browser).result())\n    except (ImportError, OSError, RuntimeError) as exc:\n        print(\'Playwright browser path unavailable: \'+str(exc), flush=True)\n    for candidate in candidates:\n        if candidate and Path(candidate).is_file():\n            print(\'Remotion browser: \'+str(candidate), flush=True)\n            return [\'--browser-executable=\'+str(candidate),\n                    \'--chrome-mode=\'+(\'headless-shell\' if \'headless-shell\' in str(candidate) else \'chrome-for-testing\')]\n    print(\'Remotion browser: using its pinned browser downloader\', flush=True)\n    return []\n\n\ndef _vx_masterplan_safe_render_video(self, audio, duration, target, profile):\n    import math\n    profile = dict(profile)\n    if not math.isfinite(duration) or duration <= 0:\n        raise ValueError(f\'Invalid measured Wildcard Slide 3 duration: {duration!r}\')\n    self.prepare(audio, duration, profile)\n    cli = _vx_remotion_cli(self.project)\n    browser = _vx_remotion_browser_arguments(self.project)\n    smoke = self.project / \'vortex_masterplan_preflight.png\'\n    print(\'Remotion: checking one frame before the full video render...\', flush=True)\n    _vx_remotion_command_output(self.project, \'preflight\',\n        [cli, \'still\', \'src/index.ts\', \'WildcardMasterplan\', str(smoke),\n         \'--frame=0\', \'--props={"mute":true}\', \'--log=verbose\', \'--overwrite\', *browser])\n    if not smoke.is_file() or smoke.stat().st_size == 0:\n        raise RuntimeError(\'Remotion preflight completed without producing a preview frame\')\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    # Colab Chromium / Node compete for memory: one renderer process is safer for long audio.\n    concurrency = 1 if profile.get(\'height\', 0) <= 1080 else 2\n    command = [cli, \'render\', \'src/index.ts\', \'WildcardMasterplan\', str(target),\n               \'--codec=h264\', \'--crf=\'+str(profile.get(\'crf\', 28)),\n               \'--concurrency=\'+str(concurrency), \'--timeout=60000\',\n               \'--log=info\', \'--overwrite\', *browser]\n    print(f\'Remotion: encoding {duration:.2f}s at {profile["width"]}x{profile["height"]}, concurrency={concurrency}\', flush=True)\n    _vx_remotion_command_output(self.project, \'video\', command)\n    if not target.is_file() or target.stat().st_size <= 0:\n        raise RuntimeError(\'Remotion returned success but produced no video: \'+str(target))\n    return target\n\n\ndef _vx_masterplan_safe_frame(self, seconds):\n    import math\n    import hashlib\n    import json\n    import numpy as np\n    duration = self.scene[\'narration_timeline\'][-1].get(\'end\')\n    if duration is None:\n        raise ValueError(\'Measure narration before a Slide 3 still\')\n    self.prepare(None, duration, dict(width=self.width, height=self.height, fps=24))\n    digest = hashlib.sha256((json.dumps(self.scene, sort_keys=True, default=str) +\n        str(seconds) + str(self.width)).encode()).hexdigest()[:20]\n    target = self.project / (\'still_\'+digest+\'.png\')\n    if not target.exists():\n        cli = _vx_remotion_cli(self.project)\n        _vx_remotion_command_output(self.project, \'still\',\n            [cli, \'still\', \'src/index.ts\', \'WildcardMasterplan\', str(target),\n             \'--frame=\'+str(min(math.ceil(duration*24)-1, round(seconds*24))),\n             \'--props={"mute":true}\', \'--log=verbose\', \'--overwrite\',\n             *_vx_remotion_browser_arguments(self.project)])\n    return np.asarray(Image.open(target).convert(\'RGB\'))\n\nRemotionMasterplanVisual.render_video = _vx_masterplan_safe_render_video\nRemotionMasterplanVisual.frame = _vx_masterplan_safe_frame\n'
_VX_MASTERPLAN_PROJECT_ZIP = '...'
# WILDCARD SLIDE 3 ONLY — LIVE MASTERPLAN / REMOTION
VX_WILDCARD_SLIDE3_PREVIEW_ONLY = True #@param {type:"boolean"}
import base64,io,zipfile,importlib
_VX_MASTERPLAN_MODULE = Path(VX_MEDIA_WORK)/'wildcard_engine'/'wildcard_masterplan.py'
_VX_MASTERPLAN_MODULE.write_text(_VX_MASTERPLAN_SOURCE,encoding='utf-8')
Path(importlib.util.cache_from_source(str(_VX_MASTERPLAN_MODULE))).unlink(missing_ok=True)
importlib.invalidate_caches()
import wildcard_masterplan as VX_MASTERPLAN
importlib.reload(VX_MASTERPLAN)
VX_MASTERPLAN_PROJECT=VX_MEDIA_WORK/'wildcard_masterplan_remotion'
VX_MASTERPLAN_PROJECT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_VX_MASTERPLAN_PROJECT_ZIP))) as _vx_wc3_zip:_vx_wc3_zip.extractall(VX_MASTERPLAN_PROJECT)
if VX_GENERATE_WILDCARD:
    def _vx_masterplan_scene(scene,contract):
        case=VX_MASTERPLAN.build_case(contract,_bootstrap,VX_WC_ENGINE.player_card)
        scene.update(masterplan_case=case,narration_text=case['narration_text'],narration_timeline=copy.deepcopy(case['narration_timeline']),title='Wildcard Masterplan',display_title='WILDCARD MASTERPLAN',subheading='BUILD THE TEAM. CONTROL THE NEXT SIX.')
        scene['timing'].update(estimated_duration_seconds=len(case['narration_text'].split())/175*60+12.1,status='Measured narration required')
        return scene
    if not getattr(vx_build_manifest,'_wildcard_masterplan',False):
        _vx_manifest_before_masterplan=vx_build_manifest
        def vx_build_manifest(contract,_build=_vx_manifest_before_masterplan):
            scenes=_build(contract)
            if contract.get('chip')=='wildcard':
                for scene in scenes:
                    if scene['visual_type']=='wc_squad_plan':_vx_masterplan_scene(scene,contract)
            return scenes
        vx_build_manifest._wildcard_masterplan=True
    for _vx_wc3_key,_vx_wc3_contract in VX_DATA_CONTRACTS.items():
        if not _vx_wc3_key.startswith('wildcard'):continue
        _vx_wc3_scenes=VX_SCENE_MANIFESTS[_vx_wc3_key]
        _vx_wc3_scene=next(s for s in _vx_wc3_scenes if s['visual_type']=='wc_squad_plan')
        _vx_masterplan_scene(_vx_wc3_scene,_vx_wc3_contract)
        _vx_wc3_final=VX_WILDCARD_SCENE_JSONS[_vx_wc3_key]
        _vx_wc3_final['scenes']=[copy.deepcopy(_vx_wc3_scene) if s['visual_type']=='wc_squad_plan' else s for s in _vx_wc3_final['scenes']]
        _vx_wc3_narration=VX_NARRATION_OBJECTS[_vx_wc3_key]
        _vx_wc3_section=next(s for s in _vx_wc3_narration['sections'] if s['visual_type']=='wc_squad_plan')
        _vx_wc3_section.update(narration=_vx_wc3_scene['narration_text'],source_fields=['scenes.masterplan_case.players','scenes.masterplan_case.weeks'],estimated_duration_seconds=round(len(_vx_wc3_scene['narration_text'].split())/175*60+12.1,2),pause_before_seconds=0,pause_after_seconds=0)
        _vx_wc3_narration['total_estimated_duration_seconds']=sum(s['estimated_duration_seconds'] for s in _vx_wc3_narration['sections'])
        if not VXWC.validate_scene_json(_vx_wc3_final)['passed']:raise ValueError('Wildcard scene validation failed after Slide 3 update')
        VXWC.validate_narration_object(_vx_wc3_final,_vx_wc3_narration)
        _vx_wc3_script=VX_NARRATION_PATHS[_vx_wc3_key];_vx_wc3_text=_vx_wc3_script.read_text(encoding='utf-8')
        _vx_wc3_stamp=VX_ANALYSIS_ROOT/'narration_provenance'/(_vx_wc3_script.name+'.provenance.json')
        _vx_wc3_pattern='(\\[\\[SEGMENT {id} \\|[^\\n]*\\]\\]\\s*\\n)(.*?)(?=\\n\\[\\[SEGMENT |\\Z)'.replace('{id}',re.escape(_vx_wc3_scene['scene_id']))
        _vx_wc3_text,_vx_wc3_count=re.subn(_vx_wc3_pattern,lambda m:m.group(1)+_vx_wc3_scene['narration_text']+'\n',_vx_wc3_text,flags=re.S)
        if _vx_wc3_count!=1:raise ValueError('Slide 3 narration header missing')
        _vx_wc3_script.write_text(_vx_wc3_text,encoding='utf-8')
        _vx_wc3_stamp.write_text(json.dumps(dict(data_hash=_vx_wc3_contract['data_hash'],generated_sha256=hashlib.sha256(_vx_wc3_text.encode()).hexdigest()),indent=2))
        VXWC.validate_render_script(_vx_wc3_narration,decode_script(_vx_wc3_text))
        for _vx_wc3_name,_vx_wc3_value in [(_vx_wc3_key+'_final_scene.json',_vx_wc3_final),(_vx_wc3_key+'_narration.json',_vx_wc3_narration),(_vx_wc3_key+'_scene_manifest.json',_vx_wc3_scenes),(_vx_wc3_key+'_slide3_masterplan_case.json',_vx_wc3_scene['masterplan_case'])]:
            (VX_ANALYSIS_ROOT/_vx_wc3_name).write_text(json.dumps(_vx_wc3_value,ensure_ascii=False,indent=2))
        print('Wildcard Slide 3: requested team story synchronized with live lineup, player cards, captaincy roadmap and transfer plan. Script words:', len(_vx_wc3_scene['narration_text'].split()), flush=True)
else:VX_WILDCARD_SLIDE3_PREVIEW_ONLY=False

_VX_FREEHIT_SUITE_SOURCE = '"""Free Hit slide 3: automatic GK–DEF–MID–FWD reveals and original notebook Hero Cards."""\nfrom pathlib import Path\nimport copy,json,math,shutil,wave\nVERSION=\'freehit-slides3-4-squad-script-sync-v5\'\n\ndef ensure_metadata(project):\n public=Path(project)/\'public\';public.mkdir(parents=True,exist_ok=True)\n stub=dict(width=1920,height=1080,fps=24,duration=1,gameweek=0,formation=\'\',cost=0,bank=0,xi_xpts=0,total_xpts=0,captain_id=0,vice_id=0,candidate_ids=[],deadline=None,updated=None,players=[],timeline=[dict(segment_id=\'pending\',type=\'title\',start=0,end=1,player_ids=[])],audio=\'\')\n for name in (\'squad.json\',\'captaincy.json\'):\n  if not (public/name).is_file():(public/name).write_text(json.dumps(stub))\n return public\n\ndef build_squad_data(scene,contract):\n p=contract[\'slide3_plan\'];roles=(\'GK\',\'DEF\',\'MID\',\'FWD\');normal=lambda x:\'GK\' if x==\'GKP\' else x\n starters=sorted(p[\'starting_xi\'],key=lambda x:(roles.index(normal(x[\'position\'])),-float(x[\'projected_points\']),int(x[\'id\'])))\n if len(starters)!=11 or len({x[\'id\'] for x in starters})!=11:raise ValueError(\'Selected legal XI required\')\n groups={role:[x for x in starters if normal(x[\'position\'])==role] for role in roles}\n row_y={\'GK\':91,\'DEF\':233,\'MID\':375,\'FWD\':517};slots={}\n for role,group in groups.items():\n  for i,x in enumerate(group):slots[x[\'id\']]=dict(x=350+(i-(len(group)-1)/2)*min(142,620/len(group)),y=row_y[role])\n out=[];bench_rank=0\n for x in starters+list(p[\'bench\']):\n  starting=x[\'id\'] in slots\n  if not starting and normal(x[\'position\'])!=\'GK\':bench_rank+=1\n  fs=x.get(\'analysis\',{}).get(\'fixtures\') or x.get(\'fixtures\',[])\n  out.append(dict(id=x[\'id\'],player_id=x[\'id\'],team_id=x[\'team_id\'],name=x[\'name\'],spoken_name=x.get(\'spoken_name\') or x.get(\'full_name\') or x[\'name\'],club=x[\'club\'],club_name=x[\'club_name\'],position=normal(x[\'position\']),price=x[\'price\'],xpts=x[\'projected_points\'],minutes=x.get(\'minutes\') or 0,expected_minutes=x.get(\'minutes\') or 0,start_probability=x.get(\'starting_probability\') or x.get(\'analysis\',{}).get(\'start_probability\') or 0,is_starter=starting,bench_order=None if starting or normal(x[\'position\'])==\'GK\' else bench_rank,slot=slots.get(x[\'id\']),fixture_items=[dict(label=f[\'opponent\']+(\'[H]\' if f[\'home\'] else \'[A]\'),fdr=f.get(\'fdr\'),style=f.get(\'fdr_style\',{})) for f in fs],fixtures=\' / \'.join(f["opponent"]+(\'[H]\' if f[\'home\'] else \'[A]\') for f in fs)))\n ids=[q.get(\'hero_player_id\') for q in scene[\'narration_timeline\'] if q[\'type\']==\'players\']\n if scene.get(\'visual_type\')!=\'fh_captain_final\' and ids!=[x[\'id\'] for x in starters]:raise ValueError(\'Narration order differs from automatic pitch reveal order\')\n if scene.get(\'visual_type\') == \'fh_squad_plan\':\n  bench_ids=[q.get(\'hero_player_id\') for q in scene[\'narration_timeline\'] if q[\'type\']==\'bench\']\n  if bench_ids != [x[\'id\'] for x in p[\'bench\']]:raise ValueError(\'Narration order differs from the displayed bench order\')\n  if scene[\'narration_text\'].split() != \' \'.join(q[\'narration_text\'] for q in scene[\'narration_timeline\']).split():raise ValueError(\'Squad speech differs from the measured card cues\')\n rows=copy.deepcopy(contract[\'fixture_case\'][\'rows\'])\n for row in rows:row[\'fixtures\']=[f for w in row[\'weeks\'] for f in w[\'fixtures\']]\n return dict(version=VERSION,season=contract.get(\'season\'),width=1920,height=1080,fps=24,duration=1,gameweek=contract[\'gameweek\'],formation=p[\'formation\'],cost=p[\'total_cost\'],bank=p[\'bank\'],xi_xpts=p[\'xi_xpts\'],total_xpts=p[\'captain_adjusted_xpts\'],captain_id=p[\'captain\'][\'id\'],vice_id=p[\'vice_captain\'][\'id\'],deadline=None,updated=contract.get(\'source_snapshot_at\') or contract.get(\'generated_at\'),rows=rows,players=out,timeline=copy.deepcopy(scene[\'narration_timeline\']),audio=\'\')\n\ndef prepare_squad(scene,contract,project,media,badges,brand_root,views,audio,width,height,fps,duration):\n from freehit_matchday import render_original_heroes\n public=ensure_metadata(project);data=build_squad_data(scene,contract);stem=\'captaincy\' if scene.get(\'visual_type\')==\'fh_captain_final\' else \'squad\';data[\'candidate_ids\']=[q[\'hero_player_id\'] for q in scene[\'narration_timeline\'] if q[\'type\']==\'candidate\'];data.update(width=width,height=height,fps=fps,duration=duration)\n # Use the same official event timestamp already stored on the canonical matchday scene.\n data[\'deadline\']=contract.get(\'deadline\') or scene.get(\'deadline\') or scene.get(\'matchday_deadline\')\n paths,proof=render_original_heroes(data,views,media,badges,brand_root,Path(project)/\'squad_hero_cache\')\n for p in data[\'players\']:\n  record=media.get(p[\'id\']) or media.get(str(p[\'id\'])) or {}\n  if record.get(\'photo_kind\')!=\'player\':raise FileNotFoundError(\'Official player photo required: \'+p[\'name\'])\n  p[\'photo\']=\'squad_player_\'+str(p[\'id\'])+\'.png\';shutil.copy2(record[\'photo\'],public/p[\'photo\'])\n  p[\'hero\']=\'squad_hero_\'+str(p[\'id\'])+\'.png\';shutil.copy2(paths[p[\'id\']],public/p[\'hero\'])\n  p[\'badge\']=\'squad_club_\'+str(p[\'team_id\'])+\'.png\';shutil.copy2(badges[p[\'club\']][\'path\'],public/p[\'badge\'])\n for name in (\'logo.png\',\'logo1.png\',\'pl.png\'):shutil.copy2(Path(brand_root)/\'assets\'/name,public/name)\n if audio:shutil.copy2(audio,public/(stem+\'_narration.wav\'));data[\'audio\']=stem+\'_narration.wav\'\n if not (public/\'intro.json\').is_file():(public/\'intro.json\').write_text(json.dumps(dict(width=width,height=height,fps=fps,duration=1,gameweek=data[\'gameweek\'],timeline=[dict(start=0,end=1,type=\'question\')],shortlist=[],total_points=None,audio=\'\')))\n if not (public/\'matchday.json\').is_file():(public/\'matchday.json\').write_text(json.dumps(dict(width=width,height=height,fps=fps,duration=1,gameweek=data[\'gameweek\'],rows=[],players=[],timeline=[dict(start=0,end=1,type=\'opening\',team_id=None,player_ids=[])],targets=[],avoids=[],featured_players=[],captains=[],audio=\'\',deadline=data[\'deadline\'],updated_at=data[\'updated\'])))\n data[\'hero_card_asset_proof\']=proof;scene[\'hero_card_asset_proof\']=proof;(public/(stem+\'.json\')).write_text(json.dumps(data,ensure_ascii=False,indent=2));return data\n\ndef prepare_squad_audio(scene,script,work,synthesise,decode_pcm):\n """Keep the complete Ryan speech, then allow the original card to land on the pitch."""\n from wildcard_plan_audio import prepare_segment_audio\n voice_cues={}\n def speak(segment):\n  path=synthesise(segment);voice_cues[str(Path(path))]=segment[\'narration_cue\'];return path\n def decode(source,target):\n  decode_pcm(source,target);cue=voice_cues[str(source)];hold=.85 if cue[\'type\']==\'players\' else .25 if cue[\'type\']==\'bench\' else .45\n  with wave.open(str(target),\'rb\') as track:params=track.getparams();pcm=track.readframes(track.getnframes())\n  with wave.open(str(target),\'wb\') as track:track.setparams(params);track.writeframes(pcm);track.writeframes(b\'\\0\'*(round(hold*params.framerate)*params.nchannels*params.sampwidth))\n cache=Path(work)/\'freehit_squad_measured_v3\';cache.mkdir(parents=True,exist_ok=True)\n result=prepare_segment_audio(scene,script,cache,speak,decode)\n for cue in scene[\'narration_timeline\']:\n  cue[\'flight_seconds\']=.85 if cue[\'type\']==\'players\' else 0\n return result\n\nclass RemotionFreeHitSquadVisual:\n requires_motion=True\n preserve_frame_edges=True\n def __init__(self,scene,contract,width,height,project,media,badges,brand_root,views,deadline):\n  self.scene=scene;self.manifest=scene;self.contract=dict(contract,deadline=deadline);self.width=width;self.height=height;self.project=Path(project);self.media=media;self.badges=badges;self.brand_root=brand_root;self.views=views\n def prepare(self,audio,duration,profile):return prepare_squad(self.scene,self.contract,self.project,self.media,self.badges,self.brand_root,self.views,audio,profile[\'width\'],profile[\'height\'],profile[\'fps\'],duration)\n def render_video(self,audio,duration,target,profile):\n  from wildcard_masterplan import _vx_remotion_cli,_vx_remotion_browser_arguments,_vx_remotion_command_output\n  self.prepare(audio,duration,profile);target=Path(target);target.parent.mkdir(parents=True,exist_ok=True)\n  _vx_remotion_command_output(self.project,\'freehit_squad_video\',[_vx_remotion_cli(self.project),\'render\',\'src/index.ts\',getattr(self,\'composition\',\'FreeHitSquad\'),str(target),\'--codec=h264\',\'--crf=\'+str(profile.get(\'crf\',28)),\'--concurrency=1\',\'--timeout=60000\',\'--log=info\',\'--overwrite\',*_vx_remotion_browser_arguments(self.project)])\n  if not target.is_file() or target.stat().st_size<=0:raise RuntimeError(\'Free Hit squad video missing\')\n  return target\n\n def draw(self,seconds,duration):\n  from PIL import Image\n  from wildcard_masterplan import _vx_remotion_cli,_vx_remotion_browser_arguments,_vx_remotion_command_output\n  duration=max(float(duration),float(self.scene[\'narration_timeline\'][-1][\'end\']))\n  fps=24;self.prepare(None,duration,dict(width=self.width,height=self.height,fps=fps))\n  frame=max(0,min(math.ceil(duration*fps)-1,round(float(seconds)*fps)))\n  target=self.project/\'squad_stills\'/f"squad_{self.width}x{self.height}_{frame}.png";target.parent.mkdir(exist_ok=True)\n  _vx_remotion_command_output(self.project,\'freehit_squad_still\',[_vx_remotion_cli(self.project),\'still\',\'src/index.ts\',getattr(self,\'composition\',\'FreeHitSquad\'),str(target),\'--frame=\'+str(frame),\'--image-format=png\',\'--timeout=60000\',\'--log=info\',\'--overwrite\',*_vx_remotion_browser_arguments(self.project)])\n  with Image.open(target) as image:return image.convert(\'RGB\')\n @property\n def complete(self):\n  cue=self.scene[\'narration_timeline\'][-1];return self.draw(cue[\'start\']+.5,cue[\'end\'])\n def frame(self,seconds,duration,previous=None):\n  import numpy as np\n  return np.asarray(self.draw(seconds,duration))\n\nclass RemotionFreeHitCaptaincyVisual(RemotionFreeHitSquadVisual):\n composition=\'FreeHitCaptaincy\'\n'
_VX_FREEHIT_PREMIUM_SUITE_SOURCE = '"""Four Free Hit broadcast slides. One-week facts, concise narration and deterministic cues."""\nfrom __future__ import annotations\nimport copy,html,json,math,re,base64\nfrom pathlib import Path\nfrom collections import Counter\nVERSION=\'freehit-premium-four-slides-hero-flight-v2\'\nVOICE=dict(voice=\'en-GB-RyanNeural\',rate=\'+0%\',pitch=\'+0Hz\',volume=\'+0%\')\ndef num(v):\n try:\n  v=float(v);return v if math.isfinite(v) else None\n except (ValueError,TypeError):return None\ndef full_club(code,fallback=\'\'):\n from tts_narration import CLUBS\n return CLUBS.get(code,fallback or code)\ndef name(p):return p.get(\'spoken_name\') or p.get(\'player_name\') or p.get(\'full_name\') or p[\'name\']\ndef seg(identity,kind,text,**kw):return dict(segment_id=identity,type=kind,narration_text=text,start=None,end=None,player_ids=kw.pop(\'player_ids\',[]),**kw)\ndef align_timeline(items,duration,segment_durations=None):\n out=copy.deepcopy(items);weights=[len(s[\'narration_text\'].split()) for s in out];times=list(segment_durations) if segment_durations is not None else [duration*w/sum(weights) for w in weights]\n if len(times)!=len(out) or any(not math.isfinite(t) or t<=0 for t in times) or abs(sum(times)-duration)>.05:raise ValueError(\'Invalid measured Free Hit timing\')\n cursor=0.\n for s,t in zip(out,times):s.update(start=cursor,end=cursor+t,alignment=\'measured_segments\' if segment_durations is not None else \'estimated_word_counts\');cursor+=t\n return out\n\ndef difficulty(value):\n from vortex_wildcard import fdr_display\n return fdr_display(num(value))\ndef fixtures(p):return p.get(\'analysis\',{}).get(\'fixtures\',p.get(\'current_fixtures\',p.get(\'fixtures\',[])))\ndef fixture_words(f):return full_club(f.get(\'opponent\',\'\'),f.get(\'opponent_name\',\'\'))+(\' at home\' if f.get(\'home\') else \' away\')\ndef fx_words(p):return \' and \'.join(fixture_words(f) for f in fixtures(p)) or \'no published fixture\'\ndef short_reason(p):\n a=p.get(\'analysis\',{});pos=p.get(\'position\');fs=fixtures(p)\n if a.get(\'availability\',{}).get(\'flagged\'):return \'Monitor availability and expected minutes\'\n if len(fs)>1:return \'Multiple verified fixtures this Gameweek\'\n if pos in (\'GK\',\'DEF\'):\n  cs=num(a.get(\'clean_sheet_probability\'))\n  if cs is not None:return f\'{cs:.0%} model clean-sheet chance\'\n  return \'Defensive returns and minutes estimate\'\n if a.get(\'role\') in (\'First-choice penalty taker\',\'first-choice penalty taker\',\'First penalty taker\'):return \'Penalty role and attacking involvement\'\n if num(a.get(\'xgi90\')) is not None:return f\'{a["xgi90"]:.2f} expected goal involvement per 90\'\n return \'One-week projection and budget fit\'\n\ndef intro_case(d,bootstrap=None):\n from freehit_personal_intro import build_case\n return build_case(d,bootstrap)\n\ndef fixture_segments(case, selected=()):\n    """Preparation-stage speech; the detailed slide uses its own live player cards."""\n    targets = case[\'targets\']\n    out = [seg(\'fh_fixture_open\', \'intro\',\n        f"For Gameweek {case[\'gameweeks\'][0]}, I\'m considering {len(targets)} fixture targets. "\n        "I\'ll look at the matchups first, then decide which players fit my one-week squad.")]\n    for rank, row in enumerate(targets, 1):\n        scheduled = [fixture for week in row[\'weeks\'] for fixture in week[\'fixtures\']]\n        club = full_club(row[\'club\'], row[\'name\'])\n        text = f"I\'m looking at {club}. "\n        text += \' \'.join(f"They face {fixture_words(fixture)}." for fixture in scheduled)\n        goals = [num(fixture.get(\'team_goals\')) for fixture in scheduled]\n        clean_sheets = [num(fixture.get(\'clean_sheet_probability\')) for fixture in scheduled]\n        if scheduled and all(value is not None for value in goals):\n            text += f" My calculations put them at {sum(goals):.2f} expected team goals."\n        if clean_sheets and all(value is not None for value in clean_sheets):\n            probability = 1 - math.prod(1 - value for value in clean_sheets)\n            outcome = \'at least one clean sheet\' if len(scheduled) > 1 else \'a clean sheet\'\n            text += f" The estimated chance of {outcome} is {probability:.0%}."\n        out.append(seg(\'fh_team_\' + str(row[\'team_id\']), \'team\', text, team_id=row[\'team_id\']))\n        players = row.get(\'players\', [])[:2]\n        for player in players:\n            text = f"{player[\'player_name\']} is an option at £{player[\'price\']:.1f} million. "\n            text += f"He projects {player[\'projected_points\']:.1f} points, with roughly {player[\'expected_minutes\']:.0f} minutes expected."\n            if player[\'position\'] in (\'MID\', \'FWD\') and num(player.get(\'xgi90\')) is not None:\n                text += f" His expected goal involvement is {player[\'xgi90\']:.2f} per ninety minutes."\n            else:\n                text += " I\'ll compare his defensive opportunity with the other options at this price."\n            out.append(seg(\'fh_target_\' + str(player[\'player_id\']), \'player\', text,\n                team_id=row[\'team_id\'], player_id=player[\'player_id\'], player_ids=[player[\'player_id\']]))\n        preferred = next((player for player in players if player[\'player_id\'] == row.get(\'preferred_player_id\')), None)\n        if preferred:\n            text = f"{preferred[\'player_name\']} offers the most projected points per million in this comparison. "\n            text += "I\'ll still check the position, minutes and squad fit before choosing."\n            out.append(seg(\'fh_compare_\' + str(row[\'team_id\']), \'comparison\', text,\n                team_id=row[\'team_id\'], player_ids=[preferred[\'player_id\']]))\n    for row in case[\'avoids\']:\n        scheduled = [fixture for week in row[\'weeks\'] for fixture in week[\'fixtures\']]\n        club = full_club(row[\'club\'], row[\'name\'])\n        picked = [player for player in selected if player.get(\'team_id\') == row[\'team_id\']]\n        text = f"I\'m more cautious about {club}. "\n        text += \' \'.join(f"They face {fixture_words(fixture)}." for fixture in scheduled) if scheduled else \'They have no scheduled fixture this Gameweek.\'\n        if picked:\n            text += f" {name(picked[0])} still fits my draft on his individual projection."\n            text += " A difficult team fixture doesn\'t rule out every player."\n        else:\n            text += " I\'d compare the stronger one-week matchups first."\n        out.append(seg(\'fh_avoid_\' + str(row[\'team_id\']), \'avoid\', text, team_id=row[\'team_id\']))\n    out.append(seg(\'fh_fixture_end\', \'summary\',\n        "Those are the fixtures I\'m considering. "\n        "Next, I\'ll use minutes, player roles and prices to build a squad that fits the budget."))\n    return out\n\ndef squad_segments(p):\n    """Use the creator\'s script structure with the exact selected slide data."""\n    roles = (\'GK\', \'DEF\', \'MID\', \'FWD\')\n    def role(player):\n        return \'GK\' if player[\'position\'] == \'GKP\' else player[\'position\']\n    starters = sorted(p[\'starting_xi\'], key=lambda player:\n        (roles.index(role(player)), -float(player[\'projected_points\']), int(player[\'id\'])))\n    bench = list(p[\'bench\'])\n    if len(starters) != 11 or len(bench) != 4 or len({player[\'id\'] for player in starters + bench}) != 15:\n        raise ValueError(\'Squad speech requires the same eleven starters and four distinct substitutes as the slide\')\n    groups = {position: [player for player in starters if role(player) == position] for position in roles}\n    formation = \'-\'.join(str(len(groups[position])) for position in (\'DEF\', \'MID\', \'FWD\'))\n    if len(groups[\'GK\']) != 1 or formation != p[\'formation\']:\n        raise ValueError(\'Spoken formation differs from the selected pitch lineup\')\n    expected_xi = sum(float(player[\'projected_points\']) for player in starters)\n    captain = next((player for player in starters if player[\'id\'] == p[\'captain\'][\'id\']), None)\n    if captain is None or abs(float(captain[\'projected_points\']) - float(p[\'captain\'][\'projected_points\'])) > 1e-6:\n        raise ValueError(\'Spoken captain bonus does not match a selected starter\')\n    if (abs(expected_xi - float(p[\'xi_xpts\'])) > 1e-6\n            or abs(expected_xi + float(captain[\'projected_points\']) - float(p[\'captain_adjusted_xpts\'])) > 1e-6):\n        raise ValueError(\'Spoken totals must equal the starting XI plus exactly one captain bonus\')\n    expected_cost = sum(float(player[\'price\']) for player in starters + bench)\n    if abs(expected_cost - float(p[\'total_cost\'])) > 1e-6:\n        raise ValueError(\'Spoken squad cost differs from the selected fifteen prices\')\n    out = [seg(\'fh_squad_open\', \'title\',\n        f"This is my Free Hit for Gameweek {p[\'next_gw\']}: a {p[\'formation\']} formation, with £{p[\'total_cost\']:.1f} million committed. "\n        "I\'ll build the team from the back, starting in goal, then work through the defence, midfield and attack.")]\n    openings = {\'GK\': \'In goal, I have\', \'DEF\': \'In defence, I have\', \'MID\': \'In midfield, I have\', \'FWD\': \'Up front, I have\'}\n    for index, player in enumerate(starters):\n        position = role(player)\n        group = groups[position]\n        position_index = next(i for i, candidate in enumerate(group) if candidate[\'id\'] == player[\'id\'])\n        player_name = name(player)\n        club = full_club(player[\'club\'], player[\'club_name\'])\n        if position_index == 0:\n            lead = f"{openings[position]} {player_name} from {club}."\n        elif position_index == len(group) - 1:\n            closing = \'Partnering him up front is\' if position == \'FWD\' else \'Completing the midfield is\' if position == \'MID\' else \'Completing the back line is\'\n            lead = f"{closing} {player_name} of {club}."\n        elif position_index % 2:\n            lead = f"Alongside him, {player_name} gives me another option from {club}."\n        else:\n            lead = f"I also have {player_name}, representing {club}."\n        points = num(player.get(\'projected_points\'))\n        minutes = num(player.get(\'minutes\'))\n        analysis = player.get(\'analysis\', {})\n        price = float(player[\'price\'])\n        fixture = fx_words(player)\n        if index % 3 == 0:\n            detail = f" He faces {fixture}. At £{price:.1f} million, the model gives him {points:.1f} points"\n            detail += f", with around {minutes:.0f} minutes expected." if minutes is not None else \'.\'\n        elif index % 3 == 1:\n            detail = f" The fixture is {fixture}. I\'m spending £{price:.1f} million for a {points:.1f}-point projection."\n            if minutes is not None:\n                detail += f" His playing-time estimate is {minutes:.0f} minutes."\n        else:\n            detail = f" For £{price:.1f} million, he projects {points:.1f} points against {fixture}."\n            if minutes is not None:\n                detail += f" The model expects roughly {minutes:.0f} minutes."\n        clean_sheet = num(analysis.get(\'clean_sheet_probability\'))\n        goal_involvement = num(analysis.get(\'xgi90\'))\n        if position in (\'GK\', \'DEF\') and clean_sheet is not None:\n            detail += f" The defensive opportunity is a {clean_sheet:.0%} model clean-sheet chance."\n        elif position in (\'MID\', \'FWD\') and goal_involvement is not None:\n            detail += f" His underlying expected goal involvement is {goal_involvement:.2f} per ninety."\n        if analysis.get(\'availability\', {}).get(\'flagged\'):\n            detail += \' I will check his availability before the deadline.\'\n        elif minutes is not None and minutes < 75:\n            detail += \' Playing time is the point I would watch here.\'\n        out.append(seg(\'fh_squad_\' + str(player[\'id\']), \'players\', lead + detail,\n            player_ids=[player[\'id\']], hero_player_id=player[\'id\'], position=position,\n            position_label={\'GK\': \'IN GOAL\', \'DEF\': \'IN DEFENCE\', \'MID\': \'IN MIDFIELD\', \'FWD\': \'UP FRONT\'}[position],\n            reveal_order=index, flight_after_speech=True))\n    bench_openings = (\'On the bench, I have\', \'For further cover, I have\', \'Another reserve is\', \'Completing the bench is\')\n    for index, player in enumerate(bench):\n        points = num(player.get(\'projected_points\'))\n        minutes = num(player.get(\'minutes\'))\n        text = f"{bench_openings[index]} {name(player)} from {full_club(player[\'club\'], player[\'club_name\'])}, at £{player[\'price\']:.1f} million. "\n        text += f"He projects {points:.1f} points" if points is not None else \'His projection needs review\'\n        text += f" with about {minutes:.0f} minutes expected." if minutes is not None else \'.\'\n        text += \' This is substitute cover; his points are separate from the starting-eleven total.\'\n        out.append(seg(\'fh_squad_\' + str(player[\'id\']), \'bench\', text,\n            player_ids=[player[\'id\']], hero_player_id=player[\'id\'], position=role(player),\n            position_label=\'BENCH COVER\', reveal_order=len(starters) + index))\n    out.append(seg(\'fh_squad_total\', \'summary\',\n        f"That completes the team. The eleven project {p[\'xi_xpts\']:.1f} points before captaincy. "\n        f"One additional captain score of {captain[\'projected_points\']:.1f} takes the total to {p[\'captain_adjusted_xpts\']:.1f}. "\n        "The bench stays outside that figure. Let\'s give the armband decision its own comparison."))\n    return out\n\ndef captain_plan(d):\n p=d[\'slide3_plan\'];cap=p[\'captain\'];vice=p[\'vice_captain\']\n eligible=sorted((x for x in p[\'starting_xi\'] if num(x.get(\'projected_points\')) is not None),key=lambda x:(-x[\'projected_points\'],x[\'id\']))\n if len(eligible)!=11 or cap[\'id\']!=eligible[0][\'id\']:\n  raise ValueError(\'Free Hit captain must lead the projected points of the selected legal XI\')\n if vice[\'id\']==cap[\'id\'] or vice[\'id\'] not in {x[\'id\'] for x in eligible}:\n  raise ValueError(\'Free Hit vice-captain must be a different player in the starting XI\')\n if num(cap.get(\'minutes\')) is None or num(vice.get(\'minutes\')) is None:\n  raise ValueError(\'Cannot verify captain minutes: regenerate the Free Hit squad\')\n if num(p.get(\'captain_adjusted_xpts\')) is None or abs(p[\'captain_adjusted_xpts\']-p[\'xi_xpts\']-cap[\'projected_points\'])>1e-6:\n  raise ValueError(\'Captain bonus must be counted exactly once\')\n # Include the actual chosen vice alongside the top non-captain alternative.\n alternatives=[x for x in eligible if x[\'id\'] not in (cap[\'id\'],vice[\'id\'])]\n if not alternatives:raise ValueError(\'Insufficient Free Hit captain alternatives\')\n return dict(version=VERSION,chip=\'freehit\',next_gw=d[\'gameweek\'],captain=cap,vice_captain=vice,candidates=[alternatives[0],cap,vice],squad_plan=p,decision=d[\'freehit_case\'])\n\ndef final_segments(plan):\n    """Creator\'s armband script, populated from the selected squad and card data."""\n    squad = plan[\'squad_plan\']\n    starters = squad[\'starting_xi\']\n    by_id = {player[\'id\']: player for player in starters}\n    if len(starters) != 11 or len(by_id) != 11:\n        raise ValueError(\'Captaincy narration requires the selected starting eleven\')\n    captain_id, vice_id = plan[\'captain\'][\'id\'], plan[\'vice_captain\'][\'id\']\n    if captain_id == vice_id or captain_id not in by_id or vice_id not in by_id:\n        raise ValueError(\'Captain and vice-captain must be distinct selected starters\')\n    if (squad[\'captain\'][\'id\'], squad[\'vice_captain\'][\'id\']) != (captain_id, vice_id):\n        raise ValueError(\'Spoken armbands differ from the slide armbands\')\n    captain, vice = by_id[captain_id], by_id[vice_id]\n    for player in starters:\n        if num(player.get(\'projected_points\')) is None or num(player.get(\'minutes\')) is None or num(player.get(\'price\')) is None or player[\'price\'] <= 0:\n            raise ValueError(\'Captaincy speech needs verified points, minutes and price data\')\n    if captain[\'projected_points\'] < max(player[\'projected_points\'] for player in starters):\n        raise ValueError(\'The spoken points leader must match the selected captain\')\n    xi_points = sum(player[\'projected_points\'] for player in starters)\n    if (num(squad.get(\'xi_xpts\')) is None or num(squad.get(\'captain_adjusted_xpts\')) is None\n            or abs(xi_points - squad[\'xi_xpts\']) > 1e-6\n            or abs(xi_points + captain[\'projected_points\'] - squad[\'captain_adjusted_xpts\']) > 1e-6):\n        raise ValueError(\'Captaincy total must equal the XI plus exactly one captain score\')\n    other_ids = [player[\'id\'] for player in plan[\'candidates\'] if player[\'id\'] not in (captain_id, vice_id)]\n    candidate_ids = other_ids + [vice_id, captain_id]\n    if len(candidate_ids) != len(set(candidate_ids)) or any(identity not in by_id for identity in candidate_ids):\n        raise ValueError(\'Every compared card must be a distinct selected starter\')\n    candidates = [by_id[identity] for identity in candidate_ids]\n    count_words = {1: \'one\', 2: \'two\', 3: \'three\', 4: \'four\', 5: \'five\'}\n    count = count_words.get(len(candidates), str(len(candidates)))\n    out = [seg(\'fh_captain_open\', \'intro\',\n        f"Let\'s settle the armband for Gameweek {plan[\'next_gw\']}. "\n        f"I\'ll compare {count} players from my starting eleven, then make the call.")]\n    for index, player in enumerate(candidates):\n        lead = \'The first option is\' if index == 0 else \'Completing the comparison is\' if index == len(candidates) - 1 else \'Another strong option is\'\n        text = f"{lead} {name(player)}, facing {fx_words(player)}. "\n        text += f"His projection is {player[\'projected_points\']:.1f} points, with around {player[\'minutes\']:.0f} minutes expected. "\n        text += f"Giving him the armband adds another {player[\'projected_points\']:.1f} projected points to the eleven. "\n        text += "That doubles his own score, rather than the team\'s total."\n        out.append(seg(\'fh_contender_\' + str(player[\'id\']), \'candidate\', text,\n            player_id=player[\'id\'], hero_player_id=player[\'id\'], player_ids=[player[\'id\']], slot=index))\n    out.append(seg(\'fh_captain_decision\', \'decision\',\n        f"My captain is {name(captain)}. He leads this eleven on the current projection. "\n        f"The captain bonus is {captain[\'projected_points\']:.1f}, counted once on top of the starting score.",\n        hero_player_id=captain_id, player_ids=[captain_id]))\n    out.append(seg(\'fh_vice\', \'vice\',\n        f"The vice-captain armband goes to {name(vice)}. "\n        "His extra score only comes into play if my captain makes no appearance. "\n        "I\'ll check both players\' availability before the deadline.", hero_player_id=vice_id, player_ids=[vice_id]))\n    value = max(starters, key=lambda player: player[\'projected_points\'] / player[\'price\'])\n    risk = min(starters, key=lambda player: (player[\'minutes\'], player.get(\'starting_probability\') or 0))\n    out.append(seg(\'fh_value\', \'takeaways\',\n        f"One final look at the squad. {name(value)} offers the strongest projected value for his price.",\n        hero_player_id=value[\'id\'], player_ids=[value[\'id\']]))\n    out.append(seg(\'fh_minutes_check\', \'takeaways\',\n        f"{name(risk)} has the lowest expected minutes, so he deserves an availability check.",\n        hero_player_id=risk[\'id\'], player_ids=[risk[\'id\']]))\n    out.append(seg(\'fh_captain_total\', \'takeaways\',\n        f"Including one captain bonus, the team projects {squad[\'captain_adjusted_xpts\']:.1f} points.",\n        hero_player_id=captain_id, player_ids=[captain_id]))\n    decision = plan[\'decision\']\n    text = decision[\'verdict_reason\'] if decision.get(\'scenario_comparison\') else (\n        \'The squad is ready. A verified current-team comparison is still needed before deciding whether the chip is worth using.\')\n    text += \' My original squad returns next Gameweek. The last step is checking the final team news.\'\n    out.append(seg(\'fh_final\', \'outro\', text, hero_player_id=captain_id, player_ids=[captain_id]))\n    return out\n\ndef prepare(d,bootstrap=None):\n d=copy.deepcopy(d);p=d[\'slide3_plan\'];official={int(x[\'id\']):x for x in (bootstrap or {}).get(\'elements\',[])}\n for group in [\'squad\',\'starting_xi\',\'bench\']:\n  for x in p[group]:\n   # Prefer concise common FPL names over long registered names.\n   e=official.get(int(x[\'id\']),{});first=str(e.get(\'first_name\',\'\')).split();web=x[\'name\']\n   x[\'spoken_name\']=web if len(web.split())>=2 else ((first[0]+\' \'+web) if first and first[0].casefold()!=web.casefold() else first[0]+\' \'+str(e.get(\'second_name\',\'\')).split()[-1] if first and str(e.get(\'second_name\',\'\')).strip() else x.get(\'spoken_name\',web))\n   x[\'club_name\']=full_club(x[\'club\'],x.get(\'club_name\',\'\'))\n byid={x[\'id\']:x for x in p[\'squad\']};p[\'captain\']=byid[p[\'captain\'][\'id\']];p[\'vice_captain\']=byid[p[\'vice_captain\'][\'id\']]\n p[\'narration_timeline\']=squad_segments(p);d[\'freehit_case\']=intro_case(d,bootstrap)\n d[\'fixture_case\'][\'narration_timeline\']=fixture_segments(d[\'fixture_case\'],p[\'starting_xi\']);d[\'fixture_case\'][\'narration_text\']=\' \'.join(x[\'narration_text\'] for x in d[\'fixture_case\'][\'narration_timeline\'])\n d[\'captain_pick\']=captain_plan(d);return d\n\ndef uri(path):\n p=Path(path) if path else None\n return \'data:image/png;base64,\'+base64.b64encode(p.read_bytes()).decode() if p and p.is_file() else \'\'\ndef render_slide(kind,d,timeline,media=None,badges=None,approved_cards=None):\n esc=lambda x:html.escape(str(x),quote=True);media=media or {};badges=badges or {};approved_cards=approved_cards or {};p=d[\'slide3_plan\'];gw=d[\'gameweek\']\n if kind==1:\n  case=d[\'freehit_case\'];labels=\' • \'.join(case.get(\'intro_indicators\',[]))\n  return \'<style>.fh-personal{color:#f2f6ff;padding:80px;font:42px Arial}.fh-personal h1{font-size:86px}.fh-personal h2{font-size:68px;color:#55e9ad}</style><section class="fh-personal"><h1>FREE HIT: NOW OR WAIT?</h1><p>\'+esc(labels)+\'</p><h2>MY FREE HIT DRAFT</h2><p>GAMEWEEK \'+str(gw)+\' ONLY</p><p>Your original squad returns next Gameweek</p><footer>NEXT: THIS WEEK’S FIXTURES</footer></section>\'\n\n def logo(code,cls=\'fp-logo\'):\n  src=uri(badges.get(code,{}).get(\'path\'))\n  return f\'<img class="{cls}" src="{src}" alt="{esc(full_club(code))} badge">\' if src else f\'<span class="{cls} fp-logo-fallback">{esc(code)}</span>\'\n def photo(x,cls=\'fp-photo\'):\n  pid=x.get(\'id\',x.get(\'player_id\'));record=media.get(str(pid),media.get(pid,{}));src=uri(record.get(\'photo\')) if record.get(\'photo_kind\')==\'player\' else \'\'\n  return f\'<img class="{cls}" src="{src}" alt="{esc(name(x))}">\' if src else f\'<div class="{cls} fp-avatar">{logo(x.get("club",""))}<b>{esc("".join(v[0] for v in name(x).split()[:2]))}</b></div>\'\n def fdr(f):\n  style=f.get(\'fdr_style\') or difficulty(f.get(\'fdr\'));return f\'<span class="fp-fdr" style="background:{style["background"]};color:{style["text"]}">{esc(style["label"])}</span>\'\n def fixture(x):return \' / \'.join(esc(f.get(\'opponent\',\'\'))+(\' [H]\' if f.get(\'home\') else \' [A]\')+\' \'+fdr(f) for f in fixtures(x)) or \'FIXTURE UNAVAILABLE\'\n def minis(xs):return \'<div class="fp-intro-minis">\'+\'\'.join(\'<article>\'+photo(x)+\'<b>\'+esc(x[\'name\'])+\'</b></article>\' for x in xs)+\'</div>\'\n def metric(label,value):return f\'<div class="fp-metric"><small>{label}</small><b>{esc(value)}</b></div>\'\n def approved_card(player):\n  pid=int(player.get(\'id\',player.get(\'player_id\')))\n  path=approved_cards.get(pid) or approved_cards.get(str(pid))\n  if not path or not Path(path).is_file():raise FileNotFoundError(f\'Original notebook Hero Card is missing for player {pid}\')\n  return f\'<img class="fp-approved-card" src="{uri(path)}" alt="Original notebook hero card for {esc(name(player))}">\'\n def shortmetrics(x, compact_labels=False):\n  a=x.get(\'analysis\',{});val=a.get(\'clean_sheet_probability\') if x.get(\'position\') in (\'GK\',\'DEF\') else a.get(\'xgi90\');label=\'Clean-sheet chance\' if x.get(\'position\') in (\'GK\',\'DEF\') else \'Expected goal involvement / 90\';fmt=(f\'{val:.0%}\' if x.get(\'position\') in (\'GK\',\'DEF\') else f\'{val:.2f}\') if val is not None else \'UNAVAILABLE\'\n  if compact_labels:\n   label=\'CLEAN SHEET %\' if x.get(\'position\') in (\'GK\',\'DEF\') else \'xGI / 90\'\n  return metric(\'PROJECTED PTS\' if compact_labels else \'Projected points\',f\'{x["projected_points"]:.1f}\' if num(x.get(\'projected_points\')) is not None else \'UNAVAILABLE\')+metric(\'EXPECTED MIN\' if compact_labels else \'Expected minutes\',f\'{x["minutes"]:.0f}\' if num(x.get(\'minutes\')) is not None else \'UNAVAILABLE\')+metric(label,fmt)\n if kind==1:\n  c=d[\'freehit_case\'];r=c[\'focus_team\'];fs=[f for w in r[\'weeks\'] for f in w[\'fixtures\']];goals=sum(f[\'team_goals\'] for f in fs) if fs and all(num(f.get(\'team_goals\')) is not None for f in fs) else None;cs=fs[0].get(\'clean_sheet_probability\') if len(fs)==1 else None;cap=p[\'captain\'];comp=c[\'scenario_comparison\']\n  comparison=f\'CURRENT {comp["current"]:.1f} · FREE HIT {comp["freehit"]:.1f} · GAIN {comp["gain"]:+.1f}\' if comp else \'VERIFIED CURRENT-TEAM BASELINE REQUIRED\'\n  content=f\'<header class="fp-section-header"><h2>ONE GAMEWEEK. ONE CHIP DECISION.</h2><p>Build the opportunity, then assess the benefit</p></header><div class="fp-points" data-fp-focus="comparison">{metric("Starting eleven",(""+format(p["xi_xpts"],".1f")+""))}{metric("Captain bonus",("+"+format(cap["projected_points"],".1f")+""))}{metric("Total projected points",(""+format(p["captain_adjusted_xpts"],".1f")+""))}<div><b>{comparison}</b><p>Bench excluded. Captain bonus counted once.</p></div></div><div class="fp-intro-grid"><section class="fp-panel" data-fp-focus="fixture"><h2>FIXTURE OPPORTUNITY</h2><div class="fp-team-title">{logo(r["club"])}<b>{esc(full_club(r["club"],r["name"]))}</b></div><div class="fp-intro-fixtures">\'+\'\'.join(f\'<div>{logo(f["opponent"])}<b>{esc(fixture_words(f))}</b>{fdr(f)}</div>\' for f in fs)+f\'</div><div class="fp-metrics">{metric("Projected team goals",f"{goals:.2f}" if goals is not None else "UNAVAILABLE")}{metric("Clean-sheet chance",f"{cs:.0%}" if cs is not None else "UNAVAILABLE")}</div><p class="fp-note">Model estimates for Gameweek {gw}</p>{minis([x for x in p["starting_xi"] if x["team_id"]==r["team_id"]])}</section><section class="fp-panel" data-fp-focus="structure"><h2>SQUAD STRUCTURE</h2><strong class="fp-big">{p["formation"]}</strong><div class="fp-metrics">{metric("Squad cost",("\\u00a3"+format(p["total_cost"],".1f")+"M"))}{metric("In the bank",("\\u00a3"+format(p["bank"],".1f")+"M"))}</div><ul><li>Eleven starters</li><li>Low-cost bench cover</li><li>Original team returns next week</li></ul>{minis(p["bench"])}</section><section class="fp-panel fp-intro-captain" data-fp-focus="captain"><h2>CAPTAINCY OPPORTUNITY</h2>{photo(cap)}<h3>{esc(name(cap))}</h3><div class="fp-club">{logo(cap["club"])}{esc(cap["club_name"])}</div><div class="fp-intro-fixtures">{fixture(cap)}</div><div class="fp-metrics">{metric("Projected points",(""+format(cap["projected_points"],".1f")+""))}{metric("Expected minutes",(""+format(cap["minutes"],".0f")+""))}</div></section></div><footer class="fp-decision" data-fp-focus="decision"><b>{c["decision_label"]}</b><p>\'+(\'Compare available free transfers before spending the chip.\' if not comp else esc(c[\'verdict_reason\']))+\'</p></footer>\'\n elif kind==2:\n  case=d[\'fixture_case\'];targets={r[\'team_id\']:i+1 for i,r in enumerate(case[\'targets\'])};avoids={r[\'team_id\'] for r in case[\'avoids\']};rows=[];panels=[]\n  for row in case[\'rows\']:\n   fs=[f for w in row[\'weeks\'] for f in w[\'fixtures\']];goals=sum(f[\'team_goals\'] for f in fs) if fs and all(num(f.get(\'team_goals\')) is not None for f in fs) else None;cs=fs[0].get(\'clean_sheet_probability\') if len(fs)==1 else None\n   cell=\' / \'.join(esc(f[\'opponent\'])+(\'[H]-\' if f[\'home\'] else \'[A]-\')+str(difficulty(f[\'fdr\'])[\'label\']) for f in fs) or \'BLANK\';style=difficulty(row.get(\'average_fdr\'));tag=f\'T{targets[row["team_id"]]}\' if row[\'team_id\'] in targets else \'AVOID\' if row[\'team_id\'] in avoids else \'\'\n   rows.append(f\'<tr data-fp-team="{row["team_id"]}"><td><span class="fp-rank">{tag}</span>{logo(row["club"])}<b>{esc(full_club(row["club"],row["name"]))}</b></td><td><span class="fp-fixture-cell" style="background:{style["background"]};color:{style["text"]}">{cell}</span></td><td>{f"{goals:.2f}" if goals is not None else "—"}</td><td>{f"{cs:.0%}" if cs is not None else "—"}</td></tr>\')\n  for row in case[\'targets\']:\n   players=[]\n   for x in row[\'players\'][:2]:\n    players.append(f\'<article class="fp-shortlist" data-fp-player="{x["player_id"]}">{photo(x)}<div><h3>{esc(x["player_name"])}</h3><b>£{x["price"]:.1f}M</b><div class="fp-metrics">{metric("Projected points",(""+format(x["projected_points"],".1f")+""))}{metric("Expected minutes",(""+format(x["expected_minutes"],".0f")+""))}</div><p>\'+(\'Expected goal involvement / 90: \'+f\'{x["xgi90"]:.2f}\' if num(x.get(\'xgi90\')) is not None else \'Underlying involvement unavailable\')+\'</p></div></article>\')\n   preferred=next((x for x in row[\'players\'][:2] if x[\'player_id\']==row.get(\'preferred_player_id\')),None)\n   preference=(\'<b>PREFERRED VALUE: \'+esc(preferred[\'player_name\'])+\'</b><span>\'+f"{preferred[\'value\']:.2f}"+\' PROJECTED POINTS PER MILLION</span>\') if preferred else \'\'\n   panels.append(f\'<section class="fp-team-detail" data-fp-detail="{row["team_id"]}"><header>{logo(row["club"])}<div><small>TARGET {targets[row["team_id"]]}</small><h2>{esc(full_club(row["club"],row["name"]))}</h2></div></header>\'+\'\'.join(players)+f\'<div class="fp-detail-note">{preference}<p>Shortlist candidates. Check minutes and full squad budget.</p></div></section>\')\n  summary=\'<section class="fp-fixture-summary"><h2>FIVE TEAMS TO TARGET</h2>\'+\'\'.join(f\'<div>{logo(r["club"])}<b>{esc(full_club(r["club"],r["name"]))}</b></div>\' for r in case[\'targets\'])+\'<h2>THREE CAUTIOUS MATCHUPS</h2>\'+\'\'.join(f\'<div>{logo(r["club"])}<b>{esc(full_club(r["club"],r["name"]))}</b></div>\' for r in case[\'avoids\'])+\'<p>Fixture rank guides the shortlist. Individual returns decide the final picks.</p></section>\'\n  content=f\'<header class="fp-section-header"><h2>GAMEWEEK {gw} FIXTURE LANDSCAPE</h2><p>Five targets. Three cautious matchups.</p></header><div class="fp-fixture-layout"><main class="fp-panel fp-ticker"><table><thead><tr><th>TEAM</th><th>FIXTURE / DIFFICULTY</th><th>PROJECTED<br>GOALS</th><th>CLEAN<br>SHEET</th></tr></thead><tbody>\'+\'\'.join(rows)+\'</tbody></table><p>One-week team model. Colours show fixture difficulty.</p></main><aside class="fp-panel fp-details">\'+\'\'.join(panels)+summary+\'</aside></div>\'\n elif kind==3:\n  starters=p[\'starting_xi\'];bypos={pos:[x for x in starters if x[\'position\']==pos] for pos in (\'GK\',\'DEF\',\'MID\',\'FWD\')};nodes=[];heroes=[];bench=[]\n  for pos,ys in [(\'GK\',14),(\'DEF\',38),(\'MID\',62),(\'FWD\',85)]:\n   for i,x in enumerate(bypos[pos]):\n    left=(i+1)*100/(len(bypos[pos])+1);label=\'C\' if x[\'id\']==p[\'captain\'][\'id\'] else \'VC\' if x[\'id\']==p[\'vice_captain\'][\'id\'] else \'\'\n    nodes.append(f\'<div class="fp-pitch-player" data-fp-pitch="{x["id"]}" style="left:{left}%;top:{ys}%">{photo(x)}<b>{esc(x["name"])} <em>{label}</em></b><span>{pos} · £{x["price"]:.1f}M</span></div>\')\n  for x in p[\'bench\']:bench.append(f\'<article data-fp-bench="{x["id"]}">{photo(x)}<b>{esc(x["name"])}</b><span>{x["position"]} · £{x["price"]:.1f}M</span></article>\')\n  for x in sorted(p[\'starting_xi\'],key=lambda x:(\'GK\',\'DEF\',\'MID\',\'FWD\').index(x[\'position\']))+sorted(p[\'bench\'],key=lambda x:x[\'position\']!=\'GK\'):\n   heroes.append(f\'<article class="fp-player-hero" data-fp-hero="{x["id"]}">{approved_card(x)}<div class="fp-card-meta"><b>{esc(name(x))}</b><span>{fixture(x)} · {x["projected_points"]:.1f} xPts</span></div><div class="fp-why-popup"><strong>WHY THIS FREE HIT PICK</strong><p>{esc(short_reason(x))}</p><small>{x["position"]} · £{x["price"]:.1f}M · {x["projected_points"]:.1f} xPts</small></div></article>\')\n  # Present only the selected 11 in the primary visualization; bench cover stays secondary.\n  best_projection=max(max(float(x[\'projected_points\']),0.0) for x in starters) or 1.0\n  top_ids={x[\'id\'] for x in sorted(starters,key=lambda x:(-float(x[\'projected_points\']),x[\'id\']))[:3]}\n  xi_cards=[]\n  for x in sorted(starters,key=lambda x:((\'GK\',\'DEF\',\'MID\',\'FWD\').index(x[\'position\']),-float(x[\'projected_points\']),x[\'id\'])):\n   percentage=max(0.0,min(100.0,100.0*float(x[\'projected_points\'])/best_projection))\n   highlight=\' fp-xpts-leader\' if x[\'id\'] in top_ids else \'\'\n   badge=\'C\' if x[\'id\']==p[\'captain\'][\'id\'] else \'VC\' if x[\'id\']==p[\'vice_captain\'][\'id\'] else \'\'\n   xi_cards.append(f\'<article class="fp-xpts-player{highlight}" data-fp-squad-row="{x["id"]}"><div class="fp-xpts-player-head"><span class="fp-xpts-role">{x["position"]}</span><b title="{esc(x["name"])}">{esc(x["name"])}</b><em>{badge}</em><strong>{x["projected_points"]:.1f}<small> xPts</small></strong></div><div class="fp-xpts-track"><div style="width:{percentage:.2f}%"></div></div></article>\')\n  bench_cards=[]\n  for index,x in enumerate(p[\'bench\'],1):\n   order=\'GK\' if x[\'position\']==\'GK\' else f\'SUB {index - (1 if any(b["position"]=="GK" for b in p["bench"][:index]) else 0)}\'\n   minutes_watch=\' · MINUTES WATCH\' if not x.get(\'secure\',True) else \'\'\n   bench_cards.append(f\'<article class="fp-xpts-sub" data-fp-squad-row="{x["id"]}"><small>{esc(order)} · {x["position"]}{minutes_watch}</small><b title="{esc(x["name"])}">{esc(x["name"])}</b><strong>{x["projected_points"]:.1f}<small> xPts</small></strong></article>\')\n  flights=[]\n  for x in starters:\n   flights.append(f\'<article class="fp-card-flight" data-fp-flight="{x["id"]}">{approved_card(x)}<b>{esc(x["name"])}</b><small>{x["projected_points"]:.1f} xPTS · {esc(x["club"])}</small></article>\')\n  starter_cards_html=\'\'.join(xi_cards)\n  bench_cards_html=\'\'.join(bench_cards)\n  breakdown=f\'<section class="fp-panel fp-xpts-dashboard" aria-label="Starting XI xPts breakdown"><header class="fp-xpts-dashboard-header"><h2>STARTING XI / xPTS BREAKDOWN</h2><span>11 STARTERS · {esc(p["formation"])} · GW {gw}</span></header><div class="fp-xpts-grid">{starter_cards_html}</div><section class="fp-xpts-bench"><h3>BENCH COVER <span>SEPARATE FROM XI</span></h3><div>{bench_cards_html}</div></section><footer class="fp-xpts-total"><b>XI {p["xi_xpts"]:.1f} <span>+</span> CAPTAIN {p["captain"]["projected_points"]:.1f} <span>=</span> {p["captain_adjusted_xpts"]:.1f}</b><small>BENCH EXCLUDED · CAPTAIN BONUS ONCE</small></footer></section>\'\n  content=f\'<div class="fp-squad-layout"><main class="fp-pitch-side"><header class="fp-section-header"><h2>FREE HIT STARTING XI</h2><b>{p["formation"]}</b></header><div class="fp-pitch"><div class="fp-boundary"></div><div class="fp-halfway"></div><div class="fp-circle"></div><div class="fp-top-goal"></div><div class="fp-top-box"></div>\'+\'\'.join(nodes)+\'</div><section class="fp-bench"><h3>BENCH — MINIMUM COVER</h3><div>\'+\'\'.join(bench)+f\'</div></section></main><aside class="fp-squad-side"><header class="fp-section-header"><h2>ONE-WEEK PLAN</h2><b>£{p["bank"]:.1f}M BANK</b></header><section class="fp-hero-stage">\'+\'\'.join(heroes)+f\'<article class="fp-squad-overview"><h2>THE SELECTED FIFTEEN</h2><div class="fp-metrics">{metric("Squad cost",("\\u00a3"+format(p["total_cost"],".1f")+"M"))}{metric("XI projection",(""+format(p["xi_xpts"],".1f")+""))}{metric("With captain",(""+format(p["captain_adjusted_xpts"],".1f")+""))}</div><p>Captain: {esc(name(p["captain"]))}</p><p>Vice-captain: {esc(name(p["vice_captain"]))}</p><p>One Gameweek only. Original squad returns afterwards.</p></article></section>{breakdown}</aside></div>\'+\'\'.join(flights)\n else:\n  plan=d[\'captain_pick\'];cards=[]\n  for i,x in enumerate(plan[\'candidates\']):\n   cards.append(f\'<article class="fp-captain-card {"fp-vortex-pick" if i==1 else ""}" data-fp-contender="{x["id"]}"><small>{"CAPTAIN · BEST GAMEWEEK xPts" if i==1 else "VICE-CAPTAIN" if x["id"]==plan["vice_captain"]["id"] else "ALTERNATIVE"}</small>{photo(x)}<h2>{esc(name(x))}</h2><div class="fp-club">{logo(x["club"])}{esc(x["club_name"])}</div><div class="fp-captain-fixture">{fixture(x)}</div><div class="fp-metrics">{shortmetrics(x, compact_labels=True)}</div><p>{esc(short_reason(x))}</p></article>\')\n  value=max(p[\'starting_xi\'],key=lambda x:x[\'projected_points\']/x[\'price\']);risk=min(p[\'starting_xi\'],key=lambda x:(x.get(\'starting_probability\') or 0,x.get(\'minutes\') or 0));c=d[\'freehit_case\'];v=p[\'vice_captain\']\n  visible_hero_ids={x[\'id\']:x for x in plan[\'candidates\']}\n  visible_hero_ids[v[\'id\']]=v\n  right_heroes=\'\'.join(f\'<figure class="fp-approved-captain" data-fp-captain-hero="{x["id"]}">{approved_card(x)}</figure>\' for x in visible_hero_ids.values())\n  content=f\'<header class="fp-section-header"><h2>GAMEWEEK {gw} · CAPTAINCY CALL</h2><p>One-week xPts · +{p["captain"]["projected_points"]:.1f} captain bonus, counted once</p></header><div class="fp-final-layout"><main class="fp-captain-three">\'+\'\'.join(cards)+f\'</main><aside class="fp-final-takeaways"><div class="fp-approved-hero-stage">\'+right_heroes+\'</div><section class="fp-panel fp-vice" data-fp-focus="vice"><h2>VICE-CAPTAIN</h2><header>{photo(v)}<div><h3>{esc(name(v))}</h3><div class="fp-club">{logo(v["club"])}{esc(v["club_name"])}</div></div></header><div class="fp-vice-projection"><strong>{v["projected_points"]:.1f} xPts</strong><span>{v["minutes"]:.0f} expected minutes</span></div><p>Recheck fitness and expected minutes</p></section><section class="fp-panel" data-fp-focus="takeaways"><h2>FINAL TAKEAWAYS</h2><div class="fp-takeaway"><small>BEST VALUE · xPts / £m</small><b>{esc(name(value))}</b><span>{value["projected_points"]/value["price"]:.2f} xPts / £m · £{value["price"]:.1f}M</span></div><div class="fp-takeaway"><small>MINUTES TO REVIEW</small><b>{esc(name(risk))}</b><span>{risk["minutes"]:.0f} estimated minutes · Review team news</span></div><div class="fp-metrics">{metric("XI with captain",(""+format(p["captain_adjusted_xpts"],".1f")+""))}</div></section></aside></div><footer class="fp-decision" data-fp-focus="outro"><b>{c["decision_label"]}</b><p>Original squad returns next week. Check team news before the deadline.</p></footer>\'\n payload=json.dumps(dict(kind=kind,timeline=timeline,captain_id=p[\'captain\'][\'id\']),ensure_ascii=False).replace(\'<\',\'\\\\u003c\')\n return \'<style>\'+CSS+\'</style><section class="fp-slide fp-slide-\'+str(kind)+\'">\'+content+\'<script class="fp-data" type="application/json">\'+payload+\'</script></section>\'\n\nCSS=\'\'\'\n#content .fp-slide{height:100%;width:100%;font-family:Arial,sans-serif;color:#f5f9ff;text-transform:uppercase;overflow:hidden;display:grid;gap:22px}.fp-slide *{box-sizing:border-box;min-width:0}.fp-slide h2,.fp-slide h3,.fp-slide p{margin:0;padding:0;min-height:0;height:auto}.fp-section-header{display:flex;align-items:center;justify-content:space-between;gap:24px;padding:20px 28px;border-left:6px solid #95deff;background:#071a2df2}.fp-section-header h2{font-size:48px;line-height:1.1;color:#fff}.fp-section-header p,.fp-section-header>b{font-size:35px;line-height:1.15;color:#b7e6fa}.fp-panel{border:2px solid #4c708b;border-radius:17px;background:#081b2ef0;padding:30px;overflow:hidden;min-height:0}.fp-panel h2{font-size:42px;color:#8ddfff;line-height:1.1;margin-bottom:25px}.fp-panel h3{font-size:45px;line-height:1.1}.fp-logo{width:65px;height:65px;object-fit:contain;flex-shrink:0}.fp-photo{width:210px;height:250px;object-fit:contain;flex-shrink:0}.fp-club{display:flex;align-items:center;gap:16px;font-size:32px;color:#cbe8f6;line-height:1.1}.fp-club .fp-logo{width:48px;height:48px}.fp-metrics{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:18px}.fp-metric{background:#173349;border:1px solid #3f637b;border-radius:10px;padding:20px 22px;min-height:0}.fp-metric>small{display:block;font-size:30px;line-height:1.12;color:#beddeb;font-weight:700}.fp-metric>b{display:block;font-size:66px;line-height:1.1;color:#fff;margin-top:12px}.fp-fdr{display:inline-flex;align-items:center;justify-content:center;padding:6px 14px;min-width:65px;border-radius:6px;font-size:33px;font-weight:900;vertical-align:middle}.fp-note{font-size:31px;color:#badbea;margin-top:22px!important}.fp-decision{display:flex;align-items:center;justify-content:space-between;gap:28px;border:2px solid #88bf9f;border-radius:15px;padding:24px 30px;background:#103b37;min-height:0}.fp-decision>b{font-size:43px;color:#d4ff91;line-height:1.1}.fp-decision>p{font-size:34px;line-height:1.15;color:#fff;max-width:65%}.fp-avatar{display:flex;flex-direction:column;align-items:center;justify-content:center;background:#234151;border-radius:16px}.fp-avatar b{font-size:55px}.fp-logo-fallback{display:grid;place-items:center;background:#234151;border-radius:9px;font-size:23px;font-weight:900}.fp-active{border-color:#d4ee94!important;box-shadow:inset 0 0 20px #c0ff7518,0 0 13px #d9ff8c33}\n.fp-slide-1{grid-template-rows:110px 210px minmax(0,1fr) 150px}.fp-points{display:grid;grid-template-columns:1fr 1fr 1.2fr 2.2fr;gap:22px;min-height:0}.fp-points>.fp-metric{padding:24px 28px}.fp-points>.fp-metric>b{font-size:91px;color:#dbff9d;margin-top:15px}.fp-points>div:last-child{display:flex;flex-direction:column;justify-content:center;background:#10283ced;padding:24px;border-radius:13px;border:2px solid #547486}.fp-points>div:last-child>b{font-size:36px;color:#ffe49a;line-height:1.2}.fp-points>div:last-child>p{font-size:30px;line-height:1.2;margin-top:16px;color:#c9e6f5}.fp-intro-grid{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:24px;min-height:0}.fp-team-title{display:flex;align-items:center;gap:28px;margin:48px 0 38px}.fp-team-title>.fp-logo{width:145px;height:145px}.fp-team-title>b{font-size:63px;color:#fff;line-height:1.1}.fp-intro-fixtures{font-size:39px;line-height:1.3;margin:20px 0 40px}.fp-intro-fixtures>div{display:flex;align-items:center;gap:20px;padding:22px;background:#123047;border-radius:12px}.fp-intro-fixtures .fp-fdr{margin-left:auto}.fp-big{display:block;font-size:155px;line-height:1.1;color:#d5ff94;margin:45px 0 55px}.fp-panel ul{padding-left:45px;margin:45px 0 0;display:grid;gap:33px}.fp-panel li{font-size:43px;line-height:1.2}.fp-intro-captain{position:relative;display:flex;flex-direction:column}.fp-intro-captain>.fp-photo{width:330px;height:320px;margin:0 auto 14px}.fp-intro-captain h3{font-size:57px;text-align:center;color:#fff;margin:12px 0 20px}.fp-intro-captain>.fp-club{justify-content:center}.fp-intro-captain>.fp-intro-fixtures{margin:25px 0;font-size:39px;text-align:center}.fp-intro-captain>.fp-metrics{margin-top:auto}\n.fp-slide-2{grid-template-rows:110px minmax(0,1fr)}.fp-fixture-layout{display:grid;grid-template-columns:59fr 41fr;gap:26px;min-height:0}.fp-ticker{padding:20px}.fp-ticker table{width:100%;height:calc(100% - 65px);table-layout:fixed;border-collapse:collapse}.fp-ticker th{font-size:27px;text-align:left;color:#9be0ff;line-height:1.1;padding:12px 10px}.fp-ticker th:first-child{width:37%}.fp-ticker th:nth-child(2){width:33%}.fp-ticker th:nth-child(3){width:16%}.fp-ticker td{font-size:32px;padding:6px 10px;border-bottom:2px solid #253f54;font-weight:700}.fp-ticker td:first-child{display:flex;align-items:center;gap:12px;height:100%}.fp-ticker td:first-child>b{font-size:30px;line-height:1.05}.fp-ticker .fp-logo{height:47px;width:47px}.fp-rank{font-size:21px;width:60px;color:#cffb98;flex-shrink:0}.fp-fixture-cell{display:flex;justify-content:center;align-items:center;min-height:44px;border-radius:6px;padding:6px 9px;font-size:29px;line-height:1.1}.fp-ticker>p{font-size:28px;margin-top:18px;color:#b9d7e8;line-height:1.15}.fp-details{position:relative;padding:28px}.fp-team-detail,.fp-fixture-summary{position:absolute;inset:28px;visibility:hidden;overflow:hidden}.fp-team-detail>header{display:flex;gap:24px;align-items:center;height:145px;margin-bottom:25px}.fp-team-detail>header>.fp-logo{height:125px;width:125px}.fp-team-detail header h2{font-size:61px;margin:12px 0 0}.fp-team-detail header small{font-size:32px;color:#d8ff96;font-weight:900}.fp-shortlist{display:flex;align-items:center;gap:24px;border:2px solid #4a718c;border-radius:15px;padding:25px;margin-bottom:25px;background:#102b40;min-height:490px}.fp-shortlist>.fp-photo{height:300px;width:230px}.fp-shortlist>div{flex:1}.fp-shortlist h3{font-size:48px;line-height:1.1;margin-bottom:14px}.fp-shortlist>div>b{font-size:45px;color:#e1ff9f}.fp-shortlist .fp-metrics{margin:22px 0}.fp-shortlist .fp-metric{padding:16px}.fp-shortlist .fp-metric small{font-size:28px}.fp-shortlist .fp-metric b{font-size:50px}.fp-shortlist p,.fp-detail-note{font-size:32px;line-height:1.22;color:#d1e7f5}.fp-fixture-summary>h2{font-size:43px;margin:16px 0 22px;color:#a6e5ff}.fp-fixture-summary>div{display:flex;align-items:center;gap:24px;padding:14px 18px;margin-bottom:13px;background:#133047;border-radius:10px;font-size:40px;line-height:1.1}.fp-fixture-summary .fp-logo{width:62px;height:62px}.fp-fixture-summary>p{font-size:33px;line-height:1.2;margin-top:24px;color:#d6e7f2}\n.fp-slide-3{display:block}.fp-squad-layout{height:100%;display:grid;grid-template-columns:60fr 40fr;gap:26px}.fp-pitch-side{min-height:0;display:grid;grid-template-rows:115px minmax(0,1fr) 270px;gap:18px}.fp-pitch-side>.fp-section-header>b{font-size:60px;color:#d3ff9d}.fp-pitch{position:relative;min-height:0;overflow:hidden;border:3px solid #6eaa8d;border-radius:16px;background:repeating-linear-gradient(0deg,#126342 0%,#126342 14.28%,#18724d 14.28%,#18724d 28.56%)}.fp-boundary{position:absolute;inset:3% 3% 1%;border:4px solid #cee9db99;border-radius:18px}.fp-halfway{position:absolute;left:3%;right:3%;top:50%;border-top:4px solid #cee9db99}.fp-circle{position:absolute;left:35%;top:35%;width:30%;height:30%;border:4px solid #cee9db99;border-radius:50%}.fp-top-goal{position:absolute;left:44%;top:.6%;height:3%;width:12%;border:4px solid #d8ecdf;background:repeating-linear-gradient(90deg,transparent,transparent 15px,#c2dfcf88 16px)}.fp-top-box{position:absolute;top:3%;left:28%;width:44%;height:22%;border:4px solid #cee9db99;border-top:0}.fp-pitch-player{position:absolute;transform:translate(-50%,-50%);width:330px;display:flex;flex-direction:column;align-items:center;opacity:.32}.fp-pitch-player>.fp-photo{width:200px;height:205px}.fp-pitch-player>b{width:100%;font-size:39px;text-align:center;line-height:1.1;background:#061e2af5;padding:8px 4px 4px;border-radius:7px 7px 0 0;white-space:nowrap}.fp-pitch-player>span{width:100%;text-align:center;background:#061e2af5;font-size:31px;padding:4px 0 9px;border-radius:0 0 7px 7px}.fp-pitch-player em{font-size:28px;font-style:normal;color:#ffe49a}.fp-pitch-player.fp-active{border:0;box-shadow:none}.fp-pitch-player.fp-active>.fp-photo{filter:drop-shadow(0 0 13px #d5ff8588)}.fp-pitch-player.fp-active>b{color:#e3ff9d}.fp-bench{padding:18px 22px;border:2px solid #5e788c;border-radius:14px;background:#142536ee;overflow:hidden}.fp-bench h3{font-size:33px;color:#a7ddf3}.fp-bench>div{display:flex;justify-content:space-around;margin-top:8px}.fp-bench article{text-align:center;flex:1;opacity:.5}.fp-bench .fp-photo{width:110px;height:100px;display:block;margin:auto}.fp-bench b{display:block;font-size:33px;line-height:1.1}.fp-bench span{display:block;font-size:28px;color:#d1e4f2;margin-top:6px}.fp-squad-side{display:grid;grid-template-rows:115px 550px minmax(0,1fr);gap:18px;min-height:0}.fp-squad-side>.fp-section-header h2{font-size:42px}.fp-squad-side>.fp-section-header>b{font-size:34px}.fp-hero-stage{position:relative;min-height:0;border:2px solid #567d98;border-radius:15px;background:#0d263cf5;overflow:hidden}.fp-player-hero,.fp-squad-overview{position:absolute;inset:26px;visibility:hidden;overflow:hidden}.fp-player-hero>header{display:flex;align-items:center;gap:26px;height:225px}.fp-player-hero>header>.fp-photo{width:180px;height:210px}.fp-player-hero h2{font-size:51px;line-height:1.07;margin-bottom:18px}.fp-player-hero header>div>b{display:block;margin-top:15px;font-size:43px;color:#e1ff9f}.fp-player-hero>.fp-metrics{grid-template-columns:repeat(3,minmax(0,1fr));gap:14px}.fp-player-hero .fp-metric{padding:13px 16px}.fp-player-hero .fp-metric small{font-size:26px}.fp-player-hero .fp-metric b{font-size:45px;margin-top:9px}.fp-hero-fixture{font-size:35px;margin-top:20px}.fp-player-hero>p{font-size:31px;line-height:1.15;margin-top:18px}.fp-squad-overview h2{font-size:51px;color:#fff;margin:10px 0 30px}.fp-squad-overview>.fp-metrics{grid-template-columns:repeat(3,minmax(0,1fr));gap:14px}.fp-squad-overview .fp-metric{padding:17px 15px}.fp-squad-overview .fp-metric small{font-size:27px}.fp-squad-overview .fp-metric b{font-size:53px}.fp-squad-overview>p{font-size:37px;line-height:1.2;margin-top:24px;color:#d4e9f4}.fp-squad-table{padding:17px 20px;display:flex;flex-direction:column}.fp-squad-table table{width:100%;table-layout:fixed;border-collapse:collapse;flex:1}.fp-squad-table th{font-size:23px;line-height:1.1;color:#9bddfa;text-align:left;padding:8px}.fp-squad-table th:first-child{width:110px}.fp-squad-table th:nth-child(2){width:30%}.fp-squad-table th:last-child{width:110px}.fp-squad-table td{font-size:27px;font-weight:700;padding:8px 7px;line-height:1.05;border-bottom:1px solid #35536c}.fp-squad-table td:first-child{font-size:22px;color:#bcdcec}.fp-squad-table .fp-fdr{font-size:24px;padding:3px 7px;min-width:45px}.fp-squad-table footer{display:flex;flex-direction:column;gap:12px;padding:20px 4px 4px}.fp-squad-table footer>b{font-size:30px;color:#d9ff96}.fp-squad-table footer>span{font-size:26px;color:#c4ddea}\n.fp-slide-4{grid-template-rows:110px minmax(0,1fr) 145px}.fp-final-layout{display:grid;grid-template-columns:70fr 30fr;gap:26px;min-height:0}.fp-captain-three{display:grid;grid-template-columns:1fr 1.08fr 1fr;align-items:center;gap:24px;min-height:0;padding:10px}.fp-captain-card{background:#0b243aee;border:2px solid #56768e;border-radius:18px;padding:30px 26px;text-align:center;min-height:0;height:95%;display:flex;flex-direction:column;align-items:center;overflow:hidden}.fp-vortex-pick{height:100%;border:3px solid #a9df9d;background:#123c3af0}.fp-captain-card>small{font-size:35px;color:#bde9fa;font-weight:900;line-height:1.1}.fp-vortex-pick>small{color:#e0ff9d}.fp-captain-card>.fp-photo{width:290px;height:330px;margin:20px 0}.fp-captain-card h2{font-size:53px;line-height:1.05;color:#fff}.fp-captain-card>.fp-club{font-size:29px;justify-content:center;margin-top:22px}.fp-captain-fixture{font-size:35px;margin:30px 0;line-height:1.2}.fp-captain-card>.fp-metrics{grid-template-columns:1fr;width:100%;gap:14px;text-align:left}.fp-captain-card .fp-metric{display:flex;align-items:center;justify-content:space-between;gap:18px;min-height:115px;padding:18px}.fp-captain-card .fp-metric small{font-size:29px;max-width:65%}.fp-captain-card .fp-metric b{font-size:60px;margin:0}.fp-captain-card>p{font-size:32px;line-height:1.2;color:#e0edf5;margin-top:26px}.fp-final-takeaways{display:grid;grid-template-rows:470px minmax(0,1fr);gap:22px;min-height:0}.fp-vice>header{display:flex;gap:22px;align-items:center;margin-top:20px}.fp-vice .fp-photo{width:170px;height:210px}.fp-vice h3{font-size:44px}.fp-vice .fp-club{font-size:26px;margin-top:20px}.fp-vice>p{font-size:32px;margin-top:24px;line-height:1.2;color:#d3e5f2}.fp-takeaway{padding:24px 0;border-bottom:2px solid #385c73}.fp-takeaway small{font-size:28px;color:#acd9ee;display:block;line-height:1.1}.fp-takeaway b{font-size:45px;color:#fff;display:block;margin-top:16px;line-height:1.1}.fp-takeaway span{font-size:30px;color:#d0e5f2;display:block;margin-top:14px;line-height:1.2}.fp-final-takeaways>.fp-panel>.fp-metrics{margin-top:28px;grid-template-columns:1fr}\n\'\'\'\n# Typography is scoped above the broadcast shell\'s defaults.\nCSS += \'\\n\\n#content .fp-slide-3 .fp-pitch-side{grid-template-rows:102px minmax(0,1fr) 212px;gap:12px}\\n#content .fp-slide-3 .fp-pitch-player{width:min(325px,22%)}\\n#content .fp-slide-3 .fp-pitch-player>.fp-photo{width:210px;height:218px}\\n#content .fp-slide-3 .fp-pitch-player>b{font-size:clamp(27px,1.05vw,39px);overflow:hidden;white-space:nowrap;text-overflow:ellipsis}\\n#content .fp-slide-3 .fp-bench{padding:9px 15px}\\n#content .fp-slide-3 .fp-bench .fp-photo{height:80px}\\n#content .fp-slide-3 .fp-squad-side{gap:11px}\\n#content .fp-slide-3 .fp-section-header h2{font-size:43px}\\n\'\n\nCSS += \'\\n#content .fp-slide-3 .fp-squad-layout{grid-template-columns:60fr 40fr;gap:24px}\\n#content .fp-slide-3 .fp-squad-side{grid-template-rows:102px minmax(0,370px) minmax(0,1fr);gap:12px}\\n#content .fp-slide-3 .fp-hero-stage{border-radius:16px}\\n#content .fp-slide-3 .fp-squad-overview,#content .fp-slide-3 .fp-player-hero{inset:18px 22px}\\n#content .fp-slide-3 .fp-squad-overview h2{font-size:39px;margin:5px 0 17px;line-height:1.05}\\n#content .fp-slide-3 .fp-squad-overview>.fp-metrics{gap:11px}\\n#content .fp-slide-3 .fp-squad-overview .fp-metric{padding:12px 15px;border-radius:9px}\\n#content .fp-slide-3 .fp-squad-overview .fp-metric small{font-size:24px;line-height:1.05}\\n#content .fp-slide-3 .fp-squad-overview .fp-metric b{font-size:47px;line-height:1.05}\\n#content .fp-slide-3 .fp-squad-overview .fp-metric:nth-child(3){border:2px solid #d5b96e;background:#243c40}\\n#content .fp-slide-3 .fp-squad-overview .fp-metric:nth-child(3) b{color:#f5d988}\\n#content .fp-slide-3 .fp-squad-overview>p{font-size:26px;line-height:1.1;margin-top:8px}\\n#content .fp-slide-3 .fp-player-hero>header{height:135px;gap:16px}\\n#content .fp-slide-3 .fp-player-hero>header>.fp-photo{width:104px;height:130px}\\n#content .fp-slide-3 .fp-player-hero h2{font-size:36px;margin-bottom:8px}\\n#content .fp-slide-3 .fp-player-hero header>div>b{margin-top:5px;font-size:27px}\\n#content .fp-slide-3 .fp-player-hero .fp-metric{padding:8px 11px}\\n#content .fp-slide-3 .fp-player-hero .fp-metric small{font-size:20px}\\n#content .fp-slide-3 .fp-player-hero .fp-metric b{font-size:34px;margin-top:5px}\\n#content .fp-slide-3 .fp-hero-fixture{font-size:24px;margin-top:10px}\\n#content .fp-slide-3 .fp-player-hero>p{font-size:23px;margin-top:9px}\\n#content .fp-slide-3 .fp-xpts-dashboard{display:grid;grid-template-rows:62px minmax(0,1fr) 152px 51px;gap:10px;min-height:0;padding:15px 17px 12px;background:#0c2033f4;border:2px solid #43667c;border-radius:17px;overflow:hidden}\\n#content .fp-slide-3 .fp-xpts-dashboard-header{display:flex;align-items:center;justify-content:space-between;gap:16px;border-bottom:1px solid #3f6175;padding-bottom:10px}\\n#content .fp-slide-3 .fp-xpts-dashboard-header h2{font-size:31px;line-height:1.04;color:#f3f8ff;white-space:nowrap}\\n#content .fp-slide-3 .fp-xpts-dashboard-header span{font-size:21px;color:#9ddcf0;white-space:nowrap;font-weight:800}\\n#content .fp-slide-3 .fp-xpts-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));grid-template-rows:repeat(6,minmax(0,1fr));grid-auto-flow:column;gap:9px 14px;min-height:0}\\n#content .fp-slide-3 .fp-xpts-player{background:linear-gradient(110deg,#17364e,#0c2b42);border:1px solid #54778a;border-radius:11px;padding:10px 13px;min-height:0;display:flex;flex-direction:column;justify-content:center;gap:10px;overflow:hidden}\\n#content .fp-slide-3 .fp-xpts-player.fp-xpts-leader{border-color:#b59b5f;background:linear-gradient(110deg,#213a4c,#193447)}\\n#content .fp-slide-3 .fp-xpts-player-head{display:flex;align-items:center;gap:10px;min-width:0}\\n#content .fp-slide-3 .fp-xpts-role{background:#123e4d;color:#8de2ed;border:1px solid #367887;border-radius:5px;font-size:21px;font-weight:900;padding:6px 7px;min-width:58px;text-align:center}\\n#content .fp-slide-3 .fp-xpts-player-head>b{font-size:29px;line-height:1.05;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;min-width:0;flex:1;color:#f0f8ff}\\n#content .fp-slide-3 .fp-xpts-player-head em{font-style:normal;font-size:20px;color:#f3d88a;font-weight:900}\\n#content .fp-slide-3 .fp-xpts-player-head strong{font-size:35px;color:#dff79f;white-space:nowrap;line-height:1}\\n#content .fp-slide-3 .fp-xpts-player-head strong small{font-size:18px;color:#afcfde}\\n#content .fp-slide-3 .fp-xpts-track{height:9px;background:#244457;border-radius:20px;overflow:hidden}\\n#content .fp-slide-3 .fp-xpts-track>div{height:100%;background:linear-gradient(90deg,#29a1b4,#7fe2c9);border-radius:20px}\\n#content .fp-slide-3 .fp-xpts-leader .fp-xpts-track>div{background:linear-gradient(90deg,#77c6ce,#e5d98e)}\\n#content .fp-slide-3 .fp-xpts-bench{min-height:0;border-top:1px solid #45677b;padding-top:10px;display:grid;grid-template-rows:30px minmax(0,1fr);gap:8px}\\n#content .fp-slide-3 .fp-xpts-bench h3{font-size:26px;color:#acd8ea;line-height:1.05}\\n#content .fp-slide-3 .fp-xpts-bench h3 span{font-size:17px;color:#799fb5;padding-left:14px}\\n#content .fp-slide-3 .fp-xpts-bench>div{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:9px}\\n#content .fp-slide-3 .fp-xpts-sub{display:flex;flex-direction:column;gap:6px;min-width:0;overflow:hidden;padding:10px 9px;background:#142a3c;border:1px solid #3e5a6e;border-radius:8px}\\n#content .fp-slide-3 .fp-xpts-sub>small{font-size:17px;line-height:1.08;color:#8db4c7;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}\\n#content .fp-slide-3 .fp-xpts-sub>b{font-size:23px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;color:#d5e5ef}\\n#content .fp-slide-3 .fp-xpts-sub>strong{font-size:26px;color:#d1eeb7;margin-top:auto;white-space:nowrap}\\n#content .fp-slide-3 .fp-xpts-sub>strong small{font-size:17px;color:#aecbd8}\\n#content .fp-slide-3 .fp-xpts-total{display:flex;justify-content:space-between;align-items:center;gap:10px;border-top:1px solid #426376;padding:9px 1px 0;min-width:0}\\n#content .fp-slide-3 .fp-xpts-total>b{font-size:27px;line-height:1.05;color:#e1f7b1;white-space:nowrap}\\n#content .fp-slide-3 .fp-xpts-total>b span{color:#9dc9db;padding:0 7px}\\n#content .fp-slide-3 .fp-xpts-total>small{font-size:17px;white-space:nowrap;color:#a6bfd0;font-weight:700}\\n#content .fp-slide-3 .fp-xpts-player.fp-active{border-color:#cbdba7;box-shadow:inset 0 0 0 2px #88ddbbaa}\\n\'\n\nCSS += \'\\n#content .fp-slide-3{height:calc(100% - 90px)}\\n#content .fp-slide-3 .fp-bench article{opacity:1}\\n#content .fp-slide-3 .fp-bench b{font-size:27px;line-height:1.06;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}\\n#content .fp-slide-3 .fp-bench span{font-size:23px;margin-top:4px;line-height:1.1}\\n#content .fp-slide-3 .fp-bench .fp-photo{height:76px}\\n#content .fp-slide-3 .fp-xpts-total>b{font-size:24px}\\n#content .fp-slide-3 .fp-xpts-total>small{font-size:15px}\\n#content .fp-slide-3 .fp-xpts-sub{padding:8px 9px}\\n\'\n\nCSS += \'\\n#content .fp-slide-3 .fp-pitch-player>b{font-size:29px;padding:6px 2px 4px}\\n#content .fp-slide-3 .fp-pitch-player>span{font-size:24px;padding:2px 0 6px}\\n\'\n\nCSS=re.sub(r\'(^|})([^{}]+){\',lambda m:m[1]+\',\'.join(s.strip() if s.strip().startswith(\'#content\') else \'#content \'+s.strip() for s in m[2].split(\',\'))+\'{\',CSS)\n\n# Free Hit Slide 04 only: title plaque, clear footer, and readable captaincy cards.\nCSS += \'\'\'\n#content:has(.fp-slide-4){height:1360px!important}\n#slide:has(.fp-slide-4) .title-group{isolation:isolate}\n#slide:has(.fp-slide-4) .title-group::before{content:"";position:absolute;top:4px;left:50%;transform:translateX(-50%);width:1050px;height:137px;z-index:0;background:#071523;border:2px solid #bda064b5;border-radius:17px;box-shadow:0 8px 25px #000b,inset 0 0 18px #203b5066;pointer-events:none}\n#slide:has(.fp-slide-4) .title-group .heading{position:relative;z-index:1}\n#content .fp-slide-4{grid-template-rows:108px minmax(0,1fr) 122px;gap:20px}\n#content .fp-slide-4>.fp-section-header{padding:14px 24px;background:#081b2ef7}\n#content .fp-slide-4>.fp-section-header h2{font-size:45px}\n#content .fp-slide-4>.fp-section-header p{font-size:31px}\n#content .fp-slide-4 .fp-final-layout{gap:22px;grid-template-columns:minmax(0,70fr) minmax(0,30fr)}\n#content .fp-slide-4 .fp-captain-three{align-items:stretch;gap:20px;padding:0 4px}\n#content .fp-slide-4 .fp-captain-card{height:100%;padding:22px 19px 19px;gap:10px;justify-content:space-around;background:#092138f7;border-color:#577c96}\n#content .fp-slide-4 .fp-vortex-pick{background:#123e38fa;border-color:#d1db92;box-shadow:inset 0 0 21px #75d4ab15,0 0 19px #9fcf8244}\n#content .fp-slide-4 .fp-captain-card>small{font-size:33px;line-height:1.1}\n#content .fp-slide-4 .fp-captain-card>.fp-photo{width:260px;height:275px;margin:5px auto 2px}\n#content .fp-slide-4 .fp-captain-card h2{font-size:54px;line-height:1.08;overflow-wrap:anywhere}\n#content .fp-slide-4 .fp-captain-card>.fp-club{font-size:32px;margin-top:2px;line-height:1.1}\n#content .fp-slide-4 .fp-captain-fixture{font-size:35px;line-height:1.15;margin:10px 0}\n#content .fp-slide-4 .fp-captain-card>.fp-metrics{gap:12px}\n#content .fp-slide-4 .fp-captain-card .fp-metric{min-height:105px;padding:14px 16px;gap:12px;background:#15364cf5}\n#content .fp-slide-4 .fp-captain-card .fp-metric small{font-size:32px;max-width:70%;line-height:1.15}\n#content .fp-slide-4 .fp-captain-card .fp-metric b{font-size:59px}\n#content .fp-slide-4 .fp-captain-card>p{font-size:33px;line-height:1.19;margin:10px 0 2px;color:#ecf5fb}\n#content .fp-slide-4 .fp-final-takeaways{grid-template-rows:440px minmax(0,1fr);gap:18px}\n#content .fp-slide-4 .fp-final-takeaways>.fp-panel{background:#081b30f8;padding:24px 25px}\n#content .fp-slide-4 .fp-final-takeaways>.fp-panel h2{font-size:43px;margin-bottom:19px}\n#content .fp-slide-4 .fp-vice>header{gap:20px;margin-top:10px}\n#content .fp-slide-4 .fp-vice .fp-photo{width:176px;height:185px}\n#content .fp-slide-4 .fp-vice h3{font-size:43px;line-height:1.1}\n#content .fp-slide-4 .fp-vice .fp-club{font-size:29px;margin-top:12px}\n#content .fp-slide-4 .fp-vice-projection{display:flex;justify-content:space-between;align-items:center;background:#12334b;border:1px solid #466e82;border-radius:9px;padding:11px 15px;margin-top:15px;gap:10px}\n#content .fp-slide-4 .fp-vice-projection strong{font-size:40px;color:#dff5a4;white-space:nowrap}\n#content .fp-slide-4 .fp-vice-projection span{font-size:27px;color:#cde5f0;white-space:nowrap}\n#content .fp-slide-4 .fp-vice>p{font-size:29px;margin-top:14px;line-height:1.16}\n#content .fp-slide-4 .fp-takeaway{padding:20px 0}\n#content .fp-slide-4 .fp-takeaway small{font-size:31px;color:#aee3f5;line-height:1.18}\n#content .fp-slide-4 .fp-takeaway b{font-size:45px;margin-top:13px;line-height:1.12}\n#content .fp-slide-4 .fp-takeaway span{font-size:31px;margin-top:10px;line-height:1.17}\n#content .fp-slide-4 .fp-final-takeaways>.fp-panel:last-child{display:flex;flex-direction:column;justify-content:space-evenly}\n#content .fp-slide-4 .fp-final-takeaways>.fp-panel>.fp-metrics{margin-top:20px}\n#content .fp-slide-4 .fp-final-takeaways .fp-metric{padding:17px 20px;border-color:#658e7c}\n#content .fp-slide-4 .fp-final-takeaways .fp-metric small{font-size:32px}\n#content .fp-slide-4 .fp-final-takeaways .fp-metric b{font-size:72px;margin-top:8px;color:#dff7a2}\n#content .fp-slide-4>.fp-decision{padding:16px 26px;gap:25px;background:#0d3834fb}\n#content .fp-slide-4>.fp-decision b{font-size:43px;white-space:nowrap}\n#content .fp-slide-4>.fp-decision p{font-size:34px;line-height:1.16;max-width:68%;text-align:right}\n\'\'\'\n\nSEEK_JS=\'\'\'<script>(function(){const prev=window.seekWildcard;window.seekWildcard=function(t){if(prev)prev(t);const root=document.querySelector(\'.fp-slide\');if(!root)return;const d=JSON.parse(root.querySelector(\'.fp-data\').textContent),tl=d.timeline,s=tl.find(x=>t>=x.start&&t<x.end)||(t<tl[0].start?tl[0]:tl.at(-1));const ease=x=>{x=Math.max(0,Math.min(1,x));return 1-Math.pow(1-x,3)};const phase=ease((t-s.start)/.4);root.dataset.phase=s.type;for(const el of root.querySelectorAll(\'[data-fp-focus]\'))el.classList.toggle(\'fp-active\',el.dataset.fpFocus===s.type);if(d.kind===2){for(const el of root.querySelectorAll(\'[data-fp-team]\'))el.classList.toggle(\'fp-active\',+el.dataset.fpTeam===s.team_id);const detail=[\'team\',\'player\',\'comparison\'].includes(s.type);for(const el of root.querySelectorAll(\'[data-fp-detail]\')){const show=detail&&+el.dataset.fpDetail===s.team_id;el.style.visibility=show?\'visible\':\'hidden\';el.style.opacity=String(show?.7+.3*phase:0);}root.querySelector(\'.fp-fixture-summary\').style.visibility=detail?\'hidden\':\'visible\';for(const el of root.querySelectorAll(\'[data-fp-player]\'))el.classList.toggle(\'fp-active\',+el.dataset.fpPlayer===s.player_id);}if(d.kind===3){const revealed=new Set(tl.filter(x=>x.start<=t).flatMap(x=>x.player_ids||[]));for(const el of root.querySelectorAll(\'[data-fp-pitch],[data-fp-bench]\')){const id=+(el.dataset.fpPitch||el.dataset.fpBench),cue=tl.find(x=>x.hero_player_id===id),v=cue?ease((t-cue.start)/.4):0;el.style.opacity=String(revealed.has(id)?.72+.28*v:el.dataset.fpBench?.72:.72);el.classList.toggle(\'fp-active\',id===s.hero_player_id);}for(const el of root.querySelectorAll(\'[data-fp-hero]\')){const show=+el.dataset.fpHero===s.hero_player_id;el.style.visibility=show?\'visible\':\'hidden\';el.style.opacity=String(phase);el.style.translate=`${(1-phase)*(s.direction===\'left\'?-25:25)}px 0`;}root.querySelector(\'.fp-squad-overview\').style.visibility=s.hero_player_id?\'hidden\':\'visible\';for(const el of root.querySelectorAll(\'[data-fp-squad-row]\')){el.style.opacity=revealed.has(+el.dataset.fpSquadRow)||s.type===\'summary\'?\'1\':\'.45\';el.classList.toggle(\'fp-active\',+el.dataset.fpSquadRow===s.hero_player_id);}}if(d.kind===4){for(const el of root.querySelectorAll(\'[data-fp-contender]\')){const active=+el.dataset.fpContender===s.player_id;el.classList.toggle(\'fp-active\',active);el.style.opacity=active||[\'takeaways\',\'outro\',\'vice\'].includes(s.type)?\'1\':\'.8\';el.style.scale=active?String(1+.006*phase):\'1\';}}};})();</script>\'\'\'\n\nclass FreeHitPremiumVisual:\n def __init__(self,scene,contract,width,height,renderer,metadata,media,badges,approved_cards=None):\n  self.manifest,self.contract,self.width,self.height=scene,contract,width,height;self.renderer,self.metadata=renderer,metadata;self.layout=[]\n  kind={\'fh_blueprint\':1,\'fixture_heatmap\':2,\'fh_squad_plan\':3,\'fh_captain_final\':4}[scene[\'visual_type\']];d=copy.deepcopy(contract)\n  self.timeline=scene[\'narration_timeline\']\n  if self.timeline[0].get(\'start\') is None:self.timeline=align_timeline(self.timeline,float(scene.get(\'audio_duration_seconds\') or len(scene[\'narration_text\'].split())/145*60))\n  d[\'captain_pick\']=captain_plan(d);self.content_html=render_slide(kind,d,self.timeline,media,badges,approved_cards);self.complete=self.draw(self.timeline[-1][\'end\']+2.5,0)\n def draw(self,seconds,duration):return self.renderer.animation_image(self.manifest,self.metadata,self.width,self.height,self.content_html,seconds)\n def frame(self,seconds,duration,previous=None):\n  import numpy as np\n  return np.asarray(self.draw(seconds,duration))\n\nCSS+="""#content .fp-intro-grid>.fp-panel{display:flex;flex-direction:column}#content .fp-intro-minis{margin-top:auto;padding-top:24px;display:flex;justify-content:space-around;gap:12px}#content .fp-intro-minis article{flex:1;text-align:center}#content .fp-intro-minis .fp-photo{width:145px;height:170px;display:block;margin:auto}#content .fp-intro-minis b{font-size:28px;line-height:1.1;display:block;margin-top:10px}#content .fp-intro-grid>.fp-panel:nth-child(2) .fp-intro-minis .fp-photo{width:105px;height:140px}#content .fp-intro-grid>.fp-panel:nth-child(2) .fp-intro-minis b{font-size:24px}"""\n\nCSS+="""#content .fp-detail-note>b{display:block;color:#dfff9c;font-size:34px;line-height:1.15}#content .fp-detail-note>span{display:block;margin-top:12px;font-size:29px;color:#fff}#content .fp-detail-note>p{margin-top:18px;font-size:30px}"""\n\n# VX_FH_SLIDE3_PREMIUM_V1\nCSS += \'\\n/* VX_FH_SLIDE3_PREMIUM_V1: isolated heading treatment, original masthead/footer preserved. */\\n#slide:has(.fp-slide-3) .title-group{isolation:isolate}\\n#slide:has(.fp-slide-3) .title-group::after{display:none!important}\\n#slide:has(.fp-slide-3) .title-group::before{\\n content:"";position:absolute;left:50%;top:7px;translate:-50% 0;width:1040px;height:136px;\\n background:linear-gradient(112deg,#071523 0%,#0c2738 55%,#071523 100%);\\n border:1px solid #927c4b;border-radius:16px;z-index:0;pointer-events:none;\\n box-shadow:0 12px 30px #0009,inset 0 1px 0 #d6ba7780;\\n}\\n#slide:has(.fp-slide-3) .title-group .heading{position:relative;z-index:1}\\n#content .fp-slide-3{position:relative;isolation:isolate}\\n#content .fp-slide-3 .fp-card-flight{\\n position:absolute;top:0;left:0;z-index:80;pointer-events:none;visibility:hidden;opacity:0;\\n display:flex;flex-direction:column;align-items:center;justify-content:center;\\n width:296px;min-height:328px;padding:13px;border-radius:17px;\\n border:2px solid #b8dfc9;background:linear-gradient(160deg,#123c4a,#081727 88%);\\n box-shadow:0 20px 46px #000c,0 0 28px #65e9cd75,inset 0 0 20px #8df5da20;\\n transform-origin:center center;will-change:transform,opacity;\\n}\\n#content .fp-slide-3 .fp-card-flight .fp-photo{width:190px;height:202px;object-fit:contain;filter:drop-shadow(0 8px 10px #0009)}\\n#content .fp-slide-3 .fp-card-flight b{width:100%;margin-top:8px;overflow:hidden;text-align:center;\\n font-size:31px;white-space:nowrap;text-overflow:ellipsis;color:#f7fffe}\\n#content .fp-slide-3 .fp-card-flight small{font-size:23px;font-weight:800;color:#d7ff9c;margin-top:6px}\\n#content .fp-slide-3 .fp-hero-stage{isolation:isolate}\\n#content .fp-slide-3 .fp-player-hero.fp-flying{filter:drop-shadow(0 0 10px #96ffdf40)}\\n#content .fp-slide-3 .fp-pitch-player.fp-landed>.fp-photo{filter:drop-shadow(0 0 16px #deffb966)}\\n#content .fp-slide-3 .fp-xpts-player.fp-active{box-shadow:inset 0 0 0 2px #b5f4ceaa,0 0 12px #66e6d858}\\n\'\n\n# Scoped Free Hit 3/4 premium presentation; underlying approved player-card art stays untouched.\nCSS += """\n#content .fp-slide-3 .fp-player-hero{display:flex;flex-direction:column;align-items:center;justify-content:center;inset:12px;padding:0}\n#content .fp-slide-3 .fp-approved-card{display:block;max-width:98%;height:calc(100% - 82px);object-fit:contain;filter:drop-shadow(0 6px 18px #0007)}\n#content .fp-slide-3 .fp-card-meta{width:98%;display:flex;flex-direction:column;align-items:center;gap:5px;text-align:center;padding:9px;border-top:2px solid #c4a569;background:#0b2837}\n#content .fp-slide-3 .fp-card-meta b{font-size:32px;color:white}\n#content .fp-slide-3 .fp-card-meta span{font-size:25px;color:#9de5ec}\n#content .fp-slide-4 .fp-final-takeaways{position:relative}\n#content .fp-slide-4 .fp-approved-hero-stage{position:absolute;inset:0;z-index:5;pointer-events:none;background:linear-gradient(150deg,#0b2137,#082437);border:2px solid #e0c789;border-radius:19px;overflow:hidden}\n#content .fp-slide-4 .fp-approved-captain{position:absolute;inset:0;margin:0;padding:18px;display:flex;align-items:center;justify-content:center;visibility:hidden;opacity:0}\n#content .fp-slide-4 .fp-approved-captain .fp-approved-card{width:auto;height:100%;max-width:100%;object-fit:contain;filter:drop-shadow(0 9px 24px #0009)}\n#content .fp-slide-4 .fp-captain-card{visibility:hidden;opacity:0;transition:none}\n#content .fp-slide-4 .fp-captain-card .fp-photo{width:208px;height:228px;margin:12px 0}\n#content .fp-slide-4 .fp-captain-card h2{font-size:40px}\n#content .fp-slide-4 .fp-captain-card .fp-metric{padding:8px;min-height:82px}\n#content .fp-slide-4 .fp-captain-card .fp-metric small{font-size:22px}\n#content .fp-slide-4 .fp-captain-card .fp-metric b{font-size:42px}\n#content .fp-slide-4 .fp-captain-card>p{font-size:24px;margin-top:10px}\n"""\n\n\n# VX_FH_FULL_HERO_FLIGHT_V2: only affects Free Hit Slide 3.\nCSS += """\n#content .fp-slide-3 .fp-card-flight {width:296px;height:360px;min-height:0;padding:7px;overflow:hidden}\n#content .fp-slide-3 .fp-card-flight .fp-approved-card {display:block;width:100%;height:276px;max-width:100%;object-fit:contain;filter:drop-shadow(0 8px 16px #000c)}\n#content .fp-slide-3 .fp-card-flight>b {margin-top:3px;font-size:23px;line-height:1.05}\n#content .fp-slide-3 .fp-card-flight>small {font-size:19px;margin-top:3px}\n#content .fp-slide-3 .fp-player-hero {position:absolute;overflow:hidden;gap:5px;justify-content:flex-start}\n#content .fp-slide-3 .fp-player-hero>.fp-approved-card {height:calc(100% - 206px);max-height:calc(100% - 206px);min-height:0;flex:1 1 auto}\n#content .fp-slide-3 .fp-card-meta {flex:0 0 auto;min-height:59px}\n#content .fp-slide-3 .fp-why-popup {flex:0 0 129px;width:98%;max-width:100%;padding:12px 17px 10px;background:#0a2736f7;border:2px solid #bfe8a8;border-radius:12px;box-shadow:0 12px 35px #000a;opacity:0;transform:translateX(36px);visibility:hidden}\n#content .fp-slide-3 .fp-why-popup strong {display:block;font-size:29px;line-height:1;color:#deffa2}\n#content .fp-slide-3 .fp-why-popup p {margin:8px 0 5px!important;font-size:25px;line-height:1.12;color:white;text-wrap:balance}\n#content .fp-slide-3 .fp-why-popup small {display:block;font-size:19px;color:#b5ecfa}\n"""\n\n# Stage height is large enough to show the original artwork and a legible explanation.\nCSS += """\n#content .fp-slide-3 .fp-squad-side{grid-template-rows:102px minmax(0,520px) minmax(0,1fr)}\n#content .fp-slide-3 .fp-player-hero>.fp-approved-card{height:calc(100% - 205px);max-height:calc(100% - 205px)}\n"""\n'
# Free Hit slides 3–4: original Hero Cards, reference pitch and measured sequential reveals.
import importlib as _fh_suite_importlib
from pathlib import Path as _FhSuitePath
import hashlib as _fh_suite_hashlib, json as _fh_suite_json, re as _fh_suite_re
_fh_suite_engine = VX_MEDIA_WORK/'wildcard_engine'
_fh_suite_engine.mkdir(parents=True,exist_ok=True)
for _fh_suite_name,_fh_suite_source in [('freehit_suite.py',_VX_FREEHIT_SUITE_SOURCE),('freehit_premium.py',_VX_FREEHIT_PREMIUM_SUITE_SOURCE)]:
    _fh_suite_target=_fh_suite_engine/_fh_suite_name
    _fh_suite_target.write_text(_fh_suite_source,encoding='utf-8')
    _FhSuitePath(_fh_suite_importlib.util.cache_from_source(str(_fh_suite_target))).unlink(missing_ok=True)
_fh_suite_importlib.invalidate_caches()
import freehit_suite as VX_FREEHIT_SUITE
import freehit_premium as VX_FREEHIT_PREMIUM_SUITE
VX_FREEHIT_SUITE=_fh_suite_importlib.reload(VX_FREEHIT_SUITE)
VX_FREEHIT_PREMIUM_SUITE=_fh_suite_importlib.reload(VX_FREEHIT_PREMIUM_SUITE)
from tts_narration import CLUBS as _fh_club_names
_fh_club_names.update({t['short_name']:t['name'] for t in _bootstrap['teams'] if t['short_name'] not in _fh_club_names})
VX_FREEHIT_SUITE.ensure_metadata(VX_FREEHIT_MATCHDAY_PROJECT)

def _vx_freehit_suite_scene(scene,contract):
    if contract.get('chip')!='freehit' or scene['visual_type'] not in ('fh_squad_plan','fh_captain_final'):return scene
    if scene['visual_type']=='fh_squad_plan':timeline=VX_FREEHIT_PREMIUM_SUITE.squad_segments(contract['slide3_plan'])
    else:timeline=VX_FREEHIT_PREMIUM_SUITE.final_segments(VX_FREEHIT_PREMIUM_SUITE.captain_plan(contract))
    scene.update(narration_timeline=timeline,narration_text=' '.join(cue['narration_text'] for cue in timeline),animation_version=VX_FREEHIT_SUITE.VERSION)
    scene['timing'].update(status='awaiting measured Ryan phrases',player_alignment='awaiting_tts',audio_padding_seconds=0)
    return scene

if not getattr(vx_build_manifest,'_freehit_suite',False):
    _vx_suite_previous_manifest=vx_build_manifest
    def vx_build_manifest(contract,_build=_vx_suite_previous_manifest):
        scenes=_build(contract)
        for scene in scenes:_vx_freehit_suite_scene(scene,contract)
        return scenes
    vx_build_manifest.__dict__.update(_vx_suite_previous_manifest.__dict__)
    vx_build_manifest._freehit_suite=True

if VX_GENERATE_FREEHIT:
    for _fh_suite_key,_fh_suite_contract in VX_DATA_CONTRACTS.items():
        if _fh_suite_contract.get('chip')!='freehit':continue
        _fh_suite_scenes=VX_SCENE_MANIFESTS[_fh_suite_key]
        _fh_suite_script=VX_NARRATION_PATHS[_fh_suite_key]
        _fh_suite_text=_fh_suite_script.read_text(encoding='utf-8')
        for _fh_suite_scene in _fh_suite_scenes:
            if _fh_suite_scene['visual_type'] not in ('fh_squad_plan','fh_captain_final'):continue
            _vx_freehit_suite_scene(_fh_suite_scene,_fh_suite_contract)
            for _fh_suite_segment in VX_STORY_GENERATED[_fh_suite_key]:
                if _fh_suite_segment['id']==_fh_suite_scene['scene_id']:_fh_suite_segment['text']=_fh_suite_scene['narration_text']
            _fh_suite_pattern=r'(\[\[SEGMENT '+_fh_suite_re.escape(_fh_suite_scene['scene_id'])+r' \|[^\n]*\]\]\s*\n)(.*?)(?=\n\[\[SEGMENT |\Z)'
            _fh_suite_text,_fh_suite_count=_fh_suite_re.subn(_fh_suite_pattern,lambda m:m.group(1)+_fh_suite_scene['narration_text']+'\n',_fh_suite_text,flags=_fh_suite_re.S)
            if _fh_suite_count!=1:raise ValueError('Expected exactly one Free Hit narration segment: '+_fh_suite_scene['scene_id'])
            (VX_NARRATION_OUT_DIR/(_fh_suite_key+'_'+_fh_suite_scene['scene_id']+'.txt')).write_text(_fh_suite_scene['narration_text']+'\n',encoding='utf-8')
        _fh_suite_script.write_text(_fh_suite_text,encoding='utf-8')
        _fh_suite_stamp=VX_ANALYSIS_ROOT/'narration_provenance'/(_fh_suite_script.name+'.provenance.json')
        _fh_suite_stamp.parent.mkdir(parents=True,exist_ok=True)
        _fh_suite_stamp.write_text(_fh_suite_json.dumps(dict(data_hash=_fh_suite_contract['data_hash'],generated_sha256=_fh_suite_hashlib.sha256(_fh_suite_text.encode()).hexdigest()),indent=2))
        VX_NARRATION[_fh_suite_key]=decode_script(_fh_suite_text)
        (VX_ANALYSIS_ROOT/(_fh_suite_key+'_scene_manifest.json')).write_text(_fh_suite_json.dumps(_fh_suite_scenes,ensure_ascii=False,indent=2),encoding='utf-8')
    print('Free Hit Slides 3–4: requested wording; live squad/captaincy data, player reveal cues and one captain bonus synchronized.')

def vx_editorial_narration(text: str) -> str:
    """Use the creator's voice while retaining estimates and every numerical fact."""
    import re
    if text.lstrip().startswith(("This is my Free Hit for Gameweek ", "Let's settle the armband for Gameweek ")):
        return text
    substitutions = (
        (r'\bHe projects\b', 'My calculations put him at'),
        (r'\bEvery number comes from our chosen Wildcard eleven\b', 'Every calculation is tied to my chosen Wildcard eleven'),
        (r'\bThe separate transfer model suggests rolling\b', 'My transfer calculations favour rolling'),
        (r'\bThe model gives him\b', 'My calculations give him'),
        (r'\bThe model expects\b', 'I expect'),
        (r'\bThe model estimates\b', 'My calculations estimate'),
        (r'\bThe model projects\b', 'My calculations project'),
        (r'\bThe model brings\b', 'My plan brings'),
        (r'\bThe model lineup\b', 'My planned lineup'),
        (r'\bmodel role indicators\b', 'role and minutes indicators'),
        (r'\bprevious model week\b', 'previous planned week'),
        (r'\bweekly model adjusts\b', 'weekly plan adjusts'),
        (r'\bmodel clean-sheet chance\b', 'estimated clean-sheet chance'),
        (r'\bmodel fixture difficulty\b', 'calculated fixture difficulty'),
        (r'\bmodel projection\b', 'points estimate'),
        (r'\bmodel points\b', 'expected points'),
        (r'\bmodel scenario\b', 'transfer scenario'),
        (r'\bmodel-backed\b', 'calculation-led'),
        (r'\bmodelled\b', 'estimated'),
        (r"\bthe model['’]s\b", 'my'),
        (r'\bthe notebook\b', 'my analysis'),
        (r'\bnotebook\b', 'analysis'),
        (r'\bmodels\b', 'calculations'),
        (r'\bmodel\b', 'calculation'),
    )
    for pattern, replacement in substitutions:
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text


def vx_editorial_spoken_fields(value):
    if isinstance(value, dict):
        if value.get('visual_type') in ('fh_squad_plan', 'fh_captain_final') or value.get('visual') in ('fh_squad_plan', 'fh_captain_final'):
            return value
        for field, item in value.items():
            if field in ('narration', 'narration_text', 'text') and isinstance(item, str):
                value[field] = vx_editorial_narration(item)
            elif isinstance(item, (dict, list)):
                vx_editorial_spoken_fields(item)
    elif isinstance(value, list):
        for item in value:
            vx_editorial_spoken_fields(item)
    return value


# Rebuilt scenes and retained TXT edits must use the same spoken text and measured cues.
if not getattr(vx_build_manifest, '_creator_voice', False):
    _vx_manifest_before_creator_voice = vx_build_manifest
    def vx_build_manifest(contract, _build=_vx_manifest_before_creator_voice):
        return vx_editorial_spoken_fields(_build(contract))
    vx_build_manifest._creator_voice = True

for _vx_voice_key, _vx_voice_scenes in VX_SCENE_MANIFESTS.items():
    # Slides 2 and 3 rewrite saved scripts, so refresh the in-memory narration before validation.
    VX_NARRATION[_vx_voice_key] = decode_script(
        Path(VX_NARRATION_PATHS[_vx_voice_key]).read_text(encoding='utf-8')
    )
    vx_editorial_spoken_fields(_vx_voice_scenes)
    vx_editorial_spoken_fields(VX_STORY_GENERATED.get(_vx_voice_key, []))
    vx_editorial_spoken_fields(VX_NARRATION[_vx_voice_key])
    if _vx_voice_key in VX_NARRATION_OBJECTS:
        _vx_voice_object = vx_editorial_spoken_fields(VX_NARRATION_OBJECTS[_vx_voice_key])
        vx_editorial_spoken_fields(VX_WILDCARD_SCENE_JSONS[_vx_voice_key])
        for _vx_voice_section in _vx_voice_object['sections']:
            _vx_voice_section['estimated_duration_seconds'] = round(
                len(_vx_voice_section['narration'].split())/145*60
                + _vx_voice_section.get('pause_before_seconds', 0)
                + _vx_voice_section.get('pause_after_seconds', 0), 2)
        _vx_voice_object['total_estimated_duration_seconds'] = sum(
            section['estimated_duration_seconds'] for section in _vx_voice_object['sections'])
        VXWC.validate_render_script(_vx_voice_object, VX_NARRATION[_vx_voice_key])
        (VX_ANALYSIS_ROOT/(_vx_voice_key+'_narration.json')).write_text(
            json.dumps(_vx_voice_object, ensure_ascii=False, indent=2), encoding='utf-8')
        (VX_ANALYSIS_ROOT/(_vx_voice_key+'_final_scene.json')).write_text(
            json.dumps(VX_WILDCARD_SCENE_JSONS[_vx_voice_key], ensure_ascii=False, indent=2), encoding='utf-8')
    _vx_voice_path = Path(VX_NARRATION_PATHS[_vx_voice_key])
    _vx_voice_header = '\n'.join(line for line in _vx_voice_path.read_text(encoding='utf-8').splitlines() if line.startswith('#'))
    _vx_voice_text = (_vx_voice_header+'\n' if _vx_voice_header else '')+encode_script(VX_NARRATION[_vx_voice_key])
    _vx_voice_path.write_text(_vx_voice_text, encoding='utf-8')
    _vx_voice_stamp = VX_ANALYSIS_ROOT/'narration_provenance'/(_vx_voice_path.name+'.provenance.json')
    _vx_voice_stamp.parent.mkdir(parents=True, exist_ok=True)
    _vx_voice_stamp.write_text(json.dumps(dict(data_hash=VX_DATA_CONTRACTS[_vx_voice_key]['data_hash'],
        generated_sha256=hashlib.sha256(_vx_voice_text.encode()).hexdigest()), indent=2), encoding='utf-8')
    for _vx_voice_segment in VX_NARRATION[_vx_voice_key]:
        (VX_NARRATION_OUT_DIR/(_vx_voice_key+'_'+_vx_voice_segment['id']+'.txt')).write_text(
            _vx_voice_segment['text'].strip()+'\n', encoding='utf-8')
    (VX_ANALYSIS_ROOT/(_vx_voice_key+'_scene_manifest.json')).write_text(
        json.dumps(_vx_voice_scenes, ensure_ascii=False, indent=2), encoding='utf-8')
print('Narration synchronized: my calculations, my captaincy decisions, unchanged figures and uncertainty.')


In [ ]:
#@title 🔴 CELL 14 - RYAN AUDIO & VIDEO RENDER {display-mode: "form"}

def vx_require_scene_contracts(cell_name):
    contracts = globals().get('VX_DATA_CONTRACTS')
    scenes = globals().get('VX_SCENE_MANIFESTS')
    narration = globals().get('VX_NARRATION_PATHS')
    if (not globals().get('VX_CELL2_COMPLETE', False)
            or not isinstance(contracts, dict) or not contracts
            or not isinstance(scenes, dict) or set(contracts) != set(scenes)
            or not isinstance(narration, dict) or set(contracts) != set(narration)):
        raise RuntimeError(
            f'Cell 14 requires a successful Cell 12 scene-and-narration run. '
            'Finish Cells 11 and 12 in order before rendering or exporting.'
        )
    return True

vx_require_scene_contracts("Cell 14")

# 🔴 CELL 14 - RYAN AUDIO & VIDEO RENDER
import importlib.util, subprocess, sys, asyncio, hashlib, json, math, shutil, textwrap, os, re, time
from pathlib import Path
for module, requirement in [('PIL','pillow'),('edge_tts','edge-tts>=7.2'),('imageio_ffmpeg','imageio-ffmpeg'),('nest_asyncio','nest_asyncio')]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',requirement],check=True)
import edge_tts

# Microsoft speech endpoint compatibility for edge-tts 7.2.8.
# Upstream fix: https://github.com/rany2/edge-tts/pull/491
# Patch both references because Communicate imports WSS_URL by value.
from edge_tts import constants as _vx_edge_constants
from edge_tts import communicate as _vx_edge_communicate
if _vx_edge_communicate.WSS_URL.startswith('wss://speech.platform.bing.com/'):
    _vx_edge_wss_url = (
        'wss://api.msedgeservices.com/tts/cognitiveservices/websocket/v1'
        '?Ocp-Apim-Subscription-Key=' + _vx_edge_constants.TRUSTED_CLIENT_TOKEN
    )
    _vx_edge_constants.WSS_URL = _vx_edge_wss_url
    _vx_edge_communicate.WSS_URL = _vx_edge_wss_url
    print('✅ Ryan speech: Microsoft synthesis endpoint compatibility applied.', flush=True)
import numpy as np
import pandas as pd
from PIL import Image,ImageDraw,ImageFont,ImageOps

_vx_required = ('VXWV','VXWC','VX_WC_ENGINE','VX_MEDIA_WORK','VX_ANALYSIS_ROOT','VX_DATA_CONTRACTS','VX_SCENE_MANIFESTS','VX_NARRATION_PATHS','VX_VIDEO_PROFILE','VX_4K_PNG_OUT_DIR','VX_SAVE_4K_PNG','_bootstrap')
_vx_missing = [name for name in _vx_required if name not in globals()]
if _vx_missing:
    raise RuntimeError('Render setup is incomplete: '+', '.join(_vx_missing)+'. Run Cells 0–13 in order, then Cell 14.')

# Freeze freshness at Cell 11 start so a valid source cannot expire while PNG/TTS/video work is running.
VX_RENDER_BATCH_STARTED_AT = VXWC.utc()
_VX_DRAFT_STALE_WARNED = set()

# Apply the shared font correction to an already-loaded Colab renderer.
# This keeps rerunning Cell 11 sufficient after an in-place notebook update.
# Embedded runtime sources make rerunning this cell recoverable after cache cleanup.
_VX_RENDER_RUNTIME_BUNDLE = '...'

# VX_CELL14_RENDER_FIX_V1
# Keep the Free Hit intro compatible with the cue-based Wildcard parent.
def _vx_fix_freehit_intro_source(content):
    old = '    def scene_image(self, index, u, global_t):\n'
    new = ('    def scene_image(self, cue, u, global_t):\n'
           '        index = self.timeline.index(cue) if isinstance(cue, dict) else int(cue)\n')
    if old in content:
        content = content.replace(old, new, 1)
    if '# VX_FREEHIT_DRAW_ADAPTERS_V1' not in content:
        content += '''

    # VX_FREEHIT_DRAW_ADAPTERS_V1
    # Free Hit owns these helpers; it no longer relies on removed parent methods.
    def _type(self, image, text, x, y, size, color, max_width=None, bold=True, progress=1.0):
        self.label(image, text, x, y, size, color, reveal=progress,
                   bold=bold, max_width=max_width)

    def _metric(self, image, x, y, label, value, accent):
        self._stat(image, x, y, label, value, accent, 1.0)

    def _base(self, accent, title, index, count):
        image = self.background.convert('RGBA')
        overlay = Image.new('RGBA', image.size, (0, 0, 0, 0))
        self.rounded(overlay, (82, 196, 1835, 967), (4, 16, 29, 237), (53, 103, 122, 255), 27, 2)
        self.rounded(overlay, (107, 216, 917, 941), (8, 33, 47, 249), (44, 93, 108, 255), 21, 2)
        image = Image.alpha_composite(image, overlay)
        self.label(image, title, 146, 253, 28, accent)
        self.rounded(image, (144, 910, 869, 920), '#315466', radius=4)
        progress = min(1.0, max(0.0, (index + 1) / max(1, count)))
        self.rounded(image, (144, 910, 144 + max(5, int(725 * progress)), 920), accent, radius=4)
        return image

    def _avatar(self, image, ident, name, x, y, size, accent):
        from pathlib import Path
        from PIL import ImageOps
        record = (self.player_media or {}).get(ident) or (self.player_media or {}).get(str(ident)) or {}
        photo = record.get('photo')
        self.rounded(image, (x, y, x + size, y + size), '#194557', accent, 14, 2)
        if record.get('photo_kind') == 'player' and photo and Path(photo).is_file():
            key = 'freehit_avatar_' + str(ident)
            if key not in self.assets:
                with Image.open(photo) as source:
                    self.assets[key] = ImageOps.contain(source.convert('RGBA'), (int(size - 8), int(size - 8)))
            sprite = self.assets[key]
            image.alpha_composite(sprite, (int(x + (size - sprite.width) / 2), int(y + size - sprite.height - 4)))
        else:
            initials = ''.join(word[0] for word in str(name).split()[:2]).upper()
            self.label(image, initials, x + 13, y + size * .3, 28, accent, max_width=size - 20)
'''
    return content

def _vx_fix_masterplan_import():
    hero_path = Path(globals().get('VX_MASTERPLAN_PROJECT', VX_MEDIA_WORK / 'wildcard_masterplan_remotion')) / 'src' / 'HeroFlight.tsx'
    content = hero_path.read_text(encoding='utf-8')
    required = ('export const HeroFlight', 'export const HeroAnalysis', 'data-hero-side="right"', 'data-pitch-player')
    missing = [token for token in required if token not in content]
    if missing:
        raise RuntimeError('Wildcard Slide 3 expected separated hero/pitch renderer: ' + ', '.join(missing))
    print('✅ Wildcard Slide 3: pitch-only player tiles and right-side original hero cards verified', flush=True)

_vx_fix_masterplan_import()


def _vx_restore_render_runtime():
    import base64
    import io
    import zipfile
    module_root = VX_MEDIA_WORK / 'wildcard_engine'
    module_root.mkdir(parents=True, exist_ok=True)
    restored = []
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(_VX_RENDER_RUNTIME_BUNDLE))) as bundle:
        for name in bundle.namelist():
            relative = Path(name)
            if relative.is_absolute() or '..' in relative.parts:
                raise ValueError('Unsafe embedded runtime path')
            target = module_root / relative
            if not target.is_file() or name in ('wildcard_final_slide.py', 'wildcard_plan_audio.py', 'tts_narration.py', 'wildcard_analyst_intro.py', 'wildcard_intro_motion.py', 'wildcard_intro_context.py', 'freehit_personal_intro.py', 'freehit_intro_motion.py', 'freehit_premium.py'):
                content = bundle.read(name).decode('utf-8')
                if name == 'freehit_intro_motion.py':
                    content = _vx_fix_freehit_intro_source(content)
                compile(content, str(target), 'exec')
                target.parent.mkdir(parents=True, exist_ok=True)
                temporary = target.with_suffix('.py.part')
                temporary.write_text(content, encoding='utf-8')
                temporary.replace(target)
                restored.append(name)
    if str(module_root) not in sys.path:
        sys.path.insert(0, str(module_root))
    # Preserve attached Cell 13 hooks: restoration writes files without reloading modules.
    importlib.invalidate_caches()
    for name, payload in globals().get('_vx_asset_payloads', {}).items():
        target = VX_WC_EMBEDDED_ASSET_DIR / name
        if not target.is_file():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(base64.b64decode(payload))
    fdr_blob = globals().get('_vx_fdr_blob')
    fdr_root = globals().get('_vx_fdr_dest')
    if fdr_blob is not None and fdr_root is not None:
        with zipfile.ZipFile(io.BytesIO(fdr_blob)) as bundle:
            for name in bundle.namelist():
                relative = Path(name)
                if relative.is_absolute() or '..' in relative.parts:
                    raise ValueError('Unsafe embedded FDR runtime path')
                if name.endswith('/'):
                    continue
                target = Path(fdr_root) / relative
                if not target.is_file():
                    target.parent.mkdir(parents=True, exist_ok=True)
                    target.write_bytes(bundle.read(name))
    if restored:
        print(f'Restored {len(restored)} missing render runtime source files.', flush=True)

_vx_restore_render_runtime()
# A cell earlier in this notebook may already have imported the old Free Hit visual.
import freehit_premium as _vx_fhp_loaded
_vx_fhp_loaded = importlib.reload(_vx_fhp_loaded)

# Pick up the updated intro modules in warm Colab kernels too.
for _vx_intro_module in ('wildcard_analyst_intro', 'wildcard_intro_motion', 'freehit_intro_motion'):
    if _vx_intro_module in sys.modules:
        importlib.reload(sys.modules[_vx_intro_module])

# Reload the Playwright render modules from their notebook sources, even when
# Cell 13 ran earlier in a warm Colab kernel with outdated module objects.

_VX_REMOTION_RENDER_SAFE_PATCH = '\n# VX_REMOTION_CLI_SAFE_V4_PROGRESS\n# Keep Remotion\'s diagnostics and use a browser already installed for Colab.\n\ndef _vx_remotion_command_output(project, stage, command):\n    import subprocess\n    project = Path(project)\n    log_path = project / (\'vortex_remotion_\' + stage + \'.log\')\n    try:\n        with log_path.open(\'wb\') as log:\n            import time\n            started = time.monotonic()\n            print(f\'Remotion {stage}: started; progress log: {log_path}\', flush=True)\n            with subprocess.Popen(command, cwd=project, stdout=log, stderr=subprocess.STDOUT) as completed:\n                try:\n                    while True:\n                        try:\n                            completed.wait(timeout=10)\n                            break\n                        except subprocess.TimeoutExpired:\n                            with log_path.open(\'rb\') as progress:\n                                progress.seek(max(0, log_path.stat().st_size-4096))\n                                lines = progress.read().decode(\'utf-8\', errors=\'replace\').splitlines()\n                            detail = lines[-1][-240:] if lines else \'waiting for renderer output\'\n                            print(f\'Remotion {stage}: {time.monotonic()-started:.0f}s elapsed; {detail}\', flush=True)\n                except BaseException:\n                    completed.kill()\n                    completed.wait()\n                    raise\n            print(f\'Remotion {stage}: finished in {time.monotonic()-started:.1f}s\', flush=True)\n    except OSError as exc:\n        raise RuntimeError(\'Remotion \'+stage+\' could not start: \'+str(exc)) from exc\n    if completed.returncode:\n        lines = log_path.read_text(encoding=\'utf-8\', errors=\'replace\').splitlines()\n        detail = \'\\n\'.join(lines[-110:]) or \'(Remotion produced no stdout/stderr)\'\n        raise RuntimeError(\n            f\'Remotion {stage} failed (exit {completed.returncode}).\\n\'\n            f\'Full log: {log_path}\\n\'\n            + detail\n        )\n    return log_path\n\n\ndef _vx_remotion_cli(project):\n    project = Path(project)\n    binary = project / \'node_modules\' / \'.bin\' / \'remotion\'\n    if not binary.is_file():\n        print(\'Installing pinned Remotion CLI dependencies...\', flush=True)\n        _vx_remotion_command_output(project, \'npm_install\', [\'npm\', \'ci\', \'--no-audit\', \'--no-fund\'])\n    if not binary.is_file():\n        raise RuntimeError(\'Remotion CLI is still missing after npm ci: \'+str(binary))\n    return str(binary)\n\n\ndef _vx_remotion_browser_arguments(project):\n    import os\n    import shutil\n    from concurrent.futures import ThreadPoolExecutor\n    project = Path(project)\n    pinned = project / \'node_modules\' / \'.remotion\' / \'chrome-headless-shell\'\n    if any(p.is_file() for p in pinned.glob(\'**/chrome-headless-shell\')):\n        return []\n    explicit = os.getenv(\'VX_REMOTION_BROWSER_EXECUTABLE\')\n    candidates = ([explicit] if explicit else [])\n    candidates.extend(filter(None, [shutil.which(\'google-chrome-stable\'), shutil.which(\'google-chrome\'),\n        shutil.which(\'chromium\'), shutil.which(\'chromium-browser\')]))\n    candidates.append(\'/Applications/Google Chrome.app/Contents/MacOS/Google Chrome\')\n    def playwright_browser():\n        from playwright.sync_api import sync_playwright\n        with sync_playwright() as runtime:\n            return runtime.chromium.executable_path\n    try:\n        with ThreadPoolExecutor(max_workers=1, thread_name_prefix=\'vortex-browser-detection\') as pool:\n            candidates.insert(0, pool.submit(playwright_browser).result())\n    except (ImportError, OSError, RuntimeError) as exc:\n        print(\'Playwright browser path unavailable: \'+str(exc), flush=True)\n    for candidate in candidates:\n        if candidate and Path(candidate).is_file():\n            print(\'Remotion browser: \'+str(candidate), flush=True)\n            return [\'--browser-executable=\'+str(candidate),\n                    \'--chrome-mode=\'+(\'headless-shell\' if \'headless-shell\' in str(candidate) else \'chrome-for-testing\')]\n    print(\'Remotion browser: using its pinned browser downloader\', flush=True)\n    return []\n\n\ndef _vx_masterplan_safe_render_video(self, audio, duration, target, profile):\n    import math\n    profile = dict(profile)\n    if not math.isfinite(duration) or duration <= 0:\n        raise ValueError(f\'Invalid measured Wildcard Slide 3 duration: {duration!r}\')\n    self.prepare(audio, duration, profile)\n    cli = _vx_remotion_cli(self.project)\n    browser = _vx_remotion_browser_arguments(self.project)\n    if profile.get(\'height\',0)>480:\n        smoke = self.project / \'vortex_masterplan_preflight.png\'\n        print(\'Remotion: checking one frame before the full video render...\', flush=True)\n        _vx_remotion_command_output(self.project, \'preflight\',\n            [cli, \'still\', \'src/index.ts\', \'WildcardMasterplan\', str(smoke),\n             \'--frame=0\', \'--props={"mute":true}\', \'--log=verbose\', \'--overwrite\', *browser])\n        if not smoke.is_file() or smoke.stat().st_size == 0:\n            raise RuntimeError(\'Remotion preflight completed without producing a preview frame\')\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    # Colab Chromium / Node compete for memory: one renderer process is safer for long audio.\n    concurrency = 1 if profile.get(\'height\', 0) <= 1080 else 2\n    command = [cli, \'render\', \'src/index.ts\', \'WildcardMasterplan\', str(target),\n               \'--codec=h264\', \'--crf=\'+str(profile.get(\'crf\', 28)),\n               \'--concurrency=\'+str(concurrency), \'--timeout=60000\',\n               \'--log=info\', \'--overwrite\', *browser]\n    print(f\'Remotion: encoding {duration:.2f}s at {profile["width"]}x{profile["height"]}, concurrency={concurrency}\', flush=True)\n    _vx_remotion_command_output(self.project, \'video\', command)\n    if not target.is_file() or target.stat().st_size <= 0:\n        raise RuntimeError(\'Remotion returned success but produced no video: \'+str(target))\n    return target\n\n\ndef _vx_masterplan_safe_frame(self, seconds):\n    import math\n    import hashlib\n    import json\n    import numpy as np\n    duration = self.scene[\'narration_timeline\'][-1].get(\'end\')\n    if duration is None:\n        raise ValueError(\'Measure narration before a Slide 3 still\')\n    self.prepare(None, duration, dict(width=self.width, height=self.height, fps=24))\n    digest = hashlib.sha256((json.dumps(self.scene, sort_keys=True, default=str) +\n        str(seconds) + str(self.width)).encode()).hexdigest()[:20]\n    target = self.project / (\'still_\'+digest+\'.png\')\n    if not target.exists():\n        cli = _vx_remotion_cli(self.project)\n        _vx_remotion_command_output(self.project, \'still\',\n            [cli, \'still\', \'src/index.ts\', \'WildcardMasterplan\', str(target),\n             \'--frame=\'+str(min(math.ceil(duration*24)-1, round(seconds*24))),\n             \'--props={"mute":true}\', \'--log=verbose\', \'--overwrite\',\n             *_vx_remotion_browser_arguments(self.project)])\n    return np.asarray(Image.open(target).convert(\'RGB\'))\n\nRemotionMasterplanVisual.render_video = _vx_masterplan_safe_render_video\nRemotionMasterplanVisual.frame = _vx_masterplan_safe_frame\n'

def _vx_sync_browser_workers():
    import importlib
    import concurrent.futures
    specs = (
        ('wildcard_masterplan', '_VX_MASTERPLAN_SOURCE', 'VX_MASTERPLAN', 'hero_assets',
         'case, views, media, badges, brand_root, public'),
        ('wildcard_fdr_elite', '_VX_ELITE_FDR_SOURCE', 'VX_ELITE_FDR', 'render_hero_assets',
         'case, views, media, badges, brand_root, work'),
    )
    for name, source_variable, module_alias, entry, arguments in specs:
        module_path = VX_MEDIA_WORK / 'wildcard_engine' / (name + '.py')
        source = globals().get(source_variable)
        if not isinstance(source, str):
            if not module_path.is_file():
                raise RuntimeError(f'{name} is missing. Run Cell 13 before Cell 14.')
            source = module_path.read_text(encoding='utf-8')
        if name == 'wildcard_masterplan' and '# VX_REMOTION_CLI_SAFE_V4_PROGRESS' not in source:
            source += '\n' + _VX_REMOTION_RENDER_SAFE_PATCH
        marker = '# VX_PLAYWRIGHT_JUPYTER_WORKER_V1'
        if marker not in source:
            if f'def {entry}(' not in source or 'sync_playwright()' not in source:
                raise RuntimeError(f'{name} has an unrecognized Hero Card renderer.')
            extra = (f'\n\n{marker}\n'
                'from concurrent.futures import ThreadPoolExecutor as _VxBrowserThreadPool\n'
                f'_vx_original_{entry} = {entry}\n\n'
                f'def {entry}({arguments}):\n'
                '    # Colab runs this cell inside an active asyncio event loop.\n'
                f'    with _VxBrowserThreadPool(max_workers=1, thread_name_prefix="vortex-playwright") as executor:\n'
                f'        return executor.submit(_vx_original_{entry}, {arguments}).result()\n')
            source += extra
        compile(source, str(module_path), 'exec')
        module_path.parent.mkdir(parents=True, exist_ok=True)
        if not module_path.is_file() or module_path.read_text(encoding='utf-8') != source:
            temporary = module_path.with_suffix('.py.part')
            temporary.write_text(source, encoding='utf-8')
            temporary.replace(module_path)
        globals()[source_variable] = source
        Path(importlib.util.cache_from_source(str(module_path))).unlink(missing_ok=True)
        importlib.invalidate_caches()
        module = importlib.import_module(name)
        globals()[module_alias] = importlib.reload(module)
        if marker not in Path(module.__file__).read_text(encoding='utf-8'):
            raise RuntimeError(f'{name} Playwright worker patch was not loaded.')
    print('✅ Cell 14: Playwright Hero Card workers synchronized for Wildcard Slides 2 and 3', flush=True)

_vx_sync_browser_workers()


# Restore Free Hit Remotion protection even when Cell 13 was run earlier.
_VX_FREEHIT_RUNTIME_PATCH = "\n# VX_FREEHIT_REMOTION_SAFE_V3\n\ndef _vx_freehit_safe_render_video(self, audio, duration, target, profile):\n    import math\n    from wildcard_masterplan import _vx_remotion_cli, _vx_remotion_browser_arguments, _vx_remotion_command_output\n    if not math.isfinite(duration) or duration <= 0:\n        raise ValueError(f'Invalid Free Hit Slide 2 narration duration: {duration!r}')\n    profile = dict(profile)\n    if profile.get('height', 0) <= 480:\n        profile.update(width=854, height=480)\n    prepare_project(self.scene, self.project, self.media, self.badges, self.brand_root, audio,\n        profile['width'], profile['height'], profile['fps'], duration)\n    cli = _vx_remotion_cli(self.project)\n    browser_args = _vx_remotion_browser_arguments(self.project)\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    _vx_remotion_command_output(self.project, 'freehit_video',\n        [cli, 'render', 'src/index.ts', 'FreeHitMatchday', str(target),\n         '--codec=h264', '--crf='+str(profile.get('crf',28)),\n         '--concurrency=1', '--timeout=60000', '--log=info', '--overwrite', *browser_args])\n    if not target.is_file() or target.stat().st_size <= 0:\n        raise RuntimeError('Free Hit Remotion did not produce a video: '+str(target))\n    return target\n\nRemotionMatchdayVisual.render_video = _vx_freehit_safe_render_video\n"
if 'VX_FREEHIT_MATCHDAY' in globals():
    import importlib
    _fh_path = VX_MEDIA_WORK / 'wildcard_engine' / 'freehit_matchday.py'
    _fh_source = globals().get('_VX_FREEHIT_MATCHDAY_SOURCE')
    if not isinstance(_fh_source, str):
        _fh_source = _fh_path.read_text(encoding='utf-8')
    if '# VX_FREEHIT_REMOTION_SAFE_V3' not in _fh_source:
        _fh_source += '\n' + _VX_FREEHIT_RUNTIME_PATCH
    if not _fh_path.is_file() or _fh_path.read_text(encoding='utf-8') != _fh_source:
        _fh_path.write_text(_fh_source,encoding='utf-8')
    _VX_FREEHIT_MATCHDAY_SOURCE = _fh_source
    Path(importlib.util.cache_from_source(str(_fh_path))).unlink(missing_ok=True)
    importlib.invalidate_caches()
    VX_FREEHIT_MATCHDAY = importlib.reload(importlib.import_module('freehit_matchday'))

# A warm Colab kernel may still have the old speech helpers loaded.
for _vx_speech_module in ('wildcard_plan_audio', 'tts_narration', 'wildcard_analyst_intro', 'wildcard_intro_motion', 'wildcard_intro_context', 'freehit_personal_intro', 'freehit_intro_motion', 'freehit_premium'):
    _vx_speech_path = VX_MEDIA_WORK / 'wildcard_engine' / (_vx_speech_module + '.py')
    Path(importlib.util.cache_from_source(str(_vx_speech_path))).unlink(missing_ok=True)
    if _vx_speech_module in sys.modules:
        importlib.reload(sys.modules[_vx_speech_module])

def _vx_refresh_visual_renderer():
    module_path = Path(VXWV.__file__)
    if not module_path.is_file():
        module_path = VX_MEDIA_WORK / 'wildcard_engine' / 'vortex_wildcard_visuals.py'
        VXWV.__file__ = str(module_path)
        if VXWV.__spec__ is not None:
            VXWV.__spec__.origin = str(module_path)
    source = module_path.read_text(encoding='utf-8')
    updated = source.replace('max(8,int(size))', 'max(1,int(size))')
    updated = updated.replace('max(8, int(size))', 'max(1, int(size))')
    updated = updated.replace('pixels=max(8,round(size*self.sy))', 'pixels=max(1,round(size*self.sy))')
    old_footer = "self.text(im,(64,976),self.contract['fdr_source']+' • xPts/minutes are model estimates',22,MUTED)"
    updated = updated.replace(old_footer, old_footer[:-1] + ",anchor='lt')")
    stale_fonts = getattr(VXWV.font(4),'size',None) != 4 or getattr(VXWV.fixture_font(4),'size',None) != 4
    if updated != source:
        compile(updated, str(module_path), 'exec')
        temporary = module_path.with_suffix('.py.part')
        temporary.write_text(updated, encoding='utf-8')
        temporary.replace(module_path)
    if updated != source or stale_fonts:
        # A corrected file can coexist with an old imported module in a warm
        # kernel. Discard its generated bytecode before loading the new source.
        Path(importlib.util.cache_from_source(str(module_path))).unlink(missing_ok=True)
        importlib.invalidate_caches();importlib.reload(VXWV)
    if VXWV.font(4).size != 4 or VXWV.fixture_font(4).size != 4:
        raise RuntimeError('The shared renderer did not accept scaled fonts. Rerun Cell 4, then Cell 14.')

_vx_refresh_visual_renderer()

# Runtime-local resolver: this cell must work even when an older Cell 0 definition
# is still loaded in the active Colab kernel.
def _vx_resolve_assets_runtime(folder_url, work):
    import json
    import os
    import shutil
    from pathlib import Path

    # Both four-slide videos begin with Slide 1 (include_opening=False).
    # Resolve only the clips that the current builders actually use.
    names=['outro.mp4']
    if globals().get('VX_INCLUDE_POPUPS',False):
        names.append('popup.mp4')

    work=Path(work)
    work.mkdir(parents=True,exist_ok=True)

    local=Path(os.environ.get('FPL_MEDIA_ASSET_DIR',str(Path.cwd()/'media_assets')))
    if all((local/name).is_file() for name in names):
        return {name:local/name for name in names}

    drive_root=Path('/content/drive/MyDrive')
    if not drive_root.is_dir():
        raise RuntimeError('Google Drive is not mounted. Run Cell 0 first and approve the single Drive mount prompt.')

    asset_dir_candidates=[
        drive_root/'FPL_VORTEX'/'Assets ',
        drive_root/'FPL_VORTEX'/'Assets',
    ]
    asset_dir=next((candidate for candidate in asset_dir_candidates if candidate.is_dir()),None)
    if asset_dir is None:
        raise RuntimeError('Could not find the mounted FPL_VORTEX Assets folder in My Drive.')

    source_paths={
        'outro.mp4': asset_dir/'outro.mp4',
        'popup.mp4': asset_dir/'popup.mp4',
    }

    assets={}
    for name in names:
        source=source_paths[name]
        if not source.is_file():
            raise RuntimeError(
                f'Missing required video asset {name} at {source}. '
                'Restore it in Drive or set FPL_MEDIA_ASSET_DIR to a directory containing all required clips.'
            )

        target=work/name
        source_stat=source.stat()
        stamp=work/(name+'.source.json')
        expected={
            'source':str(source),
            'size':source_stat.st_size,
            'mtime_ns':source_stat.st_mtime_ns,
        }
        cached_ok=False
        if target.is_file() and stamp.is_file():
            try:
                cached_ok=(json.loads(stamp.read_text())==expected and target.stat().st_size==source_stat.st_size)
            except (ValueError,OSError):
                cached_ok=False

        if not cached_ok:
            temporary=work/(name+'.part')
            temporary.unlink(missing_ok=True)
            shutil.copy2(source,temporary)
            if temporary.stat().st_size<=0:
                temporary.unlink(missing_ok=True)
                raise RuntimeError(f'Copied video asset {name} is empty.')
            temporary.replace(target)
            stamp.write_text(json.dumps(expected))

        assets[name]=target
    return assets

from functools import lru_cache

def vx_find_font(bold=False):
    names=['DejaVuSans-Bold.ttf','LiberationSans-Bold.ttf','Arial Bold.ttf'] if bold else ['DejaVuSans.ttf','LiberationSans-Regular.ttf','Arial.ttf']
    roots=[Path('/usr/share/fonts'),Path('/usr/local/share/fonts'),Path('/System/Library/Fonts/Supplemental')]
    for directory in roots:
        if directory.is_dir():
            for name in names:
                matches=sorted(directory.rglob(name))
                if matches:return str(matches[0])
    return None

FONT_REGULAR=vx_find_font()
FONT_BOLD=vx_find_font(True) or FONT_REGULAR
VX_RENDERER_VERSION='wildcard-slide3-premium-sixweek-hero-v22-wc2-broadcast-v2-2026-10-11'

@lru_cache(maxsize=256)
def _font(size,bold=False,scale=1.0):
    pixels=max(8,round(size*scale));path=FONT_BOLD if bold else FONT_REGULAR
    if path:
        try:return ImageFont.truetype(path,pixels)
        except OSError:pass
    # Pillow's bundled scalable font also works when system fonts are absent.
    try:return ImageFont.load_default(size=pixels)
    except TypeError:return ImageFont.load_default()


def _wrap(draw,text,font,width):
    words=str(text).split();lines=[];line=''
    for word in words:
        test=(line+' '+word).strip()
        if draw.textbbox((0,0),test,font=font)[2]<=width:line=test
        else:
            if line:lines.append(line)
            line=word
    if line:lines.append(line)
    return lines


def _image_for_player(pid):
    record=VX_PLAYER_MEDIA.get(int(pid),{})
    path=record.get('photo')
    if not path or not Path(path).exists():
        raise RuntimeError(f'No resolved real image for player {pid}; rerun media preparation.')
    return Image.open(path).convert('RGBA'),record






def native_body_segment(visual,audio_path,duration,target,profile,previous=None):
    if hasattr(visual,'render_video'):return visual.render_video(audio_path,duration,target,profile)
    args=[ffmpeg_binary(),'-hide_banner','-loglevel','error','-y','-f','rawvideo','-pix_fmt','rgb24','-s',f"{profile['width']}x{profile['height']}",'-r',str(profile['fps']),'-i','pipe:0','-i',str(audio_path),'-t',str(duration),'-vf','setsar='+profile['sar']+':max=10000','-c:v','libx264','-preset',profile['preset'],'-crf',str(profile['crf']),'-pix_fmt','yuv420p','-threads','4','-r',str(profile['fps']),'-fps_mode','cfr','-video_track_timescale',str(profile['fps']*2048),'-aspect','16:9','-af','apad','-c:a','aac','-ar','48000','-ac','2','-movflags','+faststart',str(target)]
    args=video_encode_args(args,profile)
    log=target.with_suffix('.encode.log')
    with log.open('wb') as handle:
        process=subprocess.Popen(args,stdin=subprocess.PIPE,stderr=handle)
        try:
            total_frames = math.ceil(duration*profile['fps'])
            frame_started = last_progress = time.monotonic()
            print(f"Rendering {total_frames} animated frames at {profile['width']}×{profile['height']} / {profile['fps']} fps", flush=True)
            for index in range(total_frames):
                process.stdin.write(visual.frame(index/profile['fps'],duration,previous).tobytes())
                now = time.monotonic()
                if index == 0 or index+1 == total_frames or now-last_progress >= 10:
                    elapsed = max(.001, now-frame_started)
                    remaining = (total_frames-index-1)*elapsed/(index+1)
                    print(f"Animation frames: {index+1}/{total_frames}; {elapsed:.1f}s elapsed; ~{remaining:.0f}s remaining", flush=True)
                    last_progress = now
            process.stdin.close();code=process.wait()
        except BrokenPipeError as exc:
            process.wait()
            raise RuntimeError('Video encoder closed its input: '+log.read_text()[-3000:]) from exc
        except BaseException:process.kill();process.wait();raise
    if code:raise RuntimeError(log.read_text()[-3000:])


def ffprobe_binary():
    executable=shutil.which('ffprobe')
    if executable:return executable
    sibling=Path(ffmpeg_binary()).with_name('ffprobe')
    if sibling.exists():return str(sibling)
    raise RuntimeError('FFprobe was not found. Colab FFmpeg normally includes ffprobe.')


def probe_media_info(path):
    path=Path(path);stat=path.stat()
    key=(str(path.resolve()),stat.st_size,stat.st_mtime_ns,'media-timebase-v2')
    cache=globals().setdefault('_VX_MEDIA_PROBE_CACHE',{})
    if key in cache:return cache[key]
    result=subprocess.run([ffprobe_binary(),'-v','error','-show_entries','format=duration:stream=codec_type,width,height,r_frame_rate,sample_aspect_ratio,display_aspect_ratio,time_base,duration','-of','json',str(path)],capture_output=True,text=True,timeout=30)
    if result.returncode:raise RuntimeError('Cannot read media: '+result.stderr[-1000:])
    payload=json.loads(result.stdout)
    if len(cache)>=512:cache.clear()
    cache[key]=payload
    return payload


def probe_media(path):
    payload=probe_media_info(path)
    return float(payload.get('format',{}).get('duration',0)),any(s.get('codec_type')=='audio' for s in payload.get('streams',[]))


def media_duration(path):return probe_media(path)[0]

def normalize_asset(source,target,profile):
    w,h=profile['width'],profile['height'];duration,has_audio=probe_media(source);inputs=['-i',source]
    if not has_audio:inputs+=['-f','lavfi','-i','anullsrc=channel_layout=stereo:sample_rate=48000']
    run_ffmpeg([*inputs,'-t',duration,'-vf',f"scale={w}:{h}:force_original_aspect_ratio=decrease,pad={w}:{h}:(ow-iw)/2:(oh-ih)/2:color=0x080f1c,setsar={profile['sar']}:max=10000",'-r',profile['fps'],'-fps_mode','cfr','-video_track_timescale',profile['fps']*2048,'-aspect','16:9','-c:v','libx264','-crf',profile['crf'],'-preset',profile['preset'],'-pix_fmt','yuv420p','-threads','4','-c:a','aac','-ar','48000','-ac','2','-movflags','+faststart',target]);return duration




def _vx_run_async(coro):
    # A fresh loop avoids nested-loop timeout failures on newer Python versions.
    try:asyncio.get_running_loop()
    except RuntimeError:return asyncio.run(coro)
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=1) as worker:return worker.submit(asyncio.run,coro).result()

VX_TTS_VOICE='en-GB-RyanNeural';VX_VIDEO_PROFILE=video_profile(VX_VIDEO_MODE)
if VX_VIDEO_MODE == 'Draft':
    VX_VIDEO_PROFILE.update(draft_keyframes=True, light_motion=True)



@lru_cache(maxsize=256)
def fitted_image(path,modified,width,height,top_crop=False,fill=False):
    pic=Image.open(path).convert('RGBA')
    if top_crop:
        w,h=pic.size;crop=min(w,h);pic=pic.crop(((w-crop)//2,0,(w+crop)//2,min(h,crop)))
    if fill:return ImageOps.fit(pic,(width,height),method=Image.Resampling.LANCZOS)
    return ImageOps.contain(pic,(width,height),Image.Resampling.LANCZOS)

def media_tile(path,size,top_crop=False,fill=False):
    path=Path(path)
    return fitted_image(str(path),path.stat().st_mtime_ns,int(size[0]),int(size[1]),top_crop,fill)



# Shared premium renderer for both chips, grounded in actual reference frames and the user's brand.
def vx_prepare_club_media(bootstrap,work):
    import io,requests
    from concurrent.futures import ThreadPoolExecutor
    directory=Path(work)/'club_media';directory.mkdir(exist_ok=True)
    def fetch(team):
        path=directory/f"t{int(team['code'])}.png";source=None
        if not path.exists():
            for url in [f"https://resources.premierleague.com/premierleague/badges/100/t{int(team['code'])}.png",f"https://resources.premierleague.com/premierleague/badges/t{int(team['code'])}.png"]:
                try:
                    r=requests.get(url,timeout=12);r.raise_for_status();pic=Image.open(io.BytesIO(r.content)).convert('RGBA')
                    if min(pic.size)<16:continue
                    pic.save(path);source=url;break
                except (requests.RequestException,OSError,ValueError):continue
        return str(team['short_name']),{'path':str(path) if path.exists() else None,'team_code':int(team['code']),'source':source or ('validated cache' if path.exists() else 'unavailable')}
    with ThreadPoolExecutor(max_workers=6) as executor:result=dict(executor.map(fetch,bootstrap['teams']))
    (Path(work)/'club_media_audit.json').write_text(json.dumps(result,indent=2))
    return result





def vx_check_text_layout(image):
    records=image.info.get('_vx_text_boxes',[]);seen=set();unique=[]
    for item in records:
        key=(item['text'],tuple(item['box']))
        if key not in seen:seen.add(key);unique.append(item)
    for i,a in enumerate(unique):
        for b in unique[i+1:]:
            ax,ay,ax2,ay2=a['box'];bx,by,bx2,by2=b['box']
            if min(ax2,bx2)-max(ax,bx)>2 and min(ay2,by2)-max(ay,by)>2:raise ValueError('Free Hit text overlaps: '+a['text']+' / '+b['text'])
    return len(unique)

def vx_valid_media_cache(path):
    if not path.exists():
        return False
    try:
        info=probe_media_info(path)
        if float(info.get('format',{}).get('duration',0))>0:
            streams=info.get('streams',[])
            if path.suffix.lower()=='.mp4':
                video=next((x for x in streams if x.get('codec_type')=='video'),None)
                audio=next((x for x in streams if x.get('codec_type')=='audio'),None)
                return bool(video and audio and video.get('width')==VX_VIDEO_PROFILE['width'] and video.get('height')==VX_VIDEO_PROFILE['height'])
            return any(x.get('codec_type')=='audio' for x in streams)
    except (RuntimeError, ValueError, OSError):
        pass
    path.unlink(missing_ok=True)
    return False

def file_sha256(path):
    path=Path(path);stat=path.stat()
    key=(str(path.resolve()),stat.st_size,stat.st_mtime_ns)
    cache=globals().setdefault('_VX_FILE_HASH_CACHE',{})
    if key in cache:return cache[key]
    digest=hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda:handle.read(8*1024*1024),b''):digest.update(block)
    if len(cache)>=512:cache.clear()
    cache[key]=digest.hexdigest()
    return cache[key]


def vx_concat_scenes(parts,work,key,profile):
    """Real video/audio crossfades; each input includes 0.65s tail padding for the 0.5s crossfade."""
    current=parts[0]
    for i,part in enumerate(parts[1:],1):
        sig=hashlib.sha256((file_sha256(current)+file_sha256(part)+str(profile)).encode()).hexdigest()[:14]
        out=work/f'{key}_crossfade_{i}_{sig}.mp4'
        if not vx_valid_media_cache(out):
            offset=media_duration(current)-.5
            if profile.get('light_motion',False):
                # Draft's native frame-blend avoids FFmpeg xfade's undefined-rate issue;
                # Final keeps the original animated 4K xfade implementation.
                fps=int(profile['fps'])
                fade_frames=max(2,round(0.5*fps))
                head_frames=round(media_duration(current)*fps)-fade_frames
                if head_frames<1 or round(media_duration(part)*fps)<=fade_frames:
                    raise ValueError('Draft clips must be longer than their transition')
                fade_denominator=fade_frames-1
                video_blend=f"A*(1-min(1,N/{fade_denominator}))+B*min(1,N/{fade_denominator})"
                graph=(
                    f"[0:v]fps={fps},format=yuv420p,split=2[a0][a1];"
                    f"[a0]trim=end_frame={head_frames},setpts=PTS-STARTPTS[head];"
                    f"[a1]trim=start_frame={head_frames}:end_frame={head_frames+fade_frames},setpts=PTS-STARTPTS[tail];"
                    f"[1:v]fps={fps},format=yuv420p,split=2[b0][b1];"
                    f"[b0]trim=end_frame={fade_frames},setpts=PTS-STARTPTS[entry];"
                    f"[b1]trim=start_frame={fade_frames},setpts=PTS-STARTPTS[rest];"
                    f"[tail][entry]blend=all_expr='{video_blend}'[mix];"
                    f"[head][mix][rest]concat=n=3:v=1:a=0,format=yuv420p,setsar={profile['sar']}:max=10000[v];"
                    "[0:a][1:a]acrossfade=d=0.5:c1=tri:c2=tri[audio]"
                )
            else:
                graph=f"[0:v]fps={profile['fps']},setpts=PTS-STARTPTS[a];[1:v]fps={profile['fps']},setpts=PTS-STARTPTS[b];[a][b]xfade=transition=fade:duration=0.5:offset={offset},format=yuv420p,fps={profile['fps']},setsar={profile['sar']}:max=10000[v];[0:a][1:a]acrossfade=d=0.5:c1=tri:c2=tri[audio]"
            run_ffmpeg(['-i',current,'-i',part,'-filter_complex_threads','1','-filter_complex',graph,'-map','[v]','-map','[audio]','-c:v','libx264','-preset',profile['preset'],'-crf',profile['crf'],'-threads','4','-r',profile['fps'],'-fps_mode','cfr','-aspect','16:9','-video_track_timescale',profile['fps']*2048,'-c:a','aac','-ar','48000','-ac','2',out])
        current=out
    return current


def vx_assert_current_snapshot(contract):
    stamp=min((contract['generated_at'],contract.get('source_snapshot_at',contract['generated_at'])),key=VXWC.utc)
    check_at=globals().get('VX_RENDER_BATCH_STARTED_AT') or VXWC.utc()
    age=(check_at-VXWC.utc(stamp)).total_seconds()
    hours=float(getattr(getattr(VX_WC_ENGINE,'cfg',None),'cache_hours',1.))
    if age < 0:
        raise RuntimeError(f"{contract['chip']} source snapshot timestamp is in the future. Rerun Cell 8, then Cells 11–14 before rendering.")
    if globals().get('VX_VIDEO_MODE') == 'Draft':
        if age > hours*3600:
            warning_key=(contract.get('chip'),contract.get('data_hash'))
            if warning_key not in _VX_DRAFT_STALE_WARNED:
                print(
                    f"⚠️ Draft preview: {contract['chip']} source snapshot is {age/3600:.2f} hours old "
                    f"(normal freshness window {hours:g} hour). Continuing because Draft is review-only; "
                    "Final mode still requires a fresh snapshot.",
                    flush=True,
                )
                _VX_DRAFT_STALE_WARNED.add(warning_key)
        return age
    if age > hours*3600:
        raise RuntimeError(
            f"{contract['chip']} source snapshot was already {age/3600:.2f} hours old when Cell 14 started; "
            f"Final mode requires <= {hours:g} hour. Rerun Cell 8, then Cells 11–14."
        )
    return age

def vx_assert_script_contract(path,contract):
    text=Path(path).read_text(encoding='utf-8')
    match=re.search(r'^# DATA_HASH:\s*(\S+)',text,re.M)
    if match:actual=match.group(1)
    else:
        provenance=VX_ANALYSIS_ROOT/'narration_provenance'/(Path(path).name+'.provenance.json')
        actual=json.loads(provenance.read_text()).get('data_hash') if provenance.exists() else None
    if actual!=contract['data_hash']:
        raise ValueError('Narration refers to older or unverified facts. Rerun Cells 12–13 and review the TXT before rendering.')
    return text

async def vx_prefetch_scene_ryan(scene: dict, segment: dict) -> None:
    """Warm this slide's exact speech cache without changing its audio clock."""
    if VX_VIDEO_MODE != 'Draft':
        return
    cues = scene.get('narration_timeline') or []
    if not cues:
        return
    unique_segments = {}
    for cue in cues:
        prepared = vx_prepare_tts(dict(segment, id=scene['scene_id']+'_'+cue['segment_id'],
            text=cue['narration_text'], scene=scene, narration_cue=cue))
        cache_key = json.dumps([prepared['tts_text'], vx_narration_profile(prepared)], sort_keys=True)
        unique_segments.setdefault(cache_key, prepared)
    semaphore = asyncio.Semaphore(2)
    completed = 0
    started = time.monotonic()
    print(f"{scene['title']}: preparing {len(unique_segments)} Ryan cues (up to two at once)", flush=True)
    async def prepare(prepared: dict) -> None:
        nonlocal completed
        async with semaphore:
            await _vx_wc_audio_for_segment(prepared)
            completed += 1
            print(f"Ryan cues ready: {completed}/{len(unique_segments)}; {time.monotonic()-started:.1f}s", flush=True)
    tasks = [asyncio.create_task(prepare(prepared)) for prepared in unique_segments.values()]
    try:
        await asyncio.gather(*tasks)
    except BaseException:
        for task in tasks:
            task.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)
        raise


def vx_show_draft_slide_audit(scene: dict, key: str, elapsed: float, reused: bool) -> None:
    """Expose each verified Draft slide as soon as it is ready."""
    if VX_VIDEO_MODE != 'Draft':
        return
    from html import escape
    from IPython.display import HTML, Video, display
    title = str(scene.get('title') or scene['scene_id'])
    path = Path(scene['media_file'])
    if not path.is_file():
        raise FileNotFoundError(f'Draft audit slide is missing: {path}')
    rows = []
    for cue in scene.get('narration_timeline') or []:
        rows.append('<tr><td>'+escape(str(cue.get('segment_id', '')))+
            '</td><td>'+escape(f"{float(cue['start']):.1f}–{float(cue['end']):.1f}s")+
            '</td><td>'+escape(str(cue.get('narration_text', '')))+'</td></tr>')
    status = 'Cached video reused' if reused else 'New video rendered'
    markup = '<h3>'+escape(key+' · '+title)+'</h3><p>'+escape(f'{status}; slide ready in {elapsed:.1f}s.')+'</p>'
    markup += '<details><summary>Audit narration and measured cue timing</summary><p style="white-space:pre-wrap">'+escape(str(scene['narration_text']))+'</p>'
    markup += '<table><thead><tr><th>Cue</th><th>Time</th><th>Narration</th></tr></thead><tbody>'+''.join(rows)+'</tbody></table></details>'
    display(HTML(markup))
    display(Video(str(path), embed=True, width=854, html_attributes='controls preload="none"'))
    print('Slide available above for audit; remaining slides continue automatically.', flush=True)



def _vx_render_verified_freehit(plan,key,label,assets):
    contract_path=VX_ANALYSIS_ROOT/('wildcard_data.json' if key.startswith('wildcard') else 'freehit_data.json')
    contract=json.loads(contract_path.read_text());vx_validate_contract(contract)
    if contract['chip']=='freehit':vx_validate_freehit(contract)
    vx_assert_current_snapshot(contract)
    script=decode_script(vx_assert_script_contract(VX_NARRATION_PATHS[key],contract))
    manifest_path=VX_ANALYSIS_ROOT/(key+'_scene_manifest.json')
    # Rebuild visuals from persisted data, then apply only explicit saved narration edits.
    scenes=vx_build_manifest(contract)
    if [s['id'] for s in script]!=[s['scene_id'] for s in scenes]:raise ValueError('TXT scene headers must match the selected chip manifest scenes. Regenerate the script or restore the headers.')
    body_parts=[];previous=None;start=0
    for index,(segment,scene) in enumerate(zip(script,scenes)):
        _vx_slide_started = time.monotonic()
        print(f'{key}: slide {index+1}/{len(scenes)} — {scene["title"]}', flush=True)
        scene['narration_text']=segment['text']
        # Validate edited sentences without altering the user's prose.
        # Premium narration is decision-led; do not force an arbitrary sentence-length cap.
        spoken=segment['text']
        if scene['visual_type'] in ('fh_blueprint','fixture_heatmap','fh_squad_plan','fh_captain_final'):
            _vx_run_async(vx_prefetch_scene_ryan(scene, segment))
            from wildcard_plan_audio import prepare_segment_audio
            audio=(VX_FREEHIT_SUITE.prepare_squad_audio if scene["visual_type"]=="fh_squad_plan" else VX_MASTERPLAN.prepare_audio if scene.get("masterplan_case") else prepare_segment_audio)(scene,segment,VX_MEDIA_WORK,
                lambda item:_vx_run_async(_vx_wc_audio_for_segment(item)),
                lambda source,target:run_ffmpeg(['-i',source,'-ar','48000','-ac','2','-c:a','pcm_s16le',target]))
        else:
            audio=_vx_run_async(_vx_wc_audio_for_segment(dict(segment,text=spoken,scene=scene)))
        if scene['visual_type']=='wc_blueprint':
            scene['animation_version'] = 'wc-slide1-ryan-cue-v6'
            _slide1_export={'scene_id':scene['scene_id'],'display_text':scene['narration_text'],
                'narration_timeline':scene['narration_timeline'],'verdict':scene['wildcard_case']['verdict'],
                'comparison_available':scene['wildcard_case']['scenario_comparison'] is not None,
                'alignment':'measured_segments','voice':VX_TTS_VOICE,'voice_settings':vx_narration_profile(segment)}
            (VX_ANALYSIS_ROOT/(key+'_slide1_script_and_cues.json')).write_text(json.dumps(_slide1_export,ensure_ascii=False,indent=2),encoding='utf-8')
        if scene['visual_type']=='wc_squad_plan':
            (VX_ANALYSIS_ROOT/(key+'_slide3_script_and_cues.json')).write_text(json.dumps({'display_text':scene['narration_text'],'narration_timeline':scene['narration_timeline'],'formation':scene['squad_plan']['formation'],'starting_xi_ids':[p['id'] for p in scene['squad_plan']['starting_xi']],'bench_ids':[p['id'] for p in scene['squad_plan']['bench']],'squad_cost':scene['squad_plan']['total_cost'],'bank':scene['squad_plan']['bank'],'alignment':'measured_segments'},ensure_ascii=False,indent=2),encoding='utf-8')
        original_seconds=media_duration(audio)
        word_count=len(vx_prepare_tts(dict(segment,scene=scene))['tts_text'].split());tempo=1.0
        paced=audio
        digest=file_sha256(audio)[:18]
        speech_duration=media_duration(paced)
        duration=speech_duration+2.5
        if scene['visual_type']=='fh_blueprint':duration=speech_duration
        scene.update(duration_seconds=duration,audio_duration_seconds=speech_duration,narration_word_count=word_count,narration_speed=tempo,start_seconds=start)
        scene['timing']['status']='measured';scene['timing']['voice_profile']=vx_narration_profile(dict(segment,scene=scene));scene['timing']['audio_padding_seconds']=0.0 if scene['visual_type']=='fh_blueprint' else 2.5;scene['timing']['audio_file']=str(paced)
        scene['timing']['row_highlight_alignment']='measured narration segments' if scene['timing'].get('player_alignment')=='measured_segments' else 'estimated from narration word position'
        if scene['visual_type'] in ('wc_analysis','fixture_heatmap'):
            from fixture_slide import bind_scene_timeline
            _fixture_durations=([item['end']-item['start'] for item in scene['narration_timeline']] if scene['timing'].get('player_alignment')=='measured_segments' else None)
            bind_scene_timeline(scene,scene['audio_duration_seconds'],_fixture_durations)
            if scene['visual_type']=='wc_analysis':
                (VX_ANALYSIS_ROOT/(key+'_slide2_script_and_cues.json')).write_text(json.dumps({'display_text':scene['narration_text'],'narration_timeline':scene['narration_timeline'],'ranking':scene['fixture_case']['diagnostics']['ranking']},ensure_ascii=False,indent=2),encoding='utf-8')
        (VX_ANALYSIS_ROOT/(key+'_'+scene['scene_id']+'_script_and_cues.json')).write_text(json.dumps({'scene_id':scene['scene_id'],'display_text':scene['narration_text'],'narration_timeline':scene.get('narration_timeline',[]),'alignment':scene['timing'].get('player_alignment'),'data_hash':contract['data_hash']},ensure_ascii=False,indent=2),encoding='utf-8')
        _vx_draft_png = None
        visual=vx_visual_from_manifest(scene,contract,VX_VIDEO_PROFILE['width'],VX_VIDEO_PROFILE['height'])
        fingerprint=hashlib.sha256((json.dumps(scene,sort_keys=True)+json.dumps({k:v for k,v in contract.items() if k not in ('model_audit','data_sources')},sort_keys=True)+VX_STORY_VERSION+VX_RENDERER_VERSION+str((FONT_REGULAR,FONT_BOLD))+json.dumps(VX_VIDEO_PROFILE,sort_keys=True)+((file_sha256(_vx_draft_png)) if _vx_draft_png is not None else '')+vx_render_asset_signature(assets)+json.dumps(VX_PLAYER_MEDIA,sort_keys=True)+('wc-slide2-full20-premium-v4' if scene.get('elite_fixture_case') else '')).encode()).hexdigest()[:18]
        part=VX_MEDIA_WORK/f'{key}_manifest_{index:02}_{fingerprint}.mp4'
        _vx_scene_reused = vx_valid_media_cache(part)
        if not _vx_scene_reused:
            if _vx_draft_png is not None:
                vx_fast_draft_from_png(_vx_draft_png,paced,duration,part,VX_VIDEO_PROFILE,motion_index=index)
            else:
                _vx_wc_native_body_segment(visual,paced,duration,part,VX_VIDEO_PROFILE)
        body_parts.append(part);start+=media_duration(part)-(.5 if VX_VIDEO_PROFILE.get('scene_crossfades',True) and index<len(scenes)-1 else 0)
        scene['encoded_duration_seconds']=media_duration(part)
        scene['cache_media_file']=str(part)
        scene['media_file']=str(vx_publish_scene_mp4(part,key,contract['gameweek'],index+1,scene['scene_id']))
        manifest_path.write_text(json.dumps(scenes,indent=2,ensure_ascii=False))
        vx_show_draft_slide_audit(scene, key, time.monotonic()-_vx_slide_started, _vx_scene_reused)
        print(f"{key} {scene['scene_id']}: {scene['title']} / {duration:.1f}s",flush=True)
    body=_vx_wc_balanced_concat(body_parts,VX_MEDIA_WORK,key,VX_VIDEO_PROFILE)
    path=VX_VIDEO_OUT_DIR/key/VX_VIDEO_MODE.lower()/f'{"Wildcard" if key.startswith("wildcard") else "Free Hit"} Combined {VX_VIDEO_MODE} GW{contract["gameweek"]}.mp4'
    audit=_vx_wc_assemble_branded([body],assets,path,VX_MEDIA_WORK,VX_VIDEO_PROFILE,include_opening=False)
    audit.update(narration_profile_mode=VX_NARRATION_PROFILE,narration_profiles=VX_NARRATION_PROFILES,voice=VX_TTS_VOICE,speech_rate=VX_RYAN_RATE,pitch_hz=VX_RYAN_PITCH_HZ,volume=VX_VIDEO_MOODS['analysis']['volume'],story_version=VX_FREEHIT_STORY_VERSION if contract['chip']=='freehit' else VX_STORY_VERSION,presentation_version=VX_PRESENTATION_VERSION,scenes=len(scenes),scene_manifest=str(manifest_path),data_contract=str(contract_path),transition=('0.5 second video/audio crossfade' if VX_VIDEO_PROFILE.get('scene_crossfades',True) else 'hard cuts'),total_duration=media_duration(path),template='7.png')
    if audit['total_duration']<=0:raise RuntimeError('Rendered video has no duration')
    (VX_ANALYSIS_ROOT/(key+'_media_audit.json')).write_text(json.dumps(audit,indent=2))
    print(f"Saved {path} — {audit['total_duration']/60:.1f} min, {len(scenes)} manifest scenes")
    return path

# One dispatch path; each chip keeps its own presentation strategy.
def vx_render_asset_signature(assets):
    hashes = {name: file_sha256(Path(path)) for name, path in sorted(assets.items())
              if Path(path).is_file()}
    hashes.update({'broadcast:' + name: digest for name, digest in VX_BROADCAST_RENDERER.asset_hashes.items()})
    hashes['broadcast:template'] = VX_BROADCAST_RENDERER.source_hash
    return json.dumps(hashes, sort_keys=True)


def vx_approved_freehit_hero_cards(scene,contract):
    """Render original, notebook-defined Hero Cards; never auto-generate substitutes."""
    from freehit_premium import captain_plan
    from freehit_matchday import render_original_heroes
    import copy
    kind=scene['visual_type']
    plan=contract['slide3_plan']
    if kind=='fh_squad_plan':
        selected=list(plan['starting_xi'])+list(plan['bench'])
    elif kind=='fh_captain_final':
        captaincy=captain_plan(contract)
        selected=list(captaincy['candidates'])+[captaincy['vice_captain']]
    else:
        raise ValueError('Original Free Hit cards are only required on Slides 3 and 4')
    unique={int(player['id']):player for player in selected}
    rows=copy.deepcopy(contract['fixture_case']['rows'])
    if len(rows)!=20 or len({int(row['team_id']) for row in rows})!=20:
        raise ValueError('Free Hit original cards require the exact 20-club matchday source')
    for row in rows:
        rows_for_week=[week for week in row['weeks'] if int(week['gameweek'])==int(contract['gameweek'])]
        if len(rows_for_week)!=1:raise ValueError('Hero card fixtures must match the one-week plan')
        row['fixtures']=rows_for_week[0]['fixtures']
    hero_data=dict(season=contract.get('season'),rows=rows,
        players=[dict(player_id=int(p['id']),team_id=int(p['team_id']),club=p['club'],
                      name=p['name'],price=float(p['price']),xpts=float(p['projected_points']))
                 for p in unique.values()])
    paths,proof=render_original_heroes(hero_data,VX_CARD_VIEWS,VX_PLAYER_MEDIA,
        VX_CLUB_MEDIA,VX_BROADCAST_ROOT,VX_MEDIA_WORK/'freehit_approved_hero_cards')
    if set(map(int,paths))!=set(unique):
        raise RuntimeError('Original Hero Card count does not match selected scene players')
    scene['hero_card_asset_proof']=proof
    return paths

def vx_visual_from_manifest(scene,contract,width,height):
    kind=scene['visual_type']
    if contract.get('chip') == 'freehit' and kind in ('fh_blueprint', 'fixture_heatmap'):
        scene['animation_version'] = 'freehit-slides1-2-fullframe-70-30-v2'
    if kind=='fixture_heatmap' and scene.get('matchday_case'):
        VX_FREEHIT_MATCHDAY.CARD_VIEWS = VX_CARD_VIEWS
        _expected_matchday_heroes = {int(player['player_id']) for player in scene['matchday_case'].get('players',[])}
        _missing_matchday_heroes = sorted(_expected_matchday_heroes - set(map(int,VX_CARD_VIEWS)))
        if _missing_matchday_heroes:
            raise RuntimeError('Free Hit matchday hero views missing from official snapshot: '
                               + ', '.join(map(str,_missing_matchday_heroes))
                               + '. Rerun Cell 14 to rebuild the complete matchday shortlist.')
        if any(c.get('start') is None for c in scene['narration_timeline']):raise ValueError('Measure the Free Hit narration before rendering Slide 2')
        return VX_FREEHIT_MATCHDAY.RemotionMatchdayVisual(scene,contract,width,height,VX_FREEHIT_MATCHDAY_PROJECT,VX_PLAYER_MEDIA,VX_CLUB_MEDIA,VX_BROADCAST_ROOT)
    if kind=='fh_blueprint':
        import copy
        from freehit_intro_motion import FreeHitIntroMotion
        from wildcard_plan import align_timeline
        scene=copy.deepcopy(scene)
        if any(cue.get('start') is None for cue in scene['narration_timeline']):
            raise ValueError('Generate measured Ryan narration before Free Hit animation')
        shell=dict(scene,display_title='FREE HIT GW'+str(contract['gameweek']),title='FREE HIT GW'+str(contract['gameweek']),subheading='ONE WEEK. ONE BIG OPPORTUNITY.')
        background=VX_BROADCAST_RENDERER.image(shell,_vx_bg_metadata(contract),1920,1080)
        fallback = FreeHitIntroMotion(scene,contract,width,height,VX_PLAYER_MEDIA,VX_CLUB_MEDIA,VX_BROADCAST_ROOT/'assets',background)
        return VX_FREEHIT_MATCHDAY.RemotionFreeHitIntroVisual(fallback,scene,contract,VX_FREEHIT_MATCHDAY_PROJECT,VX_PLAYER_MEDIA,background)
    if contract.get('chip')=='freehit' and kind in ('fh_squad_plan','fh_captain_final'):
        if any(cue.get('start') is None or cue.get('end') is None for cue in scene['narration_timeline']):
            raise ValueError('Measure the Free Hit narration before rendering slides 3–4')
        scene['animation_version']=VX_FREEHIT_SUITE.VERSION
        renderer=(VX_FREEHIT_SUITE.RemotionFreeHitSquadVisual if kind=='fh_squad_plan'
                  else VX_FREEHIT_SUITE.RemotionFreeHitCaptaincyVisual)
        return renderer(scene,contract,width,height,VX_FREEHIT_MATCHDAY_PROJECT,
                        VX_PLAYER_MEDIA,VX_CLUB_MEDIA,VX_BROADCAST_ROOT,VX_CARD_VIEWS,
                        _vx_bg_metadata(contract).get('deadline'))
    if kind in ('fh_blueprint','fixture_heatmap','fh_squad_plan','fh_captain_final'):
        from freehit_premium import FreeHitPremiumVisual
        approved_cards=(vx_approved_freehit_hero_cards(scene,contract)
                        if kind in ('fh_squad_plan','fh_captain_final') else None)
        visual = FreeHitPremiumVisual(scene,contract,width,height,VX_BROADCAST_RENDERER,
            _vx_bg_metadata(contract),VX_PLAYER_MEDIA,VX_CLUB_MEDIA,approved_cards=approved_cards)
        if kind == 'fh_squad_plan':
            visual.requires_motion = True
        return visual
    if kind=='wc_captaincy_final':
        from wildcard_final_slide import WildcardCaptaincyVisual
        # Original notebook Hero Cards, never generated substitutes or downloaded profile cards.
        captain_plan=scene['captaincy_plan']['captaincy_plan']
        unique={int(player['id']):player for week in captain_plan for player in week['candidates']+[week['captain']]}
        hero_case={
            'elite_players':[
                dict(player_id=pid,team_id=int(player['team_id']),club=player['club'],
                     player_name=player['spoken_name'],price=float(player['price']),
                     projected_points=float(player['projected_points']))
                for pid,player in unique.items()
            ],
            'rows':contract['fixture_case']['rows'],
            'season':contract.get('season'),
        }
        approved,proof=VX_ELITE_FDR.render_hero_assets(
            hero_case,VX_CARD_VIEWS,VX_PLAYER_MEDIA,VX_CLUB_MEDIA,
            VX_BROADCAST_ROOT,VX_MEDIA_WORK/'wildcard_captaincy_original_heroes')
        if set(map(int,approved))!=set(unique):
            raise RuntimeError('Wildcard captaincy original Hero Card validation failed')
        scene['hero_card_asset_proof']=proof
        scene['animation_version']='wc-captaincy-v4-original-hero-one-by-one'
        return WildcardCaptaincyVisual(
            scene,contract,width,height,VX_BROADCAST_RENDERER,
            _vx_bg_metadata(contract),VX_PLAYER_MEDIA,VX_CLUB_MEDIA,
            approved_heroes=approved)
    if kind=='wc_squad_plan' and scene.get('masterplan_case'):
        # Masterplan owns its gold heading; the broadcast shell must not draw a second title.
        shell=dict(scene,display_title=' ',title=scene['title'],subheading=' ')
        background=VX_BROADCAST_RENDERER.image(shell,_vx_bg_metadata(contract),1920,1080)
        return VX_MASTERPLAN.RemotionMasterplanVisual(scene,contract,width,height,VX_MASTERPLAN_PROJECT,VX_PLAYER_MEDIA,VX_CLUB_MEDIA,VX_BROADCAST_ROOT,VX_CARD_VIEWS,background)
    if kind=='wc_squad_plan':
        from wildcard_slide3 import WildcardSlide3Visual
        return WildcardSlide3Visual(scene,contract,width,height,VX_BROADCAST_RENDERER,_vx_bg_metadata(contract),VX_PLAYER_MEDIA,VX_CLUB_MEDIA)
    if kind=='wc_analysis' and scene.get('elite_fixture_case'):
        hero_images,hero_proof=VX_ELITE_FDR.render_hero_assets(scene['elite_fixture_case'],VX_CARD_VIEWS,VX_PLAYER_MEDIA,VX_CLUB_MEDIA,VX_BROADCAST_ROOT,VX_MEDIA_WORK/'elite_fdr_hero_cards')
        scene['hero_card_asset_proof']=hero_proof
        shell=dict(scene,display_title=' ',title=scene['title'])  # Slide 2 owns its premium headline below the broadcast header.
        background=VX_BROADCAST_RENDERER.image(shell,_vx_bg_metadata(contract),1920,1080)
        return VX_ELITE_FDR.EliteFDRVisual(scene,contract,width,height,background,hero_images,VX_CLUB_MEDIA)
    if kind in ('wc_analysis','fixture_heatmap') and scene['fixture_case'].get('analyst_version'):
        from wildcard_fixture_analysis import WildcardFixtureVisual
        return WildcardFixtureVisual(scene,contract,width,height,VX_BROADCAST_RENDERER,_vx_bg_metadata(contract),media=VX_WC_PLAYER_MEDIA,badges=VX_CLUB_MEDIA)
    if kind in ('wc_analysis','fixture_heatmap'):
        from fixture_visual import FixtureSlideVisual
        return FixtureSlideVisual(scene,contract,width,height,VXWV.WildcardVisual,
            VX_BROADCAST_RENDERER,_vx_bg_metadata(contract),VX_ASSET_PATHS,VX_PLAYER_MEDIA)
    if kind=='wc_blueprint':return WildcardBlueprintVisual(scene,contract,width,height)
    raise ValueError('Unsupported slide; each chip has exactly four content slides: '+str(kind))


# Load the speech source restored above, including in an already-running kernel.
# Cell 2 and the recovery bundle carry the same official-web-name normalizer.
import tts_narration as _vx_tts_module
_vx_tts_module = importlib.reload(_vx_tts_module)
if _vx_tts_module.VERSION != 'spoken-fpl-v7-audio-exact-web-name':
    raise RuntimeError('Speech runtime version mismatch: rerun Cell 2, then Cell 14.')
print('✅ Speech runtime synchronized: '+_vx_tts_module.VERSION, flush=True)
# Spoken football XI means eleven; keep displayed narration and validation intact.
from tts_narration import TERMS as _VX_TTS_TERMS
_VX_TTS_TERMS['XI'] = 'eleven'
from tts_narration import prepare_tts, PROFILE as VX_FIXED_TTS_PROFILE, VERSION as VX_TTS_TEXT_VERSION

def vx_prepare_tts(segment):
    scene=vx_narration_scene(segment)
    ids=[]
    def collect(value):
        if isinstance(value,dict):
            for key,item in value.items():
                if key in ('player_id','hero_player_id','element','id') and str(item).isdigit():
                    ids.append(int(item))
                elif key in ('player_ids', 'comparison_ids', 'candidate_ids') and isinstance(item,(list,tuple)):
                    ids.extend(int(player_id) for player_id in item if str(player_id).isdigit())
                elif isinstance(item,(dict,list,tuple)):
                    collect(item)
        elif isinstance(value,(list,tuple)):
            for item in value:collect(item)
    cue=segment.get('narration_cue')
    if cue is not None:
        collect(cue)
    else:
        collect(scene.get('players',[]))
        collect(scene.get('narration_timeline',[]))
    ids=list(dict.fromkeys(ids))
    display_text=segment.get('display_text',segment['text'])
    _tts_bootstrap=globals().get('_bootstrap',{})
    audit=prepare_tts(segment['text'],_tts_bootstrap,ids)
    directory=VX_ANALYSIS_ROOT/'pronunciation_audits';directory.mkdir(parents=True,exist_ok=True)
    key=hashlib.sha256((str(segment.get('id',''))+audit['original_narration']+VX_TTS_TEXT_VERSION).encode()).hexdigest()[:24]
    (directory/(key+'.json')).write_text(json.dumps(dict(segment_id=segment.get('id'),**audit),indent=2,ensure_ascii=False),encoding='utf-8')
    if not audit['validation_passed']:
        raise ValueError('Narration blocked before Edge TTS in '+str(segment.get('id', '?'))+' / '+str((cue or {}).get('segment_id', 'whole slide'))+': '+', '.join(audit['unresolved_tokens']))
    return dict(segment,display_text=display_text,tts_text=audit['tts_cleaned_narration'])

def vx_limit_narration_pauses(raw):
    """Shorten only silence; preserve Ryan's spoken rate and pitch."""
    revision='max-one-second-pauses-v2'
    digest=hashlib.sha256((file_sha256(raw)+revision).encode()).hexdigest()[:24]
    directory=VX_MEDIA_WORK/'audio_pause_limited';directory.mkdir(exist_ok=True)
    target=directory/(digest+'.wav')
    if vx_valid_media_cache(target):return target
    temporary=target.with_suffix('.part.wav')
    trim='silenceremove=start_periods=1:start_duration=0:start_threshold=-50dB'
    # Long interior pauses keep a short breath; leading/trailing silence is removed.
    filters=trim+',silenceremove=stop_periods=-1:stop_duration=0.5:stop_threshold=-50dB:stop_silence=0.25,areverse,'+trim+',areverse'
    run_ffmpeg(['-i',raw,'-af',filters,'-ar','48000','-ac','2',temporary])
    check=subprocess.run([ffmpeg_binary(),'-hide_banner','-nostats','-i',str(temporary),'-af','silencedetect=noise=-50dB:d=1','-f','null','-'],capture_output=True,text=True)
    if check.returncode or any(float(value)>1.0 for value in re.findall(r'silence_duration:\s*([0-9.]+)',check.stderr)):
        temporary.unlink(missing_ok=True)
        raise RuntimeError('Narration pause limit check failed; no movie generated from this audio')
    temporary.replace(target);return target

def vx_narration_scene(segment):
    if segment.get('scene') is not None:return segment['scene']
    return next((scene for scenes in VX_SCENE_MANIFESTS.values() for scene in scenes if scene['scene_id']==segment.get('id')), {})

# Copy the effective Day 3 GitHub runner profile; keep other narration unchanged.
VX_INTRO_TTS_PROFILE = {"rate": "+16%", "pitch": "+4Hz", "volume": "+0%"}
VX_INTRO_TTS_PROFILE_SOURCE = "https://github.com/mukundacity-glitch/FPL-VORTEX-AUTO/blob/ae05001643e90c4172cf9bdb18f53ee19af87585/day3/run_notebook.py"

def vx_narration_profile(segment):
    if vx_narration_scene(segment).get('visual_type') in ('wc_blueprint','fh_blueprint','wc_analysis','fixture_heatmap','wc_squad_plan'):
        return dict(VX_INTRO_TTS_PROFILE)
    return dict(VX_FIXED_TTS_PROFILE)

async def _vx_wc_audio_for_segment(segment):
    spoken=dict(segment)
    spoken['voice_profile']=vx_narration_profile(spoken)
    raw=await _vx_wc_raw_audio_for_segment(spoken)
    if vx_narration_scene(spoken).get('visual_type')in ('wc_blueprint','wc_analysis','fixture_heatmap','wc_squad_plan'):
        return raw  # Preserve the established Ryan pacing and natural phrase pauses.
    return await asyncio.to_thread(vx_limit_narration_pauses,raw)


async def _vx_wc_raw_audio_for_segment(segment):
    segment=vx_prepare_tts(segment)
    profile=vx_narration_profile(segment)
    # Finish downloads atomically so interrupted speech never becomes valid cache.
    if VX_WC_TTS_PROVIDER!='Microsoft speech (sends narration text)' or VX_TTS_VOICE!='en-GB-RyanNeural':
        raise RuntimeError('Ryan-only voice policy: Microsoft Edge en-GB-RyanNeural required, no eSpeak fallback.')
    digest=hashlib.sha256(json.dumps([segment['tts_text'],VX_TTS_VOICE,profile,VXWC.VERSION,VX_TTS_TEXT_VERSION],sort_keys=True).encode()).hexdigest()[:24]
    directory=VX_MEDIA_WORK/'audio_verified_wc';directory.mkdir(exist_ok=True)
    path=directory/(digest+'.mp3')
    word_sync = vx_narration_scene(segment).get('visual_type') == 'wc_blueprint'
    word_sidecar = path.with_suffix('.words.json')
    if vx_valid_media_cache(path) and (not word_sync or word_sidecar.is_file()):return path
    # Honour a configured trusted CA bundle without weakening TLS verification.
    ca=os.environ.get('SSL_CERT_FILE')
    if ca and Path(ca).exists():
        from edge_tts import communicate as edge_communicate
        context=getattr(edge_communicate,'_SSL_CTX',None)
        if context is not None:context.load_verify_locations(cafile=ca)
    error=None
    for attempt in range(4):
        temp=path.with_suffix('.part.mp3')
        try:
            comm=edge_tts.Communicate(segment['tts_text'],VX_TTS_VOICE,rate=profile['rate'],pitch=profile['pitch'],volume=profile['volume'],proxy=os.environ.get('HTTPS_PROXY'))
            if word_sync:
                async def _stream_ryan_words():
                    boundaries=[]
                    with temp.open('wb') as voice_file:
                        async for event in comm.stream():
                            if event.get('type') == 'audio':
                                voice_file.write(event['data'])
                            elif event.get('type') == 'WordBoundary':
                                boundaries.append({'text': str(event.get('text', '')),
                                                   'offset_ticks': int(event.get('offset', 0))})
                    return boundaries
                boundaries = await asyncio.wait_for(_stream_ryan_words(),timeout=60)
            else:
                await asyncio.wait_for(comm.save(str(temp)),timeout=60)
            if media_duration(temp)<=0:
                raise RuntimeError('TTS audio has no duration')
            if word_sync:
                word_temporary = word_sidecar.with_suffix('.part.words.json')
                word_temporary.write_text(json.dumps(boundaries),encoding='utf-8')
                word_temporary.replace(word_sidecar)
            temp.replace(path)
            return path
        except Exception as exc:
            temp.unlink(missing_ok=True)
            error=exc
            if attempt < 3:
                await asyncio.sleep(1.5 * (attempt + 1))
    raise RuntimeError('Current narration audio unavailable after 4 attempts: '+type(error).__name__) from error

def vx_concat_timebase_parts(parts, work, profile):
    """Losslessly align MP4 video timestamps before concat stream-copy."""
    from fractions import Fraction
    timescale = int(profile['fps']) * 2048
    aligned = []
    for source in map(Path, parts):
        info = probe_media_info(source)
        video = next(s for s in info['streams'] if s.get('codec_type') == 'video')
        if Fraction(video['time_base']) == Fraction(1, timescale):
            aligned.append(source)
            continue
        signature = hashlib.sha256((file_sha256(source) + str(timescale) + 'concat-timebase-v1').encode()).hexdigest()[:24]
        target = Path(work) / ('concat_timebase_' + signature + '.mp4')
        if not vx_valid_media_cache(target):
            temporary = target.with_suffix('.partial.mp4')
            run_ffmpeg(['-i', source, '-map', '0:v:0', '-map', '0:a:0',
                        '-c', 'copy', '-video_track_timescale', timescale,
                        '-movflags', '+faststart', temporary])
            if abs(media_duration(temporary) - media_duration(source)) > 0.5:
                raise RuntimeError('Timestamp normalization changed the source duration: ' + str(source))
            temporary.replace(target)
        aligned.append(target)
    return aligned


def vx_verify_concat_duration(target, parts):
    """Reject stretched timelines and long gaps caused by incompatible concat inputs."""
    expected = sum(media_duration(Path(part)) for part in parts)
    actual = media_duration(Path(target))
    tolerance = max(1.0, 0.25 * len(parts))
    if abs(actual - expected) > tolerance:
        raise RuntimeError(f'Combined video duration mismatch: {actual:.2f}s versus {expected:.2f}s from its input clips')
    info = probe_media_info(target)
    for stream in info.get('streams', []):
        if stream.get('codec_type') in ('video', 'audio') and stream.get('duration') is not None:
            if abs(float(stream['duration']) - expected) > tolerance:
                raise RuntimeError('Combined ' + stream['codec_type'] + ' track duration does not match its input clips')
    return actual


def _vx_wc_balanced_concat(parts,work,key,profile):
    if not parts:raise ValueError('No scene clips to concatenate')
    if not profile.get('scene_crossfades',True):
        parts = vx_concat_timebase_parts(parts, work, profile)
        signature=hashlib.sha256((''.join(file_sha256(p) for p in parts)+json.dumps(profile,sort_keys=True)).encode()).hexdigest()[:18]
        joined=work/f'{key}_draft_concat_{signature}.mp4'
        if vx_valid_media_cache(joined):
            vx_verify_concat_duration(joined, parts)
            return joined
        playlist=work/f'{key}_draft_concat_{signature}.txt'
        playlist.write_text('\n'.join("file '"+Path(p).name.replace("'","'\\''")+"'" for p in parts))
        run_ffmpeg(['-f','concat','-safe','0','-i',playlist,'-c','copy','-movflags','+faststart',joined])
        if not vx_valid_media_cache(joined):raise RuntimeError('Draft concat did not produce valid media')
        vx_verify_concat_duration(joined, parts)
        print(f'{key}: draft fast concat complete ({len(parts)} scenes, no crossfade re-encodes)',flush=True)
        return joined
    from concurrent.futures import ThreadPoolExecutor
    layer=list(parts);level=0
    # Two independent joins keep the working set bounded while avoiding idle cores.
    with ThreadPoolExecutor(max_workers=2) as executor:
        while len(layer)>1:
            def join_pair(i):
                if i+1==len(layer):return layer[i]
                for attempt in range(2):
                    joined=vx_concat_scenes(layer[i:i+2],work,key+f'_tree{level}_{i}',profile)
                    if vx_valid_media_cache(joined):return joined
                raise RuntimeError('Crossfade could not produce valid media')
            layer=list(executor.map(join_pair,range(0,len(layer),2)));level+=1
            print(f'{key}: crossfade stage {level}, {len(layer)} clips remain',flush=True)
    return layer[0]

def vx_draft_motion_filter(profile,duration,motion_index=0):
    """Make a stable, subtle Ken Burns move on the cached 480p still."""
    if not profile.get('light_motion',False):
        return f"setsar={profile['sar']}:max=10000"
    seconds=float(duration)
    if not math.isfinite(seconds) or seconds<=0:
        raise ValueError('Draft motion duration must be positive and finite')
    fps=int(profile['fps'])
    if fps<=0:
        raise ValueError('Draft motion FPS must be positive')
    frames=max(1,math.ceil(seconds*fps))
    left_to_right=int(motion_index)%2==0
    x_start,x_end=(0.24,0.76) if left_to_right else (0.76,0.24)
    increment=0.04/frames
    return (
        f"zoompan=z='min(zoom+{increment:.12f},1.04)'"
        f":x='(iw-iw/zoom)*({x_start:.2f}+({x_end-x_start:.2f})*on/{frames})'"
        f":y='(ih-ih/zoom)*(0.44+0.12*on/{frames})'"
        f":d=1:s={int(profile['width'])}x{int(profile['height'])}:fps={fps},"
        f"setsar={profile['sar']}:max=10000"
    )


def vx_fast_draft_from_png(png_path,audio,duration,target,profile,motion_index=0):
    """Encode a narrated, subtly animated Draft scene from one cached 4K still."""
    png_path=Path(png_path)
    if not png_path.is_file():
        raise FileNotFoundError(f'Draft requires its 4K scene still: {png_path}')
    target=Path(target)
    temporary=target.with_name(target.stem+'.partial.mp4')
    # Never rescale the original 4K asset for every output frame.
    draft_stills=VX_MEDIA_WORK/'draft_stills'
    draft_stills.mkdir(parents=True,exist_ok=True)
    image_digest=file_sha256(png_path)[:20]
    preview=draft_stills/f"{image_digest}_{profile['width']}x{profile['height']}.png"
    if not preview.is_file():
        with Image.open(png_path) as image:
            resized=image.convert('RGB').resize((profile['width'],profile['height']),Image.Resampling.BICUBIC)
            temp_png=preview.with_suffix('.part.png')
            resized.save(temp_png,format='PNG',compress_level=1)
            temp_png.replace(preview)
    video_filter=vx_draft_motion_filter(profile,duration,motion_index)
    try:
        run_ffmpeg(['-loop','1','-framerate',profile['fps'],'-i',preview,
                    '-i',audio,'-t',duration,'-vf',video_filter,
                    '-c:v','libx264','-preset',profile['preset'],'-crf',str(profile['crf']),
                    '-pix_fmt','yuv420p','-threads','2',
                    '-r',profile['fps'],'-fps_mode','cfr',
                    '-video_track_timescale',str(profile['fps']*2048),
                    '-aspect','16:9','-af','apad','-c:a','aac','-ar','48000','-ac','2',
                    '-movflags','+faststart',temporary])
        if not vx_valid_media_cache(temporary):
            raise RuntimeError('Draft slide video was not encoded correctly: '+str(temporary))
        temporary.replace(target)
    finally:
        temporary.unlink(missing_ok=True)
    return target

# VX_FAST_DRAFT_CUE_FRAMES_V1
def vx_draft_sections(scene, duration, fps):
    """Quantize measured cue boundaries once, without accumulating audio drift."""
    total = max(1, math.ceil(duration * fps))
    timeline = scene.get('narration_timeline') or []
    boundaries = {0, total}
    for cue in timeline:
        for field in ('start', 'end'):
            value = cue.get(field)
            if isinstance(value, (int, float)) and math.isfinite(value):
                boundaries.add(max(0, min(total, round(value * fps))))
    boundaries = sorted(boundaries)
    result = []
    for first, last in zip(boundaries, boundaries[1:]):
        if last <= first:
            continue
        start = first / fps
        cue = next((c for c in timeline if c.get('start', duration + 1) <= start + .5 / fps < c.get('end', -1)), None)
        sample = min(duration - .001, last / fps - .001, start + 2.0)
        if cue is not None:
            sample = min(sample, float(cue['end']) - .001)
        result.append(dict(first=first, frames=last-first, sample=max(0., sample)))
    return result


def vx_draft_remotion_frames(visual, duration, profile, sections, root):
    """Bundle once and reuse one browser for the small set of narrated-section stills."""
    project = Path(visual.project)
    scene = visual.scene
    if scene.get('masterplan_case'):
        visual.prepare(None, duration, dict(profile, fps=24))
        composition = 'WildcardMasterplan'
    elif scene.get('matchday_case'):
        VX_FREEHIT_MATCHDAY.prepare_project(scene, project, visual.media, visual.badges,
            visual.brand_root, None, profile['width'], profile['height'], 24, duration)
        composition = 'FreeHitMatchday'
    else:
        raise RuntimeError('Unrecognized Remotion Draft scene')
    cli = VX_MASTERPLAN._vx_remotion_cli(project)
    browser_args = VX_MASTERPLAN._vx_remotion_browser_arguments(project)
    options = {}
    for arg in browser_args:
        if arg.startswith('--browser-executable='):
            options['browserExecutable'] = arg.split('=', 1)[1]
        if arg.startswith('--chrome-mode='):
            options['chromeMode'] = arg.split('=', 1)[1]
    shots = [dict(frame=min(math.ceil(duration*24)-1, round(row['sample']*24)),
                  output=str((root / f'{i:03d}.png').resolve())) for i, row in enumerate(sections)]
    request = root / 'remotion_keyframes.json'
    request.write_text(json.dumps(dict(composition=composition, options=options, shots=shots)))
    worker = project / 'vx_draft_keyframes.mjs'
    worker.write_text(_VX_DRAFT_KEYFRAMES_JS, encoding='utf-8')
    VX_MASTERPLAN._vx_remotion_command_output(project, 'draft_keyframes',
        ['node', str(worker.resolve()), str(request.resolve())])


def vx_render_fast_draft_scene(visual, audio, duration, target, profile):
    scene = getattr(visual, 'manifest', None) or getattr(visual, 'scene', {})
    fps = int(profile['fps'])
    sections = vx_draft_sections(scene, duration, fps)
    root = target.parent / (target.stem + '_draft_cues_v1')
    root.mkdir(parents=True, exist_ok=True)
    print(f"Fast Draft: {len(sections)} narration-section frames instead of {math.ceil(duration*fps)} browser frames; animated pan/zoom", flush=True)
    if hasattr(visual, 'project') and (scene.get('masterplan_case') or scene.get('matchday_case')):
        vx_draft_remotion_frames(visual, duration, profile, sections, root)
    clips = []
    for index, row in enumerate(sections):
        png = root / f'{index:03d}.png'
        if not png.is_file():
            image = visual.draw(row['sample'], duration).convert('RGB')
            if image.size != (profile['width'], profile['height']):
                raise RuntimeError('Draft narration-section dimensions do not match the profile')
            image.save(png, format='PNG', compress_level=1)
        clip = root / f'{index:03d}.mp4'
        if not clip.is_file():
            partial = root / f'{index:03d}.partial.mp4'
            run_ffmpeg(['-loop', '1', '-framerate', fps, '-i', png,
                '-frames:v', row['frames'], '-vf', vx_draft_motion_filter(dict(profile, light_motion=True), row['frames']/fps, index),
                '-an', '-c:v', 'libx264', '-preset', profile['preset'], '-crf', profile['crf'],
                '-pix_fmt', 'yuv420p', '-threads', '2', '-r', fps, '-fps_mode', 'cfr',
                '-video_track_timescale', fps*2048, '-aspect', '16:9', partial])
            partial.replace(clip)
        clips.append(clip)
        print(f"Fast Draft section {index+1}/{len(sections)} ready", flush=True)
    playlist = root / 'sections.txt'
    playlist.write_text('\n'.join("file '" + path.name + "'" for path in clips))
    partial = target.with_name(target.stem + '.partial.mp4')
    try:
        run_ffmpeg(['-f', 'concat', '-safe', '0', '-i', playlist, '-i', audio,
            '-map', '0:v:0', '-map', '1:a:0', '-c:v', 'copy', '-af', 'apad',
            '-c:a', 'aac', '-ar', '48000', '-ac', '2', '-t', duration,
            '-movflags', '+faststart', partial])
        vx_verify_delivery_mp4(partial, profile)
        if abs(media_duration(partial) - duration) > max(.3, 1/fps):
            raise RuntimeError('Fast Draft duration differs from measured narration')
        partial.replace(target)
    finally:
        partial.unlink(missing_ok=True)
    return target


_VX_DRAFT_KEYFRAMES_JS = r'''
import fs from 'node:fs';
import path from 'node:path';
import {bundle} from '@remotion/bundler';
import {openBrowser, selectComposition, renderStill} from '@remotion/renderer';
const request = JSON.parse(fs.readFileSync(process.argv[2], 'utf8'));
const missing = request.shots.filter(shot => !fs.existsSync(shot.output));
if (missing.length) {
  const serveUrl = await bundle({entryPoint: path.resolve('src/index.ts'), publicDir: path.resolve('public')});
  const browser = await openBrowser('chrome', request.options);
  try {
    const inputProps = {mute: true};
    const composition = await selectComposition({serveUrl, id: request.composition,
      inputProps, puppeteerInstance: browser, ...request.options});
    for (const [index, shot] of missing.entries()) {
      const temporary = shot.output + '.part.png';
      await renderStill({serveUrl, composition, inputProps, frame: shot.frame,
        output: temporary, imageFormat: 'png', puppeteerInstance: browser,
        timeoutInMilliseconds: 60000, ...request.options});
      fs.renameSync(temporary, shot.output);
      console.log(`Draft keyframe ${index + 1}/${missing.length} ready`);
    }
  } finally {
    await browser.close({silent: true});
  }
}
'''


def _vx_wc_native_body_segment(visual,audio,duration,target,profile):
    # Free Hit slides 1–2 retain their true counters and protected frame edges in Draft.
    if getattr(visual, 'preserve_frame_edges', False):
        return native_body_segment(visual,audio,duration,target,profile)
    if profile.get('draft_keyframes') and not getattr(visual, 'requires_motion', False):
        return vx_render_fast_draft_scene(visual,audio,duration,target,profile)
    temp=target.with_name(target.stem+'.partial.mp4')
    frame_path=target.with_name(target.stem+'.draft_frame.png')
    try:
        if profile.get('static_preview') and not getattr(visual,'requires_motion',False):
            # Draft is for rapid review: encode one fully-rendered slide per scene
            # instead of regenerating every Python frame. Final keeps full motion.
            frame=visual.complete.convert('RGB')
            if frame.size!=(profile['width'],profile['height']):
                raise RuntimeError(f'Draft frame size mismatch: {frame.size}')
            frame_tmp=frame_path.with_suffix('.part.png')
            frame.save(frame_tmp,format='PNG',compress_level=1)
            frame_tmp.replace(frame_path)
            run_ffmpeg([
                '-loop','1','-framerate',profile['fps'],'-i',frame_path,'-i',audio,
                '-t',duration,'-vf','setsar='+profile['sar']+':max=10000',
                '-c:v','libx264','-preset',profile['preset'],'-crf',profile['crf'],
                '-tune','stillimage','-pix_fmt','yuv420p','-threads','0',
                '-r',profile['fps'],'-fps_mode','cfr','-video_track_timescale',profile['fps']*2048,
                '-aspect','16:9','-af','apad','-c:a','aac','-ar','48000','-ac','2',
                '-movflags','+faststart',temp
            ])
        else:
            native_body_segment(visual,audio,duration,temp,profile)
        if media_duration(temp)<=0:raise RuntimeError('Encoded scene has no duration')
        temp.replace(target)
    finally:
        temp.unlink(missing_ok=True)
        if not profile.get('static_preview'):frame_path.unlink(missing_ok=True)

def _vx_wc_assemble_branded(body_parts,assets,target,work,profile,include_opening=True):
    stamp=work/(target.stem+'_assembly_cache.json')
    required_assets=(['opening.mp4'] if include_opening else [])+['outro.mp4']+(['popup.mp4'] if VX_INCLUDE_POPUPS else [])
    missing=[name for name in required_assets if name not in assets]
    if missing:
        raise FileNotFoundError('Required branded video assets are missing: '+', '.join(missing))
    assembly_signature=hashlib.sha256(json.dumps({'body':[file_sha256(p) for p in body_parts],
        'assets':[file_sha256(assets[name]) for name in required_assets],
        'profile':profile,'popups':VX_INCLUDE_POPUPS,'external_intro':include_opening,
        'assembly':'chip-specific-first-frame-v4-timebase'},sort_keys=True).encode()).hexdigest()
    if stamp.exists() and target.exists():
        try:
            saved=json.loads(stamp.read_text())
            if saved.get('signature')==assembly_signature and saved.get('output_sha256')==file_sha256(target) and vx_valid_media_cache(target):
                expected = sum(media_duration(p) for p in body_parts) + float(saved['audit'].get('opening_duration', 0)) + float(saved['audit'].get('outro_duration', 0))
                if abs(media_duration(target) - expected) > 1.0:
                    raise RuntimeError('Cached combined movie duration is invalid')
                print(f'{target.stem}: reused the completed, verified movie',flush=True)
                return saved['audit']
        except (ValueError,OSError,RuntimeError):pass
    print(f'{target.stem}: assembling and normalizing the full movie ({profile["fps"]} fps)',flush=True)
    suffix=f"{profile['width']}_{profile['height']}_{profile['fps']}_{profile.get('encoder','libx264')}_storyv4";normalized=[]
    for name in (['opening.mp4'] if include_opening else [])+['outro.mp4']:
        signature=hashlib.sha256(assets[name].read_bytes()).hexdigest()[:12]
        path=work/(name.replace('.mp4','')+'_'+suffix+'_'+signature+'.mp4')
        if not vx_valid_media_cache(path):normalize_asset(assets[name],path,profile)
        normalized.append(path)
    opening=normalized[0] if include_opening else None
    outro=normalized[-1]
    opening_duration=media_duration(opening) if opening is not None else 0.0
    body_duration=sum(media_duration(p) for p in body_parts)
    playlist=work/(target.stem+'.concat.txt')
    parts=([opening] if opening is not None else [])+list(body_parts)+[outro]
    parts = vx_concat_timebase_parts(parts, work, profile)
    playlist.write_text('\n'.join("file '"+p.name.replace("'","'\\''")+"'" for p in parts))
    joined=work/(target.stem+'_joined.mp4');run_ffmpeg(['-f','concat','-safe','0','-i',playlist,'-c','copy','-movflags','+faststart',joined])
    vx_verify_concat_duration(joined, parts)
    starts=popup_times(opening_duration,body_duration) if VX_INCLUDE_POPUPS else [];staged=work/(target.stem+'_finished.mp4')
    if starts:
        popup_duration=media_duration(assets['popup.mp4']);width=2*round(profile['width']*.80/2);margin=round(profile['height']*.02);filters=[f"[1:v]scale={width}:-2,setsar=1,split={len(starts)}"+''.join(f'[p{i}]' for i in range(len(starts)))];previous='0:v'
        for i,start in enumerate(starts):
            stop=min(start+popup_duration,opening_duration+body_duration);filters.append(f'[p{i}]trim=duration={stop-start},setpts=PTS-STARTPTS+{start}/TB[o{i}]');filters.append(f"[{previous}][o{i}]overlay=x=(main_w-overlay_w)/2:y=main_h-overlay_h-{margin}:eof_action=pass:enable='between(t,{start},{stop})'[v{i}]");previous=f'v{i}'
        filters.append(f'[{previous}]fps={profile["fps"]},setsar={profile["sar"]}:max=10000[out]')
        run_ffmpeg(['-i',joined,'-i',assets['popup.mp4'],'-filter_complex',';'.join(filters),'-map','[out]','-map','0:a','-c:v','libx264','-crf',profile['crf'],'-preset',profile['preset'],'-pix_fmt','yuv420p','-r',profile['fps'],'-fps_mode','cfr','-video_track_timescale',profile['fps']*2048,'-aspect','16:9','-threads','4','-c:a','aac','-ar','48000','-ac','2','-movflags','+faststart',staged])
    elif profile.get('static_preview') or (VX_VIDEO_MODE=='Draft' and not starts):
        shutil.copy2(joined,staged)
    else:run_ffmpeg(['-i',joined,'-vf',f"fps={profile['fps']},setsar={profile['sar']}:max=10000",'-c:v','libx264','-crf',profile['crf'],'-preset',profile['preset'],'-pix_fmt','yuv420p','-r',profile['fps'],'-fps_mode','cfr','-video_track_timescale',profile['fps']*2048,'-aspect','16:9','-threads','4','-c:a','aac','-ar','48000','-ac','2','-movflags','+faststart',staged])
    vx_verify_concat_duration(staged, parts)
    target.parent.mkdir(parents=True,exist_ok=True);target_tmp=target.with_name(target.name+'.tmp');shutil.copy2(staged,target_tmp);target_tmp.replace(target);staged.unlink(missing_ok=True)
    audit={'opening_duration':opening_duration,'body_duration':body_duration,'outro_duration':media_duration(outro),
           'popup_times':starts,'profile':profile,'path':str(target),'external_intro':include_opening}
    temporary=stamp.with_suffix('.part.json');temporary.write_text(json.dumps({'signature':assembly_signature,'output_sha256':file_sha256(target),'audit':audit}));temporary.replace(stamp)
    return audit



def _vx_vtt_timestamp(seconds):
    milliseconds=max(0,int(round(float(seconds)*1000)))
    hours,milliseconds=divmod(milliseconds,3600000);minutes,milliseconds=divmod(milliseconds,60000);secs,milliseconds=divmod(milliseconds,1000)
    return f"{hours:02}:{minutes:02}:{secs:02}.{milliseconds:03}"

def _vx_render_verified_wildcard(plan,key,label,assets):
    script_text=VX_NARRATION_PATHS[key].read_text(encoding='utf-8')
    script=decode_script(script_text)
    narration_object=VX_NARRATION_OBJECTS.get(key)
    if narration_object is None:
        raise RuntimeError('Narration blocked: Wildcard Scene JSON validation failed.\nResolve the listed data inconsistencies before generating video assets.\nMissing validated narration object')
    VXWC.validate_render_script(narration_object,script)
    section_by_slide={section['slide_id']:section for section in narration_object['sections']}
    contract_path=VX_ANALYSIS_ROOT/('wildcard_data.json' if key.startswith('wildcard') else 'freehit_data.json')
    contract=json.loads(contract_path.read_text());vx_validate_contract(contract)
    if '# DATA_HASH: '+contract['data_hash'] not in script_text:
        raise ValueError('Narration refers to older facts; rerun the verified visual/narration preparation')
    vx_assert_current_snapshot(contract)
    if contract['chip']=='freehit':vx_validate_freehit(contract)
    manifest_path=VX_ANALYSIS_ROOT/(key+'_scene_manifest.json')
    # Rebuild visuals from persisted data, then apply only explicit saved narration edits.
    scenes=vx_build_manifest(contract)
    if [s['id'] for s in script]!=[s['scene_id'] for s in scenes]:raise ValueError('TXT scene headers must match the selected chip manifest scenes. Regenerate the script or restore the headers.')
    body_parts=[];previous=None;start=0
    for index,(segment,scene) in enumerate(zip(script,scenes)):
        _vx_slide_started = time.monotonic()
        print(f'{key}: slide {index+1}/{len(scenes)} — {scene["title"]}', flush=True)
        scene['narration_text']=segment['text']
        # Validate edited sentences without altering the user's prose.
        plain=re.sub(r'(?<=\d)\.(?=\d)','DECIMAL',segment['text'])
        if contract['chip']!='wildcard' and any(len(s.split())>25 for s in re.split(r'[.!?]+(?:\s|$)',plain) if s.strip()):raise ValueError(f'{scene["title"]}: a TXT sentence exceeds twenty-five words. Please split it.')
        spoken=segment['text']
        section=section_by_slide[scene['scene_id']]
        pause_before=0.0;pause_after=0.0  # TTS pacing supplies breaths; no extra section pauses
        if scene['visual_type'] in ('wc_blueprint','wc_analysis','wc_squad_plan','wc_captaincy_final','fh_blueprint','fixture_heatmap','fh_squad_plan','fh_captain_final'):
            _vx_run_async(vx_prefetch_scene_ryan(scene, segment))
            from wildcard_plan_audio import prepare_segment_audio
            audio=(VX_FREEHIT_SUITE.prepare_squad_audio if scene["visual_type"]=="fh_squad_plan" else VX_MASTERPLAN.prepare_audio if scene.get("masterplan_case") else prepare_segment_audio)(scene,segment,VX_MEDIA_WORK,
                lambda item:_vx_run_async(_vx_wc_audio_for_segment(item)),
                lambda source,target:run_ffmpeg(['-i',source,'-ar','48000','-ac','2','-c:a','pcm_s16le',target]))
        else:
            audio=_vx_run_async(_vx_wc_audio_for_segment(dict(segment,text=spoken,scene=scene)))
        if scene['visual_type']=='wc_blueprint':
            _slide1_export={'scene_id':scene['scene_id'],'display_text':scene['narration_text'],
                'narration_timeline':scene['narration_timeline'],'verdict':scene['wildcard_case']['verdict'],
                'comparison_available':scene['wildcard_case']['scenario_comparison'] is not None,
                'alignment':'measured_segments','voice':VX_TTS_VOICE,'voice_settings':vx_narration_profile(segment)}
            (VX_ANALYSIS_ROOT/(key+'_slide1_script_and_cues.json')).write_text(json.dumps(_slide1_export,ensure_ascii=False,indent=2),encoding='utf-8')
        if scene['visual_type']=='wc_squad_plan':
            (VX_ANALYSIS_ROOT/(key+'_slide3_script_and_cues.json')).write_text(json.dumps({'display_text':scene['narration_text'],'narration_timeline':scene['narration_timeline'],'formation':scene['squad_plan']['formation'],'starting_xi_ids':[p['id'] for p in scene['squad_plan']['starting_xi']],'bench_ids':[p['id'] for p in scene['squad_plan']['bench']],'squad_cost':scene['squad_plan']['total_cost'],'bank':scene['squad_plan']['bank'],'alignment':'measured_segments'},ensure_ascii=False,indent=2),encoding='utf-8')
        original_seconds=media_duration(audio)
        word_count=len(vx_prepare_tts(dict(segment,scene=scene))['tts_text'].split());tempo=1.0
        paced=audio
        digest=file_sha256(audio)[:18]
        padded=VX_MEDIA_WORK/f'ryan_{digest}_pauses_{int(round(pause_before*1000))}_{int(round(pause_after*1000))}.wav'
        if not vx_valid_media_cache(padded):
            if pause_before or pause_after:
                delay_ms=int(round(pause_before*1000))
                filters=[f'adelay={delay_ms}|{delay_ms}']
                if pause_after:filters.append(f'apad=pad_dur={pause_after:.3f}')
                run_ffmpeg(['-i',paced,'-af',','.join(filters),'-ar','48000','-ac','2',padded])
            else:shutil.copy2(paced,padded)
        speech_seconds=media_duration(paced);duration=media_duration(padded)+(2.5 if scene['visual_type'] in ('wc_blueprint','wc_analysis','wc_squad_plan','wc_captaincy_final') else .65)
        speech_start=start+pause_before;speech_end=speech_start+speech_seconds
        scene.update(duration_seconds=duration,audio_duration_seconds=media_duration(padded),narration_word_count=word_count,narration_speed=tempo,start_seconds=start,
                     pause_before_seconds=pause_before,pause_after_seconds=pause_after,speech_start_seconds=speech_start,speech_end_seconds=speech_end)
        scene['timing']['status']='measured';scene['timing']['voice_profile']=vx_narration_profile(dict(segment,scene=scene));scene['timing']['audio_padding_seconds']=0.0 if scene['visual_type']=='fh_blueprint' else 2.5;scene['timing']['audio_file']=str(padded)
        scene['timing']['row_highlight_alignment']='measured narration segments' if scene['timing'].get('player_alignment')=='measured_segments' else 'estimated from narration word position'
        if scene['visual_type'] in ('wc_blueprint','wc_analysis','wc_squad_plan','wc_captaincy_final','fh_blueprint','fixture_heatmap','fh_squad_plan','fh_captain_final'):
            from wildcard_plan import align_timeline
            _expected=' '.join(item['narration_text'] for item in scene['narration_timeline'])
            if ' '.join(_expected.split())!=' '.join(scene['narration_text'].split()):
                raise ValueError('Update the structured Slide 3 segments when editing its narration')
            if scene['timing'].get('player_alignment')!='measured_segments':
                scene['narration_timeline']=align_timeline(scene['narration_timeline'],speech_seconds)
        if scene['visual_type'] in ('wc_analysis','fixture_heatmap'):
            from fixture_slide import bind_scene_timeline
            _fixture_durations=([item['end']-item['start'] for item in scene['narration_timeline']] if scene['timing'].get('player_alignment')=='measured_segments' else None)
            bind_scene_timeline(scene,scene['audio_duration_seconds'],_fixture_durations)
            if scene['visual_type']=='wc_analysis':
                (VX_ANALYSIS_ROOT/(key+'_slide2_script_and_cues.json')).write_text(json.dumps({'display_text':scene['narration_text'],'narration_timeline':scene['narration_timeline'],'ranking':scene['fixture_case']['diagnostics']['ranking']},ensure_ascii=False,indent=2),encoding='utf-8')
        (VX_ANALYSIS_ROOT/(key+'_'+scene['scene_id']+'_script_and_cues.json')).write_text(json.dumps({'scene_id':scene['scene_id'],'display_text':scene['narration_text'],'narration_timeline':scene.get('narration_timeline',[]),'alignment':scene['timing'].get('player_alignment'),'data_hash':contract['data_hash']},ensure_ascii=False,indent=2),encoding='utf-8')
        if scene['visual_type'] == 'wc_blueprint':
            scene['animation_version'] = 'wc-slide1-ryan-cue-v6'
        if scene['visual_type'] == 'wc_captaincy_final':
            scene['animation_version'] = 'wc-captaincy-masterclass-original-heroes-v7'
        if scene['visual_type'] == 'wc_squad_plan' and scene.get('masterplan_case'):
            scene['animation_version'] = 'wc-masterplan-v6-single-heading-large-hero'
        _vx_draft_png = None
        visual=vx_visual_from_manifest(scene,contract,VX_VIDEO_PROFILE['width'],VX_VIDEO_PROFILE['height'])
        # Source-sensitive fingerprints prevent valid-but-stale videos after graphics edits.
        _vx_graphics_files = (
            sorted((VX_MASTERPLAN_PROJECT / 'src').glob('*.tsx')) if scene['visual_type'] == 'wc_squad_plan'
            else [Path(sys.modules['wildcard_final_slide'].__file__), VX_BROADCAST_ROOT / 'broadcast.html']
            if scene['visual_type'] == 'wc_captaincy_final' else []
        )
        _vx_graphics_sha = hashlib.sha256(b''.join(path.read_bytes() for path in _vx_graphics_files)).hexdigest()
        fingerprint=hashlib.sha256((json.dumps(scene,sort_keys=True)+json.dumps({k:v for k,v in contract.items() if k not in ('model_audit','data_sources')},sort_keys=True)+VX_STORY_VERSION+VX_RENDERER_VERSION+str((FONT_REGULAR,FONT_BOLD))+json.dumps(VX_VIDEO_PROFILE,sort_keys=True)+((file_sha256(_vx_draft_png)) if _vx_draft_png is not None else '')+hashlib.sha256(Path(VXWV.__file__).read_bytes()).hexdigest()+vx_render_asset_signature(assets)+json.dumps(VX_WC_PLAYER_MEDIA,sort_keys=True)+_vx_graphics_sha).encode()).hexdigest()[:18]
        part=VX_MEDIA_WORK/f'{key}_manifest_{index:02}_{fingerprint}.mp4'
        _vx_scene_reused = vx_valid_media_cache(part)
        if not _vx_scene_reused:
            if _vx_draft_png is not None:
                vx_fast_draft_from_png(_vx_draft_png,padded,duration,part,VX_VIDEO_PROFILE,motion_index=index)
            else:
                _vx_wc_native_body_segment(visual,padded,duration,part,VX_VIDEO_PROFILE)
        body_parts.append(part);start+=media_duration(part)-(.5 if VX_VIDEO_PROFILE.get('scene_crossfades',True) and index<len(scenes)-1 else 0)
        scene['encoded_duration_seconds']=media_duration(part)
        scene['cache_media_file']=str(part)
        scene['media_file']=str(vx_publish_scene_mp4(part,key,contract['gameweek'],index+1,scene['scene_id']))
        manifest_path.write_text(json.dumps(scenes,indent=2,ensure_ascii=False))
        vx_show_draft_slide_audit(scene, key, time.monotonic()-_vx_slide_started, _vx_scene_reused)
        print(f"{key} {scene['scene_id']}: {scene['title']} / {duration:.1f}s",flush=True)
    narration_seconds=sum(float(scene.get('audio_duration_seconds',0)) for scene in scenes)
    if narration_seconds<=0:raise RuntimeError('Narration has no measured audio duration')
    vtt_path=VX_ANALYSIS_ROOT/(key+'_narration.vtt')
    body=_vx_wc_balanced_concat(body_parts,VX_MEDIA_WORK,key,VX_VIDEO_PROFILE)
    path=VX_VIDEO_OUT_DIR/key/VX_VIDEO_MODE.lower()/f'{"Wildcard" if key.startswith("wildcard") else "Free Hit"} Combined {VX_VIDEO_MODE} GW{contract["gameweek"]}.mp4'
    audit=_vx_wc_assemble_branded([body],assets,path,VX_MEDIA_WORK,VX_VIDEO_PROFILE,include_opening=False)
    # VTT times are absolute against the assembled video, including the branded opening.
    opening_offset=float(audit.get('opening_duration',0) or 0);cues=['WEBVTT','']
    for cue_index,scene in enumerate(scenes,1):
        cues.extend([str(cue_index),f"{_vx_vtt_timestamp(opening_offset+scene['speech_start_seconds'])} --> {_vx_vtt_timestamp(opening_offset+scene['speech_end_seconds'])}",scene['narration_text'],''])
    vtt_path.write_text('\n'.join(cues),encoding='utf-8')
    audit.update(voice=VX_TTS_VOICE,narration_profile_mode=VX_NARRATION_PROFILE,narration_profiles=VX_NARRATION_PROFILES,story_version=VXWC.WILDCARD_NARRATION_VERSION,presentation_version=VX_PRESENTATION_VERSION,scenes=len(scenes),scene_manifest=str(manifest_path),data_contract=str(contract_path),narration_object=str(VX_ANALYSIS_ROOT/(key+'_narration.json')),vtt_timing=str(vtt_path),speech_rate=VX_RYAN_RATE,pitch_hz=VX_RYAN_PITCH_HZ,narration_duration_seconds=narration_seconds,transition=('0.5 second video/audio crossfade' if VX_VIDEO_PROFILE.get('scene_crossfades',True) else 'hard cuts'),total_duration=media_duration(path),template='7.png')
    (VX_ANALYSIS_ROOT/(key+'_media_audit.json')).write_text(json.dumps(audit,indent=2))
    print(f"Saved {path} — {audit['total_duration']/60:.1f} min, {len(scenes)} manifest scenes")
    return path


def vx_delivery_slide_name(key, scene_number):
    chip = 'Wildcard' if str(key).startswith('wildcard') else 'Free Hit'
    titles = ('Intro', 'Fixture Difficulty', 'Squad Plan', 'Captaincy')
    number = int(scene_number)
    if number not in (1, 2, 3, 4):
        raise ValueError(f'Unexpected slide number for {chip}: {number}')
    return f'Slide {number} {chip} {titles[number - 1]}'


def vx_save_slide_scripts(key, contract, scenes):
    """Keep complete editable script and export separately named slide TXT files."""
    out = VX_NARRATION_OUT_DIR / key / VX_VIDEO_MODE.lower()
    out.mkdir(parents=True, exist_ok=True)
    paths = []
    for index, scene in enumerate(scenes, 1):
        narration = str(scene.get('narration_text', '')).strip()
        if not narration:
            raise RuntimeError(f'{key}: narration empty on slide {index}')
        target = out / f'{vx_delivery_slide_name(key, index)}.txt'
        temp = target.with_suffix('.partial.txt')
        temp.write_text(narration + '\n', encoding='utf-8')
        temp.replace(target)
        paths.append(target)
    return paths



def vx_export_narration_mp3(source_video, output_path):
    """Cache a 160kb/s MP3 derived from the actual delivered, synchronized MP4 audio."""
    source_video=Path(source_video);output_path=Path(output_path)
    if not source_video.is_file() or not vx_valid_media_cache(source_video):
        raise RuntimeError(f'Cannot export narration from invalid video: {source_video}')
    output_path.parent.mkdir(parents=True,exist_ok=True)
    digest=file_sha256(source_video)
    stamp=output_path.with_suffix('.source.sha256')
    if output_path.is_file() and output_path.stat().st_size>1000 and stamp.is_file():
        if stamp.read_text(encoding='utf-8').strip()==digest and probe_media(output_path)[1]:
            return output_path
    temporary=output_path.with_suffix('.partial.mp3')
    try:
        run_ffmpeg(['-i',source_video,'-map','0:a:0','-vn','-c:a','libmp3lame',
                    '-b:a','160k','-ar','48000','-ac','2',temporary])
        if not temporary.is_file() or temporary.stat().st_size<1000 or not probe_media(temporary)[1]:
            raise RuntimeError(f'MP3 export failed: {output_path}')
        temporary.replace(output_path)
        stamp.write_text(digest,encoding='utf-8')
    finally:
        temporary.unlink(missing_ok=True)
    return output_path


def vx_save_combined_script(key, contract):
    source=Path(VX_NARRATION_PATHS[key])
    if not source.is_file() or not source.read_text(encoding='utf-8').strip():
        raise RuntimeError(f'{key}: combined narration source is empty')
    destination=VX_NARRATION_OUT_DIR/key/VX_VIDEO_MODE.lower()/f'{"Wildcard" if key.startswith("wildcard") else "Free Hit"} Combined GW{contract["gameweek"]}.txt'
    destination.parent.mkdir(parents=True,exist_ok=True)
    if not destination.is_file() or destination.read_bytes()!=source.read_bytes():
        temporary=destination.with_suffix('.partial.txt')
        temporary.write_bytes(source.read_bytes())
        temporary.replace(destination)
    return destination


def vx_verify_delivery_mp4(path,profile):
    from fractions import Fraction
    info=probe_media_info(path)
    video=next((s for s in info.get('streams',[]) if s.get('codec_type')=='video'),None)
    if video is None:raise RuntimeError(f'Missing video track in {path}')
    if (int(video.get('width',0)),int(video.get('height',0)))!=(profile['width'],profile['height']):
        raise RuntimeError(f'Unexpected frame size in {path}: {video.get("width")}x{video.get("height")}')
    rate=Fraction(str(video.get('r_frame_rate','0/1')))
    if rate!=profile['fps']:raise RuntimeError(f'Unexpected FPS in {path}: {rate}, expected {profile["fps"]}')
    aspect=Fraction(str(video.get('display_aspect_ratio') or '0/1').replace(':','/'))
    if not aspect:
        sar=Fraction(str(video.get('sample_aspect_ratio') or '1/1').replace(':','/'))
        aspect=Fraction(profile['width'],profile['height'])*sar
    if abs(float(aspect)-16/9)>0.003:
        raise RuntimeError(f'Expected 16:9 display aspect ratio in {path}, observed {aspect}')
    if not any(s.get('codec_type')=='audio' for s in info.get('streams',[])):
        raise RuntimeError(f'Missing audio track in {path}')
    return {'width':profile['width'],'height':profile['height'],'fps':profile['fps'],'aspect':str(aspect)}


def vx_publish_scene_mp4(source, key, gameweek, scene_number, scene_id):
    """Copy a verified scene to persistent Drive storage, not Colab's disposable /content cache."""
    source = Path(source)
    if not source.is_file() or not vx_valid_media_cache(source):
        raise RuntimeError(f'Cannot publish invalid scene MP4: {source}')
    safe_id = re.sub(r'[^A-Za-z0-9_-]+', '_', str(scene_id)).strip('_')
    if not safe_id:
        raise ValueError(f'Invalid scene ID: {scene_id!r}')
    destination_dir = VX_VIDEO_OUT_DIR / key / VX_VIDEO_MODE.lower() / 'scenes'
    destination_dir.mkdir(parents=True, exist_ok=True)
    destination = destination_dir / f'{vx_delivery_slide_name(key, scene_number)} {VX_VIDEO_MODE}.mp4'
    if destination.is_file() and destination.stat().st_size == source.stat().st_size:
        if file_sha256(destination) == file_sha256(source) and vx_valid_media_cache(destination):
            print(f'✅ Reused published scene: {destination}', flush=True)
            return destination
    temporary = destination.with_suffix('.partial.mp4')
    try:
        shutil.copy2(source, temporary)
        if temporary.stat().st_size != source.stat().st_size or not vx_valid_media_cache(temporary):
            raise RuntimeError(f'Incomplete scene copy: {temporary}')
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    print(f'✅ Published scene MP4: {destination}', flush=True)
    return destination

def vx_render_squad_video(plan,key,label,assets):
    import fcntl
    chip='wildcard' if key.startswith('wildcard') else 'free_hit'
    with (VX_MEDIA_WORK/(chip+'.render.lock')).open('a') as lock:
        try:fcntl.flock(lock.fileno(),fcntl.LOCK_EX|fcntl.LOCK_NB)
        except BlockingIOError:raise RuntimeError(chip+' video is already rendering in this workspace')
        renderer=_vx_render_verified_wildcard if chip=='wildcard' else _vx_render_verified_freehit
        return renderer(plan,key,label,assets)


def _vx_freehit_matchday_still_4k(visual):
    """Render the last 4K Free Hit Slide 2 frame using its Remotion composition.

    RemotionMatchdayVisual is a motion renderer, not a Pillow visual. It has no
    ``.complete`` attribute; the generic Pillow PNG exporter must not call it.
    This uses one still frame, never a long animation render in Draft mode.
    """
    from wildcard_masterplan import (
        _vx_remotion_cli, _vx_remotion_browser_arguments, _vx_remotion_command_output
    )
    if not isinstance(visual, VX_FREEHIT_MATCHDAY.RemotionMatchdayVisual):
        raise TypeError('Free Hit Slide 2 requires RemotionMatchdayVisual')
    duration = float(visual.scene.get('duration_seconds', 0))
    if not math.isfinite(duration) or duration <= 0:
        raise ValueError('Free Hit Slide 2 requires measured narration before its 4K PNG')
    timeline = visual.scene.get('narration_timeline') or []
    if not timeline or any(cue.get('start') is None or cue.get('end') is None for cue in timeline):
        raise ValueError('Free Hit Slide 2 narration timeline must be measured before still export')
    fps = 24
    # A final 4K still captures the completed slide, not an earlier highlight.
    cue_index = len(timeline)-1
    final_frame = max(0, math.ceil(duration*fps)-2)
    VX_FREEHIT_MATCHDAY.prepare_project(
        visual.scene, visual.project, visual.media, visual.badges, visual.brand_root,
        audio=None, width=3840, height=2160, fps=fps, duration=duration,
    )
    cli = _vx_remotion_cli(visual.project)
    target = visual.project / 'vortex_freehit_matchday_slide2_4k.png'
    target.unlink(missing_ok=True)
    command = [
        cli, 'still', 'src/index.ts', 'FreeHitMatchday', str(target),
        '--frame=' + str(final_frame),
        '--props=' + json.dumps({'cueIndex': cue_index, 'mute': True}, separators=(',', ':')),
        '--log=info', '--overwrite', *_vx_remotion_browser_arguments(visual.project),
    ]
    _vx_remotion_command_output(visual.project, 'freehit_4k_still', command)
    if not target.is_file() or target.stat().st_size <= 0:
        raise RuntimeError('Remotion did not create the Free Hit Slide 2 4K PNG: ' + str(target))
    with Image.open(target) as image:
        frame = image.convert('RGB')
    if frame.size != (3840, 2160):
        raise RuntimeError(f'Free Hit Slide 2 Remotion still is {frame.size}, expected 3840x2160')
    return frame


def vx_export_4k_png_slides(key,contract,scenes,start_index=1):
    """Persist native 3840x2160 slide PNGs in Final only."""
    output=VX_4K_PNG_OUT_DIR/key/VX_VIDEO_MODE.lower()
    output.mkdir(parents=True,exist_ok=True)
    report=[]
    media_state={}
    for pid,record in VX_PLAYER_MEDIA.items():
        path=record.get('photo')
        if path and Path(path).is_file():
            media_state[str(pid)]=file_sha256(path)
    static_assets={}
    for name,path in VX_ASSET_PATHS.items():
        p=Path(path)
        if p.is_file() and p.suffix.lower() in ('.png','.jpg','.jpeg','.webp'):
            static_assets[name]=file_sha256(p)
    common=json.dumps({
        'data_hash':contract.get('data_hash'),'renderer':VX_RENDERER_VERSION,
        'presentation':VX_PRESENTATION_VERSION,'media':media_state,'assets':static_assets,
        'size':[3840,2160]
    },sort_keys=True,default=str)
    for index, scene in enumerate(scenes, start_index):
        stable_scene={field:value for field,value in scene.items()
                      if field not in ('media_file','cache_media_file','encoded_duration_seconds','hero_card_asset_proof')}
        signature=hashlib.sha256((common+json.dumps(stable_scene,sort_keys=True,default=str)).encode()).hexdigest()
        image=output/(vx_delivery_slide_name(key, index)+' 4K.png')
        stamp=VX_ANALYSIS_ROOT/'png_cache'/key/(scene['scene_id']+'.sha256');stamp.parent.mkdir(parents=True,exist_ok=True)
        reuse=image.is_file() and stamp.is_file() and stamp.read_text().strip()==signature
        if reuse:
            try:reuse=Image.open(image).size==(3840,2160)
            except OSError:reuse=False
        if not reuse:
            visual=vx_visual_from_manifest(scene,contract,3840,2160)
            if scene['visual_type']=='fixture_heatmap' and scene.get('matchday_case'):
                # This is a Remotion composition, not a Pillow object.
                frame=_vx_freehit_matchday_still_4k(visual)
            else:
                if not hasattr(visual,'complete'):
                    raise TypeError(f'{key}/{scene["scene_id"]}: {type(visual).__name__} has no Pillow .complete; provide a native still renderer')
                frame=visual.complete.convert('RGB')
            if frame.size!=(3840,2160):raise RuntimeError(f'{key}/{scene["scene_id"]}: 4K PNG frame size is {frame.size}')
            temporary=image.with_suffix('.part.png')
            frame.save(temporary,format='PNG',compress_level=3)
            temporary.replace(image)
            stamp.write_text(signature)
        report.append({'scene_id':scene['scene_id'],'path':str(image),'width':3840,'height':2160,'reused':reuse})
    if len(scenes) == 4:
        (VX_ANALYSIS_ROOT/(key+'_4k_png_manifest.json')).write_text(json.dumps(report,indent=2))
    print(f'✅ {key}: {len(report)} 4K PNG slides ready at {output}',flush=True)
    return report


# Shared broadcast master: all generated Wildcard and Free Hit slides.
import os
import sys
import subprocess
import importlib.util
from pathlib import Path
import base64 as _vx_bg_base64
import io as _vx_bg_io
import zipfile as _vx_bg_zipfile
import shutil as _vx_bg_shutil
import importlib as _vx_bg_importlib
from PIL import Image as _vx_bg_Image

VX_BACKGROUND_THEME = globals().get('VX_BACKGROUND_THEME', 'auto')
VX_BACKGROUND_THEME_SEED = globals().get('VX_BACKGROUND_THEME_SEED', 0)
VX_BROADCAST_ROOT = VX_MEDIA_WORK / 'broadcast_master_4k'
VX_BROADCAST_ROOT.mkdir(parents=True, exist_ok=True)
_VX_BROADCAST_BUNDLE = '...'
with _vx_bg_zipfile.ZipFile(_vx_bg_io.BytesIO(_vx_bg_base64.b64decode(_VX_BROADCAST_BUNDLE))) as _vx_bg_bundle:
    if any(Path(name).is_absolute() or '..' in Path(name).parts for name in _vx_bg_bundle.namelist()):
        raise ValueError('Unsafe broadcast bundle path')
    _vx_bg_bundle.extractall(VX_BROADCAST_ROOT)

# VX_WCX_CONTROLLER_V1: legacy captaincy animation must not target the current showdown layout.
import wildcard_final_slide as _vx_captaincy_module
_vx_captaincy_module = importlib.reload(_vx_captaincy_module)
_vx_bg_template_path = VX_BROADCAST_ROOT / 'broadcast.html'
_vx_bg_template = _vx_bg_template_path.read_text(encoding='utf-8')
# The full-page wrapper must preserve async errors and wait for image decoding.
_vx_bg_template = _vx_bg_template.replace('window.renderSlide=function(data){previous(data);', 'window.renderSlide=async function(data){await previous(data);')
_vx_old_captaincy = "const root=document.querySelector('.wildcard-captaincy-slide');if(!root)return;"
_vx_scoped_captaincy = "const root=document.querySelector('.wildcard-captaincy-slide');if(!root||root.classList.contains('wcx-slide'))return;"
if _vx_old_captaincy in _vx_bg_template:
    _vx_bg_template = _vx_bg_template.replace(_vx_old_captaincy, _vx_scoped_captaincy)
elif _vx_scoped_captaincy not in _vx_bg_template:
    raise RuntimeError('Wildcard captaincy animation controller could not be identified')
_vx_current_captaincy_js = _vx_captaincy_module.SEEK_JS
if "document.querySelector('.wcx-v6')" not in _vx_current_captaincy_js:
    raise RuntimeError('The current Wildcard captaincy controller is missing')
if '<!-- VX_WCX_CONTROLLER_V7 -->' not in _vx_bg_template:
    _vx_bg_template += '\n<!-- VX_WCX_CONTROLLER_V7 -->\n' + _vx_current_captaincy_js
_vx_bg_template_path.write_text(_vx_bg_template, encoding='utf-8')
print('✅ Wildcard captaincy: current animation controller loaded; legacy layout excluded', flush=True)

_vx_bg_local = Path(os.environ.get('FPL_MEDIA_ASSET_DIR', str(Path.cwd() / 'media_assets')))
_vx_bg_candidates = [_vx_bg_local, Path('/content/drive/MyDrive/FPL_VORTEX/Assets '),
                      Path('/content/drive/MyDrive/FPL_VORTEX/Assets')]
for _vx_bg_name in ('logo1.png', 'pl.png', 'logo.png'):
    _vx_bg_original = next((folder / _vx_bg_name for folder in _vx_bg_candidates
                            if (folder / _vx_bg_name).is_file()), None)
    if _vx_bg_original is None:
        raise FileNotFoundError(f'Original {_vx_bg_name} unavailable in the mounted approved Drive Assets folder; no replacement logo will be used.')
    _vx_bg_target = VX_BROADCAST_ROOT / 'assets' / _vx_bg_name
    _vx_bg_shutil.copyfile(_vx_bg_original, _vx_bg_target)
    if _vx_bg_original.read_bytes() != _vx_bg_target.read_bytes():
        raise RuntimeError('Original logo copy changed unexpectedly')

if importlib.util.find_spec('playwright') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'playwright==1.62.0'], check=True)
# Browser binaries alone do not install libatk and the other Linux shared libraries.
# Colab/Jupyter already has an asyncio loop. Playwright's sync API must run
# on a dedicated thread (no event loop in that thread), including this probe.
from concurrent.futures import ThreadPoolExecutor as _VX_BrowserProbePool
from playwright.sync_api import sync_playwright

def _vx_chromium_path_from_worker():
    with sync_playwright() as _vx_browser_probe:
        return Path(_vx_browser_probe.chromium.executable_path)

with _VX_BrowserProbePool(max_workers=1, thread_name_prefix='vortex-chromium-probe') as _vx_browser_pool:
    _vx_browser_path=_vx_browser_pool.submit(_vx_chromium_path_from_worker).result()
if _vx_browser_path.is_file():
    print('✅ Reusing existing Playwright Chromium:',_vx_browser_path,flush=True)
else:
    print('Installing Chromium and its system dependencies once...', flush=True)
    try:
        subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Chromium dependency installation failed. Check the installer output above and rerun Cell 14 after resolving it.') from exc
if str(VX_BROADCAST_ROOT) not in sys.path:
    sys.path.insert(0, str(VX_BROADCAST_ROOT))
_vx_bg_importlib.invalidate_caches()
import broadcast_runtime as _vx_bg_runtime
if globals().get('VX_BROADCAST_RENDERER') is not None:
    VX_BROADCAST_RENDERER.close()
_vx_bg_importlib.reload(_vx_bg_runtime)
VX_BROADCAST_RENDERER = _vx_bg_runtime.BroadcastRenderer(VX_BROADCAST_ROOT, VX_MEDIA_WORK / 'broadcast_shell_cache')
try:
    VX_BROADCAST_RENDERER.check_ready()
except Exception:
    VX_BROADCAST_RENDERER.close()
    raise
print('Chromium launch, template and screenshot preflight passed.', flush=True)
VX_RENDERER_VERSION += ':' + _vx_bg_runtime.VERSION + ':wc-slide1-ryan-cue-v6:fh3-full-hero-flight-v2'
VX_PRESENTATION_VERSION = VX_PRESENTATION_VERSION.split(':vortex-master-', 1)[0] + ':' + _vx_bg_runtime.VERSION

_vx_bg_titles = {'wc_blueprint':'WILDCARD TIMING','fh_blueprint':'FREE HIT TIMING','wc_analysis':'FIXTURE DIFFICULTY','fixture_heatmap':'FIXTURE DIFFICULTY','wc_squad_plan':'WILDCARD SQUAD PLAN','fh_squad_plan':'FREE HIT SQUAD PLAN','wc_captaincy_final':'CAPTAINCY PLAN','fh_captain_final':'CAPTAIN PICK'}
for _vx_bg_key, _vx_bg_scenes in VX_SCENE_MANIFESTS.items():
    for _vx_bg_index, _vx_bg_scene in enumerate(_vx_bg_scenes):
        _vx_bg_scene['broadcast_index'] = _vx_bg_index
        _theme_facts = {
            'chip': _vx_bg_scene.get('chip', ''),
            'gameweek': _vx_bg_scene.get('gameweek', ''),
            'visual_type': _vx_bg_scene.get('visual_type', ''),
            'data_hash': _vx_bg_scene.get('data_hash', ''),
            'decision': (_vx_bg_scene.get('wildcard_case') or _vx_bg_scene.get('freehit_case') or {}).get('verdict', ''),
        }
        _vx_bg_scene['theme_seed'] = int(hashlib.sha256(
            json.dumps(_theme_facts, sort_keys=True, default=str).encode('utf-8')
        ).hexdigest()[:8], 16)
        # Only Wildcard’s four existing slide types receive a stable choice from the approved palette.
        # A separately configured scene theme is never overwritten; Free Hit retains its existing logic.
        _vx_bg_palette = {
            'wc_blueprint': 'Emerald Royal',
            'wc_analysis': 'Sapphire Elite',
            'wc_squad_plan': 'Purple Dynasty',
            'wc_captaincy_final': 'Black & Gold Legend',
        }
        _vx_bg_current = _vx_bg_scene.get('background_theme')
        if _vx_bg_key.startswith('wildcard') and _vx_bg_scene.get('visual_type') in _vx_bg_palette:
            _vx_bg_scene['background_theme'] = _vx_bg_current if _vx_bg_current not in (None, 'auto') else _vx_bg_palette[_vx_bg_scene['visual_type']]
        else:
            _vx_bg_scene.setdefault('background_theme', 'auto')
        _vx_bg_scene.setdefault('display_title', _vx_bg_titles.get(_vx_bg_scene['visual_type'], _vx_bg_scene['title']))
        _vx_bg_scene['background'] = {'type': 'html_css_master', 'version': _vx_bg_runtime.VERSION,
            'theme': _vx_bg_runtime.select_theme(_vx_bg_scene['background_theme'], _vx_bg_index, _vx_bg_scene['theme_seed'])}

# The foreground uses an empty canvas; the old header/footer are fully replaced.
VXWV.WildcardVisual.base = lambda self: _vx_bg_Image.new('RGB', (self.width, self.height), 'black')

def _vx_bg_metadata(contract):
    metadata = dict(contract)
    selected_event = next((event for event in _bootstrap.get('events', [])
                           if event['id'] == contract.get('gameweek')), None)
    metadata['deadline'] = selected_event.get('deadline_time') if selected_event else None
    metadata['season'] = contract.get('season') or getattr(VX_WC_ENGINE, 'season', None)
    return metadata




print('Shared 4K broadcast master enabled: 10 themes, original logos, dynamic HTML header/footer, America/New_York timestamps.')


# Fail Final immediately if source data was stale when Cell 14 started.
# Draft remains usable for visual/audio review and only warns on age.
if VX_RENDER_VIDEOS:
    for _vx_preflight_key,_vx_preflight_contract in VX_DATA_CONTRACTS.items():
        vx_assert_current_snapshot(_vx_preflight_contract)

# This class uses the existing broadcast renderer and native FFmpeg video clock.
from wildcard_intro import render_html as _vx_intro_html

class WildcardBlueprintVisual:
    requires_motion=True
    def __new__(cls,scene,contract,width,height):
        from wildcard_intro_motion import WildcardIntroMotion, VERSION as _vx_intro_animation_version
        if _vx_intro_animation_version != 'wildcard-slide1-ryan-cue-v7':
            raise RuntimeError('Stale Wildcard Slide 1 animation module; rerun Cell 2 and Cell 14.')
        if any(c.get('start') is None for c in scene['narration_timeline']):
            import copy
            from wildcard_plan import align_timeline
            scene=copy.deepcopy(scene)
            scene['narration_timeline']=align_timeline(scene['narration_timeline'],len(scene['narration_text'].split())/145*60)
        # Keep the current six-Gameweek contract and attach the live official xG leaderboard.
        scene = dict(scene, xg_leaders=VX_INTRO_XG_LEADERS)
        shell_scene=dict(scene)
        shell_scene['display_title']='WILDCARD: NOW OR WAIT?'
        shell_scene['subheading']=f"My draft • GW {contract['horizon_gameweeks'][0]}–{contract['horizon_gameweeks'][-1]}"
        background=VX_BROADCAST_RENDERER.image(shell_scene,_vx_bg_metadata(contract),1920,1080)
        return WildcardIntroMotion(scene,contract,width,height,VX_WC_PLAYER_MEDIA,VX_CLUB_MEDIA,VX_BROADCAST_ROOT/'assets',background)

VXWV.BLUEPRINT_VISUAL_FACTORY = WildcardBlueprintVisual
VXWV.SHARED_VISUAL_FACTORY = vx_visual_from_manifest
VX_CLUB_MEDIA=vx_prepare_club_media(_bootstrap,VX_MEDIA_WORK)
VXWV.CLUB_MEDIA=VX_CLUB_MEDIA
_vx_card_ids=set()
for _vx_contract in VX_DATA_CONTRACTS.values():
    _vx_card_ids.update(int(pid) for pid in _vx_contract.get('media_entities',{}))
    _vx_card_ids.update(int(p['id']) for p in _vx_contract.get('squad',[]) if p.get('id') is not None)
for _contract in VX_DATA_CONTRACTS.values():
    _upgrade=_contract.get('slide3_plan',{}).get('bench_boost_upgrade_target')
    if _upgrade:_vx_card_ids.add(int(_upgrade['id']))
for _vx_scenes in VX_SCENE_MANIFESTS.values():
    for _vx_scene in _vx_scenes:
        if _vx_scene.get('visual_type')=='wc_analysis':
            _vx_card_ids.update(p['player_id'] for p in _vx_scene.get('elite_fixture_case',{}).get('elite_players',[]))
        elif _vx_scene.get('visual_type')=='fixture_heatmap' and _vx_scene.get('matchday_case'):
            _vx_card_ids.update(int(player['player_id']) for player in _vx_scene['matchday_case'].get('players',[]))
_vx_card_bootstrap={**_bootstrap,'elements':[p for p in _bootstrap['elements'] if int(p['id']) in _vx_card_ids]}
VX_CARD_VIEWS=VXWV.build_player_card_views(_vx_card_bootstrap,VX_WC_ENGINE.fdr,VX_ANALYSIS_ROOT/'current_source_cache')
VXWV.PLAYER_CARD_VIEWS=VX_CARD_VIEWS
print(f'Player-card data prepared for {len(VX_CARD_VIEWS)} selected/referenced players (not the full player pool).',flush=True)
VX_ASSET_PATHS={name:VX_BROADCAST_ROOT / 'assets' / name for name in ('logo1.png','pl.png','logo.png')}
_vx_local_brand=Path(os.environ.get("FPL_MEDIA_ASSET_DIR","."))/"7.png"
if _vx_local_brand.is_file():VX_ASSET_PATHS["7.png"]=_vx_local_brand
if VX_RENDER_VIDEOS and not globals().get('VX_INTRO_PREVIEW_ONLY',False):
    VX_ASSET_PATHS.update(_vx_resolve_assets_runtime(VX_ASSETS_FOLDER_URL,VX_MEDIA_WORK))
def vx_intro_xg_ranking(bootstrap):
    """Top five official season-to-date expected goals, preserving FPL IDs for photos."""
    import math
    teams = {int(team['id']): team for team in bootstrap.get('teams', [])}
    ranked = []
    for player in bootstrap.get('elements', []):
        try:
            xg = float(player['expected_goals'])
            player_id = int(player['id'])
            team = teams[int(player['team'])]
        except (ValueError, TypeError, KeyError):
            continue
        if not math.isfinite(xg) or xg < 0:
            continue
        try:
            photo_code = int(player.get('code') or 0)
        except (ValueError, TypeError):
            photo_code = 0
        ranked.append(dict(
            id=player_id,
            name=str(player.get('web_name') or player.get('second_name') or player_id),
            club=str(team.get('short_name') or team.get('name') or ''),
            xg=xg,
            photo_code=photo_code if photo_code > 0 else None,
        ))
    ranked.sort(key=lambda record: (-record['xg'], record['id']))
    return ranked[:5]

VX_INTRO_XG_LEADERS = vx_intro_xg_ranking(_bootstrap) if VX_GENERATE_WILDCARD else []

if VX_GENERATE_WILDCARD or VX_GENERATE_FREEHIT:
    for name in ['grass_pitch.png']:
        VX_ASSET_PATHS.setdefault(name,VX_WC_EMBEDDED_ASSET_DIR/name)
    if VX_GENERATE_WILDCARD:
        VX_ASSET_PATHS.setdefault('WC_clean.png',VX_WC_EMBEDDED_ASSET_DIR/'WC_clean.png')
    _media_ids=set().union(*(set(map(int,VX_DATA_CONTRACTS[k]['media_entities'])) for k in VX_DATA_CONTRACTS)) if VX_DATA_CONTRACTS else set()
    for _intro_contract in VX_DATA_CONTRACTS.values():
        _intro_case=_intro_contract.get('wildcard_case') or _intro_contract.get('freehit_case') or {}
        for _intro_reason in _intro_case.get('reasons',[]):
            _media_ids.update(int(player['id']) for player in _intro_reason.get('source_data_summary',{}).get('players',[]) if player.get('status')=='i' or (player.get('status')=='d' and 'injur' in str(player.get('news','')).lower()))
    for _contract in VX_DATA_CONTRACTS.values():
        _upgrade=_contract.get('slide3_plan',{}).get('bench_boost_upgrade_target')
        if _upgrade:_media_ids.add(int(_upgrade['id']))
    for _fixture_contract in VX_DATA_CONTRACTS.values():
        for _fixture_row in _fixture_contract.get('fixture_case',{}).get('targets',[]):
            _media_ids.update(int(p['player_id']) for p in _fixture_row.get('players',[]))
            if _fixture_row.get('price_peer'):_media_ids.add(int(_fixture_row['price_peer']['player_id']))
    for _s3_contract in VX_DATA_CONTRACTS.values():
        _media_ids.update(int(p['id']) for p in _s3_contract.get('slide3_plan',{}).get('bench_boost_path',{}).get('targets',[]))
    for _vx_scenes in VX_SCENE_MANIFESTS.values():
        for _vx_scene in _vx_scenes:
            if _vx_scene.get('visual_type')=='wc_analysis':
                _media_ids.update(p['player_id'] for p in _vx_scene.get('elite_fixture_case',{}).get('elite_players',[]))
    for _vx_fh_scenes in VX_SCENE_MANIFESTS.values():
        for _vx_fh_scene in _vx_fh_scenes:
            _media_ids.update(p['player_id'] for p in _vx_fh_scene.get('matchday_case',{}).get('players',[]))
    # Resolve official xG-leader headshots using the existing photo pipeline; missing codes use avatars.
    _media_ids.update(row['id'] for row in VX_INTRO_XG_LEADERS if row.get('photo_code'))
    VX_PLAYER_MEDIA=VXWV.safe_player_media(VX_WC_ENGINE.pool,_media_ids,VX_MEDIA_WORK,VX_WC_ENGINE.boot)
    VX_WC_PLAYER_MEDIA=VX_PLAYER_MEDIA
# Hero cards read the existing normalized FPL core records, media and club assets.
from hero_card import card_html as _vx_hero_html

_vx_hero_logo = VX_BROADCAST_ROOT / 'assets' / 'logo.png'
VX_BROADCAST_RENDERER.hero_provider = None


print('Hero cards enabled: 12 available FPL core stats per position; max 30% slide width; original logo.png.')

if VX_GENERATE_WILDCARD:
    _wc_keys=[key for key in VX_DATA_CONTRACTS if key.startswith('wildcard')]
    for key in _wc_keys:
        contract=VX_DATA_CONTRACTS[key];scenes=VX_SCENE_MANIFESTS[key]
        if VX_VIDEO_MODE=='Final' and not globals().get('VX_INTRO_PREVIEW_ONLY',False):
            proof=VXWV.verify_visuals(contract,scenes,VX_ASSET_PATHS,VX_WC_PLAYER_MEDIA,VX_MEDIA_WORK/'qa_previews'/key,width=VX_VIDEO_PROFILE['width'],height=VX_VIDEO_PROFILE['height'],preview_limit=None)
            if not all(p['verified'] for p in proof):
                raise RuntimeError(f"{key}: final visual verification failed")
        else:
            # Skip expensive screenshots, 4K imagery and HTML previews in 480p Draft.
            # Data contract, Scene JSON, script, and asset checks remain enforced.
            VXWC.validate_scene_json(VX_WILDCARD_SCENE_JSONS[key])
            proof=[{'verified':True,'scene_id':s['scene_id']} for s in scenes]
        VX_WILDCARD_PREVIEW=None
        script_path=VX_NARRATION_PATHS.get(key)
        if script_path is None or not Path(script_path).is_file() or key not in VX_NARRATION_OBJECTS:
            raise RuntimeError(f"{key}: validated Scene-JSON narration is missing; rerun Cell 12, then Cell 13 before rendering")
        narration=decode_script(Path(script_path).read_text(encoding='utf-8'))
        VXWC.validate_render_script(VX_NARRATION_OBJECTS[key],narration)
        VX_NARRATION[key]=narration
        for scene,segment in zip(scenes,narration):scene['narration_text']=segment['text']
        (VX_ANALYSIS_ROOT/(key+'_scene_manifest.json')).write_text(json.dumps(scenes,indent=2,ensure_ascii=False))
        print(f"✅ {key}: {len(proof)} verified slides; Scene-JSON narration already validated at {script_path}")
        if VX_VIDEO_MODE=='Final':print('Approved visual proof:',len(proof),'scenes')




def vx_render_intro_preview():
    import copy
    from wildcard_plan_audio import prepare_segment_audio
    from wildcard_analyst_intro import enrich_case, generate_segments
    key=next(k for k in VX_DATA_CONTRACTS if k.startswith('wildcard'))
    contract=VX_DATA_CONTRACTS[key]
    vx_validate_contract(contract);vx_assert_current_snapshot(contract)
    scene=copy.deepcopy(next(s for s in VX_SCENE_MANIFESTS[key] if s['visual_type']=='wc_blueprint'))
    scene['wildcard_case']=enrich_case(contract['wildcard_case'],contract,_bootstrap)
    scene['narration_timeline']=generate_segments(scene['wildcard_case'])
    scene['narration_text']=' '.join(c['narration_text'] for c in scene['narration_timeline'])
    script=dict(id=scene['scene_id'],text=scene['narration_text'])
    for cue in scene['narration_timeline']:
        vx_prepare_tts(dict(script,text=cue['narration_text'],scene=scene,narration_cue=cue))
    print(f"Intro pronunciation preflight passed: 6 cues. Generating Ryan at {VX_INTRO_TTS_PROFILE['rate']} / {VX_INTRO_TTS_PROFILE['pitch']}.",flush=True)
    audio=prepare_segment_audio(scene,script,VX_MEDIA_WORK,
        lambda cue:_vx_run_async(_vx_wc_audio_for_segment(cue)),
        lambda source,target:run_ffmpeg(['-i',source,'-ar','48000','-ac','2','-c:a','pcm_s16le',target]))
    duration=media_duration(audio)
    profile=dict(VX_VIDEO_PROFILE) if VX_VIDEO_MODE=='Draft' else dict(width=1920,height=1080,fps=30,sar='1/1',preset='fast',crf=18)
    # Every intro frame uses the same object timeline in both quality modes.
    profile.update(static_preview=False,scene_crossfades=True)
    print(f"Ryan intro audio ready: {duration:.2f}s. Rendering {profile['width']}×{profile['height']} / {profile['fps']} fps.",flush=True)
    visual=vx_visual_from_manifest(scene,contract,profile['width'],profile['height'])
    target=VX_VIDEO_OUT_DIR/(key+'_GW'+str(contract['gameweek'])+'_intro_animated.mp4')
    partial=VX_MEDIA_WORK/'wildcard_intro_preview.partial.mp4'
    native_body_segment(visual,audio,duration,partial,profile)
    target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(partial,target)
    metadata=probe_media_info(target)
    frames=VX_ANALYSIS_ROOT/'wildcard_intro_frames';frames.mkdir(exist_ok=True)
    checks=[]
    left_reference=np.asarray(visual.draw(0.0,duration).convert('RGB'))[:,:int(profile['width']*946/1920),:].copy()
    for i,cue in enumerate(scene['narration_timeline']):
        a=visual.draw(cue['start']+.35,duration);b=visual.draw(min(cue['end']-.05,cue['start']+2.5),duration)
        # The entire left-side data panel must be identical at zero and throughout.
        for frame in (a,b):
            left_now=np.asarray(frame.convert('RGB'))[:,:left_reference.shape[1],:]
            if not np.array_equal(left_reference,left_now):
                raise RuntimeError('Wildcard Slide 1 left panel changes after 00:00')
        changed=int(np.count_nonzero(np.any(np.asarray(a)!=np.asarray(b),axis=2)))
        if changed<max(100,int(profile['width']*profile['height']*.004)):raise RuntimeError('Opening scene lacks object motion: '+cue['segment_id'])
        a.save(frames/(str(i+1)+'_entrance.png'));b.save(frames/(str(i+1)+'_settled.png'))
        checks.append(dict(scene=cue['segment_id'],changed_pixels=changed,start=cue['start'],end=cue['end']))
    report=dict(scope='Wildcard slide 1 only; later slides and Free Hit not rendered in this preview',duration_seconds=duration,profile=profile,metadata=metadata,voice=VX_TTS_VOICE,voice_settings=vx_narration_profile(script | {"scene": scene}),voice_settings_source=VX_INTRO_TTS_PROFILE_SOURCE,timeline=scene['narration_timeline'],motion_checks=checks,data_snapshot=contract.get('source_snapshot_at'),generated_at=contract.get('generated_at'),comparison_verified=scene['wildcard_case'].get('scenario_comparison') is not None,decision=scene['wildcard_case']['verdict'],decision_reason=scene['wildcard_case']['verdict_reason'],intro_context=scene['wildcard_case']['intro_context'],assets='Original VORTEX logos; official FPL player-code photographs and club crests')
    (VX_ANALYSIS_ROOT/'wildcard_intro_verification.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
    (VX_ANALYSIS_ROOT/'wildcard_intro_narration.txt').write_text(scene['narration_text'])
    VX_INTRO_PREVIEW_PATH=target
    print('Animated intro complete:',target,flush=True);print(json.dumps(report,ensure_ascii=False,indent=2),flush=True)
    from IPython.display import Video,display
    display(Video(str(target),embed=True,width=960))
    return target

def vx_render_wildcard_slide3_preview():
    """Render only Wildcard Slide 3, preserving Ryan and Draft 480p/24 FPS."""
    import copy,subprocess
    key=next(k for k in VX_DATA_CONTRACTS if k.startswith('wildcard'))
    contract=VX_DATA_CONTRACTS[key];vx_validate_contract(contract);vx_assert_current_snapshot(contract)
    scene=copy.deepcopy(next(s for s in VX_SCENE_MANIFESTS[key] if s['visual_type']=='wc_squad_plan'))
    script=dict(id=scene['scene_id'],text=scene['narration_text'],scene=scene)
    for cue in scene['narration_timeline']:vx_prepare_tts(dict(script,text=cue['narration_text'],narration_cue=cue))
    print('Wildcard Slide 3: measuring Ryan phrases; profile unchanged',VX_INTRO_TTS_PROFILE,flush=True)
    audio=VX_MASTERPLAN.prepare_audio(scene,script,VX_MEDIA_WORK,
        lambda item:_vx_run_async(_vx_wc_raw_audio_for_segment(item)),
        lambda source,target:run_ffmpeg(['-i',source,'-ar','48000','-ac','2','-c:a','pcm_s16le',target]))
    duration=media_duration(audio)
    (VX_ANALYSIS_ROOT/'wildcard_slide3_narration.txt').write_text(scene['narration_text'])
    (VX_ANALYSIS_ROOT/'wildcard_slide3_script_and_cues.json').write_text(json.dumps(dict(narration=scene['narration_text'],timeline=scene['narration_timeline']),ensure_ascii=False,indent=2))
    if not (math.isfinite(duration) and duration > 0):raise RuntimeError(f'Wildcard Slide 3 narration has invalid measured duration: {duration!r}s.')
    profile=dict(VX_VIDEO_PROFILE)
    if VX_VIDEO_MODE=='Draft':profile.update(width=VX_VIDEO_PROFILE['width'],height=VX_VIDEO_PROFILE['height'],fps=VX_VIDEO_PROFILE['fps'],sar=VX_VIDEO_PROFILE['sar'])
    else:profile.update(width=1920,height=1080,fps=30,sar='1/1',crf=18)
    profile['static_preview']=False
    print(f"Wildcard Slide 3: {duration:.2f}s; {profile['width']}×{profile['height']} / {profile['fps']} FPS.",flush=True)
    visual=vx_visual_from_manifest(scene,contract,profile['width'],profile['height']);visual.prepare(audio,duration,profile)
    if not (VX_MASTERPLAN_PROJECT/'node_modules/remotion').exists():subprocess.run(['npm','install','--no-audit','--no-fund'],cwd=VX_MASTERPLAN_PROJECT,check=True)
    subprocess.run(['node','qa.mjs'],cwd=VX_MASTERPLAN_PROJECT,check=True)
    checks=[];frame_root=VX_MASTERPLAN_PROJECT/'qa_frames'
    for i,cue in enumerate(scene['narration_timeline']):
        a=np.asarray(Image.open(frame_root/f'{i:02d}_entrance.png').convert('RGB'));b=np.asarray(Image.open(frame_root/f'{i:02d}_settled.png').convert('RGB'))
        changed=int(np.count_nonzero(np.any(a!=b,axis=2)))
        if changed<100:raise ValueError('No visible entrance animation for '+cue['segment_id'])
        check=dict(segment=cue['segment_id'],type=cue['type'],start=cue['start'],speech_end=cue['speech_end'],end=cue['end'],changed_pixels=changed)
        if cue['type']=='players':
            # The original hero remains in the right panel; only the compact pitch token appears on the left.
            scale=a.shape[1]/1920
            pitch_edge=int(1142*scale);stage_start=int(1176*scale)
            pitch_entrance=a[:,:pitch_edge];pitch_settled=b[:,:pitch_edge]
            right_entrance=a[:,stage_start:];right_settled=b[:,stage_start:]
            pitch_motion=int(np.count_nonzero(np.any(pitch_entrance!=pitch_settled,axis=2)))
            right_motion=int(np.count_nonzero(np.any(right_entrance!=right_settled,axis=2)))
            if pitch_motion<100 or right_motion<100:raise ValueError('Pitch tile or right-side hero reveal missing')
            check.update(player_id=cue['player_id'],pitch_token_changed_pixels=pitch_motion,right_hero_changed_pixels=right_motion,hero_stays_right=True)
        checks.append(check)
    target=VX_VIDEO_OUT_DIR/(key+'_GW'+str(contract['gameweek'])+'_slide3_masterplan_animated.mp4');target.parent.mkdir(parents=True,exist_ok=True)
    visual.render_video(audio,duration,target,profile)
    metadata=probe_media_info(target)
    report=dict(scope='Wildcard Slide 3 only',duration_seconds=duration,profile=profile,metadata=metadata,voice=VX_TTS_VOICE,voice_settings=VX_INTRO_TTS_PROFILE,voice_rate_changed_to_fit_duration=False,clock='Measured narration PCM with 1.1-second hero flight after each explanation',data_checks=scene['masterplan_case']['verification'],weekly_plan=scene['masterplan_case']['weeks'],motion_checks=checks,source_hash=contract['data_hash'],source_snapshot_at=contract['source_snapshot_at'],background='Existing broadcast_master_4k shell, original logos and dynamic season header',hero_component='Existing hero_card.card_html + broadcast.html CSS, fixed inside the right analysis panel',renderer='Remotion',source_project=str(VX_MASTERPLAN_PROJECT))
    (VX_ANALYSIS_ROOT/'wildcard_slide3_verification.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
    from IPython.display import Video,display
    display(Video(str(target),embed=True,width=960));print('Animated Wildcard Slide 3 complete:',target,flush=True);return target

def vx_render_wildcard_slide2_preview():
    """Render and verify only the upgraded Wildcard fixture analysis."""
    import copy
    from wildcard_plan_audio import prepare_segment_audio
    key=next(k for k in VX_DATA_CONTRACTS if k.startswith('wildcard'))
    contract=VX_DATA_CONTRACTS[key]
    vx_validate_contract(contract);vx_assert_current_snapshot(contract)
    scene=copy.deepcopy(next(s for s in VX_SCENE_MANIFESTS[key] if s['visual_type']=='wc_analysis'))
    if not scene.get('elite_fixture_case'):raise RuntimeError('Run the Wildcard Slide 2 cell before its preview')
    script=dict(id=scene['scene_id'],text=scene['narration_text'])
    for cue in scene['narration_timeline']:
        vx_prepare_tts(dict(script,text=cue['narration_text'],scene=scene,narration_cue=cue))
    print('Slide 2: generating measured Ryan phrases with the unchanged Slide 1 voice profile:',VX_INTRO_TTS_PROFILE,flush=True)
    audio=prepare_segment_audio(scene,script,VX_MEDIA_WORK,
        lambda cue:_vx_run_async(_vx_wc_raw_audio_for_segment(cue)),
        lambda source,target:run_ffmpeg(['-i',source,'-ar','48000','-ac','2','-c:a','pcm_s16le',target]))
    duration=media_duration(audio)
    if not (math.isfinite(duration) and duration > 0):
        raise RuntimeError(f'Wildcard Slide 2 has an invalid measured narration duration: {duration!r}s.')
    profile=dict(VX_VIDEO_PROFILE) if VX_VIDEO_MODE=='Draft' else dict(width=1920,height=1080,fps=30,sar='1/1',preset='fast',crf=18)
    if VX_VIDEO_MODE=='Draft':profile.update(width=VX_VIDEO_PROFILE['width'],height=VX_VIDEO_PROFILE['height'],fps=VX_VIDEO_PROFILE['fps'],sar=VX_VIDEO_PROFILE['sar'])
    profile.update(static_preview=False,scene_crossfades=True)
    print(f"Slide 2 Ryan track: {duration:.2f}s. Rendering {profile['width']}×{profile['height']} / {profile['fps']} FPS.",flush=True)
    visual=vx_visual_from_manifest(scene,contract,profile['width'],profile['height'])
    frames=VX_ANALYSIS_ROOT/'wildcard_slide2_frames';frames.mkdir(exist_ok=True);checks=[]
    for index,cue in enumerate(scene['narration_timeline']):
        first=min(cue['end']-.05,cue['start']+.1);second=min(cue['end']-.03,cue['start']+2.5)
        a=visual.draw(first,duration);b=visual.draw(second,duration)
        changed=int(np.count_nonzero(np.any(np.asarray(a)!=np.asarray(b),axis=2)))
        if changed<100:raise RuntimeError('Slide 2 cue has no visible motion: '+cue['segment_id'])
        a.save(frames/f'{index+1:02d}_{cue["segment_id"]}_entrance.png')
        b.save(frames/f'{index+1:02d}_{cue["segment_id"]}_settled.png')
        checks.append(dict(segment=cue['segment_id'],type=cue['type'],team_id=cue.get('team_id'),player_id=cue.get('player_id'),fixture_ids=cue.get('fixture_ids'),start=cue['start'],end=cue['end'],changed_pixels=changed))
    target=VX_VIDEO_OUT_DIR/(key+'_GW'+str(contract['gameweek'])+'_slide2_fdr_animated.mp4')
    partial=VX_MEDIA_WORK/'wildcard_slide2_preview.partial.mp4'
    native_body_segment(visual,audio,duration,partial,profile)
    target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(partial,target)
    metadata=probe_media_info(target)
    report=dict(scope='Wildcard Slide 02 only; no other slides rendered or changed',duration_seconds=duration,profile=profile,metadata=metadata,voice=VX_TTS_VOICE,voice_settings=vx_narration_profile(dict(script,scene=scene)),voice_rate_changed_to_fit_duration=False,clock='Measured audio phrase durations; no guessed timestamps',timeline=scene['narration_timeline'],motion_checks=checks,data_snapshot=contract['source_snapshot_at'],source_hash=contract['data_hash'],data_checks=scene['elite_fixture_case']['verification'],featured_players=scene['elite_fixture_case']['elite_players'],hero_card_proof=scene['hero_card_asset_proof'],layout=dict(design_canvas=[1920,1080],table=[55,225,1135,976],analysis_stage=[1160,225,1865,976],hero_card=[1167,278,1652,971]))
    (VX_ANALYSIS_ROOT/'wildcard_slide2_verification.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    (VX_ANALYSIS_ROOT/'wildcard_slide2_narration.txt').write_text(scene['narration_text'],encoding='utf-8')
    (VX_ANALYSIS_ROOT/(key+'_slide2_script_and_cues.json')).write_text(json.dumps(dict(narration=scene['narration_text'],timeline=scene['narration_timeline'],ranking=scene['elite_fixture_case']['diagnostics']['ranking']),ensure_ascii=False,indent=2),encoding='utf-8')
    from IPython.display import Video,display
    display(Video(str(target),embed=True,width=960))
    print('Animated Wildcard Slide 2 complete:',target,flush=True)
    return target


def vx_render_freehit_slide2_preview():
    """Only Free Hit Slide 2; measured Ryan audio, unchanged Draft settings."""
    import copy,subprocess
    from wildcard_plan_audio import prepare_segment_audio
    key=next(k for k in VX_DATA_CONTRACTS if k.startswith('free_hit'))
    contract=VX_DATA_CONTRACTS[key];vx_validate_contract(contract);vx_assert_current_snapshot(contract)
    scene=copy.deepcopy(next(s for s in VX_SCENE_MANIFESTS[key] if s['visual_type']=='fixture_heatmap'))
    if not scene.get('matchday_case'):raise RuntimeError('Run Free Hit matchday Slide 2 setup first')
    script=dict(id=scene['scene_id'],text=scene['narration_text'],scene=scene)
    for cue in scene['narration_timeline']:vx_prepare_tts(dict(script,text=cue['narration_text'],narration_cue=cue))
    print('Free Hit Slide 2: measuring Ryan phrases with unchanged Slide 1 profile',VX_INTRO_TTS_PROFILE,flush=True)
    audio=prepare_segment_audio(scene,script,VX_MEDIA_WORK,
        lambda cue:_vx_run_async(_vx_wc_raw_audio_for_segment(cue)),
        lambda source,target:run_ffmpeg(['-i',source,'-ar','48000','-ac','2','-c:a','pcm_s16le',target]))
    duration=media_duration(audio)
    if not (math.isfinite(duration) and duration > 0):
        raise RuntimeError(f'Free Hit Slide 2 has an invalid measured narration duration: {duration!r}s.')
    profile=dict(VX_VIDEO_PROFILE)
    if VX_VIDEO_MODE=='Draft':profile.update(width=VX_VIDEO_PROFILE['width'],height=VX_VIDEO_PROFILE['height'],fps=VX_VIDEO_PROFILE['fps'],sar=VX_VIDEO_PROFILE['sar'])
    else:profile.update(width=1920,height=1080,fps=30,sar='1/1',crf=18)
    profile['static_preview']=False
    print(f"Free Hit Slide 2: {duration:.2f}s; {profile['width']}×{profile['height']} / {profile['fps']} FPS. Rendering Remotion matchday scenes.",flush=True)
    visual=vx_visual_from_manifest(scene,contract,profile['width'],profile['height'])
    VX_FREEHIT_MATCHDAY.prepare_project(scene,VX_FREEHIT_MATCHDAY_PROJECT,VX_PLAYER_MEDIA,VX_CLUB_MEDIA,VX_BROADCAST_ROOT,audio,profile['width'],profile['height'],profile['fps'],duration)
    if not (VX_FREEHIT_MATCHDAY_PROJECT/'node_modules'/'remotion').exists():
        subprocess.run(['npm','install','--no-audit','--no-fund'],cwd=VX_FREEHIT_MATCHDAY_PROJECT,check=True)
    VX_MASTERPLAN._vx_remotion_command_output(VX_FREEHIT_MATCHDAY_PROJECT, 'freehit_qa', ['node','qa.mjs'])
    frame_root=VX_FREEHIT_MATCHDAY_PROJECT/'qa_frames';checks=[]
    for i,cue in enumerate(scene['narration_timeline']):
        a=np.asarray(Image.open(frame_root/f'{i:02d}_entrance.png').convert('RGB'))
        b=np.asarray(Image.open(frame_root/f'{i:02d}_settled.png').convert('RGB'))
        changed=int(np.count_nonzero(np.any(a!=b,axis=2)))
        if changed<100:raise ValueError('No visible motion for '+cue['segment_id'])
        checks.append(dict(segment=cue['segment_id'],type=cue['type'],team_id=cue['team_id'],player_id=cue['player_id'],start=cue['start'],end=cue['end'],changed_pixels=changed))
    target=VX_VIDEO_OUT_DIR/(key+'_GW'+str(contract['gameweek'])+'_slide2_matchday_animated.mp4')
    target.parent.mkdir(parents=True,exist_ok=True)
    visual.render_video(audio,duration,target,profile)
    metadata=probe_media_info(target)
    report=dict(scope='Free Hit Slide 2 only',duration_seconds=duration,profile=profile,metadata=metadata,voice=VX_TTS_VOICE,voice_settings=VX_INTRO_TTS_PROFILE,voice_rate_changed_to_fit_duration=False,clock='Actual measured phrase audio durations',data_checks=scene['matchday_case']['verification'],timeline=scene['narration_timeline'],motion_checks=checks,source_snapshot_at=contract['source_snapshot_at'],source_hash=contract['data_hash'],players=scene['matchday_case']['featured_players'],captains=scene['matchday_case']['captains'],renderer='Remotion Matchday Control Room',source_project=str(VX_FREEHIT_MATCHDAY_PROJECT))
    (VX_ANALYSIS_ROOT/'freehit_slide2_verification.json').write_text(json.dumps(report,indent=2,ensure_ascii=False))
    (VX_ANALYSIS_ROOT/'freehit_slide2_narration.txt').write_text(scene['narration_text'])
    (VX_ANALYSIS_ROOT/'freehit_slide2_script_and_cues.json').write_text(json.dumps(dict(narration=scene['narration_text'],timeline=scene['narration_timeline']),indent=2,ensure_ascii=False))
    from IPython.display import Video,display
    display(Video(str(target),embed=True,width=960))
    print('Animated Free Hit Slide 2 complete:',target,flush=True)
    return target


def vx_render_freehit_intro_preview():
    import copy
    from freehit_personal_intro import build_case
    from wildcard_plan_audio import prepare_segment_audio
    key=next((name for name in VX_DATA_CONTRACTS if name.startswith('free_hit')),None)
    if key is None:raise ValueError('Select Free Hit or Both in Cell 0 before rendering its intro')
    contract=VX_DATA_CONTRACTS[key]
    vx_validate_contract(contract);vx_validate_freehit(contract);vx_assert_current_snapshot(contract)
    case=build_case(contract,_bootstrap)
    scene=copy.deepcopy(next(item for item in VX_SCENE_MANIFESTS[key] if item['visual_type']=='fh_blueprint'))
    scene.update(freehit_case=case,narration_timeline=case['narration_timeline'],narration_text=case['narration_text'])
    script=dict(id=scene['scene_id'],text=scene['narration_text'])
    audio=prepare_segment_audio(scene,script,VX_MEDIA_WORK,
        lambda cue:_vx_run_async(_vx_wc_audio_for_segment(cue)),
        lambda source,target:run_ffmpeg(['-i',source,'-ar','48000','-ac','2','-c:a','pcm_s16le',target]))
    duration=media_duration(audio)
    profile=dict(VX_VIDEO_PROFILE) if VX_VIDEO_MODE=='Draft' else dict(width=1920,height=1080,fps=30,sar='1/1',preset='fast',crf=18)
    if VX_VIDEO_MODE=='Draft':
        profile.update(width=VX_VIDEO_PROFILE['width'],height=VX_VIDEO_PROFILE['height'],fps=VX_VIDEO_PROFILE['fps'],sar=VX_VIDEO_PROFILE['sar'])
    profile.update(static_preview=False,scene_crossfades=True)
    visual=vx_visual_from_manifest(scene,contract,profile['width'],profile['height'])
    target=VX_VIDEO_OUT_DIR/(key+'_GW'+str(contract['gameweek'])+'_intro_animated.mp4')
    partial=VX_MEDIA_WORK/'freehit_intro_preview.partial.mp4'
    native_body_segment(visual,audio,duration,partial,profile)
    target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(partial,target)
    frames=VX_ANALYSIS_ROOT/'freehit_intro_frames';frames.mkdir(exist_ok=True);checks=[]
    for index,cue in enumerate(scene['narration_timeline']):
        entrance=visual.draw(cue['start']+.2,duration);settled=visual.draw(min(cue['end']-.05,cue['start']+2.5),duration)
        changed=int(np.count_nonzero(np.any(np.asarray(entrance)!=np.asarray(settled),axis=2)))
        if changed<max(100,int(profile['width']*profile['height']*.004)):raise RuntimeError('Free Hit opening lacks object motion: '+cue['segment_id'])
        entrance.save(frames/f'{index+1}_entrance.png');settled.save(frames/f'{index+1}_settled.png');checks.append(dict(scene=cue['segment_id'],changed_pixels=changed))
    report=dict(scope='Free Hit slide 1 only',duration_seconds=duration,profile=profile,metadata=probe_media_info(target),voice=VX_TTS_VOICE,voice_settings=vx_narration_profile(dict(script,scene=scene)),timeline=scene['narration_timeline'],motion_checks=checks,decision=case['verdict'],decision_reason=case['verdict_reason'],intro_context=case['intro_context'],chip_usage=case.get('chip_usage'),data_snapshot=contract['source_snapshot_at'],generated_at=contract['generated_at'])
    (VX_ANALYSIS_ROOT/'freehit_intro_verification.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
    (VX_ANALYSIS_ROOT/'freehit_intro_narration.txt').write_text(scene['narration_text'])
    from IPython.display import Video,display
    display(Video(str(target),embed=True,width=960))
    print('Animated Free Hit introduction complete:',target,flush=True)
    return target

VX_VIDEO_PATHS={};VX_VIDEO_ERRORS={}
VX_DELIVERY_REPORT = {}
if VX_RENDER_VIDEOS and not globals().get('VX_INTRO_PREVIEW_ONLY',False):
    # Assets were resolved before Wildcard verification; preserve that exact template.
    selected_ids=set().union(*(set(p.player_id.astype(int)) for p in [VX_WILDCARD_AUTO_SQUAD,VX_FREEHIT_AUTO_SQUAD,VX_WILDCARD_MANUAL_SQUAD,VX_FREEHIT_MANUAL_SQUAD] if p is not None))
    for a in globals().get('VX_WILDCARD_FUTURE_PLAN',[]):
        if a.get('in_player_id'):selected_ids.add(int(a['in_player_id']))
        if a.get('out_player_id'):selected_ids.add(int(a['out_player_id']))
    for contract in VX_DATA_CONTRACTS.values():selected_ids.update(map(int,contract.get('media_entities',{})))
    selected_ids &= set(VX_POOL.player_id.astype(int))
    _render_targets = []
    if VX_GENERATE_WILDCARD:
        if USE_MANUAL_WILDCARD:
            _render_targets.append(('wildcard_manual', VX_WILDCARD_MANUAL_PLAN, 'Wildcard / Manual'))
        else:
            _render_targets.append(('wildcard_auto', VX_WILDCARD_AUTO_PLAN, 'Wildcard / Model'))
    if VX_GENERATE_FREEHIT:
        if USE_MANUAL_FREEHIT:
            _render_targets.append(('free_hit_manual', VX_FREEHIT_MANUAL_PLAN, 'Free Hit / Manual'))
        else:
            _render_targets.append(('free_hit_auto', VX_FREEHIT_AUTO_PLAN, 'Free Hit / Model'))
    # Check every spoken cue before any selected movie starts rendering.
    _vx_checked_cues=0
    for _vx_target_key, _vx_plan, _vx_label in _render_targets:
        _vx_scripts=decode_script(VX_NARRATION_PATHS[_vx_target_key].read_text(encoding='utf-8'))
        _vx_scenes={scene['scene_id']:scene for scene in VX_SCENE_MANIFESTS[_vx_target_key]}
        for _vx_script in _vx_scripts:
            _vx_scene=_vx_scenes[_vx_script['id']]
            _vx_cues=_vx_scene.get('narration_timeline') or []
            if _vx_cues:
                for _vx_cue in _vx_cues:
                    vx_prepare_tts(dict(_vx_script,text=_vx_cue['narration_text'],scene=_vx_scene,narration_cue=_vx_cue))
                    _vx_checked_cues+=1
            else:
                vx_prepare_tts(dict(_vx_script,scene=_vx_scene))
                _vx_checked_cues+=1
    print(f'Pronunciation preflight passed for {_vx_checked_cues} narration cues.',flush=True)
    print(f'Render stage: {VX_VIDEO_MODE}, {VX_VIDEO_PROFILE["width"]}×{VX_VIDEO_PROFILE["height"]}, {VX_VIDEO_PROFILE["fps"]} fps; each slide will appear as it finishes',flush=True)
    for key,plan,label in _render_targets:
        if not isinstance(plan, pd.DataFrame) or plan.empty or key not in VX_NARRATION_PATHS:
            continue
        try:VX_VIDEO_PATHS[key]=vx_render_squad_video(plan,key,label,VX_ASSET_PATHS)
        except Exception as exc:
            import traceback
            traceback.print_exc()
            VX_VIDEO_ERRORS[key]=f'{type(exc).__name__}: {exc}'
            print(f'Video failed for {key}: {VX_VIDEO_ERRORS[key]}. Narration TXT is preserved.')
    if len(VX_VIDEO_PATHS)!=len(_render_targets) and not VX_VIDEO_ERRORS:raise RuntimeError('Not all selected video targets were rendered')
    if VX_VIDEO_ERRORS:raise RuntimeError('Video render failures: ' + '; '.join(f'{key}: {message}' for key,message in VX_VIDEO_ERRORS.items()) + '. See the traceback above; rerun Cell 14 after fixing.')
elif not VX_RENDER_VIDEOS:print('Scripts-only run complete. Edit your TXT, then enable VX_RENDER_VIDEOS and rerun this cell.')
if VX_RENDER_VIDEOS and globals().get('VX_INTRO_PREVIEW_ONLY',False):
    if globals().get('VX_WILDCARD_SLIDE3_PREVIEW_ONLY',False) and VX_GENERATE_WILDCARD:
        VX_WILDCARD_SLIDE3_PREVIEW_PATH=vx_render_wildcard_slide3_preview()
    elif globals().get('VX_FREEHIT_SLIDE2_PREVIEW_ONLY',False) and VX_GENERATE_FREEHIT:
        VX_FREEHIT_SLIDE2_PREVIEW_PATH=vx_render_freehit_slide2_preview()
    elif globals().get('VX_WILDCARD_SLIDE2_PREVIEW_ONLY',False) and VX_GENERATE_WILDCARD:
        VX_WILDCARD_SLIDE2_PREVIEW_PATH=vx_render_wildcard_slide2_preview()
    else:
        VX_INTRO_PREVIEW_PATH=vx_render_freehit_intro_preview() if globals().get('VX_INTRO_PREVIEW_CHIP','Wildcard')=='Free Hit' else vx_render_intro_preview()


if VX_RENDER_VIDEOS and not VX_INTRO_PREVIEW_ONLY:
    # Both quality profiles export full-resolution stills from measured slide data.
    VX_4K_PNG_REPORTS = {}
    if VX_SAVE_4K_PNG:
        for key in VX_VIDEO_PATHS:
            contract = VX_DATA_CONTRACTS[key]
            measured_scenes = json.loads((VX_ANALYSIS_ROOT / (key + '_scene_manifest.json')).read_text(encoding='utf-8'))
            VX_4K_PNG_REPORTS[key] = vx_export_4k_png_slides(key, contract, measured_scenes)

    # Verify actual persistent files on Google Drive, not temporary Colab work files.
    VX_DELIVERY_REPORT = {}
    for key, final_path in VX_VIDEO_PATHS.items():
        contract = VX_DATA_CONTRACTS[key]
        final_path = Path(final_path)
        if not final_path.is_file() or not vx_valid_media_cache(final_path):
            raise RuntimeError(f'{key}: final video missing or invalid: {final_path}')
        measured_scenes = json.loads((VX_ANALYSIS_ROOT / (key + '_scene_manifest.json')).read_text(encoding='utf-8'))
        slide_videos = [Path(scene['media_file']) for scene in measured_scenes]
        for video_path in slide_videos:
            if not video_path.is_file() or not vx_valid_media_cache(video_path):
                raise RuntimeError(f'{key}: per-slide MP4 missing or invalid: {video_path}')
        pngs = [Path(record['path']) for record in VX_4K_PNG_REPORTS.get(key, [])]
        txts = vx_save_slide_scripts(key, contract, measured_scenes)
        combined_txt = vx_save_combined_script(key, contract)
        for video_path in [final_path, *slide_videos]:
            vx_verify_delivery_mp4(video_path,VX_VIDEO_PROFILE)
        audio_root=VX_AUDIO_OUT_DIR/key/VX_VIDEO_MODE.lower()
        slide_mp3s=[vx_export_narration_mp3(path,audio_root/(vx_delivery_slide_name(key,index)+'.mp3')) for index,path in enumerate(slide_videos,1)]
        combined_mp3=vx_export_narration_mp3(final_path,audio_root/(f'{"Wildcard" if key.startswith("wildcard") else "Free Hit"} Combined GW{contract["gameweek"]}.mp3'))
        if VX_SAVE_4K_PNG and len(pngs) != len(measured_scenes):
            raise RuntimeError(f'{key}: not all {len(measured_scenes)} 4K PNGs were exported')
        for text_path in txts:
            if not text_path.is_file() or not text_path.read_text(encoding='utf-8').strip():
                raise RuntimeError(f'{key}: slide script missing or empty: {text_path}')
        for image_path in pngs:
            if not image_path.is_file():
                raise RuntimeError(f'{key}: 4K PNG missing: {image_path}')
            with Image.open(image_path) as verified_image:
                if verified_image.size != (3840, 2160):
                    raise RuntimeError(f'{key}: PNG is not 4K: {image_path}')
        VX_DELIVERY_REPORT[key] = {
            'gameweek': contract['gameweek'],
            'mode': VX_VIDEO_MODE,
            'final_mp4': str(final_path),
            'individual_mp4s': [str(path) for path in slide_videos],
            'slide_pngs_4k': [str(path) for path in pngs],
            'slide_txts': [str(path) for path in txts],
            'slide_mp3s': [str(path) for path in slide_mp3s],
            'combined_mp3': str(combined_mp3),
            'combined_txt': str(combined_txt),
            'scene_count': len(measured_scenes),
        }
        print(f'✅ {key}: combined + {len(slide_videos)} slide MP4s, combined + {len(slide_mp3s)} MP3s, combined + {len(txts)} TXT scripts, {len(pngs)} final 4K PNGs', flush=True)
    expected_keys = {key for key, plan, label in _render_targets}
    if set(VX_DELIVERY_REPORT) != expected_keys:
        raise RuntimeError(f'Delivery incomplete; expected {expected_keys}, found {set(VX_DELIVERY_REPORT)}')
    delivery_index = VX_DELIVERY_ROOT / 'delivery_manifest.json'
    temporary_index = delivery_index.with_suffix('.partial.json')
    temporary_index.write_text(json.dumps(VX_DELIVERY_REPORT, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary_index.replace(delivery_index)
    print(f'✅ ALL OUTPUTS VERIFIED. Open: {delivery_index}', flush=True)

if VX_RENDER_VIDEOS and not VX_INTRO_PREVIEW_ONLY and VX_GENERATE_FREEHIT and VX_VIDEO_MODE=='Final':
        for key,contract in VX_DATA_CONTRACTS.items():
            if contract['chip']!='freehit':continue
            output=VX_MEDIA_WORK/'qa_previews'/key;output.mkdir(parents=True,exist_ok=True);proof=[]
            scenes=json.loads((VX_ANALYSIS_ROOT/(key+'_scene_manifest.json')).read_text(encoding='utf-8'))
            selected=scenes
            for scene in selected:
                width,height=VX_VIDEO_PROFILE['width'],VX_VIDEO_PROFILE['height']
                visual=vx_visual_from_manifest(scene,contract,width,height)
                _vx_preview_times=(26,) if VX_VIDEO_PROFILE.get('static_preview') else (0,13.5,26)
                for at in _vx_preview_times:
                    test_image=visual.draw(at,27);vx_check_text_layout(test_image)
                    frame=visual.frame(at,27)
                    if frame.shape[:2]!=(height,width):raise RuntimeError('Free Hit preview dimensions are invalid')
                image=output/(scene['scene_id']+'.png');visual.complete.convert('RGB').save(image)
                proof.append({'scene_id':scene['scene_id'],'output':str(image),'dimensions':[width,height],'passed':True})
            (output/'visual_validation.json').write_text(json.dumps(proof,indent=2))
            print(f'✅ {key}: {len(proof)} Free Hit preview slides rendered')
print("✅ CELL 14 COMPLETE")

In [ ]:
#@title 🔴 CELL 15 - SCENE DATA EXPORT {display-mode: "form"}

def vx_require_scene_contracts(cell_name):
    contracts = globals().get('VX_DATA_CONTRACTS')
    scenes = globals().get('VX_SCENE_MANIFESTS')
    narration = globals().get('VX_NARRATION_PATHS')
    if (not globals().get('VX_CELL2_COMPLETE', False)
            or not isinstance(contracts, dict) or not contracts
            or not isinstance(scenes, dict) or set(contracts) != set(scenes)
            or not isinstance(narration, dict) or set(contracts) != set(narration)):
        raise RuntimeError(
            f'Cell 15 requires a successful Cell 12 scene-and-narration run. '
            'Finish Cells 11 and 12 in order before rendering or exporting.'
        )
    return True

vx_require_scene_contracts("Cell 15")

# 🔴 CELL 15 - SCENE DATA EXPORT
"""
CELL 12 — EXPORT SCENE DATA FOR THE PITCH VIEWER

Packages only the chip target(s) enabled in Cell 0 — auto and manual where
available — into one JSON structure the pitch-viewer artifact
consumes. One pitch view, GW selector tabs switch which XI/captain/bench is
shown, matching the Day 3 spec: real animated state change per GW, driven
directly from this live data.
"""

import json
from pathlib import Path
import numpy as np
import pandas as pd


def _vx_row_for_pitch(row, pool=VX_POOL):
    pid = int(row["player_id"])
    meta = pool.loc[pool["player_id"] == pid]
    web = meta.iloc[0]["player_name"] if not meta.empty else f"#{pid}"
    team = meta.iloc[0]["team_name"] if not meta.empty and "team_name" in meta.columns else ""
    return {
        "player_id": pid,
        "name": web,
        "team": team,
        "position": str(row.get("position", "")).upper(),
        "score": round(float(row.get("score", row.get(next((c for c in row.index if "xpts" in c.lower()), "score"), 0.0)) or 0.0), 2)
        if hasattr(row, "index") else 0.0,
    }


def _vx_plan_to_scenes(plan_rows_frame, squad_frame, label):
    """plan_rows_frame: the per-GW dict rows produced by vx_static_plan_score
    (contains 'xi' / 'bench' sub-DataFrames plus scalar summary fields)."""
    scenes = []
    if not isinstance(plan_rows_frame, pd.DataFrame) or plan_rows_frame.empty:
        return scenes
    for _, r in plan_rows_frame.iterrows():
        xi_df = r["xi"]
        bench_df = r["bench"] if isinstance(r["bench"], pd.DataFrame) else pd.DataFrame()
        xi_players = []
        for _, xr in xi_df.iterrows():
            xi_players.append({
                "player_id": int(xr["player_id"]),
                "name": xr.get("player_name", f"#{int(xr['player_id'])}"),
                "position": str(xr.get("position", "")).upper(),
                "score": round(float(xr.get("score", 0.0)), 2),
                "is_captain": int(xr["player_id"]) == int(r["captain_player_id"]),
            })
        bench_players = []
        for _, br in bench_df.iterrows():
            bench_players.append({
                "player_id": int(br["player_id"]),
                "name": br.get("player_name", f"#{int(br['player_id'])}"),
                "position": str(br.get("position", "")).upper(),
                "score": round(float(br.get("score", 0.0)), 2),
            })
        scenes.append({
            "gameweek": int(r["gameweek"]),
            "label": label,
            "xi": xi_players,
            "bench": bench_players,
            "captain_name": r.get("captain_name"),
            "vice_captain_name": r.get("vice_captain_name"),
            "formation": r.get("formation"),
            "captain_incremental_xpts": round(float(r["captain_incremental_xpts"]), 2),
            "xi_xpts": round(float(r["xi_xpts"]), 2),
            "xi_plus_captain_xpts": round(float(r["xi_plus_captain_xpts"]), 2),
            "bench_xpts": None if pd.isna(r["bench_xpts"]) else round(float(r["bench_xpts"]), 2),
        })
    return scenes


VX_SCENE_EXPORT = {
    "current_gw": VX_CURRENT_GW,
    "future_gws": VX_FUTURE_GWS,
    "budget": round(VX_BUDGET, 1),
    "wildcard_working_budget": round(float(globals().get("VX_WILDCARD_BUDGET", VX_BUDGET)), 1),
    "freehit_working_budget": round(float(globals().get("VX_FREEHIT_BUDGET", VX_BUDGET)), 1),
    "wildcard": {
        "auto": _vx_plan_to_scenes(VX_WILDCARD_AUTO_PLAN, VX_WILDCARD_AUTO_SQUAD, "Model"),
        "manual": _vx_plan_to_scenes(VX_WILDCARD_MANUAL_PLAN, VX_WILDCARD_MANUAL_SQUAD, "Manual")
                  if VX_WILDCARD_MANUAL_SQUAD is not None else [],
    },
    "free_hit": {
        "auto": _vx_plan_to_scenes(VX_FREEHIT_AUTO_PLAN, VX_FREEHIT_AUTO_SQUAD, "Model"),
        "manual": _vx_plan_to_scenes(VX_FREEHIT_MANUAL_PLAN, VX_FREEHIT_MANUAL_SQUAD, "Manual")
                  if VX_FREEHIT_MANUAL_SQUAD is not None else [],
    },
    "narration": VX_NARRATION,
}

# Full-precision canonical Wildcard facts accompany legacy pitch-export fields.
VX_SCENE_EXPORT['wildcard']['canonical_contracts']={k:v for k,v in VX_DATA_CONTRACTS.items() if k.startswith('wildcard')}
VX_SCENE_EXPORT['wildcard']['manifests']={k:v for k,v in VX_SCENE_MANIFESTS.items() if k.startswith('wildcard')}
VX_SCENE_EXPORT_PATH = VX_INPUT_ROOT / "wildcard_freehit_scene_export.json"
VX_SCENE_EXPORT_PATH.write_text(json.dumps(VX_SCENE_EXPORT, indent=2, default=str), encoding="utf-8")
print(f"✅ Scene export written: {VX_SCENE_EXPORT_PATH}")
print(f"   Wildcard auto scenes: {len(VX_SCENE_EXPORT['wildcard']['auto'])}")
print(f"   Wildcard manual scenes: {len(VX_SCENE_EXPORT['wildcard']['manual'])}")
print(f"   Free Hit auto scenes: {len(VX_SCENE_EXPORT['free_hit']['auto'])}")
print(f"   Free Hit manual scenes: {len(VX_SCENE_EXPORT['free_hit']['manual'])}")

# Print the JSON so it can be copy-pasted straight into the pitch-viewer
# artifact's embedded data if you're not running this notebook + artifact
# in the same session.
print("\n--- SCENE_DATA (paste into pitch viewer if needed) ---")
print(json.dumps(VX_SCENE_EXPORT, indent=2, default=str)[:2000] + "\n... (truncated preview; full file written to disk)")
print("✅ CELL 15 COMPLETE")

In [ ]:
#@title 🔴 CELL 16 - INTERACTIVE SQUAD VIEWER {display-mode: "form"}

def vx_require_scene_contracts(cell_name):
    contracts = globals().get('VX_DATA_CONTRACTS')
    scenes = globals().get('VX_SCENE_MANIFESTS')
    narration = globals().get('VX_NARRATION_PATHS')
    if (not globals().get('VX_CELL2_COMPLETE', False)
            or not isinstance(contracts, dict) or not contracts
            or not isinstance(scenes, dict) or set(contracts) != set(scenes)
            or not isinstance(narration, dict) or set(contracts) != set(narration)):
        raise RuntimeError(
            f'Cell 16 requires a successful Cell 12 scene-and-narration run. '
            'Finish Cells 11 and 12 in order before rendering or exporting.'
        )
    return True

vx_require_scene_contracts("Cell 16")

# 🔴 CELL 16 - INTERACTIVE SQUAD VIEWER
if VX_VIDEO_MODE=="Draft":
    print("Fast Draft: optional interactive HTML viewer skipped to conserve Colab memory")
else:
    VX_PLANNER_TEMPLATE = '<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>FPL Squad Planner</title>\n<style>\n*{box-sizing:border-box}body{font:16px Arial,sans-serif;color:#161b1e;background:#fff;margin:0;padding:12px}button,input,select{font:inherit}button,select{background:#fff;border:1px solid #dce2e5;border-radius:11px;padding:10px;cursor:pointer}button:hover{border-color:#007e9c}button:focus-visible,input:focus-visible,select:focus-visible{outline:3px solid #007e9c}.weekbar{display:flex;gap:12px;overflow:auto;margin-bottom:18px}.week{min-width:125px;padding:13px;color:#6e8187;font-weight:bold}.week.active{border:2px solid #007e9c;background:#f3f8fa;color:#007e9c}.week span{display:block;margin-top:9px;color:#222;font-size:14px}.layout{display:grid;grid-template-columns:minmax(370px,37fr) minmax(600px,63fr);gap:18px;align-items:start}.panel{border:1px solid #dde4e7;border-radius:15px;overflow:hidden}.filters{padding:16px}.searchrow{display:flex;gap:8px}#search{min-width:0;flex:1;border:1px solid #dae2e6;border-radius:10px;padding:12px}.ranges{display:flex;gap:16px;margin:12px 0}.ranges label{width:50%;color:#6e8187;font-size:14px;font-weight:bold}.ranges b{float:right;color:#222}input[type=range]{width:100%;accent-color:#00839e;margin:12px 0 3px}.tabs{display:flex;gap:5px;align-items:center;flex-wrap:wrap}.tabs button{padding:7px 11px;color:#6c7f85;border:0}.tabs button.active{background:#00829d;color:white}.tabs select{padding:6px;max-width:160px}.columns{padding-top:12px;display:none}.columns.open{display:flex;gap:14px;flex-wrap:wrap}.scroll{overflow:auto}table{border-collapse:collapse;width:100%;white-space:nowrap}thead{background:#f1f3f4}th{padding:0;text-align:left;color:#222}th button{width:100%;text-align:left;border:0;border-radius:0;background:transparent;padding:14px 8px;font-weight:bold}th button .arrow{margin-left:5px;color:#00829d}td{padding:9px 8px;border-top:1px solid #e1e7ea}tr.selected{background:#e7f2f5}.identity{display:flex;align-items:center;gap:8px}.headshot{height:35px;width:35px;object-fit:contain;border-radius:8px;background:#e6edf1}.identity strong{font-size:16px;display:block}.identity small{color:#708189;font-size:12px}.warning{color:#b98800}.fixtures{display:flex;gap:4px}.fx{display:inline-flex;min-width:34px;flex-direction:column;align-items:center;justify-content:center;padding:4px 5px;border-radius:4px;font-size:12px;font-weight:bold;line-height:1.15}.fx small{font-size:11px;margin-top:2px}.footer{padding:12px 16px;display:flex;align-items:center;justify-content:space-between;color:#6e8187;font-size:14px}.footer button{padding:5px 12px}.righthead{display:flex;justify-content:space-between;align-items:center;padding:14px 18px;border-bottom:1px solid #dfe4e7}.righthead h2{font-size:20px;margin:0}.switch button{padding:6px 12px}.switch .active{background:#eef4f6;color:#007f9b}.stats{display:grid;grid-template-columns:repeat(3,1fr);margin:16px;background:#f1f3f4;border-radius:13px;padding:14px;text-align:center}.stats>div+div{border-left:1px solid #e3e7e9}.stats strong{font-size:26px;display:block}.stats small{display:block;color:#6d8087;margin-top:5px}.bank{color:#007854}.pitch{margin:16px;background:linear-gradient(110deg,#00896c,#00ce62 70%,#00a16c);border-radius:12px;position:relative;overflow:hidden}.field{position:relative;aspect-ratio:1.92}.marks{position:absolute;inset:2.5% 1.5%;border:3px solid white;border-radius:9px}.half{position:absolute;left:0;right:0;top:50%;height:2px;background:#fff}.circle{position:absolute;left:50%;top:50%;width:11%;aspect-ratio:1;border:3px solid white;border-radius:50%;transform:translate(-50%,-50%)}.goal{position:absolute;left:45.5%;width:9%;height:9%;border:3px solid #ffffffb0}.goal.top{top:0}.goal.bottom{bottom:0}.brand{position:absolute;right:3%;top:3%;max-width:9%;max-height:11%;object-fit:contain}.token{position:absolute;transform:translate(-50%,-50%);width:12%;height:22%;background:#3b945bc4;border-radius:12px;overflow:hidden;box-shadow:0 4px 8px #00563325}.token .portrait{width:100%;height:57%;object-fit:contain;object-position:center bottom;display:block}.token .badge{position:absolute;top:5%;right:5%;width:22%;height:25%;object-fit:contain}.namebar{height:20%;background:#fff;color:#46004c;display:flex;align-items:center;justify-content:center;padding:0 3%;font-weight:bold;font-size:clamp(9px,1.15vw,19px);line-height:1.1;text-align:center}.flagged .namebar{background:#ffdb00;color:#222}.fixturebar{height:23%;display:flex}.fixturebar .fx{flex:1;padding:1px;border-radius:0;min-width:0;font-size:clamp(8px,1vw,16px)}.fixturebar .fx small{font-size:clamp(8px,.9vw,14px);margin:0}.captain{position:absolute;top:4%;left:4%;width:20%;aspect-ratio:1;border-radius:50%;display:grid;place-items:center;background:#000;color:white;font-weight:bold;font-size:clamp(8px,1vw,17px)}.bench{position:relative;height:clamp(105px,10vw,200px);border-top:2px solid #fff9}.bench .token{height:86%;width:12%;top:50%}.chips{display:flex;gap:10px;margin:12px 16px}.chips button{flex:1;color:#007f9b}.chips button.active{background:#e9f5f9;border:2px solid #00829d}.note{margin:0 16px 16px;color:#708189;font-size:12px}.pitchtable{margin:16px}.hidden{display:none!important}.empty{padding:20px;color:#6c7f85}@media(max-width:1050px){.layout{grid-template-columns:1fr}.token .namebar{font-size:clamp(10px,1.6vw,18px)}.fixturebar .fx{font-size:clamp(9px,1.5vw,16px)}.fixturebar .fx small{font-size:clamp(9px,1.3vw,14px)}.bench{height:clamp(105px,15vw,200px)}}\n\n</style>\n<div class="weekbar" id="weeks" aria-label="Gameweeks"></div><div class="layout"><section class="panel"><div class="filters"><div class="searchrow"><input id="search" aria-label="Search players" placeholder="Search players…"><button id="reset">↶ Reset</button></div><div class="ranges"><label>Max Ownership <b id="ownValue"></b><input id="ownership" type="range" min="0" max="100" value="100" step=".1"></label><label>Max Price <b id="priceValue"></b><input id="price" type="range" min="0" step=".1"></label></div><div class="tabs" id="positions"></div><div class="columns" id="columns"></div></div><div class="scroll"><table><thead><tr id="headers"></tr></thead><tbody id="players"></tbody></table></div><div class="footer"><span id="count"></span><span><span id="pages"></span> <button id="prev" aria-label="Previous page">‹</button> <button id="next" aria-label="Next page">›</button></span></div></section><section class="panel"><div class="righthead"><h2 id="gwtitle"></h2><div class="switch"><button id="pitchToggle" class="active">Pitch</button><button id="tableToggle">Table</button></div></div><div class="stats"><div><strong id="formation"></strong><small>Formation</small></div><div><strong id="xiPoints"></strong><small>XI expected points</small></div><div><strong class="bank" id="bank"></strong><small>Bank</small></div></div><div class="pitch" id="pitch"><div class="field" id="field"><div class="marks"><div class="half"></div><div class="circle"></div><div class="goal top"></div><div class="goal bottom"></div></div></div><div class="bench" id="bench" aria-label="Bench, goalkeeper then substitution order"></div></div><div class="pitchtable hidden" id="pitchtable"></div><div class="chips" id="chips"></div><p class="note">Live squad and fixture data. Fixture value below opponent [home/away] is FDR. Pts = actual season points; xPts = model forecast. Bench: goalkeeper, then substitution order.</p></section></div>\n<style>\n.player-open{border:0;background:transparent;color:inherit;padding:0;text-align:left}.player-open:hover strong{color:#0085a4}.token[role=button]{cursor:pointer}.token[role=button]:focus-visible{outline:3px solid #ffdf4f;outline-offset:3px}\n/* Premium 2:3 sports-card tokens shared by every interactive player modal. */\n\n</style>\n<script>\n\'use strict\';const DATA=__PLANNER_DATA__;\nconst el=id=>document.getElementById(id),esc=v=>String(v??\'\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\nconst byId=new Map(DATA.players.map(p=>[p.id,p]));const definitions=[[\'name\',\'Name\'],[\'fixtures\',\'Fixtures\'],[\'price\',\'£\'],[\'ownership\',\'%\'],[\'points\',\'Pts\'],[\'expected\',\'xPts\'],[\'net\',\'Net GW\'],[\'priceChange\',\'GW Δ £\']];\nlet mode=DATA.modes[0],gw=mode.gw,position=\'All\',team=\'All\',sortKey=\'points\',ascending=false,page=0;const size=14,visible=new Set(definitions.slice(0,6).map(x=>x[0]));\nfunction fixtures(p){return DATA.weeks.filter(g=>g>=gw).slice(0,5).map(g=>({gw:g,rows:p.fixtures[g]||[]}))}\nfunction fixtureValue(p){const f=fixtures(p).flatMap(w=>w.rows).map(f=>f.fdr).filter(v=>v!==null&&v!==undefined);return f.length?f.reduce((a,b)=>a+b,0)/f.length:null}\nfunction value(p,key){if(key===\'fixtures\')return fixtureValue(p);if(key===\'expected\')return p.expected[gw];return p[key]}\nfunction comparePlayers(a,b,key=sortKey,asc=ascending){const av=value(a,key),bv=value(b,key);if(av===null||av===undefined)return bv===null||bv===undefined?a.id-b.id:1;if(bv===null||bv===undefined)return -1;const c=typeof av===\'string\'?av.localeCompare(bv,undefined,{sensitivity:\'base\',numeric:true}):av-bv;return (asc?c:-c)||a.id-b.id}\nfunction filtered(){const query=el(\'search\').value.trim().toLocaleLowerCase();return DATA.players.filter(p=>(!query||(p.fullName+\' \'+p.name+\' \'+p.teamName).toLocaleLowerCase().includes(query))&&(position===\'All\'||p.position===position)&&(team===\'All\'||String(p.team)===team)&&p.price<=Number(el(\'price\').value)&&p.ownership<=Number(el(\'ownership\').value)).sort(comparePlayers)}\nfunction fxHTML(f){let bg=\'#f0f1f2\',fg=\'#111\';if(f.fdr!=null){if(f.fdr<=2.5)bg=\'#00f875\';else if(f.fdr>4.5){bg=\'#850027\';fg=\'white\'}else if(f.fdr>3.5){bg=\'#ff004c\';fg=\'white\'}}const label=f.opponent+(f.home===true?\' [H]\':f.home===false?\' [A]\':\'\');return `<span class="fx" style="background:${bg};color:${fg}" title="${esc(f.opponent)} ${f.home?\'home\':\'away\'} • FDR ${esc(f.fdr??\'unavailable\')}">${esc(label)}<small>${esc(f.fdr??\'—\')}</small></span>`}\nfunction fixtureHTML(p){return fixtures(p).map(w=>w.rows.length?w.rows.map(fxHTML).join(\'\'):\'<span class="fx" title="Gameweek \'+w.gw+\'">—</span>\').join(\'\')}\nfunction identity(p){return `<div class="player-open"><div class="identity">${playerPhoto(p,"headshot")}<div><strong>${p.flagged?\'<span class="warning">⚠ </span>\':\'\'}${esc(p.name)}</strong><small>${esc(p.club)} · ${esc(p.position)}</small></div></div></div>`}\nfunction currentPlan(){return mode.plans.find(p=>Number(p.gameweek)===gw)||mode.plans[0]}\nfunction renderTable(){const rows=filtered(),n=Math.max(1,Math.ceil(rows.length/size));page=Math.max(0,Math.min(page,n-1));const selected=new Set([...currentPlan().xi_ids,...currentPlan().bench_ids]);el(\'headers\').innerHTML=definitions.filter(([k])=>visible.has(k)).map(([k,label])=>`<th aria-sort="${sortKey===k?(ascending?\'ascending\':\'descending\'):\'none\'}"><button data-sort="${k}">${label}<span class="arrow">${sortKey===k?(ascending?\'↑\':\'↓\'):\'↕\'}</span></button></th>`).join(\'\');el(\'headers\').querySelectorAll(\'button\').forEach(b=>b.onclick=()=>{ascending=sortKey===b.dataset.sort?!ascending:b.dataset.sort===\'name\'||b.dataset.sort===\'fixtures\';sortKey=b.dataset.sort;page=0;renderTable()});el(\'players\').innerHTML=rows.slice(page*size,(page+1)*size).map(p=>`<tr data-player-id="${p.id}" class="${selected.has(p.id)?\'selected\':\'\'}">${definitions.filter(([k])=>visible.has(k)).map(([k])=>`<td>${k===\'name\'?identity(p):k===\'fixtures\'?\'<div class="fixtures">\'+fixtureHTML(p)+\'</div>\':k===\'price\'?\'£\'+p.price.toFixed(1)+\'M\':k===\'ownership\'?p.ownership.toFixed(1)+\'%\':k===\'expected\'?(p.expected[gw]??null)===null?\'—\':p.expected[gw].toFixed(1):k===\'net\'?(p.net>0?\'+\':\'\')+p.net.toLocaleString():k===\'priceChange\'?(p.priceChange>0?\'+\':\'\')+\'£\'+p.priceChange.toFixed(1)+\'m\':p.points}</td>`).join(\'\')}</tr>`).join(\'\')||`<tr><td colspan="${visible.size}" class="empty">No matching players</td></tr>`;el(\'count\').textContent=rows.length+\' players\';el(\'pages\').textContent=\'Page \'+(page+1)+\' of \'+n;el(\'prev\').disabled=page===0;el(\'next\').disabled=page===n-1;el(\'ownValue\').textContent=Number(el(\'ownership\').value).toFixed(1)+\'%\';el(\'priceValue\').textContent=\'£\'+Number(el(\'price\').value).toFixed(1)+\'M\'}\nfunction token(p,x,y){const f=p.fixtures[gw]||[],captain=p.id===mode.captain?\'C\':p.id===mode.vice?\'V\':\'\';return `<div class="token ${p.flagged?\'flagged\':\'\'}" style="left:${x}%;top:${y}%" title="${esc(p.fullName)} • £${p.price.toFixed(1)}m • ${p.expected[gw]?.toFixed(1)??\'—\'} expected points">${playerPhoto(p,"portrait")}${DATA.badges[p.club]?`<img class="badge" src="${DATA.badges[p.club]}" alt="${esc(p.teamName)}">`:\'\'}${captain?`<span class="captain">${captain}</span>`:\'\'}<div class="namebar">${esc(p.name)}</div><div class="fixturebar">${f.length?f.map(fxHTML).join(\'\'):\'<span class="fx" style="flex:1;background:#f4f5f5">BLANK<small>—</small></span>\'}</div></div>`}\nfunction renderPitch(){const plan=currentPlan(),xi=plan.xi_ids.map(id=>byId.get(id)),bench=plan.bench_ids.map(id=>byId.get(id)).sort((a,b)=>Number(a.position!==\'GK\')-Number(b.position!==\'GK\'));el(\'field\').querySelectorAll(\'.token,.brand\').forEach(n=>n.remove());[[\'GK\',14],[\'DEF\',38],[\'MID\',62],[\'FWD\',86]].forEach(([pos,y])=>{const group=xi.filter(p=>p.position===pos);group.forEach((p,i)=>el(\'field\').insertAdjacentHTML(\'beforeend\',token(p,100*(i+1)/(group.length+1),y)))});if(DATA.logo)el(\'field\').insertAdjacentHTML(\'beforeend\',`<img class="brand" src="${DATA.logo}" alt="Brand logo">`);el(\'bench\').innerHTML=bench.map((p,i)=>token(p,100*(i+.5)/bench.length,50)).join(\'\');el(\'gwtitle\').textContent=\'Gameweek \'+gw;el(\'formation\').textContent=[\'DEF\',\'MID\',\'FWD\'].map(pos=>xi.filter(p=>p.position===pos).length).join(\'–\');const points=xi.map(p=>p.expected[gw]);el(\'xiPoints\').textContent=points.every(v=>v!=null)?points.reduce((a,b)=>a+b,0).toFixed(1):\'—\';el(\'bank\').textContent=\'£\'+(mode.budget-mode.used).toFixed(1)+\'m\';el(\'pitchtable\').innerHTML=\'<table><thead><tr><th>Player</th><th>Position</th><th>xPts</th><th>Fixture</th></tr></thead><tbody>\'+[...xi,...bench].map(p=>\'<tr><td>\'+identity(p)+\'</td><td>\'+p.position+\'</td><td>\'+(p.expected[gw]?.toFixed(1)??\'—\')+\'</td><td>\'+((p.fixtures[gw]||[]).map(fxHTML).join(\'\')||\'—\')+\'</td></tr>\').join(\'\')+\'</tbody></table>\';el(\'weeks\').innerHTML=mode.plans.map(p=>`<button class="week ${p.gameweek===gw?\'active\':\'\'}" data-gw="${p.gameweek}">GW ${p.gameweek}<span>${mode.ft==null?\'FT not supplied\':p.gameweek===mode.gw?mode.ft+\' FT\':\'FT not projected\'}</span></button>`).join(\'\');el(\'weeks\').querySelectorAll(\'button\').forEach(b=>b.onclick=()=>{gw=Number(b.dataset.gw);page=0;renderPitch();renderTable()});el(\'chips\').innerHTML=DATA.modes.map(m=>`<button class="${m.key===mode.key?\'active\':\'\'}" data-mode="${m.key}">${esc(m.name)}</button>`).join(\'\');el(\'chips\').querySelectorAll(\'button\').forEach(b=>b.onclick=()=>{mode=DATA.modes.find(m=>m.key===b.dataset.mode);gw=mode.gw;page=0;renderPitch();renderTable()})}\nel(\'positions\').innerHTML=[\'All\',\'GK\',\'DEF\',\'MID\',\'FWD\'].map(p=>`<button data-pos="${p}" class="${p===\'All\'?\'active\':\'\'}">${p}</button>`).join(\'\')+`<select id="teams" aria-label="Filter teams"><option value="All">All teams</option>${DATA.teams.map(t=>`<option value="${t.id}">${esc(t.name)}</option>`).join(\'\')}</select><button id="columnToggle">⚙ Columns</button>`;el(\'positions\').querySelectorAll(\'[data-pos]\').forEach(b=>b.onclick=()=>{position=b.dataset.pos;el(\'positions\').querySelectorAll(\'[data-pos]\').forEach(n=>n.classList.toggle(\'active\',n===b));page=0;renderTable()});el(\'teams\').onchange=()=>{team=el(\'teams\').value;page=0;renderTable()};el(\'columnToggle\').onclick=()=>el(\'columns\').classList.toggle(\'open\');el(\'columns\').innerHTML=definitions.map(([key,label])=>`<label><input type="checkbox" data-column="${key}" ${visible.has(key)?\'checked\':\'\'} ${key===\'name\'?\'disabled\':\'\'}> ${label}</label>`).join(\'\');el(\'columns\').querySelectorAll(\'input\').forEach(c=>c.onchange=()=>{c.checked?visible.add(c.dataset.column):visible.delete(c.dataset.column);renderTable()});el(\'price\').max=Math.ceil(Math.max(...DATA.players.map(p=>p.price)));el(\'price\').value=el(\'price\').max;[\'search\',\'ownership\',\'price\'].forEach(id=>el(id).oninput=()=>{page=0;renderTable()});el(\'reset\').onclick=()=>{el(\'search\').value=\'\';el(\'ownership\').value=100;el(\'price\').value=el(\'price\').max;position=\'All\';team=\'All\';el(\'teams\').value=\'All\';el(\'positions\').querySelectorAll(\'[data-pos]\').forEach(b=>b.classList.toggle(\'active\',b.dataset.pos===\'All\'));page=0;renderTable()};el(\'prev\').onclick=()=>{page--;renderTable()};el(\'next\').onclick=()=>{page++;renderTable()};el(\'pitchToggle\').onclick=()=>{el(\'pitch\').classList.remove(\'hidden\');el(\'pitchtable\').classList.add(\'hidden\');el(\'pitchToggle\').classList.add(\'active\');el(\'tableToggle\').classList.remove(\'active\')};el(\'tableToggle\').onclick=()=>{el(\'pitch\').classList.add(\'hidden\');el(\'pitchtable\').classList.remove(\'hidden\');el(\'tableToggle\').classList.add(\'active\');el(\'pitchToggle\').classList.remove(\'active\')};renderPitch();renderTable();\n\nfunction playerPhoto(p,css){if(!p)return \'\';const urls=[...new Set([...(p.photo_urls||[]),p.photo].filter(Boolean))];return `<img class="${css}" data-player-photo="${p.id}" data-photo-index="0" src="${esc(urls[0]||\'\')}" alt="${esc(p.name)}">`}\ndocument.addEventListener(\'error\',e=>{const img=e.target;if(!(img instanceof HTMLImageElement)||!img.hasAttribute(\'data-player-photo\'))return;const p=byId.get(Number(img.dataset.playerPhoto));const urls=[...new Set([...(p?.photo_urls||[]),p?.photo].filter(Boolean))];const next=Number(img.dataset.photoIndex||0)+1;img.dataset.photoIndex=String(next);if(next<urls.length){img.src=urls[next]}else{img.style.visibility=\'hidden\'}},true);\n\n\n</script></html>'
    """Interactive live-data player table and reference pitch. Shared canonical squad data."""
    import base64, json, html, math
    from pathlib import Path
    from IPython.display import display, HTML

    def vx_preview_image(path):
        if not path or not Path(path).is_file():return ''
        from PIL import Image, ImageOps
        import io
        pic=Image.open(path).convert('RGBA');pic.thumbnail((600,600),Image.Resampling.LANCZOS)
        out=io.BytesIO();pic.save(out,format='PNG')
        return 'data:image/png;base64,'+base64.b64encode(out.getvalue()).decode()

    _boot_teams={int(t['id']):t for t in _bootstrap['teams']}
    _scores={(int(r.player_id),int(r.gameweek)):float(r.gameweek_expected_points) for r in _VX_GW_SCORES.itertuples()}
    _weeks=sorted({gw for _,gw in _scores})
    from vortex_player_images import official_photo_urls
    _photo_season=next(iter(VX_DATA_CONTRACTS.values()),{}).get('season','')
    _player_data=[]
    for _p in _bootstrap['elements']:
        _tid=int(_p['team']);_team=_boot_teams[_tid];_pid=int(_p['id']);_fx={}
        for _gw in _weeks:
            _f=VX_WC_ENGINE.fdr.for_team(_tid,_gw,{1:'GK',2:'DEF',3:'MID',4:'FWD'}[_p['element_type']])
            _fx[str(_gw)]=[{'opponent':f['opponent'],'home':f['home'],'fdr':f.get('fdr')} for f in _f]
        _media=globals().get('VX_PLAYER_MEDIA',{}).get(_pid,{}) or globals().get('VX_WC_PLAYER_MEDIA',{}).get(_pid,{})
        _player_data.append({'id':_pid,'photo_code':int(_p['code']),'photo_urls':official_photo_urls(_p,_photo_season),'name':_p['web_name'],'fullName':(_p['first_name']+' '+_p['second_name']).strip(),'club':_team['short_name'],'team':_tid,'teamName':_team['name'],'position':{1:'GK',2:'DEF',3:'MID',4:'FWD'}[_p['element_type']],'price':float(_p['now_cost'])/10,'ownership':float(_p.get('selected_by_percent') or 0),'points':int(_p.get('total_points') or 0),'net':int(_p.get('transfers_in_event') or 0)-int(_p.get('transfers_out_event') or 0),'priceChange':float(_p.get('cost_change_event') or 0)/10,'expected':{str(g):_scores.get((_pid,g)) for g in _weeks},'flagged':_p.get('status')!='a','news':_p.get('news') or '', 'photo':vx_preview_image(_media.get('photo')) or '','fixtures':_fx})
    _modes=[]
    for _key,_d in VX_DATA_CONTRACTS.items():
        _cp=_d.get('captain_picks',{});_plans=_d.get('rotation_plan') or [{'gameweek':_d['gameweek'],'xi_ids':_d['starting_xi_ids'],'bench_ids':_d['bench_ids']}]
        _modes.append({'key':_key,'name':('Wildcard' if _d['chip']=='wildcard' else 'Free Hit')+(' Manual' if 'manual' in _key else ''),'gw':_d['gameweek'],'plans':_plans,'captain':(_cp.get('captain') or {}).get('id'),'vice':(_cp.get('vice_captain') or {}).get('id'),'budget':_d['budget_total'],'used':_d['budget_used'],'ft':_d.get('free_transfers') if (_d.get('free_transfers') or 0)>=0 else None})
    _badges={t['short_name']:vx_preview_image(globals().get('VX_CLUB_MEDIA',{}).get(t['short_name'],{}).get('path')) for t in _bootstrap['teams']}
    if 'VX_CARD_VIEWS' not in globals():
        VX_CARD_VIEWS=VXWV.build_player_card_views(_bootstrap,VX_WC_ENGINE.fdr,VX_ANALYSIS_ROOT/'current_source_cache')
    _card_contract=next(iter(VX_DATA_CONTRACTS.values()),{})
    _payload={'players':_player_data,'modes':_modes,'weeks':_weeks,'badges':_badges,'logo':vx_preview_image(VX_ASSET_PATHS.get('logo.png')),'teams':[{'id':t['id'],'name':t['name']} for t in _bootstrap['teams']]}
    _template=Path(__file__).with_name('reference_planner_template.html').read_text() if False else VX_PLANNER_TEMPLATE
    _data=json.dumps(_payload,ensure_ascii=False,allow_nan=False).replace('<','\\u003c')
    _html=_template.replace('__PLANNER_DATA__',_data)
    VX_PITCH_VIEWER_OUT=VX_INPUT_ROOT/'wildcard_freehit_pitch_viewer.html'
    VX_PITCH_VIEWER_OUT.write_text(_html,encoding='utf-8')
    display(HTML(_html));print('Interactive player table and pitch:',VX_PITCH_VIEWER_OUT)
print("✅ CELL 16 COMPLETE")

In [ ]:
#@title 🟢 CELL 17 - VERIFY OUTPUTS FOR THE EXACT CELL 0 SELECTION
from pathlib import Path
import json

_delivery_root = VX_DELIVERY_ROOT
_delivery_manifest = _delivery_root / 'delivery_manifest.json'
_expected_keys = []
if VX_GENERATE_WILDCARD:
    _expected_keys.append('wildcard_manual' if USE_MANUAL_WILDCARD else 'wildcard_auto')
if VX_GENERATE_FREEHIT:
    _expected_keys.append('free_hit_manual' if USE_MANUAL_FREEHIT else 'free_hit_auto')
if not _delivery_manifest.is_file():
    raise RuntimeError(f'Delivery manifest not found: {_delivery_manifest}. Run through Cell 14 first.')
_delivery_data = json.loads(_delivery_manifest.read_text(encoding='utf-8'))
if set(_delivery_data) != set(_expected_keys):
    raise RuntimeError(f'Delivery manifest is for {sorted(_delivery_data)}; selected chips are {sorted(_expected_keys)}. Rerun Cell 14.')
_missing_exports = []
for _chip_key in _expected_keys:
    _chip_export = _delivery_data[_chip_key]
    if _chip_export['mode'] != VX_VIDEO_MODE:
        raise RuntimeError(f'{_chip_key}: manifest is {_chip_export["mode"]}, but Cell 0 requests {VX_VIDEO_MODE}. Rerun Cell 14.')
    _video_paths = [Path(_chip_export['final_mp4'])] + [Path(p) for p in _chip_export['individual_mp4s']]
    _media_audit = json.loads((VX_ANALYSIS_ROOT / (_chip_key + '_media_audit.json')).read_text(encoding='utf-8'))
    _expected_seconds = sum(media_duration(p) for p in _video_paths[1:]) + float(_media_audit.get('opening_duration', 0)) + float(_media_audit.get('outro_duration', 0))
    _actual_seconds = media_duration(_video_paths[0])
    if abs(_actual_seconds - _expected_seconds) > 2.0:
        raise RuntimeError(f'{_chip_key}: combined duration {_actual_seconds:.2f}s does not match scene total {_expected_seconds:.2f}s. Rerun Cell 14.')
    print(f'Combined duration verified: {_actual_seconds / 60:.2f} minutes')
    _png_paths = [Path(p) for p in _chip_export.get('slide_pngs_4k', [])]
    _txt_paths = [Path(p) for p in _chip_export.get('slide_txts', [])]
    _combined_txt = Path(_chip_export['combined_txt'])
    _mp3_paths=[Path(p) for p in _chip_export.get('slide_mp3s',[])]
    _combined_mp3=Path(_chip_export['combined_mp3'])
    _png_count=4 if VX_VIDEO_MODE=='Final' else 0
    if not (len(_chip_export['individual_mp4s'])==len(_txt_paths)==len(_mp3_paths)==4 and len(_png_paths)==_png_count):
        raise RuntimeError(f'{_chip_key}: expected four scene MP4s/MP3s/TXTs, and {_png_count} 4K PNGs.')
    print(f'\n=== {_chip_key.upper()} / GW{_chip_export["gameweek"]} / {_chip_export["mode"]} ===')
    print('COMBINED MP4:', _chip_export['final_mp4'])
    print('COMBINED TXT:', _combined_txt)
    for _slide_number, (_mp4_path, _mp3_path, _txt_path) in enumerate(zip(_video_paths[1:],_mp3_paths,_txt_paths),1):
        print(f'SLIDE {_slide_number}: MP4={_mp4_path.name}, MP3={_mp3_path.name}, TXT={_txt_path.name}')
    if _png_paths:print('4K PNGs:',*[p.name for p in _png_paths],sep='\n  ')
    print('COMBINED MP3:',_combined_mp3)
    for _required_path in _video_paths + _png_paths + _txt_paths + _mp3_paths + [_combined_txt,_combined_mp3]:
        if not _required_path.is_file() or _required_path.stat().st_size == 0:
            _missing_exports.append(str(_required_path))
    for _png_path in _png_paths:
        if _png_path.is_file():
            from PIL import Image
            with Image.open(_png_path) as _slide_image:
                if _slide_image.size != (3840, 2160):
                    _missing_exports.append(str(_png_path) + ' (not 3840x2160)')
if _missing_exports:
    raise RuntimeError('Missing/invalid files:\n'+'\n'.join(_missing_exports))
print('\n✅ Mode-correct PNG exports, four animated MP4s, four MP3s, four text files and all combined exports verified.')
print('Google Drive folder:', _delivery_root)

In [ ]:
# Temporary inspection of the exported Wildcard slides
import json,copy,shutil
from pathlib import Path
from PIL import Image as PILImage,ImageDraw
from IPython.display import Image as DisplayImage,display
_vx_i_key=next(k for k in VX_DATA_CONTRACTS if k.startswith('wildcard'))
_vx_i_scenes=json.loads((VX_ANALYSIS_ROOT/(_vx_i_key+'_scene_manifest.json')).read_text())
_vx_i_root=VX_ANALYSIS_ROOT/'wildcard_graphics_inspection';_vx_i_root.mkdir(exist_ok=True)
_vx_i_frames=[]
for scene in _vx_i_scenes:
    if scene['visual_type'] not in ('wc_squad_plan','wc_captaincy_final'):continue
    cues=scene['narration_timeline']
    if scene['visual_type']=='wc_squad_plan':times=[12.0]
    else:times=[min(c['end']-.1,c['start']+2.0) for c in cues if c['type'] in ('candidate','week','outro')][:4]+[cues[-1]['start']+2.0]
    print(scene['scene_id'],scene['visual_type'],'timeline types',sorted({c['type'] for c in cues}))
    for i,at in enumerate(times):
        target=_vx_i_root/(scene['scene_id']+'_'+str(i)+'.png')
        run_ffmpeg(['-ss',at,'-i',scene['media_file'],'-frames:v','1',target])
        frame=PILImage.open(target).convert('RGB');frame=frame.resize((960,540));ImageDraw.Draw(frame).text((10,510),scene['scene_id']+' '+str(round(at,1))+'s',fill='yellow');_vx_i_frames.append(frame)
_vx_i_sheet=PILImage.new('RGB',(1920,540*((len(_vx_i_frames)+1)//2)))
for i,frame in enumerate(_vx_i_frames):_vx_i_sheet.paste(frame,((i%2)*960,(i//2)*540))
_vx_i_sheet_path=VX_ANALYSIS_ROOT/'Wildcard Masterplan Captaincy Before.png';_vx_i_sheet.save(_vx_i_sheet_path)
_vx_i_capture=dict(scenes=_vx_i_scenes,contract=VX_DATA_CONTRACTS[_vx_i_key],masterplan_source=_VX_MASTERPLAN_SOURCE,project_root=str(VX_MASTERPLAN_PROJECT),captaincy_module=Path(__import__('wildcard_final_slide').__file__).read_text())
(VX_ANALYSIS_ROOT/'Wildcard Graphics Inspection.json').write_text(json.dumps(_vx_i_capture,ensure_ascii=False,default=str))
print('INSPECTION WRITTEN:',_vx_i_sheet_path)
display(DisplayImage(filename=str(_vx_i_sheet_path),width=1280))
